In [ ]:
from pathlib import Path

ROOT = Path("/Users/olusolaolabanjo/Desktop/eeg-data")
EEG = ROOT / "sub-001" / "eeg"

SET_FILE = EEG / "sub-001_task-activeai_eeg.set"
FDT_FILE = EEG / "sub-001_task-activeai_eeg.fdt"

for f in [SET_FILE, FDT_FILE]:
    print(f.name)
    print("  exists :", f.exists())
    print("  symlink:", f.is_symlink())

    if f.exists():
        print(
            "  size   :",
            f"{f.stat().st_size / (1024**2):.2f} MB"
        )

In [ ]:
# pip install mne

In [ ]:
# ================================================================
# TASK 1A — ROBUST EEGLAB LOADER
# Bypasses malformed/incompatible embedded EEG.event structures
#
# Dataset: OpenNeuro ds008496
# Subject: sub-001
# Task: activeai
#
# IMPORTANT:
#   - Reads the actual .set metadata
#   - Reads the actual external .fdt signal
#   - Does NOT use embedded EEGLAB events
#   - Uses BIDS events.tsv separately
#   - Performs NO preprocessing
# ================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy
from scipy.io import loadmat

import mne

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"

EEG_DIR = (
    ROOT
    / f"sub-{SUBJECT}"
    / "eeg"
)

SET_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.set"
)

FDT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.fdt"
)

EVENT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_events.tsv"
)

CHANNEL_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_channels.tsv"
)

EEG_JSON = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.json"
)


# ================================================================
# 2. VERIFY FILES
# ================================================================

print("=" * 80)
print("ROBUST EEGLAB LOADER")
print("=" * 80)

for path in [
    SET_FILE,
    FDT_FILE,
    EVENT_FILE,
    CHANNEL_FILE,
    EEG_JSON
]:

    print(
        f"{path.name:<50}",
        "OK" if path.exists() else "MISSING",
        (
            f"{path.stat().st_size / 1024**2:.2f} MB"
            if path.exists()
            else ""
        )
    )

    if not path.exists():
        raise FileNotFoundError(path)


# ================================================================
# 3. LOAD BIDS JSON
# ================================================================

with open(
    EEG_JSON,
    "r"
) as f:

    eeg_json = json.load(f)


print("\n" + "=" * 80)
print("BIDS EEG METADATA")
print("=" * 80)

for key in [
    "TaskName",
    "SamplingFrequency",
    "EEGChannelCount",
    "RecordingDuration",
    "PowerLineFrequency",
    "EEGReference"
]:

    if key in eeg_json:
        print(
            f"{key:25s}: {eeg_json[key]}"
        )


# ================================================================
# 4. LOAD CHANNEL + EVENT TABLES
# ================================================================

channels_df = pd.read_csv(
    CHANNEL_FILE,
    sep="\t"
)

events_df = pd.read_csv(
    EVENT_FILE,
    sep="\t"
)


print("\n" + "=" * 80)
print("CHANNEL TABLE")
print("=" * 80)

display(channels_df)


print("\n" + "=" * 80)
print("BIDS EVENT TABLE")
print("=" * 80)

display(events_df)


# ================================================================
# 5. READ EEGLAB .SET USING SCIPY
#
# This avoids MNE's embedded EEG.event parser.
# ================================================================

print("\n" + "=" * 80)
print("READING EEGLAB .SET STRUCTURE")
print("=" * 80)

mat = loadmat(
    SET_FILE,
    squeeze_me=True,
    struct_as_record=False
)


print("Top-level MAT keys:")
print(
    [
        key
        for key in mat.keys()
        if not key.startswith("__")
    ]
)


# ================================================================
# 6. FIND EEG STRUCTURE
# ================================================================

if "EEG" in mat:

    EEG = mat["EEG"]

    print(
        "\nEEG structure found."
    )

else:

    # Some EEGLAB .set files save EEG fields
    # directly at the top level.

    EEG = None

    print(
        "\nNo top-level 'EEG' object."
    )

    print(
        "Checking for direct EEGLAB fields..."
    )


# ================================================================
# 7. HELPER FOR READING EEGLAB FIELDS
# ================================================================

def eeg_field(name, default=None):

    if EEG is not None:

        if hasattr(EEG, name):
            return getattr(EEG, name)

    if name in mat:
        return mat[name]

    return default


# ================================================================
# 8. EXTRACT ESSENTIAL EEGLAB PARAMETERS
# ================================================================

nbchan = eeg_field(
    "nbchan"
)

pnts = eeg_field(
    "pnts"
)

trials = eeg_field(
    "trials",
    1
)

srate = eeg_field(
    "srate"
)

data_field = eeg_field(
    "data"
)


# Convert safely to scalar values

nbchan = int(
    np.asarray(nbchan).squeeze()
)

pnts = int(
    np.asarray(pnts).squeeze()
)

trials = int(
    np.asarray(trials).squeeze()
)

srate = float(
    np.asarray(srate).squeeze()
)


print("\n" + "=" * 80)
print("EEGLAB SIGNAL PARAMETERS")
print("=" * 80)

print(
    f"nbchan : {nbchan}"
)

print(
    f"pnts   : {pnts:,}"
)

print(
    f"trials : {trials}"
)

print(
    f"srate  : {srate:.2f} Hz"
)

print(
    f"data   : {data_field}"
)


# ================================================================
# 9. INSPECT EMBEDDED EVENT STRUCTURE
#
# This is diagnostic only.
# We WILL NOT use it for experimental conditions.
# ================================================================

embedded_events = eeg_field(
    "event"
)


print("\n" + "=" * 80)
print("EMBEDDED EEGLAB EVENT DIAGNOSTIC")
print("=" * 80)


if embedded_events is None:

    print(
        "No embedded EEG.event structure found."
    )

else:

    embedded_array = np.atleast_1d(
        embedded_events
    )

    print(
        "Number of embedded event objects:",
        len(embedded_array)
    )

    print(
        "\nFirst event objects:"
    )

    for i, ev in enumerate(
        embedded_array[:10]
    ):

        print(
            f"\nEvent {i}:"
        )

        if hasattr(ev, "_fieldnames"):

            print(
                "Fields:",
                ev._fieldnames
            )

            for field in ev._fieldnames:

                try:
                    value = getattr(
                        ev,
                        field
                    )

                    print(
                        f"  {field}: {value}"
                    )

                except Exception:
                    pass

        else:

            print(
                type(ev),
                ev
            )


# ================================================================
# 10. DETERMINE FDT SIGNAL SHAPE
# ================================================================

expected_values = (
    nbchan
    * pnts
    * trials
)


fdt_bytes = (
    FDT_FILE.stat().st_size
)


print("\n" + "=" * 80)
print("FDT BINARY CHECK")
print("=" * 80)

print(
    f"FDT size: "
    f"{fdt_bytes:,} bytes"
)

print(
    f"Expected scalar values: "
    f"{expected_values:,}"
)


# EEGLAB normally stores external .fdt as float32.

expected_float32_bytes = (
    expected_values
    * 4
)


expected_float64_bytes = (
    expected_values
    * 8
)


print(
    f"Expected float32 size: "
    f"{expected_float32_bytes:,} bytes"
)

print(
    f"Expected float64 size: "
    f"{expected_float64_bytes:,} bytes"
)


# ================================================================
# 11. DETERMINE BINARY PRECISION
# ================================================================

if fdt_bytes == expected_float32_bytes:

    dtype = np.dtype("<f4")

    print(
        "\nDetected format: float32"
    )

elif fdt_bytes == expected_float64_bytes:

    dtype = np.dtype("<f8")

    print(
        "\nDetected format: float64"
    )

else:

    raise RuntimeError(
        "\nFDT file size does not match the "
        "dimensions stored in the .set file.\n\n"
        f"Actual bytes   : {fdt_bytes:,}\n"
        f"Float32 expect : {expected_float32_bytes:,}\n"
        f"Float64 expect : {expected_float64_bytes:,}\n\n"
        "Stop here rather than guessing the "
        "binary layout."
    )


# ================================================================
# 12. READ EXTERNAL FDT SIGNAL
# ================================================================

print("\nReading external FDT signal...")


flat = np.fromfile(
    FDT_FILE,
    dtype=dtype
)


print(
    f"Values read: {flat.size:,}"
)


if flat.size != expected_values:

    raise RuntimeError(
        "Unexpected number of samples in FDT."
    )


# ================================================================
# 13. RESHAPE EEGLAB DATA
#
# EEGLAB external FDT data are stored with
# channels varying fastest.
#
# Equivalent MATLAB shape:
#
#     nbchan × pnts × trials
#
# Therefore use Fortran ordering.
# ================================================================

if trials == 1:

    eeg_data_uv = flat.reshape(
        (
            nbchan,
            pnts
        ),
        order="F"
    )

else:

    eeg_data_uv = flat.reshape(
        (
            nbchan,
            pnts,
            trials
        ),
        order="F"
    )


    # This study should contain continuous recordings.
    # Do not silently concatenate epoched data.

    raise RuntimeError(
        f"Recording contains {trials} trials. "
        "This loader expected continuous data."
    )


print(
    "Signal matrix shape:",
    eeg_data_uv.shape
)


# ================================================================
# 14. CHANNEL NAMES
# ================================================================

channel_names = []


# First preference:
# BIDS channels.tsv

if "name" in channels_df.columns:

    channel_names = (
        channels_df["name"]
        .astype(str)
        .tolist()
    )


# If channels.tsv uses another capitalization
elif "Name" in channels_df.columns:

    channel_names = (
        channels_df["Name"]
        .astype(str)
        .tolist()
    )


# Ensure channel count agrees

if len(channel_names) != nbchan:

    print(
        "\nWARNING:"
    )

    print(
        f"channels.tsv contains "
        f"{len(channel_names)} names "
        f"but EEGLAB reports {nbchan} channels."
    )

    print(
        "Attempting to recover labels "
        "from EEG.chanlocs."
    )


    channel_names = []

    chanlocs = eeg_field(
        "chanlocs"
    )


    if chanlocs is not None:

        for ch in np.atleast_1d(
            chanlocs
        ):

            if hasattr(
                ch,
                "labels"
            ):

                channel_names.append(
                    str(
                        ch.labels
                    )
                )


if len(channel_names) != nbchan:

    raise RuntimeError(
        "\nCould not establish a reliable "
        "one-to-one channel-name mapping.\n"
        f"Signal channels: {nbchan}\n"
        f"Recovered names: {len(channel_names)}"
    )


print("\nChannel names:")
print(channel_names)


# ================================================================
# 15. CONVERT MICROVOLTS -> VOLTS FOR MNE
#
# EEGLAB EEG amplitudes are conventionally stored in µV.
# MNE RawArray expects volts.
# ================================================================

eeg_data_v = (
    eeg_data_uv
    * 1e-6
)


# ================================================================
# 16. CREATE MNE INFO
# ================================================================

info = mne.create_info(

    ch_names=channel_names,

    sfreq=srate,

    ch_types=[
        "eeg"
    ] * nbchan
)


# ================================================================
# 17. CREATE MNE RAWARRAY
#
# No embedded EEGLAB events are parsed.
# ================================================================

raw = mne.io.RawArray(

    eeg_data_v,

    info,

    verbose=False
)


# ================================================================
# 18. ADD STANDARD SENSOR LOCATIONS
# ================================================================

montage = (
    mne.channels
    .make_standard_montage(
        "standard_1020"
    )
)


raw.set_montage(

    montage,

    match_case=False,

    on_missing="warn",

    verbose=False
)


# ================================================================
# 19. BASIC SIGNAL VALIDATION
# ================================================================

duration_sec = (
    raw.n_times
    / raw.info["sfreq"]
)


print("\n" + "=" * 80)
print("MNE RAW OBJECT CREATED")
print("=" * 80)

print(raw)

print(
    f"\nChannels : {len(raw.ch_names)}"
)

print(
    f"Samples  : {raw.n_times:,}"
)

print(
    f"Rate     : {raw.info['sfreq']:.2f} Hz"
)

print(
    f"Duration : {duration_sec:.2f} s"
)

print(
    f"Duration : {duration_sec / 60:.2f} min"
)


# ================================================================
# 20. AMPLITUDE SANITY CHECK
# ================================================================

check_uv = (
    raw.get_data()
    * 1e6
)


print("\n" + "=" * 80)
print("AMPLITUDE SANITY CHECK")
print("=" * 80)

print(
    "Median absolute amplitude:",
    f"{np.median(np.abs(check_uv)):.3f} µV"
)

print(
    "1st percentile:",
    f"{np.percentile(check_uv, 1):.3f} µV"
)

print(
    "99th percentile:",
    f"{np.percentile(check_uv, 99):.3f} µV"
)

print(
    "Minimum:",
    f"{np.min(check_uv):.3f} µV"
)

print(
    "Maximum:",
    f"{np.max(check_uv):.3f} µV"
)


# ================================================================
# 21. QUICK 10-SECOND SIGNAL VISUALIZATION
# ================================================================

plot_start = 30.0
plot_duration = 10.0

start_sample = int(
    plot_start
    * srate
)

stop_sample = int(
    (
        plot_start
        + plot_duration
    )
    * srate
)


segment = (
    raw.get_data(
        start=start_sample,
        stop=stop_sample
    )
    * 1e6
)


times = (
    np.arange(
        segment.shape[1]
    )
    / srate
    + plot_start
)


robust_amp = np.percentile(
    np.abs(segment),
    95
)


spacing = max(
    robust_amp * 3,
    50
)


fig, ax = plt.subplots(
    figsize=(15, 9)
)


for i, ch in enumerate(
    raw.ch_names
):

    offset = (
        (
            len(raw.ch_names)
            - 1
            - i
        )
        * spacing
    )

    ax.plot(
        times,
        segment[i] + offset,
        linewidth=0.6
    )


ax.set_yticks(
    np.arange(
        len(raw.ch_names)
    )[::-1]
    * spacing
)


ax.set_yticklabels(
    raw.ch_names
)


ax.set_xlabel(
    "Time (s)"
)


ax.set_ylabel(
    "EEG channel"
)


ax.set_title(
    "Raw EEG loaded directly from EEGLAB FDT — sub-001"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()
plt.show()


# ================================================================
# 22. BIDS EVENT SUMMARY
#
# These events — NOT malformed embedded EEG.event objects —
# will be used for subsequent experimental segmentation.
# ================================================================

print("\n" + "=" * 80)
print("BIDS EVENTS TO BE USED FOR ANALYSIS")
print("=" * 80)


display(
    events_df
)


if "type" in events_df.columns:

    print(
        "\nEvent counts:"
    )

    display(
        events_df["type"]
        .astype(str)
        .value_counts()
        .rename_axis("event_type")
        .reset_index(name="count")
    )


# ================================================================
# 23. SUCCESS
# ================================================================

print("\n" + "=" * 80)
print("TASK 1A COMPLETE")
print("=" * 80)

print(
    "\nThe EEG signal was loaded WITHOUT "
    "using MNE's EEGLAB event parser."
)

print(
    "\nExperimental events will come from:"
)

print(
    EVENT_FILE
)

print(
    "\nNo filtering, ICA, rereferencing, "
    "interpolation, or artifact rejection "
    "has been performed."
)

In [ ]:
# ================================================================
# TASK 1B — EEG-ONLY RAW QC + MARKER STREAM AUDIT
# ================================================================
#
# Dataset: OpenNeuro ds008496
# Subject: sub-001
# Task: activeai
#
# PURPOSE
# -------
# 1. Load the .set/.fdt without MNE's broken event parser
# 2. Separate the 14 actual EEG channels from 44 auxiliary streams
# 3. Verify EEG amplitude scale
# 4. Examine raw traces
# 5. Examine EEG-only PSD
# 6. Inspect 50-Hz line noise
# 7. Inspect marker streams for possible experimental boundaries
# 8. Inspect BIDS events.tsv
# 9. Save QC tables and figures
#
# NO PREPROCESSING IS PERFORMED.
# ================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.signal import welch

warnings.filterwarnings("ignore")


# ================================================================
# 1. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"

EEG_NAMES = [
    "AF3", "F7", "F3", "FC5",
    "T7", "P7", "O1", "O2",
    "P8", "T8", "FC6", "F4",
    "F8", "AF4"
]

MARKER_NAMES = [
    "MarkerIndex",
    "MarkerType",
    "MarkerValueInt",
    "MARKER_HARDWARE"
]

TRACE_START = 30.0
TRACE_DURATION = 20.0


# ================================================================
# 2. PATHS
# ================================================================

EEG_DIR = (
    ROOT
    / f"sub-{SUBJECT}"
    / "eeg"
)

SET_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.set"
)

FDT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.fdt"
)

CHANNEL_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_channels.tsv"
)

EVENT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_events.tsv"
)

EEG_JSON = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.json"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task01b_eeg_marker_audit"
    / f"sub-{SUBJECT}"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. VERIFY FILES
# ================================================================

for path in [
    SET_FILE,
    FDT_FILE,
    CHANNEL_FILE,
    EVENT_FILE,
    EEG_JSON
]:
    if not path.exists():
        raise FileNotFoundError(path)


# ================================================================
# 4. LOAD METADATA
# ================================================================

channels = pd.read_csv(
    CHANNEL_FILE,
    sep="\t"
)

events = pd.read_csv(
    EVENT_FILE,
    sep="\t"
)

with open(
    EEG_JSON,
    "r"
) as f:
    metadata = json.load(f)


sfreq_json = float(
    metadata.get(
        "SamplingFrequency",
        256
    )
)

line_frequency = metadata.get(
    "PowerLineFrequency",
    None
)


print("=" * 80)
print("TASK 1B — EEG-ONLY RAW QC + MARKER AUDIT")
print("=" * 80)

print(
    f"Subject              : sub-{SUBJECT}"
)

print(
    f"Task                 : {TASK}"
)

print(
    f"Metadata sampling    : {sfreq_json:.2f} Hz"
)

print(
    f"Power-line frequency : {line_frequency} Hz"
)


# ================================================================
# 5. LOAD EEGLAB HEADER
# ================================================================

mat = loadmat(
    SET_FILE,
    squeeze_me=True,
    struct_as_record=False
)


def field(name, default=None):
    return mat.get(name, default)


nbchan = int(
    np.asarray(
        field("nbchan")
    ).squeeze()
)

pnts = int(
    np.asarray(
        field("pnts")
    ).squeeze()
)

trials = int(
    np.asarray(
        field("trials", 1)
    ).squeeze()
)

sfreq = float(
    np.asarray(
        field("srate")
    ).squeeze()
)


if trials != 1:
    raise RuntimeError(
        f"Expected continuous data, "
        f"but trials={trials}"
    )


# ================================================================
# 6. CHANNEL NAMES
# ================================================================

if "name" not in channels.columns:
    raise RuntimeError(
        "channels.tsv has no 'name' column."
    )

channel_names = (
    channels["name"]
    .astype(str)
    .tolist()
)


if len(channel_names) != nbchan:
    raise RuntimeError(
        f"channels.tsv has {len(channel_names)} channels "
        f"but .set reports {nbchan}."
    )


# ================================================================
# 7. LOAD FDT
# ================================================================

expected_values = (
    nbchan * pnts
)

expected_bytes = (
    expected_values * 4
)

actual_bytes = (
    FDT_FILE.stat().st_size
)


if actual_bytes != expected_bytes:
    raise RuntimeError(
        "FDT byte count does not match "
        "float32 EEGLAB dimensions."
    )


flat = np.fromfile(
    FDT_FILE,
    dtype="<f4"
)


data = flat.reshape(
    (
        nbchan,
        pnts
    ),
    order="F"
)


duration = (
    pnts / sfreq
)


print("\n" + "=" * 80)
print("SIGNAL DIMENSIONS")
print("=" * 80)

print(
    f"All streams : {nbchan}"
)

print(
    f"Samples     : {pnts:,}"
)

print(
    f"Sampling    : {sfreq:.2f} Hz"
)

print(
    f"Duration    : {duration:.2f} s"
)

print(
    f"Duration    : {duration/60:.2f} min"
)


# ================================================================
# 8. EXTRACT ONLY THE 14 EEG CHANNELS
# ================================================================

missing_eeg = [
    ch
    for ch in EEG_NAMES
    if ch not in channel_names
]

if missing_eeg:
    raise RuntimeError(
        f"Missing EEG channels: {missing_eeg}"
    )


eeg_indices = [
    channel_names.index(ch)
    for ch in EEG_NAMES
]


eeg_uv = data[
    eeg_indices,
    :
].astype(
    np.float64
)


print("\nEEG indices:")
for ch, idx in zip(
    EEG_NAMES,
    eeg_indices
):
    print(
        f"{ch:4s} -> stream {idx}"
    )


# ================================================================
# 9. EEG-ONLY AMPLITUDE QC
# ================================================================

qc_rows = []


for i, ch in enumerate(
    EEG_NAMES
):

    x = eeg_uv[i]

    qc_rows.append({
        "channel":
            ch,

        "mean_uV":
            np.mean(x),

        "median_uV":
            np.median(x),

        "sd_uV":
            np.std(x),

        "rms_uV":
            np.sqrt(
                np.mean(x ** 2)
            ),

        "p01_uV":
            np.percentile(
                x,
                1
            ),

        "p99_uV":
            np.percentile(
                x,
                99
            ),

        "minimum_uV":
            np.min(x),

        "maximum_uV":
            np.max(x),

        "peak_to_peak_uV":
            np.ptp(x)
    })


qc = pd.DataFrame(
    qc_rows
)


print("\n" + "=" * 80)
print("EEG-ONLY RAW AMPLITUDE QC")
print("=" * 80)

display(
    qc.round(3)
)


print(
    "\nGLOBAL EEG-ONLY AMPLITUDE SUMMARY"
)

print(
    f"Median |amplitude| : "
    f"{np.median(np.abs(eeg_uv)):.3f} µV"
)

print(
    f"1st percentile     : "
    f"{np.percentile(eeg_uv,1):.3f} µV"
)

print(
    f"99th percentile    : "
    f"{np.percentile(eeg_uv,99):.3f} µV"
)

print(
    f"Minimum            : "
    f"{np.min(eeg_uv):.3f} µV"
)

print(
    f"Maximum            : "
    f"{np.max(eeg_uv):.3f} µV"
)


qc.to_csv(
    OUT / "01_eeg_only_raw_qc.csv",
    index=False
)


# ================================================================
# 10. FIGURE 1 — EEG-ONLY RAW TRACES
# ================================================================

start = int(
    TRACE_START * sfreq
)

stop = int(
    min(
        (
            TRACE_START
            + TRACE_DURATION
        ) * sfreq,
        pnts
    )
)

segment = eeg_uv[
    :,
    start:stop
]


times = (
    np.arange(
        segment.shape[1]
    )
    / sfreq
    + TRACE_START
)


robust_scale = np.percentile(
    np.abs(
        segment
        - np.median(
            segment,
            axis=1,
            keepdims=True
        )
    ),
    95
)


if (
    not np.isfinite(robust_scale)
    or robust_scale <= 0
):
    robust_scale = 50


spacing = max(
    robust_scale * 3,
    50
)


fig, ax = plt.subplots(
    figsize=(15, 10)
)


for i, ch in enumerate(
    EEG_NAMES
):

    centered = (
        segment[i]
        - np.median(
            segment[i]
        )
    )

    offset = (
        (
            len(EEG_NAMES)
            - 1
            - i
        )
        * spacing
    )

    ax.plot(
        times,
        centered + offset,
        linewidth=0.55
    )


ax.set_yticks(
    np.arange(
        len(EEG_NAMES)
    )[::-1]
    * spacing
)

ax.set_yticklabels(
    EEG_NAMES
)

ax.set_xlabel(
    "Time (s)"
)

ax.set_ylabel(
    "EEG channel"
)

ax.set_title(
    "Raw EEG-only traces — sub-001"
)

ax.grid(
    axis="x",
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    OUT / "02_raw_eeg_only_traces.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 11. FIGURE 2 — CHANNEL RMS
# ================================================================

fig, ax = plt.subplots(
    figsize=(11, 6)
)

ax.bar(
    qc["channel"],
    qc["rms_uV"]
)

ax.set_xlabel(
    "EEG channel"
)

ax.set_ylabel(
    "RMS amplitude (µV)"
)

ax.set_title(
    "Raw EEG-only channel RMS amplitude"
)

ax.tick_params(
    axis="x",
    rotation=45
)

ax.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    OUT / "03_eeg_channel_rms.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 12. COMPUTE EEG-ONLY PSD
# ================================================================

nperseg = min(
    int(
        sfreq * 8
    ),
    pnts
)


freqs, psd = welch(
    eeg_uv,
    fs=sfreq,
    nperseg=nperseg,
    noverlap=nperseg // 2,
    axis=1,
    detrend="constant"
)


# µV² / Hz -> dB
psd_db = (
    10
    * np.log10(
        np.maximum(
            psd,
            np.finfo(float).eps
        )
    )
)


# ================================================================
# 13. FIGURE 3 — PSD BY CHANNEL
# ================================================================

mask = (
    (freqs >= 0.5)
    &
    (freqs <= 80)
)


fig, ax = plt.subplots(
    figsize=(13, 7)
)


for i, ch in enumerate(
    EEG_NAMES
):

    ax.plot(
        freqs[mask],
        psd_db[i, mask],
        linewidth=0.8,
        alpha=0.75,
        label=ch
    )


if (
    line_frequency is not None
    and float(line_frequency) <= 80
):

    ax.axvline(
        float(line_frequency),
        linestyle="--",
        linewidth=1.2,
        label=(
            f"{line_frequency} Hz "
            "power line"
        )
    )


ax.set_xlabel(
    "Frequency (Hz)"
)

ax.set_ylabel(
    "PSD (dB µV²/Hz)"
)

ax.set_title(
    "Raw EEG power spectral density by channel"
)

ax.grid(
    alpha=0.2
)

ax.legend(
    ncol=2,
    fontsize=8
)

plt.tight_layout()

plt.savefig(
    OUT / "04_raw_psd_by_channel.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 14. FIGURE 4 — MEAN PSD
# ================================================================

mean_psd = np.mean(
    psd_db,
    axis=0
)

sem_psd = (
    np.std(
        psd_db,
        axis=0,
        ddof=1
    )
    / np.sqrt(
        len(EEG_NAMES)
    )
)


fig, ax = plt.subplots(
    figsize=(12, 6)
)


ax.plot(
    freqs[mask],
    mean_psd[mask],
    linewidth=1.5
)


ax.fill_between(
    freqs[mask],
    (
        mean_psd
        - sem_psd
    )[mask],
    (
        mean_psd
        + sem_psd
    )[mask],
    alpha=0.2
)


if (
    line_frequency is not None
    and float(line_frequency) <= 80
):

    ax.axvline(
        float(line_frequency),
        linestyle="--",
        linewidth=1.2,
        label=(
            f"{line_frequency} Hz "
            "power line"
        )
    )

    ax.legend()


ax.set_xlabel(
    "Frequency (Hz)"
)

ax.set_ylabel(
    "PSD (dB µV²/Hz)"
)

ax.set_title(
    "Mean raw EEG power spectral density"
)

ax.grid(
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    OUT / "05_mean_raw_psd.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 15. QUANTIFY POWER AROUND 50 HZ
# ================================================================

line_noise_rows = []


if line_frequency is not None:

    lf = float(
        line_frequency
    )

    line_mask = (
        (freqs >= lf - 1)
        &
        (freqs <= lf + 1)
    )

    lower_mask = (
        (freqs >= lf - 5)
        &
        (freqs <= lf - 2)
    )

    upper_mask = (
        (freqs >= lf + 2)
        &
        (freqs <= lf + 5)
    )

    neighbor_mask = (
        lower_mask
        |
        upper_mask
    )


    for i, ch in enumerate(
        EEG_NAMES
    ):

        line_db = np.mean(
            psd_db[
                i,
                line_mask
            ]
        )

        neighbor_db = np.mean(
            psd_db[
                i,
                neighbor_mask
            ]
        )

        line_noise_rows.append({
            "channel":
                ch,

            "line_frequency_hz":
                lf,

            "line_band_db":
                line_db,

            "neighbor_band_db":
                neighbor_db,

            "line_excess_db":
                line_db
                - neighbor_db
        })


line_noise = pd.DataFrame(
    line_noise_rows
)


if not line_noise.empty:

    print("\n" + "=" * 80)
    print("POWER-LINE NOISE AUDIT")
    print("=" * 80)

    display(
        line_noise.round(3)
    )

    line_noise.to_csv(
        OUT / "06_line_noise_audit.csv",
        index=False
    )


# ================================================================
# 16. EXTRACT MARKER STREAMS
# ================================================================

print("\n" + "=" * 80)
print("MARKER STREAM AUDIT")
print("=" * 80)


marker_results = []


for marker in MARKER_NAMES:

    if marker not in channel_names:

        print(
            f"{marker}: NOT FOUND"
        )

        continue


    idx = channel_names.index(
        marker
    )

    x = data[
        idx,
        :
    ].astype(
        np.float64
    )


    finite = x[
        np.isfinite(x)
    ]


    if finite.size == 0:

        print(
            f"{marker}: no finite values"
        )

        continue


    unique_values, counts = (
        np.unique(
            finite,
            return_counts=True
        )
    )


    print(
        f"\n{marker}"
    )

    print(
        f"  index      : {idx}"
    )

    print(
        f"  min        : {finite.min()}"
    )

    print(
        f"  max        : {finite.max()}"
    )

    print(
        f"  unique     : {len(unique_values)}"
    )

    print(
        f"  nonzero    : "
        f"{np.count_nonzero(finite)}"
    )


    # Show most frequent values
    order = np.argsort(
        counts
    )[::-1]


    print(
        "  most frequent values:"
    )


    for j in order[:10]:

        print(
            f"    "
            f"{unique_values[j]} "
            f"-> {counts[j]:,}"
        )


    marker_results.append({
        "marker":
            marker,

        "stream_index":
            idx,

        "minimum":
            finite.min(),

        "maximum":
            finite.max(),

        "n_unique":
            len(unique_values),

        "n_nonzero":
            np.count_nonzero(
                finite
            )
    })


marker_summary = pd.DataFrame(
    marker_results
)


display(
    marker_summary
)


marker_summary.to_csv(
    OUT / "07_marker_stream_summary.csv",
    index=False
)


# ================================================================
# 17. DETECT CHANGES IN MARKER STREAMS
# ================================================================

change_rows = []


for marker in MARKER_NAMES:

    if marker not in channel_names:
        continue


    idx = channel_names.index(
        marker
    )

    x = data[
        idx,
        :
    ].astype(
        np.float64
    )


    # Samples at which marker value changes
    changes = (
        np.where(
            np.diff(x) != 0
        )[0]
        + 1
    )


    print(
        f"\n{marker}: "
        f"{len(changes):,} value transitions"
    )


    # Store first 1000 transitions
    for sample in changes[:1000]:

        change_rows.append({
            "marker":
                marker,

            "sample":
                int(sample),

            "time_sec":
                sample / sfreq,

            "old_value":
                x[sample - 1],

            "new_value":
                x[sample]
        })


marker_changes = pd.DataFrame(
    change_rows
)


print("\nFIRST MARKER TRANSITIONS")

if len(marker_changes):

    display(
        marker_changes.head(
            100
        )
    )

else:

    print(
        "No marker transitions detected."
    )


marker_changes.to_csv(
    OUT / "08_marker_transitions.csv",
    index=False
)


# ================================================================
# 18. FIGURE 5 — MARKER STREAMS ACROSS RECORDING
# ================================================================

available_markers = [
    marker
    for marker in MARKER_NAMES
    if marker in channel_names
]


if available_markers:

    fig, axes = plt.subplots(
        len(available_markers),
        1,
        figsize=(
            15,
            2.5 * len(
                available_markers
            )
        ),
        sharex=True
    )


    if len(
        available_markers
    ) == 1:

        axes = [
            axes
        ]


    time_minutes = (
        np.arange(
            pnts
        )
        / sfreq
        / 60
    )


    for ax, marker in zip(
        axes,
        available_markers
    ):

        idx = channel_names.index(
            marker
        )

        x = data[
            idx,
            :
        ]

        ax.plot(
            time_minutes,
            x,
            linewidth=0.7
        )

        ax.set_ylabel(
            marker
        )

        ax.grid(
            alpha=0.2
        )


    axes[-1].set_xlabel(
        "Recording time (minutes)"
    )


    fig.suptitle(
        "Raw marker streams — sub-001",
        y=1.01
    )


    plt.tight_layout()


    plt.savefig(
        OUT / "09_marker_streams.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 19. BIDS EVENT AUDIT
# ================================================================

print("\n" + "=" * 80)
print("BIDS EVENT AUDIT")
print("=" * 80)


print(
    "Columns:",
    list(
        events.columns
    )
)


print(
    "Number of rows:",
    len(events)
)


display(
    events
)


if "onset" in events.columns:

    event_onsets = pd.to_numeric(
        events["onset"],
        errors="coerce"
    )


    print(
        "\nValid onset values:"
    )

    print(
        event_onsets.dropna().tolist()
    )


# ================================================================
# 20. COMPARE BIDS EVENT WITH MARKER TRANSITIONS
# ================================================================

if (
    "onset" in events.columns
    and
    len(marker_changes) > 0
):

    bids_onsets = (
        pd.to_numeric(
            events["onset"],
            errors="coerce"
        )
        .dropna()
        .to_numpy()
    )


    comparisons = []


    for onset in bids_onsets:

        temp = marker_changes.copy()

        temp["distance_sec"] = (
            np.abs(
                temp["time_sec"]
                - onset
            )
        )


        nearest = (
            temp.sort_values(
                "distance_sec"
            )
            .iloc[0]
        )


        comparisons.append({
            "bids_onset_sec":
                onset,

            "nearest_marker":
                nearest[
                    "marker"
                ],

            "nearest_transition_sec":
                nearest[
                    "time_sec"
                ],

            "difference_sec":
                nearest[
                    "distance_sec"
                ],

            "old_value":
                nearest[
                    "old_value"
                ],

            "new_value":
                nearest[
                    "new_value"
                ]
        })


    comparison_df = pd.DataFrame(
        comparisons
    )


    print(
        "\nBIDS EVENT ↔ NEAREST "
        "MARKER TRANSITION"
    )


    display(
        comparison_df
    )


    comparison_df.to_csv(
        OUT
        / "10_bids_marker_comparison.csv",
        index=False
    )


# ================================================================
# 21. SAVE RECORDING SUMMARY
# ================================================================

summary = pd.DataFrame(
    [{
        "subject":
            f"sub-{SUBJECT}",

        "task":
            TASK,

        "total_streams":
            nbchan,

        "eeg_channels":
            len(
                EEG_NAMES
            ),

        "auxiliary_streams":
            nbchan
            - len(
                EEG_NAMES
            ),

        "samples":
            pnts,

        "sampling_frequency_hz":
            sfreq,

        "duration_seconds":
            duration,

        "duration_minutes":
            duration / 60,

        "power_line_frequency_hz":
            line_frequency,

        "bids_event_rows":
            len(
                events
            ),

        "global_eeg_p01_uV":
            np.percentile(
                eeg_uv,
                1
            ),

        "global_eeg_p99_uV":
            np.percentile(
                eeg_uv,
                99
            ),

        "global_eeg_min_uV":
            np.min(
                eeg_uv
            ),

        "global_eeg_max_uV":
            np.max(
                eeg_uv
            )
    }]
)


summary.to_csv(
    OUT / "11_recording_summary.csv",
    index=False
)


print("\n" + "=" * 80)
print("TASK 1B COMPLETE")
print("=" * 80)

display(
    summary.T
)

print(
    "\nOutput directory:"
)

print(
    OUT
)

print(
    "\nNo EEG preprocessing has been performed."
)

In [ ]:
# ================================================================
# TASK 1C — RECONSTRUCT EXPERIMENTAL MARKER TABLE
# ================================================================
#
# Dataset: OpenNeuro ds008496
# Subject: sub-001
# Task: activeai
#
# PURPOSE
# -------
# Recover the 14 actual marker pulses from:
#
#   MarkerIndex
#   MarkerType
#   MarkerValueInt
#   MARKER_HARDWARE
#
# For each pulse:
#   • sample
#   • onset time
#   • MarkerIndex
#   • MarkerType
#   • MarkerValueInt
#   • pulse duration
#   • interval from previous marker
#
# Also:
#   • compares markers with BIDS events.tsv
#   • examines marker sequences
#   • visualizes marker timing
#
# NO EEG PREPROCESSING IS PERFORMED.
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"


# ================================================================
# 3. PATHS
# ================================================================

EEG_DIR = (
    ROOT
    / f"sub-{SUBJECT}"
    / "eeg"
)

SET_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.set"
)

FDT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.fdt"
)

CHANNEL_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_channels.tsv"
)

EVENT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_events.tsv"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task01c_marker_decoding"
    / f"sub-{SUBJECT}"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 4. VERIFY FILES
# ================================================================

for f in [
    SET_FILE,
    FDT_FILE,
    CHANNEL_FILE,
    EVENT_FILE
]:

    if not f.exists():

        raise FileNotFoundError(
            f
        )


# ================================================================
# 5. LOAD EEGLAB HEADER
# ================================================================

mat = loadmat(
    SET_FILE,
    squeeze_me=True,
    struct_as_record=False
)


nbchan = int(
    np.asarray(
        mat["nbchan"]
    ).squeeze()
)

pnts = int(
    np.asarray(
        mat["pnts"]
    ).squeeze()
)

sfreq = float(
    np.asarray(
        mat["srate"]
    ).squeeze()
)


# ================================================================
# 6. LOAD CHANNEL TABLE
# ================================================================

channels = pd.read_csv(
    CHANNEL_FILE,
    sep="\t"
)

channel_names = (
    channels["name"]
    .astype(str)
    .tolist()
)


if len(channel_names) != nbchan:

    raise RuntimeError(
        "Channel count mismatch."
    )


# ================================================================
# 7. LOAD FDT
# ================================================================

flat = np.fromfile(
    FDT_FILE,
    dtype="<f4"
)


expected = (
    nbchan
    * pnts
)


if flat.size != expected:

    raise RuntimeError(
        f"Expected {expected:,} values "
        f"but found {flat.size:,}."
    )


data = flat.reshape(
    (
        nbchan,
        pnts
    ),
    order="F"
)


# ================================================================
# 8. EXTRACT MARKER STREAMS
# ================================================================

required_markers = [
    "MarkerIndex",
    "MarkerType",
    "MarkerValueInt",
    "MARKER_HARDWARE"
]


for marker in required_markers:

    if marker not in channel_names:

        raise RuntimeError(
            f"{marker} is missing."
        )


def stream(name):

    idx = channel_names.index(
        name
    )

    return data[
        idx,
        :
    ].astype(
        np.float64
    )


marker_index = stream(
    "MarkerIndex"
)

marker_type = stream(
    "MarkerType"
)

marker_value = stream(
    "MarkerValueInt"
)

marker_hardware = stream(
    "MARKER_HARDWARE"
)


# ================================================================
# 9. IDENTIFY MARKER PULSES
#
# MarkerValueInt is zero between markers and non-zero at marker
# samples. We identify contiguous non-zero runs so that a pulse is
# counted once even if it spans multiple samples.
# ================================================================

active = (
    (marker_index != 0)
    |
    (marker_type != 0)
    |
    (marker_value != 0)
    |
    (marker_hardware != 0)
)


# Rising edge of each active run
starts = np.where(

    active
    &
    np.concatenate(
        (
            [True],
            ~active[:-1]
        )
    )

)[0]


# Falling edge
ends = np.where(

    active
    &
    np.concatenate(
        (
            ~active[1:],
            [True]
        )
    )

)[0]


if len(starts) != len(ends):

    raise RuntimeError(
        "Marker pulse start/end mismatch."
    )


print("=" * 80)
print("TASK 1C — EXPERIMENTAL MARKER RECONSTRUCTION")
print("=" * 80)

print(
    f"Marker pulses detected : {len(starts)}"
)

print(
    f"Sampling frequency     : {sfreq:.2f} Hz"
)

print(
    f"Recording duration     : {pnts/sfreq:.2f} s"
)


# ================================================================
# 10. BUILD ONE ROW PER MARKER
# ================================================================

rows = []


for event_number, (
    start,
    end
) in enumerate(
    zip(
        starts,
        ends
    ),
    start=1
):

    # Find sample in pulse carrying the strongest marker
    pulse_samples = np.arange(
        start,
        end + 1
    )


    # Marker streams here appear to be single-sample pulses,
    # but using the maximum absolute value makes this robust.
    representative = pulse_samples[
        np.argmax(
            np.abs(
                marker_index[
                    pulse_samples
                ]
            )
            +
            np.abs(
                marker_type[
                    pulse_samples
                ]
            )
            +
            np.abs(
                marker_value[
                    pulse_samples
                ]
            )
        )
    ]


    rows.append({

        "event_number":
            event_number,

        "start_sample":
            int(start),

        "end_sample":
            int(end),

        "representative_sample":
            int(representative),

        "onset_sec":
            representative
            / sfreq,

        "onset_min":
            representative
            / sfreq
            / 60,

        "pulse_samples":
            int(
                end
                - start
                + 1
            ),

        "pulse_duration_ms":
            (
                end
                - start
                + 1
            )
            / sfreq
            * 1000,

        "MarkerIndex":
            marker_index[
                representative
            ],

        "MarkerType":
            marker_type[
                representative
            ],

        "MarkerValueInt":
            marker_value[
                representative
            ],

        "MARKER_HARDWARE":
            marker_hardware[
                representative
            ]
    })


markers = pd.DataFrame(
    rows
)


# ================================================================
# 11. TIME BETWEEN MARKERS
# ================================================================

markers[
    "interval_from_previous_sec"
] = (
    markers[
        "onset_sec"
    ]
    .diff()
)


markers[
    "interval_to_next_sec"
] = (
    markers[
        "onset_sec"
    ]
    .shift(-1)
    -
    markers[
        "onset_sec"
    ]
)


# ================================================================
# 12. CREATE COMBINED MARKER CODE
# ================================================================

markers[
    "marker_code"
] = (

    "T"
    +
    markers[
        "MarkerType"
    ]
    .astype(int)
    .astype(str)

    +

    "_V"

    +

    markers[
        "MarkerValueInt"
    ]
    .astype(int)
    .astype(str)
)


# ================================================================
# 13. DISPLAY COMPLETE MARKER TABLE
# ================================================================

print("\n" + "=" * 80)
print("COMPLETE MARKER TABLE")
print("=" * 80)


display(
    markers.round(4)
)


# ================================================================
# 14. MARKER TYPE/VALUE COMBINATIONS
# ================================================================

combination_counts = (

    markers
    .groupby(
        [
            "MarkerType",
            "MarkerValueInt",
            "marker_code"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        [
            "MarkerType",
            "MarkerValueInt"
        ]
    )
)


print("\n" + "=" * 80)
print("MARKER CODE FREQUENCIES")
print("=" * 80)


display(
    combination_counts
)


# ================================================================
# 15. MARKER INDEX SEQUENCE
# ================================================================

print("\n" + "=" * 80)
print("MARKER INDEX SEQUENCE")
print("=" * 80)


print(
    markers[
        "MarkerIndex"
    ]
    .astype(int)
    .tolist()
)


print("\nMarker type sequence:")

print(
    markers[
        "MarkerType"
    ]
    .astype(int)
    .tolist()
)


print("\nMarker value sequence:")

print(
    markers[
        "MarkerValueInt"
    ]
    .astype(int)
    .tolist()
)


print("\nCombined code sequence:")

print(
    markers[
        "marker_code"
    ]
    .tolist()
)


# ================================================================
# 16. LOAD BIDS EVENTS
# ================================================================

bids_events = pd.read_csv(
    EVENT_FILE,
    sep="\t"
)


print("\n" + "=" * 80)
print("ORIGINAL BIDS EVENTS.TSV")
print("=" * 80)


display(
    bids_events
)


# ================================================================
# 17. MATCH EACH BIDS EVENT TO RAW MARKER
# ================================================================

matches = []


if "onset" in bids_events.columns:

    bids_onsets = pd.to_numeric(

        bids_events[
            "onset"
        ],

        errors="coerce"
    )


    for bids_row, onset in enumerate(
        bids_onsets
    ):

        if pd.isna(
            onset
        ):
            continue


        differences = np.abs(

            markers[
                "onset_sec"
            ].to_numpy()

            - onset
        )


        nearest_position = int(
            np.argmin(
                differences
            )
        )


        nearest = markers.iloc[
            nearest_position
        ]


        matches.append({

            "bids_row":
                bids_row,

            "bids_onset_sec":
                onset,

            "raw_event_number":
                int(
                    nearest[
                        "event_number"
                    ]
                ),

            "raw_onset_sec":
                nearest[
                    "onset_sec"
                ],

            "difference_sec":
                differences[
                    nearest_position
                ],

            "MarkerIndex":
                nearest[
                    "MarkerIndex"
                ],

            "MarkerType":
                nearest[
                    "MarkerType"
                ],

            "MarkerValueInt":
                nearest[
                    "MarkerValueInt"
                ],

            "marker_code":
                nearest[
                    "marker_code"
                ]
        })


bids_matches = pd.DataFrame(
    matches
)


print("\n" + "=" * 80)
print("BIDS ↔ RAW MARKER MATCH")
print("=" * 80)


if len(
    bids_matches
):

    display(
        bids_matches.round(6)
    )

else:

    print(
        "No BIDS events could be matched."
    )


# ================================================================
# 18. FIGURE 1 — ALL 14 MARKERS ON RECORDING TIMELINE
# ================================================================

fig, ax = plt.subplots(
    figsize=(15, 6)
)


y = markers[
    "MarkerValueInt"
].to_numpy()


ax.scatter(

    markers[
        "onset_min"
    ],

    y,

    s=90,

    zorder=3
)


for _, row in markers.iterrows():

    ax.annotate(

        (
            f"{int(row['MarkerIndex'])}\n"
            f"T{int(row['MarkerType'])}"
        ),

        (
            row[
                "onset_min"
            ],
            row[
                "MarkerValueInt"
            ]
        ),

        xytext=(
            0,
            10
        ),

        textcoords="offset points",

        ha="center",

        fontsize=8
    )


ax.set_xlabel(
    "Recording time (minutes)"
)

ax.set_ylabel(
    "MarkerValueInt"
)

ax.set_title(
    "Experimental marker sequence — sub-001"
)

ax.set_yticks(
    sorted(
        markers[
            "MarkerValueInt"
        ]
        .astype(int)
        .unique()
    )
)

ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    / "01_experimental_marker_sequence.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 19. FIGURE 2 — INTER-MARKER INTERVALS
# ================================================================

interval_data = markers.dropna(
    subset=[
        "interval_from_previous_sec"
    ]
)


fig, ax = plt.subplots(
    figsize=(13, 6)
)


ax.bar(

    interval_data[
        "event_number"
    ].astype(str),

    interval_data[
        "interval_from_previous_sec"
    ]
)


ax.set_xlabel(
    "Marker event number"
)

ax.set_ylabel(
    "Seconds since previous marker"
)

ax.set_title(
    "Intervals between experimental markers"
)

ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    / "02_inter_marker_intervals.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 20. FIGURE 3 — MARKER TYPE × VALUE
# ================================================================

fig, ax = plt.subplots(
    figsize=(12, 6)
)


for marker_type_value in sorted(
    markers[
        "MarkerType"
    ]
    .unique()
):

    subset = markers[
        markers[
            "MarkerType"
        ]
        ==
        marker_type_value
    ]


    ax.scatter(

        subset[
            "onset_min"
        ],

        subset[
            "MarkerValueInt"
        ],

        s=100,

        label=(
            f"MarkerType "
            f"{int(marker_type_value)}"
        )
    )


ax.set_xlabel(
    "Recording time (minutes)"
)

ax.set_ylabel(
    "MarkerValueInt"
)

ax.set_title(
    "Marker type and value across recording"
)

ax.legend()

ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    / "03_marker_type_value_timeline.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 21. SAVE TABLES
# ================================================================

markers.to_csv(

    OUT
    / "reconstructed_markers.csv",

    index=False
)


combination_counts.to_csv(

    OUT
    / "marker_code_frequencies.csv",

    index=False
)


if len(
    bids_matches
):

    bids_matches.to_csv(

        OUT
        / "bids_raw_marker_matches.csv",

        index=False
    )


# ================================================================
# 22. PRINT POTENTIAL SEGMENTS
#
# IMPORTANT:
# These are purely temporal intervals.
# We DO NOT assign AI/no-AI labels yet.
# ================================================================

segments = []


for i in range(
    len(markers) - 1
):

    start = markers.iloc[
        i
    ]

    end = markers.iloc[
        i + 1
    ]


    segments.append({

        "segment":
            i + 1,

        "start_event":
            int(
                start[
                    "event_number"
                ]
            ),

        "end_event":
            int(
                end[
                    "event_number"
                ]
            ),

        "start_marker_code":
            start[
                "marker_code"
            ],

        "end_marker_code":
            end[
                "marker_code"
            ],

        "start_sec":
            start[
                "onset_sec"
            ],

        "end_sec":
            end[
                "onset_sec"
            ],

        "duration_sec":
            (
                end[
                    "onset_sec"
                ]
                -
                start[
                    "onset_sec"
                ]
            )
    })


segments = pd.DataFrame(
    segments
)


print("\n" + "=" * 80)
print("TEMPORAL SEGMENTS BETWEEN MARKERS")
print("=" * 80)


display(
    segments.round(3)
)


segments.to_csv(

    OUT
    / "temporal_segments_unlabelled.csv",

    index=False
)


# ================================================================
# 23. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("TASK 1C COMPLETE")
print("=" * 80)


print(
    f"Reconstructed marker pulses : "
    f"{len(markers)}"
)


print(
    f"Unique MarkerType values    : "
    f"{sorted(markers['MarkerType'].unique())}"
)


print(
    f"Unique MarkerValueInt       : "
    f"{sorted(markers['MarkerValueInt'].unique())}"
)


print(
    f"Temporal intervals          : "
    f"{len(segments)}"
)


print(
    "\nFiles saved to:"
)

print(
    OUT
)


print(
    "\nIMPORTANT:"
)

print(
    "The intervals have NOT been assigned "
    "experimental condition labels."
)

print(
    "We must decode the marker semantics "
    "before calling any interval AI, no-AI, "
    "active, passive, eyes-open, or eyes-closed."
)

In [ ]:
# ================================================================
# TASK 1D — LOCATE AND DISPLAY EXPERIMENT CONFIGURATION
# ================================================================

from pathlib import Path
import json

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

print("=" * 80)
print("TASK 1D — EXPERIMENT CONFIGURATION AUDIT")
print("=" * 80)

# ------------------------------------------------
# Find potentially relevant configuration/protocol files
# ------------------------------------------------

patterns = [
    "*experiment*.json",
    "*config*.json",
    "*task*.json",
    "*event*.json",
    "*README*",
    "*readme*"
]

found = []

for pattern in patterns:
    found.extend(
        ROOT.rglob(pattern)
    )

# Remove duplicates
found = sorted(
    set(found)
)

print("\nPotential protocol/configuration files:\n")

for i, path in enumerate(found, start=1):
    try:
        rel = path.relative_to(ROOT)
    except Exception:
        rel = path

    print(
        f"{i:02d}. {rel}"
    )


# ------------------------------------------------
# Find experiment_config.json specifically
# ------------------------------------------------

config_files = list(
    ROOT.rglob(
        "experiment_config.json"
    )
)

print("\n" + "=" * 80)
print("EXPERIMENT_CONFIG.JSON")
print("=" * 80)

if not config_files:

    print(
        "\nexperiment_config.json was not found "
        "under the dataset root."
    )

else:

    for config_file in config_files:

        print(
            f"\nFILE:\n{config_file}\n"
        )

        try:

            with open(
                config_file,
                "r",
                encoding="utf-8"
            ) as f:

                config = json.load(f)

            print(
                json.dumps(
                    config,
                    indent=2,
                    ensure_ascii=False
                )
            )

        except Exception as e:

            print(
                "Could not parse as JSON:"
            )

            print(
                e
            )

            print(
                "\nRaw text follows:\n"
            )

            print(
                config_file.read_text(
                    encoding="utf-8",
                    errors="replace"
                )
            )


# ------------------------------------------------
# Also display root README if available
# ------------------------------------------------

print("\n" + "=" * 80)
print("ROOT README")
print("=" * 80)

readme_candidates = [
    ROOT / "README",
    ROOT / "README.md",
    ROOT / "README.txt"
]

readme_found = False

for readme in readme_candidates:

    if readme.exists():

        readme_found = True

        print(
            f"\nFILE:\n{readme}\n"
        )

        print(
            readme.read_text(
                encoding="utf-8",
                errors="replace"
            )
        )

if not readme_found:

    print(
        "No root README found."
    )


# ------------------------------------------------
# Display task/event JSON files
# ------------------------------------------------

print("\n" + "=" * 80)
print("EVENT/TASK JSON FILES")
print("=" * 80)

json_candidates = sorted(
    set(
        list(ROOT.rglob("*events.json"))
        +
        list(ROOT.rglob("*task*.json"))
    )
)

for path in json_candidates:

    print(
        "\n" + "-" * 80
    )

    print(
        path
    )

    print(
        "-" * 80
    )

    try:

        with open(
            path,
            "r",
            encoding="utf-8"
        ) as f:

            obj = json.load(f)

        print(
            json.dumps(
                obj,
                indent=2,
                ensure_ascii=False
            )
        )

    except Exception as e:

        print(
            f"Could not parse: {e}"
        )


print("\n" + "=" * 80)
print("TASK 1D COMPLETE")
print("=" * 80)

In [ ]:
# ================================================================
# TASK 1E — ALL-SUBJECT RECORDING + MARKER AUDIT
# ================================================================
#
# Dataset: OpenNeuro ds008496
#
# PURPOSE
# -------
# Audit all 15 participants BEFORE EEG preprocessing.
#
# For every participant:
#   1. Determine activeai/passiveai group
#   2. Read recording dimensions
#   3. Identify EEG and auxiliary channels
#   4. Reconstruct raw Emotiv marker pulses
#   5. Extract MarkerIndex / MarkerType / MarkerValueInt
#   6. Compare BIDS events.tsv with raw markers
#   7. Summarize marker sequences
#   8. Compare active vs passive marker structure
#
# IMPORTANT
# ---------
# This cell does NOT:
#   - filter EEG
#   - rereference EEG
#   - run ICA
#   - interpolate channels
#   - assign AI/no-AI labels to numeric markers
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat

warnings.filterwarnings("ignore")


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECTS = [
    f"{i:03d}"
    for i in range(1, 16)
]

EEG_NAMES = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]

MARKER_NAMES = [
    "MarkerIndex",
    "MarkerType",
    "MarkerValueInt",
    "MARKER_HARDWARE"
]


# ================================================================
# 3. OUTPUT DIRECTORY
# ================================================================

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task01e_all_subject_marker_audit"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 4. HELPER — DETERMINE TASK
# ================================================================

def get_task(subject):

    if int(subject) <= 8:
        return "activeai"

    return "passiveai"


# ================================================================
# 5. HELPER — SAFE NUMBER
# ================================================================

def safe_int(x):

    try:
        return int(
            np.asarray(x).squeeze()
        )

    except Exception:
        return np.nan


def safe_float(x):

    try:
        return float(
            np.asarray(x).squeeze()
        )

    except Exception:
        return np.nan


# ================================================================
# 6. STORAGE
# ================================================================

recording_rows = []

marker_rows = []

sequence_rows = []

bids_rows = []

error_rows = []


# ================================================================
# 7. LOOP THROUGH ALL SUBJECTS
# ================================================================

print("=" * 90)
print("TASK 1E — ALL-SUBJECT RECORDING + MARKER AUDIT")
print("=" * 90)


for subject in SUBJECTS:

    task = get_task(
        subject
    )

    group = (
        "Active"
        if task == "activeai"
        else "Passive"
    )

    print(
        f"\nProcessing sub-{subject} "
        f"({task}) ..."
    )


    # ------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------

    eeg_dir = (
        ROOT
        / f"sub-{subject}"
        / "eeg"
    )

    set_file = (
        eeg_dir
        / f"sub-{subject}_task-{task}_eeg.set"
    )

    fdt_file = (
        eeg_dir
        / f"sub-{subject}_task-{task}_eeg.fdt"
    )

    channel_file = (
        eeg_dir
        / f"sub-{subject}_task-{task}_channels.tsv"
    )

    event_file = (
        eeg_dir
        / f"sub-{subject}_task-{task}_events.tsv"
    )

    json_file = (
        eeg_dir
        / f"sub-{subject}_task-{task}_eeg.json"
    )


    # ------------------------------------------------------------
    # Verify required files
    # ------------------------------------------------------------

    required = [
        set_file,
        fdt_file,
        channel_file,
        json_file
    ]

    missing = [
        str(f)
        for f in required
        if not f.exists()
    ]


    if missing:

        print(
            "  MISSING FILES"
        )

        error_rows.append({
            "subject":
                f"sub-{subject}",

            "task":
                task,

            "error":
                "Missing required file(s)",

            "details":
                " | ".join(
                    missing
                )
        })

        continue


    # ------------------------------------------------------------
    # Process subject
    # ------------------------------------------------------------

    try:

        # ========================================================
        # 8. LOAD JSON
        # ========================================================

        with open(
            json_file,
            "r"
        ) as f:

            meta = json.load(f)


        # ========================================================
        # 9. LOAD CHANNEL TABLE
        # ========================================================

        channels = pd.read_csv(
            channel_file,
            sep="\t"
        )


        if "name" not in channels.columns:

            raise RuntimeError(
                "channels.tsv has no name column"
            )


        channel_names = (
            channels["name"]
            .astype(str)
            .tolist()
        )


        # ========================================================
        # 10. LOAD EEGLAB HEADER
        # ========================================================

        mat = loadmat(
            set_file,
            squeeze_me=True,
            struct_as_record=False
        )


        nbchan = safe_int(
            mat.get(
                "nbchan"
            )
        )

        pnts = safe_int(
            mat.get(
                "pnts"
            )
        )

        trials = safe_int(
            mat.get(
                "trials",
                1
            )
        )

        sfreq = safe_float(
            mat.get(
                "srate"
            )
        )


        if trials != 1:

            raise RuntimeError(
                f"Expected continuous data; "
                f"trials={trials}"
            )


        if len(channel_names) != nbchan:

            raise RuntimeError(
                f"Channel mismatch: "
                f"TSV={len(channel_names)}, "
                f"SET={nbchan}"
            )


        # ========================================================
        # 11. CHECK FDT
        # ========================================================

        expected_values = (
            nbchan
            * pnts
        )

        expected_bytes = (
            expected_values
            * 4
        )

        actual_bytes = (
            fdt_file.stat().st_size
        )


        if actual_bytes != expected_bytes:

            raise RuntimeError(
                f"FDT size mismatch: "
                f"{actual_bytes:,} vs "
                f"{expected_bytes:,}"
            )


        # ========================================================
        # 12. LOAD FDT
        # ========================================================

        flat = np.fromfile(
            fdt_file,
            dtype="<f4"
        )


        if flat.size != expected_values:

            raise RuntimeError(
                "Unexpected number "
                "of FDT values"
            )


        data = flat.reshape(
            (
                nbchan,
                pnts
            ),
            order="F"
        )


        duration = (
            pnts
            / sfreq
        )


        # ========================================================
        # 13. EEG CHANNEL CHECK
        # ========================================================

        available_eeg = [
            ch
            for ch in EEG_NAMES
            if ch in channel_names
        ]

        missing_eeg = [
            ch
            for ch in EEG_NAMES
            if ch not in channel_names
        ]


        # ========================================================
        # 14. RECORDING SUMMARY
        # ========================================================

        recording_rows.append({

            "subject":
                f"sub-{subject}",

            "group":
                group,

            "task":
                task,

            "nbchan":
                nbchan,

            "eeg_channels":
                len(
                    available_eeg
                ),

            "misc_channels":
                nbchan
                - len(
                    available_eeg
                ),

            "missing_eeg":
                ",".join(
                    missing_eeg
                ),

            "samples":
                pnts,

            "sfreq_hz":
                sfreq,

            "duration_sec":
                duration,

            "duration_min":
                duration
                / 60,

            "json_duration_sec":
                meta.get(
                    "RecordingDuration",
                    np.nan
                ),

            "powerline_hz":
                meta.get(
                    "PowerLineFrequency",
                    np.nan
                )
        })


        # ========================================================
        # 15. EXTRACT MARKER STREAMS
        # ========================================================

        marker_available = {

            name:
                (
                    name
                    in channel_names
                )

            for name in MARKER_NAMES
        }


        if not all(
            marker_available.values()
        ):

            print(
                "  Some marker streams missing:",
                marker_available
            )


        def get_stream(name):

            if name not in channel_names:

                return np.zeros(
                    pnts,
                    dtype=float
                )

            idx = channel_names.index(
                name
            )

            return data[
                idx,
                :
            ].astype(
                np.float64
            )


        marker_index = get_stream(
            "MarkerIndex"
        )

        marker_type = get_stream(
            "MarkerType"
        )

        marker_value = get_stream(
            "MarkerValueInt"
        )

        marker_hardware = get_stream(
            "MARKER_HARDWARE"
        )


        # ========================================================
        # 16. DETECT ACTIVE MARKER PULSES
        # ========================================================

        active = (
            (marker_index != 0)
            |
            (marker_type != 0)
            |
            (marker_value != 0)
            |
            (marker_hardware != 0)
        )


        starts = np.where(

            active
            &
            np.concatenate(
                (
                    [True],
                    ~active[:-1]
                )
            )

        )[0]


        ends = np.where(

            active
            &
            np.concatenate(
                (
                    ~active[1:],
                    [True]
                )
            )

        )[0]


        if len(starts) != len(ends):

            raise RuntimeError(
                "Marker pulse start/end mismatch"
            )


        # ========================================================
        # 17. RECONSTRUCT MARKERS
        # ========================================================

        subject_markers = []


        for event_number, (
            start,
            end
        ) in enumerate(
            zip(
                starts,
                ends
            ),
            start=1
        ):

            pulse_samples = np.arange(
                start,
                end + 1
            )


            scores = (
                np.abs(
                    marker_index[
                        pulse_samples
                    ]
                )
                +
                np.abs(
                    marker_type[
                        pulse_samples
                    ]
                )
                +
                np.abs(
                    marker_value[
                        pulse_samples
                    ]
                )
            )


            representative = (
                pulse_samples[
                    np.argmax(
                        scores
                    )
                ]
            )


            row = {

                "subject":
                    f"sub-{subject}",

                "group":
                    group,

                "task":
                    task,

                "event_number":
                    event_number,

                "sample":
                    int(
                        representative
                    ),

                "onset_sec":
                    representative
                    / sfreq,

                "onset_min":
                    representative
                    / sfreq
                    / 60,

                "pulse_samples":
                    int(
                        end
                        - start
                        + 1
                    ),

                "pulse_duration_ms":
                    (
                        end
                        - start
                        + 1
                    )
                    / sfreq
                    * 1000,

                "MarkerIndex":
                    marker_index[
                        representative
                    ],

                "MarkerType":
                    marker_type[
                        representative
                    ],

                "MarkerValueInt":
                    marker_value[
                        representative
                    ],

                "MARKER_HARDWARE":
                    marker_hardware[
                        representative
                    ]
            }


            row[
                "marker_code"
            ] = (
                f"T"
                f"{int(row['MarkerType'])}"
                f"_V"
                f"{int(row['MarkerValueInt'])}"
            )


            marker_rows.append(
                row
            )

            subject_markers.append(
                row
            )


        # ========================================================
        # 18. SUBJECT MARKER SEQUENCES
        # ========================================================

        index_sequence = [
            int(
                x[
                    "MarkerIndex"
                ]
            )
            for x in subject_markers
        ]

        type_sequence = [
            int(
                x[
                    "MarkerType"
                ]
            )
            for x in subject_markers
        ]

        value_sequence = [
            int(
                x[
                    "MarkerValueInt"
                ]
            )
            for x in subject_markers
        ]

        code_sequence = [
            x[
                "marker_code"
            ]
            for x in subject_markers
        ]


        sequence_rows.append({

            "subject":
                f"sub-{subject}",

            "group":
                group,

            "task":
                task,

            "n_markers":
                len(
                    subject_markers
                ),

            "index_sequence":
                " ".join(
                    map(
                        str,
                        index_sequence
                    )
                ),

            "type_sequence":
                " ".join(
                    map(
                        str,
                        type_sequence
                    )
                ),

            "value_sequence":
                " ".join(
                    map(
                        str,
                        value_sequence
                    )
                ),

            "code_sequence":
                " ".join(
                    code_sequence
                )
        })


        # ========================================================
        # 19. BIDS EVENT AUDIT
        # ========================================================

        if event_file.exists():

            bids = pd.read_csv(
                event_file,
                sep="\t"
            )


            for bids_index, bids_event in (
                bids.iterrows()
            ):

                onset = pd.to_numeric(
                    bids_event.get(
                        "onset",
                        np.nan
                    ),
                    errors="coerce"
                )


                match = {
                    "subject":
                        f"sub-{subject}",

                    "group":
                        group,

                    "task":
                        task,

                    "bids_row":
                        bids_index,

                    "bids_onset_sec":
                        onset,

                    "bids_columns":
                        "|".join(
                            bids.columns
                        )
                }


                if (
                    pd.notna(onset)
                    and
                    len(subject_markers) > 0
                ):

                    marker_times = np.array([
                        x[
                            "onset_sec"
                        ]
                        for x
                        in subject_markers
                    ])


                    nearest_idx = int(
                        np.argmin(
                            np.abs(
                                marker_times
                                - onset
                            )
                        )
                    )


                    nearest = (
                        subject_markers[
                            nearest_idx
                        ]
                    )


                    match.update({

                        "nearest_raw_event":
                            nearest[
                                "event_number"
                            ],

                        "nearest_raw_onset_sec":
                            nearest[
                                "onset_sec"
                            ],

                        "difference_sec":
                            abs(
                                nearest[
                                    "onset_sec"
                                ]
                                - onset
                            ),

                        "nearest_marker_code":
                            nearest[
                                "marker_code"
                            ],

                        "nearest_MarkerIndex":
                            nearest[
                                "MarkerIndex"
                            ]
                    })


                bids_rows.append(
                    match
                )


        print(
            f"  duration = "
            f"{duration/60:.2f} min; "
            f"markers = "
            f"{len(subject_markers)}"
        )


    except Exception as exc:

        print(
            f"  ERROR: {exc}"
        )

        error_rows.append({

            "subject":
                f"sub-{subject}",

            "task":
                task,

            "error":
                type(
                    exc
                ).__name__,

            "details":
                str(
                    exc
                )
        })


# ================================================================
# 20. DATAFRAMES
# ================================================================

recordings = pd.DataFrame(
    recording_rows
)

markers = pd.DataFrame(
    marker_rows
)

sequences = pd.DataFrame(
    sequence_rows
)

bids_audit = pd.DataFrame(
    bids_rows
)

errors = pd.DataFrame(
    error_rows
)


# ================================================================
# 21. RECORDING SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("ALL RECORDINGS")
print("=" * 90)

display(
    recordings.round(3)
)


# ================================================================
# 22. MARKER SEQUENCES — MOST IMPORTANT OUTPUT
# ================================================================

print("\n" + "=" * 90)
print("MARKER SEQUENCES BY PARTICIPANT")
print("=" * 90)

display(
    sequences
)


# ================================================================
# 23. COMPLETE RAW MARKER TABLE
# ================================================================

print("\n" + "=" * 90)
print("ALL RECONSTRUCTED MARKERS")
print("=" * 90)

display(
    markers.round(4)
)


# ================================================================
# 24. GROUP × MARKER COUNTS
# ================================================================

if not markers.empty:

    marker_counts = (

        markers
        .groupby(
            [
                "group",
                "marker_code"
            ]
        )
        .size()
        .reset_index(
            name="count"
        )
    )


    print("\n" + "=" * 90)
    print("MARKER CODE COUNTS BY GROUP")
    print("=" * 90)

    display(
        marker_counts
    )

else:

    marker_counts = (
        pd.DataFrame()
    )


# ================================================================
# 25. MARKER NUMBER BY SUBJECT
# ================================================================

if not sequences.empty:

    print("\n" + "=" * 90)
    print("NUMBER OF MARKERS PER SUBJECT")
    print("=" * 90)


    display(

        sequences[
            [
                "subject",
                "group",
                "n_markers"
            ]
        ]

    )


# ================================================================
# 26. BIDS EVENT AUDIT
# ================================================================

print("\n" + "=" * 90)
print("BIDS EVENT AUDIT")
print("=" * 90)


if not bids_audit.empty:

    display(
        bids_audit.round(6)
    )

else:

    print(
        "No BIDS event rows found."
    )


# ================================================================
# 27. ERRORS
# ================================================================

if not errors.empty:

    print("\n" + "=" * 90)
    print("ERRORS / MISSING DATA")
    print("=" * 90)

    display(
        errors
    )


# ================================================================
# 28. FIGURE 1 — RECORDING DURATION
# ================================================================

if not recordings.empty:

    plot_df = (
        recordings
        .sort_values(
            "subject"
        )
        .copy()
    )


    fig, ax = plt.subplots(
        figsize=(13, 6)
    )


    x = np.arange(
        len(
            plot_df
        )
    )


    ax.bar(
        x,
        plot_df[
            "duration_min"
        ]
    )


    ax.set_xticks(
        x
    )


    ax.set_xticklabels(
        plot_df[
            "subject"
        ],
        rotation=45
    )


    ax.set_ylabel(
        "Recording duration (minutes)"
    )


    ax.set_xlabel(
        "Participant"
    )


    ax.set_title(
        "Recording duration across participants"
    )


    # Active/passive boundary
    if len(plot_df) >= 9:

        ax.axvline(
            7.5,
            linestyle="--",
            linewidth=1
        )


    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "01_recording_duration_by_subject.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 29. FIGURE 2 — NUMBER OF MARKERS
# ================================================================

if not sequences.empty:

    plot_df = (
        sequences
        .sort_values(
            "subject"
        )
        .copy()
    )


    fig, ax = plt.subplots(
        figsize=(13, 6)
    )


    x = np.arange(
        len(
            plot_df
        )
    )


    ax.bar(
        x,
        plot_df[
            "n_markers"
        ]
    )


    ax.set_xticks(
        x
    )


    ax.set_xticklabels(
        plot_df[
            "subject"
        ],
        rotation=45
    )


    ax.set_ylabel(
        "Number of reconstructed markers"
    )


    ax.set_xlabel(
        "Participant"
    )


    ax.set_title(
        "Raw experimental markers across participants"
    )


    if len(plot_df) >= 9:

        ax.axvline(
            7.5,
            linestyle="--",
            linewidth=1
        )


    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "02_marker_count_by_subject.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 30. FIGURE 3 — ALL MARKER TIMELINES
# ================================================================

if not markers.empty:

    subject_order = [
        f"sub-{x}"
        for x in SUBJECTS
    ]


    y_map = {

        subject:
            i

        for i, subject
        in enumerate(
            subject_order
        )
    }


    fig, ax = plt.subplots(
        figsize=(15, 9)
    )


    for _, row in markers.iterrows():

        subject = row[
            "subject"
        ]


        ax.scatter(
            row[
                "onset_min"
            ],
            y_map[
                subject
            ],
            s=45
        )


        ax.annotate(
            row[
                "marker_code"
            ],
            (
                row[
                    "onset_min"
                ],
                y_map[
                    subject
                ]
            ),
            xytext=(
                3,
                3
            ),
            textcoords="offset points",
            fontsize=6
        )


    ax.set_yticks(
        range(
            len(
                subject_order
            )
        )
    )


    ax.set_yticklabels(
        subject_order
    )


    ax.invert_yaxis()


    ax.set_xlabel(
        "Recording time (minutes)"
    )


    ax.set_ylabel(
        "Participant"
    )


    ax.set_title(
        "Reconstructed marker timelines across all participants"
    )


    ax.grid(
        axis="x",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "03_all_subject_marker_timelines.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 31. FIGURE 4 — MARKER VALUE SEQUENCE MATRIX
# ================================================================

if not markers.empty:

    max_events = int(
        markers[
            "event_number"
        ].max()
    )


    matrix = np.full(
        (
            len(
                SUBJECTS
            ),
            max_events
        ),
        np.nan
    )


    for s_idx, subject in enumerate(
        SUBJECTS
    ):

        subject_name = (
            f"sub-{subject}"
        )


        temp = markers[
            markers[
                "subject"
            ]
            ==
            subject_name
        ]


        for _, row in temp.iterrows():

            event_idx = (
                int(
                    row[
                        "event_number"
                    ]
                )
                - 1
            )


            matrix[
                s_idx,
                event_idx
            ] = row[
                "MarkerValueInt"
            ]


    fig, ax = plt.subplots(
        figsize=(14, 8)
    )


    image = ax.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest"
    )


    ax.set_xticks(
        np.arange(
            max_events
        )
    )


    ax.set_xticklabels(
        np.arange(
            1,
            max_events + 1
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                SUBJECTS
            )
        )
    )


    ax.set_yticklabels(
        [
            f"sub-{x}"
            for x in SUBJECTS
        ]
    )


    ax.set_xlabel(
        "Marker event number"
    )


    ax.set_ylabel(
        "Participant"
    )


    ax.set_title(
        "MarkerValueInt sequence across participants"
    )


    cbar = plt.colorbar(
        image,
        ax=ax
    )


    cbar.set_label(
        "MarkerValueInt"
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "04_marker_value_sequence_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 32. SAVE ALL TABLES
# ================================================================

recordings.to_csv(
    OUT
    / "all_recording_summary.csv",
    index=False
)


markers.to_csv(
    OUT
    / "all_reconstructed_markers.csv",
    index=False
)


sequences.to_csv(
    OUT
    / "all_marker_sequences.csv",
    index=False
)


bids_audit.to_csv(
    OUT
    / "all_bids_event_audit.csv",
    index=False
)


if not marker_counts.empty:

    marker_counts.to_csv(
        OUT
        / "marker_code_counts_by_group.csv",
        index=False
    )


if not errors.empty:

    errors.to_csv(
        OUT
        / "errors.csv",
        index=False
    )


# ================================================================
# 33. GROUP-LEVEL DURATION SUMMARY
# ================================================================

if not recordings.empty:

    group_duration = (

        recordings
        .groupby(
            "group"
        )[
            "duration_min"
        ]
        .agg(
            [
                "count",
                "mean",
                "std",
                "min",
                "median",
                "max"
            ]
        )
        .reset_index()
    )


    print("\n" + "=" * 90)
    print("RECORDING DURATION BY GROUP")
    print("=" * 90)


    display(
        group_duration.round(3)
    )


    group_duration.to_csv(
        OUT
        / "group_recording_duration_summary.csv",
        index=False
    )


# ================================================================
# 34. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 90)
print("TASK 1E COMPLETE")
print("=" * 90)


print(
    f"Participants successfully audited : "
    f"{len(recordings)} / 15"
)


print(
    f"Total markers reconstructed       : "
    f"{len(markers)}"
)


print(
    f"Participants with errors          : "
    f"{len(errors)}"
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print(
    "\nIMPORTANT:"
)

print(
    "No numeric marker has been assigned "
    "an experimental meaning."
)

print(
    "The purpose of this audit is to determine "
    "whether marker structure is reproducible "
    "across Active and Passive participants."
)

print(
    "\nNo EEG preprocessing has been performed."
)

In [ ]:
# ================================================================
# TASK 2A — PREPROCESSING + QC, SUB-001
# ================================================================
#
# Dataset: OpenNeuro ds008496
# Subject: sub-001
# Task: activeai
#
# PIPELINE
# --------
# 1. Load the 14 EEG channels directly from EEGLAB .fdt
# 2. Convert µV -> V for MNE
# 3. Apply standard 10-20 sensor montage
# 4. High-pass: 1 Hz
# 5. Low-pass: 45 Hz
# 6. Average reference
# 7. Quantify channel quality
# 8. Compare raw vs preprocessed traces
# 9. Compare raw vs preprocessed PSD
# 10. Generate channel-level QC
#
# IMPORTANT
# ---------
# No ICA components are removed here.
# No channels are automatically interpolated here.
# No experimental-condition labels are used.
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.signal import welch

import mne

warnings.filterwarnings("ignore")


# Force notebook figures to PNG rather than SVG
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"

L_FREQ = 1.0
H_FREQ = 45.0

TRACE_START = 180.0
TRACE_DURATION = 20.0


EEG_NAMES = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


# ================================================================
# 3. PATHS
# ================================================================

EEG_DIR = (
    ROOT
    / f"sub-{SUBJECT}"
    / "eeg"
)

SET_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.set"
)

FDT_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.fdt"
)

CHANNEL_FILE = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_channels.tsv"
)

EEG_JSON = (
    EEG_DIR
    / f"sub-{SUBJECT}_task-{TASK}_eeg.json"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02a_preprocessing_qc"
    / f"sub-{SUBJECT}"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 4. VERIFY FILES
# ================================================================

for path in [
    SET_FILE,
    FDT_FILE,
    CHANNEL_FILE,
    EEG_JSON
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


print("=" * 80)
print("TASK 2A — SUB-001 PREPROCESSING + QC")
print("=" * 80)


# ================================================================
# 5. LOAD METADATA
# ================================================================

channels = pd.read_csv(
    CHANNEL_FILE,
    sep="\t"
)


with open(
    EEG_JSON,
    "r"
) as f:

    metadata = json.load(f)


print(
    f"Task                 : "
    f"{metadata.get('TaskName')}"
)

print(
    f"Sampling frequency   : "
    f"{metadata.get('SamplingFrequency')} Hz"
)

print(
    f"Power-line frequency : "
    f"{metadata.get('PowerLineFrequency')} Hz"
)

print(
    f"Reference             : "
    f"{metadata.get('EEGReference')}"
)


# ================================================================
# 6. LOAD EEGLAB HEADER
# ================================================================

mat = loadmat(
    SET_FILE,
    squeeze_me=True,
    struct_as_record=False
)


nbchan = int(
    np.asarray(
        mat["nbchan"]
    ).squeeze()
)

pnts = int(
    np.asarray(
        mat["pnts"]
    ).squeeze()
)

sfreq = float(
    np.asarray(
        mat["srate"]
    ).squeeze()
)

trials = int(
    np.asarray(
        mat.get(
            "trials",
            1
        )
    ).squeeze()
)


if trials != 1:

    raise RuntimeError(
        "Expected continuous EEG."
    )


# ================================================================
# 7. CHANNEL NAMES
# ================================================================

channel_names = (
    channels["name"]
    .astype(str)
    .tolist()
)


if len(channel_names) != nbchan:

    raise RuntimeError(
        "Channel count mismatch."
    )


missing = [
    ch
    for ch in EEG_NAMES
    if ch not in channel_names
]


if missing:

    raise RuntimeError(
        f"Missing EEG channels: {missing}"
    )


eeg_indices = [
    channel_names.index(ch)
    for ch in EEG_NAMES
]


# ================================================================
# 8. LOAD FDT
# ================================================================

expected_values = (
    nbchan
    * pnts
)


if (
    FDT_FILE.stat().st_size
    !=
    expected_values * 4
):

    raise RuntimeError(
        "Unexpected FDT file size."
    )


flat = np.fromfile(
    FDT_FILE,
    dtype="<f4"
)


data = flat.reshape(
    (
        nbchan,
        pnts
    ),
    order="F"
)


# ================================================================
# 9. EXTRACT EEG
# ================================================================

raw_uv = data[
    eeg_indices,
    :
].astype(
    np.float64
)


# ================================================================
# 10. CREATE MNE RAW
# ================================================================

info = mne.create_info(
    ch_names=EEG_NAMES,
    sfreq=sfreq,
    ch_types="eeg"
)


raw = mne.io.RawArray(
    raw_uv * 1e-6,
    info,
    verbose=False
)


# ================================================================
# 11. SENSOR MONTAGE
# ================================================================

montage = (
    mne.channels
    .make_standard_montage(
        "standard_1020"
    )
)


raw.set_montage(
    montage,
    match_case=False,
    on_missing="raise",
    verbose=False
)


# ================================================================
# 12. PRESERVE RAW COPY
# ================================================================

raw_original = (
    raw.copy()
)


# ================================================================
# 13. FILTER
#
# 1–45 Hz.
#
# No 50-Hz notch is applied because:
#   1. H_FREQ is below 50 Hz
#   2. sub-001 showed no positive 50-Hz spectral excess
# ================================================================

print("\nFiltering 1–45 Hz...")


raw_filtered = (
    raw.copy()
    .filter(
        l_freq=L_FREQ,
        h_freq=H_FREQ,
        method="fir",
        phase="zero",
        fir_design="firwin",
        verbose=False
    )
)


# ================================================================
# 14. AVERAGE REFERENCE
# ================================================================

print(
    "Applying average reference..."
)


raw_filtered.set_eeg_reference(
    ref_channels="average",
    projection=False,
    verbose=False
)


# ================================================================
# 15. GET DATA IN µV
# ================================================================

before_uv = (
    raw_original.get_data()
    * 1e6
)


after_uv = (
    raw_filtered.get_data()
    * 1e6
)


# ================================================================
# 16. CHANNEL QC FUNCTION
# ================================================================

def channel_qc(
    data_uv,
    names
):

    rows = []


    for i, ch in enumerate(
        names
    ):

        x = data_uv[
            i
        ]


        centered = (
            x
            - np.median(
                x
            )
        )


        rows.append({

            "channel":
                ch,

            "mean_uV":
                np.mean(
                    x
                ),

            "median_uV":
                np.median(
                    x
                ),

            "sd_uV":
                np.std(
                    x
                ),

            "robust_mad_uV":
                np.median(
                    np.abs(
                        centered
                    )
                ),

            "rms_uV":
                np.sqrt(
                    np.mean(
                        x ** 2
                    )
                ),

            "p01_uV":
                np.percentile(
                    x,
                    1
                ),

            "p99_uV":
                np.percentile(
                    x,
                    99
                ),

            "peak_to_peak_uV":
                np.ptp(
                    x
                )
        })


    return pd.DataFrame(
        rows
    )


qc_before = channel_qc(
    before_uv,
    EEG_NAMES
)


qc_after = channel_qc(
    after_uv,
    EEG_NAMES
)


comparison = (
    qc_before
    .merge(
        qc_after,
        on="channel",
        suffixes=(
            "_before",
            "_after"
        )
    )
)


# ================================================================
# 17. ROBUST CHANNEL OUTLIER SCORES
#
# These are QC FLAGS, not automatic rejection.
# ================================================================

metric = (
    qc_after[
        "robust_mad_uV"
    ]
    .to_numpy()
)


median_metric = np.median(
    metric
)


mad_metric = np.median(
    np.abs(
        metric
        - median_metric
    )
)


if mad_metric > 0:

    robust_z = (
        0.6745
        *
        (
            metric
            - median_metric
        )
        /
        mad_metric
    )

else:

    robust_z = np.zeros_like(
        metric
    )


qc_after[
    "robust_z_mad"
] = robust_z


qc_after[
    "qc_flag_abs_z_gt_3_5"
] = (
    np.abs(
        robust_z
    )
    >
    3.5
)


print("\n" + "=" * 80)
print("POST-FILTER CHANNEL QC")
print("=" * 80)


display(
    qc_after.round(3)
)


flagged = qc_after[
    qc_after[
        "qc_flag_abs_z_gt_3_5"
    ]
]


print("\nPotential amplitude-dispersion outliers:")


if len(flagged):

    display(
        flagged[
            [
                "channel",
                "robust_mad_uV",
                "sd_uV",
                "peak_to_peak_uV",
                "robust_z_mad"
            ]
        ].round(3)
    )

else:

    print(
        "None by this criterion."
    )


# ================================================================
# 18. FIGURE — BEFORE/AFTER TRACE
# ================================================================

start = int(
    TRACE_START
    * sfreq
)

stop = int(
    (
        TRACE_START
        +
        TRACE_DURATION
    )
    * sfreq
)


stop = min(
    stop,
    pnts
)


times = (
    np.arange(
        stop - start
    )
    / sfreq
    +
    TRACE_START
)


before_segment = before_uv[
    :,
    start:stop
]


after_segment = after_uv[
    :,
    start:stop
]


# Use the same vertical spacing
# for both figures.

robust_amp = np.percentile(
    np.abs(
        after_segment
    ),
    95
)


spacing = max(
    robust_amp * 4,
    50
)


# ------------------------------------------------
# Raw
# ------------------------------------------------

fig, ax = plt.subplots(
    figsize=(15, 9)
)


for i, ch in enumerate(
    EEG_NAMES
):

    x = (
        before_segment[
            i
        ]
        -
        np.median(
            before_segment[
                i
            ]
        )
    )


    offset = (
        (
            len(
                EEG_NAMES
            )
            - 1
            - i
        )
        * spacing
    )


    ax.plot(
        times,
        x + offset,
        linewidth=0.55
    )


ax.set_yticks(
    np.arange(
        len(
            EEG_NAMES
        )
    )[::-1]
    * spacing
)


ax.set_yticklabels(
    EEG_NAMES
)


ax.set_xlabel(
    "Time (s)"
)


ax.set_ylabel(
    "Channel"
)


ax.set_title(
    "Before preprocessing — raw EEG"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_before_preprocessing_trace.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ------------------------------------------------
# Filtered + average referenced
# ------------------------------------------------

fig, ax = plt.subplots(
    figsize=(15, 9)
)


for i, ch in enumerate(
    EEG_NAMES
):

    x = after_segment[
        i
    ]


    offset = (
        (
            len(
                EEG_NAMES
            )
            - 1
            - i
        )
        * spacing
    )


    ax.plot(
        times,
        x + offset,
        linewidth=0.55
    )


ax.set_yticks(
    np.arange(
        len(
            EEG_NAMES
        )
    )[::-1]
    * spacing
)


ax.set_yticklabels(
    EEG_NAMES
)


ax.set_xlabel(
    "Time (s)"
)


ax.set_ylabel(
    "Channel"
)


ax.set_title(
    "After 1–45 Hz filtering and average reference"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_after_filter_reference_trace.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 19. PSD FUNCTION
# ================================================================

def calculate_psd(
    data_uv,
    sampling_rate
):

    nperseg = min(
        int(
            sampling_rate
            * 8
        ),
        data_uv.shape[
            1
        ]
    )


    freqs, psd = welch(
        data_uv,
        fs=sampling_rate,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        axis=1,
        detrend="constant"
    )


    psd_db = (
        10
        *
        np.log10(
            np.maximum(
                psd,
                np.finfo(
                    float
                ).eps
            )
        )
    )


    return (
        freqs,
        psd_db
    )


freq_before, psd_before = (
    calculate_psd(
        before_uv,
        sfreq
    )
)


freq_after, psd_after = (
    calculate_psd(
        after_uv,
        sfreq
    )
)


# ================================================================
# 20. FIGURE — MEAN PSD BEFORE VS AFTER
# ================================================================

mask_before = (
    (freq_before >= 0.2)
    &
    (freq_before <= 80)
)


mask_after = (
    (freq_after >= 0.2)
    &
    (freq_after <= 80)
)


mean_before = np.mean(
    psd_before,
    axis=0
)


mean_after = np.mean(
    psd_after,
    axis=0
)


fig, ax = plt.subplots(
    figsize=(13, 7)
)


ax.plot(
    freq_before[
        mask_before
    ],
    mean_before[
        mask_before
    ],
    linewidth=1.5,
    label="Raw"
)


ax.plot(
    freq_after[
        mask_after
    ],
    mean_after[
        mask_after
    ],
    linewidth=1.5,
    label="Filtered + average reference"
)


power_line = metadata.get(
    "PowerLineFrequency",
    None
)


if power_line is not None:

    ax.axvline(
        float(
            power_line
        ),
        linestyle="--",
        linewidth=1,
        label=(
            f"{power_line} Hz "
            "power line"
        )
    )


ax.axvline(
    L_FREQ,
    linestyle=":",
    linewidth=1
)


ax.axvline(
    H_FREQ,
    linestyle=":",
    linewidth=1
)


ax.set_xlabel(
    "Frequency (Hz)"
)


ax.set_ylabel(
    "PSD (dB µV²/Hz)"
)


ax.set_title(
    "Power spectrum before and after preprocessing"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "03_psd_before_after.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 21. PSD BY CHANNEL AFTER PREPROCESSING
# ================================================================

mask = (
    (freq_after >= 1)
    &
    (freq_after <= 45)
)


fig, ax = plt.subplots(
    figsize=(13, 7)
)


for i, ch in enumerate(
    EEG_NAMES
):

    ax.plot(
        freq_after[
            mask
        ],
        psd_after[
            i,
            mask
        ],
        linewidth=0.8,
        label=ch
    )


ax.set_xlabel(
    "Frequency (Hz)"
)


ax.set_ylabel(
    "PSD (dB µV²/Hz)"
)


ax.set_title(
    "Post-filter EEG power spectrum by channel"
)


ax.legend(
    ncol=2,
    fontsize=8
)


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "04_postfilter_psd_channels.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 22. CHANNEL VARIANCE COMPARISON
# ================================================================

fig, ax = plt.subplots(
    figsize=(12, 6)
)


x = np.arange(
    len(
        EEG_NAMES
    )
)


width = 0.38


ax.bar(
    x - width / 2,
    qc_before[
        "sd_uV"
    ],
    width,
    label="Raw"
)


ax.bar(
    x + width / 2,
    qc_after[
        "sd_uV"
    ],
    width,
    label="Filtered + referenced"
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    EEG_NAMES,
    rotation=45
)


ax.set_ylabel(
    "Standard deviation (µV)"
)


ax.set_xlabel(
    "EEG channel"
)


ax.set_title(
    "Channel variability before and after preprocessing"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "05_channel_variability_before_after.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 23. SENSOR POSITIONS
# ================================================================

positions = []


for ch in EEG_NAMES:

    idx = raw_filtered.ch_names.index(
        ch
    )


    loc = raw_filtered.info[
        "chs"
    ][
        idx
    ][
        "loc"
    ][:3]


    positions.append({
        "channel":
            ch,

        "x":
            loc[0],

        "y":
            loc[1],

        "z":
            loc[2]
    })


positions = pd.DataFrame(
    positions
)


# Manual 2D projection
fig, ax = plt.subplots(
    figsize=(7, 7)
)


ax.scatter(
    positions[
        "x"
    ],
    positions[
        "y"
    ],
    s=100
)


for _, row in positions.iterrows():

    ax.text(
        row[
            "x"
        ],
        row[
            "y"
        ],
        row[
            "channel"
        ],
        fontsize=9,
        ha="center",
        va="bottom"
    )


ax.set_xlabel(
    "X"
)


ax.set_ylabel(
    "Y"
)


ax.set_title(
    "EEG sensor locations"
)


ax.set_aspect(
    "equal"
)


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "06_sensor_locations.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 24. SAVE PREPROCESSED RAW
# ================================================================

FIF_FILE = (
    OUT
    / f"sub-{SUBJECT}_task-{TASK}_desc-filtered_eeg_raw.fif"
)


raw_filtered.save(
    FIF_FILE,
    overwrite=True,
    verbose=False
)


# ================================================================
# 25. SAVE QC TABLES
# ================================================================

qc_before.to_csv(
    OUT
    / "qc_before.csv",
    index=False
)


qc_after.to_csv(
    OUT
    / "qc_after.csv",
    index=False
)


comparison.to_csv(
    OUT
    / "qc_before_after.csv",
    index=False
)


positions.to_csv(
    OUT
    / "sensor_positions.csv",
    index=False
)


# ================================================================
# 26. SUMMARY
# ================================================================

summary = pd.DataFrame(
    [{
        "subject":
            f"sub-{SUBJECT}",

        "task":
            TASK,

        "sampling_frequency_hz":
            sfreq,

        "duration_sec":
            raw_filtered.times[
                -1
            ],

        "n_eeg_channels":
            len(
                EEG_NAMES
            ),

        "highpass_hz":
            L_FREQ,

        "lowpass_hz":
            H_FREQ,

        "reference":
            "average",

        "notch_filter":
            "none",

        "ica_applied":
            False,

        "channels_interpolated":
            0,

        "amplitude_qc_flags":
            ",".join(
                flagged[
                    "channel"
                ].tolist()
            )
    }]
)


summary.to_csv(
    OUT
    / "preprocessing_summary.csv",
    index=False
)


print("\n" + "=" * 80)
print("TASK 2A COMPLETE")
print("=" * 80)


display(
    summary.T
)


print(
    "\nPost-filter QC:"
)


display(
    qc_after.round(3)
)


print(
    "\nSaved preprocessed continuous EEG:"
)

print(
    FIF_FILE
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print(
    "\nIMPORTANT:"
)

print(
    "Filtering and average referencing are complete."
)

print(
    "No ICA component has been removed."
)

print(
    "No channel has been automatically interpolated."
)

print(
    "Review the traces, PSD, and QC flags before proceeding."
)

In [ ]:
# ================================================================
# TASK 2B — CONSERVATIVE ICA + ARTIFACT DIAGNOSTICS
# CORRECTED COMPLETE VERSION
# ================================================================
#
# Dataset: OpenNeuro ds008496
# Subject: sub-001
# Task: activeai
#
# INPUT:
# Task 2A filtered + average-referenced FIF
#
# PURPOSE
# -------
# 1. Load filtered/referenced EEG from Task 2A
# 2. Detect transient high-amplitude artifact windows
# 3. Quantify artifact burden by channel
# 4. Fit ICA using cleaner portions of the recording
# 5. Calculate component temporal/spectral diagnostics
# 6. Rank possible artifact components
# 7. Generate ICA topographies, time courses, and spectra
#
# IMPORTANT
# ---------
# - No ICA component is removed automatically.
# - No EEG channel is interpolated automatically.
# - AF3/AF4/F7/F8 are EEG channels, NOT dedicated EOG.
# - ICA rankings are diagnostic only.
# - Experimental condition labels are not used.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

from mne.preprocessing import ICA
from scipy.signal import welch
from scipy.stats import kurtosis


warnings.filterwarnings("ignore")


# Force notebook figures to PNG instead of SVG
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats

    set_matplotlib_formats("png")

except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"


# ------------------------------------------------
# Input from Task 2A
# ------------------------------------------------

INPUT_FIF = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02a_preprocessing_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}_desc-filtered_eeg_raw.fif"
)


# ------------------------------------------------
# Output directory
# ------------------------------------------------

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02b_ica_artifact_qc"
    / f"sub-{SUBJECT}"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------
# Artifact-screening parameters
# ------------------------------------------------

WINDOW_SEC = 2.0

PEAK_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


# ------------------------------------------------
# ICA parameters
# ------------------------------------------------

ICA_METHOD = "fastica"

ICA_RANDOM_STATE = 42

ICA_MAX_ITER = 1000


# ------------------------------------------------
# Visualization
# ------------------------------------------------

DISPLAY_MINUTES = 5

TOP_COMPONENTS_TO_DISPLAY = 6


# ================================================================
# 3. VERIFY INPUT
# ================================================================

if not INPUT_FIF.exists():

    raise FileNotFoundError(
        f"Task 2A FIF file not found:\n{INPUT_FIF}"
    )


# ================================================================
# 4. LOAD PREPROCESSED EEG
# ================================================================

raw = mne.io.read_raw_fif(
    INPUT_FIF,
    preload=True,
    verbose=False
)


sfreq = float(
    raw.info["sfreq"]
)

ch_names = list(
    raw.ch_names
)


data_v = raw.get_data()

data_uv = (
    data_v
    * 1e6
)


n_channels = (
    data_uv.shape[0]
)

n_samples = (
    data_uv.shape[1]
)

duration_sec = (
    n_samples
    / sfreq
)

duration_min = (
    duration_sec
    / 60
)


print("=" * 80)

print(
    "TASK 2B — CONSERVATIVE ICA + ARTIFACT DIAGNOSTICS"
)

print("=" * 80)

print(
    f"Subject      : sub-{SUBJECT}"
)

print(
    f"Task         : {TASK}"
)

print(
    f"Channels     : {n_channels}"
)

print(
    f"Sampling     : {sfreq:.2f} Hz"
)

print(
    f"Samples      : {n_samples:,}"
)

print(
    f"Duration     : {duration_sec:.2f} s"
)

print(
    f"Duration     : {duration_min:.2f} min"
)

print(
    f"Reference    : average "
    f"(from Task 2A)"
)

print(
    f"Filter       : 1–45 Hz "
    f"(from Task 2A)"
)


# ================================================================
# 5. WINDOW CONFIGURATION
# ================================================================

window_samples = int(
    round(
        WINDOW_SEC
        * sfreq
    )
)


if window_samples <= 0:

    raise RuntimeError(
        "Invalid artifact window size."
    )


n_complete_windows = (
    n_samples
    // window_samples
)


print("\n" + "=" * 80)

print(
    "ARTIFACT WINDOW CONFIGURATION"
)

print("=" * 80)

print(
    f"Window duration       : "
    f"{WINDOW_SEC:.2f} s"
)

print(
    f"Samples per window    : "
    f"{window_samples}"
)

print(
    f"Complete windows      : "
    f"{n_complete_windows}"
)

print(
    f"Peak threshold        : "
    f"{PEAK_THRESHOLD_UV:.1f} µV"
)

print(
    f"Peak-to-peak threshold: "
    f"{PTP_THRESHOLD_UV:.1f} µV"
)


# ================================================================
# 6. GLOBAL TRANSIENT ARTIFACT SCREEN
# ================================================================

window_rows = []

bad_window_mask = np.zeros(
    n_complete_windows,
    dtype=bool
)


for window_number in range(
    n_complete_windows
):

    start = (
        window_number
        * window_samples
    )

    stop = (
        start
        + window_samples
    )


    segment = data_uv[
        :,
        start:stop
    ]


    channel_max_abs = np.max(
        np.abs(
            segment
        ),
        axis=1
    )


    channel_ptp = np.ptp(
        segment,
        axis=1
    )


    max_abs = float(
        np.max(
            channel_max_abs
        )
    )


    max_ptp = float(
        np.max(
            channel_ptp
        )
    )


    n_peak_channels = int(
        np.sum(
            channel_max_abs
            >
            PEAK_THRESHOLD_UV
        )
    )


    n_ptp_channels = int(
        np.sum(
            channel_ptp
            >
            PTP_THRESHOLD_UV
        )
    )


    bad = bool(
        (max_abs > PEAK_THRESHOLD_UV)
        or
        (max_ptp > PTP_THRESHOLD_UV)
    )


    bad_window_mask[
        window_number
    ] = bad


    window_rows.append({

        "window":
            window_number,

        "start_sec":
            start / sfreq,

        "end_sec":
            stop / sfreq,

        "start_min":
            start / sfreq / 60,

        "max_abs_uV":
            max_abs,

        "max_ptp_uV":
            max_ptp,

        "channels_above_peak_threshold":
            n_peak_channels,

        "channels_above_ptp_threshold":
            n_ptp_channels,

        "artifact_flag":
            bad
    })


windows = pd.DataFrame(
    window_rows
)


n_bad_windows = int(
    bad_window_mask.sum()
)


artifact_percent = (
    100
    * n_bad_windows
    / n_complete_windows
)


print("\n" + "=" * 80)

print(
    "TRANSIENT ARTIFACT SCREEN"
)

print("=" * 80)


print(
    f"2-s windows                 : "
    f"{n_complete_windows}"
)

print(
    f"Flagged windows             : "
    f"{n_bad_windows}"
)

print(
    f"Unflagged windows           : "
    f"{n_complete_windows - n_bad_windows}"
)

print(
    f"Flagged proportion          : "
    f"{artifact_percent:.2f}%"
)


# ================================================================
# 7. CHANNEL-SPECIFIC ARTIFACT BURDEN
# ================================================================

channel_rows = []


for ch_idx, ch in enumerate(
    ch_names
):

    peak_flags = np.zeros(
        n_complete_windows,
        dtype=bool
    )

    ptp_flags = np.zeros(
        n_complete_windows,
        dtype=bool
    )


    for window_number in range(
        n_complete_windows
    ):

        start = (
            window_number
            * window_samples
        )

        stop = (
            start
            + window_samples
        )


        segment = data_uv[
            ch_idx,
            start:stop
        ]


        peak_flags[
            window_number
        ] = (
            np.max(
                np.abs(
                    segment
                )
            )
            >
            PEAK_THRESHOLD_UV
        )


        ptp_flags[
            window_number
        ] = (
            np.ptp(
                segment
            )
            >
            PTP_THRESHOLD_UV
        )


    combined_flags = (
        peak_flags
        |
        ptp_flags
    )


    channel_rows.append({

        "channel":
            ch,

        "peak_flag_windows":
            int(
                peak_flags.sum()
            ),

        "ptp_flag_windows":
            int(
                ptp_flags.sum()
            ),

        "artifact_windows":
            int(
                combined_flags.sum()
            ),

        "artifact_percent":
            float(
                combined_flags.mean()
                * 100
            )
    })


channel_artifacts = pd.DataFrame(
    channel_rows
)


channel_artifacts = (
    channel_artifacts
    .sort_values(
        "artifact_percent",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 80)

print(
    "CHANNEL-SPECIFIC ARTIFACT BURDEN"
)

print("=" * 80)


display(
    channel_artifacts.round(3)
)


# ================================================================
# 8. FIGURE — TRANSIENT AMPLITUDE SCREEN
# ================================================================

fig, ax = plt.subplots(
    figsize=(15, 5)
)


ax.scatter(
    windows[
        "start_min"
    ],
    windows[
        "max_abs_uV"
    ],
    s=14
)


ax.axhline(
    PEAK_THRESHOLD_UV,
    linestyle="--",
    linewidth=1.2,
    label=(
        f"{PEAK_THRESHOLD_UV:.0f} µV threshold"
    )
)


ax.set_xlabel(
    "Recording time (minutes)"
)


ax.set_ylabel(
    "Maximum |amplitude| in 2-s window (µV)"
)


ax.set_title(
    "Transient amplitude artifact screen"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_transient_amplitude_screen.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 9. FIGURE — PEAK-TO-PEAK ARTIFACT SCREEN
# ================================================================

fig, ax = plt.subplots(
    figsize=(15, 5)
)


ax.scatter(
    windows[
        "start_min"
    ],
    windows[
        "max_ptp_uV"
    ],
    s=14
)


ax.axhline(
    PTP_THRESHOLD_UV,
    linestyle="--",
    linewidth=1.2,
    label=(
        f"{PTP_THRESHOLD_UV:.0f} µV threshold"
    )
)


ax.set_xlabel(
    "Recording time (minutes)"
)


ax.set_ylabel(
    "Maximum peak-to-peak amplitude "
    "in 2-s window (µV)"
)


ax.set_title(
    "Transient peak-to-peak artifact screen"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_transient_peak_to_peak_screen.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 10. FIGURE — ARTIFACT BURDEN BY CHANNEL
# ================================================================

plot_data = (
    channel_artifacts
    .sort_values(
        "artifact_percent",
        ascending=False
    )
)


fig, ax = plt.subplots(
    figsize=(12, 6)
)


ax.bar(
    plot_data[
        "channel"
    ],
    plot_data[
        "artifact_percent"
    ]
)


ax.set_xlabel(
    "EEG channel"
)


ax.set_ylabel(
    "Flagged 2-s windows (%)"
)


ax.set_title(
    "Transient artifact burden by EEG channel"
)


ax.tick_params(
    axis="x",
    rotation=45
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "03_artifact_burden_by_channel.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 11. BUILD CLEAN DATA FOR ICA FITTING
#
# ICA is fitted only using windows that passed the conservative
# transient artifact screen.
#
# The resulting ICA solution is evaluated against the full
# recording but NOTHING is removed automatically.
# ================================================================

clean_segments = []


for window_number in range(
    n_complete_windows
):

    if bad_window_mask[
        window_number
    ]:

        continue


    start = (
        window_number
        * window_samples
    )

    stop = (
        start
        + window_samples
    )


    clean_segments.append(

        raw.get_data(
            start=start,
            stop=stop
        )

    )


if len(
    clean_segments
) == 0:

    raise RuntimeError(
        "No clean windows remain for ICA fitting."
    )


clean_data = np.concatenate(
    clean_segments,
    axis=1
)


clean_samples = (
    clean_data.shape[1]
)


clean_duration_sec = (
    clean_samples
    / sfreq
)


clean_duration_min = (
    clean_duration_sec
    / 60
)


clean_percent = (
    100
    * clean_duration_sec
    / duration_sec
)


print("\n" + "=" * 80)

print(
    "ICA FITTING DATA"
)

print("=" * 80)


print(
    f"Clean samples  : "
    f"{clean_samples:,}"
)

print(
    f"Clean duration : "
    f"{clean_duration_sec:.2f} s"
)

print(
    f"Clean duration : "
    f"{clean_duration_min:.2f} min"
)

print(
    f"Retained       : "
    f"{clean_percent:.2f}%"
)


# ================================================================
# 12. CREATE CLEAN RAW FOR ICA
# ================================================================

clean_info = (
    raw.info.copy()
)


raw_clean_fit = mne.io.RawArray(
    clean_data,
    clean_info,
    verbose=False
)


# ================================================================
# 13. ESTIMATE DATA RANK
# ================================================================

rank = mne.compute_rank(
    raw_clean_fit,
    rank=None,
    verbose=False
)


print("\nEstimated EEG rank:")

print(
    rank
)


# ================================================================
# 14. FIT ICA
#
# Average referencing reduces the theoretical rank by one.
# n_components=0.999 lets MNE retain components explaining
# 99.9% of variance while respecting the numerical rank.
# ================================================================

print("\nFitting ICA...")


ica = ICA(
    n_components=0.999,
    method=ICA_METHOD,
    random_state=ICA_RANDOM_STATE,
    max_iter=ICA_MAX_ITER
)


ica.fit(
    raw_clean_fit,
    picks="eeg",
    verbose=False
)


n_components = int(
    ica.n_components_
)


print(
    f"ICA components fitted: "
    f"{n_components}"
)


# ================================================================
# 15. EXTRACT ICA SOURCES FROM FULL RECORDING
# ================================================================

source_raw = ica.get_sources(
    raw
)


sources = (
    source_raw
    .get_data()
)


if sources.shape[0] != n_components:

    raise RuntimeError(
        "ICA source/component count mismatch."
    )


# ================================================================
# 16. BASIC COMPONENT STATISTICS
# ================================================================

component_rows = []


for component in range(
    n_components
):

    x = sources[
        component
    ]


    component_rows.append({

        "component":
            f"ICA{component:03d}",

        "index":
            component,

        "mean":
            float(
                np.mean(x)
            ),

        "sd":
            float(
                np.std(x)
            ),

        "kurtosis":
            float(
                kurtosis(
                    x,
                    fisher=True,
                    bias=False
                )
            ),

        "max_abs":
            float(
                np.max(
                    np.abs(x)
                )
            ),

        "p99_abs":
            float(
                np.percentile(
                    np.abs(x),
                    99
                )
            )
    })


component_stats = pd.DataFrame(
    component_rows
)


# ================================================================
# 17. FRONTAL EEG PROXY CORRELATION
#
# AF3/AF4/F7/F8 are NOT EOG.
#
# This measure is only a diagnostic indicator of components
# strongly represented in frontal EEG.
# ================================================================

frontal_names = [

    ch

    for ch in [
        "AF3",
        "AF4",
        "F7",
        "F8"
    ]

    if ch in ch_names
]


frontal_indices = [

    ch_names.index(
        ch
    )

    for ch in frontal_names
]


if len(
    frontal_indices
) > 0:

    frontal_signal = np.mean(
        data_uv[
            frontal_indices,
            :
        ],
        axis=0
    )

else:

    frontal_signal = np.zeros(
        n_samples
    )


frontal_corr = []


for component in range(
    n_components
):

    x = sources[
        component
    ]


    if (
        np.std(x) == 0
        or
        np.std(
            frontal_signal
        ) == 0
    ):

        r = 0.0

    else:

        r = float(
            np.corrcoef(
                x,
                frontal_signal
            )[0, 1]
        )


    frontal_corr.append(
        r
    )


component_stats[
    "frontal_proxy_corr"
] = frontal_corr


component_stats[
    "abs_frontal_proxy_corr"
] = np.abs(
    component_stats[
        "frontal_proxy_corr"
    ]
)


# ================================================================
# 18. COMPONENT PSD
# ================================================================

nperseg = min(
    int(
        sfreq
        * 8
    ),
    sources.shape[1]
)


if nperseg < 2:

    raise RuntimeError(
        "Insufficient samples for component PSD."
    )


noverlap = (
    nperseg
    // 2
)


freqs, source_psd = welch(
    sources,
    fs=sfreq,
    nperseg=nperseg,
    noverlap=noverlap,
    axis=1,
    detrend="constant"
)


# ================================================================
# 19. FREQUENCY MASKS
# ================================================================

analysis_mask = (
    (freqs >= 1)
    &
    (freqs <= 45)
)


low_mask = (
    (freqs >= 1)
    &
    (freqs <= 4)
)


theta_mask = (
    (freqs >= 4)
    &
    (freqs < 8)
)


alpha_mask = (
    (freqs >= 8)
    &
    (freqs < 13)
)


beta_mask = (
    (freqs >= 13)
    &
    (freqs < 30)
)


high_mask = (
    (freqs >= 30)
    &
    (freqs <= 45)
)


# ================================================================
# 20. SAFE NUMERICAL INTEGRATION
#
# NumPy 2.x uses np.trapezoid().
#
# This helper also retains compatibility with older NumPy
# versions if the notebook is moved to another environment.
# ================================================================

def integrate_spectrum(
    y,
    x
):

    if hasattr(
        np,
        "trapezoid"
    ):

        return float(
            np.trapezoid(
                y,
                x
            )
        )

    # Compatibility fallback for older NumPy
    return float(
        np.trapz(
            y,
            x
        )
    )


# ================================================================
# 21. COMPONENT SPECTRAL FRACTIONS
# ================================================================

spectral_rows = []


for component in range(
    n_components
):

    spectrum = source_psd[
        component
    ]


    total_power = integrate_spectrum(
        spectrum[
            analysis_mask
        ],
        freqs[
            analysis_mask
        ]
    )


    low_power = integrate_spectrum(
        spectrum[
            low_mask
        ],
        freqs[
            low_mask
        ]
    )


    theta_power = integrate_spectrum(
        spectrum[
            theta_mask
        ],
        freqs[
            theta_mask
        ]
    )


    alpha_power = integrate_spectrum(
        spectrum[
            alpha_mask
        ],
        freqs[
            alpha_mask
        ]
    )


    beta_power = integrate_spectrum(
        spectrum[
            beta_mask
        ],
        freqs[
            beta_mask
        ]
    )


    high_power = integrate_spectrum(
        spectrum[
            high_mask
        ],
        freqs[
            high_mask
        ]
    )


    if (
        np.isfinite(
            total_power
        )
        and
        total_power > 0
    ):

        low_fraction = (
            low_power
            / total_power
        )

        theta_fraction = (
            theta_power
            / total_power
        )

        alpha_fraction = (
            alpha_power
            / total_power
        )

        beta_fraction = (
            beta_power
            / total_power
        )

        high_fraction = (
            high_power
            / total_power
        )

    else:

        low_fraction = np.nan
        theta_fraction = np.nan
        alpha_fraction = np.nan
        beta_fraction = np.nan
        high_fraction = np.nan


    spectral_rows.append({

        "index":
            component,

        "total_1_45_power":
            total_power,

        "lowfreq_1_4_fraction":
            low_fraction,

        "theta_4_8_fraction":
            theta_fraction,

        "alpha_8_13_fraction":
            alpha_fraction,

        "beta_13_30_fraction":
            beta_fraction,

        "highfreq_30_45_fraction":
            high_fraction
    })


spectral_stats = pd.DataFrame(
    spectral_rows
)


component_stats = (
    component_stats
    .merge(
        spectral_stats,
        on="index",
        how="left"
    )
)


# ================================================================
# 22. ROBUST COMPONENT-SCORE FUNCTION
# ================================================================

def robust_score(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    med = np.nanmedian(
        values
    )


    mad = np.nanmedian(
        np.abs(
            values
            - med
        )
    )


    if (
        not np.isfinite(
            mad
        )
        or
        mad == 0
    ):

        return np.zeros_like(
            values
        )


    return (
        0.6745
        *
        (
            values
            - med
        )
        /
        mad
    )


# ================================================================
# 23. COMPONENT DIAGNOSTIC SCORES
# ================================================================

component_stats[
    "z_kurtosis"
] = robust_score(
    component_stats[
        "kurtosis"
    ]
)


component_stats[
    "z_frontal"
] = robust_score(
    component_stats[
        "abs_frontal_proxy_corr"
    ]
)


component_stats[
    "z_lowfreq"
] = robust_score(
    component_stats[
        "lowfreq_1_4_fraction"
    ]
)


component_stats[
    "z_highfreq"
] = robust_score(
    component_stats[
        "highfreq_30_45_fraction"
    ]
)


# ================================================================
# 24. CANDIDATE SCORE
#
# This is NOT an artifact probability.
#
# It simply prioritizes components showing one or more unusual
# characteristics:
#
# - high kurtosis
# - strong frontal representation
# - unusually high 1–4 Hz fraction
# - unusually high 30–45 Hz fraction
# ================================================================

component_stats[
    "candidate_score"
] = (

    np.maximum(
        component_stats[
            "z_kurtosis"
        ],
        0
    )

    +

    np.maximum(
        component_stats[
            "z_frontal"
        ],
        0
    )

    +

    np.maximum(
        component_stats[
            "z_lowfreq"
        ],
        0
    )

    +

    np.maximum(
        component_stats[
            "z_highfreq"
        ],
        0
    )
)


component_stats = (
    component_stats
    .sort_values(
        "candidate_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 80)

print(
    "ICA COMPONENT DIAGNOSTIC RANKING"
)

print("=" * 80)


display(
    component_stats[
        [
            "component",
            "index",
            "kurtosis",
            "frontal_proxy_corr",
            "lowfreq_1_4_fraction",
            "theta_4_8_fraction",
            "alpha_8_13_fraction",
            "beta_13_30_fraction",
            "highfreq_30_45_fraction",
            "candidate_score"
        ]
    ].round(4)
)


# ================================================================
# 25. SELECT TOP COMPONENTS FOR VISUALIZATION
# ================================================================

top_n = min(
    TOP_COMPONENTS_TO_DISPLAY,
    n_components
)


top_components = (

    component_stats[
        "index"
    ]

    .head(
        top_n
    )

    .astype(int)

    .tolist()
)


print("\nTop components for inspection:")

print(
    [
        f"ICA{x:03d}"
        for x in top_components
    ]
)


# ================================================================
# 26. ICA TOPOGRAPHIES
# ================================================================

print(
    "\nGenerating ICA topographies..."
)


component_figures = ica.plot_components(
    show=False
)


if not isinstance(
    component_figures,
    list
):

    component_figures = [
        component_figures
    ]


for figure_number, figure in enumerate(
    component_figures,
    start=1
):

    figure.savefig(

        OUT
        / (
            f"04_ica_topographies_"
            f"{figure_number}.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 27. TOP-CANDIDATE TOPOGRAPHIES
# ================================================================

try:

    top_figures = ica.plot_components(
        picks=top_components,
        show=False
    )


    if not isinstance(
        top_figures,
        list
    ):

        top_figures = [
            top_figures
        ]


    for figure_number, figure in enumerate(
        top_figures,
        start=1
    ):

        figure.savefig(

            OUT
            / (
                "05_top_candidate_"
                f"topographies_"
                f"{figure_number}.png"
            ),

            dpi=300,

            bbox_inches="tight"
        )


        plt.show()


except Exception as exc:

    print(
        "Could not generate separate "
        "candidate topographies:"
    )

    print(
        exc
    )


# ================================================================
# 28. TOP-CANDIDATE TIME COURSES
# ================================================================

display_samples = min(

    int(
        sfreq
        * DISPLAY_MINUTES
        * 60
    ),

    sources.shape[1]
)


display_time_min = (

    np.arange(
        display_samples
    )

    / sfreq

    / 60
)


fig, ax = plt.subplots(
    figsize=(15, 9)
)


spacing = 8.0


for row_idx, component in enumerate(
    top_components
):

    x = sources[
        component,
        :display_samples
    ].copy()


    scale = np.std(
        x
    )


    if (
        np.isfinite(scale)
        and
        scale > 0
    ):

        x = (
            x
            / scale
        )


    offset = (

        (
            top_n
            - 1
            - row_idx
        )

        * spacing
    )


    ax.plot(
        display_time_min,
        x + offset,
        linewidth=0.5
    )


ax.set_yticks(

    np.arange(
        top_n
    )[::-1]

    * spacing
)


ax.set_yticklabels(

    [
        f"ICA{x:03d}"
        for x in top_components
    ]
)


ax.set_xlabel(
    "Time (minutes)"
)


ax.set_ylabel(
    "ICA component"
)


ax.set_title(
    f"Highest-ranked ICA candidates — "
    f"first {DISPLAY_MINUTES} minutes"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "06_top_candidate_timecourses.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 29. COMPONENT SPECTRA
# ================================================================

frequency_mask = (
    (freqs >= 1)
    &
    (freqs <= 45)
)


fig, ax = plt.subplots(
    figsize=(13, 7)
)


for component in top_components:

    spectrum_db = (

        10

        * np.log10(

            np.maximum(
                source_psd[
                    component
                ],
                np.finfo(
                    float
                ).eps
            )

        )
    )


    ax.plot(

        freqs[
            frequency_mask
        ],

        spectrum_db[
            frequency_mask
        ],

        linewidth=1,

        label=(
            f"ICA{component:03d}"
        )
    )


ax.axvline(
    4,
    linestyle=":",
    linewidth=0.8
)


ax.axvline(
    8,
    linestyle=":",
    linewidth=0.8
)


ax.axvline(
    13,
    linestyle=":",
    linewidth=0.8
)


ax.axvline(
    30,
    linestyle=":",
    linewidth=0.8
)


ax.set_xlabel(
    "Frequency (Hz)"
)


ax.set_ylabel(
    "PSD (dB)"
)


ax.set_title(
    "Spectra of highest-ranked ICA candidates"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "07_candidate_component_spectra.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 30. COMPONENT DIAGNOSTIC SUMMARY FIGURE
# ================================================================

plot_stats = (
    component_stats
    .sort_values(
        "candidate_score",
        ascending=False
    )
)


fig, ax = plt.subplots(
    figsize=(12, 6)
)


ax.bar(

    plot_stats[
        "component"
    ],

    plot_stats[
        "candidate_score"
    ]
)


ax.set_xlabel(
    "ICA component"
)


ax.set_ylabel(
    "Diagnostic candidate score"
)


ax.set_title(
    "ICA component diagnostic ranking"
)


ax.tick_params(
    axis="x",
    rotation=45
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "08_ica_candidate_scores.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 31. SAVE ICA SOLUTION
# ================================================================

ICA_FILE = (

    OUT

    / (
        f"sub-{SUBJECT}_"
        f"task-{TASK}-ica.fif"
    )
)


ica.save(
    ICA_FILE,
    overwrite=True
)


# ================================================================
# 32. SAVE TABLES
# ================================================================

windows.to_csv(
    OUT
    / "artifact_windows.csv",
    index=False
)


channel_artifacts.to_csv(
    OUT
    / "channel_artifact_burden.csv",
    index=False
)


component_stats.to_csv(
    OUT
    / "ica_component_diagnostics.csv",
    index=False
)


# ================================================================
# 33. CREATE SUMMARY
# ================================================================

summary = pd.DataFrame(
    [{
        "subject":
            f"sub-{SUBJECT}",

        "task":
            TASK,

        "duration_sec":
            duration_sec,

        "duration_min":
            duration_min,

        "sampling_frequency_hz":
            sfreq,

        "eeg_channels":
            n_channels,

        "window_sec":
            WINDOW_SEC,

        "n_complete_windows":
            n_complete_windows,

        "artifact_windows":
            n_bad_windows,

        "artifact_percent":
            artifact_percent,

        "clean_duration_sec":
            clean_duration_sec,

        "clean_duration_min":
            clean_duration_min,

        "clean_percent":
            clean_percent,

        "ica_components":
            n_components,

        "ica_method":
            ICA_METHOD,

        "ica_random_state":
            ICA_RANDOM_STATE,

        "components_removed":
            0,

        "channels_interpolated":
            0,

        "peak_threshold_uV":
            PEAK_THRESHOLD_UV,

        "ptp_threshold_uV":
            PTP_THRESHOLD_UV
    }]
)


summary.to_csv(
    OUT
    / "task02b_summary.csv",
    index=False
)


# ================================================================
# 34. FINAL OUTPUT
# ================================================================

print("\n" + "=" * 80)

print(
    "TASK 2B COMPLETE"
)

print("=" * 80)


print("\nSUMMARY")

display(
    summary.T
)


print("\n" + "=" * 80)

print(
    "CHANNEL ARTIFACT BURDEN"
)

print("=" * 80)


display(
    channel_artifacts.round(3)
)


print("\n" + "=" * 80)

print(
    "HIGHEST-RANKED ICA CANDIDATES"
)

print("=" * 80)


display(

    component_stats[
        [
            "component",
            "kurtosis",
            "frontal_proxy_corr",
            "lowfreq_1_4_fraction",
            "theta_4_8_fraction",
            "alpha_8_13_fraction",
            "beta_13_30_fraction",
            "highfreq_30_45_fraction",
            "candidate_score"
        ]
    ]

    .head(
        min(
            8,
            n_components
        )
    )

    .round(4)
)


print(
    "\nSaved ICA solution:"
)

print(
    ICA_FILE
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print("\n" + "=" * 80)

print(
    "IMPORTANT INTERPRETATION"
)

print("=" * 80)


print(
    "1. No ICA component has been removed."
)

print(
    "2. No EEG channel has been interpolated."
)

print(
    "3. Artifact windows were excluded only "
    "from ICA fitting."
)

print(
    "4. The full continuous recording remains "
    "unchanged."
)

print(
    "5. AF3, AF4, F7, and F8 are frontal EEG "
    "channels, not dedicated EOG channels."
)

print(
    "6. Candidate scores are diagnostic rankings, "
    "not artifact probabilities."
)

print(
    "7. Component removal should require converging "
    "topographic, temporal, and spectral evidence."
)

In [ ]:
# ================================================================
# TASK 2C — ICA COMPONENT INSPECTION AND PROVISIONAL CLASSIFICATION
# ================================================================
#
# Subject: sub-001
# Task: activeai
#
# PURPOSE
# -------
# Combine:
#   - ICA topography
#   - frontal loading
#   - temporal loading
#   - posterior loading
#   - lateralization
#   - kurtosis
#   - low-frequency power
#   - high-frequency power
#   - frontal EEG correlation
#
# OUTPUT
# ------
# A structured component-review table.
#
# IMPORTANT
# ---------
# NO ICA COMPONENT IS REMOVED.
# NO CHANNEL IS INTERPOLATED.
# Classification is provisional and conservative.
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from mne.preprocessing import read_ica


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"


ICA_DIR = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02b_ica_artifact_qc"
    / f"sub-{SUBJECT}"
)


INPUT_FIF = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02a_preprocessing_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}_desc-filtered_eeg_raw.fif"
)


ICA_FILE = (
    ICA_DIR
    / f"sub-{SUBJECT}_task-{TASK}-ica.fif"
)


DIAGNOSTIC_FILE = (
    ICA_DIR
    / "ica_component_diagnostics.csv"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02c_component_review"
    / f"sub-{SUBJECT}"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. VERIFY INPUT FILES
# ================================================================

for path in [
    INPUT_FIF,
    ICA_FILE,
    DIAGNOSTIC_FILE
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


# ================================================================
# 4. LOAD EEG + ICA + TASK 2B DIAGNOSTICS
# ================================================================

raw = mne.io.read_raw_fif(
    INPUT_FIF,
    preload=True,
    verbose=False
)


ica = read_ica(
    ICA_FILE,
    verbose=False
)


diagnostics = pd.read_csv(
    DIAGNOSTIC_FILE
)


n_components = int(
    ica.n_components_
)


ch_names = list(
    raw.ch_names
)


print("=" * 80)
print("TASK 2C — ICA COMPONENT REVIEW")
print("=" * 80)

print(
    f"Subject        : sub-{SUBJECT}"
)

print(
    f"Task           : {TASK}"
)

print(
    f"EEG channels   : {len(ch_names)}"
)

print(
    f"ICA components : {n_components}"
)


# ================================================================
# 5. EXTRACT ICA SENSOR PATTERNS
#
# get_components() returns:
#
# channels × ICA components
#
# We normalize absolute loadings within each component so that
# spatial distributions can be compared across components.
# ================================================================

patterns = ica.get_components()


if patterns.shape[1] != n_components:

    raise RuntimeError(
        "Unexpected ICA component matrix."
    )


if patterns.shape[0] != len(
    ica.ch_names
):

    raise RuntimeError(
        "ICA channel count mismatch."
    )


ica_channels = list(
    ica.ch_names
)


# ================================================================
# 6. DEFINE SENSOR REGIONS
#
# These are descriptive scalp regions, not anatomical source
# localizations.
# ================================================================

FRONTAL = [
    "AF3",
    "AF4",
    "F7",
    "F3",
    "F4",
    "F8"
]


FRONTOCENTRAL = [
    "FC5",
    "FC6"
]


TEMPORAL = [
    "T7",
    "T8"
]


POSTERIOR = [
    "P7",
    "P8",
    "O1",
    "O2"
]


LEFT = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1"
]


RIGHT = [
    "AF4",
    "F8",
    "F4",
    "FC6",
    "T8",
    "P8",
    "O2"
]


# ================================================================
# 7. HELPER FUNCTIONS
# ================================================================

def indices_for(names):

    return [
        ica_channels.index(ch)
        for ch in names
        if ch in ica_channels
    ]


frontal_idx = indices_for(
    FRONTAL
)

frontocentral_idx = indices_for(
    FRONTOCENTRAL
)

temporal_idx = indices_for(
    TEMPORAL
)

posterior_idx = indices_for(
    POSTERIOR
)

left_idx = indices_for(
    LEFT
)

right_idx = indices_for(
    RIGHT
)


def region_fraction(
    weights,
    indices
):

    if len(indices) == 0:

        return np.nan

    total = np.sum(
        np.abs(weights)
    )

    if total == 0:

        return np.nan

    return float(
        np.sum(
            np.abs(
                weights[
                    indices
                ]
            )
        )
        /
        total
    )


# ================================================================
# 8. CALCULATE TOPOGRAPHIC CHARACTERISTICS
# ================================================================

topography_rows = []


for component in range(
    n_components
):

    weights = patterns[
        :,
        component
    ]


    absolute_weights = np.abs(
        weights
    )


    total_abs = np.sum(
        absolute_weights
    )


    if total_abs > 0:

        normalized = (
            absolute_weights
            /
            total_abs
        )

    else:

        normalized = (
            absolute_weights
        )


    dominant_idx = int(
        np.argmax(
            absolute_weights
        )
    )


    dominant_channel = (
        ica_channels[
            dominant_idx
        ]
    )


    dominant_fraction = float(
        normalized[
            dominant_idx
        ]
    )


    frontal_fraction = region_fraction(
        weights,
        frontal_idx
    )


    frontocentral_fraction = region_fraction(
        weights,
        frontocentral_idx
    )


    temporal_fraction = region_fraction(
        weights,
        temporal_idx
    )


    posterior_fraction = region_fraction(
        weights,
        posterior_idx
    )


    left_fraction = region_fraction(
        weights,
        left_idx
    )


    right_fraction = region_fraction(
        weights,
        right_idx
    )


    lateralization = (
        right_fraction
        -
        left_fraction
    )


    # Spatial concentration:
    # larger values = component dominated by fewer electrodes.

    spatial_concentration = float(
        np.sum(
            normalized ** 2
        )
    )


    topography_rows.append({

        "index":
            component,

        "component":
            f"ICA{component:03d}",

        "dominant_channel":
            dominant_channel,

        "dominant_channel_fraction":
            dominant_fraction,

        "frontal_loading_fraction":
            frontal_fraction,

        "frontocentral_loading_fraction":
            frontocentral_fraction,

        "temporal_loading_fraction":
            temporal_fraction,

        "posterior_loading_fraction":
            posterior_fraction,

        "left_loading_fraction":
            left_fraction,

        "right_loading_fraction":
            right_fraction,

        "lateralization_right_minus_left":
            lateralization,

        "spatial_concentration":
            spatial_concentration
    })


topography = pd.DataFrame(
    topography_rows
)


# ================================================================
# 9. MERGE WITH TASK 2B DIAGNOSTICS
# ================================================================

review = diagnostics.merge(
    topography,
    on=[
        "index",
        "component"
    ],
    how="left"
)


# ================================================================
# 10. PROVISIONAL COMPONENT CHARACTERISTICS
#
# These are deliberately conservative.
#
# They are NOT automatic artifact diagnoses.
# ================================================================

review[
    "very_lowfreq"
] = (
    review[
        "lowfreq_1_4_fraction"
    ]
    >= 0.85
)


review[
    "strong_frontal_corr"
] = (
    review[
        "abs_frontal_proxy_corr"
    ]
    >= 0.40
)


review[
    "high_frontal_loading"
] = (
    review[
        "frontal_loading_fraction"
    ]
    >= 0.45
)


review[
    "high_temporal_loading"
] = (
    review[
        "temporal_loading_fraction"
    ]
    >= 0.30
)


review[
    "high_posterior_loading"
] = (
    review[
        "posterior_loading_fraction"
    ]
    >= 0.40
)


review[
    "high_highfreq"
] = (
    review[
        "highfreq_30_45_fraction"
    ]
    >= 0.10
)


review[
    "extreme_kurtosis"
] = (
    review[
        "kurtosis"
    ]
    >= 500
)


review[
    "very_extreme_kurtosis"
] = (
    review[
        "kurtosis"
    ]
    >= 1000
)


review[
    "strong_lateralization"
] = (
    np.abs(
        review[
            "lateralization_right_minus_left"
        ]
    )
    >= 0.40
)


# ================================================================
# 11. PROVISIONAL ARTIFACT PATTERN LABEL
#
# These labels describe the diagnostic pattern.
# They do NOT establish biological origin.
# ================================================================

pattern_labels = []

review_recommendations = []

reasons = []


for _, row in review.iterrows():

    evidence = []

    pattern = "mixed/neural-like"

    recommendation = "KEEP"


    # ------------------------------------------------------------
    # Extreme transient-like component
    # ------------------------------------------------------------

    if row[
        "very_extreme_kurtosis"
    ]:

        evidence.append(
            "very extreme kurtosis"
        )


    elif row[
        "extreme_kurtosis"
    ]:

        evidence.append(
            "extreme kurtosis"
        )


    # ------------------------------------------------------------
    # Low-frequency behavior
    # ------------------------------------------------------------

    if row[
        "very_lowfreq"
    ]:

        evidence.append(
            "very strong 1–4 Hz dominance"
        )


    # ------------------------------------------------------------
    # Frontal evidence
    # ------------------------------------------------------------

    if row[
        "strong_frontal_corr"
    ]:

        evidence.append(
            "strong frontal EEG correlation"
        )


    if row[
        "high_frontal_loading"
    ]:

        evidence.append(
            "frontal scalp loading"
        )


    # ------------------------------------------------------------
    # Temporal evidence
    # ------------------------------------------------------------

    if row[
        "high_temporal_loading"
    ]:

        evidence.append(
            "temporal scalp concentration"
        )


    # ------------------------------------------------------------
    # High-frequency evidence
    # ------------------------------------------------------------

    if row[
        "high_highfreq"
    ]:

        evidence.append(
            "elevated 30–45 Hz fraction"
        )


    # ------------------------------------------------------------
    # Pattern classification
    # ------------------------------------------------------------

    ocular_like = (
        row[
            "very_lowfreq"
        ]
        and
        (
            row[
                "strong_frontal_corr"
            ]
            or
            row[
                "high_frontal_loading"
            ]
        )
    )


    muscle_like = (
        row[
            "high_highfreq"
        ]
        and
        (
            row[
                "high_temporal_loading"
            ]
            or
            row[
                "strong_lateralization"
            ]
        )
    )


    transient_like = (
        row[
            "very_extreme_kurtosis"
        ]
        or
        (
            row[
                "extreme_kurtosis"
            ]
            and
            row[
                "spatial_concentration"
            ]
            >= 0.15
        )
    )


    if ocular_like:

        pattern = (
            "ocular/movement-like"
        )

        recommendation = (
            "REVIEW"
        )


    elif muscle_like:

        pattern = (
            "muscle-like"
        )

        recommendation = (
            "REVIEW"
        )


    elif transient_like:

        pattern = (
            "transient/artifact-like"
        )

        recommendation = (
            "REVIEW"
        )


    # ------------------------------------------------------------
    # Build reason
    # ------------------------------------------------------------

    if len(
        evidence
    ) == 0:

        reason = (
            "No strong artifact signature "
            "under current conservative rules"
        )

    else:

        reason = "; ".join(
            evidence
        )


    pattern_labels.append(
        pattern
    )

    review_recommendations.append(
        recommendation
    )

    reasons.append(
        reason
    )


review[
    "provisional_pattern"
] = pattern_labels


review[
    "recommendation"
] = review_recommendations


review[
    "reason"
] = reasons


# ================================================================
# 12. SORT FOR REVIEW
# ================================================================

review[
    "review_priority"
] = np.where(
    review[
        "recommendation"
    ]
    == "REVIEW",
    1,
    0
)


review = (
    review
    .sort_values(
        [
            "review_priority",
            "candidate_score"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 13. DISPLAY COMPONENT REVIEW TABLE
# ================================================================

display_columns = [

    "component",

    "dominant_channel",

    "frontal_loading_fraction",

    "temporal_loading_fraction",

    "posterior_loading_fraction",

    "lateralization_right_minus_left",

    "spatial_concentration",

    "kurtosis",

    "frontal_proxy_corr",

    "lowfreq_1_4_fraction",

    "highfreq_30_45_fraction",

    "candidate_score",

    "provisional_pattern",

    "recommendation",

    "reason"
]


print("\n" + "=" * 80)

print(
    "ICA COMPONENT REVIEW TABLE"
)

print("=" * 80)


display(
    review[
        display_columns
    ].round(4)
)


# ================================================================
# 14. COMPONENTS REQUIRING MANUAL REVIEW
# ================================================================

review_candidates = review[
    review[
        "recommendation"
    ]
    == "REVIEW"
].copy()


print("\n" + "=" * 80)

print(
    "COMPONENTS REQUIRING MANUAL REVIEW"
)

print("=" * 80)


if len(
    review_candidates
) == 0:

    print(
        "No components met the conservative "
        "review criteria."
    )

else:

    display(

        review_candidates[
            [
                "component",
                "dominant_channel",
                "provisional_pattern",
                "kurtosis",
                "frontal_proxy_corr",
                "lowfreq_1_4_fraction",
                "highfreq_30_45_fraction",
                "reason"
            ]
        ].round(4)

    )


# ================================================================
# 15. VISUALIZE SPATIAL LOADING FRACTIONS
# ================================================================

plot_review = (
    review
    .sort_values(
        "index"
    )
)


components = (
    plot_review[
        "component"
    ]
    .tolist()
)


x = np.arange(
    len(
        components
    )
)


fig, ax = plt.subplots(
    figsize=(14, 7)
)


ax.plot(
    x,
    plot_review[
        "frontal_loading_fraction"
    ],
    marker="o",
    label="Frontal"
)


ax.plot(
    x,
    plot_review[
        "temporal_loading_fraction"
    ],
    marker="o",
    label="Temporal"
)


ax.plot(
    x,
    plot_review[
        "posterior_loading_fraction"
    ],
    marker="o",
    label="Posterior"
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    components,
    rotation=45
)


ax.set_xlabel(
    "ICA component"
)


ax.set_ylabel(
    "Absolute loading fraction"
)


ax.set_title(
    "Regional scalp loading of ICA components"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_component_regional_loadings.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 16. LOW VS HIGH FREQUENCY DIAGNOSTIC PLOT
# ================================================================

fig, ax = plt.subplots(
    figsize=(10, 7)
)


ax.scatter(

    review[
        "lowfreq_1_4_fraction"
    ],

    review[
        "highfreq_30_45_fraction"
    ],

    s=80
)


for _, row in review.iterrows():

    ax.text(

        row[
            "lowfreq_1_4_fraction"
        ],

        row[
            "highfreq_30_45_fraction"
        ],

        row[
            "component"
        ],

        fontsize=8,

        ha="left",

        va="bottom"
    )


ax.set_xlabel(
    "1–4 Hz power fraction"
)


ax.set_ylabel(
    "30–45 Hz power fraction"
)


ax.set_title(
    "ICA component spectral characteristics"
)


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_component_spectral_characteristics.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 17. KURTOSIS VS FRONTAL CORRELATION
# ================================================================

fig, ax = plt.subplots(
    figsize=(10, 7)
)


ax.scatter(

    review[
        "abs_frontal_proxy_corr"
    ],

    review[
        "kurtosis"
    ],

    s=80
)


for _, row in review.iterrows():

    ax.text(

        row[
            "abs_frontal_proxy_corr"
        ],

        row[
            "kurtosis"
        ],

        row[
            "component"
        ],

        fontsize=8,

        ha="left",

        va="bottom"
    )


ax.set_xlabel(
    "|Correlation with frontal EEG proxy|"
)


ax.set_ylabel(
    "Kurtosis"
)


ax.set_title(
    "ICA transient and frontal characteristics"
)


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "03_kurtosis_vs_frontal_correlation.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 18. SAVE COMPLETE REVIEW TABLE
# ================================================================

review.to_csv(
    OUT
    / "ica_component_review.csv",
    index=False
)


review_candidates.to_csv(
    OUT
    / "ica_components_for_manual_review.csv",
    index=False
)


# ================================================================
# 19. CREATE SUMMARY
# ================================================================

summary = pd.DataFrame(
    [{
        "subject":
            f"sub-{SUBJECT}",

        "task":
            TASK,

        "ica_components":
            n_components,

        "components_for_review":
            len(
                review_candidates
            ),

        "components_automatically_removed":
            0,

        "channels_interpolated":
            0
    }]
)


summary.to_csv(
    OUT
    / "task02c_summary.csv",
    index=False
)


# ================================================================
# 20. FINAL OUTPUT
# ================================================================

print("\n" + "=" * 80)

print(
    "TASK 2C COMPLETE"
)

print("=" * 80)


display(
    summary.T
)


print(
    "\nComponents currently marked for REVIEW:"
)


if len(
    review_candidates
) > 0:

    print(

        review_candidates[
            "component"
        ].tolist()

    )

else:

    print(
        "None"
    )


print(
    "\nNo component has been removed."
)


print(
    "No channel has been interpolated."
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print("\nNEXT DECISION:")

print(
    "Use the topographies, time courses, spectra, "
    "and this review table together to determine "
    "whether any ICA component should actually "
    "be excluded."
)

In [ ]:
# ================================================================
# TASK 2D — CONSERVATIVE ICA CLEANING + VALIDATION
# ================================================================
#
# Subject: sub-001
# Task: activeai
#
# DECISION FROM TASK 2C
# ---------------------
# Exclude:
#     ICA005
#     ICA012
#
# Retain:
#     ICA006
#
# RATIONALE
# ---------
# ICA005:
#   strong frontal loading
#   AF4 dominance
#   strong low-frequency dominance
#   strong frontal EEG association
#   very high kurtosis
#
# ICA012:
#   very strong frontal loading
#   extreme kurtosis
#   strong low-frequency dominance
#   strong lateralization
#
# ICA006 is retained because its dominant T7/temporal
# distribution makes its interpretation less certain.
#
# IMPORTANT
# ---------
# This cell:
#   1. applies only the two conservative exclusions;
#   2. compares EEG before/after ICA;
#   3. measures spectral preservation;
#   4. reassesses transient artifact burden;
#   5. saves cleaned continuous EEG;
#   6. DOES NOT interpolate channels;
#   7. DOES NOT permanently delete artifact windows.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

from mne.preprocessing import read_ica
from scipy.signal import welch


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"


# Components selected conservatively
ICA_EXCLUDE = [
    5,
    12
]


# Artifact-screen parameters from Task 2B
WINDOW_SEC = 2.0

PEAK_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


# Trace visualization
TRACE_START_SEC = 180.0

TRACE_DURATION_SEC = 20.0


# ================================================================
# 3. INPUT PATHS
# ================================================================

INPUT_FIF = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02a_preprocessing_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}_desc-filtered_eeg_raw.fif"
)


ICA_FILE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02b_ica_artifact_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}-ica.fif"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02d_final_cleaning"
    / f"sub-{SUBJECT}"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 4. VERIFY FILES
# ================================================================

for path in [
    INPUT_FIF,
    ICA_FILE
]:

    if not path.exists():

        raise FileNotFoundError(
            path
        )


# ================================================================
# 5. LOAD FILTERED EEG + ICA
# ================================================================

raw_before = mne.io.read_raw_fif(
    INPUT_FIF,
    preload=True,
    verbose=False
)


ica = read_ica(
    ICA_FILE,
    verbose=False
)


sfreq = float(
    raw_before.info["sfreq"]
)


ch_names = list(
    raw_before.ch_names
)


n_channels = len(
    ch_names
)


n_samples = raw_before.n_times


duration_sec = (
    n_samples
    / sfreq
)


duration_min = (
    duration_sec
    / 60
)


print("=" * 80)

print(
    "TASK 2D — CONSERVATIVE ICA CLEANING + VALIDATION"
)

print("=" * 80)


print(
    f"Subject              : sub-{SUBJECT}"
)

print(
    f"Task                 : {TASK}"
)

print(
    f"Channels             : {n_channels}"
)

print(
    f"Sampling frequency   : {sfreq:.2f} Hz"
)

print(
    f"Duration             : {duration_min:.2f} min"
)

print(
    f"ICA components       : {ica.n_components_}"
)

print(
    f"Components selected  : "
    f"{['ICA%03d' % x for x in ICA_EXCLUDE]}"
)

print(
    "ICA006              : RETAINED"
)


# ================================================================
# 6. APPLY ICA TO A COPY
# ================================================================

raw_after = (
    raw_before.copy()
)


ica.exclude = ICA_EXCLUDE.copy()


ica.apply(
    raw_after,
    exclude=ICA_EXCLUDE,
    verbose=False
)


print(
    "\nICA cleaning applied successfully."
)


# ================================================================
# 7. DATA IN MICROVOLTS
# ================================================================

before_uv = (
    raw_before.get_data()
    * 1e6
)


after_uv = (
    raw_after.get_data()
    * 1e6
)


# ================================================================
# 8. BASIC CHANNEL STATISTICS
# ================================================================

def channel_statistics(
    data_uv,
    channel_names
):

    rows = []


    for idx, ch in enumerate(
        channel_names
    ):

        x = data_uv[
            idx
        ]


        median = np.median(
            x
        )


        mad = np.median(
            np.abs(
                x - median
            )
        )


        rows.append({

            "channel":
                ch,

            "mean_uV":
                float(
                    np.mean(x)
                ),

            "median_uV":
                float(
                    median
                ),

            "sd_uV":
                float(
                    np.std(x)
                ),

            "mad_uV":
                float(
                    mad
                ),

            "rms_uV":
                float(
                    np.sqrt(
                        np.mean(
                            x ** 2
                        )
                    )
                ),

            "p01_uV":
                float(
                    np.percentile(
                        x,
                        1
                    )
                ),

            "p99_uV":
                float(
                    np.percentile(
                        x,
                        99
                    )
                ),

            "peak_to_peak_uV":
                float(
                    np.ptp(x)
                )
        })


    return pd.DataFrame(
        rows
    )


stats_before = channel_statistics(
    before_uv,
    ch_names
)


stats_after = channel_statistics(
    after_uv,
    ch_names
)


channel_comparison = (
    stats_before
    .merge(
        stats_after,
        on="channel",
        suffixes=(
            "_before",
            "_after"
        )
    )
)


channel_comparison[
    "sd_change_percent"
] = (

    100

    * (
        channel_comparison[
            "sd_uV_after"
        ]
        -
        channel_comparison[
            "sd_uV_before"
        ]
    )

    /
    channel_comparison[
        "sd_uV_before"
    ]
)


channel_comparison[
    "rms_change_percent"
] = (

    100

    * (
        channel_comparison[
            "rms_uV_after"
        ]
        -
        channel_comparison[
            "rms_uV_before"
        ]
    )

    /
    channel_comparison[
        "rms_uV_before"
    ]
)


print("\n" + "=" * 80)

print(
    "CHANNEL STATISTICS — BEFORE VS AFTER ICA"
)

print("=" * 80)


display(

    channel_comparison[
        [
            "channel",
            "sd_uV_before",
            "sd_uV_after",
            "sd_change_percent",
            "mad_uV_before",
            "mad_uV_after",
            "peak_to_peak_uV_before",
            "peak_to_peak_uV_after"
        ]
    ].round(3)

)


# ================================================================
# 9. SIGNAL DIFFERENCE
# ================================================================

difference_uv = (
    before_uv
    -
    after_uv
)


difference_rms = np.sqrt(
    np.mean(
        difference_uv ** 2,
        axis=1
    )
)


original_rms = np.sqrt(
    np.mean(
        before_uv ** 2,
        axis=1
    )
)


relative_difference = (

    100
    * difference_rms
    / original_rms
)


difference_table = pd.DataFrame({

    "channel":
        ch_names,

    "removed_signal_rms_uV":
        difference_rms,

    "original_signal_rms_uV":
        original_rms,

    "removed_relative_to_original_percent":
        relative_difference
})


print("\n" + "=" * 80)

print(
    "AMOUNT OF SIGNAL REMOVED BY ICA"
)

print("=" * 80)


display(
    difference_table.round(3)
)


# ================================================================
# 10. PSD FUNCTION
# ================================================================

def compute_psd(
    data_uv,
    sampling_frequency
):

    nperseg = min(
        int(
            sampling_frequency
            * 8
        ),
        data_uv.shape[1]
    )


    freqs, psd = welch(
        data_uv,
        fs=sampling_frequency,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        axis=1,
        detrend="constant"
    )


    return (
        freqs,
        psd
    )


freqs_before, psd_before = compute_psd(
    before_uv,
    sfreq
)


freqs_after, psd_after = compute_psd(
    after_uv,
    sfreq
)


# ================================================================
# 11. SAFE SPECTRAL INTEGRATION
# ================================================================

def integrate_spectrum(
    y,
    x
):

    if hasattr(
        np,
        "trapezoid"
    ):

        return float(
            np.trapezoid(
                y,
                x
            )
        )

    return float(
        np.trapz(
            y,
            x
        )
    )


# ================================================================
# 12. BAND DEFINITIONS
# ================================================================

BANDS = {

    "delta_1_4":
        (1.0, 4.0),

    "theta_4_8":
        (4.0, 8.0),

    "alpha_8_13":
        (8.0, 13.0),

    "beta_13_30":
        (13.0, 30.0),

    "low_gamma_30_45":
        (30.0, 45.0)
}


# ================================================================
# 13. BAND POWER BEFORE/AFTER
# ================================================================

band_rows = []


for ch_idx, ch in enumerate(
    ch_names
):

    for band_name, (
        f_low,
        f_high
    ) in BANDS.items():

        mask_before = (
            (freqs_before >= f_low)
            &
            (freqs_before < f_high)
        )


        mask_after = (
            (freqs_after >= f_low)
            &
            (freqs_after < f_high)
        )


        power_before = integrate_spectrum(

            psd_before[
                ch_idx,
                mask_before
            ],

            freqs_before[
                mask_before
            ]
        )


        power_after = integrate_spectrum(

            psd_after[
                ch_idx,
                mask_after
            ],

            freqs_after[
                mask_after
            ]
        )


        if power_before > 0:

            change_percent = (

                100
                * (
                    power_after
                    -
                    power_before
                )
                /
                power_before
            )

        else:

            change_percent = np.nan


        band_rows.append({

            "channel":
                ch,

            "band":
                band_name,

            "power_before":
                power_before,

            "power_after":
                power_after,

            "change_percent":
                change_percent
        })


band_power = pd.DataFrame(
    band_rows
)


# ================================================================
# 14. GROUP BAND SUMMARY
# ================================================================

band_summary = (

    band_power

    .groupby(
        "band"
    )

    .agg(

        mean_change_percent=(
            "change_percent",
            "mean"
        ),

        median_change_percent=(
            "change_percent",
            "median"
        ),

        min_change_percent=(
            "change_percent",
            "min"
        ),

        max_change_percent=(
            "change_percent",
            "max"
        )

    )

    .reset_index()
)


print("\n" + "=" * 80)

print(
    "BAND-POWER PRESERVATION"
)

print("=" * 80)


display(
    band_summary.round(3)
)


# ================================================================
# 15. REASSESS ARTIFACT WINDOWS
# ================================================================

window_samples = int(
    round(
        WINDOW_SEC
        * sfreq
    )
)


n_windows = (
    n_samples
    // window_samples
)


def artifact_screen(
    data_uv
):

    rows = []


    for window_number in range(
        n_windows
    ):

        start = (
            window_number
            * window_samples
        )

        stop = (
            start
            + window_samples
        )


        segment = data_uv[
            :,
            start:stop
        ]


        max_abs = float(
            np.max(
                np.abs(
                    segment
                )
            )
        )


        max_ptp = float(
            np.max(
                np.ptp(
                    segment,
                    axis=1
                )
            )
        )


        artifact = bool(

            (max_abs > PEAK_THRESHOLD_UV)

            or

            (max_ptp > PTP_THRESHOLD_UV)

        )


        rows.append({

            "window":
                window_number,

            "start_sec":
                start / sfreq,

            "max_abs_uV":
                max_abs,

            "max_ptp_uV":
                max_ptp,

            "artifact_flag":
                artifact
        })


    return pd.DataFrame(
        rows
    )


artifacts_before = artifact_screen(
    before_uv
)


artifacts_after = artifact_screen(
    after_uv
)


before_count = int(
    artifacts_before[
        "artifact_flag"
    ].sum()
)


after_count = int(
    artifacts_after[
        "artifact_flag"
    ].sum()
)


before_percent = (
    100
    * before_count
    / n_windows
)


after_percent = (
    100
    * after_count
    / n_windows
)


if before_count > 0:

    reduction_percent = (

        100
        * (
            before_count
            -
            after_count
        )
        / before_count
    )

else:

    reduction_percent = np.nan


artifact_summary = pd.DataFrame(
    [{
        "windows":
            n_windows,

        "artifact_windows_before":
            before_count,

        "artifact_percent_before":
            before_percent,

        "artifact_windows_after":
            after_count,

        "artifact_percent_after":
            after_percent,

        "artifact_window_reduction_percent":
            reduction_percent
    }]
)


print("\n" + "=" * 80)

print(
    "ARTIFACT-WINDOW IMPROVEMENT"
)

print("=" * 80)


display(
    artifact_summary.T
)


# ================================================================
# 16. TRACE COMPARISON
# ================================================================

start_sample = int(
    TRACE_START_SEC
    * sfreq
)


stop_sample = min(

    int(
        (
            TRACE_START_SEC
            +
            TRACE_DURATION_SEC
        )
        * sfreq
    ),

    n_samples
)


times = (

    np.arange(
        stop_sample
        -
        start_sample
    )

    / sfreq

    +
    TRACE_START_SEC
)


segment_before = before_uv[
    :,
    start_sample:stop_sample
]


segment_after = after_uv[
    :,
    start_sample:stop_sample
]


robust_amplitude = np.percentile(
    np.abs(
        segment_before
    ),
    95
)


spacing = max(
    robust_amplitude * 4,
    50
)


# ------------------------------------------------
# BEFORE
# ------------------------------------------------

fig, ax = plt.subplots(
    figsize=(15, 9)
)


for i, ch in enumerate(
    ch_names
):

    offset = (

        (
            len(ch_names)
            - 1
            - i
        )

        * spacing
    )


    ax.plot(

        times,

        segment_before[
            i
        ]
        + offset,

        linewidth=0.5
    )


ax.set_yticks(

    np.arange(
        len(ch_names)
    )[::-1]

    * spacing
)


ax.set_yticklabels(
    ch_names
)


ax.set_xlabel(
    "Time (s)"
)


ax.set_ylabel(
    "EEG channel"
)


ax.set_title(
    "Before ICA cleaning"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_before_ica_trace.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ------------------------------------------------
# AFTER
# ------------------------------------------------

fig, ax = plt.subplots(
    figsize=(15, 9)
)


for i, ch in enumerate(
    ch_names
):

    offset = (

        (
            len(ch_names)
            - 1
            - i
        )

        * spacing
    )


    ax.plot(

        times,

        segment_after[
            i
        ]
        + offset,

        linewidth=0.5
    )


ax.set_yticks(

    np.arange(
        len(ch_names)
    )[::-1]

    * spacing
)


ax.set_yticklabels(
    ch_names
)


ax.set_xlabel(
    "Time (s)"
)


ax.set_ylabel(
    "EEG channel"
)


ax.set_title(
    "After conservative ICA cleaning"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_after_ica_trace.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 17. PSD BEFORE/AFTER
# ================================================================

mean_psd_before = np.mean(
    psd_before,
    axis=0
)


mean_psd_after = np.mean(
    psd_after,
    axis=0
)


frequency_mask = (
    (freqs_before >= 1)
    &
    (freqs_before <= 45)
)


fig, ax = plt.subplots(
    figsize=(13, 7)
)


ax.plot(

    freqs_before[
        frequency_mask
    ],

    10
    * np.log10(
        np.maximum(
            mean_psd_before[
                frequency_mask
            ],
            np.finfo(float).eps
        )
    ),

    linewidth=1.5,

    label="Before ICA"
)


ax.plot(

    freqs_after[
        frequency_mask
    ],

    10
    * np.log10(
        np.maximum(
            mean_psd_after[
                frequency_mask
            ],
            np.finfo(float).eps
        )
    ),

    linewidth=1.5,

    label="After ICA"
)


ax.set_xlabel(
    "Frequency (Hz)"
)


ax.set_ylabel(
    "PSD (dB µV²/Hz)"
)


ax.set_title(
    "Power spectrum before and after conservative ICA cleaning"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "03_psd_before_after_ica.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 18. BAND POWER CHANGE FIGURE
# ================================================================

band_plot = (

    band_power

    .pivot(
        index="channel",
        columns="band",
        values="change_percent"
    )

    .reindex(
        ch_names
    )
)


fig, ax = plt.subplots(
    figsize=(14, 7)
)


for band in band_plot.columns:

    ax.plot(

        band_plot.index,

        band_plot[
            band
        ],

        marker="o",

        linewidth=1,

        label=band
    )


ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "EEG channel"
)


ax.set_ylabel(
    "Power change after ICA (%)"
)


ax.set_title(
    "Band-power change produced by ICA cleaning"
)


ax.tick_params(
    axis="x",
    rotation=45
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "04_band_power_change.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 19. ARTIFACT WINDOWS BEFORE/AFTER
# ================================================================

fig, ax = plt.subplots(
    figsize=(15, 5)
)


ax.scatter(

    artifacts_before[
        "start_sec"
    ]
    / 60,

    artifacts_before[
        "max_abs_uV"
    ],

    s=12,

    label="Before ICA"
)


ax.scatter(

    artifacts_after[
        "start_sec"
    ]
    / 60,

    artifacts_after[
        "max_abs_uV"
    ],

    s=12,

    label="After ICA"
)


ax.axhline(
    PEAK_THRESHOLD_UV,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Recording time (minutes)"
)


ax.set_ylabel(
    "Maximum |amplitude| per 2-s window (µV)"
)


ax.set_title(
    "Transient amplitudes before and after ICA cleaning"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "05_artifact_windows_before_after.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 20. SAVE FINAL CLEANED CONTINUOUS EEG
# ================================================================

FINAL_FIF = (

    OUT

    / (
        f"sub-{SUBJECT}_"
        f"task-{TASK}_"
        f"desc-cleaned_eeg_raw.fif"
    )
)


raw_after.save(
    FINAL_FIF,
    overwrite=True,
    verbose=False
)


# ================================================================
# 21. SAVE ARTIFACT ANNOTATIONS
#
# IMPORTANT:
# We retain the full continuous recording.
# These annotations identify windows that still exceed the
# transient threshold after ICA.
# ================================================================

remaining_bad = artifacts_after[
    artifacts_after[
        "artifact_flag"
    ]
].copy()


if len(
    remaining_bad
) > 0:

    annotations = mne.Annotations(

        onset=remaining_bad[
            "start_sec"
        ].to_numpy(),

        duration=np.full(
            len(
                remaining_bad
            ),
            WINDOW_SEC
        ),

        description=[
            "BAD_transient"
        ]
        * len(
            remaining_bad
        )
    )


    raw_annotated = (
        raw_after.copy()
    )


    raw_annotated.set_annotations(
        annotations
    )


else:

    raw_annotated = (
        raw_after.copy()
    )


ANNOTATED_FIF = (

    OUT

    / (
        f"sub-{SUBJECT}_"
        f"task-{TASK}_"
        f"desc-cleaned_annotated_eeg_raw.fif"
    )
)


raw_annotated.save(
    ANNOTATED_FIF,
    overwrite=True,
    verbose=False
)


# ================================================================
# 22. SAVE TABLES
# ================================================================

channel_comparison.to_csv(
    OUT
    / "channel_before_after_ica.csv",
    index=False
)


difference_table.to_csv(
    OUT
    / "ica_removed_signal_by_channel.csv",
    index=False
)


band_power.to_csv(
    OUT
    / "band_power_before_after_ica.csv",
    index=False
)


band_summary.to_csv(
    OUT
    / "band_power_summary.csv",
    index=False
)


artifacts_before.to_csv(
    OUT
    / "artifact_windows_before_ica.csv",
    index=False
)


artifacts_after.to_csv(
    OUT
    / "artifact_windows_after_ica.csv",
    index=False
)


remaining_bad.to_csv(
    OUT
    / "remaining_bad_windows.csv",
    index=False
)


# ================================================================
# 23. FINAL SUMMARY
# ================================================================

summary = pd.DataFrame(
    [{
        "subject":
            f"sub-{SUBJECT}",

        "task":
            TASK,

        "duration_min":
            duration_min,

        "eeg_channels":
            n_channels,

        "ica_components_total":
            int(
                ica.n_components_
            ),

        "ica_components_removed":
            ",".join(
                [
                    f"ICA{x:03d}"
                    for x in ICA_EXCLUDE
                ]
            ),

        "n_ica_components_removed":
            len(
                ICA_EXCLUDE
            ),

        "ica006_status":
            "retained",

        "channels_interpolated":
            0,

        "artifact_windows_before":
            before_count,

        "artifact_windows_after":
            after_count,

        "artifact_percent_before":
            before_percent,

        "artifact_percent_after":
            after_percent,

        "artifact_window_reduction_percent":
            reduction_percent,

        "remaining_bad_windows":
            len(
                remaining_bad
            )
    }]
)


summary.to_csv(
    OUT
    / "task02d_summary.csv",
    index=False
)


# ================================================================
# 24. FINAL REPORT
# ================================================================

print("\n" + "=" * 80)

print(
    "TASK 2D COMPLETE"
)

print("=" * 80)


print("\nFINAL CLEANING SUMMARY")


display(
    summary.T
)


print("\n" + "=" * 80)

print(
    "BAND-POWER PRESERVATION"
)

print("=" * 80)


display(
    band_summary.round(3)
)


print("\n" + "=" * 80)

print(
    "SIGNAL REMOVED BY CHANNEL"
)

print("=" * 80)


display(
    difference_table.round(3)
)


print("\n" + "=" * 80)

print(
    "REMAINING TRANSIENT WINDOWS"
)

print("=" * 80)


print(
    f"Before ICA : "
    f"{before_count}/{n_windows} "
    f"({before_percent:.2f}%)"
)


print(
    f"After ICA  : "
    f"{after_count}/{n_windows} "
    f"({after_percent:.2f}%)"
)


print(
    f"Reduction  : "
    f"{reduction_percent:.2f}%"
)


print(
    "\nClean continuous EEG:"
)

print(
    FINAL_FIF
)


print(
    "\nClean EEG with BAD_transient annotations:"
)

print(
    ANNOTATED_FIF
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print("\n" + "=" * 80)

print(
    "IMPORTANT"
)

print("=" * 80)


print(
    "ICA005 and ICA012 were removed."
)


print(
    "ICA006 was deliberately retained."
)


print(
    "No EEG channel was interpolated."
)


print(
    "Remaining high-amplitude windows are annotated "
    "as BAD_transient rather than deleted."
)


print(
    "The annotated FIF should be used for subsequent "
    "epoch-based spectral/connectivity analyses."
)

In [ ]:
# ================================================================
# TASK 2E — ICA EXCLUSION SENSITIVITY ANALYSIS
# ================================================================
#
# Subject: sub-001
# Task: activeai
#
# Compare:
#   A. No ICA removal
#   B. ICA005 only
#   C. ICA012 only
#   D. ICA005 + ICA012
#
# PURPOSE
# -------
# Determine which ICA exclusion provides useful artifact reduction
# and which causes excessive frontal signal removal.
#
# NOTHING IS OVERWRITTEN.
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from mne.preprocessing import read_ica
from scipy.signal import welch

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 1. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"

WINDOW_SEC = 2.0
PEAK_THRESHOLD_UV = 250.0
PTP_THRESHOLD_UV = 500.0


INPUT_FIF = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02a_preprocessing_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}_desc-filtered_eeg_raw.fif"
)


ICA_FILE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02b_ica_artifact_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}-ica.fif"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02e_ica_sensitivity"
    / f"sub-{SUBJECT}"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. LOAD DATA
# ================================================================

if not INPUT_FIF.exists():
    raise FileNotFoundError(INPUT_FIF)

if not ICA_FILE.exists():
    raise FileNotFoundError(ICA_FILE)


raw_original = mne.io.read_raw_fif(
    INPUT_FIF,
    preload=True,
    verbose=False
)

ica = read_ica(
    ICA_FILE,
    verbose=False
)


sfreq = float(
    raw_original.info["sfreq"]
)

ch_names = list(
    raw_original.ch_names
)

original_uv = (
    raw_original.get_data()
    * 1e6
)

n_samples = original_uv.shape[1]


print("=" * 80)
print("TASK 2E — ICA EXCLUSION SENSITIVITY ANALYSIS")
print("=" * 80)

print(f"Subject        : sub-{SUBJECT}")
print(f"Task           : {TASK}")
print(f"Channels       : {len(ch_names)}")
print(f"ICA components : {ica.n_components_}")
print(f"Duration       : {n_samples/sfreq/60:.2f} min")


# ================================================================
# 3. CONDITIONS TO TEST
# ================================================================

CONDITIONS = {
    "No_ICA": [],
    "ICA005_only": [5],
    "ICA012_only": [12],
    "ICA005_ICA012": [5, 12]
}


# ================================================================
# 4. APPLY EACH ICA CONDITION
# ================================================================

condition_data = {}


for name, exclusions in CONDITIONS.items():

    candidate = raw_original.copy()

    if len(exclusions) > 0:

        ica.apply(
            candidate,
            exclude=exclusions,
            verbose=False
        )

    condition_data[name] = (
        candidate.get_data()
        * 1e6
    )


print("\nConditions created:")

for name, exclusions in CONDITIONS.items():

    labels = [
        f"ICA{x:03d}"
        for x in exclusions
    ]

    print(
        f"{name:18s}: "
        f"{labels if labels else 'none'}"
    )


# ================================================================
# 5. ARTIFACT WINDOW FUNCTION
# ================================================================

window_samples = int(
    WINDOW_SEC * sfreq
)

n_windows = (
    n_samples
    // window_samples
)


def artifact_metrics(data_uv):

    flags = []
    max_abs_values = []
    max_ptp_values = []

    for w in range(n_windows):

        start = (
            w * window_samples
        )

        stop = (
            start + window_samples
        )

        segment = data_uv[
            :,
            start:stop
        ]

        max_abs = float(
            np.max(
                np.abs(segment)
            )
        )

        max_ptp = float(
            np.max(
                np.ptp(
                    segment,
                    axis=1
                )
            )
        )

        flag = (
            max_abs > PEAK_THRESHOLD_UV
            or
            max_ptp > PTP_THRESHOLD_UV
        )

        flags.append(flag)
        max_abs_values.append(max_abs)
        max_ptp_values.append(max_ptp)

    flags = np.asarray(
        flags,
        dtype=bool
    )

    return {
        "n_artifact_windows":
            int(flags.sum()),

        "artifact_percent":
            float(
                flags.mean() * 100
            ),

        "median_window_max_abs_uV":
            float(
                np.median(
                    max_abs_values
                )
            ),

        "p95_window_max_abs_uV":
            float(
                np.percentile(
                    max_abs_values,
                    95
                )
            ),

        "max_window_abs_uV":
            float(
                np.max(
                    max_abs_values
                )
            ),

        "flags":
            flags
    }


# ================================================================
# 6. SIGNAL REMOVAL FUNCTION
# ================================================================

def signal_metrics(
    original,
    cleaned
):

    original_rms = np.sqrt(
        np.mean(
            original ** 2,
            axis=1
        )
    )

    difference = (
        original - cleaned
    )

    removed_rms = np.sqrt(
        np.mean(
            difference ** 2,
            axis=1
        )
    )

    relative_removed = (
        100
        * removed_rms
        / original_rms
    )

    return (
        original_rms,
        removed_rms,
        relative_removed
    )


# ================================================================
# 7. PSD FUNCTION
# ================================================================

def calculate_psd(
    data_uv
):

    nperseg = min(
        int(
            sfreq * 8
        ),
        data_uv.shape[1]
    )

    freqs, psd = welch(
        data_uv,
        fs=sfreq,
        nperseg=nperseg,
        noverlap=nperseg // 2,
        axis=1,
        detrend="constant"
    )

    return freqs, psd


def integrate_spectrum(
    y,
    x
):

    if hasattr(
        np,
        "trapezoid"
    ):

        return float(
            np.trapezoid(
                y,
                x
            )
        )

    return float(
        np.trapz(
            y,
            x
        )
    )


BANDS = {
    "delta_1_4": (1, 4),
    "theta_4_8": (4, 8),
    "alpha_8_13": (8, 13),
    "beta_13_30": (13, 30),
    "gamma_30_45": (30, 45)
}


# ================================================================
# 8. ORIGINAL PSD
# ================================================================

freqs_original, psd_original = calculate_psd(
    original_uv
)


# ================================================================
# 9. CONDITION-LEVEL SUMMARY
# ================================================================

summary_rows = []
channel_rows = []
band_rows = []


baseline_artifacts = artifact_metrics(
    original_uv
)

baseline_bad = (
    baseline_artifacts[
        "n_artifact_windows"
    ]
)


for condition_name, cleaned_uv in condition_data.items():

    # ------------------------------------------------------------
    # Artifact burden
    # ------------------------------------------------------------

    artifact = artifact_metrics(
        cleaned_uv
    )


    if baseline_bad > 0:

        artifact_reduction = (
            100
            * (
                baseline_bad
                -
                artifact[
                    "n_artifact_windows"
                ]
            )
            /
            baseline_bad
        )

    else:

        artifact_reduction = np.nan


    # ------------------------------------------------------------
    # Signal removed
    # ------------------------------------------------------------

    (
        original_rms,
        removed_rms,
        relative_removed

    ) = signal_metrics(
        original_uv,
        cleaned_uv
    )


    # ------------------------------------------------------------
    # Channel-level table
    # ------------------------------------------------------------

    for idx, channel in enumerate(
        ch_names
    ):

        channel_rows.append({

            "condition":
                condition_name,

            "channel":
                channel,

            "original_rms_uV":
                original_rms[idx],

            "removed_rms_uV":
                removed_rms[idx],

            "removed_percent":
                relative_removed[idx]
        })


    # ------------------------------------------------------------
    # Spectral preservation
    # ------------------------------------------------------------

    freqs_clean, psd_clean = calculate_psd(
        cleaned_uv
    )


    all_band_changes = []


    for band_name, (
        fmin,
        fmax
    ) in BANDS.items():

        mask_original = (
            (freqs_original >= fmin)
            &
            (freqs_original < fmax)
        )

        mask_clean = (
            (freqs_clean >= fmin)
            &
            (freqs_clean < fmax)
        )


        channel_band_changes = []


        for ch_idx in range(
            len(ch_names)
        ):

            before_power = integrate_spectrum(

                psd_original[
                    ch_idx,
                    mask_original
                ],

                freqs_original[
                    mask_original
                ]
            )


            after_power = integrate_spectrum(

                psd_clean[
                    ch_idx,
                    mask_clean
                ],

                freqs_clean[
                    mask_clean
                ]
            )


            if before_power > 0:

                change = (
                    100
                    * (
                        after_power
                        -
                        before_power
                    )
                    /
                    before_power
                )

            else:

                change = np.nan


            channel_band_changes.append(
                change
            )


            band_rows.append({

                "condition":
                    condition_name,

                "channel":
                    ch_names[
                        ch_idx
                    ],

                "band":
                    band_name,

                "change_percent":
                    change
            })


        all_band_changes.extend(
            channel_band_changes
        )


    all_band_changes = np.asarray(
        all_band_changes,
        dtype=float
    )


    # ------------------------------------------------------------
    # Important frontal channels
    # ------------------------------------------------------------

    def channel_removed(
        channel
    ):

        idx = ch_names.index(
            channel
        )

        return float(
            relative_removed[
                idx
            ]
        )


    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------

    summary_rows.append({

        "condition":
            condition_name,

        "excluded_components":
            ",".join(
                [
                    f"ICA{x:03d}"
                    for x in CONDITIONS[
                        condition_name
                    ]
                ]
            )
            if len(
                CONDITIONS[
                    condition_name
                ]
            ) > 0
            else "none",

        "artifact_windows":
            artifact[
                "n_artifact_windows"
            ],

        "artifact_percent":
            artifact[
                "artifact_percent"
            ],

        "artifact_reduction_percent":
            artifact_reduction,

        "median_removed_signal_percent":
            float(
                np.median(
                    relative_removed
                )
            ),

        "mean_removed_signal_percent":
            float(
                np.mean(
                    relative_removed
                )
            ),

        "max_removed_signal_percent":
            float(
                np.max(
                    relative_removed
                )
            ),

        "AF3_removed_percent":
            channel_removed(
                "AF3"
            ),

        "F3_removed_percent":
            channel_removed(
                "F3"
            ),

        "AF4_removed_percent":
            channel_removed(
                "AF4"
            ),

        "T7_removed_percent":
            channel_removed(
                "T7"
            ),

        "median_band_change_percent":
            float(
                np.nanmedian(
                    all_band_changes
                )
            ),

        "median_absolute_band_change_percent":
            float(
                np.nanmedian(
                    np.abs(
                        all_band_changes
                    )
                )
            ),

        "max_absolute_band_change_percent":
            float(
                np.nanmax(
                    np.abs(
                        all_band_changes
                    )
                )
            )
    })


summary = pd.DataFrame(
    summary_rows
)


channel_effects = pd.DataFrame(
    channel_rows
)


band_effects = pd.DataFrame(
    band_rows
)


# ================================================================
# 10. DISPLAY PRIMARY COMPARISON
# ================================================================

print("\n" + "=" * 80)
print("PRIMARY SENSITIVITY COMPARISON")
print("=" * 80)


display(
    summary.round(3)
)


# ================================================================
# 11. COMPONENT-SPECIFIC CHANNEL EFFECTS
# ================================================================

important_channels = [
    "AF3",
    "F3",
    "AF4",
    "T7",
    "F7",
    "F8"
]


important_effects = channel_effects[
    channel_effects[
        "channel"
    ].isin(
        important_channels
    )
].copy()


important_pivot = important_effects.pivot(
    index="channel",
    columns="condition",
    values="removed_percent"
)


print("\n" + "=" * 80)
print("SIGNAL REMOVED FROM IMPORTANT CHANNELS (%)")
print("=" * 80)


display(
    important_pivot.round(3)
)


# ================================================================
# 12. BAND-LEVEL SUMMARY
# ================================================================

band_summary = (

    band_effects

    .groupby(
        [
            "condition",
            "band"
        ]
    )

    .agg(

        mean_change_percent=(
            "change_percent",
            "mean"
        ),

        median_change_percent=(
            "change_percent",
            "median"
        ),

        min_change_percent=(
            "change_percent",
            "min"
        ),

        max_change_percent=(
            "change_percent",
            "max"
        )

    )

    .reset_index()
)


print("\n" + "=" * 80)
print("BAND-POWER SENSITIVITY")
print("=" * 80)


display(
    band_summary.round(3)
)


# ================================================================
# 13. FIGURE — ARTIFACT REDUCTION
# ================================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


ax.bar(
    summary[
        "condition"
    ],
    summary[
        "artifact_percent"
    ]
)


ax.set_ylabel(
    "Flagged 2-s windows (%)"
)


ax.set_xlabel(
    "ICA exclusion strategy"
)


ax.set_title(
    "Residual transient artifact burden"
)


ax.tick_params(
    axis="x",
    rotation=30
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_artifact_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 14. FIGURE — SIGNAL REMOVAL BY CHANNEL
# ================================================================

fig, ax = plt.subplots(
    figsize=(14, 7)
)


for condition in [
    "ICA005_only",
    "ICA012_only",
    "ICA005_ICA012"
]:

    subset = channel_effects[
        channel_effects[
            "condition"
        ]
        ==
        condition
    ]


    subset = (
        subset
        .set_index(
            "channel"
        )
        .reindex(
            ch_names
        )
        .reset_index()
    )


    ax.plot(

        subset[
            "channel"
        ],

        subset[
            "removed_percent"
        ],

        marker="o",

        linewidth=1.2,

        label=condition
    )


ax.set_xlabel(
    "EEG channel"
)


ax.set_ylabel(
    "Removed RMS relative to original (%)"
)


ax.set_title(
    "Spatial effect of ICA exclusion strategy"
)


ax.tick_params(
    axis="x",
    rotation=45
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_channel_signal_removal_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 15. FIGURE — MEDIAN BAND-POWER CHANGE
# ================================================================

median_band = (

    band_summary

    .pivot(
        index="band",
        columns="condition",
        values="median_change_percent"
    )
)


fig, ax = plt.subplots(
    figsize=(12, 7)
)


for condition in [
    "ICA005_only",
    "ICA012_only",
    "ICA005_ICA012"
]:

    if condition not in median_band.columns:
        continue


    ax.plot(

        median_band.index,

        median_band[
            condition
        ],

        marker="o",

        linewidth=1.3,

        label=condition
    )


ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Frequency band"
)


ax.set_ylabel(
    "Median power change (%)"
)


ax.set_title(
    "Spectral sensitivity to ICA exclusion strategy"
)


ax.tick_params(
    axis="x",
    rotation=30
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "03_band_power_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 16. FIGURE — AF4 EFFECT
# ================================================================

nonbaseline = summary[
    summary[
        "condition"
    ]
    !=
    "No_ICA"
]


fig, ax = plt.subplots(
    figsize=(9, 6)
)


ax.bar(

    nonbaseline[
        "condition"
    ],

    nonbaseline[
        "AF4_removed_percent"
    ]
)


ax.set_ylabel(
    "AF4 signal removed (%)"
)


ax.set_xlabel(
    "ICA exclusion strategy"
)


ax.set_title(
    "Sensitivity of AF4 to ICA component removal"
)


ax.tick_params(
    axis="x",
    rotation=30
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "04_AF4_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 17. SAVE TABLES
# ================================================================

summary.to_csv(
    OUT
    / "ica_exclusion_sensitivity_summary.csv",
    index=False
)


channel_effects.to_csv(
    OUT
    / "ica_exclusion_channel_effects.csv",
    index=False
)


band_effects.to_csv(
    OUT
    / "ica_exclusion_band_effects.csv",
    index=False
)


band_summary.to_csv(
    OUT
    / "ica_exclusion_band_summary.csv",
    index=False
)


# ================================================================
# 18. FINAL REPORT
# ================================================================

print("\n" + "=" * 80)
print("TASK 2E COMPLETE")
print("=" * 80)


print(
    "\nThe preferred exclusion should achieve useful "
    "artifact reduction while minimizing excessive "
    "channel-specific and spectral signal loss."
)


print(
    "\nDo NOT choose the strategy solely from the "
    "artifact-window count."
)


print(
    "\nOutput directory:"
)

print(
    OUT
)

In [ ]:
# ================================================================
# TASK 2F — FINAL EPOCH-LEVEL ARTIFACT QC
# ================================================================
#
# FINAL PRIMARY PREPROCESSING DECISION FOR SUB-001
# ------------------------------------------------
# 1–45 Hz band-pass
# average reference
# NO ICA component removal
# NO channel interpolation
# transient artifact rejection at epoch level
#
# This cell validates the final epoch rejection strategy.
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 1. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

SUBJECT = "001"
TASK = "activeai"

EPOCH_SEC = 2.0

PEAK_THRESHOLD_UV = 250.0
PTP_THRESHOLD_UV = 500.0


INPUT_FIF = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02a_preprocessing_qc"
    / f"sub-{SUBJECT}"
    / f"sub-{SUBJECT}_task-{TASK}_desc-filtered_eeg_raw.fif"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task02f_final_epoch_qc"
    / f"sub-{SUBJECT}"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 2. LOAD PRIMARY EEG
# ================================================================

if not INPUT_FIF.exists():
    raise FileNotFoundError(INPUT_FIF)


raw = mne.io.read_raw_fif(
    INPUT_FIF,
    preload=True,
    verbose=False
)


sfreq = float(raw.info["sfreq"])
ch_names = list(raw.ch_names)

data_uv = raw.get_data() * 1e6

n_channels, n_samples = data_uv.shape

duration_sec = n_samples / sfreq


print("=" * 80)
print("TASK 2F — FINAL EPOCH-LEVEL ARTIFACT QC")
print("=" * 80)

print(f"Subject       : sub-{SUBJECT}")
print(f"Task          : {TASK}")
print(f"Channels      : {n_channels}")
print(f"Sampling      : {sfreq:.2f} Hz")
print(f"Duration      : {duration_sec/60:.2f} min")
print(f"Epoch length  : {EPOCH_SEC:.2f} s")
print("ICA removal   : NONE")
print("Interpolation : NONE")


# ================================================================
# 3. CONSTRUCT NON-OVERLAPPING EPOCHS
# ================================================================

epoch_samples = int(
    round(EPOCH_SEC * sfreq)
)

n_epochs = n_samples // epoch_samples

used_samples = n_epochs * epoch_samples

unused_samples = n_samples - used_samples


print("\n" + "=" * 80)
print("EPOCH CONSTRUCTION")
print("=" * 80)

print(f"Samples per epoch : {epoch_samples}")
print(f"Complete epochs   : {n_epochs}")
print(f"Unused samples    : {unused_samples}")


# ================================================================
# 4. EPOCH QC
# ================================================================

epoch_rows = []

channel_failure_matrix = np.zeros(
    (n_epochs, n_channels),
    dtype=bool
)


for epoch_idx in range(n_epochs):

    start = epoch_idx * epoch_samples
    stop = start + epoch_samples

    segment = data_uv[:, start:stop]

    max_abs_by_channel = np.max(
        np.abs(segment),
        axis=1
    )

    ptp_by_channel = np.ptp(
        segment,
        axis=1
    )

    peak_fail = (
        max_abs_by_channel
        > PEAK_THRESHOLD_UV
    )

    ptp_fail = (
        ptp_by_channel
        > PTP_THRESHOLD_UV
    )

    channel_fail = (
        peak_fail | ptp_fail
    )

    channel_failure_matrix[
        epoch_idx
    ] = channel_fail

    n_failed_channels = int(
        channel_fail.sum()
    )

    failed_names = [
        ch_names[i]
        for i in np.where(channel_fail)[0]
    ]

    epoch_rows.append({

        "epoch":
            epoch_idx,

        "start_sec":
            start / sfreq,

        "end_sec":
            stop / sfreq,

        "start_min":
            start / sfreq / 60,

        "max_abs_uV":
            float(
                np.max(
                    max_abs_by_channel
                )
            ),

        "max_ptp_uV":
            float(
                np.max(
                    ptp_by_channel
                )
            ),

        "n_failed_channels":
            n_failed_channels,

        "failed_channels":
            ",".join(
                failed_names
            ),

        "peak_fail":
            bool(
                peak_fail.any()
            ),

        "ptp_fail":
            bool(
                ptp_fail.any()
            ),

        "reject":
            bool(
                channel_fail.any()
            )
    })


epochs_qc = pd.DataFrame(
    epoch_rows
)


# ================================================================
# 5. RETENTION SUMMARY
# ================================================================

n_rejected = int(
    epochs_qc["reject"].sum()
)

n_retained = (
    n_epochs - n_rejected
)

retained_percent = (
    100 * n_retained / n_epochs
)

rejected_percent = (
    100 * n_rejected / n_epochs
)


retained_duration_sec = (
    n_retained * EPOCH_SEC
)

retained_duration_min = (
    retained_duration_sec / 60
)


summary = pd.DataFrame([{

    "subject":
        f"sub-{SUBJECT}",

    "task":
        TASK,

    "total_epochs":
        n_epochs,

    "retained_epochs":
        n_retained,

    "rejected_epochs":
        n_rejected,

    "retained_percent":
        retained_percent,

    "rejected_percent":
        rejected_percent,

    "retained_duration_sec":
        retained_duration_sec,

    "retained_duration_min":
        retained_duration_min,

    "ica_components_removed":
        0,

    "channels_interpolated":
        0

}])


print("\n" + "=" * 80)
print("FINAL EPOCH RETENTION")
print("=" * 80)

display(summary.T)


# ================================================================
# 6. CHANNEL-SPECIFIC FAILURE BURDEN
# ================================================================

channel_rows = []


for ch_idx, ch in enumerate(ch_names):

    failures = int(
        channel_failure_matrix[
            :,
            ch_idx
        ].sum()
    )

    channel_rows.append({

        "channel":
            ch,

        "failed_epochs":
            failures,

        "failed_percent":
            100 * failures / n_epochs

    })


channel_failure = pd.DataFrame(
    channel_rows
).sort_values(
    "failed_percent",
    ascending=False
).reset_index(
    drop=True
)


print("\n" + "=" * 80)
print("CHANNEL-SPECIFIC EPOCH FAILURE")
print("=" * 80)

display(
    channel_failure.round(3)
)


# ================================================================
# 7. NUMBER OF CHANNELS RESPONSIBLE PER REJECTED EPOCH
# ================================================================

rejected_only = epochs_qc[
    epochs_qc["reject"]
].copy()


failure_distribution = (

    rejected_only[
        "n_failed_channels"
    ]

    .value_counts()

    .sort_index()

    .rename_axis(
        "failed_channels_per_epoch"
    )

    .reset_index(
        name="n_epochs"
    )
)


if len(failure_distribution) > 0:

    failure_distribution[
        "percent_of_rejected_epochs"
    ] = (
        100
        * failure_distribution["n_epochs"]
        / n_rejected
    )


print("\n" + "=" * 80)
print("HOW MANY CHANNELS CAUSE EACH REJECTION?")
print("=" * 80)

display(
    failure_distribution.round(3)
)


# ================================================================
# 8. SINGLE-CHANNEL VS MULTI-CHANNEL FAILURES
# ================================================================

single_channel_rejections = int(
    (
        rejected_only[
            "n_failed_channels"
        ] == 1
    ).sum()
)


multi_channel_rejections = int(
    (
        rejected_only[
            "n_failed_channels"
        ] > 1
    ).sum()
)


if n_rejected > 0:

    single_percent = (
        100
        * single_channel_rejections
        / n_rejected
    )

    multi_percent = (
        100
        * multi_channel_rejections
        / n_rejected
    )

else:

    single_percent = 0.0
    multi_percent = 0.0


rejection_structure = pd.DataFrame([{

    "rejected_epochs":
        n_rejected,

    "single_channel_rejections":
        single_channel_rejections,

    "single_channel_percent":
        single_percent,

    "multi_channel_rejections":
        multi_channel_rejections,

    "multi_channel_percent":
        multi_percent

}])


print("\n" + "=" * 80)
print("REJECTION STRUCTURE")
print("=" * 80)

display(
    rejection_structure.T
)


# ================================================================
# 9. WHICH CHANNELS CAUSE SINGLE-CHANNEL REJECTIONS?
# ================================================================

single_rows = []


for _, row in rejected_only[
    rejected_only[
        "n_failed_channels"
    ] == 1
].iterrows():

    single_rows.append(
        row["failed_channels"]
    )


if len(single_rows) > 0:

    single_channel_causes = (

        pd.Series(
            single_rows
        )

        .value_counts()

        .rename_axis(
            "channel"
        )

        .reset_index(
            name="single_channel_rejections"
        )
    )

else:

    single_channel_causes = pd.DataFrame(
        columns=[
            "channel",
            "single_channel_rejections"
        ]
    )


print("\n" + "=" * 80)
print("CAUSES OF SINGLE-CHANNEL REJECTIONS")
print("=" * 80)

display(
    single_channel_causes
)


# ================================================================
# 10. RETENTION IF EACH CHANNEL WERE IGNORED
#
# Diagnostic only.
#
# This does NOT mean we will remove that channel.
# It tells us whether one electrode disproportionately controls
# whole-epoch rejection.
# ================================================================

leave_one_out_rows = []


for ignore_idx, ignore_ch in enumerate(
    ch_names
):

    remaining = np.delete(
        channel_failure_matrix,
        ignore_idx,
        axis=1
    )

    rejection_without_channel = (
        remaining.any(axis=1)
    )

    rejected_without = int(
        rejection_without_channel.sum()
    )

    retained_without = (
        n_epochs
        - rejected_without
    )

    leave_one_out_rows.append({

        "ignored_channel":
            ignore_ch,

        "rejected_epochs":
            rejected_without,

        "retained_epochs":
            retained_without,

        "retained_percent":
            100
            * retained_without
            / n_epochs,

        "additional_epochs_saved":
            n_rejected
            - rejected_without
    })


leave_one_out = pd.DataFrame(
    leave_one_out_rows
).sort_values(
    "additional_epochs_saved",
    ascending=False
).reset_index(
    drop=True
)


print("\n" + "=" * 80)
print("LEAVE-ONE-CHANNEL-OUT DIAGNOSTIC")
print("=" * 80)

display(
    leave_one_out.round(3)
)


# ================================================================
# 11. ARTIFACT TIMELINE
# ================================================================

fig, ax = plt.subplots(
    figsize=(15, 5)
)


accepted = epochs_qc[
    ~epochs_qc["reject"]
]

rejected = epochs_qc[
    epochs_qc["reject"]
]


ax.scatter(
    accepted["start_min"],
    accepted["max_abs_uV"],
    s=12,
    label="Retained"
)


ax.scatter(
    rejected["start_min"],
    rejected["max_abs_uV"],
    s=18,
    label="Rejected"
)


ax.axhline(
    PEAK_THRESHOLD_UV,
    linestyle="--",
    linewidth=1,
    label=f"{PEAK_THRESHOLD_UV:.0f} µV"
)


ax.set_xlabel(
    "Recording time (minutes)"
)

ax.set_ylabel(
    "Maximum |amplitude| per 2-s epoch (µV)"
)

ax.set_title(
    "Final epoch-level artifact screening"
)

ax.legend()

ax.grid(
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    OUT / "01_epoch_artifact_timeline.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 12. CHANNEL FAILURE FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(12, 6)
)


ax.bar(
    channel_failure["channel"],
    channel_failure["failed_percent"]
)


ax.set_xlabel(
    "EEG channel"
)

ax.set_ylabel(
    "Epochs exceeding artifact criterion (%)"
)

ax.set_title(
    "Channel-specific artifact burden"
)

ax.tick_params(
    axis="x",
    rotation=45
)

ax.grid(
    axis="y",
    alpha=0.2
)

plt.tight_layout()

plt.savefig(
    OUT / "02_channel_epoch_failure.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ================================================================
# 13. FAILURE-COUNT FIGURE
# ================================================================

if len(failure_distribution) > 0:

    fig, ax = plt.subplots(
        figsize=(10, 6)
    )

    ax.bar(
        failure_distribution[
            "failed_channels_per_epoch"
        ].astype(str),

        failure_distribution[
            "n_epochs"
        ]
    )

    ax.set_xlabel(
        "Number of channels exceeding threshold"
    )

    ax.set_ylabel(
        "Rejected epochs"
    )

    ax.set_title(
        "Spatial extent of rejected epochs"
    )

    ax.grid(
        axis="y",
        alpha=0.2
    )

    plt.tight_layout()

    plt.savefig(
        OUT / "03_rejection_spatial_extent.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ================================================================
# 14. CREATE BAD_TRANSIENT ANNOTATIONS
# ================================================================

bad_epochs = epochs_qc[
    epochs_qc["reject"]
].copy()


annotations = mne.Annotations(

    onset=bad_epochs[
        "start_sec"
    ].to_numpy(),

    duration=np.full(
        len(bad_epochs),
        EPOCH_SEC
    ),

    description=[
        "BAD_transient"
    ] * len(bad_epochs)

)


raw_annotated = raw.copy()

raw_annotated.set_annotations(
    annotations
)


# ================================================================
# 15. SAVE ANNOTATED PRIMARY EEG
# ================================================================

FINAL_FIF = (
    OUT
    / (
        f"sub-{SUBJECT}_"
        f"task-{TASK}_"
        f"desc-primary_annotated_eeg_raw.fif"
    )
)


raw_annotated.save(
    FINAL_FIF,
    overwrite=True,
    verbose=False
)


# ================================================================
# 16. SAVE TABLES
# ================================================================

epochs_qc.to_csv(
    OUT / "epoch_qc.csv",
    index=False
)


channel_failure.to_csv(
    OUT / "channel_epoch_failure.csv",
    index=False
)


failure_distribution.to_csv(
    OUT / "rejection_spatial_extent.csv",
    index=False
)


single_channel_causes.to_csv(
    OUT / "single_channel_rejection_causes.csv",
    index=False
)


leave_one_out.to_csv(
    OUT / "leave_one_channel_out_diagnostic.csv",
    index=False
)


summary.to_csv(
    OUT / "task02f_summary.csv",
    index=False
)


# ================================================================
# 17. FINAL REPORT
# ================================================================

print("\n" + "=" * 80)
print("TASK 2F COMPLETE")
print("=" * 80)


print("\nFINAL PRIMARY PREPROCESSING:")

print("Band-pass       : 1–45 Hz")
print("Reference       : average")
print("ICA removal     : none")
print("Interpolation   : none")
print(
    f"Epoch criterion : |amplitude| > "
    f"{PEAK_THRESHOLD_UV:.0f} µV OR "
    f"peak-to-peak > {PTP_THRESHOLD_UV:.0f} µV"
)


print("\nRETENTION:")

print(
    f"{n_retained}/{n_epochs} epochs retained "
    f"({retained_percent:.2f}%)"
)

print(
    f"{n_rejected}/{n_epochs} epochs rejected "
    f"({rejected_percent:.2f}%)"
)

print(
    f"Usable EEG = {retained_duration_min:.2f} min"
)


print("\nAnnotated primary EEG:")

print(
    FINAL_FIF
)


print("\nOutput directory:")

print(
    OUT
)


print("\nNEXT STEP:")

print(
    "If retention and channel-specific rejection are acceptable, "
    "this preprocessing strategy can be generalized to all 15 "
    "participants with subject-specific QC."
)

In [ ]:
# ================================================================
# TASK 3A — BATCH PREPROCESSING + QC FOR ALL 15 PARTICIPANTS
# ================================================================
#
# Frozen primary preprocessing protocol:
#
#   1. Load EEGLAB .set + external .fdt manually
#   2. Retain 14 EEG channels
#   3. Convert µV -> V
#   4. Apply standard 10-20 montage
#   5. Band-pass 1–45 Hz
#   6. Average reference
#   7. NO ICA subtraction
#   8. NO channel interpolation
#   9. Reconstruct Emotiv marker pulses
#  10. Apply diagnostic 2-s artifact QC:
#          |amplitude| > 250 µV
#             OR
#          peak-to-peak > 500 µV
#
# IMPORTANT:
# ---------
# The 2-s QC windows in this task characterize subject-level
# recording quality.
#
# They are NOT yet the final experimental epochs.
#
# Final analysis epochs will later be constructed WITHIN
# marker-defined phases so that no epoch crosses a phase boundary.
#
# No T1/T2 marker code is assigned experimental meaning here.
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

from scipy.io import loadmat


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03a_batch_preprocessing_qc"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


EEG_CHANNELS = [

    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


MARKER_CHANNELS = [

    "MarkerIndex",
    "MarkerType",
    "MarkerValueInt"
]


L_FREQ = 1.0

H_FREQ = 45.0


QC_WINDOW_SEC = 2.0

PEAK_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


SUBJECTS = [
    f"{x:03d}"
    for x in range(1, 16)
]


# ================================================================
# 3. HELPER — FIND SUBJECT EEG FILE
# ================================================================

def find_set_file(
    root,
    subject
):

    subject_dir = (
        root
        / f"sub-{subject}"
    )


    if not subject_dir.exists():

        raise FileNotFoundError(
            f"Missing subject directory: {subject_dir}"
        )


    candidates = list(
        subject_dir.rglob(
            "*_eeg.set"
        )
    )


    if len(candidates) == 0:

        raise FileNotFoundError(
            f"No EEG .set file found for sub-{subject}"
        )


    if len(candidates) > 1:

        print(
            f"WARNING: multiple .set files for sub-{subject}:"
        )

        for candidate in candidates:
            print("   ", candidate)


    return sorted(
        candidates
    )[0]


# ================================================================
# 4. HELPER — MATLAB STRING CONVERSION
# ================================================================

def matlab_string(
    value
):

    if isinstance(
        value,
        str
    ):

        return value


    if isinstance(
        value,
        bytes
    ):

        return value.decode(
            errors="ignore"
        )


    if isinstance(
        value,
        np.ndarray
    ):

        if value.size == 0:

            return ""


        if value.dtype.kind in [
            "U",
            "S"
        ]:

            return "".join(
                value.ravel().astype(str)
            )


        if value.size == 1:

            return matlab_string(
                value.item()
            )


    return str(
        value
    )


# ================================================================
# 5. HELPER — EXTRACT CHANNEL LABELS
# ================================================================

def extract_channel_labels(
    chanlocs
):

    labels = []


    arr = np.atleast_1d(
        chanlocs
    ).ravel()


    for item in arr:

        label = None


        if hasattr(
            item,
            "labels"
        ):

            label = item.labels


        elif isinstance(
            item,
            np.void
        ):

            if (
                item.dtype.names
                and
                "labels"
                in item.dtype.names
            ):

                label = item[
                    "labels"
                ]


        if label is None:

            labels.append(
                ""
            )

        else:

            labels.append(
                matlab_string(
                    label
                ).strip()
            )


    return labels


# ================================================================
# 6. HELPER — LOAD EEGLAB PAYLOAD
# ================================================================

def load_eeglab_payload(
    set_file
):

    mat = loadmat(
        set_file,
        squeeze_me=True,
        struct_as_record=False
    )


    nbchan = int(
        np.asarray(
            mat["nbchan"]
        ).squeeze()
    )


    pnts = int(
        np.asarray(
            mat["pnts"]
        ).squeeze()
    )


    trials = int(
        np.asarray(
            mat["trials"]
        ).squeeze()
    )


    sfreq = float(
        np.asarray(
            mat["srate"]
        ).squeeze()
    )


    if trials != 1:

        raise RuntimeError(
            f"Expected continuous EEG; trials={trials}"
        )


    labels = extract_channel_labels(
        mat["chanlocs"]
    )


    if len(labels) != nbchan:

        raise RuntimeError(
            f"Channel-label mismatch: "
            f"{len(labels)} labels vs {nbchan} channels"
        )


    data_field = mat["data"]


    # ------------------------------------------------------------
    # External FDT
    # ------------------------------------------------------------

    if isinstance(
        data_field,
        str
    ):

        fdt_name = data_field.strip()


    elif isinstance(
        data_field,
        bytes
    ):

        fdt_name = data_field.decode().strip()


    elif (
        isinstance(
            data_field,
            np.ndarray
        )
        and
        data_field.dtype.kind
        in ["U", "S"]
    ):

        fdt_name = matlab_string(
            data_field
        ).strip()


    else:

        fdt_name = None


    if fdt_name:

        fdt_file = (
            set_file.parent
            / fdt_name
        )


        if not fdt_file.exists():

            raise FileNotFoundError(
                f"Missing FDT file: {fdt_file}"
            )


        expected_values = (
            nbchan
            * pnts
            * trials
        )


        raw_values = np.fromfile(
            fdt_file,
            dtype="<f4"
        )


        if raw_values.size != expected_values:

            raise RuntimeError(
                f"FDT size mismatch for {set_file.name}: "
                f"{raw_values.size:,} values found; "
                f"{expected_values:,} expected."
            )


        data = raw_values.reshape(
            (
                nbchan,
                pnts
            ),
            order="F"
        )


    # ------------------------------------------------------------
    # Embedded data
    # ------------------------------------------------------------

    else:

        data = np.asarray(
            data_field,
            dtype=float
        )


        if data.shape != (
            nbchan,
            pnts
        ):

            try:

                data = data.reshape(
                    (
                        nbchan,
                        pnts
                    ),
                    order="F"
                )

            except Exception as exc:

                raise RuntimeError(
                    f"Could not reshape embedded EEG data: {exc}"
                )


    return {

        "data":
            data,

        "labels":
            labels,

        "sfreq":
            sfreq,

        "pnts":
            pnts,

        "nbchan":
            nbchan,

        "mat":
            mat
    }


# ================================================================
# 7. HELPER — RECONSTRUCT MARKER PULSES
# ================================================================

def reconstruct_markers(
    full_data,
    labels,
    sfreq
):

    missing = [

        channel

        for channel in MARKER_CHANNELS

        if channel not in labels
    ]


    if missing:

        raise RuntimeError(
            f"Missing marker streams: {missing}"
        )


    idx_marker_index = labels.index(
        "MarkerIndex"
    )

    idx_marker_type = labels.index(
        "MarkerType"
    )

    idx_marker_value = labels.index(
        "MarkerValueInt"
    )


    marker_index = full_data[
        idx_marker_index
    ]

    marker_type = full_data[
        idx_marker_type
    ]

    marker_value = full_data[
        idx_marker_value
    ]


    # Marker pulses are represented by non-zero MarkerIndex samples.
    pulse_samples = np.flatnonzero(
        marker_index != 0
    )


    rows = []


    for sample in pulse_samples:

        m_index = int(
            round(
                marker_index[
                    sample
                ]
            )
        )


        m_type = int(
            round(
                marker_type[
                    sample
                ]
            )
        )


        m_value = int(
            round(
                marker_value[
                    sample
                ]
            )
        )


        rows.append({

            "sample":
                int(sample),

            "onset_sec":
                float(
                    sample / sfreq
                ),

            "MarkerIndex":
                m_index,

            "MarkerType":
                m_type,

            "MarkerValueInt":
                m_value,

            "marker_code":
                f"T{m_type}_V{m_value}"
        })


    return pd.DataFrame(
        rows
    )


# ================================================================
# 8. HELPER — CREATE MNE EEG
# ================================================================

def create_mne_raw(
    full_data,
    labels,
    sfreq
):

    missing = [

        channel

        for channel in EEG_CHANNELS

        if channel not in labels
    ]


    if missing:

        raise RuntimeError(
            f"Missing EEG channels: {missing}"
        )


    eeg_indices = [

        labels.index(
            channel
        )

        for channel in EEG_CHANNELS
    ]


    # Original Emotiv EEG values are µV.
    eeg_uv = full_data[
        eeg_indices
    ].astype(
        np.float64
    )


    eeg_v = (
        eeg_uv
        * 1e-6
    )


    info = mne.create_info(

        ch_names=EEG_CHANNELS,

        sfreq=sfreq,

        ch_types=[
            "eeg"
        ] * len(
            EEG_CHANNELS
        )
    )


    raw = mne.io.RawArray(
        eeg_v,
        info,
        verbose=False
    )


    montage = mne.channels.make_standard_montage(
        "standard_1020"
    )


    raw.set_montage(
        montage,
        on_missing="raise",
        verbose=False
    )


    return raw


# ================================================================
# 9. HELPER — PREPROCESS
# ================================================================

def preprocess_raw(
    raw
):

    processed = raw.copy()


    processed.filter(

        l_freq=L_FREQ,

        h_freq=H_FREQ,

        method="fir",

        phase="zero",

        fir_design="firwin",

        verbose=False
    )


    processed.set_eeg_reference(

        ref_channels="average",

        projection=False,

        verbose=False
    )


    return processed


# ================================================================
# 10. HELPER — QC WINDOWS
# ================================================================

def calculate_qc(
    raw
):

    sfreq = float(
        raw.info["sfreq"]
    )


    data_uv = (
        raw.get_data()
        * 1e6
    )


    n_channels, n_samples = (
        data_uv.shape
    )


    window_samples = int(
        round(
            QC_WINDOW_SEC
            * sfreq
        )
    )


    n_windows = (
        n_samples
        // window_samples
    )


    channel_failure = np.zeros(
        (
            n_windows,
            n_channels
        ),
        dtype=bool
    )


    rows = []


    for window in range(
        n_windows
    ):

        start = (
            window
            * window_samples
        )

        stop = (
            start
            + window_samples
        )


        segment = data_uv[
            :,
            start:stop
        ]


        max_abs_by_channel = np.max(
            np.abs(
                segment
            ),
            axis=1
        )


        ptp_by_channel = np.ptp(
            segment,
            axis=1
        )


        peak_fail = (
            max_abs_by_channel
            >
            PEAK_THRESHOLD_UV
        )


        ptp_fail = (
            ptp_by_channel
            >
            PTP_THRESHOLD_UV
        )


        fail = (
            peak_fail
            |
            ptp_fail
        )


        channel_failure[
            window
        ] = fail


        failed_names = [

            EEG_CHANNELS[i]

            for i in np.where(
                fail
            )[0]
        ]


        rows.append({

            "window":
                window,

            "start_sec":
                start / sfreq,

            "end_sec":
                stop / sfreq,

            "max_abs_uV":
                float(
                    np.max(
                        max_abs_by_channel
                    )
                ),

            "max_ptp_uV":
                float(
                    np.max(
                        ptp_by_channel
                    )
                ),

            "n_failed_channels":
                int(
                    fail.sum()
                ),

            "failed_channels":
                ",".join(
                    failed_names
                ),

            "reject":
                bool(
                    fail.any()
                )
        })


    qc_windows = pd.DataFrame(
        rows
    )


    channel_rows = []


    for channel_idx, channel in enumerate(
        EEG_CHANNELS
    ):

        n_fail = int(
            channel_failure[
                :,
                channel_idx
            ].sum()
        )


        channel_rows.append({

            "channel":
                channel,

            "failed_windows":
                n_fail,

            "failed_percent":
                (
                    100
                    * n_fail
                    / n_windows
                )
                if n_windows
                else np.nan
        })


    channel_qc = pd.DataFrame(
        channel_rows
    )


    return (
        qc_windows,
        channel_qc,
        channel_failure
    )


# ================================================================
# 11. PROCESS ALL PARTICIPANTS
# ================================================================

subject_summaries = []

all_channel_qc = []

all_markers = []

errors = []


for subject in SUBJECTS:

    print("\n" + "=" * 80)
    print(f"PROCESSING sub-{subject}")
    print("=" * 80)


    try:

        # --------------------------------------------------------
        # Locate .set
        # --------------------------------------------------------

        set_file = find_set_file(
            ROOT,
            subject
        )


        print(
            f"SET: {set_file}"
        )


        # --------------------------------------------------------
        # Load EEGLAB payload
        # --------------------------------------------------------

        payload = load_eeglab_payload(
            set_file
        )


        full_data = payload[
            "data"
        ]

        labels = payload[
            "labels"
        ]

        sfreq = payload[
            "sfreq"
        ]

        pnts = payload[
            "pnts"
        ]


        duration_sec = (
            pnts
            / sfreq
        )


        # --------------------------------------------------------
        # Task group
        # --------------------------------------------------------

        if int(
            subject
        ) <= 8:

            group = "Active"

            task = "activeai"

        else:

            group = "Passive"

            task = "passiveai"


        # --------------------------------------------------------
        # Reconstruct markers
        # --------------------------------------------------------

        markers = reconstruct_markers(

            full_data,

            labels,

            sfreq
        )


        markers.insert(
            0,
            "subject",
            f"sub-{subject}"
        )


        markers.insert(
            1,
            "group",
            group
        )


        all_markers.append(
            markers
        )


        # --------------------------------------------------------
        # Create EEG Raw
        # --------------------------------------------------------

        raw = create_mne_raw(

            full_data,

            labels,

            sfreq
        )


        # --------------------------------------------------------
        # Preprocess
        # --------------------------------------------------------

        processed = preprocess_raw(
            raw
        )


        # --------------------------------------------------------
        # QC
        # --------------------------------------------------------

        (
            qc_windows,
            channel_qc,
            failure_matrix

        ) = calculate_qc(
            processed
        )


        n_windows = len(
            qc_windows
        )


        n_rejected = int(
            qc_windows[
                "reject"
            ].sum()
        )


        n_retained = (
            n_windows
            -
            n_rejected
        )


        retained_percent = (
            100
            * n_retained
            / n_windows
        )


        rejected_percent = (
            100
            * n_rejected
            / n_windows
        )


        # --------------------------------------------------------
        # Single-channel rejection structure
        # --------------------------------------------------------

        rejected_rows = qc_windows[
            qc_windows[
                "reject"
            ]
        ]


        single_channel_rejections = int(
            (
                rejected_rows[
                    "n_failed_channels"
                ]
                ==
                1
            ).sum()
        )


        multi_channel_rejections = int(
            (
                rejected_rows[
                    "n_failed_channels"
                ]
                >
                1
            ).sum()
        )


        # --------------------------------------------------------
        # Worst channel
        # --------------------------------------------------------

        worst_row = (
            channel_qc
            .sort_values(
                "failed_percent",
                ascending=False
            )
            .iloc[0]
        )


        worst_channel = (
            worst_row[
                "channel"
            ]
        )


        worst_channel_percent = float(
            worst_row[
                "failed_percent"
            ]
        )


        # --------------------------------------------------------
        # Leave-worst-channel-out diagnostic
        # --------------------------------------------------------

        worst_idx = EEG_CHANNELS.index(
            worst_channel
        )


        remaining_matrix = np.delete(

            failure_matrix,

            worst_idx,

            axis=1
        )


        rejected_without_worst = int(
            remaining_matrix.any(
                axis=1
            ).sum()
        )


        retained_without_worst = (
            n_windows
            -
            rejected_without_worst
        )


        retention_without_worst = (
            100
            * retained_without_worst
            / n_windows
        )


        additional_saved = (
            n_rejected
            -
            rejected_without_worst
        )


        # --------------------------------------------------------
        # Add subject metadata to QC tables
        # --------------------------------------------------------

        qc_windows.insert(
            0,
            "subject",
            f"sub-{subject}"
        )


        qc_windows.insert(
            1,
            "group",
            group
        )


        channel_qc.insert(
            0,
            "subject",
            f"sub-{subject}"
        )


        channel_qc.insert(
            1,
            "group",
            group
        )


        all_channel_qc.append(
            channel_qc
        )


        # --------------------------------------------------------
        # Subject output directory
        # --------------------------------------------------------

        subject_out = (
            OUT
            / f"sub-{subject}"
        )


        subject_out.mkdir(
            parents=True,
            exist_ok=True
        )


        # --------------------------------------------------------
        # Create BAD_transient annotations
        #
        # Diagnostic whole-recording windows only.
        # Later analysis will construct epochs inside marker phases.
        # --------------------------------------------------------

        bad_windows = qc_windows[
            qc_windows[
                "reject"
            ]
        ]


        if len(
            bad_windows
        ) > 0:

            annotations = mne.Annotations(

                onset=bad_windows[
                    "start_sec"
                ].to_numpy(),

                duration=np.full(
                    len(
                        bad_windows
                    ),
                    QC_WINDOW_SEC
                ),

                description=[
                    "BAD_transient"
                ]
                * len(
                    bad_windows
                )
            )


            processed_annotated = (
                processed.copy()
            )


            processed_annotated.set_annotations(
                annotations
            )


        else:

            processed_annotated = (
                processed.copy()
            )


        # --------------------------------------------------------
        # Save preprocessed annotated EEG
        # --------------------------------------------------------

        output_fif = (

            subject_out

            / (
                f"sub-{subject}_"
                f"task-{task}_"
                f"desc-primary_annotated_eeg_raw.fif"
            )
        )


        processed_annotated.save(
            output_fif,
            overwrite=True,
            verbose=False
        )


        # --------------------------------------------------------
        # Save subject tables
        # --------------------------------------------------------

        qc_windows.to_csv(

            subject_out
            / "qc_windows.csv",

            index=False
        )


        channel_qc.to_csv(

            subject_out
            / "channel_qc.csv",

            index=False
        )


        markers.to_csv(

            subject_out
            / "reconstructed_markers.csv",

            index=False
        )


        # --------------------------------------------------------
        # Summary
        # --------------------------------------------------------

        subject_summaries.append({

            "subject":
                f"sub-{subject}",

            "group":
                group,

            "task":
                task,

            "sfreq_hz":
                sfreq,

            "duration_sec":
                duration_sec,

            "duration_min":
                duration_sec / 60,

            "marker_count":
                len(
                    markers
                ),

            "qc_windows":
                n_windows,

            "retained_windows":
                n_retained,

            "rejected_windows":
                n_rejected,

            "retained_percent":
                retained_percent,

            "rejected_percent":
                rejected_percent,

            "retained_minutes":
                (
                    n_retained
                    * QC_WINDOW_SEC
                    / 60
                ),

            "single_channel_rejections":
                single_channel_rejections,

            "multi_channel_rejections":
                multi_channel_rejections,

            "worst_channel":
                worst_channel,

            "worst_channel_failed_percent":
                worst_channel_percent,

            "retention_without_worst_channel_percent":
                retention_without_worst,

            "additional_windows_saved_without_worst":
                additional_saved,

            "ica_removed":
                0,

            "channels_interpolated":
                0,

            "status":
                "SUCCESS"
        })


        print(
            f"Markers       : {len(markers)}"
        )


        print(
            f"QC retention  : "
            f"{n_retained}/{n_windows} "
            f"({retained_percent:.2f}%)"
        )


        print(
            f"Worst channel : "
            f"{worst_channel} "
            f"({worst_channel_percent:.2f}% failed)"
        )


        print(
            "Saved          : "
            f"{output_fif.name}"
        )


    except Exception as exc:

        print(
            f"ERROR sub-{subject}: {exc}"
        )


        errors.append({

            "subject":
                f"sub-{subject}",

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 12. COMBINE TABLES
# ================================================================

summary_df = pd.DataFrame(
    subject_summaries
)


if len(
    all_channel_qc
) > 0:

    channel_qc_df = pd.concat(
        all_channel_qc,
        ignore_index=True
    )

else:

    channel_qc_df = pd.DataFrame()


if len(
    all_markers
) > 0:

    markers_df = pd.concat(
        all_markers,
        ignore_index=True
    )

else:

    markers_df = pd.DataFrame()


errors_df = pd.DataFrame(
    errors
)


# ================================================================
# 13. SAVE MASTER TABLES
# ================================================================

summary_df.to_csv(
    OUT
    / "all_subjects_preprocessing_qc.csv",
    index=False
)


channel_qc_df.to_csv(
    OUT
    / "all_subjects_channel_qc.csv",
    index=False
)


markers_df.to_csv(
    OUT
    / "all_subjects_reconstructed_markers.csv",
    index=False
)


errors_df.to_csv(
    OUT
    / "processing_errors.csv",
    index=False
)


# ================================================================
# 14. DISPLAY SUBJECT SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("ALL-SUBJECT PREPROCESSING SUMMARY")
print("=" * 80)


if len(
    summary_df
) > 0:

    display(

        summary_df[
            [
                "subject",
                "group",
                "duration_min",
                "marker_count",
                "retained_windows",
                "rejected_windows",
                "retained_percent",
                "worst_channel",
                "worst_channel_failed_percent",
                "retention_without_worst_channel_percent"
            ]
        ].round(3)

    )


# ================================================================
# 15. GROUP-LEVEL QC SUMMARY
# ================================================================

if len(
    summary_df
) > 0:

    group_summary = (

        summary_df

        .groupby(
            "group"
        )

        .agg(

            n_subjects=(
                "subject",
                "count"
            ),

            mean_duration_min=(
                "duration_min",
                "mean"
            ),

            mean_retention_percent=(
                "retained_percent",
                "mean"
            ),

            sd_retention_percent=(
                "retained_percent",
                "std"
            ),

            min_retention_percent=(
                "retained_percent",
                "min"
            ),

            median_retention_percent=(
                "retained_percent",
                "median"
            ),

            max_retention_percent=(
                "retained_percent",
                "max"
            )

        )

        .reset_index()
    )


    print("\n" + "=" * 80)
    print("GROUP-LEVEL QC")
    print("=" * 80)


    display(
        group_summary.round(3)
    )


    group_summary.to_csv(
        OUT
        / "group_level_qc.csv",
        index=False
    )


# ================================================================
# 16. CHANNEL FAILURE SUMMARY ACROSS SUBJECTS
# ================================================================

if len(
    channel_qc_df
) > 0:

    channel_summary = (

        channel_qc_df

        .groupby(
            "channel"
        )

        .agg(

            mean_failed_percent=(
                "failed_percent",
                "mean"
            ),

            median_failed_percent=(
                "failed_percent",
                "median"
            ),

            max_failed_percent=(
                "failed_percent",
                "max"
            )

        )

        .reset_index()

        .sort_values(
            "mean_failed_percent",
            ascending=False
        )
    )


    print("\n" + "=" * 80)
    print("CHANNEL QC ACROSS ALL SUBJECTS")
    print("=" * 80)


    display(
        channel_summary.round(3)
    )


    channel_summary.to_csv(
        OUT
        / "channel_qc_group_summary.csv",
        index=False
    )


# ================================================================
# 17. FIGURE — SUBJECT RETENTION
# ================================================================

if len(
    summary_df
) > 0:

    plot_summary = summary_df.copy()


    fig, ax = plt.subplots(
        figsize=(14, 6)
    )


    ax.bar(

        plot_summary[
            "subject"
        ],

        plot_summary[
            "retained_percent"
        ]
    )


    ax.axhline(
        80,
        linestyle="--",
        linewidth=1,
        label="80% reference"
    )


    ax.set_xlabel(
        "Participant"
    )


    ax.set_ylabel(
        "Retained 2-s QC windows (%)"
    )


    ax.set_title(
        "EEG data retention after primary artifact screening"
    )


    ax.tick_params(
        axis="x",
        rotation=45
    )


    ax.set_ylim(
        0,
        105
    )


    ax.legend()


    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "01_subject_retention.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 18. FIGURE — WORST CHANNEL BY SUBJECT
# ================================================================

if len(
    summary_df
) > 0:

    fig, ax = plt.subplots(
        figsize=(14, 6)
    )


    ax.bar(

        summary_df[
            "subject"
        ],

        summary_df[
            "worst_channel_failed_percent"
        ]
    )


    for i, row in summary_df.reset_index(
        drop=True
    ).iterrows():

        ax.text(

            i,

            row[
                "worst_channel_failed_percent"
            ]
            + 0.5,

            row[
                "worst_channel"
            ],

            ha="center",

            va="bottom",

            fontsize=8
        )


    ax.set_xlabel(
        "Participant"
    )


    ax.set_ylabel(
        "Worst-channel failed windows (%)"
    )


    ax.set_title(
        "Most artifact-prone EEG channel by participant"
    )


    ax.tick_params(
        axis="x",
        rotation=45
    )


    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "02_worst_channel_by_subject.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 19. FIGURE — CHANNEL QC ACROSS SUBJECTS
# ================================================================

if len(
    channel_qc_df
) > 0:

    channel_plot = (

        channel_qc_df

        .groupby(
            "channel"
        )[
            "failed_percent"
        ]

        .mean()

        .reindex(
            EEG_CHANNELS
        )
    )


    fig, ax = plt.subplots(
        figsize=(12, 6)
    )


    ax.bar(
        channel_plot.index,
        channel_plot.values
    )


    ax.set_xlabel(
        "EEG channel"
    )


    ax.set_ylabel(
        "Mean failed 2-s windows (%)"
    )


    ax.set_title(
        "Mean artifact burden by EEG channel"
    )


    ax.tick_params(
        axis="x",
        rotation=45
    )


    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(
        OUT
        / "03_mean_channel_artifact_burden.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 20. MARKER SEQUENCES
# ================================================================

print("\n" + "=" * 80)
print("RECONSTRUCTED MARKER SEQUENCES")
print("=" * 80)


if len(
    markers_df
) > 0:

    for subject in markers_df[
        "subject"
    ].unique():

        sequence = (

            markers_df[
                markers_df[
                    "subject"
                ]
                ==
                subject
            ][
                "marker_code"
            ]

            .tolist()
        )


        print(
            f"{subject}: "
            +
            " → ".join(
                sequence
            )
        )


# ================================================================
# 21. ERROR REPORT
# ================================================================

print("\n" + "=" * 80)
print("PROCESSING ERRORS")
print("=" * 80)


if len(
    errors_df
) == 0:

    print(
        "None — all participants processed successfully."
    )

else:

    display(
        errors_df
    )


# ================================================================
# 22. FINAL REPORT
# ================================================================

print("\n" + "=" * 80)
print("TASK 3A COMPLETE")
print("=" * 80)


print(
    f"Successful participants : "
    f"{len(summary_df)}/15"
)


print(
    f"Processing errors       : "
    f"{len(errors_df)}"
)


print(
    "\nPrimary preprocessing:"
)


print(
    "  Band-pass       : 1–45 Hz"
)


print(
    "  Reference       : average"
)


print(
    "  ICA removal     : none"
)


print(
    "  Interpolation   : none"
)


print(
    "  QC window       : 2 s"
)


print(
    "  Peak threshold  : "
    f"{PEAK_THRESHOLD_UV:.0f} µV"
)


print(
    "  PTP threshold   : "
    f"{PTP_THRESHOLD_UV:.0f} µV"
)


print(
    "\nIMPORTANT:"
)


print(
    "These whole-recording 2-s windows are QC diagnostics."
)


print(
    "Final scientific epochs will be generated separately "
    "within marker-defined experimental phases."
)


print(
    "No T1/T2 numeric marker has been assigned an "
    "experimental meaning."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)

In [ ]:
# ================================================================
# TASK 3B — MARKER-DEFINED PHASE RECONSTRUCTION + CLEAN EPOCH AUDIT
# ================================================================
#
# PURPOSE
# -------
# Reconstruct experimental intervals from the raw Emotiv markers
# for all 15 participants WITHOUT assigning experimental meaning
# to T1_V1–V4 or T2_V1/V3.
#
# For every marker-defined interval:
#
#   1. determine start/end/duration;
#   2. generate 2-s epochs INSIDE the interval;
#   3. ensure epochs never cross marker boundaries;
#   4. apply the frozen artifact rule:
#          |amplitude| > 250 µV OR
#          peak-to-peak > 500 µV;
#   5. calculate clean epochs and clean duration;
#   6. compare marker structure across groups.
#
# IMPORTANT
# ---------
# Repeated identical markers are retained as distinct boundaries.
# No numeric marker is assigned semantic meaning.
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK3A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03a_batch_preprocessing_qc"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03b_phase_reconstruction"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


EPOCH_SEC = 2.0

PEAK_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


SUBJECTS = [
    f"{x:03d}"
    for x in range(1, 16)
]


# ================================================================
# 3. LOAD MASTER MARKER TABLE
# ================================================================

MARKER_FILE = (
    TASK3A
    / "all_subjects_reconstructed_markers.csv"
)

if not MARKER_FILE.exists():

    raise FileNotFoundError(
        MARKER_FILE
    )


markers = pd.read_csv(
    MARKER_FILE
)


required_columns = [

    "subject",
    "group",
    "sample",
    "onset_sec",
    "MarkerIndex",
    "MarkerType",
    "MarkerValueInt",
    "marker_code"
]


missing = [

    column

    for column in required_columns

    if column not in markers.columns
]


if missing:

    raise RuntimeError(
        f"Missing marker columns: {missing}"
    )


# ================================================================
# 4. HELPER — FIND PREPROCESSED FIF
# ================================================================

def find_fif(
    subject
):

    subject_dir = (
        TASK3A
        / f"sub-{subject}"
    )


    candidates = list(
        subject_dir.glob(
            "*desc-primary_annotated_eeg_raw.fif"
        )
    )


    if len(candidates) != 1:

        raise RuntimeError(
            f"Expected exactly one FIF for sub-{subject}; "
            f"found {len(candidates)}"
        )


    return candidates[0]


# ================================================================
# 5. HELPER — EPOCH QC
# ================================================================

def epoch_is_clean(
    segment_uv
):

    max_abs = float(
        np.max(
            np.abs(
                segment_uv
            )
        )
    )


    max_ptp = float(
        np.max(
            np.ptp(
                segment_uv,
                axis=1
            )
        )
    )


    clean = (

        max_abs
        <=
        PEAK_THRESHOLD_UV

        and

        max_ptp
        <=
        PTP_THRESHOLD_UV
    )


    return (
        clean,
        max_abs,
        max_ptp
    )


# ================================================================
# 6. RECONSTRUCT PHASE INTERVALS
# ================================================================

phase_rows = []

epoch_rows = []

errors = []


for subject in SUBJECTS:

    subject_name = (
        f"sub-{subject}"
    )


    print("\n" + "=" * 80)

    print(
        f"PROCESSING {subject_name}"
    )

    print("=" * 80)


    try:

        # --------------------------------------------------------
        # Subject markers
        # --------------------------------------------------------

        subject_markers = (

            markers[
                markers[
                    "subject"
                ]
                ==
                subject_name
            ]

            .sort_values(
                "sample"
            )

            .reset_index(
                drop=True
            )
        )


        if len(
            subject_markers
        ) == 0:

            raise RuntimeError(
                "No reconstructed markers found."
            )


        # --------------------------------------------------------
        # Load primary preprocessed EEG
        # --------------------------------------------------------

        fif_file = find_fif(
            subject
        )


        raw = mne.io.read_raw_fif(
            fif_file,
            preload=True,
            verbose=False
        )


        sfreq = float(
            raw.info[
                "sfreq"
            ]
        )


        data_uv = (
            raw.get_data()
            * 1e6
        )


        n_samples = (
            raw.n_times
        )


        recording_end_sec = (
            n_samples
            / sfreq
        )


        epoch_samples = int(
            round(
                EPOCH_SEC
                * sfreq
            )
        )


        group = str(
            subject_markers.loc[
                0,
                "group"
            ]
        )


        # --------------------------------------------------------
        # Each marker starts an interval.
        #
        # End = next marker.
        # Final marker interval ends at recording end.
        # --------------------------------------------------------

        for marker_position in range(
            len(
                subject_markers
            )
        ):

            marker = subject_markers.iloc[
                marker_position
            ]


            start_sample = int(
                marker[
                    "sample"
                ]
            )


            start_sec = (
                start_sample
                / sfreq
            )


            if (
                marker_position
                <
                len(
                    subject_markers
                )
                - 1
            ):

                next_marker = (
                    subject_markers.iloc[
                        marker_position + 1
                    ]
                )


                end_sample = int(
                    next_marker[
                        "sample"
                    ]
                )


                interval_end_source = (
                    "next_marker"
                )


            else:

                end_sample = (
                    n_samples
                )


                interval_end_source = (
                    "recording_end"
                )


            # ----------------------------------------------------
            # Protect against malformed boundaries
            # ----------------------------------------------------

            end_sample = min(
                end_sample,
                n_samples
            )


            if (
                end_sample
                <=
                start_sample
            ):

                continue


            end_sec = (
                end_sample
                / sfreq
            )


            duration_sec = (
                end_sec
                -
                start_sec
            )


            # ----------------------------------------------------
            # Generate complete 2-s epochs WITHIN this interval
            # ----------------------------------------------------

            available_samples = (
                end_sample
                -
                start_sample
            )


            n_complete_epochs = (
                available_samples
                //
                epoch_samples
            )


            clean_epochs = 0

            rejected_epochs = 0


            for local_epoch in range(
                n_complete_epochs
            ):

                epoch_start_sample = (

                    start_sample

                    +

                    local_epoch
                    * epoch_samples
                )


                epoch_end_sample = (

                    epoch_start_sample

                    +

                    epoch_samples
                )


                segment_uv = data_uv[
                    :,
                    epoch_start_sample:
                    epoch_end_sample
                ]


                (
                    clean,
                    max_abs,
                    max_ptp

                ) = epoch_is_clean(
                    segment_uv
                )


                if clean:

                    clean_epochs += 1

                else:

                    rejected_epochs += 1


                epoch_rows.append({

                    "subject":
                        subject_name,

                    "group":
                        group,

                    "marker_position":
                        marker_position + 1,

                    "marker_code":
                        marker[
                            "marker_code"
                        ],

                    "MarkerIndex":
                        int(
                            marker[
                                "MarkerIndex"
                            ]
                        ),

                    "MarkerType":
                        int(
                            marker[
                                "MarkerType"
                            ]
                        ),

                    "MarkerValueInt":
                        int(
                            marker[
                                "MarkerValueInt"
                            ]
                        ),

                    "local_epoch":
                        local_epoch + 1,

                    "epoch_start_sample":
                        epoch_start_sample,

                    "epoch_end_sample":
                        epoch_end_sample,

                    "epoch_start_sec":
                        epoch_start_sample
                        / sfreq,

                    "epoch_end_sec":
                        epoch_end_sample
                        / sfreq,

                    "max_abs_uV":
                        max_abs,

                    "max_ptp_uV":
                        max_ptp,

                    "clean":
                        clean
                })


            # ----------------------------------------------------
            # Phase retention
            # ----------------------------------------------------

            if n_complete_epochs > 0:

                clean_percent = (

                    100
                    * clean_epochs
                    / n_complete_epochs
                )

            else:

                clean_percent = np.nan


            phase_rows.append({

                "subject":
                    subject_name,

                "group":
                    group,

                "marker_position":
                    marker_position + 1,

                "marker_code":
                    marker[
                        "marker_code"
                    ],

                "MarkerIndex":
                    int(
                        marker[
                            "MarkerIndex"
                        ]
                    ),

                "MarkerType":
                    int(
                        marker[
                            "MarkerType"
                        ]
                    ),

                "MarkerValueInt":
                    int(
                        marker[
                            "MarkerValueInt"
                        ]
                    ),

                "start_sample":
                    start_sample,

                "end_sample":
                    end_sample,

                "start_sec":
                    start_sec,

                "end_sec":
                    end_sec,

                "duration_sec":
                    duration_sec,

                "duration_min":
                    duration_sec / 60,

                "interval_end_source":
                    interval_end_source,

                "complete_2s_epochs":
                    n_complete_epochs,

                "clean_2s_epochs":
                    clean_epochs,

                "rejected_2s_epochs":
                    rejected_epochs,

                "clean_percent":
                    clean_percent,

                "clean_duration_sec":
                    clean_epochs
                    * EPOCH_SEC,

                "clean_duration_min":
                    clean_epochs
                    * EPOCH_SEC
                    / 60
            })


        print(
            f"Markers        : "
            f"{len(subject_markers)}"
        )


        print(
            f"Intervals      : "
            f"{len(subject_markers)}"
        )


        print(
            f"Recording      : "
            f"{recording_end_sec/60:.2f} min"
        )


    except Exception as exc:

        print(
            f"ERROR: {exc}"
        )


        errors.append({

            "subject":
                subject_name,

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 7. CREATE DATAFRAMES
# ================================================================

phases = pd.DataFrame(
    phase_rows
)


epochs = pd.DataFrame(
    epoch_rows
)


errors_df = pd.DataFrame(
    errors
)


# ================================================================
# 8. SAVE MASTER TABLES
# ================================================================

phases.to_csv(
    OUT
    / "all_subjects_marker_defined_phases.csv",
    index=False
)


epochs.to_csv(
    OUT
    / "all_subjects_phase_epochs.csv",
    index=False
)


errors_df.to_csv(
    OUT
    / "task03b_errors.csv",
    index=False
)


# ================================================================
# 9. DISPLAY ALL PHASES
# ================================================================

print("\n" + "=" * 80)

print(
    "MARKER-DEFINED PHASE INTERVALS"
)

print("=" * 80)


display(

    phases[
        [
            "subject",
            "group",
            "marker_position",
            "marker_code",
            "MarkerIndex",
            "start_sec",
            "end_sec",
            "duration_sec",
            "complete_2s_epochs",
            "clean_2s_epochs",
            "clean_percent"
        ]
    ].round(3)

)


# ================================================================
# 10. PHASE SUMMARY BY GROUP + MARKER CODE
# ================================================================

phase_summary = (

    phases

    .groupby(
        [
            "group",
            "marker_code"
        ]
    )

    .agg(

        n_intervals=(
            "marker_code",
            "size"
        ),

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_duration_sec=(
            "duration_sec",
            "mean"
        ),

        median_duration_sec=(
            "duration_sec",
            "median"
        ),

        min_duration_sec=(
            "duration_sec",
            "min"
        ),

        max_duration_sec=(
            "duration_sec",
            "max"
        ),

        total_clean_epochs=(
            "clean_2s_epochs",
            "sum"
        ),

        median_clean_epochs=(
            "clean_2s_epochs",
            "median"
        ),

        mean_clean_percent=(
            "clean_percent",
            "mean"
        )

    )

    .reset_index()
)


print("\n" + "=" * 80)

print(
    "PHASE SUMMARY BY GROUP AND MARKER CODE"
)

print("=" * 80)


display(
    phase_summary.round(3)
)


phase_summary.to_csv(
    OUT
    / "phase_summary_by_group_marker.csv",
    index=False
)


# ================================================================
# 11. SUBJECT × MARKER OCCURRENCE COUNTS
# ================================================================

occurrence_counts = (

    phases

    .groupby(
        [
            "subject",
            "group",
            "marker_code"
        ]
    )

    .size()

    .reset_index(
        name="n_intervals"
    )
)


occurrence_pivot = (

    occurrence_counts

    .pivot_table(

        index=[
            "subject",
            "group"
        ],

        columns="marker_code",

        values="n_intervals",

        fill_value=0
    )

    .reset_index()
)


print("\n" + "=" * 80)

print(
    "MARKER OCCURRENCES BY PARTICIPANT"
)

print("=" * 80)


display(
    occurrence_pivot
)


occurrence_pivot.to_csv(
    OUT
    / "marker_occurrences_by_subject.csv",
    index=False
)


# ================================================================
# 12. CLEAN EPOCHS BY SUBJECT × MARKER
# ================================================================

clean_by_marker = (

    phases

    .groupby(
        [
            "subject",
            "group",
            "marker_code"
        ]
    )

    .agg(

        intervals=(
            "marker_code",
            "size"
        ),

        total_duration_sec=(
            "duration_sec",
            "sum"
        ),

        complete_epochs=(
            "complete_2s_epochs",
            "sum"
        ),

        clean_epochs=(
            "clean_2s_epochs",
            "sum"
        ),

        rejected_epochs=(
            "rejected_2s_epochs",
            "sum"
        )

    )

    .reset_index()
)


clean_by_marker[
    "clean_percent"
] = (

    100

    * clean_by_marker[
        "clean_epochs"
    ]

    /

    clean_by_marker[
        "complete_epochs"
    ].replace(
        0,
        np.nan
    )
)


clean_by_marker[
    "clean_duration_sec"
] = (

    clean_by_marker[
        "clean_epochs"
    ]

    * EPOCH_SEC
)


print("\n" + "=" * 80)

print(
    "CLEAN DATA BY SUBJECT AND MARKER"
)

print("=" * 80)


display(
    clean_by_marker.round(3)
)


clean_by_marker.to_csv(
    OUT
    / "clean_data_by_subject_marker.csv",
    index=False
)


# ================================================================
# 13. CROSS-GROUP MARKER AVAILABILITY
# ================================================================

cross_group = (

    phases

    .groupby(
        [
            "marker_code",
            "group"
        ]
    )[
        "subject"
    ]

    .nunique()

    .unstack(
        fill_value=0
    )

    .reset_index()
)


for column in [
    "Active",
    "Passive"
]:

    if column not in cross_group.columns:

        cross_group[
            column
        ] = 0


cross_group[
    "present_in_both_groups"
] = (

    (
        cross_group[
            "Active"
        ]
        >
        0
    )

    &

    (
        cross_group[
            "Passive"
        ]
        >
        0
    )
)


print("\n" + "=" * 80)

print(
    "CROSS-GROUP MARKER AVAILABILITY"
)

print("=" * 80)


display(
    cross_group
)


cross_group.to_csv(
    OUT
    / "cross_group_marker_availability.csv",
    index=False
)


# ================================================================
# 14. INTERVAL POSITION TABLE
#
# Useful because repeated identical codes may represent
# different protocol stages.
# ================================================================

position_summary = (

    phases

    .groupby(
        [
            "group",
            "marker_position",
            "marker_code"
        ]
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        median_duration_sec=(
            "duration_sec",
            "median"
        ),

        median_clean_epochs=(
            "clean_2s_epochs",
            "median"
        ),

        mean_clean_percent=(
            "clean_percent",
            "mean"
        )

    )

    .reset_index()
)


print("\n" + "=" * 80)

print(
    "MARKER POSITION SUMMARY"
)

print("=" * 80)


display(
    position_summary.round(3)
)


position_summary.to_csv(
    OUT
    / "marker_position_summary.csv",
    index=False
)


# ================================================================
# 15. FIGURE — INTERVAL DURATIONS
# ================================================================

fig, ax = plt.subplots(
    figsize=(13, 7)
)


marker_order = sorted(
    phases[
        "marker_code"
    ].unique()
)


plot_data = [

    phases[
        phases[
            "marker_code"
        ]
        ==
        marker
    ][
        "duration_sec"
    ].to_numpy()

    for marker in marker_order
]


ax.boxplot(

    plot_data,

    tick_labels=marker_order,

    showfliers=True
)


ax.set_xlabel(
    "Raw marker code"
)


ax.set_ylabel(
    "Interval duration (s)"
)


ax.set_title(
    "Duration of marker-defined EEG intervals"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_marker_interval_durations.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 16. FIGURE — CLEAN EPOCH AVAILABILITY
# ================================================================

fig, ax = plt.subplots(
    figsize=(14, 7)
)


for group in [
    "Active",
    "Passive"
]:

    subset = clean_by_marker[
        clean_by_marker[
            "group"
        ]
        ==
        group
    ]


    if len(
        subset
    ) == 0:

        continue


    group_marker = (

        subset

        .groupby(
            "marker_code"
        )[
            "clean_epochs"
        ]

        .median()

        .reindex(
            marker_order
        )
    )


    ax.plot(

        marker_order,

        group_marker,

        marker="o",

        linewidth=1.5,

        label=group
    )


ax.set_xlabel(
    "Raw marker code"
)


ax.set_ylabel(
    "Median clean 2-s epochs per participant"
)


ax.set_title(
    "Usable EEG by raw marker code and group"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_clean_epoch_availability.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 17. FIGURE — PARTICIPANT PHASE TIMELINES
# ================================================================

fig, ax = plt.subplots(
    figsize=(15, 9)
)


subject_order = (

    phases[
        "subject"
    ]

    .drop_duplicates()

    .tolist()
)


y_map = {

    subject:
        idx

    for idx, subject in enumerate(
        subject_order
    )
}


for _, row in phases.iterrows():

    y = y_map[
        row[
            "subject"
        ]
    ]


    ax.plot(

        [
            row[
                "start_sec"
            ]
            / 60,

            row[
                "end_sec"
            ]
            / 60
        ],

        [
            y,
            y
        ],

        linewidth=6
    )


    ax.text(

        (
            row[
                "start_sec"
            ]
            +
            row[
                "end_sec"
            ]
        )
        /
        2
        /
        60,

        y + 0.12,

        row[
            "marker_code"
        ],

        fontsize=6,

        ha="center"
    )


ax.set_yticks(
    range(
        len(
            subject_order
        )
    )
)


ax.set_yticklabels(
    subject_order
)


ax.set_xlabel(
    "Recording time (minutes)"
)


ax.set_ylabel(
    "Participant"
)


ax.set_title(
    "Raw marker-defined recording structure"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "03_participant_phase_timelines.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 18. BASIC COMPARABILITY AUDIT
# ================================================================

comparison_rows = []


for marker_code in sorted(
    phases[
        "marker_code"
    ].unique()
):

    active = clean_by_marker[
        (
            clean_by_marker[
                "group"
            ]
            ==
            "Active"
        )
        &
        (
            clean_by_marker[
                "marker_code"
            ]
            ==
            marker_code
        )
    ]


    passive = clean_by_marker[
        (
            clean_by_marker[
                "group"
            ]
            ==
            "Passive"
        )
        &
        (
            clean_by_marker[
                "marker_code"
            ]
            ==
            marker_code
        )
    ]


    comparison_rows.append({

        "marker_code":
            marker_code,

        "active_subjects":
            active[
                "subject"
            ].nunique(),

        "passive_subjects":
            passive[
                "subject"
            ].nunique(),

        "active_median_clean_epochs":
            (
                active[
                    "clean_epochs"
                ].median()
                if len(active)
                else np.nan
            ),

        "passive_median_clean_epochs":
            (
                passive[
                    "clean_epochs"
                ].median()
                if len(passive)
                else np.nan
            ),

        "candidate_cross_group":
            bool(
                active[
                    "subject"
                ].nunique()
                > 0
                and
                passive[
                    "subject"
                ].nunique()
                > 0
            )
    })


comparability = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 80)

print(
    "PRELIMINARY CROSS-GROUP COMPARABILITY"
)

print("=" * 80)


display(
    comparability.round(3)
)


comparability.to_csv(
    OUT
    / "preliminary_cross_group_comparability.csv",
    index=False
)


# ================================================================
# 19. ERRORS
# ================================================================

print("\n" + "=" * 80)

print(
    "PROCESSING ERRORS"
)

print("=" * 80)


if len(
    errors_df
) == 0:

    print(
        "None."
    )

else:

    display(
        errors_df
    )


# ================================================================
# 20. FINAL REPORT
# ================================================================

print("\n" + "=" * 80)

print(
    "TASK 3B COMPLETE"
)

print("=" * 80)


print(
    f"Participants processed : "
    f"{phases['subject'].nunique()}/15"
)


print(
    f"Marker intervals       : "
    f"{len(phases)}"
)


print(
    f"Phase-contained epochs : "
    f"{len(epochs)}"
)


if len(
    epochs
) > 0:

    print(
        f"Clean epochs           : "
        f"{int(epochs['clean'].sum())}"
    )


    print(
        f"Rejected epochs        : "
        f"{int((~epochs['clean']).sum())}"
    )


print(
    "\nNo numeric marker meaning was assigned."
)


print(
    "Repeated marker codes were retained as "
    "separate protocol intervals."
)


print(
    "Epochs were generated within marker boundaries "
    "and cannot cross from one phase into another."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)

In [ ]:
# ================================================================
# TASK 3C — BASELINE SPECTRAL VALIDATION
# ================================================================
#
# Compare the two standardized ~30-s baseline blocks:
#
#   Baseline A = marker position 1 = T2_V1
#   Baseline B = marker position 3 = T2_V3
#
# PURPOSE
# -------
# Determine whether the two baseline blocks exhibit systematically
# different EEG spectral patterns, particularly posterior alpha.
#
# IMPORTANT
# ---------
# We DO NOT assign Eyes Open / Eyes Closed labels.
#
# A stronger posterior-alpha condition will only be described as
# "physiologically consistent with eyes-closed resting EEG" unless
# an authoritative protocol mapping is recovered.
#
# Primary statistical unit = participant (N=15)
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK3A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03a_batch_preprocessing_qc"
)

TASK3B = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03b_phase_reconstruction"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03c_baseline_spectral_validation"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. ANALYSIS SETTINGS
# ================================================================

EPOCH_SEC = 2.0

PEAK_THRESHOLD_UV = 250.0
PTP_THRESHOLD_UV = 500.0

PSD_FMIN = 1.0
PSD_FMAX = 45.0


BANDS = {

    "delta_1_4": (1.0, 4.0),

    "theta_4_8": (4.0, 8.0),

    "alpha_8_13": (8.0, 13.0),

    "beta_13_30": (13.0, 30.0),

    "gamma_30_45": (30.0, 45.0)
}


POSTERIOR_CHANNELS = [
    "P7",
    "O1",
    "O2",
    "P8"
]


BASELINES = {

    "Baseline_A_T2_V1": {
        "marker_position": 1,
        "marker_code": "T2_V1"
    },

    "Baseline_B_T2_V3": {
        "marker_position": 3,
        "marker_code": "T2_V3"
    }
}


SUBJECTS = [
    f"{x:03d}"
    for x in range(1, 16)
]


# ================================================================
# 4. LOAD TASK 3B PHASE TABLE
# ================================================================

PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)

if not PHASE_FILE.exists():
    raise FileNotFoundError(
        PHASE_FILE
    )


phases = pd.read_csv(
    PHASE_FILE
)


# ================================================================
# 5. HELPER — FIND PREPROCESSED FIF
# ================================================================

def find_fif(subject):

    subject_dir = (
        TASK3A
        / f"sub-{subject}"
    )

    candidates = list(
        subject_dir.glob(
            "*desc-primary_annotated_eeg_raw.fif"
        )
    )

    if len(candidates) != 1:

        raise RuntimeError(
            f"Expected one FIF for sub-{subject}; "
            f"found {len(candidates)}"
        )

    return candidates[0]


# ================================================================
# 6. HELPER — ARTIFACT CHECK
# ================================================================

def epoch_is_clean(segment_uv):

    max_abs = np.max(
        np.abs(segment_uv)
    )

    max_ptp = np.max(
        np.ptp(
            segment_uv,
            axis=1
        )
    )

    return (
        max_abs <= PEAK_THRESHOLD_UV
        and
        max_ptp <= PTP_THRESHOLD_UV
    )


# ================================================================
# 7. HELPER — BAND INTEGRATION
# ================================================================

def integrate_band(
    freqs,
    psd,
    low,
    high
):

    mask = (
        (freqs >= low)
        &
        (freqs < high)
    )

    if mask.sum() < 2:
        return np.nan

    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(
            psd[..., mask],
            freqs[mask],
            axis=-1
        )

    return np.trapz(
        psd[..., mask],
        freqs[mask],
        axis=-1
    )


# ================================================================
# 8. EXTRACT CLEAN BASELINE EPOCHS
# ================================================================

spectral_rows = []

epoch_count_rows = []

subject_psd = {}

info_template = None


for subject in SUBJECTS:

    subject_name = (
        f"sub-{subject}"
    )

    print("\n" + "=" * 80)
    print(f"PROCESSING {subject_name}")
    print("=" * 80)


    # ------------------------------------------------------------
    # Load EEG
    # ------------------------------------------------------------

    fif_file = find_fif(
        subject
    )

    raw = mne.io.read_raw_fif(
        fif_file,
        preload=True,
        verbose=False
    )

    sfreq = float(
        raw.info["sfreq"]
    )

    ch_names = list(
        raw.ch_names
    )

    data_uv = (
        raw.get_data()
        * 1e6
    )

    if info_template is None:

        info_template = (
            raw.info.copy()
        )


    epoch_samples = int(
        round(
            EPOCH_SEC
            * sfreq
        )
    )


    # ------------------------------------------------------------
    # Subject group
    # ------------------------------------------------------------

    subject_phase_rows = phases[
        phases["subject"]
        ==
        subject_name
    ]

    if len(subject_phase_rows) == 0:

        raise RuntimeError(
            f"No Task 3B phases for {subject_name}"
        )

    group = str(
        subject_phase_rows.iloc[0][
            "group"
        ]
    )


    # ------------------------------------------------------------
    # Process Baseline A and B
    # ------------------------------------------------------------

    for baseline_name, definition in BASELINES.items():

        marker_position = (
            definition[
                "marker_position"
            ]
        )

        expected_code = (
            definition[
                "marker_code"
            ]
        )


        phase = phases[
            (
                phases["subject"]
                ==
                subject_name
            )
            &
            (
                phases["marker_position"]
                ==
                marker_position
            )
        ]


        if len(phase) != 1:

            raise RuntimeError(
                f"{subject_name}: expected one interval "
                f"at position {marker_position}; "
                f"found {len(phase)}"
            )


        phase = phase.iloc[0]


        observed_code = str(
            phase[
                "marker_code"
            ]
        )


        if observed_code != expected_code:

            raise RuntimeError(
                f"{subject_name}: position "
                f"{marker_position} expected "
                f"{expected_code}, observed "
                f"{observed_code}"
            )


        start_sample = int(
            phase[
                "start_sample"
            ]
        )

        end_sample = int(
            phase[
                "end_sample"
            ]
        )


        available_samples = (
            end_sample
            -
            start_sample
        )


        n_epochs = (
            available_samples
            //
            epoch_samples
        )


        clean_segments = []


        for local_epoch in range(
            n_epochs
        ):

            epoch_start = (
                start_sample
                +
                local_epoch
                * epoch_samples
            )

            epoch_end = (
                epoch_start
                +
                epoch_samples
            )


            segment_uv = data_uv[
                :,
                epoch_start:epoch_end
            ]


            if epoch_is_clean(
                segment_uv
            ):

                clean_segments.append(
                    segment_uv
                )


        n_clean = len(
            clean_segments
        )


        epoch_count_rows.append({

            "subject":
                subject_name,

            "group":
                group,

            "baseline":
                baseline_name,

            "marker_position":
                marker_position,

            "marker_code":
                observed_code,

            "available_epochs":
                n_epochs,

            "clean_epochs":
                n_clean,

            "rejected_epochs":
                n_epochs - n_clean,

            "clean_percent":
                (
                    100
                    * n_clean
                    / n_epochs
                    if n_epochs
                    else np.nan
                )
        })


        if n_clean == 0:

            print(
                f"{baseline_name}: "
                "NO CLEAN EPOCHS"
            )

            continue


        clean_segments = np.stack(
            clean_segments,
            axis=0
        )


        # shape:
        # epochs × channels × samples


        # --------------------------------------------------------
        # Welch PSD for each clean epoch
        # --------------------------------------------------------

        nperseg = min(
            epoch_samples,
            int(
                2 * sfreq
            )
        )


        freqs, psd = welch(

            clean_segments,

            fs=sfreq,

            nperseg=nperseg,

            noverlap=nperseg // 2,

            axis=-1,

            detrend="constant"
        )


        # PSD units:
        # µV² / Hz


        freq_mask = (
            (freqs >= PSD_FMIN)
            &
            (freqs <= PSD_FMAX)
        )


        freqs_selected = (
            freqs[
                freq_mask
            ]
        )


        psd_selected = (
            psd[
                ...,
                freq_mask
            ]
        )


        # --------------------------------------------------------
        # Subject-level PSD:
        # median across clean epochs
        # --------------------------------------------------------

        median_psd = np.median(
            psd_selected,
            axis=0
        )


        subject_psd[
            (
                subject_name,
                baseline_name
            )
        ] = {

            "freqs":
                freqs_selected,

            "psd":
                median_psd
        }


        # --------------------------------------------------------
        # Total 1–45 Hz power for relative power
        # --------------------------------------------------------

        total_power = integrate_band(

            freqs_selected,

            psd_selected,

            PSD_FMIN,

            PSD_FMAX + 1e-9
        )


        # --------------------------------------------------------
        # Band powers
        # --------------------------------------------------------

        for band_name, (
            low,
            high
        ) in BANDS.items():

            band_power = integrate_band(

                freqs_selected,

                psd_selected,

                low,

                high
            )


            relative_power = (

                band_power
                /
                total_power
            )


            # Median across clean epochs
            absolute_subject = np.median(
                band_power,
                axis=0
            )


            relative_subject = np.median(
                relative_power,
                axis=0
            )


            for ch_idx, channel in enumerate(
                ch_names
            ):

                spectral_rows.append({

                    "subject":
                        subject_name,

                    "group":
                        group,

                    "baseline":
                        baseline_name,

                    "marker_position":
                        marker_position,

                    "marker_code":
                        observed_code,

                    "band":
                        band_name,

                    "channel":
                        channel,

                    "absolute_power_uV2":
                        float(
                            absolute_subject[
                                ch_idx
                            ]
                        ),

                    "relative_power":
                        float(
                            relative_subject[
                                ch_idx
                            ]
                        ),

                    "clean_epochs":
                        n_clean
                })


        print(
            f"{baseline_name}: "
            f"{n_clean}/{n_epochs} clean epochs"
        )


# ================================================================
# 9. DATAFRAMES
# ================================================================

spectral = pd.DataFrame(
    spectral_rows
)

epoch_counts = pd.DataFrame(
    epoch_count_rows
)


spectral.to_csv(
    OUT
    / "baseline_channel_band_power.csv",
    index=False
)


epoch_counts.to_csv(
    OUT
    / "baseline_epoch_counts.csv",
    index=False
)


# ================================================================
# 10. DISPLAY EPOCH AVAILABILITY
# ================================================================

print("\n" + "=" * 80)
print("BASELINE EPOCH AVAILABILITY")
print("=" * 80)


epoch_pivot = (

    epoch_counts

    .pivot_table(

        index=[
            "subject",
            "group"
        ],

        columns="baseline",

        values="clean_epochs"
    )

    .reset_index()
)


display(
    epoch_pivot
)


# ================================================================
# 11. POSTERIOR ALPHA
# ================================================================

posterior_alpha = spectral[
    (
        spectral["band"]
        ==
        "alpha_8_13"
    )
    &
    (
        spectral["channel"]
        .isin(
            POSTERIOR_CHANNELS
        )
    )
].copy()


posterior_alpha_subject = (

    posterior_alpha

    .groupby(
        [
            "subject",
            "group",
            "baseline"
        ]
    )

    .agg(

        posterior_alpha_absolute=(
            "absolute_power_uV2",
            "mean"
        ),

        posterior_alpha_relative=(
            "relative_power",
            "mean"
        )

    )

    .reset_index()
)


posterior_alpha_subject.to_csv(
    OUT
    / "posterior_alpha_by_subject.csv",
    index=False
)


print("\n" + "=" * 80)
print("POSTERIOR ALPHA BY PARTICIPANT")
print("=" * 80)


display(
    posterior_alpha_subject.round(6)
)


# ================================================================
# 12. PAIRED POSTERIOR-ALPHA TABLE
# ================================================================

alpha_pivot = (

    posterior_alpha_subject

    .pivot(

        index="subject",

        columns="baseline",

        values="posterior_alpha_relative"
    )

    .dropna()
)


alpha_pivot[
    "B_minus_A"
] = (

    alpha_pivot[
        "Baseline_B_T2_V3"
    ]

    -

    alpha_pivot[
        "Baseline_A_T2_V1"
    ]
)


alpha_pivot[
    "B_over_A"
] = (

    alpha_pivot[
        "Baseline_B_T2_V3"
    ]

    /

    alpha_pivot[
        "Baseline_A_T2_V1"
    ]
)


print("\n" + "=" * 80)
print("PAIRED POSTERIOR-ALPHA COMPARISON")
print("=" * 80)


display(
    alpha_pivot.round(6)
)


# ================================================================
# 13. WILCOXON PAIRED TEST
# ================================================================

A = alpha_pivot[
    "Baseline_A_T2_V1"
].to_numpy()


B = alpha_pivot[
    "Baseline_B_T2_V3"
].to_numpy()


if len(A) >= 2:

    try:

        stat,
        p_value = wilcoxon(
            B,
            A,
            alternative="two-sided",
            zero_method="wilcox"
        )

    except Exception:

        stat = np.nan
        p_value = np.nan

else:

    stat = np.nan
    p_value = np.nan


# ================================================================
# 14. PAIRED EFFECT SIZE — DZ
# ================================================================

difference = (
    B - A
)


if (
    len(difference) > 1
    and
    np.std(
        difference,
        ddof=1
    ) > 0
):

    cohen_dz = (

        np.mean(
            difference
        )

        /

        np.std(
            difference,
            ddof=1
        )
    )

else:

    cohen_dz = np.nan


n_B_gt_A = int(
    np.sum(
        B > A
    )
)


n_A_gt_B = int(
    np.sum(
        A > B
    )
)


alpha_test = pd.DataFrame([{

    "n_subjects":
        len(
            alpha_pivot
        ),

    "mean_A":
        np.mean(
            A
        ),

    "mean_B":
        np.mean(
            B
        ),

    "median_A":
        np.median(
            A
        ),

    "median_B":
        np.median(
            B
        ),

    "mean_B_minus_A":
        np.mean(
            difference
        ),

    "median_B_minus_A":
        np.median(
            difference
        ),

    "B_greater_A_subjects":
        n_B_gt_A,

    "A_greater_B_subjects":
        n_A_gt_B,

    "wilcoxon_statistic":
        stat,

    "wilcoxon_p":
        p_value,

    "cohen_dz":
        cohen_dz
}])


print("\n" + "=" * 80)
print("POSTERIOR-ALPHA STATISTICAL SUMMARY")
print("=" * 80)


display(
    alpha_test.T
)


alpha_test.to_csv(
    OUT
    / "posterior_alpha_statistical_test.csv",
    index=False
)


# ================================================================
# 15. ALL-BAND SUBJECT-LEVEL SUMMARY
# ================================================================

subject_band = (

    spectral

    .groupby(
        [
            "subject",
            "group",
            "baseline",
            "band"
        ]
    )

    .agg(

        mean_absolute_power=(
            "absolute_power_uV2",
            "mean"
        ),

        mean_relative_power=(
            "relative_power",
            "mean"
        )

    )

    .reset_index()
)


subject_band.to_csv(
    OUT
    / "whole_scalp_band_power_by_subject.csv",
    index=False
)


band_summary = (

    subject_band

    .groupby(
        [
            "baseline",
            "band"
        ]
    )

    .agg(

        mean_relative_power=(
            "mean_relative_power",
            "mean"
        ),

        sd_relative_power=(
            "mean_relative_power",
            "std"
        ),

        median_relative_power=(
            "mean_relative_power",
            "median"
        )

    )

    .reset_index()
)


print("\n" + "=" * 80)
print("WHOLE-SCALP RELATIVE BAND POWER")
print("=" * 80)


display(
    band_summary.round(6)
)


band_summary.to_csv(
    OUT
    / "whole_scalp_band_summary.csv",
    index=False
)


# ================================================================
# 16. FIGURE — PAIRED POSTERIOR ALPHA
# ================================================================

fig, ax = plt.subplots(
    figsize=(8, 7)
)


for subject in alpha_pivot.index:

    y1 = alpha_pivot.loc[
        subject,
        "Baseline_A_T2_V1"
    ]

    y2 = alpha_pivot.loc[
        subject,
        "Baseline_B_T2_V3"
    ]


    ax.plot(
        [0, 1],
        [y1, y2],
        marker="o",
        linewidth=1,
        alpha=0.75
    )


ax.set_xticks(
    [0, 1]
)


ax.set_xticklabels(
    [
        "Baseline A\nT2_V1",
        "Baseline B\nT2_V3"
    ]
)


ax.set_ylabel(
    "Posterior relative alpha power"
)


ax.set_title(
    "Within-participant posterior alpha"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "01_paired_posterior_alpha.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 17. FIGURE — GROUP-MEAN PSD
# ================================================================

fig, ax = plt.subplots(
    figsize=(11, 7)
)


for baseline_name in BASELINES.keys():

    psds = []


    for subject in SUBJECTS:

        key = (
            f"sub-{subject}",
            baseline_name
        )


        if key not in subject_psd:
            continue


        entry = subject_psd[
            key
        ]


        posterior_idx = [

            list(
                info_template[
                    "ch_names"
                ]
            ).index(
                channel
            )

            for channel in POSTERIOR_CHANNELS
        ]


        posterior_psd = np.mean(

            entry[
                "psd"
            ][
                posterior_idx,
                :
            ],

            axis=0
        )


        psds.append(
            posterior_psd
        )


    psds = np.asarray(
        psds
    )


    mean_psd = np.mean(
        psds,
        axis=0
    )


    sem_psd = (

        np.std(
            psds,
            axis=0,
            ddof=1
        )

        /

        np.sqrt(
            psds.shape[0]
        )
    )


    freqs_plot = entry[
        "freqs"
    ]


    ax.plot(
        freqs_plot,
        10 * np.log10(
            mean_psd
        ),
        linewidth=2,
        label=baseline_name
    )


ax.axvspan(
    8,
    13,
    alpha=0.1
)


ax.set_xlim(
    1,
    30
)


ax.set_xlabel(
    "Frequency (Hz)"
)


ax.set_ylabel(
    "Posterior PSD (dB, µV²/Hz)"
)


ax.set_title(
    "Posterior power spectra during the two baseline blocks"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    / "02_posterior_psd_baselines.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 18. TOPOGRAPHIES — RELATIVE ALPHA
# ================================================================

alpha_channel_subject = spectral[
    spectral[
        "band"
    ]
    ==
    "alpha_8_13"
].copy()


topo_values = {}


for baseline_name in BASELINES.keys():

    subset = alpha_channel_subject[
        alpha_channel_subject[
            "baseline"
        ]
        ==
        baseline_name
    ]


    channel_means = (

        subset

        .groupby(
            "channel"
        )[
            "relative_power"
        ]

        .mean()

        .reindex(
            info_template[
                "ch_names"
            ]
        )
    )


    topo_values[
        baseline_name
    ] = channel_means.to_numpy()


# Common scale
all_topo_values = np.concatenate(
    list(
        topo_values.values()
    )
)


vmin = float(
    np.nanmin(
        all_topo_values
    )
)


vmax = float(
    np.nanmax(
        all_topo_values
    )
)


for baseline_name, values in topo_values.items():

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )


    mne.viz.plot_topomap(

        values,

        info_template,

        axes=ax,

        show=False,

        contours=6,

        vlim=(
            vmin,
            vmax
        )
    )


    ax.set_title(
        f"Relative alpha power\n{baseline_name}"
    )


    plt.tight_layout()


    safe_name = (
        baseline_name
        .replace(
            " ",
            "_"
        )
    )


    plt.savefig(
        OUT
        / f"03_topomap_{safe_name}.png",
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 19. DIFFERENCE TOPOGRAPHY — B MINUS A
# ================================================================

difference_topo = (

    topo_values[
        "Baseline_B_T2_V3"
    ]

    -

    topo_values[
        "Baseline_A_T2_V1"
    ]
)


abs_limit = float(
    np.nanmax(
        np.abs(
            difference_topo
        )
    )
)


fig, ax = plt.subplots(
    figsize=(7, 6)
)


mne.viz.plot_topomap(

    difference_topo,

    info_template,

    axes=ax,

    show=False,

    contours=6,

    vlim=(
        -abs_limit,
        abs_limit
    )
)


ax.set_title(
    "Relative alpha difference\nBaseline B − Baseline A"
)


plt.tight_layout()


plt.savefig(
    OUT
    / "04_topomap_alpha_difference_B_minus_A.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 20. CHANNEL-WISE ALPHA DIFFERENCES
# ================================================================

channel_alpha = (

    alpha_channel_subject

    .pivot_table(

        index=[
            "subject",
            "channel"
        ],

        columns="baseline",

        values="relative_power"
    )

    .dropna()

    .reset_index()
)


channel_alpha[
    "B_minus_A"
] = (

    channel_alpha[
        "Baseline_B_T2_V3"
    ]

    -

    channel_alpha[
        "Baseline_A_T2_V1"
    ]
)


channel_alpha_summary = (

    channel_alpha

    .groupby(
        "channel"
    )

    .agg(

        mean_B_minus_A=(
            "B_minus_A",
            "mean"
        ),

        median_B_minus_A=(
            "B_minus_A",
            "median"
        ),

        n_B_greater_A=(
            "B_minus_A",
            lambda x:
                int(
                    np.sum(
                        x > 0
                    )
                )
        )

    )

    .reset_index()

    .sort_values(
        "mean_B_minus_A",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("CHANNEL-WISE RELATIVE ALPHA DIFFERENCE")
print("=" * 80)


display(
    channel_alpha_summary.round(6)
)


channel_alpha_summary.to_csv(
    OUT
    / "channel_alpha_difference_summary.csv",
    index=False
)


# ================================================================
# 21. AUTOMATIC PHYSIOLOGICAL INTERPRETATION
# ================================================================

median_difference = float(
    np.median(
        difference
    )
)


print("\n" + "=" * 80)
print("PHYSIOLOGICAL VALIDATION")
print("=" * 80)


print(
    f"Participants with B > A posterior alpha : "
    f"{n_B_gt_A}/{len(alpha_pivot)}"
)


print(
    f"Participants with A > B posterior alpha : "
    f"{n_A_gt_B}/{len(alpha_pivot)}"
)


print(
    f"Median posterior-alpha difference B-A   : "
    f"{median_difference:.6f}"
)


print(
    f"Wilcoxon p-value                        : "
    f"{p_value:.6g}"
)


print(
    f"Paired Cohen's dz                       : "
    f"{cohen_dz:.3f}"
)


if (
    n_B_gt_A
    >
    n_A_gt_B
):

    print(
        "\nBaseline B (T2_V3) shows the stronger "
        "posterior-alpha tendency."
    )

    print(
        "If the effect is spatially posterior and sufficiently "
        "consistent, this would be physiologically compatible "
        "with an eyes-closed resting state."
    )

else:

    print(
        "\nBaseline A (T2_V1) shows the stronger "
        "posterior-alpha tendency."
    )

    print(
        "If the effect is spatially posterior and sufficiently "
        "consistent, this would be physiologically compatible "
        "with an eyes-closed resting state."
    )


print(
    "\nThis is physiological inference only."
)


print(
    "Do NOT rename either baseline Eyes Open or Eyes Closed "
    "without independent protocol evidence."
)


# ================================================================
# 22. FINAL REPORT
# ================================================================

print("\n" + "=" * 80)
print("TASK 3C COMPLETE")
print("=" * 80)


print(
    f"Participants analyzed : "
    f"{alpha_pivot.shape[0]}/15"
)


print(
    "Baseline A           : position 1 / T2_V1"
)


print(
    "Baseline B           : position 3 / T2_V3"
)


print(
    "Epoch duration       : 2 s"
)


print(
    "Posterior channels   : "
    + ", ".join(
        POSTERIOR_CHANNELS
    )
)


print(
    "\nPrimary question:"
)


print(
    "Does either baseline show a reproducible posterior-alpha "
    "enhancement compatible with the expected eyes-closed EEG "
    "signature?"
)


print(
    "\nOutput directory:"
)


print(
    OUT
)

In [ ]:
# ================================================================
# TASK 3C — BASELINE SPECTRAL VALIDATION
# ================================================================
#
# PURPOSE
# -------
# Compare the two standardized baseline blocks shared by all
# participants:
#
#   Baseline A = marker position 1 = T2_V1
#   Baseline B = marker position 3 = T2_V3
#
# We test whether either baseline exhibits a reproducible
# posterior-alpha enhancement.
#
# IMPORTANT
# ---------
# We DO NOT assign Eyes Open / Eyes Closed labels.
#
# If one condition shows stronger posterior alpha, it will only be
# described as "physiologically consistent with an eyes-closed
# resting state" unless an authoritative protocol mapping is found.
#
# Statistical unit:
#   Participant (N = 15)
#
# Primary posterior ROI:
#   P7, O1, O2, P8
#
# Primary spectral measure:
#   Relative alpha power (8–13 Hz)
#
# Primary paired comparison:
#   Baseline B - Baseline A
#
# Preprocessing inherited from Task 3A:
#   Band-pass       : 1–45 Hz
#   Reference       : average
#   ICA removal     : none
#   Interpolation   : none
#
# Artifact rejection:
#   2-s phase-contained epochs
#   Reject if:
#       |amplitude| > 250 µV
#       OR
#       peak-to-peak > 500 µV
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from scipy.stats import wilcoxon


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. CONFIGURATION
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)


TASK3A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03a_batch_preprocessing_qc"
)


TASK3B = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03b_phase_reconstruction"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03c_baseline_spectral_validation"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. ANALYSIS SETTINGS
# ================================================================

EPOCH_SEC = 2.0


PEAK_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


PSD_FMIN = 1.0

PSD_FMAX = 45.0


BANDS = {

    "delta_1_4":
        (1.0, 4.0),

    "theta_4_8":
        (4.0, 8.0),

    "alpha_8_13":
        (8.0, 13.0),

    "beta_13_30":
        (13.0, 30.0),

    "gamma_30_45":
        (30.0, 45.0)
}


POSTERIOR_CHANNELS = [

    "P7",
    "O1",
    "O2",
    "P8"
]


BASELINES = {

    "Baseline_A_T2_V1": {

        "marker_position":
            1,

        "marker_code":
            "T2_V1"
    },

    "Baseline_B_T2_V3": {

        "marker_position":
            3,

        "marker_code":
            "T2_V3"
    }
}


SUBJECTS = [

    f"{x:03d}"

    for x in range(
        1,
        16
    )
]


# ================================================================
# 4. LOAD TASK 3B PHASE TABLE
# ================================================================

PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)


if not PHASE_FILE.exists():

    raise FileNotFoundError(
        f"Task 3B phase file not found:\n{PHASE_FILE}"
    )


phases = pd.read_csv(
    PHASE_FILE
)


required_phase_columns = [

    "subject",
    "group",
    "marker_position",
    "marker_code",
    "start_sample",
    "end_sample"
]


missing_columns = [

    column

    for column in required_phase_columns

    if column not in phases.columns
]


if missing_columns:

    raise RuntimeError(
        "Missing required Task 3B columns: "
        + ", ".join(
            missing_columns
        )
    )


# ================================================================
# 5. HELPER — FIND PREPROCESSED FIF
# ================================================================

def find_fif(subject):

    subject_dir = (
        TASK3A
        / f"sub-{subject}"
    )


    candidates = list(
        subject_dir.glob(
            "*desc-primary_annotated_eeg_raw.fif"
        )
    )


    if len(candidates) != 1:

        raise RuntimeError(

            f"Expected exactly one primary FIF for "
            f"sub-{subject}; found {len(candidates)}"
        )


    return candidates[0]


# ================================================================
# 6. HELPER — ARTIFACT CHECK
# ================================================================

def epoch_is_clean(segment_uv):

    if segment_uv.size == 0:

        return False


    max_abs = float(
        np.max(
            np.abs(
                segment_uv
            )
        )
    )


    max_ptp = float(
        np.max(
            np.ptp(
                segment_uv,
                axis=1
            )
        )
    )


    return bool(

        (
            max_abs
            <=
            PEAK_THRESHOLD_UV
        )

        and

        (
            max_ptp
            <=
            PTP_THRESHOLD_UV
        )
    )


# ================================================================
# 7. HELPER — BAND INTEGRATION
# ================================================================

def integrate_band(
    freqs,
    psd,
    low,
    high
):

    mask = (

        (freqs >= low)

        &

        (freqs < high)
    )


    if np.sum(
        mask
    ) < 2:

        return np.full(
            psd.shape[:-1],
            np.nan
        )


    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(

            psd[
                ...,
                mask
            ],

            freqs[
                mask
            ],

            axis=-1
        )


    return np.trapz(

        psd[
            ...,
            mask
        ],

        freqs[
            mask
        ],

        axis=-1
    )


# ================================================================
# 8. STORAGE
# ================================================================

spectral_rows = []

epoch_count_rows = []

subject_psd = {}

errors = []


info_template = None

channel_order = None


# ================================================================
# 9. PROCESS ALL PARTICIPANTS
# ================================================================

for subject in SUBJECTS:

    subject_name = (
        f"sub-{subject}"
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"PROCESSING {subject_name}"
    )

    print(
        "=" * 80
    )


    try:

        # --------------------------------------------------------
        # Load EEG
        # --------------------------------------------------------

        fif_file = find_fif(
            subject
        )


        raw = mne.io.read_raw_fif(

            fif_file,

            preload=True,

            verbose=False
        )


        sfreq = float(
            raw.info[
                "sfreq"
            ]
        )


        ch_names = list(
            raw.ch_names
        )


        if channel_order is None:

            channel_order = (
                ch_names.copy()
            )


        if ch_names != channel_order:

            raise RuntimeError(
                f"{subject_name}: channel order differs "
                "from previous participants."
            )


        missing_posterior = [

            channel

            for channel in POSTERIOR_CHANNELS

            if channel not in ch_names
        ]


        if missing_posterior:

            raise RuntimeError(

                f"{subject_name}: missing posterior channels: "
                f"{missing_posterior}"
            )


        if info_template is None:

            info_template = (
                raw.info.copy()
            )


        # --------------------------------------------------------
        # EEG in µV
        # --------------------------------------------------------

        data_uv = (

            raw.get_data()

            * 1e6
        )


        epoch_samples = int(
            round(
                EPOCH_SEC
                * sfreq
            )
        )


        # --------------------------------------------------------
        # Subject metadata
        # --------------------------------------------------------

        subject_phase_rows = phases[
            phases[
                "subject"
            ]
            ==
            subject_name
        ]


        if len(
            subject_phase_rows
        ) == 0:

            raise RuntimeError(
                f"No Task 3B phases found for {subject_name}."
            )


        group = str(
            subject_phase_rows.iloc[
                0
            ][
                "group"
            ]
        )


        # --------------------------------------------------------
        # Process both baseline blocks
        # --------------------------------------------------------

        for (
            baseline_name,
            definition
        ) in BASELINES.items():

            marker_position = int(
                definition[
                    "marker_position"
                ]
            )


            expected_code = str(
                definition[
                    "marker_code"
                ]
            )


            phase = phases[
                (
                    phases[
                        "subject"
                    ]
                    ==
                    subject_name
                )

                &

                (
                    phases[
                        "marker_position"
                    ]
                    ==
                    marker_position
                )
            ]


            if len(
                phase
            ) != 1:

                raise RuntimeError(

                    f"{subject_name}: expected one interval "
                    f"at marker position {marker_position}; "
                    f"found {len(phase)}."
                )


            phase = phase.iloc[
                0
            ]


            observed_code = str(
                phase[
                    "marker_code"
                ]
            )


            if observed_code != expected_code:

                raise RuntimeError(

                    f"{subject_name}: position "
                    f"{marker_position} expected "
                    f"{expected_code}, but observed "
                    f"{observed_code}."
                )


            start_sample = int(
                phase[
                    "start_sample"
                ]
            )


            end_sample = int(
                phase[
                    "end_sample"
                ]
            )


            available_samples = (

                end_sample
                -
                start_sample
            )


            n_epochs = int(

                available_samples
                //
                epoch_samples
            )


            clean_segments = []


            # ----------------------------------------------------
            # Construct phase-contained 2-s epochs
            # ----------------------------------------------------

            for local_epoch in range(
                n_epochs
            ):

                epoch_start = (

                    start_sample

                    +

                    local_epoch
                    * epoch_samples
                )


                epoch_end = (

                    epoch_start

                    +

                    epoch_samples
                )


                segment_uv = data_uv[
                    :,
                    epoch_start:
                    epoch_end
                ]


                if (
                    segment_uv.shape[
                        1
                    ]
                    !=
                    epoch_samples
                ):

                    continue


                if epoch_is_clean(
                    segment_uv
                ):

                    clean_segments.append(
                        segment_uv
                    )


            n_clean = len(
                clean_segments
            )


            n_rejected = (

                n_epochs
                -
                n_clean
            )


            clean_percent = (

                100
                * n_clean
                / n_epochs

                if n_epochs > 0

                else np.nan
            )


            epoch_count_rows.append({

                "subject":
                    subject_name,

                "group":
                    group,

                "baseline":
                    baseline_name,

                "marker_position":
                    marker_position,

                "marker_code":
                    observed_code,

                "available_epochs":
                    n_epochs,

                "clean_epochs":
                    n_clean,

                "rejected_epochs":
                    n_rejected,

                "clean_percent":
                    clean_percent
            })


            if n_clean == 0:

                print(

                    f"{baseline_name}: "
                    f"0/{n_epochs} clean epochs"
                )

                continue


            clean_segments = np.stack(

                clean_segments,

                axis=0
            )


            # Shape:
            #
            # epochs × channels × samples


            # ----------------------------------------------------
            # Welch PSD
            # ----------------------------------------------------

            nperseg = int(
                min(
                    epoch_samples,
                    round(
                        2.0
                        * sfreq
                    )
                )
            )


            freqs, psd = welch(

                clean_segments,

                fs=sfreq,

                window="hann",

                nperseg=nperseg,

                noverlap=nperseg // 2,

                detrend="constant",

                scaling="density",

                axis=-1
            )


            # ----------------------------------------------------
            # Keep 1–45 Hz
            # ----------------------------------------------------

            frequency_mask = (

                (freqs >= PSD_FMIN)

                &

                (freqs <= PSD_FMAX)
            )


            freqs_selected = (
                freqs[
                    frequency_mask
                ]
            )


            psd_selected = (
                psd[
                    ...,
                    frequency_mask
                ]
            )


            # ----------------------------------------------------
            # Subject-level median PSD across clean epochs
            # ----------------------------------------------------

            median_psd = np.median(

                psd_selected,

                axis=0
            )


            subject_psd[
                (
                    subject_name,
                    baseline_name
                )
            ] = {

                "freqs":
                    freqs_selected.copy(),

                "psd":
                    median_psd.copy()
            }


            # ----------------------------------------------------
            # Total 1–45 Hz power
            # ----------------------------------------------------

            total_power = integrate_band(

                freqs_selected,

                psd_selected,

                PSD_FMIN,

                PSD_FMAX
                + 1e-9
            )


            # ----------------------------------------------------
            # Band powers
            # ----------------------------------------------------

            for (
                band_name,
                (
                    low,
                    high
                )
            ) in BANDS.items():

                band_power = integrate_band(

                    freqs_selected,

                    psd_selected,

                    low,

                    high
                )


                relative_power = np.divide(

                    band_power,

                    total_power,

                    out=np.full_like(
                        band_power,
                        np.nan,
                        dtype=float
                    ),

                    where=(
                        total_power
                        >
                        0
                    )
                )


                # ------------------------------------------------
                # Robust subject-level aggregation:
                # median across clean epochs
                # ------------------------------------------------

                absolute_subject = np.nanmedian(

                    band_power,

                    axis=0
                )


                relative_subject = np.nanmedian(

                    relative_power,

                    axis=0
                )


                for (
                    ch_idx,
                    channel
                ) in enumerate(
                    ch_names
                ):

                    spectral_rows.append({

                        "subject":
                            subject_name,

                        "group":
                            group,

                        "baseline":
                            baseline_name,

                        "marker_position":
                            marker_position,

                        "marker_code":
                            observed_code,

                        "band":
                            band_name,

                        "channel":
                            channel,

                        "absolute_power_uV2":
                            float(
                                absolute_subject[
                                    ch_idx
                                ]
                            ),

                        "relative_power":
                            float(
                                relative_subject[
                                    ch_idx
                                ]
                            ),

                        "clean_epochs":
                            n_clean
                    })


            print(

                f"{baseline_name}: "
                f"{n_clean}/{n_epochs} clean epochs "
                f"({clean_percent:.1f}%)"
            )


    except Exception as exc:

        print(
            f"ERROR {subject_name}: {exc}"
        )


        errors.append({

            "subject":
                subject_name,

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 10. CREATE DATAFRAMES
# ================================================================

spectral = pd.DataFrame(
    spectral_rows
)


epoch_counts = pd.DataFrame(
    epoch_count_rows
)


errors_df = pd.DataFrame(
    errors
)


if len(
    spectral
) == 0:

    raise RuntimeError(
        "No spectral results were generated."
    )


# ================================================================
# 11. SAVE BASIC OUTPUTS
# ================================================================

spectral.to_csv(

    OUT
    / "baseline_channel_band_power.csv",

    index=False
)


epoch_counts.to_csv(

    OUT
    / "baseline_epoch_counts.csv",

    index=False
)


errors_df.to_csv(

    OUT
    / "processing_errors.csv",

    index=False
)


# ================================================================
# 12. BASELINE EPOCH AVAILABILITY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "BASELINE EPOCH AVAILABILITY"
)

print(
    "=" * 80
)


epoch_pivot = (

    epoch_counts

    .pivot_table(

        index=[
            "subject",
            "group"
        ],

        columns="baseline",

        values="clean_epochs"
    )

    .reset_index()
)


display(
    epoch_pivot
)


epoch_pivot.to_csv(

    OUT
    / "baseline_clean_epoch_availability.csv",

    index=False
)


# ================================================================
# 13. POSTERIOR ALPHA ROI
# ================================================================

posterior_alpha = spectral[
    (
        spectral[
            "band"
        ]
        ==
        "alpha_8_13"
    )

    &

    (
        spectral[
            "channel"
        ].isin(
            POSTERIOR_CHANNELS
        )
    )
].copy()


posterior_alpha_subject = (

    posterior_alpha

    .groupby(
        [
            "subject",
            "group",
            "baseline"
        ]
    )

    .agg(

        posterior_alpha_absolute=(
            "absolute_power_uV2",
            "mean"
        ),

        posterior_alpha_relative=(
            "relative_power",
            "mean"
        ),

        clean_epochs=(
            "clean_epochs",
            "first"
        )

    )

    .reset_index()
)


posterior_alpha_subject.to_csv(

    OUT
    / "posterior_alpha_by_subject.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "POSTERIOR ALPHA BY PARTICIPANT"
)

print(
    "=" * 80
)


display(
    posterior_alpha_subject.round(
        6
    )
)


# ================================================================
# 14. PAIRED POSTERIOR-ALPHA TABLE
# ================================================================

alpha_pivot = (

    posterior_alpha_subject

    .pivot(

        index="subject",

        columns="baseline",

        values="posterior_alpha_relative"
    )

    .dropna(
        subset=[
            "Baseline_A_T2_V1",
            "Baseline_B_T2_V3"
        ]
    )
)


alpha_pivot[
    "B_minus_A"
] = (

    alpha_pivot[
        "Baseline_B_T2_V3"
    ]

    -

    alpha_pivot[
        "Baseline_A_T2_V1"
    ]
)


alpha_pivot[
    "B_over_A"
] = np.divide(

    alpha_pivot[
        "Baseline_B_T2_V3"
    ],

    alpha_pivot[
        "Baseline_A_T2_V1"
    ],

    out=np.full(
        len(
            alpha_pivot
        ),
        np.nan,
        dtype=float
    ),

    where=(

        alpha_pivot[
            "Baseline_A_T2_V1"
        ].to_numpy()

        !=
        0
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "PAIRED POSTERIOR-ALPHA COMPARISON"
)

print(
    "=" * 80
)


display(
    alpha_pivot.round(
        6
    )
)


alpha_pivot.to_csv(

    OUT
    / "paired_posterior_alpha.csv"
)


# ================================================================
# 15. CORRECTED WILCOXON PAIRED TEST
# ================================================================

A = alpha_pivot[
    "Baseline_A_T2_V1"
].to_numpy(
    dtype=float
)


B = alpha_pivot[
    "Baseline_B_T2_V3"
].to_numpy(
    dtype=float
)


difference = (
    B - A
)


valid = (

    np.isfinite(
        A
    )

    &

    np.isfinite(
        B
    )
)


A = A[
    valid
]


B = B[
    valid
]


difference = (
    B - A
)


if len(
    A
) >= 2:

    try:

        wilcoxon_result = wilcoxon(

            B,

            A,

            alternative="two-sided",

            zero_method="wilcox"
        )


        # Explicitly extract scalar values.
        #
        # This avoids the WilcoxonResult formatting problem
        # encountered in the previous version.

        stat = float(
            np.asarray(
                wilcoxon_result.statistic
            ).squeeze()
        )


        p_value = float(
            np.asarray(
                wilcoxon_result.pvalue
            ).squeeze()
        )


    except Exception as exc:

        print(
            f"Wilcoxon test failed: {exc}"
        )

        stat = np.nan

        p_value = np.nan


else:

    stat = np.nan

    p_value = np.nan


# ================================================================
# 16. PAIRED EFFECT SIZE — COHEN'S dz
# ================================================================

if (

    len(
        difference
    )
    >
    1

    and

    np.std(
        difference,
        ddof=1
    )
    >
    0
):

    cohen_dz = float(

        np.mean(
            difference
        )

        /

        np.std(
            difference,
            ddof=1
        )
    )


else:

    cohen_dz = np.nan


# ================================================================
# 17. DIRECTION COUNTS
# ================================================================

n_B_gt_A = int(
    np.sum(
        B > A
    )
)


n_A_gt_B = int(
    np.sum(
        A > B
    )
)


n_equal = int(
    np.sum(
        A == B
    )
)


# ================================================================
# 18. POSTERIOR-ALPHA STATISTICAL SUMMARY
# ================================================================

alpha_test = pd.DataFrame([{

    "n_subjects":
        int(
            len(
                A
            )
        ),

    "mean_A":
        float(
            np.mean(
                A
            )
        ),

    "mean_B":
        float(
            np.mean(
                B
            )
        ),

    "median_A":
        float(
            np.median(
                A
            )
        ),

    "median_B":
        float(
            np.median(
                B
            )
        ),

    "mean_B_minus_A":
        float(
            np.mean(
                difference
            )
        ),

    "median_B_minus_A":
        float(
            np.median(
                difference
            )
        ),

    "B_greater_A_subjects":
        n_B_gt_A,

    "A_greater_B_subjects":
        n_A_gt_B,

    "equal_subjects":
        n_equal,

    "wilcoxon_statistic":
        stat,

    "wilcoxon_p":
        p_value,

    "cohen_dz":
        cohen_dz
}])


print(
    "\n"
    + "=" * 80
)

print(
    "POSTERIOR-ALPHA STATISTICAL SUMMARY"
)

print(
    "=" * 80
)


display(
    alpha_test.T
)


alpha_test.to_csv(

    OUT
    / "posterior_alpha_statistical_test.csv",

    index=False
)


# ================================================================
# 19. WHOLE-SCALP BAND POWER BY SUBJECT
# ================================================================

subject_band = (

    spectral

    .groupby(
        [
            "subject",
            "group",
            "baseline",
            "band"
        ]
    )

    .agg(

        mean_absolute_power=(
            "absolute_power_uV2",
            "mean"
        ),

        mean_relative_power=(
            "relative_power",
            "mean"
        )

    )

    .reset_index()
)


subject_band.to_csv(

    OUT
    / "whole_scalp_band_power_by_subject.csv",

    index=False
)


band_summary = (

    subject_band

    .groupby(
        [
            "baseline",
            "band"
        ]
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_relative_power=(
            "mean_relative_power",
            "mean"
        ),

        sd_relative_power=(
            "mean_relative_power",
            "std"
        ),

        median_relative_power=(
            "mean_relative_power",
            "median"
        )

    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)

print(
    "WHOLE-SCALP RELATIVE BAND POWER"
)

print(
    "=" * 80
)


display(
    band_summary.round(
        6
    )
)


band_summary.to_csv(

    OUT
    / "whole_scalp_band_summary.csv",

    index=False
)


# ================================================================
# 20. FIGURE — PAIRED POSTERIOR ALPHA
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        7
    )
)


for subject in alpha_pivot.index:

    y1 = float(
        alpha_pivot.loc[
            subject,
            "Baseline_A_T2_V1"
        ]
    )


    y2 = float(
        alpha_pivot.loc[
            subject,
            "Baseline_B_T2_V3"
        ]
    )


    ax.plot(

        [
            0,
            1
        ],

        [
            y1,
            y2
        ],

        marker="o",

        linewidth=1,

        alpha=0.75
    )


ax.set_xticks(
    [
        0,
        1
    ]
)


ax.set_xticklabels(
    [
        "Baseline A\nT2_V1",
        "Baseline B\nT2_V3"
    ]
)


ax.set_ylabel(
    "Posterior relative alpha power"
)


ax.set_title(
    "Within-participant posterior alpha"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    / "01_paired_posterior_alpha.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 21. FIGURE — POSTERIOR GROUP-MEAN PSD
# ================================================================

posterior_indices = [

    channel_order.index(
        channel
    )

    for channel in POSTERIOR_CHANNELS
]


fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


for baseline_name in BASELINES.keys():

    baseline_psds = []

    common_freqs = None


    for subject in SUBJECTS:

        key = (
            f"sub-{subject}",
            baseline_name
        )


        if key not in subject_psd:

            continue


        entry = subject_psd[
            key
        ]


        posterior_psd = np.mean(

            entry[
                "psd"
            ][
                posterior_indices,
                :
            ],

            axis=0
        )


        baseline_psds.append(
            posterior_psd
        )


        common_freqs = entry[
            "freqs"
        ]


    if len(
        baseline_psds
    ) == 0:

        continue


    baseline_psds = np.asarray(
        baseline_psds
    )


    # Mean participant PSD.
    mean_psd = np.mean(

        baseline_psds,

        axis=0
    )


    ax.plot(

        common_freqs,

        10
        *
        np.log10(
            np.maximum(
                mean_psd,
                np.finfo(float).tiny
            )
        ),

        linewidth=2,

        label=baseline_name
    )


ax.axvspan(

    8,
    13,

    alpha=0.1
)


ax.set_xlim(
    1,
    30
)


ax.set_xlabel(
    "Frequency (Hz)"
)


ax.set_ylabel(
    "Posterior PSD (dB, µV²/Hz)"
)


ax.set_title(
    "Posterior spectra during the two baseline blocks"
)


ax.legend()


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    / "02_posterior_psd_baselines.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 22. ALPHA TOPOGRAPHY DATA
# ================================================================

alpha_channel_subject = spectral[
    spectral[
        "band"
    ]
    ==
    "alpha_8_13"
].copy()


topo_values = {}


for baseline_name in BASELINES.keys():

    subset = alpha_channel_subject[
        alpha_channel_subject[
            "baseline"
        ]
        ==
        baseline_name
    ]


    channel_means = (

        subset

        .groupby(
            "channel"
        )[
            "relative_power"
        ]

        .mean()

        .reindex(
            channel_order
        )
    )


    if channel_means.isna().any():

        missing = (
            channel_means[
                channel_means.isna()
            ]
            .index
            .tolist()
        )

        raise RuntimeError(

            f"Missing alpha values for "
            f"{baseline_name}: {missing}"
        )


    topo_values[
        baseline_name
    ] = (
        channel_means.to_numpy(
            dtype=float
        )
    )


# ================================================================
# 23. BASELINE A TOPOGRAPHY
# ================================================================

all_topo_values = np.concatenate(
    list(
        topo_values.values()
    )
)


topo_vmin = float(
    np.nanmin(
        all_topo_values
    )
)


topo_vmax = float(
    np.nanmax(
        all_topo_values
    )
)


for baseline_name, values in topo_values.items():

    fig, ax = plt.subplots(
        figsize=(
            7,
            6
        )
    )


    mne.viz.plot_topomap(

        values,

        info_template,

        axes=ax,

        show=False,

        contours=6,

        vlim=(
            topo_vmin,
            topo_vmax
        )
    )


    ax.set_title(
        f"Relative alpha power\n{baseline_name}"
    )


    plt.tight_layout()


    safe_name = (
        baseline_name
        .replace(
            " ",
            "_"
        )
    )


    plt.savefig(

        OUT
        / f"03_topomap_{safe_name}.png",

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 24. DIFFERENCE TOPOGRAPHY — B MINUS A
# ================================================================

difference_topo = (

    topo_values[
        "Baseline_B_T2_V3"
    ]

    -

    topo_values[
        "Baseline_A_T2_V1"
    ]
)


abs_limit = float(
    np.nanmax(
        np.abs(
            difference_topo
        )
    )
)


if not np.isfinite(
    abs_limit
) or abs_limit == 0:

    abs_limit = 1e-6


fig, ax = plt.subplots(
    figsize=(
        7,
        6
    )
)


mne.viz.plot_topomap(

    difference_topo,

    info_template,

    axes=ax,

    show=False,

    contours=6,

    vlim=(
        -abs_limit,
        abs_limit
    )
)


ax.set_title(
    "Relative alpha difference\nBaseline B − Baseline A"
)


plt.tight_layout()


plt.savefig(

    OUT
    / "04_topomap_alpha_difference_B_minus_A.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 25. CHANNEL-WISE ALPHA DIFFERENCES
# ================================================================

channel_alpha = (

    alpha_channel_subject

    .pivot_table(

        index=[
            "subject",
            "channel"
        ],

        columns="baseline",

        values="relative_power"
    )

    .dropna(
        subset=[
            "Baseline_A_T2_V1",
            "Baseline_B_T2_V3"
        ]
    )

    .reset_index()
)


channel_alpha[
    "B_minus_A"
] = (

    channel_alpha[
        "Baseline_B_T2_V3"
    ]

    -

    channel_alpha[
        "Baseline_A_T2_V1"
    ]
)


channel_alpha_summary = (

    channel_alpha

    .groupby(
        "channel"
    )

    .agg(

        mean_B_minus_A=(
            "B_minus_A",
            "mean"
        ),

        median_B_minus_A=(
            "B_minus_A",
            "median"
        ),

        n_B_greater_A=(
            "B_minus_A",

            lambda x:
                int(
                    np.sum(
                        x > 0
                    )
                )
        ),

        n_A_greater_B=(
            "B_minus_A",

            lambda x:
                int(
                    np.sum(
                        x < 0
                    )
                )
        ),

        n_subjects=(
            "B_minus_A",
            "size"
        )

    )

    .reset_index()

    .sort_values(
        "mean_B_minus_A",
        ascending=False
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "CHANNEL-WISE RELATIVE ALPHA DIFFERENCE"
)

print(
    "=" * 80
)


display(
    channel_alpha_summary.round(
        6
    )
)


channel_alpha_summary.to_csv(

    OUT
    / "channel_alpha_difference_summary.csv",

    index=False
)


# ================================================================
# 26. POSTERIOR CHANNEL-SPECIFIC DIFFERENCES
# ================================================================

posterior_channel_summary = (

    channel_alpha_summary[
        channel_alpha_summary[
            "channel"
        ].isin(
            POSTERIOR_CHANNELS
        )
    ]

    .copy()
)


print(
    "\n"
    + "=" * 80
)

print(
    "POSTERIOR CHANNEL ALPHA DIFFERENCES"
)

print(
    "=" * 80
)


display(
    posterior_channel_summary.round(
        6
    )
)


posterior_channel_summary.to_csv(

    OUT
    / "posterior_channel_alpha_differences.csv",

    index=False
)


# ================================================================
# 27. PHYSIOLOGICAL VALIDATION
# ================================================================

median_difference = float(
    np.median(
        difference
    )
)


mean_difference = float(
    np.mean(
        difference
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "PHYSIOLOGICAL VALIDATION"
)

print(
    "=" * 80
)


print(
    f"Participants analyzed                    : "
    f"{len(A)}/15"
)


print(
    f"Participants with B > A posterior alpha : "
    f"{n_B_gt_A}/{len(A)}"
)


print(
    f"Participants with A > B posterior alpha : "
    f"{n_A_gt_B}/{len(A)}"
)


print(
    f"Participants with B = A                 : "
    f"{n_equal}/{len(A)}"
)


print(
    f"Mean posterior-alpha difference B-A     : "
    f"{mean_difference:.6f}"
)


print(
    f"Median posterior-alpha difference B-A   : "
    f"{median_difference:.6f}"
)


print(
    f"Wilcoxon statistic                      : "
    f"{stat:.3f}"
)


print(
    f"Wilcoxon p-value                        : "
    f"{p_value:.6g}"
)


print(
    f"Paired Cohen's dz                       : "
    f"{cohen_dz:.3f}"
)


# ================================================================
# 28. CONSERVATIVE AUTOMATIC INTERPRETATION
# ================================================================

print(
    "\n"
    + "-" * 80
)


if (
    n_B_gt_A
    >
    n_A_gt_B
):

    stronger_baseline = (
        "Baseline B (T2_V3)"
    )


    print(
        f"{stronger_baseline} shows the stronger "
        "posterior-alpha tendency."
    )


    print(
        "If the difference is spatially concentrated over "
        "posterior electrodes and consistent across participants, "
        "this pattern is physiologically compatible with an "
        "eyes-closed resting EEG state."
    )


elif (
    n_A_gt_B
    >
    n_B_gt_A
):

    stronger_baseline = (
        "Baseline A (T2_V1)"
    )


    print(
        f"{stronger_baseline} shows the stronger "
        "posterior-alpha tendency."
    )


    print(
        "If the difference is spatially concentrated over "
        "posterior electrodes and consistent across participants, "
        "this pattern is physiologically compatible with an "
        "eyes-closed resting EEG state."
    )


else:

    stronger_baseline = (
        "Neither baseline"
    )


    print(
        "Neither baseline shows a clear directional "
        "posterior-alpha majority."
    )


print(
    "\nIMPORTANT:"
)


print(
    "This analysis provides physiological evidence only."
)


print(
    "It does NOT establish the acquisition-system mapping "
    "between T2_V1/T2_V3 and Eyes Open/Eyes Closed."
)


print(
    "The neutral labels Baseline A and Baseline B should "
    "therefore be retained unless independent protocol "
    "documentation establishes the mapping."
)


# ================================================================
# 29. ERROR REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PROCESSING ERRORS"
)

print(
    "=" * 80
)


if len(
    errors_df
) == 0:

    print(
        "None — all participants processed successfully."
    )


else:

    display(
        errors_df
    )


# ================================================================
# 30. FINAL REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "TASK 3C COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participants analyzed : "
    f"{len(A)}/15"
)


print(
    "Baseline A           : "
    "marker position 1 / T2_V1"
)


print(
    "Baseline B           : "
    "marker position 3 / T2_V3"
)


print(
    f"Epoch duration       : "
    f"{EPOCH_SEC:.1f} s"
)


print(
    "Posterior ROI        : "
    + ", ".join(
        POSTERIOR_CHANNELS
    )
)


print(
    "Primary measure      : "
    "relative alpha power (8–13 Hz)"
)


print(
    "Statistical unit     : participant"
)


print(
    "Paired test          : "
    "two-sided Wilcoxon signed-rank"
)


print(
    "ICA removal          : none"
)


print(
    "Interpolation        : none"
)


print(
    "\nPrimary question:"
)


print(
    "Does either standardized baseline show a reproducible "
    "posterior-alpha enhancement compatible with the expected "
    "eyes-closed resting EEG signature?"
)


print(
    "\nNo Eyes Open / Eyes Closed label has been assigned."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)

In [ ]:
# ================================================================
# TASK 3D — MATCHED BASELINE → TASK ANALYTICAL EPOCH CONSTRUCTION
# COMPLETE CORRECTED SELF-CONTAINED VERSION
# ================================================================
#
# PURPOSE
# -------
# Construct the final matched epoch dataset for subsequent
# Active-vs-Passive EEG analyses.
#
# STRUCTURAL CONDITIONS
# ---------------------
# Baseline:
#   marker position 1
#   expected code = T2_V1
#
# Task onset:
#   marker position 5
#   Active  -> expected code = T1_V1
#   Passive -> expected code = T1_V4
#
# IMPORTANT
# ---------
# We do NOT assign undocumented semantic meanings to T1_V1/T1_V4.
#
# T2_V1 remains "Baseline A" in saved data.
# Task 3C showed that T2_V3 has a very strong posterior-alpha
# enhancement relative to T2_V1, making T2_V1 physiologically
# consistent with the eyes-open-like baseline.
#
# This physiological interpretation does NOT replace the neutral
# marker labels in the analytical files.
#
# MATCHING STRATEGY
# -----------------
# 1. Use at most the first 30 s of each condition.
# 2. Divide each condition into non-overlapping 2-s epochs.
# 3. Apply frozen artifact thresholds:
#
#       absolute amplitude <= 250 µV
#       peak-to-peak       <= 500 µV
#
# 4. Within each participant:
#
#       n_matched = min(
#           clean baseline epochs,
#           clean task epochs,
#           15
#       )
#
# 5. Select the earliest n_matched clean epochs from each condition.
#
# This prevents:
#   - longer Active recordings from dominating;
#   - unequal baseline/task counts within a participant;
#   - epochs crossing marker-defined phase boundaries.
#
# OUTPUTS
# -------
# - matched Epochs FIF for each participant
# - participant-level epoch metadata CSV
# - combined metadata CSV
# - subject-level matching summary
# - group-level matching summary
# - matching integrity checks
# - QC figures
# - exclusion report
# - error report
#
# No scientific spectral hypothesis test is performed here.
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne


warnings.filterwarnings("ignore")


# Force notebook figures to PNG rather than SVG.
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. PATHS
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)


TASK3A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03a_batch_preprocessing_qc"
)


TASK3B = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03b_phase_reconstruction"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03d_matched_baseline_task_epochs"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. FROZEN ANALYSIS SETTINGS
# ================================================================

EPOCH_SEC = 2.0

MAX_CONDITION_SEC = 30.0


MAX_EPOCHS_PER_CONDITION = int(
    MAX_CONDITION_SEC
    /
    EPOCH_SEC
)


PEAK_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


# ------------------------------------------------
# Structural marker definitions
# ------------------------------------------------

BASELINE_POSITION = 1

BASELINE_CODE = "T2_V1"


TASK_POSITION = 5


EXPECTED_TASK_CODE = {

    "Active":
        "T1_V1",

    "Passive":
        "T1_V4"
}


SUBJECTS = [

    f"{number:03d}"

    for number in range(
        1,
        16
    )
]


# ================================================================
# 4. PRINT CONFIGURATION
# ================================================================

print(
    "=" * 80
)

print(
    "TASK 3D — MATCHED BASELINE → TASK ANALYTICAL EPOCHS"
)

print(
    "=" * 80
)


print(
    f"Root directory              : {ROOT}"
)


print(
    f"Task 3A directory           : {TASK3A}"
)


print(
    f"Task 3B directory           : {TASK3B}"
)


print(
    f"Output directory            : {OUT}"
)


print(
    f"Epoch duration              : {EPOCH_SEC:.1f} s"
)


print(
    f"Maximum condition duration  : {MAX_CONDITION_SEC:.1f} s"
)


print(
    f"Maximum epochs/condition    : {MAX_EPOCHS_PER_CONDITION}"
)


print(
    f"Peak threshold              : {PEAK_THRESHOLD_UV:.1f} µV"
)


print(
    f"Peak-to-peak threshold      : {PTP_THRESHOLD_UV:.1f} µV"
)


print(
    f"Baseline                    : position {BASELINE_POSITION} / "
    f"{BASELINE_CODE}"
)


print(
    f"Active task onset           : position {TASK_POSITION} / "
    f"{EXPECTED_TASK_CODE['Active']}"
)


print(
    f"Passive task onset          : position {TASK_POSITION} / "
    f"{EXPECTED_TASK_CODE['Passive']}"
)


# ================================================================
# 5. LOAD TASK 3B PHASE TABLE
# ================================================================

PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)


if not PHASE_FILE.exists():

    raise FileNotFoundError(
        f"\nTask 3B phase table not found:\n{PHASE_FILE}"
    )


phases = pd.read_csv(
    PHASE_FILE
)


required_columns = [

    "subject",
    "group",
    "marker_position",
    "marker_code",
    "start_sample",
    "end_sample"
]


missing_columns = [

    column

    for column in required_columns

    if column not in phases.columns
]


if missing_columns:

    raise RuntimeError(

        "Missing required columns in Task 3B table: "

        + ", ".join(
            missing_columns
        )
    )


print(
    f"\nLoaded Task 3B phase rows    : {len(phases)}"
)


print(
    f"Subjects represented        : "
    f"{phases['subject'].nunique()}"
)


# ================================================================
# 6. HELPER — FIND PRIMARY PREPROCESSED FIF
# ================================================================

def find_primary_fif(subject):

    subject_dir = (
        TASK3A
        / f"sub-{subject}"
    )


    if not subject_dir.exists():

        raise FileNotFoundError(
            f"Subject directory not found: {subject_dir}"
        )


    candidates = list(
        subject_dir.glob(
            "*desc-primary_annotated_eeg_raw.fif"
        )
    )


    if len(candidates) != 1:

        raise RuntimeError(

            f"sub-{subject}: expected exactly one primary FIF; "
            f"found {len(candidates)}"
        )


    return candidates[0]


# ================================================================
# 7. HELPER — ARTIFACT CHECK
# ================================================================

def evaluate_epoch(segment_uv):

    if segment_uv.size == 0:

        return (
            False,
            np.nan,
            np.nan
        )


    max_abs_uv = float(
        np.max(
            np.abs(
                segment_uv
            )
        )
    )


    max_ptp_uv = float(
        np.max(
            np.ptp(
                segment_uv,
                axis=1
            )
        )
    )


    clean = bool(

        (
            max_abs_uv
            <=
            PEAK_THRESHOLD_UV
        )

        and

        (
            max_ptp_uv
            <=
            PTP_THRESHOLD_UV
        )
    )


    return (
        clean,
        max_abs_uv,
        max_ptp_uv
    )


# ================================================================
# 8. HELPER — GET EXACT MARKER INTERVAL
# ================================================================

def get_phase_row(
    subject_name,
    marker_position,
    expected_code
):

    subset = phases[
        (
            phases[
                "subject"
            ]
            ==
            subject_name
        )

        &

        (
            phases[
                "marker_position"
            ]
            ==
            marker_position
        )
    ]


    if len(subset) != 1:

        raise RuntimeError(

            f"{subject_name}: expected exactly one phase "
            f"at marker position {marker_position}; "
            f"found {len(subset)}."
        )


    row = subset.iloc[0]


    observed_code = str(
        row[
            "marker_code"
        ]
    )


    if observed_code != expected_code:

        raise RuntimeError(

            f"{subject_name}: marker position "
            f"{marker_position} expected {expected_code}, "
            f"but observed {observed_code}."
        )


    return row


# ================================================================
# 9. STORAGE
# ================================================================

all_epoch_metadata = []

subject_summary_rows = []

exclusion_rows = []

error_rows = []


reference_channel_order = None

reference_sfreq = None


# ================================================================
# 10. PROCESS ALL PARTICIPANTS
# ================================================================

for subject in SUBJECTS:

    subject_name = (
        f"sub-{subject}"
    )


    print(
        "\n"
        + "=" * 80
    )

    print(
        f"PROCESSING {subject_name}"
    )

    print(
        "=" * 80
    )


    try:

        # ========================================================
        # 10A. DETERMINE GROUP
        # ========================================================

        subject_rows = phases[
            phases[
                "subject"
            ]
            ==
            subject_name
        ]


        if len(subject_rows) == 0:

            raise RuntimeError(
                "No Task 3B marker rows found."
            )


        unique_groups = (

            subject_rows[
                "group"
            ]

            .dropna()

            .astype(str)

            .unique()
        )


        if len(unique_groups) != 1:

            raise RuntimeError(

                f"Expected exactly one group; found "
                f"{unique_groups.tolist()}."
            )


        group = str(
            unique_groups[0]
        )


        if group not in EXPECTED_TASK_CODE:

            raise RuntimeError(
                f"Unexpected group: {group}"
            )


        task_code = (
            EXPECTED_TASK_CODE[
                group
            ]
        )


        print(
            f"Group                     : {group}"
        )


        print(
            f"Expected task marker      : {task_code}"
        )


        # ========================================================
        # 10B. GET BASELINE AND TASK PHASES
        # ========================================================

        baseline_phase = get_phase_row(

            subject_name,

            BASELINE_POSITION,

            BASELINE_CODE
        )


        task_phase = get_phase_row(

            subject_name,

            TASK_POSITION,

            task_code
        )


        # ========================================================
        # 10C. LOAD PRIMARY EEG
        # ========================================================

        fif_file = find_primary_fif(
            subject
        )


        raw = mne.io.read_raw_fif(

            fif_file,

            preload=True,

            verbose=False
        )


        sfreq = float(
            raw.info[
                "sfreq"
            ]
        )


        ch_names = list(
            raw.ch_names
        )


        # --------------------------------------------------------
        # Verify consistent channel ordering
        # --------------------------------------------------------

        if reference_channel_order is None:

            reference_channel_order = (
                ch_names.copy()
            )


        elif ch_names != reference_channel_order:

            raise RuntimeError(
                "Channel order differs from reference participant."
            )


        # --------------------------------------------------------
        # Verify consistent sampling frequency
        # --------------------------------------------------------

        if reference_sfreq is None:

            reference_sfreq = sfreq


        elif not np.isclose(
            sfreq,
            reference_sfreq
        ):

            raise RuntimeError(

                f"Sampling frequency differs: "
                f"{sfreq} vs {reference_sfreq}"
            )


        # --------------------------------------------------------
        # Data
        # --------------------------------------------------------

        data_v = raw.get_data()


        data_uv = (
            data_v
            *
            1e6
        )


        epoch_samples = int(
            round(
                EPOCH_SEC
                *
                sfreq
            )
        )


        max_condition_samples = int(
            round(
                MAX_CONDITION_SEC
                *
                sfreq
            )
        )


        print(
            f"Sampling frequency        : {sfreq:.1f} Hz"
        )


        print(
            f"EEG channels              : {len(ch_names)}"
        )


        # ========================================================
        # 10D. DEFINE CONDITIONS
        # ========================================================

        condition_definitions = {

            "baseline": {

                "phase":
                    baseline_phase,

                "marker_position":
                    BASELINE_POSITION,

                "marker_code":
                    BASELINE_CODE
            },

            "task": {

                "phase":
                    task_phase,

                "marker_position":
                    TASK_POSITION,

                "marker_code":
                    task_code
            }
        }


        condition_results = {}


        # ========================================================
        # 10E. EXTRACT CLEAN EPOCHS
        # ========================================================

        for (
            condition,
            definition
        ) in condition_definitions.items():

            phase_row = (
                definition[
                    "phase"
                ]
            )


            phase_start = int(
                phase_row[
                    "start_sample"
                ]
            )


            phase_end = int(
                phase_row[
                    "end_sample"
                ]
            )


            if phase_end <= phase_start:

                raise RuntimeError(

                    f"{subject_name}: invalid "
                    f"{condition} phase interval."
                )


            phase_samples = (
                phase_end
                -
                phase_start
            )


            phase_duration_sec = (

                phase_samples
                /
                sfreq
            )


            # ----------------------------------------------------
            # Restrict analysis to first 30 seconds maximum
            # ----------------------------------------------------

            analysis_end = min(

                phase_end,

                phase_start
                +
                max_condition_samples
            )


            analysis_samples = (

                analysis_end
                -
                phase_start
            )


            candidate_epochs = int(

                analysis_samples
                //
                epoch_samples
            )


            candidate_epochs = min(

                candidate_epochs,

                MAX_EPOCHS_PER_CONDITION
            )


            clean_epoch_records = []

            rejected_epoch_records = []


            # ----------------------------------------------------
            # Construct non-overlapping 2-s epochs
            # ----------------------------------------------------

            for local_epoch in range(
                candidate_epochs
            ):

                epoch_start = (

                    phase_start

                    +

                    local_epoch
                    *
                    epoch_samples
                )


                epoch_end = (

                    epoch_start

                    +
                    epoch_samples
                )


                # Must remain within marker-defined phase.
                if epoch_end > phase_end:

                    break


                segment_uv = data_uv[
                    :,
                    epoch_start:
                    epoch_end
                ]


                segment_v = data_v[
                    :,
                    epoch_start:
                    epoch_end
                ]


                if (
                    segment_uv.shape[1]
                    !=
                    epoch_samples
                ):

                    continue


                (
                    is_clean,
                    max_abs_uv,
                    max_ptp_uv
                ) = evaluate_epoch(
                    segment_uv
                )


                epoch_record = {

                    "local_epoch":
                        int(
                            local_epoch
                        ),

                    "start_sample":
                        int(
                            epoch_start
                        ),

                    "end_sample":
                        int(
                            epoch_end
                        ),

                    "start_sec":
                        float(
                            epoch_start
                            /
                            sfreq
                        ),

                    "end_sec":
                        float(
                            epoch_end
                            /
                            sfreq
                        ),

                    "relative_start_sec":
                        float(
                            local_epoch
                            *
                            EPOCH_SEC
                        ),

                    "relative_end_sec":
                        float(
                            (
                                local_epoch
                                +
                                1
                            )
                            *
                            EPOCH_SEC
                        ),

                    "max_abs_uv":
                        max_abs_uv,

                    "max_ptp_uv":
                        max_ptp_uv,

                    "data_v":
                        segment_v.copy()
                }


                if is_clean:

                    clean_epoch_records.append(
                        epoch_record
                    )

                else:

                    rejected_epoch_records.append(
                        epoch_record
                    )


            condition_results[
                condition
            ] = {

                "phase_duration_sec":
                    float(
                        phase_duration_sec
                    ),

                "candidate_epochs":
                    int(
                        candidate_epochs
                    ),

                "clean_records":
                    clean_epoch_records,

                "rejected_records":
                    rejected_epoch_records
            }


        # ========================================================
        # 10F. MATCH BASELINE AND TASK COUNTS
        # ========================================================

        n_baseline_clean = len(

            condition_results[
                "baseline"
            ][
                "clean_records"
            ]
        )


        n_task_clean = len(

            condition_results[
                "task"
            ][
                "clean_records"
            ]
        )


        n_matched = min(

            n_baseline_clean,

            n_task_clean,

            MAX_EPOCHS_PER_CONDITION
        )


        print(
            f"Baseline phase duration   : "
            f"{condition_results['baseline']['phase_duration_sec']:.3f} s"
        )


        print(
            f"Task phase duration       : "
            f"{condition_results['task']['phase_duration_sec']:.3f} s"
        )


        print(
            f"Baseline candidate epochs : "
            f"{condition_results['baseline']['candidate_epochs']}"
        )


        print(
            f"Task candidate epochs     : "
            f"{condition_results['task']['candidate_epochs']}"
        )


        print(
            f"Baseline clean epochs     : "
            f"{n_baseline_clean}/"
            f"{condition_results['baseline']['candidate_epochs']}"
        )


        print(
            f"Task clean epochs         : "
            f"{n_task_clean}/"
            f"{condition_results['task']['candidate_epochs']}"
        )


        print(
            f"Matched epochs/condition  : {n_matched}"
        )


        # ========================================================
        # 10G. EXCLUSION SAFEGUARD
        # ========================================================

        if n_matched < 1:

            exclusion_rows.append({

                "subject":
                    subject_name,

                "group":
                    group,

                "reason":
                    "No matched clean baseline-task epochs",

                "baseline_clean":
                    n_baseline_clean,

                "task_clean":
                    n_task_clean
            })


            print(
                "EXCLUDED: no matched clean epochs."
            )


            continue


        # ========================================================
        # 10H. SELECT EARLIEST CLEAN MATCHED EPOCHS
        # ========================================================

        selected_baseline = (

            condition_results[
                "baseline"
            ][
                "clean_records"
            ][
                :n_matched
            ]
        )


        selected_task = (

            condition_results[
                "task"
            ][
                "clean_records"
            ][
                :n_matched
            ]
        )


        # ========================================================
        # 10I. ASSEMBLE ARRAYS
        # ========================================================

        baseline_array = np.stack(

            [

                record[
                    "data_v"
                ]

                for record in selected_baseline
            ],

            axis=0
        )


        task_array = np.stack(

            [

                record[
                    "data_v"
                ]

                for record in selected_task
            ],

            axis=0
        )


        # Expected:
        #
        # baseline_array:
        #   epochs × channels × samples
        #
        # task_array:
        #   epochs × channels × samples


        if (
            baseline_array.shape
            !=
            task_array.shape
        ):

            raise RuntimeError(

                f"Matched arrays differ in shape: "
                f"{baseline_array.shape} vs "
                f"{task_array.shape}"
            )


        combined_array = np.concatenate(

            [
                baseline_array,
                task_array
            ],

            axis=0
        )


        # ========================================================
        # 10J. STRUCTURAL EVENT CODES
        # ========================================================

        event_id = {

            "baseline":
                1,

            "task":
                2
        }


        event_codes = np.concatenate(

            [

                np.full(

                    n_matched,

                    event_id[
                        "baseline"
                    ],

                    dtype=int
                ),

                np.full(

                    n_matched,

                    event_id[
                        "task"
                    ],

                    dtype=int
                )
            ]
        )


        # --------------------------------------------------------
        # Synthetic sequential events are used only because
        # EpochsArray requires an event matrix.
        #
        # Original acquisition sample locations are preserved
        # in the metadata.
        # --------------------------------------------------------

        synthetic_events = np.column_stack(

            [

                np.arange(
                    len(
                        event_codes
                    )
                )
                *
                epoch_samples,

                np.zeros(
                    len(
                        event_codes
                    ),
                    dtype=int
                ),

                event_codes
            ]
        )


        # ========================================================
        # 10K. BUILD METADATA
        # ========================================================

        participant_metadata_rows = []


        condition_metadata_definitions = [

            (
                "baseline",
                selected_baseline,
                BASELINE_POSITION,
                BASELINE_CODE
            ),

            (
                "task",
                selected_task,
                TASK_POSITION,
                task_code
            )
        ]


        for (
            condition,
            selected_records,
            marker_position,
            marker_code
        ) in condition_metadata_definitions:

            for (
                matched_index,
                record
            ) in enumerate(
                selected_records,
                start=1
            ):

                row = {

                    "subject":
                        subject_name,

                    "group":
                        group,

                    "condition":
                        condition,

                    "marker_position":
                        int(
                            marker_position
                        ),

                    "marker_code":
                        marker_code,

                    "matched_epoch_index":
                        int(
                            matched_index
                        ),

                    "original_local_epoch":
                        int(
                            record[
                                "local_epoch"
                            ]
                        ),

                    "original_start_sample":
                        int(
                            record[
                                "start_sample"
                            ]
                        ),

                    "original_end_sample":
                        int(
                            record[
                                "end_sample"
                            ]
                        ),

                    "original_start_sec":
                        float(
                            record[
                                "start_sec"
                            ]
                        ),

                    "original_end_sec":
                        float(
                            record[
                                "end_sec"
                            ]
                        ),

                    "relative_phase_start_sec":
                        float(
                            record[
                                "relative_start_sec"
                            ]
                        ),

                    "relative_phase_end_sec":
                        float(
                            record[
                                "relative_end_sec"
                            ]
                        ),

                    "max_abs_uv":
                        float(
                            record[
                                "max_abs_uv"
                            ]
                        ),

                    "max_ptp_uv":
                        float(
                            record[
                                "max_ptp_uv"
                            ]
                        ),

                    "n_matched_per_condition":
                        int(
                            n_matched
                        )
                }


                participant_metadata_rows.append(
                    row
                )


                all_epoch_metadata.append(
                    row.copy()
                )


        participant_metadata = pd.DataFrame(
            participant_metadata_rows
        )


        # ========================================================
        # 10L. CREATE MNE EPOCHSARRAY
        # ========================================================

        epochs = mne.EpochsArray(

            combined_array,

            info=raw.info.copy(),

            events=synthetic_events,

            event_id=event_id,

            tmin=0.0,

            metadata=participant_metadata,

            verbose=False
        )


        # ========================================================
        # 10M. VERIFY CREATED EPOCHS
        # ========================================================

        if len(
            epochs[
                "baseline"
            ]
        ) != n_matched:

            raise RuntimeError(
                "Saved baseline epoch count mismatch."
            )


        if len(
            epochs[
                "task"
            ]
        ) != n_matched:

            raise RuntimeError(
                "Saved task epoch count mismatch."
            )


        # ========================================================
        # 10N. CREATE PARTICIPANT OUTPUT DIRECTORY
        # ========================================================

        subject_out = (
            OUT
            /
            subject_name
        )


        subject_out.mkdir(
            parents=True,
            exist_ok=True
        )


        # ========================================================
        # 10O. SAVE MATCHED EPOCHS
        # ========================================================

        epochs_file = (

            subject_out

            /
            (
                f"{subject_name}_group-{group.lower()}"
                "_desc-matched_baseline_task_epo.fif"
            )
        )


        epochs.save(

            epochs_file,

            overwrite=True,

            verbose=False
        )


        # ========================================================
        # 10P. SAVE PARTICIPANT METADATA
        # ========================================================

        metadata_file = (

            subject_out

            /
            (
                f"{subject_name}"
                "_desc-matched_epoch_metadata.csv"
            )
        )


        participant_metadata.to_csv(

            metadata_file,

            index=False
        )


        # ========================================================
        # 10Q. CALCULATE SELECTED TEMPORAL COVERAGE
        # ========================================================

        baseline_last_selected_time = float(

            selected_baseline[
                -1
            ][
                "relative_end_sec"
            ]
        )


        task_last_selected_time = float(

            selected_task[
                -1
            ][
                "relative_end_sec"
            ]
        )


        baseline_first_selected_time = float(

            selected_baseline[
                0
            ][
                "relative_start_sec"
            ]
        )


        task_first_selected_time = float(

            selected_task[
                0
            ][
                "relative_start_sec"
            ]
        )


        # ========================================================
        # 10R. PARTICIPANT SUMMARY
        # ========================================================

        subject_summary_rows.append({

            "subject":
                subject_name,

            "group":
                group,

            "baseline_marker":
                BASELINE_CODE,

            "task_marker":
                task_code,

            "baseline_phase_duration_sec":
                float(
                    condition_results[
                        "baseline"
                    ][
                        "phase_duration_sec"
                    ]
                ),

            "task_phase_duration_sec":
                float(
                    condition_results[
                        "task"
                    ][
                        "phase_duration_sec"
                    ]
                ),

            "baseline_candidate_epochs":
                int(
                    condition_results[
                        "baseline"
                    ][
                        "candidate_epochs"
                    ]
                ),

            "task_candidate_epochs":
                int(
                    condition_results[
                        "task"
                    ][
                        "candidate_epochs"
                    ]
                ),

            "baseline_clean_epochs":
                int(
                    n_baseline_clean
                ),

            "task_clean_epochs":
                int(
                    n_task_clean
                ),

            "baseline_rejected_epochs":
                int(
                    condition_results[
                        "baseline"
                    ][
                        "candidate_epochs"
                    ]
                    -
                    n_baseline_clean
                ),

            "task_rejected_epochs":
                int(
                    condition_results[
                        "task"
                    ][
                        "candidate_epochs"
                    ]
                    -
                    n_task_clean
                ),

            "matched_epochs_per_condition":
                int(
                    n_matched
                ),

            "matched_total_epochs":
                int(
                    2
                    *
                    n_matched
                ),

            "matched_seconds_per_condition":
                float(
                    n_matched
                    *
                    EPOCH_SEC
                ),

            "baseline_first_selected_time_sec":
                baseline_first_selected_time,

            "baseline_last_selected_time_sec":
                baseline_last_selected_time,

            "task_first_selected_time_sec":
                task_first_selected_time,

            "task_last_selected_time_sec":
                task_last_selected_time,

            "epochs_file":
                str(
                    epochs_file
                )
        })


        print(
            f"Matched duration/condition: "
            f"{n_matched * EPOCH_SEC:.1f} s"
        )


        print(
            f"Saved total epochs        : "
            f"{2 * n_matched}"
        )


        print(
            "STATUS                    : SUCCESS"
        )


    except Exception as exc:

        print(
            f"ERROR                     : {exc}"
        )


        error_rows.append({

            "subject":
                subject_name,

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 11. CREATE COMBINED TABLES
# ================================================================

subject_summary = pd.DataFrame(
    subject_summary_rows
)


combined_metadata = pd.DataFrame(
    all_epoch_metadata
)


exclusions = pd.DataFrame(
    exclusion_rows
)


errors = pd.DataFrame(
    error_rows
)


# ================================================================
# 12. SAFETY CHECK — DID ANY SUBJECT PROCESS?
# ================================================================

if len(
    subject_summary
) == 0:

    raise RuntimeError(

        "No participants were successfully processed. "
        "Inspect the processing error table above."
    )


# ================================================================
# 13. SAVE COMBINED OUTPUTS
# ================================================================

subject_summary.to_csv(

    OUT
    /
    "matched_subject_summary.csv",

    index=False
)


combined_metadata.to_csv(

    OUT
    /
    "matched_epoch_metadata_all_subjects.csv",

    index=False
)


exclusions.to_csv(

    OUT
    /
    "matched_epoch_exclusions.csv",

    index=False
)


errors.to_csv(

    OUT
    /
    "processing_errors.csv",

    index=False
)


# ================================================================
# 14. MATCHED SUBJECT SUMMARY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "MATCHED SUBJECT SUMMARY"
)

print(
    "=" * 80
)


subject_display = subject_summary[
    [
        "subject",
        "group",
        "baseline_marker",
        "task_marker",
        "baseline_candidate_epochs",
        "task_candidate_epochs",
        "baseline_clean_epochs",
        "task_clean_epochs",
        "matched_epochs_per_condition",
        "matched_seconds_per_condition"
    ]
].copy()


display(
    subject_display
)


# ================================================================
# 15. GROUP MATCHING SUMMARY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "GROUP MATCHING SUMMARY"
)

print(
    "=" * 80
)


group_summary = (

    subject_summary

    .groupby(
        "group"
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_baseline_clean_epochs=(
            "baseline_clean_epochs",
            "mean"
        ),

        sd_baseline_clean_epochs=(
            "baseline_clean_epochs",
            "std"
        ),

        mean_task_clean_epochs=(
            "task_clean_epochs",
            "mean"
        ),

        sd_task_clean_epochs=(
            "task_clean_epochs",
            "std"
        ),

        mean_matched_epochs=(
            "matched_epochs_per_condition",
            "mean"
        ),

        sd_matched_epochs=(
            "matched_epochs_per_condition",
            "std"
        ),

        median_matched_epochs=(
            "matched_epochs_per_condition",
            "median"
        ),

        min_matched_epochs=(
            "matched_epochs_per_condition",
            "min"
        ),

        max_matched_epochs=(
            "matched_epochs_per_condition",
            "max"
        ),

        mean_matched_seconds=(
            "matched_seconds_per_condition",
            "mean"
        ),

        median_matched_seconds=(
            "matched_seconds_per_condition",
            "median"
        )

    )

    .reset_index()
)


display(
    group_summary.round(
        3
    )
)


group_summary.to_csv(

    OUT
    /
    "matched_group_summary.csv",

    index=False
)


# ================================================================
# 16. WITHIN-SUBJECT MATCHING CHECK
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "WITHIN-SUBJECT MATCHING CHECK"
)

print(
    "=" * 80
)


matching_check_rows = []


for subject_name in sorted(
    combined_metadata[
        "subject"
    ].unique()
):

    temp = combined_metadata[
        combined_metadata[
            "subject"
        ]
        ==
        subject_name
    ]


    baseline_n = int(
        np.sum(
            temp[
                "condition"
            ]
            ==
            "baseline"
        )
    )


    task_n = int(
        np.sum(
            temp[
                "condition"
            ]
            ==
            "task"
        )
    )


    matching_check_rows.append({

        "subject":
            subject_name,

        "baseline_epochs":
            baseline_n,

        "task_epochs":
            task_n,

        "exact_match":
            bool(
                baseline_n
                ==
                task_n
            )
    })


matching_check = pd.DataFrame(
    matching_check_rows
)


display(
    matching_check
)


matching_check.to_csv(

    OUT
    /
    "within_subject_matching_check.csv",

    index=False
)


# ================================================================
# 17. MARKER STRUCTURE CHECK
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "MARKER STRUCTURE CHECK"
)

print(
    "=" * 80
)


marker_check = (

    subject_summary[
        [
            "subject",
            "group",
            "baseline_marker",
            "task_marker"
        ]
    ]

    .copy()
)


marker_check[
    "expected_baseline"
] = (

    marker_check[
        "baseline_marker"
    ]
    ==
    BASELINE_CODE
)


marker_check[
    "expected_task"
] = marker_check.apply(

    lambda row:

        row[
            "task_marker"
        ]

        ==

        EXPECTED_TASK_CODE[
            row[
                "group"
            ]
        ],

    axis=1
)


display(
    marker_check
)


marker_check.to_csv(

    OUT
    /
    "marker_structure_check.csv",

    index=False
)


# ================================================================
# 18. GLOBAL MATCHING CHECKS
# ================================================================

all_exact = bool(

    matching_check[
        "exact_match"
    ].all()
)


all_marker_valid = bool(

    marker_check[
        "expected_baseline"
    ].all()

    and

    marker_check[
        "expected_task"
    ].all()
)


print(
    f"\nAll subjects exactly matched : {all_exact}"
)


print(
    f"All marker structures valid  : {all_marker_valid}"
)


# ================================================================
# 19. CHECK EPOCH METADATA COUNTS
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "EPOCH METADATA INTEGRITY CHECK"
)

print(
    "=" * 80
)


metadata_count_check = (

    combined_metadata

    .groupby(
        [
            "subject",
            "condition"
        ]
    )

    .size()

    .unstack(
        fill_value=0
    )

    .reset_index()
)


if "baseline" not in metadata_count_check.columns:

    metadata_count_check[
        "baseline"
    ] = 0


if "task" not in metadata_count_check.columns:

    metadata_count_check[
        "task"
    ] = 0


metadata_count_check[
    "difference"
] = (

    metadata_count_check[
        "task"
    ]

    -

    metadata_count_check[
        "baseline"
    ]
)


metadata_count_check[
    "valid"
] = (

    metadata_count_check[
        "difference"
    ]
    ==
    0
)


display(
    metadata_count_check
)


metadata_count_check.to_csv(

    OUT
    /
    "epoch_metadata_integrity_check.csv",

    index=False
)


# ================================================================
# 20. FIGURE 1 — MATCHED EPOCH COUNTS BY SUBJECT
# ================================================================

plot_data = (
    subject_summary.copy()
)


x = np.arange(
    len(
        plot_data
    )
)


fig, ax = plt.subplots(
    figsize=(
        13,
        6
    )
)


ax.bar(

    x,

    plot_data[
        "matched_epochs_per_condition"
    ].to_numpy()
)


ax.set_xticks(
    x
)


ax.set_xticklabels(

    plot_data[
        "subject"
    ],

    rotation=45,

    ha="right"
)


ax.axhline(

    MAX_EPOCHS_PER_CONDITION,

    linestyle="--",

    linewidth=1
)


ax.set_ylabel(
    "Matched clean epochs per condition"
)


ax.set_xlabel(
    "Participant"
)


ax.set_title(
    "Matched baseline-task epoch availability"
)


ax.set_ylim(
    0,
    MAX_EPOCHS_PER_CONDITION
    +
    1
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "01_matched_epochs_by_subject.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 21. FIGURE 2 — MATCHED SECONDS BY GROUP
# CORRECTED FOR CURRENT MATPLOTLIB
# ================================================================

groups = list(

    subject_summary[
        "group"
    ].unique()
)


group_values = [

    subject_summary.loc[
        subject_summary[
            "group"
        ]
        ==
        group,
        "matched_seconds_per_condition"
    ].to_numpy()

    for group in groups
]


fig, ax = plt.subplots(
    figsize=(
        8,
        6
    )
)


# ------------------------------------------------
# Current Matplotlib:
#     tick_labels=
#
# Compatibility fallback:
#     labels=
# ------------------------------------------------

try:

    ax.boxplot(

        group_values,

        tick_labels=groups,

        showmeans=True
    )


except TypeError:

    ax.boxplot(

        group_values,

        labels=groups,

        showmeans=True
    )


ax.set_ylabel(
    "Matched seconds per condition"
)


ax.set_xlabel(
    "Group"
)


ax.set_title(
    "Matched analytical duration by group"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "02_matched_duration_by_group.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 22. FIGURE 3 — CLEAN BASELINE VS TASK AVAILABILITY
# ================================================================

x = np.arange(
    len(
        subject_summary
    )
)


width = 0.38


fig, ax = plt.subplots(
    figsize=(
        13,
        6
    )
)


ax.bar(

    x
    -
    width / 2,

    subject_summary[
        "baseline_clean_epochs"
    ].to_numpy(),

    width,

    label="Baseline"
)


ax.bar(

    x
    +
    width / 2,

    subject_summary[
        "task_clean_epochs"
    ].to_numpy(),

    width,

    label="Task"
)


ax.set_xticks(
    x
)


ax.set_xticklabels(

    subject_summary[
        "subject"
    ],

    rotation=45,

    ha="right"
)


ax.set_ylabel(
    "Clean epochs in first 30 s"
)


ax.set_xlabel(
    "Participant"
)


ax.set_title(
    "Clean epoch availability before within-subject matching"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "03_clean_epoch_availability.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 23. SELECTED-TIME COVERAGE
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "SELECTED-TIME COVERAGE"
)

print(
    "=" * 80
)


coverage_table = (

    subject_summary[
        [
            "subject",
            "group",
            "matched_epochs_per_condition",
            "matched_seconds_per_condition",
            "baseline_first_selected_time_sec",
            "baseline_last_selected_time_sec",
            "task_first_selected_time_sec",
            "task_last_selected_time_sec"
        ]
    ]

    .copy()
)


display(
    coverage_table.round(
        3
    )
)


coverage_table.to_csv(

    OUT
    /
    "selected_time_coverage.csv",

    index=False
)


# ================================================================
# 24. FIGURE 4 — TEMPORAL COVERAGE OF SELECTED TASK EPOCHS
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        13,
        6
    )
)


x = np.arange(
    len(
        coverage_table
    )
)


ax.bar(

    x,

    coverage_table[
        "task_last_selected_time_sec"
    ].to_numpy()
)


ax.set_xticks(
    x
)


ax.set_xticklabels(

    coverage_table[
        "subject"
    ],

    rotation=45,

    ha="right"
)


ax.axhline(

    MAX_CONDITION_SEC,

    linestyle="--",

    linewidth=1
)


ax.set_ylabel(
    "End time of last selected task epoch (s)"
)


ax.set_xlabel(
    "Participant"
)


ax.set_title(
    "Temporal coverage of matched task-onset epochs"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "04_task_temporal_coverage.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 25. GROUP-SPECIFIC PARTICIPANT COUNTS
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "GROUP PARTICIPANT COUNTS"
)

print(
    "=" * 80
)


group_counts = (

    subject_summary

    .groupby(
        "group"
    )[
        "subject"
    ]

    .nunique()

    .reset_index(
        name="n_subjects"
    )
)


display(
    group_counts
)


# ================================================================
# 26. EXCLUSION REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "EXCLUSIONS"
)

print(
    "=" * 80
)


if len(
    exclusions
) == 0:

    print(
        "None."
    )


else:

    display(
        exclusions
    )


# ================================================================
# 27. PROCESSING ERROR REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PROCESSING ERRORS"
)

print(
    "=" * 80
)


if len(
    errors
) == 0:

    print(
        "None."
    )


else:

    display(
        errors
    )


# ================================================================
# 28. FINAL NUMERICAL INTEGRITY CHECKS
# ================================================================

n_processed = int(

    subject_summary[
        "subject"
    ].nunique()
)


n_active = int(

    subject_summary.loc[
        subject_summary[
            "group"
        ]
        ==
        "Active",
        "subject"
    ].nunique()
)


n_passive = int(

    subject_summary.loc[
        subject_summary[
            "group"
        ]
        ==
        "Passive",
        "subject"
    ].nunique()
)


total_matched_epochs = int(

    subject_summary[
        "matched_total_epochs"
    ].sum()
)


total_matched_condition_pairs = int(

    subject_summary[
        "matched_epochs_per_condition"
    ].sum()
)


minimum_matched = int(

    subject_summary[
        "matched_epochs_per_condition"
    ].min()
)


median_matched = float(

    subject_summary[
        "matched_epochs_per_condition"
    ].median()
)


mean_matched = float(

    subject_summary[
        "matched_epochs_per_condition"
    ].mean()
)


maximum_matched = int(

    subject_summary[
        "matched_epochs_per_condition"
    ].max()
)


minimum_seconds = float(

    subject_summary[
        "matched_seconds_per_condition"
    ].min()
)


median_seconds = float(

    subject_summary[
        "matched_seconds_per_condition"
    ].median()
)


maximum_seconds = float(

    subject_summary[
        "matched_seconds_per_condition"
    ].max()
)


all_metadata_valid = bool(

    metadata_count_check[
        "valid"
    ].all()
)


# ================================================================
# 29. FINAL PASS/FAIL FLAGS
# ================================================================

participant_count_valid = (
    n_processed
    ==
    15
)


group_counts_valid = (

    n_active
    ==
    8

    and

    n_passive
    ==
    7
)


matching_valid = (
    all_exact
)


markers_valid = (
    all_marker_valid
)


metadata_valid = (
    all_metadata_valid
)


overall_integrity_pass = bool(

    participant_count_valid

    and

    group_counts_valid

    and

    matching_valid

    and

    markers_valid

    and

    metadata_valid

    and

    len(
        exclusions
    )
    ==
    0

    and

    len(
        errors
    )
    ==
    0
)


# ================================================================
# 30. SAVE FINAL INTEGRITY SUMMARY
# ================================================================

integrity_summary = pd.DataFrame([{

    "participants_processed":
        n_processed,

    "expected_participants":
        15,

    "active_participants":
        n_active,

    "expected_active":
        8,

    "passive_participants":
        n_passive,

    "expected_passive":
        7,

    "total_matched_epochs":
        total_matched_epochs,

    "total_matched_condition_pairs":
        total_matched_condition_pairs,

    "minimum_matched_epochs_per_condition":
        minimum_matched,

    "mean_matched_epochs_per_condition":
        mean_matched,

    "median_matched_epochs_per_condition":
        median_matched,

    "maximum_matched_epochs_per_condition":
        maximum_matched,

    "minimum_matched_seconds_per_condition":
        minimum_seconds,

    "median_matched_seconds_per_condition":
        median_seconds,

    "maximum_matched_seconds_per_condition":
        maximum_seconds,

    "within_subject_matching_valid":
        matching_valid,

    "marker_structure_valid":
        markers_valid,

    "metadata_integrity_valid":
        metadata_valid,

    "participant_count_valid":
        participant_count_valid,

    "group_counts_valid":
        group_counts_valid,

    "n_exclusions":
        len(
            exclusions
        ),

    "n_errors":
        len(
            errors
        ),

    "overall_integrity_pass":
        overall_integrity_pass
}])


integrity_summary.to_csv(

    OUT
    /
    "task03d_integrity_summary.csv",

    index=False
)


# ================================================================
# 31. TASK 3D FINAL REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "TASK 3D COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participants processed          : "
    f"{n_processed}/15"
)


print(
    f"Active participants             : "
    f"{n_active}/8"
)


print(
    f"Passive participants            : "
    f"{n_passive}/7"
)


print(
    f"Excluded participants           : "
    f"{len(exclusions)}"
)


print(
    f"Processing errors               : "
    f"{len(errors)}"
)


print(
    f"Total matched epochs            : "
    f"{total_matched_epochs}"
)


print(
    f"Matched baseline-task pairs     : "
    f"{total_matched_condition_pairs}"
)


print(
    f"Minimum epochs/condition        : "
    f"{minimum_matched}"
)


print(
    f"Mean epochs/condition           : "
    f"{mean_matched:.2f}"
)


print(
    f"Median epochs/condition         : "
    f"{median_matched:.1f}"
)


print(
    f"Maximum epochs/condition        : "
    f"{maximum_matched}"
)


print(
    f"Minimum matched duration        : "
    f"{minimum_seconds:.1f} s/condition"
)


print(
    f"Median matched duration         : "
    f"{median_seconds:.1f} s/condition"
)


print(
    f"Maximum matched duration        : "
    f"{maximum_seconds:.1f} s/condition"
)


print(
    f"Maximum permitted duration      : "
    f"{MAX_CONDITION_SEC:.1f} s/condition"
)


print(
    f"Epoch duration                  : "
    f"{EPOCH_SEC:.1f} s"
)


print(
    "\n"
    + "-" * 80
)


print(
    f"All within-subject counts equal : "
    f"{matching_valid}"
)


print(
    f"All marker structures valid     : "
    f"{markers_valid}"
)


print(
    f"All metadata counts valid       : "
    f"{metadata_valid}"
)


print(
    f"Participant count valid         : "
    f"{participant_count_valid}"
)


print(
    f"Group counts valid               : "
    f"{group_counts_valid}"
)


print(
    "\n"
    + "-" * 80
)


print(
    "ANALYTICAL CONDITIONS"
)


print(
    "\nBaseline:"
)


print(
    "  Marker position : 1"
)


print(
    "  Marker code     : T2_V1"
)


print(
    "  Analytical name : Baseline A"
)


print(
    "  Interpretation  : physiologically consistent with the "
    "eyes-open-like baseline based on Task 3C"
)


print(
    "\nTask onset:"
)


print(
    "  Active group    : position 5 / T1_V1"
)


print(
    "  Passive group   : position 5 / T1_V4"
)


print(
    "\nNo undocumented semantic meaning has been assigned "
    "to T1_V1 or T1_V4."
)


print(
    "\n"
    + "-" * 80
)


print(
    "MATCHING RULE"
)


print(
    "\nFor participant i:"
)


print(
    "n_i = min(clean baseline epochs, clean task epochs, 15)"
)


print(
    "\nThe earliest n_i clean epochs from each condition "
    "were retained."
)


print(
    "\nThe participant—not the individual 2-s epoch—will "
    "remain the inferential unit in subsequent group statistics."
)


print(
    "\n"
    + "-" * 80
)


if overall_integrity_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )


    print(
        "\nThe matched analytical dataset is ready for "
        "Task 4A."
    )


else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


    print(
        "\nDo NOT proceed to Task 4A until the failed "
        "integrity condition(s) above have been examined."
    )


print(
    "\n"
    + "-" * 80
)


print(
    "NEXT:"
)


print(
    "Task 4A — Baseline-normalized spectral dynamics and "
    "Active-vs-Passive inference."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# ================================================================
# TASK 4A — BASELINE-NORMALIZED SPECTRAL DYNAMICS
# ACTIVE VS PASSIVE
# ================================================================
#
# PRIMARY QUESTION
# ----------------
# Do Active and Passive GenAI engagement modes differ in the
# change in EEG spectral activity from the common pre-task
# Baseline A (T2_V1) to the first marker-defined task phase?
#
# DATA
# ----
# Task 3D matched epochs:
#
#   Baseline:
#       position 1 / T2_V1
#
#   Task:
#       Active  -> position 5 / T1_V1
#       Passive -> position 5 / T1_V4
#
# IMPORTANT
# ---------
# T1_V1 and T1_V4 are treated as protocol-defined task-onset
# markers. No undocumented semantic interpretation is assigned.
#
# STATISTICAL UNIT
# ----------------
# Participant, NOT individual 2-s epoch.
#
# PRIMARY BANDS
# -------------
# Theta : 4–8 Hz
# Alpha : 8–13 Hz
# Beta  : 13–30 Hz
#
# PRIMARY ROIs
# ------------
# Frontal:
#   AF3, F7, F3, F4, F8, AF4
#
# Central/temporal:
#   FC5, T7, T8, FC6
#
# Posterior:
#   P7, O1, O2, P8
#
# NORMALIZATION
# -------------
# For each participant, band and ROI:
#
#   log_ratio = ln(Task power / Baseline power)
#
# Interpretation:
#
#   > 0 : task-related increase
#   < 0 : task-related decrease
#   = 0 : no change
#
# INFERENCE
# ---------
# Active vs Passive:
#
#   exact two-sided permutation test
#
# There are:
#
#   C(15, 8) = 6435
#
# possible assignments of 8 Active participants among 15.
#
# EFFECT SIZE
# -----------
# Hedges' g:
#
#   positive = larger task-related change in Active
#   negative = larger task-related change in Passive
#
# MULTIPLE COMPARISONS
# --------------------
# BH-FDR across the 9 prespecified band × ROI contrasts.
#
# BOOTSTRAP
# ---------
# 10,000 stratified bootstrap resamples for the difference in
# group means:
#
#   Active - Passive
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
from itertools import combinations
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from scipy.stats import ttest_1samp


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. PATHS
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)


TASK3D = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03d_matched_baseline_task_epochs"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task04a_baseline_normalized_spectral"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. ANALYSIS SETTINGS
# ================================================================

PSD_FMIN = 1.0

PSD_FMAX = 45.0


BANDS = {

    "Theta":
        (4.0, 8.0),

    "Alpha":
        (8.0, 13.0),

    "Beta":
        (13.0, 30.0)
}


ROIS = {

    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "CentralTemporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}


EXPECTED_GROUP_COUNTS = {

    "Active":
        8,

    "Passive":
        7
}


N_BOOTSTRAP = 10000

RANDOM_SEED = 20260913


rng = np.random.default_rng(
    RANDOM_SEED
)


# ================================================================
# 4. HELPER — BAND INTEGRATION
# ================================================================

def integrate_band(
    freqs,
    psd,
    low,
    high
):

    mask = (

        (freqs >= low)

        &

        (freqs < high)
    )


    if np.sum(mask) < 2:

        return np.full(
            psd.shape[:-1],
            np.nan
        )


    if hasattr(
        np,
        "trapezoid"
    ):

        return np.trapezoid(

            psd[
                ...,
                mask
            ],

            freqs[
                mask
            ],

            axis=-1
        )


    return np.trapz(

        psd[
            ...,
            mask
        ],

        freqs[
            mask
        ],

        axis=-1
    )


# ================================================================
# 5. HELPER — HEDGES' g
# ================================================================

def hedges_g(
    x,
    y
):

    x = np.asarray(
        x,
        dtype=float
    )


    y = np.asarray(
        y,
        dtype=float
    )


    x = x[
        np.isfinite(x)
    ]


    y = y[
        np.isfinite(y)
    ]


    nx = len(x)

    ny = len(y)


    if nx < 2 or ny < 2:

        return np.nan


    vx = np.var(
        x,
        ddof=1
    )


    vy = np.var(
        y,
        ddof=1
    )


    pooled_variance = (

        (
            (nx - 1)
            *
            vx
        )

        +

        (
            (ny - 1)
            *
            vy
        )

    ) / (

        nx
        +
        ny
        -
        2
    )


    if pooled_variance <= 0:

        return np.nan


    pooled_sd = np.sqrt(
        pooled_variance
    )


    cohen_d = (

        np.mean(x)
        -
        np.mean(y)

    ) / pooled_sd


    df = (
        nx
        +
        ny
        -
        2
    )


    correction = (

        1

        -

        (
            3
            /
            (
                4
                *
                df
                -
                1
            )
        )
    )


    return float(
        correction
        *
        cohen_d
    )


# ================================================================
# 6. HELPER — EXACT PERMUTATION TEST
# ================================================================

def exact_permutation_test(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )


    passive = np.asarray(
        passive,
        dtype=float
    )


    active = active[
        np.isfinite(active)
    ]


    passive = passive[
        np.isfinite(passive)
    ]


    n_active = len(active)

    n_passive = len(passive)


    if (
        n_active == 0
        or
        n_passive == 0
    ):

        return (
            np.nan,
            np.nan,
            0
        )


    combined = np.concatenate(
        [
            active,
            passive
        ]
    )


    n_total = len(
        combined
    )


    observed = float(

        np.mean(active)

        -

        np.mean(passive)
    )


    indices = np.arange(
        n_total
    )


    permutation_statistics = []


    for active_indices_tuple in combinations(
        indices,
        n_active
    ):

        active_indices = np.asarray(
            active_indices_tuple,
            dtype=int
        )


        active_mask = np.zeros(
            n_total,
            dtype=bool
        )


        active_mask[
            active_indices
        ] = True


        perm_active = combined[
            active_mask
        ]


        perm_passive = combined[
            ~active_mask
        ]


        statistic = (

            np.mean(
                perm_active
            )

            -

            np.mean(
                perm_passive
            )
        )


        permutation_statistics.append(
            statistic
        )


    permutation_statistics = np.asarray(
        permutation_statistics,
        dtype=float
    )


    # Exact two-sided p-value.
    #
    # Numerical tolerance prevents floating-point boundary issues.

    tolerance = 1e-12


    p_value = float(

        np.mean(

            np.abs(
                permutation_statistics
            )

            >=

            (
                abs(
                    observed
                )
                -
                tolerance
            )
        )
    )


    return (
        observed,
        p_value,
        len(
            permutation_statistics
        )
    )


# ================================================================
# 7. HELPER — STRATIFIED BOOTSTRAP CI
# ================================================================

def bootstrap_mean_difference_ci(
    active,
    passive,
    n_bootstrap=10000,
    seed=20260913
):

    local_rng = np.random.default_rng(
        seed
    )


    active = np.asarray(
        active,
        dtype=float
    )


    passive = np.asarray(
        passive,
        dtype=float
    )


    active = active[
        np.isfinite(active)
    ]


    passive = passive[
        np.isfinite(passive)
    ]


    if (
        len(active) == 0
        or
        len(passive) == 0
    ):

        return (
            np.nan,
            np.nan
        )


    differences = np.empty(
        n_bootstrap,
        dtype=float
    )


    for iteration in range(
        n_bootstrap
    ):

        active_sample = local_rng.choice(

            active,

            size=len(active),

            replace=True
        )


        passive_sample = local_rng.choice(

            passive,

            size=len(passive),

            replace=True
        )


        differences[
            iteration
        ] = (

            np.mean(
                active_sample
            )

            -

            np.mean(
                passive_sample
            )
        )


    lower = float(
        np.percentile(
            differences,
            2.5
        )
    )


    upper = float(
        np.percentile(
            differences,
            97.5
        )
    )


    return (
        lower,
        upper
    )


# ================================================================
# 8. HELPER — BENJAMINI-HOCHBERG FDR
# ================================================================

def benjamini_hochberg(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    adjusted = np.full(
        len(p_values),
        np.nan,
        dtype=float
    )


    valid_mask = np.isfinite(
        p_values
    )


    valid_p = p_values[
        valid_mask
    ]


    if len(valid_p) == 0:

        return adjusted


    order = np.argsort(
        valid_p
    )


    ranked_p = valid_p[
        order
    ]


    m = len(
        ranked_p
    )


    adjusted_ranked = (

        ranked_p

        *

        m

        /

        np.arange(
            1,
            m + 1
        )
    )


    # Enforce monotonicity.
    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]


    adjusted_ranked = np.minimum(
        adjusted_ranked,
        1.0
    )


    valid_adjusted = np.empty(
        m,
        dtype=float
    )


    valid_adjusted[
        order
    ] = adjusted_ranked


    adjusted[
        valid_mask
    ] = valid_adjusted


    return adjusted


# ================================================================
# 9. FIND MATCHED EPOCH FILES
# ================================================================

epoch_files = sorted(
    TASK3D.glob(
        "sub-*/"
        "*desc-matched_baseline_task_epo.fif"
    )
)


print(
    "=" * 80
)

print(
    "TASK 4A — BASELINE-NORMALIZED SPECTRAL DYNAMICS"
)

print(
    "=" * 80
)


print(
    f"Matched epoch files found : {len(epoch_files)}"
)


if len(
    epoch_files
) != 15:

    raise RuntimeError(

        f"Expected 15 Task 3D epoch files; "
        f"found {len(epoch_files)}."
    )


# ================================================================
# 10. PROCESS PARTICIPANTS
# ================================================================

epoch_band_rows = []

subject_band_channel_rows = []

subject_roi_rows = []

subject_global_rows = []

error_rows = []


channel_order = None

info_template = None


for epoch_file in epoch_files:

    print(
        "\n"
        + "-" * 80
    )


    try:

        epochs = mne.read_epochs(

            epoch_file,

            preload=True,

            verbose=False
        )


        metadata = (
            epochs.metadata.copy()
        )


        if metadata is None:

            raise RuntimeError(
                "Epoch metadata missing."
            )


        subjects = (
            metadata[
                "subject"
            ]
            .astype(str)
            .unique()
        )


        groups = (
            metadata[
                "group"
            ]
            .astype(str)
            .unique()
        )


        if len(subjects) != 1:

            raise RuntimeError(
                f"Expected one subject; found {subjects}."
            )


        if len(groups) != 1:

            raise RuntimeError(
                f"Expected one group; found {groups}."
            )


        subject = str(
            subjects[0]
        )


        group = str(
            groups[0]
        )


        if group not in EXPECTED_GROUP_COUNTS:

            raise RuntimeError(
                f"Unexpected group: {group}"
            )


        print(
            f"{subject} | {group}"
        )


        ch_names = list(
            epochs.ch_names
        )


        sfreq = float(
            epochs.info[
                "sfreq"
            ]
        )


        if channel_order is None:

            channel_order = (
                ch_names.copy()
            )


            info_template = (
                epochs.info.copy()
            )


        elif ch_names != channel_order:

            raise RuntimeError(
                "Channel order mismatch."
            )


        # --------------------------------------------------------
        # Verify ROIs
        # --------------------------------------------------------

        for (
            roi_name,
            roi_channels
        ) in ROIS.items():

            missing = [

                channel

                for channel in roi_channels

                if channel not in ch_names
            ]


            if missing:

                raise RuntimeError(

                    f"{subject}: ROI {roi_name} "
                    f"missing channels {missing}"
                )


        # --------------------------------------------------------
        # Process baseline and task separately
        # --------------------------------------------------------

        condition_channel_band = {}


        for condition in [
            "baseline",
            "task"
        ]:

            condition_epochs = epochs[
                condition
            ]


            data_uv = (

                condition_epochs.get_data()

                *
                1e6
            )


            if data_uv.shape[0] == 0:

                raise RuntimeError(
                    f"No {condition} epochs."
                )


            n_samples = int(
                data_uv.shape[
                    -1
                ]
            )


            nperseg = min(

                n_samples,

                int(
                    round(
                        2.0
                        *
                        sfreq
                    )
                )
            )


            freqs, psd = welch(

                data_uv,

                fs=sfreq,

                window="hann",

                nperseg=nperseg,

                noverlap=nperseg // 2,

                detrend="constant",

                scaling="density",

                axis=-1
            )


            freq_mask = (

                (freqs >= PSD_FMIN)

                &

                (freqs <= PSD_FMAX)
            )


            selected_freqs = (
                freqs[
                    freq_mask
                ]
            )


            selected_psd = (
                psd[
                    ...,
                    freq_mask
                ]
            )


            # ----------------------------------------------------
            # Integrate bands for every epoch and channel
            # ----------------------------------------------------

            condition_channel_band[
                condition
            ] = {}


            for (
                band,
                (
                    low,
                    high
                )
            ) in BANDS.items():

                band_power = integrate_band(

                    selected_freqs,

                    selected_psd,

                    low,

                    high
                )


                # Shape:
                # epochs × channels


                condition_channel_band[
                    condition
                ][
                    band
                ] = (
                    band_power.copy()
                )


                # Save epoch-level values for QC/reproducibility.
                for epoch_index in range(
                    band_power.shape[0]
                ):

                    for channel_index, channel in enumerate(
                        ch_names
                    ):

                        epoch_band_rows.append({

                            "subject":
                                subject,

                            "group":
                                group,

                            "condition":
                                condition,

                            "epoch_index":
                                int(
                                    epoch_index + 1
                                ),

                            "band":
                                band,

                            "channel":
                                channel,

                            "absolute_power_uV2":
                                float(
                                    band_power[
                                        epoch_index,
                                        channel_index
                                    ]
                                )
                        })


        # --------------------------------------------------------
        # Participant-level channel summaries
        # --------------------------------------------------------

        for band in BANDS.keys():

            baseline_power = (

                condition_channel_band[
                    "baseline"
                ][
                    band
                ]
            )


            task_power = (

                condition_channel_band[
                    "task"
                ][
                    band
                ]
            )


            # Robust participant estimate:
            # median across matched epochs.
            baseline_channel = np.median(

                baseline_power,

                axis=0
            )


            task_channel = np.median(

                task_power,

                axis=0
            )


            epsilon = np.finfo(
                float
            ).tiny


            channel_log_ratio = np.log(

                np.maximum(
                    task_channel,
                    epsilon
                )

                /

                np.maximum(
                    baseline_channel,
                    epsilon
                )
            )


            for channel_index, channel in enumerate(
                ch_names
            ):

                subject_band_channel_rows.append({

                    "subject":
                        subject,

                    "group":
                        group,

                    "band":
                        band,

                    "channel":
                        channel,

                    "baseline_power_uV2":
                        float(
                            baseline_channel[
                                channel_index
                            ]
                        ),

                    "task_power_uV2":
                        float(
                            task_channel[
                                channel_index
                            ]
                        ),

                    "log_ratio_task_vs_baseline":
                        float(
                            channel_log_ratio[
                                channel_index
                            ]
                        ),

                    "percent_change":
                        float(
                            (
                                np.exp(
                                    channel_log_ratio[
                                        channel_index
                                    ]
                                )
                                -
                                1
                            )
                            *
                            100
                        )
                })


            # ----------------------------------------------------
            # ROI summaries
            # ----------------------------------------------------

            for (
                roi_name,
                roi_channels
            ) in ROIS.items():

                roi_indices = [

                    ch_names.index(
                        channel
                    )

                    for channel in roi_channels
                ]


                # Median across epochs first,
                # then mean across channels in the ROI.
                baseline_roi = float(

                    np.mean(
                        baseline_channel[
                            roi_indices
                        ]
                    )
                )


                task_roi = float(

                    np.mean(
                        task_channel[
                            roi_indices
                        ]
                    )
                )


                log_ratio = float(

                    np.log(

                        max(
                            task_roi,
                            epsilon
                        )

                        /

                        max(
                            baseline_roi,
                            epsilon
                        )
                    )
                )


                percent_change = float(

                    (
                        np.exp(
                            log_ratio
                        )
                        -
                        1
                    )
                    *
                    100
                )


                subject_roi_rows.append({

                    "subject":
                        subject,

                    "group":
                        group,

                    "band":
                        band,

                    "roi":
                        roi_name,

                    "baseline_power_uV2":
                        baseline_roi,

                    "task_power_uV2":
                        task_roi,

                    "log_ratio_task_vs_baseline":
                        log_ratio,

                    "percent_change":
                        percent_change,

                    "n_matched_epochs":
                        int(
                            len(
                                epochs[
                                    "baseline"
                                ]
                            )
                        )
                })


            # ----------------------------------------------------
            # Whole-scalp summary
            # ----------------------------------------------------

            baseline_global = float(

                np.mean(
                    baseline_channel
                )
            )


            task_global = float(

                np.mean(
                    task_channel
                )
            )


            global_log_ratio = float(

                np.log(

                    max(
                        task_global,
                        epsilon
                    )

                    /

                    max(
                        baseline_global,
                        epsilon
                    )
                )
            )


            subject_global_rows.append({

                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "baseline_power_uV2":
                    baseline_global,

                "task_power_uV2":
                    task_global,

                "log_ratio_task_vs_baseline":
                    global_log_ratio,

                "percent_change":
                    float(
                        (
                            np.exp(
                                global_log_ratio
                            )
                            -
                            1
                        )
                        *
                        100
                    )
            })


        print(
            f"  matched epochs/condition: "
            f"{len(epochs['baseline'])}"
        )


        print(
            "  spectral extraction: SUCCESS"
        )


    except Exception as exc:

        print(
            f"ERROR: {exc}"
        )


        error_rows.append({

            "file":
                str(
                    epoch_file
                ),

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 11. CREATE TABLES
# ================================================================

epoch_band = pd.DataFrame(
    epoch_band_rows
)


subject_channel = pd.DataFrame(
    subject_band_channel_rows
)


subject_roi = pd.DataFrame(
    subject_roi_rows
)


subject_global = pd.DataFrame(
    subject_global_rows
)


errors = pd.DataFrame(
    error_rows
)


if len(
    errors
) > 0:

    print(
        "\nPROCESSING ERRORS:"
    )

    display(
        errors
    )


if (
    subject_roi[
        "subject"
    ].nunique()
    !=
    15
):

    raise RuntimeError(

        "Task 4A did not successfully process "
        "all 15 participants."
    )


# ================================================================
# 12. SAVE RAW SPECTRAL TABLES
# ================================================================

epoch_band.to_csv(

    OUT
    /
    "epoch_channel_band_power.csv",

    index=False
)


subject_channel.to_csv(

    OUT
    /
    "subject_channel_spectral_change.csv",

    index=False
)


subject_roi.to_csv(

    OUT
    /
    "subject_roi_spectral_change.csv",

    index=False
)


subject_global.to_csv(

    OUT
    /
    "subject_global_spectral_change.csv",

    index=False
)


errors.to_csv(

    OUT
    /
    "processing_errors.csv",

    index=False
)


# ================================================================
# 13. PARTICIPANT-LEVEL ROI CHANGE TABLE
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PARTICIPANT-LEVEL ROI LOG-RATIO CHANGES"
)

print(
    "=" * 80
)


participant_change_table = (

    subject_roi

    .pivot_table(

        index=[
            "subject",
            "group"
        ],

        columns=[
            "band",
            "roi"
        ],

        values="log_ratio_task_vs_baseline"
    )

    .reset_index()
)


display(
    participant_change_table.round(
        4
    )
)


participant_change_table.to_csv(

    OUT
    /
    "participant_roi_logratio_matrix.csv",

    index=False
)


# ================================================================
# 14. DESCRIPTIVE GROUP SUMMARY
# ================================================================

group_descriptive = (

    subject_roi

    .groupby(
        [
            "group",
            "band",
            "roi"
        ]
    )

    .agg(

        n=(
            "subject",
            "nunique"
        ),

        mean_log_ratio=(
            "log_ratio_task_vs_baseline",
            "mean"
        ),

        sd_log_ratio=(
            "log_ratio_task_vs_baseline",
            "std"
        ),

        median_log_ratio=(
            "log_ratio_task_vs_baseline",
            "median"
        ),

        mean_percent_change=(
            "percent_change",
            "mean"
        ),

        median_percent_change=(
            "percent_change",
            "median"
        )

    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)

print(
    "GROUP DESCRIPTIVE SUMMARY"
)

print(
    "=" * 80
)


display(
    group_descriptive.round(
        4
    )
)


group_descriptive.to_csv(

    OUT
    /
    "group_descriptive_spectral_change.csv",

    index=False
)


# ================================================================
# 15. WITHIN-GROUP BASELINE → TASK TESTS
# EXPLORATORY/DESCRIPTIVE
# ================================================================

within_group_rows = []


for group in [
    "Active",
    "Passive"
]:

    for band in BANDS.keys():

        for roi in ROIS.keys():

            values = subject_roi.loc[
                (
                    subject_roi[
                        "group"
                    ]
                    ==
                    group
                )

                &

                (
                    subject_roi[
                        "band"
                    ]
                    ==
                    band
                )

                &

                (
                    subject_roi[
                        "roi"
                    ]
                    ==
                    roi
                ),
                "log_ratio_task_vs_baseline"
            ].to_numpy(
                dtype=float
            )


            if len(values) >= 2:

                result = ttest_1samp(

                    values,

                    popmean=0.0
                )


                statistic = float(
                    np.asarray(
                        result.statistic
                    ).squeeze()
                )


                p_value = float(
                    np.asarray(
                        result.pvalue
                    ).squeeze()
                )


                sd = float(
                    np.std(
                        values,
                        ddof=1
                    )
                )


                if sd > 0:

                    dz = float(

                        np.mean(
                            values
                        )

                        /
                        sd
                    )

                else:

                    dz = np.nan


            else:

                statistic = np.nan

                p_value = np.nan

                dz = np.nan


            within_group_rows.append({

                "group":
                    group,

                "band":
                    band,

                "roi":
                    roi,

                "n":
                    len(
                        values
                    ),

                "mean_log_ratio":
                    float(
                        np.mean(
                            values
                        )
                    ),

                "median_log_ratio":
                    float(
                        np.median(
                            values
                        )
                    ),

                "t_statistic":
                    statistic,

                "p_value":
                    p_value,

                "cohen_dz":
                    dz
            })


within_group = pd.DataFrame(
    within_group_rows
)


within_group[
    "p_fdr"
] = benjamini_hochberg(

    within_group[
        "p_value"
    ].to_numpy()
)


within_group[
    "significant_fdr_0_05"
] = (

    within_group[
        "p_fdr"
    ]
    <
    0.05
)


print(
    "\n"
    + "=" * 80
)

print(
    "WITHIN-GROUP BASELINE → TASK CHANGES"
)

print(
    "=" * 80
)


display(
    within_group.round(
        5
    )
)


within_group.to_csv(

    OUT
    /
    "within_group_baseline_task_tests.csv",

    index=False
)


# ================================================================
# 16. PRIMARY ACTIVE VS PASSIVE EXACT PERMUTATION TESTS
# ================================================================

comparison_rows = []


contrast_number = 0


for band in BANDS.keys():

    for roi in ROIS.keys():

        contrast_number += 1


        active = subject_roi.loc[
            (
                subject_roi[
                    "group"
                ]
                ==
                "Active"
            )

            &

            (
                subject_roi[
                    "band"
                ]
                ==
                band
            )

            &

            (
                subject_roi[
                    "roi"
                ]
                ==
                roi
            ),
            "log_ratio_task_vs_baseline"
        ].to_numpy(
            dtype=float
        )


        passive = subject_roi.loc[
            (
                subject_roi[
                    "group"
                ]
                ==
                "Passive"
            )

            &

            (
                subject_roi[
                    "band"
                ]
                ==
                band
            )

            &

            (
                subject_roi[
                    "roi"
                ]
                ==
                roi
            ),
            "log_ratio_task_vs_baseline"
        ].to_numpy(
            dtype=float
        )


        (
            observed_difference,
            permutation_p,
            n_permutations
        ) = exact_permutation_test(

            active,

            passive
        )


        effect_size = hedges_g(

            active,

            passive
        )


        (
            bootstrap_low,
            bootstrap_high
        ) = bootstrap_mean_difference_ci(

            active,

            passive,

            n_bootstrap=N_BOOTSTRAP,

            seed=(
                RANDOM_SEED
                +
                contrast_number
            )
        )


        comparison_rows.append({

            "band":
                band,

            "roi":
                roi,

            "n_active":
                len(
                    active
                ),

            "n_passive":
                len(
                    passive
                ),

            "active_mean_log_ratio":
                float(
                    np.mean(
                        active
                    )
                ),

            "passive_mean_log_ratio":
                float(
                    np.mean(
                        passive
                    )
                ),

            "active_median_log_ratio":
                float(
                    np.median(
                        active
                    )
                ),

            "passive_median_log_ratio":
                float(
                    np.median(
                        passive
                    )
                ),

            "active_mean_percent_change":
                float(
                    (
                        np.exp(
                            np.mean(
                                active
                            )
                        )
                        -
                        1
                    )
                    *
                    100
                ),

            "passive_mean_percent_change":
                float(
                    (
                        np.exp(
                            np.mean(
                                passive
                            )
                        )
                        -
                        1
                    )
                    *
                    100
                ),

            "active_minus_passive":
                observed_difference,

            "bootstrap_95ci_low":
                bootstrap_low,

            "bootstrap_95ci_high":
                bootstrap_high,

            "hedges_g":
                effect_size,

            "exact_permutation_p":
                permutation_p,

            "n_exact_permutations":
                n_permutations
        })


comparison = pd.DataFrame(
    comparison_rows
)


# ================================================================
# 17. BH-FDR FOR PRIMARY 9 CONTRASTS
# ================================================================

comparison[
    "p_fdr"
] = benjamini_hochberg(

    comparison[
        "exact_permutation_p"
    ].to_numpy()
)


comparison[
    "significant_uncorrected_0_05"
] = (

    comparison[
        "exact_permutation_p"
    ]
    <
    0.05
)


comparison[
    "significant_fdr_0_05"
] = (

    comparison[
        "p_fdr"
    ]
    <
    0.05
)


comparison[
    "ci_excludes_zero"
] = (

    (
        comparison[
            "bootstrap_95ci_low"
        ]
        >
        0
    )

    |

    (
        comparison[
            "bootstrap_95ci_high"
        ]
        <
        0
    )
)


# ================================================================
# 18. PRIMARY INFERENTIAL RESULTS
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PRIMARY ACTIVE VS PASSIVE SPECTRAL RESULTS"
)

print(
    "=" * 80
)


display(

    comparison[
        [
            "band",
            "roi",
            "n_active",
            "n_passive",
            "active_mean_log_ratio",
            "passive_mean_log_ratio",
            "active_minus_passive",
            "bootstrap_95ci_low",
            "bootstrap_95ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_fdr",
            "significant_fdr_0_05"
        ]
    ].round(
        5
    )
)


comparison.to_csv(

    OUT
    /
    "primary_active_passive_spectral_tests.csv",

    index=False
)


# ================================================================
# 19. VERIFY EXACT PERMUTATION COUNT
# ================================================================

unique_permutation_counts = (
    comparison[
        "n_exact_permutations"
    ].unique()
)


print(
    "\n"
    + "=" * 80
)

print(
    "EXACT PERMUTATION CHECK"
)

print(
    "=" * 80
)


print(
    f"Permutation counts observed: "
    f"{unique_permutation_counts.tolist()}"
)


expected_permutations = 6435


all_exact_permutation_counts_valid = bool(

    np.all(
        comparison[
            "n_exact_permutations"
        ]
        ==
        expected_permutations
    )
)


print(
    f"Expected permutations        : "
    f"{expected_permutations}"
)


print(
    f"All contrasts exact          : "
    f"{all_exact_permutation_counts_valid}"
)


# ================================================================
# 20. WHOLE-SCALP EXPLORATORY SUMMARY
# ================================================================

global_group_summary = (

    subject_global

    .groupby(
        [
            "group",
            "band"
        ]
    )

    .agg(

        n=(
            "subject",
            "nunique"
        ),

        mean_log_ratio=(
            "log_ratio_task_vs_baseline",
            "mean"
        ),

        sd_log_ratio=(
            "log_ratio_task_vs_baseline",
            "std"
        ),

        median_log_ratio=(
            "log_ratio_task_vs_baseline",
            "median"
        )

    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)

print(
    "WHOLE-SCALP EXPLORATORY SUMMARY"
)

print(
    "=" * 80
)


display(
    global_group_summary.round(
        5
    )
)


global_group_summary.to_csv(

    OUT
    /
    "whole_scalp_group_summary.csv",

    index=False
)


# ================================================================
# 21. FIGURE 1 — GROUP MEAN LOG-RATIO BY BAND AND ROI
# ================================================================

plot_summary = (

    subject_roi

    .groupby(
        [
            "group",
            "band",
            "roi"
        ]
    )[
        "log_ratio_task_vs_baseline"
    ]

    .agg(
        [
            "mean",
            "sem"
        ]
    )

    .reset_index()
)


band_order = [
    "Theta",
    "Alpha",
    "Beta"
]


roi_order = [
    "Frontal",
    "CentralTemporal",
    "Posterior"
]


group_order = [
    "Active",
    "Passive"
]


x = np.arange(
    len(
        band_order
    )
)


width = 0.12


fig, ax = plt.subplots(
    figsize=(
        12,
        7
    )
)


offset_counter = 0


for roi_index, roi in enumerate(
    roi_order
):

    for group_index, group in enumerate(
        group_order
    ):

        means = []

        sems = []


        for band in band_order:

            row = plot_summary[
                (
                    plot_summary[
                        "group"
                    ]
                    ==
                    group
                )

                &

                (
                    plot_summary[
                        "band"
                    ]
                    ==
                    band
                )

                &

                (
                    plot_summary[
                        "roi"
                    ]
                    ==
                    roi
                )
            ]


            means.append(
                float(
                    row[
                        "mean"
                    ].iloc[0]
                )
            )


            sems.append(
                float(
                    row[
                        "sem"
                    ].iloc[0]
                )
            )


        offset = (

            (
                offset_counter
                -
                2.5
            )
            *
            width
        )


        ax.bar(

            x
            +
            offset,

            means,

            width,

            yerr=sems,

            capsize=3,

            label=(
                f"{group} — {roi}"
            )
        )


        offset_counter += 1


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    band_order
)


ax.set_ylabel(
    "Task vs baseline log power ratio"
)


ax.set_xlabel(
    "Frequency band"
)


ax.set_title(
    "Baseline-normalized spectral dynamics"
)


ax.legend(
    fontsize=8,
    ncol=2
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "01_group_roi_spectral_logratio.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 22. FIGURE 2 — ACTIVE VS PASSIVE EFFECT SIZES
# ================================================================

effect_plot = (
    comparison.copy()
)


effect_plot[
    "contrast"
] = (

    effect_plot[
        "band"
    ]

    +
    " — "

    +
    effect_plot[
        "roi"
    ]
)


effect_plot = effect_plot.sort_values(
    "hedges_g"
)


fig, ax = plt.subplots(
    figsize=(
        10,
        7
    )
)


y = np.arange(
    len(
        effect_plot
    )
)


ax.scatter(

    effect_plot[
        "hedges_g"
    ],

    y,

    s=60
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_yticks(
    y
)


ax.set_yticklabels(
    effect_plot[
        "contrast"
    ]
)


ax.set_xlabel(
    "Hedges' g (Active − Passive)"
)


ax.set_ylabel(
    "Prespecified spectral contrast"
)


ax.set_title(
    "Effect sizes for Active versus Passive engagement"
)


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "02_active_passive_effect_sizes.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 23. FIGURE 3 — PARTICIPANT-LEVEL POSTERIOR ALPHA
# ================================================================

posterior_alpha = subject_roi[
    (
        subject_roi[
            "band"
        ]
        ==
        "Alpha"
    )

    &

    (
        subject_roi[
            "roi"
        ]
        ==
        "Posterior"
    )
].copy()


fig, ax = plt.subplots(
    figsize=(
        8,
        7
    )
)


for group_index, group in enumerate(
    group_order
):

    values = posterior_alpha.loc[
        posterior_alpha[
            "group"
        ]
        ==
        group,
        "log_ratio_task_vs_baseline"
    ].to_numpy()


    jitter = rng.normal(

        0,

        0.035,

        size=len(
            values
        )
    )


    ax.scatter(

        np.full(
            len(
                values
            ),
            group_index
        )
        +
        jitter,

        values,

        s=70,

        alpha=0.8
    )


    mean_value = float(
        np.mean(
            values
        )
    )


    ax.plot(

        [
            group_index
            -
            0.18,

            group_index
            +
            0.18
        ],

        [
            mean_value,
            mean_value
        ],

        linewidth=3
    )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    [
        0,
        1
    ]
)


ax.set_xticklabels(
    [
        "Active",
        "Passive"
    ]
)


ax.set_ylabel(
    "Posterior alpha log ratio\nTask / Baseline"
)


ax.set_xlabel(
    "Engagement group"
)


ax.set_title(
    "Participant-level posterior alpha change"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    OUT
    /
    "03_posterior_alpha_participants.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 24. CHANNEL-LEVEL GROUP DIFFERENCE TOPOGRAPHIES
# ================================================================

channel_group = (

    subject_channel

    .groupby(
        [
            "group",
            "band",
            "channel"
        ]
    )[
        "log_ratio_task_vs_baseline"
    ]

    .mean()

    .reset_index()
)


for band in band_order:

    active_values = (

        channel_group[
            (
                channel_group[
                    "group"
                ]
                ==
                "Active"
            )

            &

            (
                channel_group[
                    "band"
                ]
                ==
                band
            )
        ]

        .set_index(
            "channel"
        )

        .reindex(
            channel_order
        )[
            "log_ratio_task_vs_baseline"
        ]

        .to_numpy(
            dtype=float
        )
    )


    passive_values = (

        channel_group[
            (
                channel_group[
                    "group"
                ]
                ==
                "Passive"
            )

            &

            (
                channel_group[
                    "band"
                ]
                ==
                band
            )
        ]

        .set_index(
            "channel"
        )

        .reindex(
            channel_order
        )[
            "log_ratio_task_vs_baseline"
        ]

        .to_numpy(
            dtype=float
        )
    )


    difference_values = (

        active_values

        -

        passive_values
    )


    limit = float(
        np.nanmax(
            np.abs(
                difference_values
            )
        )
    )


    if (
        not np.isfinite(
            limit
        )

        or

        limit == 0
    ):

        limit = 1e-6


    fig, ax = plt.subplots(
        figsize=(
            7,
            6
        )
    )


    mne.viz.plot_topomap(

        difference_values,

        info_template,

        axes=ax,

        show=False,

        contours=6,

        vlim=(
            -limit,
            limit
        )
    )


    ax.set_title(
        f"{band}: Active − Passive\n"
        "baseline-normalized log power"
    )


    plt.tight_layout()


    plt.savefig(

        OUT
        /
        (
            f"topomap_{band.lower()}_"
            "active_minus_passive.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 25. RANK PRIMARY CONTRASTS
# ================================================================

ranked_results = (

    comparison

    .sort_values(
        [
            "p_fdr",
            "exact_permutation_p"
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "RANKED PRIMARY CONTRASTS"
)

print(
    "=" * 80
)


display(

    ranked_results[
        [
            "band",
            "roi",
            "active_minus_passive",
            "hedges_g",
            "bootstrap_95ci_low",
            "bootstrap_95ci_high",
            "exact_permutation_p",
            "p_fdr",
            "significant_fdr_0_05"
        ]
    ].round(
        5
    )
)


ranked_results.to_csv(

    OUT
    /
    "ranked_primary_spectral_results.csv",

    index=False
)


# ================================================================
# 26. AUTOMATIC CONSERVATIVE INTERPRETATION
# ================================================================

n_uncorrected = int(

    comparison[
        "significant_uncorrected_0_05"
    ].sum()
)


n_fdr = int(

    comparison[
        "significant_fdr_0_05"
    ].sum()
)


n_ci = int(

    comparison[
        "ci_excludes_zero"
    ].sum()
)


best = ranked_results.iloc[
    0
]


print(
    "\n"
    + "=" * 80
)

print(
    "CONSERVATIVE INTERPRETATION"
)

print(
    "=" * 80
)


print(
    f"Prespecified contrasts tested : "
    f"{len(comparison)}"
)


print(
    f"Uncorrected p < .05           : "
    f"{n_uncorrected}"
)


print(
    f"BH-FDR significant            : "
    f"{n_fdr}"
)


print(
    f"Bootstrap CIs excluding zero  : "
    f"{n_ci}"
)


print(
    "\nStrongest contrast:"
)


print(
    f"  Band                         : "
    f"{best['band']}"
)


print(
    f"  ROI                          : "
    f"{best['roi']}"
)


print(
    f"  Active − Passive log ratio   : "
    f"{best['active_minus_passive']:.4f}"
)


print(
    f"  Hedges' g                    : "
    f"{best['hedges_g']:.3f}"
)


print(
    f"  Exact permutation p          : "
    f"{best['exact_permutation_p']:.6f}"
)


print(
    f"  BH-FDR p                     : "
    f"{best['p_fdr']:.6f}"
)


print(
    f"  Bootstrap 95% CI             : "
    f"[{best['bootstrap_95ci_low']:.4f}, "
    f"{best['bootstrap_95ci_high']:.4f}]"
)


if n_fdr > 0:

    print(
        "\nAt least one prespecified Active-vs-Passive "
        "spectral contrast survives BH-FDR correction."
    )


    print(
        "Interpret the direction as a difference in "
        "baseline-normalized neural dynamics, not as direct "
        "evidence of learning, cognitive offloading, or superior "
        "educational performance."
    )


else:

    print(
        "\nNo prespecified Active-vs-Passive spectral "
        "contrast survives BH-FDR correction."
    )


    print(
        "The results should therefore be interpreted primarily "
        "through effect sizes, uncertainty intervals, and "
        "descriptive neural dynamics rather than claims of a "
        "statistically established group difference."
    )


# ================================================================
# 27. FINAL INTEGRITY CHECK
# ================================================================

n_subjects = int(
    subject_roi[
        "subject"
    ].nunique()
)


n_active = int(

    subject_roi.loc[
        subject_roi[
            "group"
        ]
        ==
        "Active",
        "subject"
    ].nunique()
)


n_passive = int(

    subject_roi.loc[
        subject_roi[
            "group"
        ]
        ==
        "Passive",
        "subject"
    ].nunique()
)


expected_subject_roi_rows = (

    15

    *
    len(
        BANDS
    )

    *
    len(
        ROIS
    )
)


row_count_valid = (

    len(
        subject_roi
    )

    ==
    expected_subject_roi_rows
)


group_count_valid = (

    n_active
    ==
    8

    and

    n_passive
    ==
    7
)


subject_count_valid = (
    n_subjects
    ==
    15
)


overall_pass = bool(

    row_count_valid

    and

    group_count_valid

    and

    subject_count_valid

    and

    all_exact_permutation_counts_valid

    and

    len(
        errors
    )
    ==
    0
)


# ================================================================
# 28. SAVE ANALYSIS INTEGRITY SUMMARY
# ================================================================

integrity = pd.DataFrame([{

    "n_subjects":
        n_subjects,

    "n_active":
        n_active,

    "n_passive":
        n_passive,

    "subject_roi_rows":
        len(
            subject_roi
        ),

    "expected_subject_roi_rows":
        expected_subject_roi_rows,

    "exact_permutations_per_contrast":
        expected_permutations,

    "n_primary_contrasts":
        len(
            comparison
        ),

    "n_uncorrected_significant":
        n_uncorrected,

    "n_fdr_significant":
        n_fdr,

    "n_bootstrap_ci_excluding_zero":
        n_ci,

    "n_errors":
        len(
            errors
        ),

    "overall_integrity_pass":
        overall_pass
}])


integrity.to_csv(

    OUT
    /
    "task04a_integrity_summary.csv",

    index=False
)


# ================================================================
# 29. TASK 4A FINAL REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "TASK 4A COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participants analyzed          : "
    f"{n_subjects}/15"
)


print(
    f"Active                         : "
    f"{n_active}/8"
)


print(
    f"Passive                        : "
    f"{n_passive}/7"
)


print(
    f"Primary bands                  : "
    f"{', '.join(BANDS.keys())}"
)


print(
    f"Primary ROIs                   : "
    f"{', '.join(ROIS.keys())}"
)


print(
    f"Primary contrasts              : "
    f"{len(comparison)}"
)


print(
    f"Exact permutations/contrast    : "
    f"{expected_permutations}"
)


print(
    f"Bootstrap iterations/contrast  : "
    f"{N_BOOTSTRAP}"
)


print(
    f"Uncorrected p < .05            : "
    f"{n_uncorrected}"
)


print(
    f"BH-FDR significant             : "
    f"{n_fdr}"
)


print(
    f"Bootstrap CIs excluding zero   : "
    f"{n_ci}"
)


print(
    f"Processing errors              : "
    f"{len(errors)}"
)


print(
    "\nStatistical unit:"
)


print(
    "  Participant"
)


print(
    "\nPrimary outcome:"
)


print(
    "  ln(Task band power / Baseline A band power)"
)


print(
    "\nGroup contrast:"
)


print(
    "  Active − Passive"
)


print(
    "\nPositive group difference:"
)


print(
    "  Larger baseline-normalized task change in Active"
)


print(
    "\nNegative group difference:"
)


print(
    "  Larger baseline-normalized task change in Passive"
)


print(
    "\nMultiple-comparison control:"
)


print(
    "  Benjamini-Hochberg FDR across the 9 prespecified "
    "band × ROI contrasts"
)


print(
    "\n"
    + "-" * 80
)


if overall_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )


else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "IMPORTANT INTERPRETIVE LIMIT:"
)


print(
    "These contrasts quantify differences in EEG spectral "
    "dynamics associated with the two engagement protocols."
)


print(
    "They do not directly measure learning outcomes, cognitive "
    "offloading, academic performance, or causal effects of AI."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# ================================================================
# TASK 4B — BASELINE SENSITIVITY ANALYSIS
# BASELINE A (T2_V1) VS BASELINE B (T2_V3)
# ================================================================
#
# PURPOSE
# -------
# Task 4A used Baseline A:
#
#     position 1 / T2_V1
#
# This analysis asks whether the Active-vs-Passive spectral
# conclusions depend materially on that baseline choice.
#
# We reconstruct a second matched analytical dataset using:
#
#     Baseline B:
#         position 3 / T2_V3
#
#     Task onset:
#         Active  -> position 5 / T1_V1
#         Passive -> position 5 / T1_V4
#
# We then:
#
#   1. create phase-contained 2-s epochs;
#   2. apply the same artifact thresholds as Tasks 3D/4A;
#   3. match baseline and task epoch counts within participant;
#   4. compute theta/alpha/beta power;
#   5. calculate ln(Task / Baseline B);
#   6. perform exact 6,435-permutation Active-vs-Passive tests;
#   7. calculate Hedges' g;
#   8. calculate stratified bootstrap 95% CIs;
#   9. apply BH-FDR across 9 prespecified contrasts;
#  10. compare Baseline-B results directly with Task 4A.
#
# IMPORTANT
# ---------
# T2_V1 and T2_V3 remain protocol-defined baseline markers.
# They are NOT renamed Eyes Open / Eyes Closed here.
#
# Statistical unit = participant.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
from itertools import combinations
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from scipy.stats import pearsonr, spearmanr


warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. PATHS
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK3A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03a_batch_preprocessing_qc"
)

TASK3B = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03b_phase_reconstruction"
)

TASK4A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task04a_baseline_normalized_spectral"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task04b_baseline_sensitivity"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. ANALYSIS SETTINGS
# ================================================================

EPOCH_SECONDS = 2.0

MAX_DURATION_SECONDS = 30.0

MAX_EPOCHS = 15

ABS_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0

PSD_FMIN = 1.0

PSD_FMAX = 45.0

N_BOOTSTRAP = 10000

RANDOM_SEED = 20260913


BANDS = {
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0)
}


ROIS = {

    "Frontal": [
        "AF3", "F7", "F3",
        "F4", "F8", "AF4"
    ],

    "CentralTemporal": [
        "FC5", "T7", "T8", "FC6"
    ],

    "Posterior": [
        "P7", "O1", "O2", "P8"
    ]
}


SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 16)
]


GROUPS = {

    **{
        f"sub-{i:03d}": "Active"
        for i in range(1, 9)
    },

    **{
        f"sub-{i:03d}": "Passive"
        for i in range(9, 16)
    }
}


BASELINE_POSITION = 3

BASELINE_MARKER = "T2_V3"

TASK_POSITION = 5


# ================================================================
# 4. HELPERS
# ================================================================

def integrate_band(freqs, psd, low, high):

    mask = (
        (freqs >= low)
        &
        (freqs < high)
    )

    if np.sum(mask) < 2:
        return np.full(
            psd.shape[:-1],
            np.nan
        )

    if hasattr(np, "trapezoid"):

        return np.trapezoid(
            psd[..., mask],
            freqs[mask],
            axis=-1
        )

    return np.trapz(
        psd[..., mask],
        freqs[mask],
        axis=-1
    )


def epoch_is_clean(epoch_uv):

    max_absolute = np.max(
        np.abs(epoch_uv)
    )

    max_ptp = np.max(
        np.ptp(
            epoch_uv,
            axis=1
        )
    )

    return bool(
        (max_absolute <= ABS_THRESHOLD_UV)
        and
        (max_ptp <= PTP_THRESHOLD_UV)
    )


def hedges_g(x, y):

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    nx = len(x)
    ny = len(y)

    if nx < 2 or ny < 2:
        return np.nan

    vx = np.var(x, ddof=1)
    vy = np.var(y, ddof=1)

    pooled_variance = (
        ((nx - 1) * vx)
        +
        ((ny - 1) * vy)
    ) / (
        nx + ny - 2
    )

    if pooled_variance <= 0:
        return np.nan

    d = (
        np.mean(x)
        -
        np.mean(y)
    ) / np.sqrt(
        pooled_variance
    )

    df = nx + ny - 2

    correction = (
        1
        -
        3
        /
        (
            4 * df - 1
        )
    )

    return float(
        correction * d
    )


def exact_permutation_test(active, passive):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )

    active = active[
        np.isfinite(active)
    ]

    passive = passive[
        np.isfinite(passive)
    ]

    n_active = len(active)

    combined = np.concatenate(
        [active, passive]
    )

    observed = float(
        np.mean(active)
        -
        np.mean(passive)
    )

    indices = np.arange(
        len(combined)
    )

    statistics = []

    for active_indices_tuple in combinations(
        indices,
        n_active
    ):

        active_indices = np.asarray(
            active_indices_tuple,
            dtype=int
        )

        mask = np.zeros(
            len(combined),
            dtype=bool
        )

        mask[
            active_indices
        ] = True

        statistic = (
            np.mean(combined[mask])
            -
            np.mean(combined[~mask])
        )

        statistics.append(
            statistic
        )

    statistics = np.asarray(
        statistics,
        dtype=float
    )

    p_value = float(
        np.mean(
            np.abs(statistics)
            >=
            (
                abs(observed)
                -
                1e-12
            )
        )
    )

    return (
        observed,
        p_value,
        len(statistics)
    )


def bootstrap_ci(
    active,
    passive,
    seed
):

    local_rng = np.random.default_rng(
        seed
    )

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )

    differences = np.empty(
        N_BOOTSTRAP
    )

    for i in range(
        N_BOOTSTRAP
    ):

        a = local_rng.choice(
            active,
            size=len(active),
            replace=True
        )

        p = local_rng.choice(
            passive,
            size=len(passive),
            replace=True
        )

        differences[i] = (
            np.mean(a)
            -
            np.mean(p)
        )

    return (
        float(
            np.percentile(
                differences,
                2.5
            )
        ),
        float(
            np.percentile(
                differences,
                97.5
            )
        )
    )


def bh_fdr(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    output = np.full(
        len(p_values),
        np.nan
    )

    valid = np.isfinite(
        p_values
    )

    p = p_values[
        valid
    ]

    if len(p) == 0:
        return output

    order = np.argsort(p)

    ranked = p[order]

    m = len(ranked)

    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.minimum(
        adjusted,
        1.0
    )

    restored = np.empty(
        m
    )

    restored[
        order
    ] = adjusted

    output[
        valid
    ] = restored

    return output


def find_preprocessed_fif(subject):

    subject_dir = (
        TASK3A
        /
        subject
    )

    matches = sorted(
        subject_dir.glob(
            "*desc-primary_annotated_eeg_raw.fif"
        )
    )

    if len(matches) != 1:

        raise RuntimeError(
            f"{subject}: expected exactly one "
            f"preprocessed FIF, found {len(matches)}."
        )

    return matches[0]


# ================================================================
# 5. LOAD TASK 3B PHASE TABLE
# ================================================================

phase_candidates = sorted(
    TASK3B.glob(
        "*phase*.csv"
    )
)

if len(phase_candidates) == 0:

    phase_candidates = sorted(
        TASK3B.glob(
            "*.csv"
        )
    )


if len(phase_candidates) == 0:

    raise FileNotFoundError(
        "No Task 3B phase CSV found."
    )


preferred = [
    p
    for p in phase_candidates
    if "all_subjects_marker_defined_phases"
    in p.name
]


if preferred:

    PHASE_FILE = preferred[0]

else:

    PHASE_FILE = phase_candidates[0]


phases = pd.read_csv(
    PHASE_FILE
)


required_columns = {
    "subject",
    "group",
    "marker_position",
    "marker_code",
    "start_sample",
    "end_sample"
}


missing_columns = (
    required_columns
    -
    set(phases.columns)
)


if missing_columns:

    raise RuntimeError(
        f"Phase table missing columns: "
        f"{sorted(missing_columns)}"
    )


phases[
    "subject"
] = phases[
    "subject"
].astype(str)


# Normalize subject labels if required.

def normalize_subject(value):

    value = str(value)

    if value.startswith("sub-"):
        return value

    try:
        return f"sub-{int(value):03d}"
    except Exception:
        return value


phases[
    "subject"
] = phases[
    "subject"
].apply(
    normalize_subject
)


# ================================================================
# 6. LOAD TASK 4A RESULTS
# ================================================================

TASK4A_RESULTS = (
    TASK4A
    /
    "primary_active_passive_spectral_tests.csv"
)


if not TASK4A_RESULTS.exists():

    raise FileNotFoundError(
        f"Task 4A results not found:\n"
        f"{TASK4A_RESULTS}"
    )


task4a_results = pd.read_csv(
    TASK4A_RESULTS
)


print(
    "=" * 80
)

print(
    "TASK 4B — BASELINE SENSITIVITY ANALYSIS"
)

print(
    "=" * 80
)

print(
    f"Phase table       : {PHASE_FILE}"
)

print(
    f"Task 4A results   : {TASK4A_RESULTS}"
)

print(
    f"Output directory  : {OUT}"
)

print(
    f"Baseline B        : "
    f"position {BASELINE_POSITION} / "
    f"{BASELINE_MARKER}"
)

print(
    f"Task onset        : position {TASK_POSITION}"
)


# ================================================================
# 7. EXTRACT MATCHED BASELINE-B AND TASK EPOCHS
# ================================================================

subject_epoch_data = {}

matching_rows = []

error_rows = []


for subject in SUBJECTS:

    print(
        "\n"
        + "-" * 80
    )

    print(
        f"PROCESSING {subject}"
    )

    try:

        group = GROUPS[
            subject
        ]

        expected_task_marker = (
            "T1_V1"
            if group == "Active"
            else "T1_V4"
        )

        baseline_rows = phases[
            (
                phases[
                    "subject"
                ]
                ==
                subject
            )
            &
            (
                phases[
                    "marker_position"
                ]
                ==
                BASELINE_POSITION
            )
            &
            (
                phases[
                    "marker_code"
                ]
                ==
                BASELINE_MARKER
            )
        ]


        task_rows = phases[
            (
                phases[
                    "subject"
                ]
                ==
                subject
            )
            &
            (
                phases[
                    "marker_position"
                ]
                ==
                TASK_POSITION
            )
            &
            (
                phases[
                    "marker_code"
                ]
                ==
                expected_task_marker
            )
        ]


        if len(baseline_rows) != 1:

            raise RuntimeError(
                f"Expected exactly one Baseline B row; "
                f"found {len(baseline_rows)}."
            )


        if len(task_rows) != 1:

            raise RuntimeError(
                f"Expected exactly one task-onset row; "
                f"found {len(task_rows)}."
            )


        baseline_row = (
            baseline_rows.iloc[0]
        )

        task_row = (
            task_rows.iloc[0]
        )


        fif = find_preprocessed_fif(
            subject
        )


        raw = mne.io.read_raw_fif(
            fif,
            preload=True,
            verbose=False
        )


        sfreq = float(
            raw.info[
                "sfreq"
            ]
        )


        ch_names = list(
            raw.ch_names
        )


        epoch_samples = int(
            round(
                EPOCH_SECONDS
                *
                sfreq
            )
        )


        max_samples = int(
            round(
                MAX_DURATION_SECONDS
                *
                sfreq
            )
        )


        def extract_clean_epochs(
            phase_row
        ):

            start_sample = int(
                phase_row[
                    "start_sample"
                ]
            )

            end_sample = int(
                phase_row[
                    "end_sample"
                ]
            )

            available_samples = (
                end_sample
                -
                start_sample
            )

            selected_samples = min(
                available_samples,
                max_samples
            )

            n_candidates = min(
                selected_samples
                //
                epoch_samples,
                MAX_EPOCHS
            )

            clean_epochs = []

            clean_indices = []


            for epoch_index in range(
                n_candidates
            ):

                start = (
                    start_sample
                    +
                    epoch_index
                    *
                    epoch_samples
                )

                stop = (
                    start
                    +
                    epoch_samples
                )

                if stop > end_sample:
                    break


                data_uv = (
                    raw.get_data(
                        start=start,
                        stop=stop
                    )
                    *
                    1e6
                )


                if epoch_is_clean(
                    data_uv
                ):

                    clean_epochs.append(
                        data_uv
                    )

                    clean_indices.append(
                        epoch_index
                    )


            return (
                clean_epochs,
                clean_indices,
                n_candidates
            )


        (
            baseline_clean,
            baseline_indices,
            baseline_candidates
        ) = extract_clean_epochs(
            baseline_row
        )


        (
            task_clean,
            task_indices,
            task_candidates
        ) = extract_clean_epochs(
            task_row
        )


        n_matched = min(
            len(baseline_clean),
            len(task_clean),
            MAX_EPOCHS
        )


        if n_matched < 1:

            raise RuntimeError(
                "No matched clean epochs."
            )


        baseline_selected = np.stack(
            baseline_clean[
                :n_matched
            ],
            axis=0
        )


        task_selected = np.stack(
            task_clean[
                :n_matched
            ],
            axis=0
        )


        subject_epoch_data[
            subject
        ] = {

            "group":
                group,

            "sfreq":
                sfreq,

            "ch_names":
                ch_names,

            "baseline":
                baseline_selected,

            "task":
                task_selected
        }


        matching_rows.append({

            "subject":
                subject,

            "group":
                group,

            "baseline_marker":
                BASELINE_MARKER,

            "task_marker":
                expected_task_marker,

            "baseline_candidate_epochs":
                baseline_candidates,

            "task_candidate_epochs":
                task_candidates,

            "baseline_clean_epochs":
                len(
                    baseline_clean
                ),

            "task_clean_epochs":
                len(
                    task_clean
                ),

            "matched_epochs_per_condition":
                n_matched,

            "matched_seconds_per_condition":
                n_matched
                *
                EPOCH_SECONDS
        })


        print(
            f"Group                     : {group}"
        )

        print(
            f"Baseline clean epochs     : "
            f"{len(baseline_clean)}/"
            f"{baseline_candidates}"
        )

        print(
            f"Task clean epochs         : "
            f"{len(task_clean)}/"
            f"{task_candidates}"
        )

        print(
            f"Matched epochs/condition  : "
            f"{n_matched}"
        )

        print(
            f"Matched duration/condition: "
            f"{n_matched * EPOCH_SECONDS:.1f} s"
        )

        print(
            "STATUS                    : SUCCESS"
        )


    except Exception as exc:

        print(
            f"ERROR: {exc}"
        )

        error_rows.append({

            "subject":
                subject,

            "error":
                repr(
                    exc
                )
        })


matching = pd.DataFrame(
    matching_rows
)

errors = pd.DataFrame(
    error_rows
)


matching.to_csv(
    OUT
    /
    "baseline_b_matching_summary.csv",
    index=False
)


errors.to_csv(
    OUT
    /
    "processing_errors.csv",
    index=False
)


# ================================================================
# 8. MATCHING SUMMARY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "BASELINE-B MATCHING SUMMARY"
)

print(
    "=" * 80
)


display(
    matching
)


if len(errors) > 0:

    print(
        "\nPROCESSING ERRORS"
    )

    display(
        errors
    )


if len(subject_epoch_data) != 15:

    raise RuntimeError(
        f"Expected 15 successful participants; "
        f"found {len(subject_epoch_data)}."
    )


# ================================================================
# 9. COMPUTE SUBJECT-LEVEL SPECTRAL CHANGE
# ================================================================

spectral_rows = []

channel_rows = []


for subject in SUBJECTS:

    data = subject_epoch_data[
        subject
    ]

    group = data[
        "group"
    ]

    sfreq = data[
        "sfreq"
    ]

    ch_names = data[
        "ch_names"
    ]


    for condition in [
        "baseline",
        "task"
    ]:

        epochs_uv = data[
            condition
        ]


        n_samples = epochs_uv.shape[
            -1
        ]


        nperseg = min(
            n_samples,
            int(
                round(
                    2.0
                    *
                    sfreq
                )
            )
        )


        freqs, psd = welch(

            epochs_uv,

            fs=sfreq,

            window="hann",

            nperseg=nperseg,

            noverlap=nperseg // 2,

            detrend="constant",

            scaling="density",

            axis=-1
        )


        mask = (
            (freqs >= PSD_FMIN)
            &
            (freqs <= PSD_FMAX)
        )


        if condition == "baseline":

            baseline_freqs = freqs[
                mask
            ]

            baseline_psd = psd[
                ...,
                mask
            ]

        else:

            task_freqs = freqs[
                mask
            ]

            task_psd = psd[
                ...,
                mask
            ]


    if not np.allclose(
        baseline_freqs,
        task_freqs
    ):

        raise RuntimeError(
            f"{subject}: PSD frequency mismatch."
        )


    epsilon = np.finfo(
        float
    ).tiny


    for band, (
        low,
        high
    ) in BANDS.items():

        baseline_band = integrate_band(
            baseline_freqs,
            baseline_psd,
            low,
            high
        )

        task_band = integrate_band(
            task_freqs,
            task_psd,
            low,
            high
        )


        baseline_channel = np.median(
            baseline_band,
            axis=0
        )

        task_channel = np.median(
            task_band,
            axis=0
        )


        channel_log_ratio = np.log(
            np.maximum(
                task_channel,
                epsilon
            )
            /
            np.maximum(
                baseline_channel,
                epsilon
            )
        )


        for channel_index, channel in enumerate(
            ch_names
        ):

            channel_rows.append({

                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "channel":
                    channel,

                "baseline_b_power_uV2":
                    float(
                        baseline_channel[
                            channel_index
                        ]
                    ),

                "task_power_uV2":
                    float(
                        task_channel[
                            channel_index
                        ]
                    ),

                "log_ratio_task_vs_baseline_b":
                    float(
                        channel_log_ratio[
                            channel_index
                        ]
                    )
            })


        for roi, roi_channels in ROIS.items():

            roi_indices = [
                ch_names.index(channel)
                for channel in roi_channels
            ]


            baseline_roi = float(
                np.mean(
                    baseline_channel[
                        roi_indices
                    ]
                )
            )


            task_roi = float(
                np.mean(
                    task_channel[
                        roi_indices
                    ]
                )
            )


            log_ratio = float(
                np.log(
                    max(
                        task_roi,
                        epsilon
                    )
                    /
                    max(
                        baseline_roi,
                        epsilon
                    )
                )
            )


            spectral_rows.append({

                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "roi":
                    roi,

                "baseline_b_power_uV2":
                    baseline_roi,

                "task_power_uV2":
                    task_roi,

                "log_ratio_task_vs_baseline_b":
                    log_ratio,

                "percent_change":
                    float(
                        (
                            np.exp(log_ratio)
                            -
                            1
                        )
                        *
                        100
                    ),

                "n_matched_epochs":
                    int(
                        data[
                            "baseline"
                        ].shape[0]
                    )
            })


spectral = pd.DataFrame(
    spectral_rows
)

channel_spectral = pd.DataFrame(
    channel_rows
)


spectral.to_csv(
    OUT
    /
    "subject_roi_baseline_b_spectral_change.csv",
    index=False
)


channel_spectral.to_csv(
    OUT
    /
    "subject_channel_baseline_b_spectral_change.csv",
    index=False
)


# ================================================================
# 10. BASELINE-B GROUP DESCRIPTIVES
# ================================================================

descriptive = (
    spectral

    .groupby(
        [
            "group",
            "band",
            "roi"
        ]
    )

    .agg(

        n=(
            "subject",
            "nunique"
        ),

        mean_log_ratio=(
            "log_ratio_task_vs_baseline_b",
            "mean"
        ),

        sd_log_ratio=(
            "log_ratio_task_vs_baseline_b",
            "std"
        ),

        median_log_ratio=(
            "log_ratio_task_vs_baseline_b",
            "median"
        ),

        mean_percent_change=(
            "percent_change",
            "mean"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)

print(
    "BASELINE-B GROUP DESCRIPTIVE SUMMARY"
)

print(
    "=" * 80
)


display(
    descriptive.round(
        5
    )
)


descriptive.to_csv(
    OUT
    /
    "baseline_b_group_descriptive_summary.csv",
    index=False
)


# ================================================================
# 11. ACTIVE VS PASSIVE USING BASELINE B
# ================================================================

comparison_rows = []

contrast_index = 0


for band in BANDS.keys():

    for roi in ROIS.keys():

        contrast_index += 1


        active = spectral.loc[
            (
                spectral[
                    "group"
                ]
                ==
                "Active"
            )
            &
            (
                spectral[
                    "band"
                ]
                ==
                band
            )
            &
            (
                spectral[
                    "roi"
                ]
                ==
                roi
            ),
            "log_ratio_task_vs_baseline_b"
        ].to_numpy(
            dtype=float
        )


        passive = spectral.loc[
            (
                spectral[
                    "group"
                ]
                ==
                "Passive"
            )
            &
            (
                spectral[
                    "band"
                ]
                ==
                band
            )
            &
            (
                spectral[
                    "roi"
                ]
                ==
                roi
            ),
            "log_ratio_task_vs_baseline_b"
        ].to_numpy(
            dtype=float
        )


        (
            difference,
            p_value,
            n_permutations
        ) = exact_permutation_test(
            active,
            passive
        )


        effect = hedges_g(
            active,
            passive
        )


        (
            ci_low,
            ci_high
        ) = bootstrap_ci(
            active,
            passive,
            RANDOM_SEED
            +
            contrast_index
        )


        comparison_rows.append({

            "band":
                band,

            "roi":
                roi,

            "n_active":
                len(active),

            "n_passive":
                len(passive),

            "active_mean_log_ratio":
                float(
                    np.mean(active)
                ),

            "passive_mean_log_ratio":
                float(
                    np.mean(passive)
                ),

            "active_minus_passive":
                difference,

            "bootstrap_95ci_low":
                ci_low,

            "bootstrap_95ci_high":
                ci_high,

            "hedges_g":
                effect,

            "exact_permutation_p":
                p_value,

            "n_exact_permutations":
                n_permutations
        })


baseline_b_results = pd.DataFrame(
    comparison_rows
)


baseline_b_results[
    "p_fdr"
] = bh_fdr(
    baseline_b_results[
        "exact_permutation_p"
    ].to_numpy()
)


baseline_b_results[
    "significant_fdr_0_05"
] = (
    baseline_b_results[
        "p_fdr"
    ]
    <
    0.05
)


print(
    "\n"
    + "=" * 80
)

print(
    "BASELINE-B ACTIVE VS PASSIVE RESULTS"
)

print(
    "=" * 80
)


display(
    baseline_b_results.round(
        5
    )
)


baseline_b_results.to_csv(
    OUT
    /
    "baseline_b_active_passive_spectral_tests.csv",
    index=False
)


# ================================================================
# 12. MERGE BASELINE A AND BASELINE B RESULTS
# ================================================================

a = task4a_results[
    [
        "band",
        "roi",
        "active_minus_passive",
        "hedges_g",
        "exact_permutation_p",
        "p_fdr"
    ]
].copy()


a = a.rename(
    columns={
        "active_minus_passive":
            "baseline_a_difference",

        "hedges_g":
            "baseline_a_hedges_g",

        "exact_permutation_p":
            "baseline_a_p",

        "p_fdr":
            "baseline_a_p_fdr"
    }
)


b = baseline_b_results[
    [
        "band",
        "roi",
        "active_minus_passive",
        "hedges_g",
        "exact_permutation_p",
        "p_fdr"
    ]
].copy()


b = b.rename(
    columns={
        "active_minus_passive":
            "baseline_b_difference",

        "hedges_g":
            "baseline_b_hedges_g",

        "exact_permutation_p":
            "baseline_b_p",

        "p_fdr":
            "baseline_b_p_fdr"
    }
)


sensitivity = a.merge(
    b,
    on=[
        "band",
        "roi"
    ],
    how="inner"
)


# ================================================================
# 13. DIRECTIONAL STABILITY
# ================================================================

sensitivity[
    "same_direction"
] = (
    np.sign(
        sensitivity[
            "baseline_a_difference"
        ]
    )
    ==
    np.sign(
        sensitivity[
            "baseline_b_difference"
        ]
    )
)


sensitivity[
    "effect_size_same_direction"
] = (
    np.sign(
        sensitivity[
            "baseline_a_hedges_g"
        ]
    )
    ==
    np.sign(
        sensitivity[
            "baseline_b_hedges_g"
        ]
    )
)


sensitivity[
    "absolute_effect_size_change"
] = np.abs(
    sensitivity[
        "baseline_b_hedges_g"
    ]
    -
    sensitivity[
        "baseline_a_hedges_g"
    ]
)


sensitivity[
    "relative_effect_magnitude"
] = (
    np.abs(
        sensitivity[
            "baseline_b_hedges_g"
        ]
    )
    /
    np.maximum(
        np.abs(
            sensitivity[
                "baseline_a_hedges_g"
            ]
        ),
        1e-12
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "BASELINE A VS BASELINE B SENSITIVITY"
)

print(
    "=" * 80
)


display(
    sensitivity.round(
        5
    )
)


sensitivity.to_csv(
    OUT
    /
    "baseline_a_vs_b_sensitivity.csv",
    index=False
)


# ================================================================
# 14. EFFECT-SIZE CORRELATION ACROSS BASELINES
# ================================================================

g_a = sensitivity[
    "baseline_a_hedges_g"
].to_numpy(
    dtype=float
)


g_b = sensitivity[
    "baseline_b_hedges_g"
].to_numpy(
    dtype=float
)


pearson_result = pearsonr(
    g_a,
    g_b
)


spearman_result = spearmanr(
    g_a,
    g_b
)


pearson_r = float(
    np.asarray(
        pearson_result.statistic
    ).squeeze()
)


pearson_p = float(
    np.asarray(
        pearson_result.pvalue
    ).squeeze()
)


spearman_rho = float(
    np.asarray(
        spearman_result.statistic
    ).squeeze()
)


spearman_p = float(
    np.asarray(
        spearman_result.pvalue
    ).squeeze()
)


direction_stability = float(
    sensitivity[
        "same_direction"
    ].mean()
)


print(
    "\n"
    + "=" * 80
)

print(
    "CROSS-BASELINE EFFECT STABILITY"
)

print(
    "=" * 80
)


print(
    f"Same-direction contrasts      : "
    f"{int(sensitivity['same_direction'].sum())}/"
    f"{len(sensitivity)}"
)


print(
    f"Directional stability         : "
    f"{100 * direction_stability:.1f}%"
)


print(
    f"Pearson r                     : "
    f"{pearson_r:.4f}"
)


print(
    f"Pearson p                     : "
    f"{pearson_p:.6f}"
)


print(
    f"Spearman rho                  : "
    f"{spearman_rho:.4f}"
)


print(
    f"Spearman p                    : "
    f"{spearman_p:.6f}"
)


# ================================================================
# 15. FOCUSED CONTRASTS FROM TASK 4A
# ================================================================

focus_pairs = [

    (
        "Beta",
        "CentralTemporal"
    ),

    (
        "Alpha",
        "Posterior"
    ),

    (
        "Theta",
        "CentralTemporal"
    )
]


focus_rows = []


for band, roi in focus_pairs:

    row = sensitivity[
        (
            sensitivity[
                "band"
            ]
            ==
            band
        )
        &
        (
            sensitivity[
                "roi"
            ]
            ==
            roi
        )
    ].iloc[0]


    focus_rows.append(
        row.to_dict()
    )


focus = pd.DataFrame(
    focus_rows
)


print(
    "\n"
    + "=" * 80
)

print(
    "FOCUSED TASK 4A EFFECTS UNDER BOTH BASELINES"
)

print(
    "=" * 80
)


display(
    focus[
        [
            "band",
            "roi",
            "baseline_a_difference",
            "baseline_b_difference",
            "baseline_a_hedges_g",
            "baseline_b_hedges_g",
            "baseline_a_p",
            "baseline_b_p",
            "same_direction"
        ]
    ].round(
        5
    )
)


focus.to_csv(
    OUT
    /
    "focused_effect_baseline_sensitivity.csv",
    index=False
)


# ================================================================
# 16. FIGURE — HEDGES' g BASELINE A VS BASELINE B
# ================================================================

plot_df = sensitivity.copy()


plot_df[
    "contrast"
] = (
    plot_df[
        "band"
    ]
    +
    " — "
    +
    plot_df[
        "roi"
    ]
)


fig, ax = plt.subplots(
    figsize=(
        10,
        7
    )
)


y = np.arange(
    len(plot_df)
)


for i, row in plot_df.iterrows():

    ax.plot(
        [
            row[
                "baseline_a_hedges_g"
            ],
            row[
                "baseline_b_hedges_g"
            ]
        ],
        [
            i,
            i
        ],
        linewidth=1.5
    )


ax.scatter(
    plot_df[
        "baseline_a_hedges_g"
    ],
    y,
    s=70,
    marker="o",
    label="Baseline A"
)


ax.scatter(
    plot_df[
        "baseline_b_hedges_g"
    ],
    y,
    s=70,
    marker="s",
    label="Baseline B"
)


ax.axvline(
    0,
    linewidth=1
)


ax.set_yticks(
    y
)


ax.set_yticklabels(
    plot_df[
        "contrast"
    ]
)


ax.set_xlabel(
    "Hedges' g (Active − Passive)"
)


ax.set_ylabel(
    "Spectral contrast"
)


ax.set_title(
    "Sensitivity of Active–Passive effect sizes to baseline choice"
)


ax.legend()


ax.grid(
    axis="x",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    /
    "01_baseline_a_vs_b_effect_sizes.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 17. FIGURE — EFFECT-SIZE CORRELATION
# ================================================================

fig, ax = plt.subplots(
    figsize=(
        7,
        7
    )
)


ax.scatter(
    sensitivity[
        "baseline_a_hedges_g"
    ],
    sensitivity[
        "baseline_b_hedges_g"
    ],
    s=80
)


for _, row in sensitivity.iterrows():

    ax.annotate(
        (
            f"{row['band']} "
            f"{row['roi']}"
        ),
        (
            row[
                "baseline_a_hedges_g"
            ],
            row[
                "baseline_b_hedges_g"
            ]
        ),
        xytext=(
            5,
            5
        ),
        textcoords="offset points",
        fontsize=8
    )


limits = [

    min(
        sensitivity[
            "baseline_a_hedges_g"
        ].min(),
        sensitivity[
            "baseline_b_hedges_g"
        ].min()
    ),

    max(
        sensitivity[
            "baseline_a_hedges_g"
        ].max(),
        sensitivity[
            "baseline_b_hedges_g"
        ].max()
    )
]


padding = (
    (
        limits[1]
        -
        limits[0]
    )
    *
    0.1
)


if padding == 0:
    padding = 0.1


lower = limits[0] - padding

upper = limits[1] + padding


ax.plot(
    [
        lower,
        upper
    ],
    [
        lower,
        upper
    ],
    linestyle="--",
    linewidth=1
)


ax.axhline(
    0,
    linewidth=0.8
)


ax.axvline(
    0,
    linewidth=0.8
)


ax.set_xlim(
    lower,
    upper
)


ax.set_ylim(
    lower,
    upper
)


ax.set_xlabel(
    "Hedges' g using Baseline A"
)


ax.set_ylabel(
    "Hedges' g using Baseline B"
)


ax.set_title(
    "Cross-baseline stability of spectral effect sizes"
)


ax.grid(
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    /
    "02_cross_baseline_effect_correlation.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 18. FIGURE — FOCUSED CONTRASTS
# ================================================================

focus_plot = focus.copy()


focus_plot[
    "contrast"
] = (
    focus_plot[
        "band"
    ]
    +
    " — "
    +
    focus_plot[
        "roi"
    ]
)


x = np.arange(
    len(focus_plot)
)


width = 0.34


fig, ax = plt.subplots(
    figsize=(
        9,
        6
    )
)


ax.bar(
    x - width / 2,
    focus_plot[
        "baseline_a_hedges_g"
    ],
    width,
    label="Baseline A"
)


ax.bar(
    x + width / 2,
    focus_plot[
        "baseline_b_hedges_g"
    ],
    width,
    label="Baseline B"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    focus_plot[
        "contrast"
    ],
    rotation=15,
    ha="right"
)


ax.set_ylabel(
    "Hedges' g (Active − Passive)"
)


ax.set_xlabel(
    "Contrast"
)


ax.set_title(
    "Baseline sensitivity of the strongest Task 4A effects"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(
    OUT
    /
    "03_focused_effect_baseline_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# ================================================================
# 19. BASELINE-B MATCHING BY GROUP
# ================================================================

matching_group_summary = (

    matching

    .groupby(
        "group"
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_matched_epochs=(
            "matched_epochs_per_condition",
            "mean"
        ),

        sd_matched_epochs=(
            "matched_epochs_per_condition",
            "std"
        ),

        median_matched_epochs=(
            "matched_epochs_per_condition",
            "median"
        ),

        min_matched_epochs=(
            "matched_epochs_per_condition",
            "min"
        ),

        max_matched_epochs=(
            "matched_epochs_per_condition",
            "max"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)

print(
    "BASELINE-B MATCHING BY GROUP"
)

print(
    "=" * 80
)


display(
    matching_group_summary.round(
        3
    )
)


matching_group_summary.to_csv(
    OUT
    /
    "baseline_b_matching_group_summary.csv",
    index=False
)


# ================================================================
# 20. AUTOMATIC SENSITIVITY INTERPRETATION
# ================================================================

n_same_direction = int(
    sensitivity[
        "same_direction"
    ].sum()
)


n_b_fdr = int(
    baseline_b_results[
        "significant_fdr_0_05"
    ].sum()
)


focus_same = int(
    focus[
        "same_direction"
    ].sum()
)


print(
    "\n"
    + "=" * 80
)

print(
    "SENSITIVITY INTERPRETATION"
)

print(
    "=" * 80
)


print(
    f"All contrasts same direction : "
    f"{n_same_direction}/9"
)


print(
    f"Focused effects same direction: "
    f"{focus_same}/3"
)


print(
    f"Baseline-B FDR significant    : "
    f"{n_b_fdr}/9"
)


print(
    f"Effect-size Pearson r         : "
    f"{pearson_r:.3f}"
)


print(
    f"Effect-size Spearman rho      : "
    f"{spearman_rho:.3f}"
)


if (
    n_same_direction >= 7
    and
    focus_same == 3
):

    print(
        "\nThe principal spectral effect pattern is "
        "directionally robust to baseline choice."
    )

    print(
        "This supports interpreting the Task 4A effect-size "
        "pattern as task-related rather than as an obvious "
        "artifact of selecting Baseline A."
    )

elif focus_same == 3:

    print(
        "\nThe three strongest Task 4A effects retain their "
        "direction under Baseline B, although some weaker "
        "contrasts are baseline-sensitive."
    )

    print(
        "The focused effect pattern is therefore more stable "
        "than the complete nine-contrast profile."
    )

else:

    print(
        "\nOne or more of the strongest Task 4A effects changes "
        "direction when Baseline B is used."
    )

    print(
        "The spectral findings should therefore be described "
        "as baseline-sensitive, and both normalizations should "
        "be reported."
    )


# ================================================================
# 21. SAVE STABILITY SUMMARY
# ================================================================

stability_summary = pd.DataFrame([{

    "n_subjects":
        len(
            subject_epoch_data
        ),

    "n_active":
        sum(
            1
            for s in subject_epoch_data
            if GROUPS[s] == "Active"
        ),

    "n_passive":
        sum(
            1
            for s in subject_epoch_data
            if GROUPS[s] == "Passive"
        ),

    "n_contrasts":
        len(
            sensitivity
        ),

    "same_direction_contrasts":
        n_same_direction,

    "directional_stability_percent":
        100
        *
        direction_stability,

    "focused_same_direction":
        focus_same,

    "pearson_r":
        pearson_r,

    "pearson_p":
        pearson_p,

    "spearman_rho":
        spearman_rho,

    "spearman_p":
        spearman_p,

    "baseline_b_fdr_significant":
        n_b_fdr,

    "processing_errors":
        len(
            errors
        )
}])


stability_summary.to_csv(
    OUT
    /
    "baseline_sensitivity_summary.csv",
    index=False
)


# ================================================================
# 22. FINAL INTEGRITY CHECK
# ================================================================

participant_valid = (
    len(
        subject_epoch_data
    )
    ==
    15
)


group_valid = (
    sum(
        GROUPS[s] == "Active"
        for s in subject_epoch_data
    )
    ==
    8

    and

    sum(
        GROUPS[s] == "Passive"
        for s in subject_epoch_data
    )
    ==
    7
)


contrast_valid = (
    len(
        baseline_b_results
    )
    ==
    9
)


permutation_valid = bool(
    np.all(
        baseline_b_results[
            "n_exact_permutations"
        ]
        ==
        6435
    )
)


error_valid = (
    len(errors)
    ==
    0
)


overall_pass = bool(
    participant_valid
    and
    group_valid
    and
    contrast_valid
    and
    permutation_valid
    and
    error_valid
)


# ================================================================
# 23. TASK 4B FINAL REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "TASK 4B COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participants analyzed          : "
    f"{len(subject_epoch_data)}/15"
)


print(
    f"Active                         : "
    f"{sum(GROUPS[s] == 'Active' for s in subject_epoch_data)}/8"
)


print(
    f"Passive                        : "
    f"{sum(GROUPS[s] == 'Passive' for s in subject_epoch_data)}/7"
)


print(
    f"Alternative baseline           : "
    f"position {BASELINE_POSITION} / "
    f"{BASELINE_MARKER}"
)


print(
    f"Primary contrasts repeated     : "
    f"{len(baseline_b_results)}"
)


print(
    f"Exact permutations/contrast    : "
    f"6435"
)


print(
    f"Same-direction contrasts       : "
    f"{n_same_direction}/9"
)


print(
    f"Focused same-direction effects : "
    f"{focus_same}/3"
)


print(
    f"Effect-size Pearson r          : "
    f"{pearson_r:.4f}"
)


print(
    f"Effect-size Spearman rho       : "
    f"{spearman_rho:.4f}"
)


print(
    f"Baseline-B FDR significant     : "
    f"{n_b_fdr}"
)


print(
    f"Processing errors              : "
    f"{len(errors)}"
)


print(
    "\n"
    + "-" * 80
)


if overall_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )

else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "INTERPRETIVE RULE:"
)


print(
    "Baseline sensitivity is evaluated from direction, "
    "effect-size stability, exact permutation inference, "
    "and uncertainty—not from whether an alternative "
    "baseline happens to produce p < .05."
)


print(
    "\nNo Eyes Open / Eyes Closed semantic label has been "
    "assigned to T2_V1 or T2_V3."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# ================================================================
# TASK 5A — dwPLI FUNCTIONAL CONNECTIVITY CONSTRUCTION AND QC
# ================================================================
#
# PURPOSE
# -------
# Construct participant-level functional connectivity matrices for:
#
#       Baseline A = position 1 / T2_V1
#       Task       = position 5
#
# Task marker:
#
#       Active  -> T1_V1
#       Passive -> T1_V4
#
# Connectivity metric:
#
#       debiased weighted phase-lag index (dwPLI)
#
# Frequency bands:
#
#       Theta : 4–8 Hz
#       Alpha : 8–13 Hz
#       Beta  : 13–30 Hz
#
# PRIMARY CONNECTIVITY CHANGE:
#
#       Delta dwPLI = Task dwPLI - Baseline dwPLI
#
# IMPORTANT
# ---------
# This is a CONSTRUCTION + QC task.
#
# NO Active-vs-Passive hypothesis tests are performed here.
# NO edge selection is performed here.
# NO network threshold is optimized here.
#
# Statistical inference will occur only after connectivity
# construction has been independently validated.
#
# The statistical unit remains the participant.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import hilbert


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. PATHS
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)


TASK3D = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03d_matched_baseline_task_epochs"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05a_dwpli_connectivity_qc"
)


MATRIX_OUT = (
    OUT
    / "matrices"
)


FIGURE_OUT = (
    OUT
    / "figures"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


MATRIX_OUT.mkdir(
    parents=True,
    exist_ok=True
)


FIGURE_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. ANALYSIS SETTINGS
# ================================================================

BANDS = {

    "Theta":
        (4.0, 8.0),

    "Alpha":
        (8.0, 13.0),

    "Beta":
        (13.0, 30.0)
}


EXPECTED_CHANNELS = [

    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


EXPECTED_GROUP_COUNTS = {

    "Active":
        8,

    "Passive":
        7
}


RANDOM_SEED = 20260913


# ================================================================
# 4. HELPER — BANDPASS FILTER EPOCH DATA
# ================================================================

def bandpass_epoch_data(
    data,
    sfreq,
    low,
    high
):

    """
    data:
        epochs × channels × samples

    Returns:
        filtered data with identical shape.
    """

    filtered = mne.filter.filter_data(

        data,

        sfreq=sfreq,

        l_freq=low,

        h_freq=high,

        method="fir",

        phase="zero",

        fir_design="firwin",

        pad="reflect_limited",

        verbose=False
    )


    return filtered


# ================================================================
# 5. HELPER — DEBIASED WPLI
# ================================================================

def debiased_wpli_from_epochs(
    epoch_data
):

    """
    Compute debiased weighted phase-lag index.

    Parameters
    ----------
    epoch_data:
        ndarray
        Shape:
            epochs × channels × samples

        Data must already be band-pass filtered.

    Method
    ------
    1. Hilbert transform each channel.
    2. For every channel pair, calculate the imaginary
       component of the cross-spectrum at each time sample.
    3. Pool observations across epochs and time.
    4. Apply the debiased squared weighted PLI estimator:

       dwPLI =
       [(sum Im)^2 - sum(Im^2)]
       /
       [(sum |Im|)^2 - sum(Im^2)]

    Notes
    -----
    The estimator can occasionally become slightly negative
    because it is bias-corrected.

    We therefore DO NOT clip negative values during computation.

    The diagonal is explicitly set to zero.
    """

    epoch_data = np.asarray(
        epoch_data,
        dtype=float
    )


    if epoch_data.ndim != 3:

        raise ValueError(
            "epoch_data must have shape "
            "epochs × channels × samples."
        )


    n_epochs = epoch_data.shape[0]

    n_channels = epoch_data.shape[1]

    n_samples = epoch_data.shape[2]


    if n_epochs < 1:

        raise ValueError(
            "At least one epoch is required."
        )


    if n_samples < 2:

        raise ValueError(
            "Epochs contain too few samples."
        )


    # ------------------------------------------------------------
    # Analytic signal
    # ------------------------------------------------------------

    analytic = hilbert(

        epoch_data,

        axis=-1
    )


    matrix = np.zeros(

        (
            n_channels,
            n_channels
        ),

        dtype=float
    )


    # ------------------------------------------------------------
    # Pairwise dwPLI
    # ------------------------------------------------------------

    for i in range(
        n_channels
    ):

        zi = analytic[
            :,
            i,
            :
        ]


        for j in range(
            i + 1,
            n_channels
        ):

            zj = analytic[
                :,
                j,
                :
            ]


            # Cross-spectrum imaginary component.
            #
            # zi * conjugate(zj)

            imaginary_cross = np.imag(

                zi

                *

                np.conjugate(
                    zj
                )
            )


            values = imaginary_cross.reshape(
                -1
            )


            values = values[
                np.isfinite(
                    values
                )
            ]


            if len(values) < 2:

                dwpli = np.nan

            else:

                sum_im = np.sum(
                    values
                )


                sum_abs_im = np.sum(
                    np.abs(
                        values
                    )
                )


                sum_im_squared = np.sum(
                    values ** 2
                )


                numerator = (

                    sum_im ** 2

                    -

                    sum_im_squared
                )


                denominator = (

                    sum_abs_im ** 2

                    -

                    sum_im_squared
                )


                if (
                    not np.isfinite(
                        denominator
                    )

                    or

                    denominator <= 0
                ):

                    dwpli = np.nan

                else:

                    dwpli = (

                        numerator

                        /

                        denominator
                    )


            matrix[
                i,
                j
            ] = dwpli


            matrix[
                j,
                i
            ] = dwpli


    np.fill_diagonal(
        matrix,
        0.0
    )


    return matrix


# ================================================================
# 6. HELPER — MATRIX QC
# ================================================================

def matrix_qc(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    n_channels = matrix.shape[0]


    upper = matrix[
        np.triu_indices(
            n_channels,
            k=1
        )
    ]


    finite_upper = upper[
        np.isfinite(
            upper
        )
    ]


    symmetry_error = float(

        np.nanmax(

            np.abs(

                matrix

                -

                matrix.T
            )
        )
    )


    diagonal_error = float(

        np.nanmax(

            np.abs(

                np.diag(
                    matrix
                )
            )
        )
    )


    if len(
        finite_upper
    ) > 0:

        edge_mean = float(
            np.mean(
                finite_upper
            )
        )


        edge_median = float(
            np.median(
                finite_upper
            )
        )


        edge_sd = float(
            np.std(
                finite_upper,
                ddof=1
            )
        )


        edge_min = float(
            np.min(
                finite_upper
            )
        )


        edge_max = float(
            np.max(
                finite_upper
            )
        )


        negative_fraction = float(

            np.mean(
                finite_upper
                <
                0
            )
        )


        above_one_fraction = float(

            np.mean(
                finite_upper
                >
                1
            )
        )


    else:

        edge_mean = np.nan

        edge_median = np.nan

        edge_sd = np.nan

        edge_min = np.nan

        edge_max = np.nan

        negative_fraction = np.nan

        above_one_fraction = np.nan


    expected_edges = (

        n_channels

        *

        (
            n_channels
            -
            1
        )

        //
        2
    )


    finite_edges = int(
        np.sum(
            np.isfinite(
                upper
            )
        )
    )


    return {

        "n_channels":
            n_channels,

        "expected_edges":
            expected_edges,

        "finite_edges":
            finite_edges,

        "missing_edges":
            expected_edges
            -
            finite_edges,

        "edge_mean":
            edge_mean,

        "edge_median":
            edge_median,

        "edge_sd":
            edge_sd,

        "edge_min":
            edge_min,

        "edge_max":
            edge_max,

        "negative_edge_fraction":
            negative_fraction,

        "above_one_edge_fraction":
            above_one_fraction,

        "symmetry_error":
            symmetry_error,

        "diagonal_error":
            diagonal_error
    }


# ================================================================
# 7. HELPER — SAVE MATRIX
# ================================================================

def save_matrix_csv(
    matrix,
    channels,
    path
):

    dataframe = pd.DataFrame(

        matrix,

        index=channels,

        columns=channels
    )


    dataframe.to_csv(
        path
    )


# ================================================================
# 8. HELPER — UPPER TRIANGLE
# ================================================================

def upper_triangle_values(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    return matrix[
        np.triu_indices(
            matrix.shape[0],
            k=1
        )
    ]


# ================================================================
# 9. FIND TASK 3D MATCHED EPOCH FILES
# ================================================================

epoch_files = sorted(

    TASK3D.glob(

        "sub-*/"
        "*desc-matched_baseline_task_epo.fif"
    )
)


print(
    "=" * 80
)

print(
    "TASK 5A — dwPLI FUNCTIONAL CONNECTIVITY CONSTRUCTION AND QC"
)

print(
    "=" * 80
)


print(
    f"Task 3D directory : {TASK3D}"
)


print(
    f"Output directory  : {OUT}"
)


print(
    f"Epoch files found : {len(epoch_files)}"
)


if len(
    epoch_files
) != 15:

    raise RuntimeError(

        f"Expected 15 Task 3D epoch files; "
        f"found {len(epoch_files)}."
    )


# ================================================================
# 10. PROCESS PARTICIPANTS
# ================================================================

qc_rows = []

edge_rows = []

subject_summary_rows = []

error_rows = []


all_matrices = {}


reference_channels = None


for epoch_file in epoch_files:

    print(
        "\n"
        + "-" * 80
    )


    try:

        # --------------------------------------------------------
        # Load epochs
        # --------------------------------------------------------

        epochs = mne.read_epochs(

            epoch_file,

            preload=True,

            verbose=False
        )


        metadata = epochs.metadata


        if metadata is None:

            raise RuntimeError(
                "Epoch metadata missing."
            )


        subjects = (
            metadata[
                "subject"
            ]
            .astype(str)
            .unique()
        )


        groups = (
            metadata[
                "group"
            ]
            .astype(str)
            .unique()
        )


        if len(
            subjects
        ) != 1:

            raise RuntimeError(
                f"Expected one subject, found {subjects}."
            )


        if len(
            groups
        ) != 1:

            raise RuntimeError(
                f"Expected one group, found {groups}."
            )


        subject = str(
            subjects[0]
        )


        group = str(
            groups[0]
        )


        print(
            f"{subject} | {group}"
        )


        if group not in EXPECTED_GROUP_COUNTS:

            raise RuntimeError(
                f"Unexpected group: {group}"
            )


        # --------------------------------------------------------
        # Channel verification
        # --------------------------------------------------------

        channels = list(
            epochs.ch_names
        )


        if set(
            channels
        ) != set(
            EXPECTED_CHANNELS
        ):

            raise RuntimeError(

                f"{subject}: unexpected channel set.\n"
                f"Observed: {channels}"
            )


        # Reorder identically across subjects.

        epochs.reorder_channels(
            EXPECTED_CHANNELS
        )


        channels = list(
            epochs.ch_names
        )


        if reference_channels is None:

            reference_channels = (
                channels.copy()
            )


        elif channels != reference_channels:

            raise RuntimeError(
                f"{subject}: channel order mismatch."
            )


        sfreq = float(
            epochs.info[
                "sfreq"
            ]
        )


        # --------------------------------------------------------
        # Extract matched conditions
        # --------------------------------------------------------

        baseline_epochs = epochs[
            "baseline"
        ]


        task_epochs = epochs[
            "task"
        ]


        n_baseline = len(
            baseline_epochs
        )


        n_task = len(
            task_epochs
        )


        if n_baseline != n_task:

            raise RuntimeError(

                f"{subject}: baseline/task epoch "
                f"count mismatch "
                f"({n_baseline} vs {n_task})."
            )


        if n_baseline < 1:

            raise RuntimeError(
                f"{subject}: no matched epochs."
            )


        baseline_data = (
            baseline_epochs.get_data()
        )


        task_data = (
            task_epochs.get_data()
        )


        if (
            baseline_data.shape
            !=
            task_data.shape
        ):

            raise RuntimeError(

                f"{subject}: baseline/task "
                f"shape mismatch."
            )


        print(
            f"  epochs/condition : {n_baseline}"
        )


        print(
            f"  sfreq            : {sfreq:.1f} Hz"
        )


        print(
            f"  channels         : {len(channels)}"
        )


        # --------------------------------------------------------
        # Subject dictionary
        # --------------------------------------------------------

        all_matrices[
            subject
        ] = {

            "group":
                group,

            "n_epochs":
                n_baseline,

            "bands":
                {}
        }


        # ========================================================
        # BAND LOOP
        # ========================================================

        for band, (
            low,
            high
        ) in BANDS.items():

            print(
                f"  {band:<6s} "
                f"{low:.1f}-{high:.1f} Hz"
            )


            # ----------------------------------------------------
            # Band-pass filter baseline
            # ----------------------------------------------------

            baseline_filtered = bandpass_epoch_data(

                baseline_data,

                sfreq,

                low,

                high
            )


            # ----------------------------------------------------
            # Band-pass filter task
            # ----------------------------------------------------

            task_filtered = bandpass_epoch_data(

                task_data,

                sfreq,

                low,

                high
            )


            # ----------------------------------------------------
            # Compute dwPLI
            # ----------------------------------------------------

            baseline_matrix = debiased_wpli_from_epochs(

                baseline_filtered
            )


            task_matrix = debiased_wpli_from_epochs(

                task_filtered
            )


            delta_matrix = (

                task_matrix

                -

                baseline_matrix
            )


            np.fill_diagonal(
                delta_matrix,
                0.0
            )


            # ----------------------------------------------------
            # Save matrices
            # ----------------------------------------------------

            band_slug = band.lower()


            subject_matrix_dir = (

                MATRIX_OUT
                /
                subject
            )


            subject_matrix_dir.mkdir(

                parents=True,

                exist_ok=True
            )


            np.save(

                subject_matrix_dir
                /
                (
                    f"{subject}_{band_slug}_"
                    "baseline_dwpli.npy"
                ),

                baseline_matrix
            )


            np.save(

                subject_matrix_dir
                /
                (
                    f"{subject}_{band_slug}_"
                    "task_dwpli.npy"
                ),

                task_matrix
            )


            np.save(

                subject_matrix_dir
                /
                (
                    f"{subject}_{band_slug}_"
                    "delta_dwpli.npy"
                ),

                delta_matrix
            )


            save_matrix_csv(

                baseline_matrix,

                channels,

                subject_matrix_dir
                /
                (
                    f"{subject}_{band_slug}_"
                    "baseline_dwpli.csv"
                )
            )


            save_matrix_csv(

                task_matrix,

                channels,

                subject_matrix_dir
                /
                (
                    f"{subject}_{band_slug}_"
                    "task_dwpli.csv"
                )
            )


            save_matrix_csv(

                delta_matrix,

                channels,

                subject_matrix_dir
                /
                (
                    f"{subject}_{band_slug}_"
                    "delta_dwpli.csv"
                )
            )


            # ----------------------------------------------------
            # QC
            # ----------------------------------------------------

            baseline_qc = matrix_qc(
                baseline_matrix
            )


            task_qc = matrix_qc(
                task_matrix
            )


            delta_qc = matrix_qc(
                delta_matrix
            )


            for condition, qc in [

                (
                    "baseline",
                    baseline_qc
                ),

                (
                    "task",
                    task_qc
                ),

                (
                    "delta",
                    delta_qc
                )
            ]:

                qc_rows.append({

                    "subject":
                        subject,

                    "group":
                        group,

                    "band":
                        band,

                    "condition":
                        condition,

                    "n_epochs":
                        n_baseline,

                    **qc
                })


            # ----------------------------------------------------
            # Save participant matrices in memory
            # ----------------------------------------------------

            all_matrices[
                subject
            ][
                "bands"
            ][
                band
            ] = {

                "baseline":
                    baseline_matrix,

                "task":
                    task_matrix,

                "delta":
                    delta_matrix
            }


            # ----------------------------------------------------
            # Edge-level long table
            # ----------------------------------------------------

            for i in range(
                len(channels)
            ):

                for j in range(
                    i + 1,
                    len(channels)
                ):

                    edge_rows.append({

                        "subject":
                            subject,

                        "group":
                            group,

                        "band":
                            band,

                        "channel_1":
                            channels[i],

                        "channel_2":
                            channels[j],

                        "baseline_dwpli":
                            float(
                                baseline_matrix[
                                    i,
                                    j
                                ]
                            ),

                        "task_dwpli":
                            float(
                                task_matrix[
                                    i,
                                    j
                                ]
                            ),

                        "delta_dwpli":
                            float(
                                delta_matrix[
                                    i,
                                    j
                                ]
                            )
                    })


            # ----------------------------------------------------
            # Subject-level global connectivity summary
            # ----------------------------------------------------

            baseline_edges = upper_triangle_values(
                baseline_matrix
            )


            task_edges = upper_triangle_values(
                task_matrix
            )


            delta_edges = upper_triangle_values(
                delta_matrix
            )


            subject_summary_rows.append({

                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "n_epochs":
                    n_baseline,

                "baseline_mean_dwpli":
                    float(
                        np.nanmean(
                            baseline_edges
                        )
                    ),

                "baseline_median_dwpli":
                    float(
                        np.nanmedian(
                            baseline_edges
                        )
                    ),

                "task_mean_dwpli":
                    float(
                        np.nanmean(
                            task_edges
                        )
                    ),

                "task_median_dwpli":
                    float(
                        np.nanmedian(
                            task_edges
                        )
                    ),

                "delta_mean_dwpli":
                    float(
                        np.nanmean(
                            delta_edges
                        )
                    ),

                "delta_median_dwpli":
                    float(
                        np.nanmedian(
                            delta_edges
                        )
                    ),

                "delta_positive_edge_fraction":
                    float(
                        np.nanmean(
                            delta_edges
                            >
                            0
                        )
                    ),

                "delta_negative_edge_fraction":
                    float(
                        np.nanmean(
                            delta_edges
                            <
                            0
                        )
                    )
            })


        print(
            "  connectivity     : SUCCESS"
        )


    except Exception as exc:

        print(
            f"ERROR: {exc}"
        )


        error_rows.append({

            "file":
                str(
                    epoch_file
                ),

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 11. CREATE DATA TABLES
# ================================================================

qc = pd.DataFrame(
    qc_rows
)


edges = pd.DataFrame(
    edge_rows
)


subject_summary = pd.DataFrame(
    subject_summary_rows
)


errors = pd.DataFrame(
    error_rows
)


# ================================================================
# 12. SAVE TABLES
# ================================================================

qc.to_csv(

    OUT
    /
    "dwpli_matrix_qc.csv",

    index=False
)


edges.to_csv(

    OUT
    /
    "dwpli_edge_level_long.csv",

    index=False
)


subject_summary.to_csv(

    OUT
    /
    "subject_global_dwpli_summary.csv",

    index=False
)


errors.to_csv(

    OUT
    /
    "processing_errors.csv",

    index=False
)


# ================================================================
# 13. PROCESSING SUMMARY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PROCESSING SUMMARY"
)

print(
    "=" * 80
)


n_subjects = int(
    subject_summary[
        "subject"
    ].nunique()
)


n_active = int(

    subject_summary.loc[
        subject_summary[
            "group"
        ]
        ==
        "Active",
        "subject"
    ].nunique()
)


n_passive = int(

    subject_summary.loc[
        subject_summary[
            "group"
        ]
        ==
        "Passive",
        "subject"
    ].nunique()
)


print(
    f"Participants processed : "
    f"{n_subjects}/15"
)


print(
    f"Active                 : "
    f"{n_active}/8"
)


print(
    f"Passive                : "
    f"{n_passive}/7"
)


print(
    f"Processing errors      : "
    f"{len(errors)}"
)


if len(
    errors
) > 0:

    print(
        "\nERROR TABLE"
    )

    display(
        errors
    )


# ================================================================
# 14. MATRIX QC SUMMARY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "dwPLI MATRIX QC SUMMARY"
)

print(
    "=" * 80
)


qc_summary = (

    qc

    .groupby(
        [
            "band",
            "condition"
        ]
    )

    .agg(

        matrices=(
            "subject",
            "count"
        ),

        mean_edges=(
            "edge_mean",
            "mean"
        ),

        median_edges=(
            "edge_median",
            "median"
        ),

        minimum_edge=(
            "edge_min",
            "min"
        ),

        maximum_edge=(
            "edge_max",
            "max"
        ),

        mean_negative_fraction=(
            "negative_edge_fraction",
            "mean"
        ),

        max_missing_edges=(
            "missing_edges",
            "max"
        ),

        max_symmetry_error=(
            "symmetry_error",
            "max"
        ),

        max_diagonal_error=(
            "diagonal_error",
            "max"
        )
    )

    .reset_index()
)


display(
    qc_summary.round(
        6
    )
)


qc_summary.to_csv(

    OUT
    /
    "dwpli_matrix_qc_summary.csv",

    index=False
)


# ================================================================
# 15. SUBJECT-LEVEL GLOBAL CONNECTIVITY
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "SUBJECT-LEVEL GLOBAL dwPLI SUMMARY"
)

print(
    "=" * 80
)


display(

    subject_summary[
        [
            "subject",
            "group",
            "band",
            "n_epochs",
            "baseline_mean_dwpli",
            "task_mean_dwpli",
            "delta_mean_dwpli",
            "delta_positive_edge_fraction",
            "delta_negative_edge_fraction"
        ]
    ].round(
        5
    )
)


# ================================================================
# 16. DESCRIPTIVE GROUP SUMMARY
# ================================================================

group_summary = (

    subject_summary

    .groupby(
        [
            "group",
            "band"
        ]
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_baseline_dwpli=(
            "baseline_mean_dwpli",
            "mean"
        ),

        sd_baseline_dwpli=(
            "baseline_mean_dwpli",
            "std"
        ),

        mean_task_dwpli=(
            "task_mean_dwpli",
            "mean"
        ),

        sd_task_dwpli=(
            "task_mean_dwpli",
            "std"
        ),

        mean_delta_dwpli=(
            "delta_mean_dwpli",
            "mean"
        ),

        sd_delta_dwpli=(
            "delta_mean_dwpli",
            "std"
        ),

        median_delta_dwpli=(
            "delta_mean_dwpli",
            "median"
        ),

        mean_positive_edge_fraction=(
            "delta_positive_edge_fraction",
            "mean"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)

print(
    "DESCRIPTIVE GROUP CONNECTIVITY SUMMARY"
)

print(
    "=" * 80
)


display(
    group_summary.round(
        5
    )
)


group_summary.to_csv(

    OUT
    /
    "descriptive_group_dwpli_summary.csv",

    index=False
)


# ================================================================
# 17. GROUP-AVERAGE MATRICES
# ================================================================

group_matrix_rows = []


for group in [
    "Active",
    "Passive"
]:

    group_subjects = [

        subject

        for subject in all_matrices

        if all_matrices[
            subject
        ][
            "group"
        ]
        ==
        group
    ]


    for band in BANDS.keys():

        for condition in [
            "baseline",
            "task",
            "delta"
        ]:

            stack = np.stack(

                [

                    all_matrices[
                        subject
                    ][
                        "bands"
                    ][
                        band
                    ][
                        condition
                    ]

                    for subject in group_subjects

                ],

                axis=0
            )


            mean_matrix = np.nanmean(

                stack,

                axis=0
            )


            np.fill_diagonal(
                mean_matrix,
                0.0
            )


            filename = (

                MATRIX_OUT
                /
                (
                    f"group-{group.lower()}_"
                    f"{band.lower()}_"
                    f"{condition}_"
                    "mean_dwpli.csv"
                )
            )


            save_matrix_csv(

                mean_matrix,

                reference_channels,

                filename
            )


            group_matrix_rows.append({

                "group":
                    group,

                "band":
                    band,

                "condition":
                    condition,

                "n_subjects":
                    len(
                        group_subjects
                    ),

                "mean_upper_triangle":
                    float(
                        np.nanmean(
                            upper_triangle_values(
                                mean_matrix
                            )
                        )
                    )
            })


group_matrix_summary = pd.DataFrame(
    group_matrix_rows
)


group_matrix_summary.to_csv(

    OUT
    /
    "group_matrix_summary.csv",

    index=False
)


# ================================================================
# 18. FIGURE 1 — SUBJECT GLOBAL DELTA dwPLI
# ================================================================

band_order = [
    "Theta",
    "Alpha",
    "Beta"
]


group_order = [
    "Active",
    "Passive"
]


rng = np.random.default_rng(
    RANDOM_SEED
)


fig, ax = plt.subplots(
    figsize=(
        10,
        7
    )
)


positions = {

    (
        "Theta",
        "Active"
    ):
        0,

    (
        "Theta",
        "Passive"
    ):
        1,

    (
        "Alpha",
        "Active"
    ):
        3,

    (
        "Alpha",
        "Passive"
    ):
        4,

    (
        "Beta",
        "Active"
    ):
        6,

    (
        "Beta",
        "Passive"
    ):
        7
}


for band in band_order:

    for group in group_order:

        values = subject_summary.loc[
            (
                subject_summary[
                    "band"
                ]
                ==
                band
            )

            &

            (
                subject_summary[
                    "group"
                ]
                ==
                group
            ),
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        x_position = positions[
            (
                band,
                group
            )
        ]


        jitter = rng.normal(

            0,

            0.045,

            size=len(
                values
            )
        )


        ax.scatter(

            np.full(
                len(
                    values
                ),
                x_position
            )
            +
            jitter,

            values,

            s=65,

            alpha=0.8
        )


        mean_value = float(
            np.mean(
                values
            )
        )


        ax.plot(

            [
                x_position
                -
                0.20,

                x_position
                +
                0.20
            ],

            [
                mean_value,
                mean_value
            ],

            linewidth=3
        )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    [
        0.5,
        3.5,
        6.5
    ]
)


ax.set_xticklabels(
    [
        "Theta",
        "Alpha",
        "Beta"
    ]
)


ax.set_ylabel(
    "Mean ΔdwPLI\n(Task − Baseline A)"
)


ax.set_xlabel(
    "Frequency band"
)


ax.set_title(
    "Participant-level global connectivity change"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    FIGURE_OUT
    /
    "01_subject_global_delta_dwpli.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 19. FIGURE 2 — GROUP-AVERAGE DELTA MATRICES
# ================================================================

for band in band_order:

    active_subjects = [

        subject

        for subject in all_matrices

        if all_matrices[
            subject
        ][
            "group"
        ]
        ==
        "Active"
    ]


    passive_subjects = [

        subject

        for subject in all_matrices

        if all_matrices[
            subject
        ][
            "group"
        ]
        ==
        "Passive"
    ]


    active_stack = np.stack(

        [

            all_matrices[
                subject
            ][
                "bands"
            ][
                band
            ][
                "delta"
            ]

            for subject in active_subjects

        ],

        axis=0
    )


    passive_stack = np.stack(

        [

            all_matrices[
                subject
            ][
                "bands"
            ][
                band
            ][
                "delta"
            ]

            for subject in passive_subjects

        ],

        axis=0
    )


    active_mean = np.nanmean(

        active_stack,

        axis=0
    )


    passive_mean = np.nanmean(

        passive_stack,

        axis=0
    )


    # ------------------------------------------------------------
    # Active
    # ------------------------------------------------------------

    limit = float(

        np.nanmax(

            np.abs(

                np.concatenate(

                    [
                        active_mean.ravel(),
                        passive_mean.ravel()
                    ]
                )
            )
        )
    )


    if (
        not np.isfinite(
            limit
        )

        or

        limit == 0
    ):

        limit = 1e-6


    fig, ax = plt.subplots(
        figsize=(
            8,
            7
        )
    )


    image = ax.imshow(

        active_mean,

        vmin=-limit,

        vmax=limit,

        aspect="equal"
    )


    ax.set_xticks(
        np.arange(
            len(
                reference_channels
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                reference_channels
            )
        )
    )


    ax.set_xticklabels(

        reference_channels,

        rotation=90
    )


    ax.set_yticklabels(
        reference_channels
    )


    ax.set_title(
        f"Active — {band} ΔdwPLI"
    )


    fig.colorbar(

        image,

        ax=ax,

        label="Task − Baseline A"
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"02_active_{band.lower()}_"
            "delta_dwpli_matrix.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


    # ------------------------------------------------------------
    # Passive
    # ------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(
            8,
            7
        )
    )


    image = ax.imshow(

        passive_mean,

        vmin=-limit,

        vmax=limit,

        aspect="equal"
    )


    ax.set_xticks(
        np.arange(
            len(
                reference_channels
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                reference_channels
            )
        )
    )


    ax.set_xticklabels(

        reference_channels,

        rotation=90
    )


    ax.set_yticklabels(
        reference_channels
    )


    ax.set_title(
        f"Passive — {band} ΔdwPLI"
    )


    fig.colorbar(

        image,

        ax=ax,

        label="Task − Baseline A"
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"03_passive_{band.lower()}_"
            "delta_dwpli_matrix.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 20. FIGURE 3 — DESCRIPTIVE ACTIVE MINUS PASSIVE DELTA MATRICES
# ================================================================
#
# IMPORTANT:
#
# These are DESCRIPTIVE difference matrices.
#
# They are NOT statistical significance maps.
#
# ================================================================

for band in band_order:

    active_subjects = [

        subject

        for subject in all_matrices

        if all_matrices[
            subject
        ][
            "group"
        ]
        ==
        "Active"
    ]


    passive_subjects = [

        subject

        for subject in all_matrices

        if all_matrices[
            subject
        ][
            "group"
        ]
        ==
        "Passive"
    ]


    active_mean = np.nanmean(

        np.stack(

            [

                all_matrices[
                    subject
                ][
                    "bands"
                ][
                    band
                ][
                    "delta"
                ]

                for subject in active_subjects

            ],

            axis=0
        ),

        axis=0
    )


    passive_mean = np.nanmean(

        np.stack(

            [

                all_matrices[
                    subject
                ][
                    "bands"
                ][
                    band
                ][
                    "delta"
                ]

                for subject in passive_subjects

            ],

            axis=0
        ),

        axis=0
    )


    difference = (

        active_mean

        -

        passive_mean
    )


    np.fill_diagonal(
        difference,
        0.0
    )


    limit = float(
        np.nanmax(
            np.abs(
                difference
            )
        )
    )


    if (
        not np.isfinite(
            limit
        )

        or

        limit == 0
    ):

        limit = 1e-6


    fig, ax = plt.subplots(
        figsize=(
            8,
            7
        )
    )


    image = ax.imshow(

        difference,

        vmin=-limit,

        vmax=limit,

        aspect="equal"
    )


    ax.set_xticks(
        np.arange(
            len(
                reference_channels
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                reference_channels
            )
        )
    )


    ax.set_xticklabels(

        reference_channels,

        rotation=90
    )


    ax.set_yticklabels(
        reference_channels
    )


    ax.set_title(

        f"{band} ΔdwPLI\n"
        "Active − Passive (descriptive)"
    )


    fig.colorbar(

        image,

        ax=ax,

        label=(
            "Difference in Task − Baseline A dwPLI"
        )
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"04_{band.lower()}_"
            "active_minus_passive_delta_dwpli.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 21. EDGE-LEVEL QC
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "EDGE-LEVEL QC"
)

print(
    "=" * 80
)


expected_edges_per_matrix = (

    len(
        EXPECTED_CHANNELS
    )

    *

    (
        len(
            EXPECTED_CHANNELS
        )
        -
        1
    )

    //
    2
)


print(
    f"Channels                         : "
    f"{len(EXPECTED_CHANNELS)}"
)


print(
    f"Unique undirected edges          : "
    f"{expected_edges_per_matrix}"
)


print(
    f"Expected subject-band edge rows  : "
    f"{15 * len(BANDS) * expected_edges_per_matrix}"
)


print(
    f"Observed subject-band edge rows  : "
    f"{len(edges)}"
)


edge_row_count_valid = (

    len(
        edges
    )

    ==

    (
        15

        *
        len(
            BANDS
        )

        *
        expected_edges_per_matrix
    )
)


print(
    f"Edge-row count valid             : "
    f"{edge_row_count_valid}"
)


# ================================================================
# 22. NUMERICAL QC
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "NUMERICAL QC"
)

print(
    "=" * 80
)


non_delta_qc = qc[
    qc[
        "condition"
    ].isin(
        [
            "baseline",
            "task"
        ]
    )
]


max_missing = int(
    non_delta_qc[
        "missing_edges"
    ].max()
)


max_symmetry = float(
    non_delta_qc[
        "symmetry_error"
    ].max()
)


max_diagonal = float(
    non_delta_qc[
        "diagonal_error"
    ].max()
)


max_dwpli = float(
    non_delta_qc[
        "edge_max"
    ].max()
)


min_dwpli = float(
    non_delta_qc[
        "edge_min"
    ].min()
)


print(
    f"Maximum missing edges       : "
    f"{max_missing}"
)


print(
    f"Maximum symmetry error      : "
    f"{max_symmetry:.12f}"
)


print(
    f"Maximum diagonal error      : "
    f"{max_diagonal:.12f}"
)


print(
    f"Minimum observed dwPLI      : "
    f"{min_dwpli:.6f}"
)


print(
    f"Maximum observed dwPLI      : "
    f"{max_dwpli:.6f}"
)


print(
    "\nNOTE:"
)


print(
    "Small negative dwPLI values are permissible for the "
    "debiased estimator and are therefore not automatically "
    "treated as errors."
)


# ================================================================
# 23. CHECK FOR EXTREME SUBJECTS
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "GLOBAL CONNECTIVITY EXTREME-SUBJECT SCREEN"
)

print(
    "=" * 80
)


extreme_rows = []


for band in band_order:

    band_data = subject_summary[
        subject_summary[
            "band"
        ]
        ==
        band
    ].copy()


    values = band_data[
        "delta_mean_dwpli"
    ].to_numpy(
        dtype=float
    )


    median_value = np.median(
        values
    )


    mad = np.median(

        np.abs(

            values

            -

            median_value
        )
    )


    if mad > 0:

        robust_z = (

            0.6745

            *

            (
                values
                -
                median_value
            )

            /

            mad
        )


    else:

        robust_z = np.zeros_like(
            values
        )


    band_data[
        "robust_z_delta_mean"
    ] = robust_z


    band_data[
        "extreme_abs_robust_z_gt_3_5"
    ] = (

        np.abs(
            robust_z
        )

        >
        3.5
    )


    extreme_rows.append(
        band_data
    )


extreme_screen = pd.concat(

    extreme_rows,

    ignore_index=True
)


display(

    extreme_screen[
        [
            "subject",
            "group",
            "band",
            "delta_mean_dwpli",
            "robust_z_delta_mean",
            "extreme_abs_robust_z_gt_3_5"
        ]
    ].round(
        5
    )
)


extreme_screen.to_csv(

    OUT
    /
    "global_connectivity_extreme_subject_screen.csv",

    index=False
)


n_extreme = int(

    extreme_screen[
        "extreme_abs_robust_z_gt_3_5"
    ].sum()
)


# ================================================================
# 24. FINAL INTEGRITY CHECK
# ================================================================

subject_count_valid = (
    n_subjects
    ==
    15
)


group_count_valid = (

    n_active
    ==
    8

    and

    n_passive
    ==
    7
)


expected_qc_rows = (

    15

    *
    len(
        BANDS
    )

    *
    3
)


qc_row_count_valid = (

    len(
        qc
    )

    ==
    expected_qc_rows
)


expected_summary_rows = (

    15

    *
    len(
        BANDS
    )
)


summary_row_count_valid = (

    len(
        subject_summary
    )

    ==
    expected_summary_rows
)


missing_edge_valid = (
    max_missing
    ==
    0
)


symmetry_valid = (
    max_symmetry
    <
    1e-12
)


diagonal_valid = (
    max_diagonal
    <
    1e-12
)


upper_bound_valid = (
    max_dwpli
    <=
    1.000001
)


overall_pass = bool(

    subject_count_valid

    and

    group_count_valid

    and

    edge_row_count_valid

    and

    qc_row_count_valid

    and

    summary_row_count_valid

    and

    missing_edge_valid

    and

    symmetry_valid

    and

    diagonal_valid

    and

    upper_bound_valid

    and

    len(
        errors
    )
    ==
    0
)


# ================================================================
# 25. SAVE INTEGRITY SUMMARY
# ================================================================

integrity = pd.DataFrame([{

    "n_subjects":
        n_subjects,

    "n_active":
        n_active,

    "n_passive":
        n_passive,

    "n_channels":
        len(
            EXPECTED_CHANNELS
        ),

    "unique_edges_per_matrix":
        expected_edges_per_matrix,

    "frequency_bands":
        len(
            BANDS
        ),

    "subject_band_rows":
        len(
            subject_summary
        ),

    "edge_rows":
        len(
            edges
        ),

    "qc_rows":
        len(
            qc
        ),

    "max_missing_edges":
        max_missing,

    "max_symmetry_error":
        max_symmetry,

    "max_diagonal_error":
        max_diagonal,

    "minimum_dwpli":
        min_dwpli,

    "maximum_dwpli":
        max_dwpli,

    "extreme_subject_band_cases":
        n_extreme,

    "processing_errors":
        len(
            errors
        ),

    "overall_integrity_pass":
        overall_pass
}])


integrity.to_csv(

    OUT
    /
    "task05a_integrity_summary.csv",

    index=False
)


# ================================================================
# 26. TASK 5A FINAL REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "TASK 5A COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participants analyzed          : "
    f"{n_subjects}/15"
)


print(
    f"Active                         : "
    f"{n_active}/8"
)


print(
    f"Passive                        : "
    f"{n_passive}/7"
)


print(
    f"EEG channels                   : "
    f"{len(EXPECTED_CHANNELS)}"
)


print(
    f"Unique edges/matrix            : "
    f"{expected_edges_per_matrix}"
)


print(
    f"Frequency bands                : "
    f"{', '.join(BANDS.keys())}"
)


print(
    f"Connectivity metric            : "
    f"debiased weighted phase-lag index (dwPLI)"
)


print(
    f"Primary baseline               : "
    f"Baseline A / T2_V1"
)


print(
    f"Connectivity change            : "
    f"Task − Baseline A"
)


print(
    f"Expected edge rows              : "
    f"{15 * len(BANDS) * expected_edges_per_matrix}"
)


print(
    f"Observed edge rows              : "
    f"{len(edges)}"
)


print(
    f"Missing connectivity edges      : "
    f"{max_missing}"
)


print(
    f"Maximum symmetry error          : "
    f"{max_symmetry:.12f}"
)


print(
    f"Minimum baseline/task dwPLI     : "
    f"{min_dwpli:.6f}"
)


print(
    f"Maximum baseline/task dwPLI     : "
    f"{max_dwpli:.6f}"
)


print(
    f"Extreme subject-band cases      : "
    f"{n_extreme}"
)


print(
    f"Processing errors               : "
    f"{len(errors)}"
)


print(
    "\n"
    + "-" * 80
)


if overall_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )

else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "IMPORTANT:"
)


print(
    "No Active-vs-Passive statistical inference was "
    "performed in Task 5A."
)


print(
    "The Active-minus-Passive matrices produced here are "
    "descriptive visualizations only."
)


print(
    "No individual connectivity edge has been selected as "
    "significant or interpreted as a group effect."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# ================================================================
# TASK 5B0 — FIXED-10-EPOCH dwPLI SENSITIVITY AND ANALYSIS FREEZE
# ================================================================
#
# PURPOSE
# -------
# Task 5A used all available matched clean epochs within each
# participant. This resulted in 10–15 epochs per condition.
#
# Before inferential testing, this task standardizes the amount
# of EEG information contributing to every connectivity estimate:
#
#       10 epochs × 2 seconds = 20 seconds
#
# for:
#
#       Baseline A
#       Task
#
# in EVERY participant.
#
# We then:
#
#   1. recompute Theta/Alpha/Beta dwPLI matrices;
#   2. compute Task - Baseline A delta matrices;
#   3. compare fixed-10 matrices with Task 5A matrices;
#   4. compare participant-level global delta dwPLI;
#   5. repeat the robust extreme-subject screen;
#   6. save the standardized matrices;
#   7. freeze the fixed-10 dataset for subsequent inference.
#
# IMPORTANT
# ---------
# NO Active-vs-Passive hypothesis test is performed here.
#
# The choice of 10 epochs is based ONLY on data availability/QC,
# before group inferential testing.
#
# ================================================================


# ================================================================
# 1. IMPORTS
# ================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import hilbert
from scipy.stats import pearsonr, spearmanr


warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# ================================================================
# 2. PATHS
# ================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)


TASK3D = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03d_matched_baseline_task_epochs"
)


TASK5A = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05a_dwpli_connectivity_qc"
)


TASK5A_MATRICES = (
    TASK5A
    / "matrices"
)


OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05b0_fixed10_dwpli_freeze"
)


MATRIX_OUT = (
    OUT
    / "matrices"
)


FIGURE_OUT = (
    OUT
    / "figures"
)


OUT.mkdir(
    parents=True,
    exist_ok=True
)


MATRIX_OUT.mkdir(
    parents=True,
    exist_ok=True
)


FIGURE_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ================================================================
# 3. SETTINGS
# ================================================================

FIXED_N_EPOCHS = 10


BANDS = {

    "Theta":
        (4.0, 8.0),

    "Alpha":
        (8.0, 13.0),

    "Beta":
        (13.0, 30.0)
}


EXPECTED_CHANNELS = [

    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


EXPECTED_SUBJECTS = [

    f"sub-{i:03d}"
    for i in range(1, 16)
]


EXPECTED_GROUPS = {

    **{
        f"sub-{i:03d}": "Active"
        for i in range(1, 9)
    },

    **{
        f"sub-{i:03d}": "Passive"
        for i in range(9, 16)
    }
}


# ================================================================
# 4. HELPER — BANDPASS
# ================================================================

def bandpass_epoch_data(
    data,
    sfreq,
    low,
    high
):

    return mne.filter.filter_data(

        data,

        sfreq=sfreq,

        l_freq=low,

        h_freq=high,

        method="fir",

        phase="zero",

        fir_design="firwin",

        pad="reflect_limited",

        verbose=False
    )


# ================================================================
# 5. HELPER — dwPLI
# ================================================================

def debiased_wpli_from_epochs(
    epoch_data
):

    epoch_data = np.asarray(
        epoch_data,
        dtype=float
    )


    if epoch_data.ndim != 3:

        raise ValueError(
            "Expected epochs × channels × samples."
        )


    analytic = hilbert(
        epoch_data,
        axis=-1
    )


    n_channels = epoch_data.shape[1]


    matrix = np.zeros(
        (
            n_channels,
            n_channels
        ),
        dtype=float
    )


    for i in range(
        n_channels
    ):

        zi = analytic[
            :,
            i,
            :
        ]


        for j in range(
            i + 1,
            n_channels
        ):

            zj = analytic[
                :,
                j,
                :
            ]


            imaginary_cross = np.imag(

                zi

                *

                np.conjugate(
                    zj
                )
            )


            values = imaginary_cross.reshape(
                -1
            )


            values = values[
                np.isfinite(
                    values
                )
            ]


            if len(values) < 2:

                value = np.nan

            else:

                sum_im = np.sum(
                    values
                )


                sum_abs_im = np.sum(
                    np.abs(
                        values
                    )
                )


                sum_sq = np.sum(
                    values ** 2
                )


                numerator = (

                    sum_im ** 2

                    -

                    sum_sq
                )


                denominator = (

                    sum_abs_im ** 2

                    -

                    sum_sq
                )


                if (
                    not np.isfinite(
                        denominator
                    )

                    or

                    denominator <= 0
                ):

                    value = np.nan

                else:

                    value = (

                        numerator

                        /

                        denominator
                    )


            matrix[
                i,
                j
            ] = value


            matrix[
                j,
                i
            ] = value


    np.fill_diagonal(
        matrix,
        0.0
    )


    return matrix


# ================================================================
# 6. HELPER — UPPER TRIANGLE
# ================================================================

def upper_triangle(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    return matrix[
        np.triu_indices(
            matrix.shape[0],
            k=1
        )
    ]


# ================================================================
# 7. HELPER — MATRIX CORRELATION
# ================================================================

def matrix_correlation(
    matrix_a,
    matrix_b
):

    a = upper_triangle(
        matrix_a
    )


    b = upper_triangle(
        matrix_b
    )


    valid = (
        np.isfinite(a)
        &
        np.isfinite(b)
    )


    if np.sum(valid) < 3:

        return np.nan


    if (
        np.std(a[valid]) == 0
        or
        np.std(b[valid]) == 0
    ):

        return np.nan


    result = pearsonr(
        a[valid],
        b[valid]
    )


    return float(
        result.statistic
    )


# ================================================================
# 8. HELPER — MATRIX DIFFERENCE
# ================================================================

def matrix_difference_metrics(
    matrix_fixed,
    matrix_original
):

    fixed = upper_triangle(
        matrix_fixed
    )


    original = upper_triangle(
        matrix_original
    )


    valid = (
        np.isfinite(fixed)
        &
        np.isfinite(original)
    )


    difference = (
        fixed[valid]
        -
        original[valid]
    )


    return {

        "edge_correlation":
            matrix_correlation(
                matrix_fixed,
                matrix_original
            ),

        "mean_absolute_edge_difference":
            float(
                np.mean(
                    np.abs(
                        difference
                    )
                )
            ),

        "rmse_edge_difference":
            float(
                np.sqrt(
                    np.mean(
                        difference ** 2
                    )
                )
            ),

        "maximum_absolute_edge_difference":
            float(
                np.max(
                    np.abs(
                        difference
                    )
                )
            )
    }


# ================================================================
# 9. FIND MATCHED EPOCH FILES
# ================================================================

epoch_files = sorted(

    TASK3D.glob(

        "sub-*/"
        "*desc-matched_baseline_task_epo.fif"
    )
)


print(
    "=" * 80
)


print(
    "TASK 5B0 — FIXED-10-EPOCH dwPLI SENSITIVITY AND ANALYSIS FREEZE"
)


print(
    "=" * 80
)


print(
    f"Task 3D directory     : {TASK3D}"
)


print(
    f"Task 5A directory     : {TASK5A}"
)


print(
    f"Output directory      : {OUT}"
)


print(
    f"Fixed epochs/condition: {FIXED_N_EPOCHS}"
)


print(
    f"Fixed seconds/condition: "
    f"{FIXED_N_EPOCHS * 2}"
)


print(
    f"Epoch files found     : {len(epoch_files)}"
)


if len(
    epoch_files
) != 15:

    raise RuntimeError(

        f"Expected 15 matched epoch files; "
        f"found {len(epoch_files)}."
    )


# ================================================================
# 10. PROCESS FIXED-10 CONNECTIVITY
# ================================================================

fixed_summary_rows = []

comparison_rows = []

edge_rows = []

error_rows = []


fixed_matrices = {}


reference_channels = None


for epoch_file in epoch_files:

    print(
        "\n"
        + "-" * 80
    )


    try:

        epochs = mne.read_epochs(

            epoch_file,

            preload=True,

            verbose=False
        )


        if epochs.metadata is None:

            raise RuntimeError(
                "Epoch metadata missing."
            )


        metadata = epochs.metadata


        subjects = (
            metadata[
                "subject"
            ]
            .astype(str)
            .unique()
        )


        groups = (
            metadata[
                "group"
            ]
            .astype(str)
            .unique()
        )


        if len(subjects) != 1:

            raise RuntimeError(
                f"Expected one subject; found {subjects}."
            )


        if len(groups) != 1:

            raise RuntimeError(
                f"Expected one group; found {groups}."
            )


        subject = str(
            subjects[0]
        )


        group = str(
            groups[0]
        )


        if subject not in EXPECTED_SUBJECTS:

            raise RuntimeError(
                f"Unexpected subject: {subject}"
            )


        if (
            EXPECTED_GROUPS[
                subject
            ]
            !=
            group
        ):

            raise RuntimeError(
                f"{subject}: group mismatch."
            )


        print(
            f"{subject} | {group}"
        )


        # --------------------------------------------------------
        # Standardize channel order
        # --------------------------------------------------------

        if set(
            epochs.ch_names
        ) != set(
            EXPECTED_CHANNELS
        ):

            raise RuntimeError(
                f"{subject}: unexpected channels."
            )


        epochs.reorder_channels(
            EXPECTED_CHANNELS
        )


        channels = list(
            epochs.ch_names
        )


        if reference_channels is None:

            reference_channels = (
                channels.copy()
            )


        elif channels != reference_channels:

            raise RuntimeError(
                f"{subject}: channel-order mismatch."
            )


        sfreq = float(
            epochs.info[
                "sfreq"
            ]
        )


        # --------------------------------------------------------
        # Retrieve matched baseline and task
        # --------------------------------------------------------

        baseline_epochs = epochs[
            "baseline"
        ]


        task_epochs = epochs[
            "task"
        ]


        n_available_baseline = len(
            baseline_epochs
        )


        n_available_task = len(
            task_epochs
        )


        if (
            n_available_baseline
            <
            FIXED_N_EPOCHS
        ):

            raise RuntimeError(

                f"{subject}: only "
                f"{n_available_baseline} baseline epochs."
            )


        if (
            n_available_task
            <
            FIXED_N_EPOCHS
        ):

            raise RuntimeError(

                f"{subject}: only "
                f"{n_available_task} task epochs."
            )


        # --------------------------------------------------------
        # FIXED-N FREEZE
        #
        # Select the FIRST 10 clean matched epochs.
        #
        # No outcome-dependent selection.
        # --------------------------------------------------------

        baseline_fixed = (

            baseline_epochs[
                :FIXED_N_EPOCHS
            ]
            .get_data()
        )


        task_fixed = (

            task_epochs[
                :FIXED_N_EPOCHS
            ]
            .get_data()
        )


        if baseline_fixed.shape != task_fixed.shape:

            raise RuntimeError(
                f"{subject}: fixed arrays differ in shape."
            )


        print(
            f"  available baseline : "
            f"{n_available_baseline}"
        )


        print(
            f"  available task     : "
            f"{n_available_task}"
        )


        print(
            f"  fixed per condition: "
            f"{FIXED_N_EPOCHS}"
        )


        fixed_matrices[
            subject
        ] = {

            "group":
                group,

            "bands":
                {}
        }


        # ========================================================
        # BAND LOOP
        # ========================================================

        for band, (
            low,
            high
        ) in BANDS.items():

            print(
                f"  {band:<6s} "
                f"{low:.1f}-{high:.1f} Hz"
            )


            baseline_filtered = bandpass_epoch_data(

                baseline_fixed,

                sfreq,

                low,

                high
            )


            task_filtered = bandpass_epoch_data(

                task_fixed,

                sfreq,

                low,

                high
            )


            baseline_matrix = debiased_wpli_from_epochs(

                baseline_filtered
            )


            task_matrix = debiased_wpli_from_epochs(

                task_filtered
            )


            delta_matrix = (

                task_matrix

                -

                baseline_matrix
            )


            np.fill_diagonal(
                delta_matrix,
                0.0
            )


            fixed_matrices[
                subject
            ][
                "bands"
            ][
                band
            ] = {

                "baseline":
                    baseline_matrix,

                "task":
                    task_matrix,

                "delta":
                    delta_matrix
            }


            # ----------------------------------------------------
            # Save fixed matrices
            # ----------------------------------------------------

            subject_dir = (

                MATRIX_OUT
                /
                subject
            )


            subject_dir.mkdir(

                parents=True,

                exist_ok=True
            )


            for condition, matrix in [

                (
                    "baseline",
                    baseline_matrix
                ),

                (
                    "task",
                    task_matrix
                ),

                (
                    "delta",
                    delta_matrix
                )
            ]:

                np.save(

                    subject_dir
                    /
                    (
                        f"{subject}_{band.lower()}_"
                        f"{condition}_fixed10_dwpli.npy"
                    ),

                    matrix
                )


                pd.DataFrame(

                    matrix,

                    index=channels,

                    columns=channels

                ).to_csv(

                    subject_dir
                    /
                    (
                        f"{subject}_{band.lower()}_"
                        f"{condition}_fixed10_dwpli.csv"
                    )
                )


            # ----------------------------------------------------
            # Fixed global summaries
            # ----------------------------------------------------

            baseline_upper = upper_triangle(
                baseline_matrix
            )


            task_upper = upper_triangle(
                task_matrix
            )


            delta_upper = upper_triangle(
                delta_matrix
            )


            fixed_summary_rows.append({

                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "available_epochs":
                    min(
                        n_available_baseline,
                        n_available_task
                    ),

                "fixed_epochs":
                    FIXED_N_EPOCHS,

                "baseline_mean_dwpli":
                    float(
                        np.nanmean(
                            baseline_upper
                        )
                    ),

                "task_mean_dwpli":
                    float(
                        np.nanmean(
                            task_upper
                        )
                    ),

                "delta_mean_dwpli":
                    float(
                        np.nanmean(
                            delta_upper
                        )
                    ),

                "delta_median_dwpli":
                    float(
                        np.nanmedian(
                            delta_upper
                        )
                    ),

                "positive_delta_edge_fraction":
                    float(
                        np.nanmean(
                            delta_upper
                            >
                            0
                        )
                    )
            })


            # ----------------------------------------------------
            # Save fixed edge-level long data
            # ----------------------------------------------------

            for i in range(
                len(channels)
            ):

                for j in range(
                    i + 1,
                    len(channels)
                ):

                    edge_rows.append({

                        "subject":
                            subject,

                        "group":
                            group,

                        "band":
                            band,

                        "channel_1":
                            channels[i],

                        "channel_2":
                            channels[j],

                        "baseline_dwpli":
                            float(
                                baseline_matrix[
                                    i,
                                    j
                                ]
                            ),

                        "task_dwpli":
                            float(
                                task_matrix[
                                    i,
                                    j
                                ]
                            ),

                        "delta_dwpli":
                            float(
                                delta_matrix[
                                    i,
                                    j
                                ]
                            )
                    })


            # ----------------------------------------------------
            # Load corresponding Task 5A matrices
            # ----------------------------------------------------

            original_subject_dir = (

                TASK5A_MATRICES
                /
                subject
            )


            original_baseline_file = (

                original_subject_dir
                /
                (
                    f"{subject}_{band.lower()}_"
                    "baseline_dwpli.npy"
                )
            )


            original_task_file = (

                original_subject_dir
                /
                (
                    f"{subject}_{band.lower()}_"
                    "task_dwpli.npy"
                )
            )


            original_delta_file = (

                original_subject_dir
                /
                (
                    f"{subject}_{band.lower()}_"
                    "delta_dwpli.npy"
                )
            )


            for required_file in [

                original_baseline_file,
                original_task_file,
                original_delta_file
            ]:

                if not required_file.exists():

                    raise FileNotFoundError(
                        required_file
                    )


            original_baseline = np.load(
                original_baseline_file
            )


            original_task = np.load(
                original_task_file
            )


            original_delta = np.load(
                original_delta_file
            )


            # ----------------------------------------------------
            # Matrix stability
            # ----------------------------------------------------

            for condition, (
                fixed_matrix,
                original_matrix
            ) in {

                "baseline":
                    (
                        baseline_matrix,
                        original_baseline
                    ),

                "task":
                    (
                        task_matrix,
                        original_task
                    ),

                "delta":
                    (
                        delta_matrix,
                        original_delta
                    )

            }.items():

                metrics = matrix_difference_metrics(

                    fixed_matrix,

                    original_matrix
                )


                comparison_rows.append({

                    "subject":
                        subject,

                    "group":
                        group,

                    "band":
                        band,

                    "condition":
                        condition,

                    "available_epochs":
                        min(
                            n_available_baseline,
                            n_available_task
                        ),

                    "fixed_epochs":
                        FIXED_N_EPOCHS,

                    **metrics
                })


        print(
            "  STATUS: SUCCESS"
        )


    except Exception as exc:

        print(
            f"ERROR: {exc}"
        )


        error_rows.append({

            "file":
                str(
                    epoch_file
                ),

            "error":
                repr(
                    exc
                )
        })


# ================================================================
# 11. TABLES
# ================================================================

fixed_summary = pd.DataFrame(
    fixed_summary_rows
)


comparison = pd.DataFrame(
    comparison_rows
)


fixed_edges = pd.DataFrame(
    edge_rows
)


errors = pd.DataFrame(
    error_rows
)


fixed_summary.to_csv(

    OUT
    /
    "fixed10_subject_global_dwpli.csv",

    index=False
)


comparison.to_csv(

    OUT
    /
    "fixed10_vs_task5a_matrix_stability.csv",

    index=False
)


fixed_edges.to_csv(

    OUT
    /
    "fixed10_edge_level_long.csv",

    index=False
)


errors.to_csv(

    OUT
    /
    "processing_errors.csv",

    index=False
)


# ================================================================
# 12. VERIFY COMPLETE SAMPLE
# ================================================================

n_subjects = int(
    fixed_summary[
        "subject"
    ].nunique()
)


n_active = int(

    fixed_summary.loc[
        fixed_summary[
            "group"
        ]
        ==
        "Active",
        "subject"
    ].nunique()
)


n_passive = int(

    fixed_summary.loc[
        fixed_summary[
            "group"
        ]
        ==
        "Passive",
        "subject"
    ].nunique()
)


print(
    "\n"
    + "=" * 80
)


print(
    "FIXED-10 SAMPLE SUMMARY"
)


print(
    "=" * 80
)


print(
    f"Participants : {n_subjects}/15"
)


print(
    f"Active       : {n_active}/8"
)


print(
    f"Passive      : {n_passive}/7"
)


print(
    f"Errors       : {len(errors)}"
)


if len(errors) > 0:

    display(
        errors
    )


# ================================================================
# 13. MATRIX STABILITY SUMMARY
# ================================================================

stability_summary = (

    comparison

    .groupby(
        [
            "band",
            "condition"
        ]
    )

    .agg(

        n=(
            "subject",
            "count"
        ),

        mean_edge_correlation=(
            "edge_correlation",
            "mean"
        ),

        median_edge_correlation=(
            "edge_correlation",
            "median"
        ),

        minimum_edge_correlation=(
            "edge_correlation",
            "min"
        ),

        mean_absolute_edge_difference=(
            "mean_absolute_edge_difference",
            "mean"
        ),

        mean_rmse=(
            "rmse_edge_difference",
            "mean"
        ),

        maximum_absolute_difference=(
            "maximum_absolute_edge_difference",
            "max"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)


print(
    "FIXED-10 VS ORIGINAL TASK 5A MATRIX STABILITY"
)


print(
    "=" * 80
)


display(
    stability_summary.round(
        6
    )
)


stability_summary.to_csv(

    OUT
    /
    "matrix_stability_summary.csv",

    index=False
)


# ================================================================
# 14. GLOBAL DELTA COMPARISON WITH TASK 5A
# ================================================================

task5a_summary_file = (

    TASK5A
    /
    "subject_global_dwpli_summary.csv"
)


if not task5a_summary_file.exists():

    raise FileNotFoundError(
        task5a_summary_file
    )


original_summary = pd.read_csv(
    task5a_summary_file
)


original_summary = original_summary[
    [
        "subject",
        "group",
        "band",
        "n_epochs",
        "delta_mean_dwpli"
    ]
].copy()


original_summary = original_summary.rename(
    columns={

        "n_epochs":
            "original_epochs",

        "delta_mean_dwpli":
            "original_delta_mean_dwpli"
    }
)


global_comparison = fixed_summary.merge(

    original_summary,

    on=[
        "subject",
        "group",
        "band"
    ],

    how="left"
)


global_comparison = global_comparison.rename(
    columns={

        "delta_mean_dwpli":
            "fixed10_delta_mean_dwpli"
    }
)


global_comparison[
    "delta_difference_fixed_minus_original"
] = (

    global_comparison[
        "fixed10_delta_mean_dwpli"
    ]

    -

    global_comparison[
        "original_delta_mean_dwpli"
    ]
)


global_comparison[
    "same_sign"
] = (

    np.sign(
        global_comparison[
            "fixed10_delta_mean_dwpli"
        ]
    )

    ==

    np.sign(
        global_comparison[
            "original_delta_mean_dwpli"
        ]
    )
)


global_comparison.to_csv(

    OUT
    /
    "fixed10_vs_original_global_delta.csv",

    index=False
)


# ================================================================
# 15. BAND-LEVEL GLOBAL STABILITY
# ================================================================

band_stability_rows = []


for band in BANDS.keys():

    temp = global_comparison[
        global_comparison[
            "band"
        ]
        ==
        band
    ].copy()


    x = temp[
        "original_delta_mean_dwpli"
    ].to_numpy(
        dtype=float
    )


    y = temp[
        "fixed10_delta_mean_dwpli"
    ].to_numpy(
        dtype=float
    )


    pearson_result = pearsonr(
        x,
        y
    )


    spearman_result = spearmanr(
        x,
        y
    )


    band_stability_rows.append({

        "band":
            band,

        "n_subjects":
            len(
                temp
            ),

        "same_sign_n":
            int(
                temp[
                    "same_sign"
                ].sum()
            ),

        "same_sign_percent":
            float(
                100
                *
                temp[
                    "same_sign"
                ].mean()
            ),

        "pearson_r":
            float(
                pearson_result.statistic
            ),

        "pearson_p":
            float(
                pearson_result.pvalue
            ),

        "spearman_rho":
            float(
                spearman_result.statistic
            ),

        "spearman_p":
            float(
                spearman_result.pvalue
            ),

        "mean_absolute_global_delta_difference":
            float(
                np.mean(
                    np.abs(
                        temp[
                            "delta_difference_fixed_minus_original"
                        ]
                    )
                )
            )
    })


band_stability = pd.DataFrame(
    band_stability_rows
)


print(
    "\n"
    + "=" * 80
)


print(
    "GLOBAL ΔdwPLI STABILITY"
)


print(
    "=" * 80
)


display(
    band_stability.round(
        6
    )
)


band_stability.to_csv(

    OUT
    /
    "global_delta_stability_by_band.csv",

    index=False
)


# ================================================================
# 16. FIXED-10 DESCRIPTIVE GROUP SUMMARY
# ================================================================

group_summary = (

    fixed_summary

    .groupby(
        [
            "group",
            "band"
        ]
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_baseline_dwpli=(
            "baseline_mean_dwpli",
            "mean"
        ),

        sd_baseline_dwpli=(
            "baseline_mean_dwpli",
            "std"
        ),

        mean_task_dwpli=(
            "task_mean_dwpli",
            "mean"
        ),

        sd_task_dwpli=(
            "task_mean_dwpli",
            "std"
        ),

        mean_delta_dwpli=(
            "fixed10_delta_mean_dwpli"
            if "fixed10_delta_mean_dwpli"
            in fixed_summary.columns
            else
            "delta_mean_dwpli",
            "mean"
        ),

        sd_delta_dwpli=(
            "fixed10_delta_mean_dwpli"
            if "fixed10_delta_mean_dwpli"
            in fixed_summary.columns
            else
            "delta_mean_dwpli",
            "std"
        )
    )

    .reset_index()
)


print(
    "\n"
    + "=" * 80
)


print(
    "FIXED-10 DESCRIPTIVE GROUP SUMMARY"
)


print(
    "=" * 80
)


display(
    group_summary.round(
        6
    )
)


group_summary.to_csv(

    OUT
    /
    "fixed10_descriptive_group_summary.csv",

    index=False
)


# ================================================================
# 17. ROBUST EXTREME-SUBJECT SCREEN
# ================================================================

extreme_rows = []


for band in BANDS.keys():

    band_data = fixed_summary[
        fixed_summary[
            "band"
        ]
        ==
        band
    ].copy()


    values = band_data[
        "delta_mean_dwpli"
    ].to_numpy(
        dtype=float
    )


    median_value = np.median(
        values
    )


    mad = np.median(

        np.abs(

            values

            -

            median_value
        )
    )


    if mad > 0:

        robust_z = (

            0.6745

            *

            (
                values
                -
                median_value
            )

            /

            mad
        )

    else:

        robust_z = np.zeros_like(
            values
        )


    band_data[
        "robust_z"
    ] = robust_z


    band_data[
        "extreme_abs_robust_z_gt_3_5"
    ] = (

        np.abs(
            robust_z
        )

        >
        3.5
    )


    extreme_rows.append(
        band_data
    )


extreme_screen = pd.concat(

    extreme_rows,

    ignore_index=True
)


print(
    "\n"
    + "=" * 80
)


print(
    "FIXED-10 EXTREME-SUBJECT SCREEN"
)


print(
    "=" * 80
)


display(

    extreme_screen[
        [
            "subject",
            "group",
            "band",
            "delta_mean_dwpli",
            "robust_z",
            "extreme_abs_robust_z_gt_3_5"
        ]
    ].round(
        6
    )
)


extreme_screen.to_csv(

    OUT
    /
    "fixed10_extreme_subject_screen.csv",

    index=False
)


n_extreme = int(

    extreme_screen[
        "extreme_abs_robust_z_gt_3_5"
    ].sum()
)


# ================================================================
# 18. FIGURE — ORIGINAL VS FIXED-10 GLOBAL DELTA
# ================================================================

for band in BANDS.keys():

    plot_data = global_comparison[
        global_comparison[
            "band"
        ]
        ==
        band
    ].copy()


    fig, ax = plt.subplots(
        figsize=(
            7,
            7
        )
    )


    for group in [
        "Active",
        "Passive"
    ]:

        group_data = plot_data[
            plot_data[
                "group"
            ]
            ==
            group
        ]


        ax.scatter(

            group_data[
                "original_delta_mean_dwpli"
            ],

            group_data[
                "fixed10_delta_mean_dwpli"
            ],

            s=75,

            label=group
        )


    for _, row in plot_data.iterrows():

        ax.annotate(

            row[
                "subject"
            ],

            (
                row[
                    "original_delta_mean_dwpli"
                ],

                row[
                    "fixed10_delta_mean_dwpli"
                ]
            ),

            xytext=(
                4,
                4
            ),

            textcoords="offset points",

            fontsize=8
        )


    all_values = np.concatenate(

        [

            plot_data[
                "original_delta_mean_dwpli"
            ].to_numpy(),

            plot_data[
                "fixed10_delta_mean_dwpli"
            ].to_numpy()
        ]
    )


    lower = float(
        np.min(
            all_values
        )
    )


    upper = float(
        np.max(
            all_values
        )
    )


    padding = (

        0.1

        *

        (
            upper
            -
            lower
        )
    )


    if padding == 0:

        padding = 0.001


    lower -= padding

    upper += padding


    ax.plot(

        [
            lower,
            upper
        ],

        [
            lower,
            upper
        ],

        linestyle="--",

        linewidth=1
    )


    ax.axhline(
        0,
        linewidth=0.8
    )


    ax.axvline(
        0,
        linewidth=0.8
    )


    ax.set_xlim(
        lower,
        upper
    )


    ax.set_ylim(
        lower,
        upper
    )


    ax.set_xlabel(
        "Original Task 5A mean ΔdwPLI"
    )


    ax.set_ylabel(
        "Fixed-10 mean ΔdwPLI"
    )


    ax.set_title(
        f"{band}: fixed-epoch connectivity stability"
    )


    ax.legend()


    ax.grid(
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"{band.lower()}_"
            "original_vs_fixed10_global_delta.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# ================================================================
# 19. FIGURE — FIXED-10 GLOBAL DELTA BY GROUP
# ================================================================

rng = np.random.default_rng(
    20260913
)


fig, ax = plt.subplots(
    figsize=(
        10,
        7
    )
)


positions = {

    (
        "Theta",
        "Active"
    ):
        0,

    (
        "Theta",
        "Passive"
    ):
        1,

    (
        "Alpha",
        "Active"
    ):
        3,

    (
        "Alpha",
        "Passive"
    ):
        4,

    (
        "Beta",
        "Active"
    ):
        6,

    (
        "Beta",
        "Passive"
    ):
        7
}


for band in BANDS.keys():

    for group in [
        "Active",
        "Passive"
    ]:

        values = fixed_summary.loc[
            (
                fixed_summary[
                    "band"
                ]
                ==
                band
            )

            &

            (
                fixed_summary[
                    "group"
                ]
                ==
                group
            ),
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        position = positions[
            (
                band,
                group
            )
        ]


        jitter = rng.normal(

            0,

            0.045,

            len(
                values
            )
        )


        ax.scatter(

            np.full(
                len(
                    values
                ),
                position
            )
            +
            jitter,

            values,

            s=70,

            alpha=0.8
        )


        mean_value = np.mean(
            values
        )


        ax.plot(

            [
                position - 0.20,
                position + 0.20
            ],

            [
                mean_value,
                mean_value
            ],

            linewidth=3
        )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    [
        0.5,
        3.5,
        6.5
    ]
)


ax.set_xticklabels(
    [
        "Theta",
        "Alpha",
        "Beta"
    ]
)


ax.set_ylabel(
    "Mean ΔdwPLI (Task − Baseline A)"
)


ax.set_xlabel(
    "Frequency band"
)


ax.set_title(
    "Fixed-10 participant-level connectivity change"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    FIGURE_OUT
    /
    "fixed10_global_delta_by_group.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# ================================================================
# 20. EDGE-COUNT / MATRIX INTEGRITY
# ================================================================

N_CHANNELS = len(
    EXPECTED_CHANNELS
)


EXPECTED_EDGES = (

    N_CHANNELS

    *

    (
        N_CHANNELS
        -
        1
    )

    //
    2
)


EXPECTED_EDGE_ROWS = (

    15

    *
    len(
        BANDS
    )

    *
    EXPECTED_EDGES
)


observed_edge_rows = len(
    fixed_edges
)


finite_edge_rows = int(

    np.sum(

        np.isfinite(

            fixed_edges[
                "delta_dwpli"
            ]
        )
    )
)


print(
    "\n"
    + "=" * 80
)


print(
    "FIXED-10 EDGE INTEGRITY"
)


print(
    "=" * 80
)


print(
    f"Channels                : {N_CHANNELS}"
)


print(
    f"Edges/matrix            : {EXPECTED_EDGES}"
)


print(
    f"Expected edge rows      : {EXPECTED_EDGE_ROWS}"
)


print(
    f"Observed edge rows      : {observed_edge_rows}"
)


print(
    f"Finite delta edge rows  : {finite_edge_rows}"
)


# ================================================================
# 21. FREEZE DECISION
# ================================================================

subject_valid = (
    n_subjects
    ==
    15
)


group_valid = (
    n_active
    ==
    8

    and

    n_passive
    ==
    7
)


edge_valid = (

    observed_edge_rows
    ==
    EXPECTED_EDGE_ROWS

    and

    finite_edge_rows
    ==
    EXPECTED_EDGE_ROWS
)


error_valid = (
    len(
        errors
    )
    ==
    0
)


all_have_fixed10 = bool(

    np.all(

        fixed_summary[
            "available_epochs"
        ]

        >=

        FIXED_N_EPOCHS
    )
)


freeze_pass = bool(

    subject_valid

    and

    group_valid

    and

    edge_valid

    and

    error_valid

    and

    all_have_fixed10
)


freeze_summary = pd.DataFrame([{

    "fixed_epochs_per_condition":
        FIXED_N_EPOCHS,

    "fixed_seconds_per_condition":
        FIXED_N_EPOCHS
        *
        2,

    "participants":
        n_subjects,

    "active":
        n_active,

    "passive":
        n_passive,

    "edges_per_matrix":
        EXPECTED_EDGES,

    "expected_edge_rows":
        EXPECTED_EDGE_ROWS,

    "observed_edge_rows":
        observed_edge_rows,

    "finite_delta_edge_rows":
        finite_edge_rows,

    "extreme_subject_band_cases":
        n_extreme,

    "processing_errors":
        len(
            errors
        ),

    "all_subjects_have_fixed10":
        all_have_fixed10,

    "freeze_pass":
        freeze_pass
}])


freeze_summary.to_csv(

    OUT
    /
    "analysis_freeze_summary.csv",

    index=False
)


# ================================================================
# 22. FINAL REPORT
# ================================================================

print(
    "\n"
    + "=" * 80
)


print(
    "TASK 5B0 COMPLETE"
)


print(
    "=" * 80
)


print(
    f"Participants analyzed          : "
    f"{n_subjects}/15"
)


print(
    f"Active                         : "
    f"{n_active}/8"
)


print(
    f"Passive                        : "
    f"{n_passive}/7"
)


print(
    f"Fixed epochs/condition         : "
    f"{FIXED_N_EPOCHS}"
)


print(
    f"Fixed seconds/condition        : "
    f"{FIXED_N_EPOCHS * 2}"
)


print(
    f"Frequency bands                : "
    f"{', '.join(BANDS.keys())}"
)


print(
    f"Connectivity metric            : "
    f"dwPLI"
)


print(
    f"Baseline                       : "
    f"Baseline A / T2_V1"
)


print(
    f"Edges per matrix               : "
    f"{EXPECTED_EDGES}"
)


print(
    f"Expected edge rows             : "
    f"{EXPECTED_EDGE_ROWS}"
)


print(
    f"Observed edge rows             : "
    f"{observed_edge_rows}"
)


print(
    f"Extreme subject-band cases     : "
    f"{n_extreme}"
)


print(
    f"Processing errors              : "
    f"{len(errors)}"
)


print(
    "\n"
    + "-" * 80
)


if freeze_pass:

    print(
        "FIXED-10 CONNECTIVITY DATASET: FREEZE PASS"
    )

else:

    print(
        "FIXED-10 CONNECTIVITY DATASET: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "PRIMARY CONNECTIVITY DATASET FOR SUBSEQUENT INFERENCE:"
)


print(
    "10 earliest clean matched 2-s epochs per condition "
    "per participant."
)


print(
    "Connectivity change = Task dwPLI − Baseline A dwPLI."
)


print(
    "No participant has been excluded on the basis of "
    "connectivity magnitude."
)


print(
    "No Active-vs-Passive hypothesis test has been "
    "performed in this task."
)


print(
    "\nOutput directory:"
)


print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# =============================================================================
# TASK 5B — EXACT PERMUTATION INFERENCE FOR GLOBAL dwPLI CHANGE
# =============================================================================
#
# PRIMARY CONNECTIVITY ENDPOINT
# -----------------------------
# Participant-level mean delta dwPLI:
#
#       Delta dwPLI = Task dwPLI - Baseline A dwPLI
#
# Frequency bands:
#       Theta  4-8 Hz
#       Alpha  8-13 Hz
#       Beta   13-30 Hz
#
# Groups:
#       Active  n = 8
#       Passive n = 7
#
# INFERENCE
# ---------
# 1. Exact label permutation test
#       C(15, 8) = 6435 possible allocations
#
# 2. Hedges' g
#
# 3. Bootstrap 95% CI for Active - Passive mean difference
#
# 4. Benjamini-Hochberg FDR across the 3 prespecified bands
#
# 5. Leave-one-participant-out robustness
#
# IMPORTANT
# ---------
# - Fixed-10 dataset is frozen before inference.
# - Participant is the statistical unit.
# - No edge selection is performed.
# - No participant is excluded based on connectivity magnitude.
# - Two-sided tests are used.
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 2. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK5B0 = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05b0_fixed10_dwpli_freeze"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05b_global_dwpli_inference"
)

FIGURE_OUT = OUT / "figures"

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. SETTINGS
# =============================================================================

BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]

N_ACTIVE = 8

N_PASSIVE = 7

EXPECTED_N = 15

N_BOOTSTRAP = 10000

RANDOM_SEED = 20260913


# =============================================================================
# 4. LOAD FROZEN FIXED-10 DATA
# =============================================================================

summary_file = (
    TASK5B0
    / "fixed10_subject_global_dwpli.csv"
)

freeze_file = (
    TASK5B0
    / "analysis_freeze_summary.csv"
)


if not summary_file.exists():
    raise FileNotFoundError(summary_file)

if not freeze_file.exists():
    raise FileNotFoundError(freeze_file)


df = pd.read_csv(
    summary_file
)

freeze = pd.read_csv(
    freeze_file
)


# =============================================================================
# 5. VERIFY FREEZE
# =============================================================================

if "freeze_pass" not in freeze.columns:
    raise RuntimeError(
        "Freeze status not found."
    )


freeze_value = freeze.loc[
    0,
    "freeze_pass"
]


if isinstance(
    freeze_value,
    str
):
    freeze_pass = (
        freeze_value
        .strip()
        .lower()
        ==
        "true"
    )
else:
    freeze_pass = bool(
        freeze_value
    )


if not freeze_pass:
    raise RuntimeError(
        "Task 5B0 dataset did not pass freeze."
    )


required_columns = {
    "subject",
    "group",
    "band",
    "delta_mean_dwpli"
}


missing_columns = (
    required_columns
    -
    set(df.columns)
)


if missing_columns:
    raise RuntimeError(
        f"Missing columns: {missing_columns}"
    )


# =============================================================================
# 6. SAMPLE INTEGRITY
# =============================================================================

subjects = sorted(
    df["subject"].unique()
)


n_subjects = len(
    subjects
)


n_active = (
    df.loc[
        df["group"] == "Active",
        "subject"
    ]
    .nunique()
)


n_passive = (
    df.loc[
        df["group"] == "Passive",
        "subject"
    ]
    .nunique()
)


print(
    "=" * 80
)

print(
    "TASK 5B — EXACT PERMUTATION INFERENCE FOR GLOBAL dwPLI CHANGE"
)

print(
    "=" * 80
)

print(
    f"Input directory       : {TASK5B0}"
)

print(
    f"Output directory      : {OUT}"
)

print(
    f"Participants          : {n_subjects}/{EXPECTED_N}"
)

print(
    f"Active                : {n_active}/{N_ACTIVE}"
)

print(
    f"Passive               : {n_passive}/{N_PASSIVE}"
)

print(
    f"Bootstrap iterations  : {N_BOOTSTRAP:,}"
)


if n_subjects != EXPECTED_N:
    raise RuntimeError(
        "Expected 15 participants."
    )

if n_active != N_ACTIVE:
    raise RuntimeError(
        "Expected 8 Active participants."
    )

if n_passive != N_PASSIVE:
    raise RuntimeError(
        "Expected 7 Passive participants."
    )


# =============================================================================
# 7. HELPER — HEDGES' g
# =============================================================================

def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    n1 = len(active)
    n2 = len(passive)


    mean1 = np.mean(active)
    mean2 = np.mean(passive)


    var1 = np.var(
        active,
        ddof=1
    )

    var2 = np.var(
        passive,
        ddof=1
    )


    pooled_variance = (

        (
            (n1 - 1)
            *
            var1
        )

        +

        (
            (n2 - 1)
            *
            var2
        )

    ) / (

        n1
        +
        n2
        -
        2
    )


    pooled_sd = np.sqrt(
        pooled_variance
    )


    if pooled_sd == 0:
        return np.nan


    cohen_d = (
        mean1
        -
        mean2
    ) / pooled_sd


    dfree = (
        n1
        +
        n2
        -
        2
    )


    correction = (

        1

        -

        (
            3
            /
            (
                4
                *
                dfree
                -
                1
            )
        )
    )


    return float(
        correction
        *
        cohen_d
    )


# =============================================================================
# 8. HELPER — EXACT PERMUTATION TEST
# =============================================================================

def exact_permutation_test(
    values,
    observed_active_indices,
    n_active
):

    values = np.asarray(
        values,
        dtype=float
    )


    n_total = len(
        values
    )


    all_indices = np.arange(
        n_total
    )


    observed_active_indices = np.asarray(
        observed_active_indices,
        dtype=int
    )


    observed_passive_indices = np.setdiff1d(
        all_indices,
        observed_active_indices
    )


    observed_difference = (

        np.mean(
            values[
                observed_active_indices
            ]
        )

        -

        np.mean(
            values[
                observed_passive_indices
            ]
        )
    )


    permutation_differences = []


    for active_combination in combinations(
        range(n_total),
        n_active
    ):

        active_idx = np.asarray(
            active_combination,
            dtype=int
        )


        passive_idx = np.setdiff1d(
            all_indices,
            active_idx
        )


        difference = (

            np.mean(
                values[
                    active_idx
                ]
            )

            -

            np.mean(
                values[
                    passive_idx
                ]
            )
        )


        permutation_differences.append(
            difference
        )


    permutation_differences = np.asarray(
        permutation_differences,
        dtype=float
    )


    tolerance = 1e-15


    extreme = np.sum(

        np.abs(
            permutation_differences
        )

        >=

        (
            np.abs(
                observed_difference
            )
            -
            tolerance
        )
    )


    exact_p = (
        extreme
        /
        len(
            permutation_differences
        )
    )


    return (
        float(
            observed_difference
        ),
        float(
            exact_p
        ),
        permutation_differences
    )


# =============================================================================
# 9. HELPER — BOOTSTRAP CI
# =============================================================================

def bootstrap_mean_difference_ci(
    active,
    passive,
    n_bootstrap=10000,
    seed=20260913
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )


    differences = np.empty(
        n_bootstrap,
        dtype=float
    )


    for i in range(
        n_bootstrap
    ):

        active_sample = rng.choice(
            active,
            size=len(active),
            replace=True
        )


        passive_sample = rng.choice(
            passive,
            size=len(passive),
            replace=True
        )


        differences[i] = (

            np.mean(
                active_sample
            )

            -

            np.mean(
                passive_sample
            )
        )


    lower = np.percentile(
        differences,
        2.5
    )


    upper = np.percentile(
        differences,
        97.5
    )


    return (
        float(lower),
        float(upper)
    )


# =============================================================================
# 10. HELPER — BENJAMINI-HOCHBERG
# =============================================================================

def benjamini_hochberg(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    sorted_p = p_values[
        order
    ]


    adjusted_sorted = np.empty(
        m,
        dtype=float
    )


    running_min = 1.0


    for i in range(
        m - 1,
        -1,
        -1
    ):

        rank = (
            i
            +
            1
        )


        adjusted = (

            sorted_p[i]
            *
            m
            /
            rank
        )


        running_min = min(
            running_min,
            adjusted
        )


        adjusted_sorted[i] = min(
            running_min,
            1.0
        )


    adjusted = np.empty(
        m,
        dtype=float
    )


    adjusted[
        order
    ] = adjusted_sorted


    return adjusted


# =============================================================================
# 11. PRIMARY EXACT INFERENCE
# =============================================================================

result_rows = []

permutation_distributions = {}


for band_index, band in enumerate(
    BANDS
):

    band_df = (

        df.loc[
            df["band"] == band
        ]

        .copy()

        .sort_values(
            "subject"
        )

        .reset_index(
            drop=True
        )
    )


    if len(
        band_df
    ) != EXPECTED_N:

        raise RuntimeError(
            f"{band}: expected 15 rows."
        )


    values = band_df[
        "delta_mean_dwpli"
    ].to_numpy(
        dtype=float
    )


    active_mask = (
        band_df[
            "group"
        ]
        .to_numpy()
        ==
        "Active"
    )


    passive_mask = (
        band_df[
            "group"
        ]
        .to_numpy()
        ==
        "Passive"
    )


    active_values = values[
        active_mask
    ]


    passive_values = values[
        passive_mask
    ]


    active_indices = np.where(
        active_mask
    )[0]


    observed_difference, exact_p, permutation_values = (
        exact_permutation_test(
            values=values,
            observed_active_indices=active_indices,
            n_active=N_ACTIVE
        )
    )


    permutation_distributions[
        band
    ] = permutation_values


    ci_low, ci_high = (
        bootstrap_mean_difference_ci(

            active_values,

            passive_values,

            n_bootstrap=N_BOOTSTRAP,

            seed=(
                RANDOM_SEED
                +
                band_index
            )
        )
    )


    g = hedges_g(
        active_values,
        passive_values
    )


    result_rows.append({

        "band":
            band,

        "n_active":
            len(
                active_values
            ),

        "n_passive":
            len(
                passive_values
            ),

        "active_mean_delta_dwpli":
            float(
                np.mean(
                    active_values
                )
            ),

        "active_sd_delta_dwpli":
            float(
                np.std(
                    active_values,
                    ddof=1
                )
            ),

        "active_median_delta_dwpli":
            float(
                np.median(
                    active_values
                )
            ),

        "passive_mean_delta_dwpli":
            float(
                np.mean(
                    passive_values
                )
            ),

        "passive_sd_delta_dwpli":
            float(
                np.std(
                    passive_values,
                    ddof=1
                )
            ),

        "passive_median_delta_dwpli":
            float(
                np.median(
                    passive_values
                )
            ),

        "mean_difference_active_minus_passive":
            observed_difference,

        "bootstrap_ci_low":
            ci_low,

        "bootstrap_ci_high":
            ci_high,

        "hedges_g":
            g,

        "exact_permutation_p":
            exact_p,

        "n_exact_permutations":
            len(
                permutation_values
            )
    })


results = pd.DataFrame(
    result_rows
)


# =============================================================================
# 12. FDR ACROSS THREE PRESPECIFIED BANDS
# =============================================================================

results[
    "p_bh_fdr"
] = benjamini_hochberg(

    results[
        "exact_permutation_p"
    ].to_numpy()
)


results[
    "significant_raw_0_05"
] = (

    results[
        "exact_permutation_p"
    ]

    <
    0.05
)


results[
    "significant_fdr_0_05"
] = (

    results[
        "p_bh_fdr"
    ]

    <
    0.05
)


results.to_csv(

    OUT
    /
    "global_dwpli_exact_permutation_results.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "PRIMARY EXACT PERMUTATION RESULTS"
)

print(
    "=" * 80
)


display(
    results.round(
        6
    )
)


# =============================================================================
# 13. LEAVE-ONE-OUT ROBUSTNESS
# =============================================================================

loo_rows = []


for band in BANDS:

    band_df = (

        df.loc[
            df["band"] == band
        ]

        .copy()

        .sort_values(
            "subject"
        )

        .reset_index(
            drop=True
        )
    )


    for omitted_subject in band_df[
        "subject"
    ]:

        reduced = (

            band_df.loc[
                band_df[
                    "subject"
                ]
                !=
                omitted_subject
            ]

            .copy()

            .reset_index(
                drop=True
            )
        )


        active_values = reduced.loc[
            reduced[
                "group"
            ]
            ==
            "Active",
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        passive_values = reduced.loc[
            reduced[
                "group"
            ]
            ==
            "Passive",
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        difference = (

            np.mean(
                active_values
            )

            -

            np.mean(
                passive_values
            )
        )


        g = hedges_g(
            active_values,
            passive_values
        )


        loo_rows.append({

            "band":
                band,

            "omitted_subject":
                omitted_subject,

            "omitted_group":
                band_df.loc[
                    band_df[
                        "subject"
                    ]
                    ==
                    omitted_subject,
                    "group"
                ].iloc[0],

            "n_active":
                len(
                    active_values
                ),

            "n_passive":
                len(
                    passive_values
                ),

            "mean_difference_active_minus_passive":
                float(
                    difference
                ),

            "hedges_g":
                float(
                    g
                )
        })


loo = pd.DataFrame(
    loo_rows
)


loo.to_csv(

    OUT
    /
    "leave_one_out_robustness.csv",

    index=False
)


# =============================================================================
# 14. LEAVE-ONE-OUT SUMMARY
# =============================================================================

loo_summary_rows = []


for band in BANDS:

    primary_row = results.loc[
        results[
            "band"
        ]
        ==
        band
    ].iloc[0]


    primary_difference = float(
        primary_row[
            "mean_difference_active_minus_passive"
        ]
    )


    temp = loo.loc[
        loo[
            "band"
        ]
        ==
        band
    ].copy()


    same_direction = (

        np.sign(
            temp[
                "mean_difference_active_minus_passive"
            ]
        )

        ==

        np.sign(
            primary_difference
        )
    )


    loo_summary_rows.append({

        "band":
            band,

        "primary_difference":
            primary_difference,

        "minimum_loo_difference":
            float(
                temp[
                    "mean_difference_active_minus_passive"
                ].min()
            ),

        "maximum_loo_difference":
            float(
                temp[
                    "mean_difference_active_minus_passive"
                ].max()
            ),

        "median_loo_difference":
            float(
                temp[
                    "mean_difference_active_minus_passive"
                ].median()
            ),

        "same_direction_n":
            int(
                same_direction.sum()
            ),

        "same_direction_percent":
            float(
                100
                *
                same_direction.mean()
            ),

        "minimum_loo_hedges_g":
            float(
                temp[
                    "hedges_g"
                ].min()
            ),

        "maximum_loo_hedges_g":
            float(
                temp[
                    "hedges_g"
                ].max()
            )
    })


loo_summary = pd.DataFrame(
    loo_summary_rows
)


loo_summary.to_csv(

    OUT
    /
    "leave_one_out_summary.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "LEAVE-ONE-PARTICIPANT-OUT ROBUSTNESS"
)

print(
    "=" * 80
)


display(
    loo_summary.round(
        6
    )
)


# =============================================================================
# 15. FIGURE — PARTICIPANT-LEVEL DELTA dwPLI
# =============================================================================

rng = np.random.default_rng(
    RANDOM_SEED
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        15,
        5
    ),
    sharey=False
)


for ax, band in zip(
    axes,
    BANDS
):

    temp = df.loc[
        df[
            "band"
        ]
        ==
        band
    ].copy()


    for position, group in enumerate(
        [
            "Active",
            "Passive"
        ]
    ):

        values = temp.loc[
            temp[
                "group"
            ]
            ==
            group,
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        jitter = rng.normal(
            0,
            0.045,
            size=len(values)
        )


        ax.scatter(

            np.full(
                len(values),
                position
            )
            +
            jitter,

            values,

            s=70,

            alpha=0.8
        )


        mean_value = np.mean(
            values
        )


        ax.plot(

            [
                position - 0.18,
                position + 0.18
            ],

            [
                mean_value,
                mean_value
            ],

            linewidth=3
        )


    result = results.loc[
        results[
            "band"
        ]
        ==
        band
    ].iloc[0]


    ax.axhline(
        0,
        linewidth=0.8
    )


    ax.set_xticks(
        [
            0,
            1
        ]
    )


    ax.set_xticklabels(
        [
            "Active",
            "Passive"
        ]
    )


    ax.set_title(
        (
            f"{band}\n"
            f"g = {result['hedges_g']:.2f}, "
            f"p = {result['exact_permutation_p']:.3f}"
        )
    )


    ax.set_ylabel(
        "Mean ΔdwPLI"
    )


    ax.grid(
        axis="y",
        alpha=0.2
    )


plt.suptitle(
    "Task-related global connectivity change"
)


plt.tight_layout()


plt.savefig(

    FIGURE_OUT
    /
    "global_dwpli_group_comparison.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 16. FIGURE — EXACT PERMUTATION DISTRIBUTIONS
# =============================================================================

for band in BANDS:

    permutation_values = (
        permutation_distributions[
            band
        ]
    )


    observed = float(

        results.loc[
            results[
                "band"
            ]
            ==
            band,
            "mean_difference_active_minus_passive"
        ].iloc[0]
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            5
        )
    )


    ax.hist(
        permutation_values,
        bins=40,
        alpha=0.8
    )


    ax.axvline(
        observed,
        linewidth=2,
        label=(
            "Observed Active − Passive"
        )
    )


    ax.axvline(
        -abs(
            observed
        ),
        linestyle="--",
        linewidth=1
    )


    ax.axvline(
        abs(
            observed
        ),
        linestyle="--",
        linewidth=1
    )


    p_value = float(

        results.loc[
            results[
                "band"
            ]
            ==
            band,
            "exact_permutation_p"
        ].iloc[0]
    )


    ax.set_xlabel(
        "Permuted Active − Passive mean ΔdwPLI"
    )


    ax.set_ylabel(
        "Number of exact allocations"
    )


    ax.set_title(
        (
            f"{band} exact permutation distribution\n"
            f"p = {p_value:.4f}"
        )
    )


    ax.legend()


    ax.grid(
        axis="y",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"{band.lower()}_"
            "exact_permutation_distribution.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# 17. COMPACT MANUSCRIPT TABLE
# =============================================================================

manuscript_table = results[
    [
        "band",
        "n_active",
        "n_passive",
        "active_mean_delta_dwpli",
        "active_sd_delta_dwpli",
        "passive_mean_delta_dwpli",
        "passive_sd_delta_dwpli",
        "mean_difference_active_minus_passive",
        "bootstrap_ci_low",
        "bootstrap_ci_high",
        "hedges_g",
        "exact_permutation_p",
        "p_bh_fdr"
    ]
].copy()


manuscript_table.columns = [

    "Band",
    "n Active",
    "n Passive",
    "Active mean",
    "Active SD",
    "Passive mean",
    "Passive SD",
    "Active-Passive difference",
    "95% CI lower",
    "95% CI upper",
    "Hedges g",
    "Exact p",
    "BH-FDR p"
]


manuscript_table.to_csv(

    OUT
    /
    "table_global_dwpli_inference.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "MANUSCRIPT-READY GLOBAL CONNECTIVITY TABLE"
)

print(
    "=" * 80
)


display(
    manuscript_table.round(
        6
    )
)


# =============================================================================
# 18. FINAL INTEGRITY CHECK
# =============================================================================

expected_permutations = math.comb(
    EXPECTED_N,
    N_ACTIVE
)


permutation_valid = bool(

    np.all(

        results[
            "n_exact_permutations"
        ]

        ==

        expected_permutations
    )
)


bands_valid = (
    set(
        results[
            "band"
        ]
    )
    ==
    set(
        BANDS
    )
)


finite_results = bool(

    np.all(

        np.isfinite(

            results[
                [
                    "mean_difference_active_minus_passive",
                    "hedges_g",
                    "exact_permutation_p",
                    "p_bh_fdr"
                ]
            ].to_numpy(
                dtype=float
            )
        )
    )
)


integrity_pass = bool(

    freeze_pass
    and
    permutation_valid
    and
    bands_valid
    and
    finite_results
)


print(
    "\n"
    + "=" * 80
)

print(
    "TASK 5B COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Frozen fixed epochs/condition : 10"
)

print(
    f"Participants                  : {EXPECTED_N}"
)

print(
    f"Active                        : {N_ACTIVE}"
)

print(
    f"Passive                       : {N_PASSIVE}"
)

print(
    f"Frequency bands               : {', '.join(BANDS)}"
)

print(
    f"Exact allocations/band        : {expected_permutations}"
)

print(
    f"Bootstrap iterations          : {N_BOOTSTRAP:,}"
)

print(
    f"Multiple-comparison control   : BH-FDR across 3 bands"
)

print(
    f"Leave-one-out analyses        : {len(loo)}"
)


print(
    "\n"
    + "-" * 80
)


if integrity_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )

else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "INTERPRETATION RULE:"
)

print(
    "Group effects are interpreted from participant-level "
    "Task-minus-Baseline-A mean dwPLI."
)

print(
    "Exact permutation p-values are primary."
)

print(
    "BH-FDR controls multiplicity across Theta, Alpha, and Beta."
)

print(
    "Bootstrap confidence intervals and Hedges' g quantify "
    "effect magnitude and uncertainty."
)

print(
    "Leave-one-out results assess dependence on individual participants."
)

print(
    "No individual edge is declared significant in Task 5B."
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# =============================================================================
# TASK 5C — ROI-LEVEL NETWORK LOCALIZATION OF CONNECTIVITY CHANGE
# =============================================================================
#
# PURPOSE
# -------
# Determine whether Active-vs-Passive differences in Task-minus-Baseline
# dwPLI are spatially concentrated within or between predefined scalp ROIs.
#
# FROZEN INPUT
# ------------
# Task 5B0 fixed-10 connectivity dataset:
#   10 earliest clean matched 2-s epochs per condition
#   Baseline A / T2_V1
#   Task - Baseline delta dwPLI
#
# PREDEFINED ROIs
# ---------------
# Frontal:
#   AF3, F7, F3, F4, F8, AF4
#
# CentralTemporal:
#   FC5, T7, T8, FC6
#
# Posterior:
#   P7, O1, O2, P8
#
# NETWORK CLASSES
# ---------------
#   Frontal-Frontal
#   CentralTemporal-CentralTemporal
#   Posterior-Posterior
#   Frontal-CentralTemporal
#   Frontal-Posterior
#   CentralTemporal-Posterior
#
# INFERENCE
# ---------
# - Participant is statistical unit.
# - Mean delta dwPLI across edges in each network class.
# - Exact Active-vs-Passive permutation:
#       C(15, 8) = 6435 allocations.
# - Hedges' g.
# - 10,000 bootstrap CI for mean difference.
# - BH-FDR across all 18 band × network tests.
# - Leave-one-out direction stability.
#
# IMPORTANT
# ---------
# This is localization/follow-up analysis.
# Individual edges are NOT tested for significance here.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 2. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK5B0 = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05b0_fixed10_dwpli_freeze"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05c_roi_network_localization"
)

FIGURE_OUT = OUT / "figures"

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. SETTINGS
# =============================================================================

BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]

ROIS = {

    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "CentralTemporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}


NETWORK_CLASSES = [

    (
        "Frontal-Frontal",
        "Frontal",
        "Frontal"
    ),

    (
        "CentralTemporal-CentralTemporal",
        "CentralTemporal",
        "CentralTemporal"
    ),

    (
        "Posterior-Posterior",
        "Posterior",
        "Posterior"
    ),

    (
        "Frontal-CentralTemporal",
        "Frontal",
        "CentralTemporal"
    ),

    (
        "Frontal-Posterior",
        "Frontal",
        "Posterior"
    ),

    (
        "CentralTemporal-Posterior",
        "CentralTemporal",
        "Posterior"
    )
]


EXPECTED_CHANNELS = [

    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


N_ACTIVE = 8

N_PASSIVE = 7

EXPECTED_N = 15

EXPECTED_PERMUTATIONS = math.comb(
    EXPECTED_N,
    N_ACTIVE
)

N_BOOTSTRAP = 10000

RANDOM_SEED = 20260913


# =============================================================================
# 4. LOAD FROZEN EDGE DATA
# =============================================================================

edge_file = (
    TASK5B0
    / "fixed10_edge_level_long.csv"
)

freeze_file = (
    TASK5B0
    / "analysis_freeze_summary.csv"
)


if not edge_file.exists():
    raise FileNotFoundError(edge_file)

if not freeze_file.exists():
    raise FileNotFoundError(freeze_file)


edges = pd.read_csv(
    edge_file
)

freeze = pd.read_csv(
    freeze_file
)


# =============================================================================
# 5. VERIFY FREEZE
# =============================================================================

freeze_value = freeze.loc[
    0,
    "freeze_pass"
]


if isinstance(
    freeze_value,
    str
):
    freeze_pass = (
        freeze_value
        .strip()
        .lower()
        ==
        "true"
    )
else:
    freeze_pass = bool(
        freeze_value
    )


if not freeze_pass:
    raise RuntimeError(
        "Task 5B0 freeze did not pass."
    )


required_columns = {

    "subject",
    "group",
    "band",
    "channel_1",
    "channel_2",
    "baseline_dwpli",
    "task_dwpli",
    "delta_dwpli"
}


missing = (
    required_columns
    -
    set(
        edges.columns
    )
)


if missing:
    raise RuntimeError(
        f"Missing columns: {missing}"
    )


# =============================================================================
# 6. CHANNEL -> ROI MAP
# =============================================================================

channel_to_roi = {}


for roi_name, roi_channels in ROIS.items():

    for channel in roi_channels:

        if channel in channel_to_roi:
            raise RuntimeError(
                f"{channel} assigned to multiple ROIs."
            )

        channel_to_roi[
            channel
        ] = roi_name


if set(
    channel_to_roi.keys()
) != set(
    EXPECTED_CHANNELS
):

    raise RuntimeError(
        "ROI definitions do not cover exactly the 14 expected channels."
    )


# =============================================================================
# 7. NETWORK CLASS LOOKUP
# =============================================================================

def network_class_for_edge(
    channel_1,
    channel_2
):

    roi_1 = channel_to_roi[
        channel_1
    ]

    roi_2 = channel_to_roi[
        channel_2
    ]


    for (
        class_name,
        roi_a,
        roi_b
    ) in NETWORK_CLASSES:

        if roi_a == roi_b:

            if (
                roi_1 == roi_a
                and
                roi_2 == roi_b
            ):
                return class_name

        else:

            if (
                (
                    roi_1 == roi_a
                    and
                    roi_2 == roi_b
                )
                or
                (
                    roi_1 == roi_b
                    and
                    roi_2 == roi_a
                )
            ):
                return class_name


    raise RuntimeError(
        f"No network class for {channel_1}-{channel_2}"
    )


edges[
    "network_class"
] = [

    network_class_for_edge(
        ch1,
        ch2
    )

    for ch1, ch2 in zip(
        edges[
            "channel_1"
        ],
        edges[
            "channel_2"
        ]
    )
]


# =============================================================================
# 8. EXPECTED EDGE COUNTS PER NETWORK CLASS
# =============================================================================

expected_edge_counts = {

    "Frontal-Frontal":
        math.comb(
            len(
                ROIS[
                    "Frontal"
                ]
            ),
            2
        ),

    "CentralTemporal-CentralTemporal":
        math.comb(
            len(
                ROIS[
                    "CentralTemporal"
                ]
            ),
            2
        ),

    "Posterior-Posterior":
        math.comb(
            len(
                ROIS[
                    "Posterior"
                ]
            ),
            2
        ),

    "Frontal-CentralTemporal":
        (
            len(
                ROIS[
                    "Frontal"
                ]
            )
            *
            len(
                ROIS[
                    "CentralTemporal"
                ]
            )
        ),

    "Frontal-Posterior":
        (
            len(
                ROIS[
                    "Frontal"
                ]
            )
            *
            len(
                ROIS[
                    "Posterior"
                ]
            )
        ),

    "CentralTemporal-Posterior":
        (
            len(
                ROIS[
                    "CentralTemporal"
                ]
            )
            *
            len(
                ROIS[
                    "Posterior"
                ]
            )
        )
}


print(
    "=" * 80
)

print(
    "TASK 5C — ROI-LEVEL NETWORK LOCALIZATION"
)

print(
    "=" * 80
)

print(
    f"Input directory      : {TASK5B0}"
)

print(
    f"Output directory     : {OUT}"
)

print(
    f"Participants         : {edges['subject'].nunique()}/15"
)

print(
    f"Edge rows            : {len(edges)}"
)

print(
    f"Exact permutations   : {EXPECTED_PERMUTATIONS}"
)

print(
    "\nExpected edges per network class:"
)


for network, count in expected_edge_counts.items():

    print(
        f"  {network:<38s}: {count}"
    )


print(
    f"\nTotal edges: "
    f"{sum(expected_edge_counts.values())}"
)


if sum(
    expected_edge_counts.values()
) != 91:

    raise RuntimeError(
        "Network classes do not partition all 91 edges."
    )


# =============================================================================
# 9. VERIFY OBSERVED EDGE CLASS COUNTS
# =============================================================================

observed_counts = (

    edges[
        [
            "subject",
            "band",
            "network_class",
            "channel_1",
            "channel_2"
        ]
    ]

    .groupby(
        [
            "subject",
            "band",
            "network_class"
        ]
    )

    .size()

    .reset_index(
        name="n_edges"
    )
)


for _, row in observed_counts.iterrows():

    expected = expected_edge_counts[
        row[
            "network_class"
        ]
    ]


    if int(
        row[
            "n_edges"
        ]
    ) != expected:

        raise RuntimeError(

            f"Edge-count mismatch: "
            f"{row['subject']} "
            f"{row['band']} "
            f"{row['network_class']} "
            f"observed={row['n_edges']} "
            f"expected={expected}"
        )


# =============================================================================
# 10. PARTICIPANT-LEVEL NETWORK SUMMARIES
# =============================================================================

subject_network = (

    edges

    .groupby(
        [
            "subject",
            "group",
            "band",
            "network_class"
        ]
    )

    .agg(

        n_edges=(
            "delta_dwpli",
            "size"
        ),

        baseline_mean_dwpli=(
            "baseline_dwpli",
            "mean"
        ),

        task_mean_dwpli=(
            "task_dwpli",
            "mean"
        ),

        delta_mean_dwpli=(
            "delta_dwpli",
            "mean"
        ),

        delta_median_dwpli=(
            "delta_dwpli",
            "median"
        )
    )

    .reset_index()
)


subject_network.to_csv(

    OUT
    /
    "subject_network_dwpli_summary.csv",

    index=False
)


expected_rows = (
    EXPECTED_N
    *
    len(BANDS)
    *
    len(NETWORK_CLASSES)
)


if len(
    subject_network
) != expected_rows:

    raise RuntimeError(

        f"Expected {expected_rows} participant-network rows; "
        f"found {len(subject_network)}."
    )


# =============================================================================
# 11. DESCRIPTIVE GROUP SUMMARY
# =============================================================================

group_summary = (

    subject_network

    .groupby(
        [
            "group",
            "band",
            "network_class"
        ]
    )

    .agg(

        n_subjects=(
            "subject",
            "nunique"
        ),

        mean_baseline_dwpli=(
            "baseline_mean_dwpli",
            "mean"
        ),

        sd_baseline_dwpli=(
            "baseline_mean_dwpli",
            "std"
        ),

        mean_task_dwpli=(
            "task_mean_dwpli",
            "mean"
        ),

        sd_task_dwpli=(
            "task_mean_dwpli",
            "std"
        ),

        mean_delta_dwpli=(
            "delta_mean_dwpli",
            "mean"
        ),

        sd_delta_dwpli=(
            "delta_mean_dwpli",
            "std"
        )
    )

    .reset_index()
)


group_summary.to_csv(

    OUT
    /
    "network_descriptive_group_summary.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "NETWORK DESCRIPTIVE GROUP SUMMARY"
)

print(
    "=" * 80
)


display(
    group_summary.round(
        6
    )
)


# =============================================================================
# 12. HEDGES' g
# =============================================================================

def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    n1 = len(active)
    n2 = len(passive)


    var1 = np.var(
        active,
        ddof=1
    )

    var2 = np.var(
        passive,
        ddof=1
    )


    pooled_variance = (

        (
            (n1 - 1)
            *
            var1
        )

        +

        (
            (n2 - 1)
            *
            var2
        )

    ) / (

        n1
        +
        n2
        -
        2
    )


    pooled_sd = np.sqrt(
        pooled_variance
    )


    if pooled_sd == 0:
        return np.nan


    d = (

        np.mean(
            active
        )

        -

        np.mean(
            passive
        )

    ) / pooled_sd


    dfree = (
        n1
        +
        n2
        -
        2
    )


    correction = (

        1

        -

        (
            3

            /

            (
                4
                *
                dfree
                -
                1
            )
        )
    )


    return float(
        correction
        *
        d
    )


# =============================================================================
# 13. EXACT PERMUTATION TEST
# =============================================================================

def exact_permutation_test(
    values,
    labels
):

    values = np.asarray(
        values,
        dtype=float
    )

    labels = np.asarray(
        labels
    )


    active_indices = np.where(
        labels
        ==
        "Active"
    )[0]


    passive_indices = np.where(
        labels
        ==
        "Passive"
    )[0]


    observed = (

        np.mean(
            values[
                active_indices
            ]
        )

        -

        np.mean(
            values[
                passive_indices
            ]
        )
    )


    all_indices = np.arange(
        len(values)
    )


    permutation_values = []


    for active_combo in combinations(
        range(
            len(values)
        ),
        N_ACTIVE
    ):

        active_idx = np.asarray(
            active_combo,
            dtype=int
        )


        passive_idx = np.setdiff1d(
            all_indices,
            active_idx
        )


        difference = (

            np.mean(
                values[
                    active_idx
                ]
            )

            -

            np.mean(
                values[
                    passive_idx
                ]
            )
        )


        permutation_values.append(
            difference
        )


    permutation_values = np.asarray(
        permutation_values,
        dtype=float
    )


    tolerance = 1e-15


    p_value = np.mean(

        np.abs(
            permutation_values
        )

        >=

        (
            np.abs(
                observed
            )
            -
            tolerance
        )
    )


    return (
        float(
            observed
        ),
        float(
            p_value
        ),
        permutation_values
    )


# =============================================================================
# 14. BOOTSTRAP CI
# =============================================================================

def bootstrap_ci(
    active,
    passive,
    seed
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )


    boot = np.empty(
        N_BOOTSTRAP,
        dtype=float
    )


    for i in range(
        N_BOOTSTRAP
    ):

        active_sample = rng.choice(
            active,
            len(active),
            replace=True
        )


        passive_sample = rng.choice(
            passive,
            len(passive),
            replace=True
        )


        boot[i] = (

            np.mean(
                active_sample
            )

            -

            np.mean(
                passive_sample
            )
        )


    return (

        float(
            np.percentile(
                boot,
                2.5
            )
        ),

        float(
            np.percentile(
                boot,
                97.5
            )
        )
    )


# =============================================================================
# 15. BH-FDR
# =============================================================================

def benjamini_hochberg(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    sorted_p = p_values[
        order
    ]


    adjusted_sorted = np.empty(
        m,
        dtype=float
    )


    running_min = 1.0


    for i in range(
        m - 1,
        -1,
        -1
    ):

        rank = (
            i
            +
            1
        )


        candidate = (

            sorted_p[i]
            *
            m
            /
            rank
        )


        running_min = min(
            running_min,
            candidate
        )


        adjusted_sorted[i] = min(
            running_min,
            1.0
        )


    adjusted = np.empty(
        m,
        dtype=float
    )


    adjusted[
        order
    ] = adjusted_sorted


    return adjusted


# =============================================================================
# 16. EXACT NETWORK INFERENCE
# =============================================================================

result_rows = []

test_number = 0


for band in BANDS:

    for (
        network_class,
        _,
        _
    ) in NETWORK_CLASSES:

        test_number += 1


        temp = (

            subject_network.loc[
                (
                    subject_network[
                        "band"
                    ]
                    ==
                    band
                )
                &
                (
                    subject_network[
                        "network_class"
                    ]
                    ==
                    network_class
                )
            ]

            .copy()

            .sort_values(
                "subject"
            )

            .reset_index(
                drop=True
            )
        )


        if len(
            temp
        ) != EXPECTED_N:

            raise RuntimeError(

                f"{band} {network_class}: "
                f"expected 15 subjects."
            )


        values = temp[
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        labels = temp[
            "group"
        ].to_numpy()


        active = values[
            labels
            ==
            "Active"
        ]


        passive = values[
            labels
            ==
            "Passive"
        ]


        difference, p_value, permutations = (
            exact_permutation_test(
                values,
                labels
            )
        )


        ci_low, ci_high = bootstrap_ci(

            active,

            passive,

            seed=(
                RANDOM_SEED
                +
                test_number
            )
        )


        result_rows.append({

            "band":
                band,

            "network_class":
                network_class,

            "n_edges":
                expected_edge_counts[
                    network_class
                ],

            "n_active":
                len(active),

            "n_passive":
                len(passive),

            "active_mean_delta_dwpli":
                float(
                    np.mean(
                        active
                    )
                ),

            "active_sd_delta_dwpli":
                float(
                    np.std(
                        active,
                        ddof=1
                    )
                ),

            "passive_mean_delta_dwpli":
                float(
                    np.mean(
                        passive
                    )
                ),

            "passive_sd_delta_dwpli":
                float(
                    np.std(
                        passive,
                        ddof=1
                    )
                ),

            "active_minus_passive":
                difference,

            "bootstrap_ci_low":
                ci_low,

            "bootstrap_ci_high":
                ci_high,

            "hedges_g":
                hedges_g(
                    active,
                    passive
                ),

            "exact_permutation_p":
                p_value,

            "n_exact_permutations":
                len(
                    permutations
                )
        })


results = pd.DataFrame(
    result_rows
)


# =============================================================================
# 17. FDR ACROSS ALL 18 LOCALIZATION TESTS
# =============================================================================

results[
    "p_bh_fdr_18"
] = benjamini_hochberg(

    results[
        "exact_permutation_p"
    ].to_numpy(
        dtype=float
    )
)


results[
    "significant_raw_0_05"
] = (

    results[
        "exact_permutation_p"
    ]

    <
    0.05
)


results[
    "significant_fdr_0_05"
] = (

    results[
        "p_bh_fdr_18"
    ]

    <
    0.05
)


results.to_csv(

    OUT
    /
    "network_exact_permutation_results.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "NETWORK EXACT PERMUTATION RESULTS"
)

print(
    "=" * 80
)


display(
    results.round(
        6
    )
)


# =============================================================================
# 18. RANK RESULTS
# =============================================================================

ranked = (

    results

    .sort_values(
        [
            "exact_permutation_p",
            "p_bh_fdr_18"
        ]
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "RANKED NETWORK LOCALIZATION RESULTS"
)

print(
    "=" * 80
)


display(

    ranked[
        [
            "band",
            "network_class",
            "active_mean_delta_dwpli",
            "passive_mean_delta_dwpli",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_fdr_18"
        ]
    ].round(
        6
    )
)


ranked.to_csv(

    OUT
    /
    "ranked_network_localization_results.csv",

    index=False
)


# =============================================================================
# 19. LEAVE-ONE-OUT DIRECTION ROBUSTNESS
# =============================================================================

loo_rows = []


for _, result in results.iterrows():

    band = result[
        "band"
    ]

    network_class = result[
        "network_class"
    ]


    primary_difference = float(
        result[
            "active_minus_passive"
        ]
    )


    temp = (

        subject_network.loc[
            (
                subject_network[
                    "band"
                ]
                ==
                band
            )
            &
            (
                subject_network[
                    "network_class"
                ]
                ==
                network_class
            )
        ]

        .copy()

        .sort_values(
            "subject"
        )
    )


    loo_differences = []

    loo_effects = []


    for omitted_subject in temp[
        "subject"
    ]:

        reduced = temp.loc[
            temp[
                "subject"
            ]
            !=
            omitted_subject
        ]


        active = reduced.loc[
            reduced[
                "group"
            ]
            ==
            "Active",
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        passive = reduced.loc[
            reduced[
                "group"
            ]
            ==
            "Passive",
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        difference = (

            np.mean(
                active
            )

            -

            np.mean(
                passive
            )
        )


        effect = hedges_g(
            active,
            passive
        )


        loo_differences.append(
            difference
        )

        loo_effects.append(
            effect
        )


    loo_differences = np.asarray(
        loo_differences
    )


    loo_effects = np.asarray(
        loo_effects
    )


    same_direction = (

        np.sign(
            loo_differences
        )

        ==

        np.sign(
            primary_difference
        )
    )


    loo_rows.append({

        "band":
            band,

        "network_class":
            network_class,

        "primary_difference":
            primary_difference,

        "minimum_loo_difference":
            float(
                np.min(
                    loo_differences
                )
            ),

        "maximum_loo_difference":
            float(
                np.max(
                    loo_differences
                )
            ),

        "median_loo_difference":
            float(
                np.median(
                    loo_differences
                )
            ),

        "same_direction_n":
            int(
                np.sum(
                    same_direction
                )
            ),

        "same_direction_percent":
            float(
                100
                *
                np.mean(
                    same_direction
                )
            ),

        "minimum_loo_hedges_g":
            float(
                np.min(
                    loo_effects
                )
            ),

        "maximum_loo_hedges_g":
            float(
                np.max(
                    loo_effects
                )
            )
    })


loo_summary = pd.DataFrame(
    loo_rows
)


loo_summary.to_csv(

    OUT
    /
    "network_leave_one_out_summary.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "NETWORK LEAVE-ONE-OUT ROBUSTNESS"
)

print(
    "=" * 80
)


display(
    loo_summary.round(
        6
    )
)


# =============================================================================
# 20. BETA-ONLY FOLLOW-UP VIEW
# =============================================================================

beta_results = (

    results.loc[
        results[
            "band"
        ]
        ==
        "Beta"
    ]

    .copy()

    .sort_values(
        "active_minus_passive"
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "BETA NETWORK LOCALIZATION — FOLLOW-UP VIEW"
)

print(
    "=" * 80
)


display(

    beta_results[
        [
            "network_class",
            "n_edges",
            "active_mean_delta_dwpli",
            "passive_mean_delta_dwpli",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_fdr_18"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 21. FIGURE — EFFECT SIZE BY NETWORK CLASS
# =============================================================================

network_order = [
    x[0]
    for x in NETWORK_CLASSES
]


for band in BANDS:

    temp = (

        results.loc[
            results[
                "band"
            ]
            ==
            band
        ]

        .set_index(
            "network_class"
        )

        .loc[
            network_order
        ]

        .reset_index()
    )


    fig, ax = plt.subplots(
        figsize=(
            10,
            6
        )
    )


    y = np.arange(
        len(
            temp
        )
    )


    ax.barh(
        y,
        temp[
            "hedges_g"
        ]
    )


    ax.axvline(
        0,
        linewidth=1
    )


    ax.set_yticks(
        y
    )


    ax.set_yticklabels(
        temp[
            "network_class"
        ]
    )


    ax.set_xlabel(
        "Hedges' g (Active − Passive)"
    )


    ax.set_title(
        f"{band}: ROI-network connectivity-change effect sizes"
    )


    ax.grid(
        axis="x",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"{band.lower()}_"
            "network_effect_sizes.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# 22. FIGURE — BETA PARTICIPANT VALUES BY NETWORK CLASS
# =============================================================================

beta_subject = subject_network.loc[
    subject_network[
        "band"
    ]
    ==
    "Beta"
].copy()


rng = np.random.default_rng(
    RANDOM_SEED
)


fig, ax = plt.subplots(
    figsize=(
        14,
        7
    )
)


base_positions = np.arange(
    len(
        network_order
    )
) * 3


for network_index, network_class in enumerate(
    network_order
):

    temp = beta_subject.loc[
        beta_subject[
            "network_class"
        ]
        ==
        network_class
    ]


    for group_offset, group in enumerate(
        [
            "Active",
            "Passive"
        ]
    ):

        values = temp.loc[
            temp[
                "group"
            ]
            ==
            group,
            "delta_mean_dwpli"
        ].to_numpy(
            dtype=float
        )


        position = (
            base_positions[
                network_index
            ]
            +
            group_offset
        )


        jitter = rng.normal(
            0,
            0.045,
            len(
                values
            )
        )


        ax.scatter(

            np.full(
                len(
                    values
                ),
                position
            )
            +
            jitter,

            values,

            s=55,

            alpha=0.8
        )


        mean_value = np.mean(
            values
        )


        ax.plot(

            [
                position - 0.18,
                position + 0.18
            ],

            [
                mean_value,
                mean_value
            ],

            linewidth=3
        )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    base_positions
    +
    0.5
)


ax.set_xticklabels(
    network_order,
    rotation=25,
    ha="right"
)


ax.set_ylabel(
    "Beta mean ΔdwPLI"
)


ax.set_title(
    "Beta connectivity change by predefined network class"
)


ax.grid(
    axis="y",
    alpha=0.2
)


plt.tight_layout()


plt.savefig(

    FIGURE_OUT
    /
    "beta_participant_network_localization.png",

    dpi=300,

    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 23. MANUSCRIPT-READY TABLE
# =============================================================================

manuscript_table = results[
    [
        "band",
        "network_class",
        "n_edges",
        "n_active",
        "n_passive",
        "active_mean_delta_dwpli",
        "active_sd_delta_dwpli",
        "passive_mean_delta_dwpli",
        "passive_sd_delta_dwpli",
        "active_minus_passive",
        "bootstrap_ci_low",
        "bootstrap_ci_high",
        "hedges_g",
        "exact_permutation_p",
        "p_bh_fdr_18"
    ]
].copy()


manuscript_table.columns = [

    "Band",
    "Network class",
    "Edges",
    "n Active",
    "n Passive",
    "Active mean",
    "Active SD",
    "Passive mean",
    "Passive SD",
    "Active-Passive difference",
    "95% CI lower",
    "95% CI upper",
    "Hedges g",
    "Exact p",
    "BH-FDR p"
]


manuscript_table.to_csv(

    OUT
    /
    "table_network_localization.csv",

    index=False
)


# =============================================================================
# 24. FINAL INTEGRITY
# =============================================================================

subjects_valid = (
    subject_network[
        "subject"
    ].nunique()
    ==
    15
)


groups_valid = (

    subject_network.loc[
        subject_network[
            "group"
        ]
        ==
        "Active",
        "subject"
    ].nunique()
    ==
    8

    and

    subject_network.loc[
        subject_network[
            "group"
        ]
        ==
        "Passive",
        "subject"
    ].nunique()
    ==
    7
)


rows_valid = (
    len(
        subject_network
    )
    ==
    270
)


tests_valid = (
    len(
        results
    )
    ==
    18
)


permutations_valid = bool(

    np.all(

        results[
            "n_exact_permutations"
        ]

        ==

        EXPECTED_PERMUTATIONS
    )
)


finite_valid = bool(

    np.all(

        np.isfinite(

            results[
                [
                    "active_minus_passive",
                    "hedges_g",
                    "exact_permutation_p",
                    "p_bh_fdr_18"
                ]
            ].to_numpy(
                dtype=float
            )
        )
    )
)


integrity_pass = bool(

    freeze_pass
    and
    subjects_valid
    and
    groups_valid
    and
    rows_valid
    and
    tests_valid
    and
    permutations_valid
    and
    finite_valid
)


print(
    "\n"
    + "=" * 80
)

print(
    "TASK 5C COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participants                 : "
    f"{subject_network['subject'].nunique()}/15"
)

print(
    "Active                       : 8/8"
)

print(
    "Passive                      : 7/7"
)

print(
    f"Frequency bands              : "
    f"{', '.join(BANDS)}"
)

print(
    f"Network classes              : "
    f"{len(NETWORK_CLASSES)}"
)

print(
    f"Participant-network rows     : "
    f"{len(subject_network)}/270"
)

print(
    f"Localization tests           : "
    f"{len(results)}/18"
)

print(
    f"Exact allocations/test       : "
    f"{EXPECTED_PERMUTATIONS}"
)

print(
    f"Bootstrap iterations/test    : "
    f"{N_BOOTSTRAP:,}"
)

print(
    "Multiplicity family          : "
    "18 band × network tests"
)

print(
    "Individual edge inference    : none"
)


print(
    "\n"
    + "-" * 80
)


if integrity_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )

else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "INTERPRETATION:"
)

print(
    "Task 5C localizes participant-level connectivity-change "
    "differences to predefined within- and between-ROI edge classes."
)

print(
    "Exact permutation p-values are inferential; "
    "BH-FDR controls the full 18-test localization family."
)

print(
    "Beta results are interpreted as follow-up localization of "
    "the stable global beta pattern observed in Task 5B."
)

print(
    "No single EEG edge is declared significant from this task."
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# =============================================================================
# TASK 6A — WEIGHTED NETWORK TOPOLOGY:
#             NODE STRENGTH AND RELATIVE NODE STRENGTH
# =============================================================================
#
# PURPOSE
# -------
# Test whether Active and Passive engagement differ in the spatial
# organization of weighted dwPLI networks.
#
# INPUT
# -----
# Frozen Task 5B0 fixed-10 edge-level connectivity.
#
# METRICS
# -------
# 1. Absolute node strength:
#       S_i = sum_j dwPLI_ij
#
# 2. Task-related node-strength change:
#       Delta S_i = S_i(task) - S_i(baseline)
#
# 3. Relative node strength:
#       R_i = S_i / sum_k S_k
#
# 4. Relative node-strength change:
#       Delta R_i = R_i(task) - R_i(baseline)
#
# ROI summaries:
#   Frontal
#   CentralTemporal
#   Posterior
#
# INFERENCE
# ---------
# Participant is the statistical unit.
#
# Exact Active-vs-Passive permutation:
#       C(15,8) = 6435
#
# Hedges' g
# 10,000 bootstrap CI
#
# BH-FDR families are kept SEPARATE:
#   - 9 tests for absolute node-strength change
#     (3 bands x 3 ROIs)
#   - 9 tests for relative node-strength change
#     (3 bands x 3 ROIs)
#
# No individual electrode is tested inferentially.
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 2. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK5B0 = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task05b0_fixed10_dwpli_freeze"
)

OUT = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task06a_weighted_network_topology"
)

FIGURE_OUT = OUT / "figures"

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. SETTINGS
# =============================================================================

BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]

ROIS = {

    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "CentralTemporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}

CHANNELS = [

    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]

EXPECTED_N = 15
N_ACTIVE = 8
N_PASSIVE = 7

EXPECTED_PERMUTATIONS = math.comb(
    EXPECTED_N,
    N_ACTIVE
)

N_BOOTSTRAP = 10000
RANDOM_SEED = 20260913


# =============================================================================
# 4. LOAD FROZEN EDGE DATA
# =============================================================================

edge_file = (
    TASK5B0
    / "fixed10_edge_level_long.csv"
)

freeze_file = (
    TASK5B0
    / "analysis_freeze_summary.csv"
)

if not edge_file.exists():
    raise FileNotFoundError(edge_file)

if not freeze_file.exists():
    raise FileNotFoundError(freeze_file)


edges = pd.read_csv(edge_file)
freeze = pd.read_csv(freeze_file)


# =============================================================================
# 5. VERIFY FREEZE
# =============================================================================

freeze_value = freeze.loc[
    0,
    "freeze_pass"
]

if isinstance(freeze_value, str):

    freeze_pass = (
        freeze_value
        .strip()
        .lower()
        ==
        "true"
    )

else:

    freeze_pass = bool(
        freeze_value
    )


if not freeze_pass:

    raise RuntimeError(
        "Task 5B0 freeze did not pass."
    )


required_columns = {

    "subject",
    "group",
    "band",
    "channel_1",
    "channel_2",
    "baseline_dwpli",
    "task_dwpli"
}

missing = (
    required_columns
    -
    set(edges.columns)
)

if missing:

    raise RuntimeError(
        f"Missing columns: {missing}"
    )


# =============================================================================
# 6. CHANNEL -> ROI MAP
# =============================================================================

channel_to_roi = {}

for roi, roi_channels in ROIS.items():

    for channel in roi_channels:

        channel_to_roi[
            channel
        ] = roi


if set(channel_to_roi) != set(CHANNELS):

    raise RuntimeError(
        "ROI mapping does not cover exactly the expected 14 channels."
    )


# =============================================================================
# 7. RECONSTRUCT NODE STRENGTHS
# =============================================================================

node_rows = []

for (
    subject,
    group,
    band
), temp in edges.groupby(
    [
        "subject",
        "group",
        "band"
    ]
):

    baseline_strength = {
        ch: 0.0
        for ch in CHANNELS
    }

    task_strength = {
        ch: 0.0
        for ch in CHANNELS
    }


    for _, row in temp.iterrows():

        ch1 = row[
            "channel_1"
        ]

        ch2 = row[
            "channel_2"
        ]

        baseline_value = float(
            row[
                "baseline_dwpli"
            ]
        )

        task_value = float(
            row[
                "task_dwpli"
            ]
        )


        baseline_strength[
            ch1
        ] += baseline_value

        baseline_strength[
            ch2
        ] += baseline_value


        task_strength[
            ch1
        ] += task_value

        task_strength[
            ch2
        ] += task_value


    total_baseline_strength = sum(
        baseline_strength.values()
    )

    total_task_strength = sum(
        task_strength.values()
    )


    if total_baseline_strength <= 0:

        raise RuntimeError(
            f"{subject} {band}: non-positive baseline total strength."
        )

    if total_task_strength <= 0:

        raise RuntimeError(
            f"{subject} {band}: non-positive task total strength."
        )


    for channel in CHANNELS:

        s_base = baseline_strength[
            channel
        ]

        s_task = task_strength[
            channel
        ]


        r_base = (
            s_base
            /
            total_baseline_strength
        )

        r_task = (
            s_task
            /
            total_task_strength
        )


        node_rows.append({

            "subject":
                subject,

            "group":
                group,

            "band":
                band,

            "channel":
                channel,

            "roi":
                channel_to_roi[
                    channel
                ],

            "baseline_strength":
                s_base,

            "task_strength":
                s_task,

            "delta_strength":
                (
                    s_task
                    -
                    s_base
                ),

            "baseline_relative_strength":
                r_base,

            "task_relative_strength":
                r_task,

            "delta_relative_strength":
                (
                    r_task
                    -
                    r_base
                )
        })


node_df = pd.DataFrame(
    node_rows
)


# =============================================================================
# 8. NODE-LEVEL INTEGRITY
# =============================================================================

expected_node_rows = (
    EXPECTED_N
    *
    len(BANDS)
    *
    len(CHANNELS)
)

if len(node_df) != expected_node_rows:

    raise RuntimeError(

        f"Expected {expected_node_rows} node rows; "
        f"found {len(node_df)}."
    )


# Relative node strengths must sum to 1 for every network.

relative_check = (

    node_df

    .groupby(
        [
            "subject",
            "band"
        ]
    )

    .agg(

        baseline_sum=(
            "baseline_relative_strength",
            "sum"
        ),

        task_sum=(
            "task_relative_strength",
            "sum"
        )
    )

    .reset_index()
)


if not np.allclose(
    relative_check[
        "baseline_sum"
    ],
    1.0
):

    raise RuntimeError(
        "Baseline relative strengths do not sum to 1."
    )


if not np.allclose(
    relative_check[
        "task_sum"
    ],
    1.0
):

    raise RuntimeError(
        "Task relative strengths do not sum to 1."
    )


node_df.to_csv(

    OUT
    /
    "participant_node_strengths.csv",

    index=False
)


# =============================================================================
# 9. ROI-LEVEL TOPOLOGY
# =============================================================================

roi_df = (

    node_df

    .groupby(
        [
            "subject",
            "group",
            "band",
            "roi"
        ]
    )

    .agg(

        n_nodes=(
            "channel",
            "size"
        ),

        baseline_mean_strength=(
            "baseline_strength",
            "mean"
        ),

        task_mean_strength=(
            "task_strength",
            "mean"
        ),

        delta_mean_strength=(
            "delta_strength",
            "mean"
        ),

        baseline_mean_relative_strength=(
            "baseline_relative_strength",
            "mean"
        ),

        task_mean_relative_strength=(
            "task_relative_strength",
            "mean"
        ),

        delta_mean_relative_strength=(
            "delta_relative_strength",
            "mean"
        )
    )

    .reset_index()
)


expected_roi_rows = (
    EXPECTED_N
    *
    len(BANDS)
    *
    len(ROIS)
)

if len(roi_df) != expected_roi_rows:

    raise RuntimeError(

        f"Expected {expected_roi_rows} ROI rows; "
        f"found {len(roi_df)}."
    )


roi_df.to_csv(

    OUT
    /
    "participant_roi_topology.csv",

    index=False
)


# =============================================================================
# 10. EFFECT SIZE
# =============================================================================

def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )

    n1 = len(active)
    n2 = len(passive)

    var1 = np.var(
        active,
        ddof=1
    )

    var2 = np.var(
        passive,
        ddof=1
    )

    pooled_variance = (

        (
            (n1 - 1)
            *
            var1
        )

        +

        (
            (n2 - 1)
            *
            var2
        )

    ) / (
        n1
        +
        n2
        -
        2
    )


    if pooled_variance <= 0:

        return np.nan


    pooled_sd = np.sqrt(
        pooled_variance
    )


    d = (

        np.mean(active)
        -
        np.mean(passive)

    ) / pooled_sd


    dfree = (
        n1
        +
        n2
        -
        2
    )


    correction = (

        1

        -

        (
            3
            /
            (
                4
                *
                dfree
                -
                1
            )
        )
    )


    return float(
        correction
        *
        d
    )


# =============================================================================
# 11. EXACT PERMUTATION
# =============================================================================

def exact_permutation_test(
    values,
    labels
):

    values = np.asarray(
        values,
        dtype=float
    )

    labels = np.asarray(
        labels
    )


    active_idx = np.where(
        labels
        ==
        "Active"
    )[0]

    passive_idx = np.where(
        labels
        ==
        "Passive"
    )[0]


    observed = (

        np.mean(
            values[
                active_idx
            ]
        )

        -

        np.mean(
            values[
                passive_idx
            ]
        )
    )


    all_idx = np.arange(
        len(values)
    )

    permuted = []


    for active_combo in combinations(
        range(
            len(values)
        ),
        N_ACTIVE
    ):

        a = np.asarray(
            active_combo,
            dtype=int
        )

        p = np.setdiff1d(
            all_idx,
            a
        )


        permuted.append(

            np.mean(
                values[a]
            )

            -

            np.mean(
                values[p]
            )
        )


    permuted = np.asarray(
        permuted,
        dtype=float
    )


    p_value = np.mean(

        np.abs(
            permuted
        )

        >=

        (
            np.abs(
                observed
            )
            -
            1e-15
        )
    )


    return (
        float(observed),
        float(p_value),
        len(permuted)
    )


# =============================================================================
# 12. BOOTSTRAP CI
# =============================================================================

def bootstrap_ci(
    active,
    passive,
    seed
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        N_BOOTSTRAP
    )


    for i in range(
        N_BOOTSTRAP
    ):

        a = rng.choice(
            active,
            len(active),
            replace=True
        )

        p = rng.choice(
            passive,
            len(passive),
            replace=True
        )


        boot[i] = (
            np.mean(a)
            -
            np.mean(p)
        )


    return (

        float(
            np.percentile(
                boot,
                2.5
            )
        ),

        float(
            np.percentile(
                boot,
                97.5
            )
        )
    )


# =============================================================================
# 13. BH-FDR
# =============================================================================

def benjamini_hochberg(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(p_values)

    order = np.argsort(
        p_values
    )

    sorted_p = p_values[
        order
    ]

    adjusted_sorted = np.empty(
        m
    )

    running_min = 1.0


    for i in range(
        m - 1,
        -1,
        -1
    ):

        rank = i + 1

        candidate = (
            sorted_p[i]
            *
            m
            /
            rank
        )

        running_min = min(
            running_min,
            candidate
        )

        adjusted_sorted[i] = min(
            running_min,
            1.0
        )


    adjusted = np.empty(
        m
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted


# =============================================================================
# 14. INFERENCE FUNCTION
# =============================================================================

def run_inference(
    dataframe,
    value_column,
    metric_name
):

    rows = []

    test_index = 0


    for band in BANDS:

        for roi in ROIS:

            test_index += 1


            temp = (

                dataframe.loc[
                    (
                        dataframe[
                            "band"
                        ]
                        ==
                        band
                    )
                    &
                    (
                        dataframe[
                            "roi"
                        ]
                        ==
                        roi
                    )
                ]

                .copy()

                .sort_values(
                    "subject"
                )

                .reset_index(
                    drop=True
                )
            )


            if len(temp) != EXPECTED_N:

                raise RuntimeError(

                    f"{band} {roi}: "
                    "expected 15 participants."
                )


            values = temp[
                value_column
            ].to_numpy(
                dtype=float
            )

            labels = temp[
                "group"
            ].to_numpy()


            active = values[
                labels
                ==
                "Active"
            ]

            passive = values[
                labels
                ==
                "Passive"
            ]


            difference, p_value, n_perm = (
                exact_permutation_test(
                    values,
                    labels
                )
            )


            ci_low, ci_high = bootstrap_ci(

                active,
                passive,

                RANDOM_SEED
                +
                test_index
            )


            rows.append({

                "metric":
                    metric_name,

                "band":
                    band,

                "roi":
                    roi,

                "n_active":
                    len(active),

                "n_passive":
                    len(passive),

                "active_mean":
                    float(
                        np.mean(active)
                    ),

                "active_sd":
                    float(
                        np.std(
                            active,
                            ddof=1
                        )
                    ),

                "passive_mean":
                    float(
                        np.mean(passive)
                    ),

                "passive_sd":
                    float(
                        np.std(
                            passive,
                            ddof=1
                        )
                    ),

                "active_minus_passive":
                    difference,

                "bootstrap_ci_low":
                    ci_low,

                "bootstrap_ci_high":
                    ci_high,

                "hedges_g":
                    hedges_g(
                        active,
                        passive
                    ),

                "exact_permutation_p":
                    p_value,

                "n_exact_permutations":
                    n_perm
            })


    output = pd.DataFrame(
        rows
    )


    output[
        "p_bh_fdr_9"
    ] = benjamini_hochberg(

        output[
            "exact_permutation_p"
        ].to_numpy()
    )


    output[
        "significant_raw_0_05"
    ] = (

        output[
            "exact_permutation_p"
        ]
        <
        0.05
    )


    output[
        "significant_fdr_0_05"
    ] = (

        output[
            "p_bh_fdr_9"
        ]
        <
        0.05
    )


    return output


# =============================================================================
# 15. ABSOLUTE NODE-STRENGTH CHANGE
# =============================================================================

absolute_results = run_inference(

    roi_df,

    "delta_mean_strength",

    "Absolute node-strength change"
)


# =============================================================================
# 16. RELATIVE NODE-STRENGTH CHANGE
# =============================================================================

relative_results = run_inference(

    roi_df,

    "delta_mean_relative_strength",

    "Relative node-strength change"
)


absolute_results.to_csv(

    OUT
    /
    "absolute_node_strength_inference.csv",

    index=False
)


relative_results.to_csv(

    OUT
    /
    "relative_node_strength_inference.csv",

    index=False
)


# =============================================================================
# 17. PRINT ABSOLUTE RESULTS
# =============================================================================

print(
    "=" * 80
)

print(
    "TASK 6A — WEIGHTED NETWORK TOPOLOGY"
)

print(
    "=" * 80
)

print(
    f"Participants       : {EXPECTED_N}"
)

print(
    f"Active             : {N_ACTIVE}"
)

print(
    f"Passive            : {N_PASSIVE}"
)

print(
    f"Channels           : {len(CHANNELS)}"
)

print(
    f"Bands              : {', '.join(BANDS)}"
)

print(
    f"Exact permutations : {EXPECTED_PERMUTATIONS}"
)


print(
    "\n"
    + "=" * 80
)

print(
    "ABSOLUTE NODE-STRENGTH CHANGE"
)

print(
    "=" * 80
)


display(
    absolute_results.round(
        6
    )
)


# =============================================================================
# 18. PRINT RELATIVE RESULTS
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "RELATIVE NODE-STRENGTH CHANGE"
)

print(
    "=" * 80
)


display(
    relative_results.round(
        6
    )
)


# =============================================================================
# 19. RANKED RESULTS
# =============================================================================

absolute_ranked = (

    absolute_results

    .sort_values(
        "exact_permutation_p"
    )

    .reset_index(
        drop=True
    )
)


relative_ranked = (

    relative_results

    .sort_values(
        "exact_permutation_p"
    )

    .reset_index(
        drop=True
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "RANKED ABSOLUTE TOPOLOGY RESULTS"
)

print(
    "=" * 80
)


display(

    absolute_ranked[
        [
            "band",
            "roi",
            "active_mean",
            "passive_mean",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_fdr_9"
        ]
    ].round(
        6
    )
)


print(
    "\n"
    + "=" * 80
)

print(
    "RANKED RELATIVE TOPOLOGY RESULTS"
)

print(
    "=" * 80
)


display(

    relative_ranked[
        [
            "band",
            "roi",
            "active_mean",
            "passive_mean",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_fdr_9"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 20. FIGURE — ABSOLUTE EFFECT SIZES
# =============================================================================

roi_order = [
    "Frontal",
    "CentralTemporal",
    "Posterior"
]


for band in BANDS:

    temp = (

        absolute_results.loc[
            absolute_results[
                "band"
            ]
            ==
            band
        ]

        .set_index(
            "roi"
        )

        .loc[
            roi_order
        ]

        .reset_index()
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            5
        )
    )


    y = np.arange(
        len(temp)
    )


    ax.barh(
        y,
        temp[
            "hedges_g"
        ]
    )


    ax.axvline(
        0,
        linewidth=1
    )


    ax.set_yticks(
        y
    )

    ax.set_yticklabels(
        temp[
            "roi"
        ]
    )


    ax.set_xlabel(
        "Hedges' g (Active − Passive)"
    )


    ax.set_title(
        f"{band}: absolute node-strength change"
    )


    ax.grid(
        axis="x",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"{band.lower()}_"
            "absolute_node_strength_effects.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# 21. FIGURE — RELATIVE EFFECT SIZES
# =============================================================================

for band in BANDS:

    temp = (

        relative_results.loc[
            relative_results[
                "band"
            ]
            ==
            band
        ]

        .set_index(
            "roi"
        )

        .loc[
            roi_order
        ]

        .reset_index()
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            5
        )
    )


    y = np.arange(
        len(temp)
    )


    ax.barh(
        y,
        temp[
            "hedges_g"
        ]
    )


    ax.axvline(
        0,
        linewidth=1
    )


    ax.set_yticks(
        y
    )

    ax.set_yticklabels(
        temp[
            "roi"
        ]
    )


    ax.set_xlabel(
        "Hedges' g (Active − Passive)"
    )


    ax.set_title(
        f"{band}: relative node-strength redistribution"
    )


    ax.grid(
        axis="x",
        alpha=0.2
    )


    plt.tight_layout()


    plt.savefig(

        FIGURE_OUT
        /
        (
            f"{band.lower()}_"
            "relative_node_strength_effects.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# 22. BETA FOLLOW-UP TABLE
# =============================================================================

beta_absolute = absolute_results.loc[
    absolute_results[
        "band"
    ]
    ==
    "Beta"
].copy()


beta_relative = relative_results.loc[
    relative_results[
        "band"
    ]
    ==
    "Beta"
].copy()


beta_followup = pd.concat(
    [
        beta_absolute,
        beta_relative
    ],
    ignore_index=True
)


beta_followup.to_csv(

    OUT
    /
    "beta_topology_followup.csv",

    index=False
)


print(
    "\n"
    + "=" * 80
)

print(
    "BETA TOPOLOGY FOLLOW-UP"
)

print(
    "=" * 80
)


display(

    beta_followup[
        [
            "metric",
            "roi",
            "active_mean",
            "passive_mean",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_fdr_9"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 23. FINAL INTEGRITY CHECK
# =============================================================================

node_rows_valid = (
    len(node_df)
    ==
    630
)

roi_rows_valid = (
    len(roi_df)
    ==
    135
)

absolute_tests_valid = (
    len(absolute_results)
    ==
    9
)

relative_tests_valid = (
    len(relative_results)
    ==
    9
)

permutation_valid = bool(

    np.all(
        absolute_results[
            "n_exact_permutations"
        ]
        ==
        EXPECTED_PERMUTATIONS
    )

    and

    np.all(
        relative_results[
            "n_exact_permutations"
        ]
        ==
        EXPECTED_PERMUTATIONS
    )
)


finite_valid = bool(

    np.all(

        np.isfinite(

            absolute_results[
                [
                    "active_minus_passive",
                    "hedges_g",
                    "exact_permutation_p",
                    "p_bh_fdr_9"
                ]
            ].to_numpy(
                dtype=float
            )
        )
    )

    and

    np.all(

        np.isfinite(

            relative_results[
                [
                    "active_minus_passive",
                    "hedges_g",
                    "exact_permutation_p",
                    "p_bh_fdr_9"
                ]
            ].to_numpy(
                dtype=float
            )
        )
    )
)


integrity_pass = bool(

    freeze_pass
    and
    node_rows_valid
    and
    roi_rows_valid
    and
    absolute_tests_valid
    and
    relative_tests_valid
    and
    permutation_valid
    and
    finite_valid
)


print(
    "\n"
    + "=" * 80
)

print(
    "TASK 6A COMPLETE"
)

print(
    "=" * 80
)


print(
    f"Participant-node rows       : "
    f"{len(node_df)}/630"
)

print(
    f"Participant-ROI rows        : "
    f"{len(roi_df)}/135"
)

print(
    f"Absolute topology tests     : "
    f"{len(absolute_results)}/9"
)

print(
    f"Relative topology tests     : "
    f"{len(relative_results)}/9"
)

print(
    f"Exact allocations/test      : "
    f"{EXPECTED_PERMUTATIONS}"
)

print(
    f"Bootstrap iterations/test   : "
    f"{N_BOOTSTRAP:,}"
)

print(
    "Absolute FDR family         : 9 tests"
)

print(
    "Relative FDR family         : 9 tests"
)

print(
    "Electrode-level inference   : none"
)


print(
    "\n"
    + "-" * 80
)


if integrity_pass:

    print(
        "OVERALL INTEGRITY CHECK: PASS"
    )

else:

    print(
        "OVERALL INTEGRITY CHECK: REVIEW REQUIRED"
    )


print(
    "\n"
    + "-" * 80
)


print(
    "INTERPRETATION:"
)

print(
    "Absolute node-strength change reflects regional changes "
    "that may partly inherit global connectivity magnitude."
)

print(
    "Relative node-strength change tests redistribution of "
    "connectivity across scalp regions after normalizing each "
    "participant's total network strength."
)

print(
    "Beta topology is interpreted as follow-up characterization "
    "of the Task 5B/5C beta connectivity pattern."
)

print(
    "No individual electrode is declared significant."
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print(
    "\n"
    + "=" * 80
)

In [ ]:
# =============================================================================
# TASK 6B — DIAGNOSTIC: FIND TASK 3B PHASE TABLE AND INSPECT SCHEMA
# =============================================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

TASK3B = (
    ROOT
    / "derivatives"
    / "tine_analysis"
    / "task03b_phase_reconstruction"
)

print("=" * 80)
print("TASK 3B CSV DISCOVERY")
print("=" * 80)

print(f"\nDirectory:\n{TASK3B}")

if not TASK3B.exists():
    raise FileNotFoundError(
        f"\nTask 3B directory does not exist:\n{TASK3B}"
    )

csv_files = sorted(
    TASK3B.rglob("*.csv")
)

print(f"\nCSV files found: {len(csv_files)}")

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found under:\n{TASK3B}"
    )

inspection_rows = []

for i, path in enumerate(csv_files, start=1):

    print("\n" + "-" * 80)
    print(f"[{i}] {path.name}")
    print(f"Path: {path}")

    try:
        df = pd.read_csv(path)

        print(f"Shape: {df.shape}")
        print("Columns:")

        for col in df.columns:
            print(f"  - {repr(col)}")

        print("\nFirst 3 rows:")
        display(df.head(3))

        inspection_rows.append({
            "number": i,
            "filename": path.name,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": " | ".join(
                map(str, df.columns)
            )
        })

    except Exception as exc:

        print(f"READ ERROR: {exc}")

        inspection_rows.append({
            "number": i,
            "filename": path.name,
            "rows": None,
            "columns": None,
            "column_names": f"ERROR: {exc}"
        })


print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

inspection_df = pd.DataFrame(
    inspection_rows
)

display(inspection_df)

print("\nLook for the CSV containing rows such as:")
print("sub-001")
print("T2_V1")
print("T2_V3")
print("T1_V1")
print("T1_V4")
print("\nand phase start/end times.")

In [ ]:
# =============================================================================
# TASK 6B — DIAGNOSTIC: WHY DID FIXED-9 EXTRACTION FAIL?
# SELF-CONTAINED
# =============================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import mne

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

DERIVATIVES = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK3A = (
    DERIVATIVES
    / "task03a_batch_preprocessing"
)

TASK3B = (
    DERIVATIVES
    / "task03b_phase_reconstruction"
)


PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)


# =============================================================================
# SETTINGS
# =============================================================================

CHANNELS = [
    "AF3", "F7", "F3", "FC5",
    "T7", "P7", "O1", "O2",
    "P8", "T8", "FC6", "F4",
    "F8", "AF4"
]

SFREQ_EXPECTED = 256.0

EPOCH_SECONDS = 2.0

FIXED_N = 9

ABS_THRESHOLD_UV = 250.0

PTP_THRESHOLD_UV = 500.0


SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 16)
]


# =============================================================================
# LOAD PHASE TABLE
# =============================================================================

if not PHASE_FILE.exists():
    raise FileNotFoundError(
        f"Phase file not found:\n{PHASE_FILE}"
    )

phase_df = pd.read_csv(PHASE_FILE)


print("=" * 100)
print("TASK 6B FIXED-9 EXTRACTION DIAGNOSTIC")
print("=" * 100)

print(f"\nTask 3A directory:\n{TASK3A}")
print(f"\nTask 3B phase file:\n{PHASE_FILE}")
print(f"\nPhase rows: {len(phase_df)}")

print("\nPhase-table columns:")
print(list(phase_df.columns))


# =============================================================================
# FIRST: SHOW EVERY FIF FILE ACTUALLY AVAILABLE
# =============================================================================

print("\n" + "=" * 100)
print("AVAILABLE FIF FILES UNDER TINE ANALYSIS")
print("=" * 100)

all_fifs = sorted(
    DERIVATIVES.rglob("*.fif")
)

print(f"\nTotal FIF files found: {len(all_fifs)}")

for p in all_fifs:
    print(p)


# =============================================================================
# HELPER: FIND ALL FIF CANDIDATES FOR ONE SUBJECT
# =============================================================================

def find_subject_fifs(subject):

    return sorted([
        p
        for p in all_fifs
        if subject.lower() in str(p).lower()
    ])


# =============================================================================
# HELPER: PHASE LOOKUP
# =============================================================================

def get_phase(subject, position, expected_code):

    temp = phase_df.loc[
        (
            phase_df["subject"].astype(str)
            ==
            subject
        )
        &
        (
            pd.to_numeric(
                phase_df["marker_position"],
                errors="coerce"
            )
            ==
            position
        )
    ].copy()

    if len(temp) != 1:
        raise RuntimeError(
            f"phase lookup: position {position}, "
            f"expected one row, found {len(temp)}"
        )

    row = temp.iloc[0]

    observed = str(
        row["marker_code"]
    )

    if observed != expected_code:
        raise RuntimeError(
            f"phase lookup: position {position} "
            f"expected {expected_code}, "
            f"found {observed}"
        )

    return row


# =============================================================================
# HELPER: COUNT CLEAN EPOCHS WITHOUT REQUIRING NINE
# =============================================================================

def inspect_phase_epochs(
    raw,
    start_sec,
    end_sec
):

    sfreq = float(
        raw.info["sfreq"]
    )

    start_sec = float(start_sec)
    end_sec = float(end_sec)

    duration = (
        end_sec
        -
        start_sec
    )

    # Only first 30 s, consistent with the intended sensitivity analysis.
    duration_used = min(
        duration,
        30.0
    )

    n_possible = int(
        np.floor(
            duration_used
            /
            EPOCH_SECONDS
        )
    )

    rows = []

    clean_data = []

    for epoch_index in range(n_possible):

        epoch_start_sec = (
            start_sec
            +
            epoch_index
            *
            EPOCH_SECONDS
        )

        epoch_end_sec = (
            epoch_start_sec
            +
            EPOCH_SECONDS
        )

        start_sample = int(
            round(
                epoch_start_sec
                *
                sfreq
            )
        )

        stop_sample = int(
            start_sample
            +
            round(
                EPOCH_SECONDS
                *
                sfreq
            )
        )

        if stop_sample > raw.n_times:
            rows.append({
                "epoch_index": epoch_index,
                "start_sec": epoch_start_sec,
                "end_sec": epoch_end_sec,
                "status": "OUTSIDE_RAW",
                "max_abs_uv": np.nan,
                "max_ptp_uv": np.nan
            })
            continue

        data = raw.get_data(
            picks=CHANNELS,
            start=start_sample,
            stop=stop_sample
        )

        expected_samples = int(
            round(
                EPOCH_SECONDS
                *
                sfreq
            )
        )

        if data.shape != (
            len(CHANNELS),
            expected_samples
        ):
            rows.append({
                "epoch_index": epoch_index,
                "start_sec": epoch_start_sec,
                "end_sec": epoch_end_sec,
                "status": f"BAD_SHAPE_{data.shape}",
                "max_abs_uv": np.nan,
                "max_ptp_uv": np.nan
            })
            continue

        data_uv = (
            data
            *
            1e6
        )

        max_abs_uv = float(
            np.max(
                np.abs(data_uv)
            )
        )

        max_ptp_uv = float(
            np.max(
                np.ptp(
                    data_uv,
                    axis=1
                )
            )
        )

        clean = (
            max_abs_uv
            <=
            ABS_THRESHOLD_UV
            and
            max_ptp_uv
            <=
            PTP_THRESHOLD_UV
        )

        rows.append({
            "epoch_index": epoch_index,
            "start_sec": epoch_start_sec,
            "end_sec": epoch_end_sec,
            "status": (
                "CLEAN"
                if clean
                else "REJECT"
            ),
            "max_abs_uv": max_abs_uv,
            "max_ptp_uv": max_ptp_uv
        })

        if clean:
            clean_data.append(data)

    return (
        pd.DataFrame(rows),
        clean_data
    )


# =============================================================================
# SUBJECT-BY-SUBJECT DIAGNOSTIC
# =============================================================================

summary_rows = []

detailed_failures = []


for subject in SUBJECTS:

    print("\n" + "-" * 100)
    print(subject)
    print("-" * 100)

    group = (
        "Active"
        if int(subject.split("-")[1]) <= 8
        else "Passive"
    )

    task_code = (
        "T1_V1"
        if group == "Active"
        else "T1_V4"
    )

    subject_result = {
        "subject": subject,
        "group": group,
        "fif_candidates": 0,
        "selected_fif": "",
        "sfreq": np.nan,
        "n_times": np.nan,
        "duration_sec": np.nan,
        "baseline_a_clean": np.nan,
        "baseline_b_clean": np.nan,
        "task_clean": np.nan,
        "status": ""
    }

    try:

        # ---------------------------------------------------------------------
        # FIF DISCOVERY
        # ---------------------------------------------------------------------

        candidates = find_subject_fifs(
            subject
        )

        subject_result[
            "fif_candidates"
        ] = len(candidates)

        print(
            f"FIF candidates: {len(candidates)}"
        )

        for candidate in candidates:
            print(
                f"    {candidate}"
            )

        if len(candidates) == 0:
            raise FileNotFoundError(
                "No FIF candidate found anywhere under "
                "derivatives/tine_analysis."
            )

        # Prefer Task 3A files explicitly.
        task3a_candidates = [
            p
            for p in candidates
            if TASK3A in p.parents
        ]

        if len(task3a_candidates) > 0:
            candidates_to_use = task3a_candidates
        else:
            candidates_to_use = candidates

        # Prefer names containing primary / filtered.
        preferred = [
            p
            for p in candidates_to_use
            if (
                "primary" in p.name.lower()
                or
                "filtered" in p.name.lower()
            )
        ]

        if len(preferred) > 0:
            candidates_to_use = preferred

        selected_fif = sorted(
            candidates_to_use
        )[0]

        subject_result[
            "selected_fif"
        ] = str(selected_fif)

        print(
            f"\nSelected FIF:\n    {selected_fif}"
        )

        # ---------------------------------------------------------------------
        # LOAD RAW
        # ---------------------------------------------------------------------

        raw = mne.io.read_raw_fif(
            selected_fif,
            preload=True,
            verbose="ERROR"
        )

        sfreq = float(
            raw.info["sfreq"]
        )

        duration_sec = (
            raw.n_times
            /
            sfreq
        )

        subject_result[
            "sfreq"
        ] = sfreq

        subject_result[
            "n_times"
        ] = raw.n_times

        subject_result[
            "duration_sec"
        ] = duration_sec

        print(
            f"Sampling rate : {sfreq:.3f} Hz"
        )

        print(
            f"Samples       : {raw.n_times}"
        )

        print(
            f"Duration      : {duration_sec:.3f} s"
        )

        print(
            f"Channels      : {len(raw.ch_names)}"
        )

        print(
            f"Channel names : {raw.ch_names}"
        )

        missing_channels = [
            ch
            for ch in CHANNELS
            if ch not in raw.ch_names
        ]

        if missing_channels:
            raise RuntimeError(
                f"Missing EEG channels: {missing_channels}"
            )

        # ---------------------------------------------------------------------
        # PHASE LOOKUPS
        # ---------------------------------------------------------------------

        phase_a = get_phase(
            subject,
            1,
            "T2_V1"
        )

        phase_b = get_phase(
            subject,
            3,
            "T2_V3"
        )

        phase_task = get_phase(
            subject,
            5,
            task_code
        )

        print("\nSelected phases:")

        print(
            "  Baseline A : "
            f"{float(phase_a['start_sec']):.3f} -> "
            f"{float(phase_a['end_sec']):.3f} s "
            f"({float(phase_a['duration_sec']):.3f} s)"
        )

        print(
            "  Baseline B : "
            f"{float(phase_b['start_sec']):.3f} -> "
            f"{float(phase_b['end_sec']):.3f} s "
            f"({float(phase_b['duration_sec']):.3f} s)"
        )

        print(
            "  Task       : "
            f"{float(phase_task['start_sec']):.3f} -> "
            f"{float(phase_task['end_sec']):.3f} s "
            f"({float(phase_task['duration_sec']):.3f} s)"
        )

        # ---------------------------------------------------------------------
        # EPOCH INSPECTION
        # ---------------------------------------------------------------------

        a_table, a_clean = inspect_phase_epochs(
            raw,
            phase_a["start_sec"],
            phase_a["end_sec"]
        )

        b_table, b_clean = inspect_phase_epochs(
            raw,
            phase_b["start_sec"],
            phase_b["end_sec"]
        )

        task_table, task_clean = inspect_phase_epochs(
            raw,
            phase_task["start_sec"],
            phase_task["end_sec"]
        )

        subject_result[
            "baseline_a_clean"
        ] = len(a_clean)

        subject_result[
            "baseline_b_clean"
        ] = len(b_clean)

        subject_result[
            "task_clean"
        ] = len(task_clean)

        print("\nClean epochs in first 30 s:")

        print(
            f"  Baseline A : "
            f"{len(a_clean)}/{len(a_table)}"
        )

        print(
            f"  Baseline B : "
            f"{len(b_clean)}/{len(b_table)}"
        )

        print(
            f"  Task       : "
            f"{len(task_clean)}/{len(task_table)}"
        )

        # Show why epochs were rejected.
        for condition, table in [
            ("Baseline A", a_table),
            ("Baseline B", b_table),
            ("Task", task_table)
        ]:

            print(
                f"\n{condition} epoch status:"
            )

            if len(table) == 0:
                print(
                    "  NO COMPLETE 2-S EPOCHS"
                )
            else:
                print(
                    table[
                        "status"
                    ].value_counts(
                        dropna=False
                    ).to_string()
                )

                rejected = table.loc[
                    table["status"] != "CLEAN"
                ]

                if len(rejected) > 0:
                    print(
                        "\n  First rejected epochs:"
                    )

                    display(
                        rejected.head(5)
                    )

        if (
            len(a_clean) >= FIXED_N
            and
            len(b_clean) >= FIXED_N
            and
            len(task_clean) >= FIXED_N
        ):
            subject_result[
                "status"
            ] = "PASS_FIXED9"

            print(
                "\nSTATUS: PASS_FIXED9"
            )

        else:
            subject_result[
                "status"
            ] = "INSUFFICIENT_FIXED9"

            print(
                "\nSTATUS: INSUFFICIENT_FIXED9"
            )

    except Exception as exc:

        subject_result[
            "status"
        ] = (
            "ERROR: "
            +
            repr(exc)
        )

        detailed_failures.append({
            "subject": subject,
            "error": repr(exc)
        })

        print(
            f"\nERROR: {repr(exc)}"
        )

    summary_rows.append(
        subject_result
    )


# =============================================================================
# SUMMARY
# =============================================================================

summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 100)
print("DIAGNOSTIC SUMMARY")
print("=" * 100)

display(
    summary_df
)


print("\n" + "=" * 100)
print("STATUS COUNTS")
print("=" * 100)

print(
    summary_df[
        "status"
    ].value_counts(
        dropna=False
    )
)


# =============================================================================
# PHASE-TABLE REFERENCE COUNTS
# =============================================================================

print("\n" + "=" * 100)
print("TASK 3B REPORTED CLEAN-EPOCH COUNTS FOR SELECTED PHASES")
print("=" * 100)

reference_rows = []

for subject in SUBJECTS:

    group = (
        "Active"
        if int(subject.split("-")[1]) <= 8
        else "Passive"
    )

    task_code = (
        "T1_V1"
        if group == "Active"
        else "T1_V4"
    )

    for condition, position, code in [
        ("Baseline A", 1, "T2_V1"),
        ("Baseline B", 3, "T2_V3"),
        ("Task", 5, task_code)
    ]:

        try:

            row = get_phase(
                subject,
                position,
                code
            )

            reference_rows.append({
                "subject": subject,
                "group": group,
                "condition": condition,
                "marker_position": position,
                "marker_code": code,
                "duration_sec": row["duration_sec"],
                "task3b_complete_2s_epochs": row["complete_2s_epochs"],
                "task3b_clean_2s_epochs": row["clean_2s_epochs"],
                "task3b_rejected_2s_epochs": row["rejected_2s_epochs"],
                "task3b_clean_percent": row["clean_percent"]
            })

        except Exception as exc:

            reference_rows.append({
                "subject": subject,
                "group": group,
                "condition": condition,
                "marker_position": position,
                "marker_code": code,
                "duration_sec": np.nan,
                "task3b_complete_2s_epochs": np.nan,
                "task3b_clean_2s_epochs": np.nan,
                "task3b_rejected_2s_epochs": np.nan,
                "task3b_clean_percent": np.nan,
                "error": repr(exc)
            })


reference_df = pd.DataFrame(
    reference_rows
)

display(
    reference_df
)


# =============================================================================
# SAVE DIAGNOSTIC
# =============================================================================

OUT = (
    DERIVATIVES
    /
    "task06b_baseline_sensitivity_dwpli"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

summary_df.to_csv(
    OUT
    /
    "fixed9_extraction_diagnostic.csv",
    index=False
)

reference_df.to_csv(
    OUT
    /
    "fixed9_task3b_reference_counts.csv",
    index=False
)


if detailed_failures:

    pd.DataFrame(
        detailed_failures
    ).to_csv(
        OUT
        /
        "fixed9_diagnostic_errors.csv",
        index=False
    )


print("\n" + "=" * 100)
print("DIAGNOSTIC COMPLETE")
print("=" * 100)

print(
    "\nDo NOT change the fixed-9 criterion yet."
)

print(
    "This diagnostic determines whether the failure is due to "
    "FIF discovery, channel naming, phase lookup, time/sample alignment, "
    "or epoch QC."
)

print(
    f"\nSaved to:\n{OUT}"
)

In [ ]:
# =============================================================================
# TASK 6B — BASELINE-B CONNECTIVITY SENSITIVITY
# Validated spectral connectivity:
# debiased squared weighted phase-lag index (wPLI²_debiased)
#
# PURPOSE
# -------
# Recompute the connectivity analysis using Baseline B (T2_V3, marker
# position 3) instead of primary Baseline A.
#
# Every participant contributes exactly:
#     9 clean Baseline-B epochs × 2 s = 18 s
#     9 clean Task epochs       × 2 s = 18 s
#
# Task definition:
#     Active  (sub-001 ... sub-008): marker position 5, T1_V1
#     Passive (sub-009 ... sub-015): marker position 5, T1_V4
#
# Connectivity:
#     MNE-Connectivity spectral_connectivity_epochs
#     method = 'wpli2_debiased'
#     mode   = 'multitaper'
#     mt_bandwidth = 2 Hz
#
# Bands:
#     Theta 4–8 Hz
#     Alpha 8–13 Hz
#     Beta  13–30 Hz
#
# Inference:
#     Subject-level global connectivity change = Task - Baseline B
#     Exact Active/Passive permutation test: C(15,8) = 6435 allocations
#     Hedges g
#     10,000 stratified bootstrap CI
#     BH-FDR across 3 bands
#
# IMPORTANT
# ---------
# Negative wPLI²_debiased estimates are valid and are NOT clipped.
# No individual edge inference is performed here.
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


try:
    from mne_connectivity import spectral_connectivity_epochs
except ImportError as exc:
    raise ImportError(
        "\nMNE-Connectivity is required for Task 6B.\n"
        "Run this in a separate Jupyter cell:\n\n"
        "    %pip install mne-connectivity\n\n"
        "Then restart the kernel and rerun Task 6B."
    ) from exc


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

DERIVATIVES = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

# IMPORTANT:
# This is the actual frozen Task 3A directory identified by the diagnostic.
TASK3A = (
    DERIVATIVES
    / "task03a_batch_preprocessing_qc"
)

TASK3B = (
    DERIVATIVES
    / "task03b_phase_reconstruction"
)

PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)

OUT = (
    DERIVATIVES
    / "task06b_baseline_b_connectivity_sensitivity"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. ANALYSIS SETTINGS
# =============================================================================

CHANNELS = [
    "AF3", "F7", "F3", "FC5",
    "T7", "P7", "O1", "O2",
    "P8", "T8", "FC6", "F4",
    "F8", "AF4"
]

BANDS = {
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0)
}

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 16)
]

ACTIVE_SUBJECTS = SUBJECTS[:8]
PASSIVE_SUBJECTS = SUBJECTS[8:]

SFREQ_EXPECTED = 256.0

EPOCH_SECONDS = 2.0

# Fixed by the diagnostic:
FIXED_N = 9

ABS_THRESHOLD_UV = 250.0
PTP_THRESHOLD_UV = 500.0

MAX_PHASE_SECONDS = 30.0

MT_BANDWIDTH = 2.0

N_BOOTSTRAP = 10_000

RANDOM_SEED = 20260913


# =============================================================================
# 3. LOAD AND VALIDATE PHASE TABLE
# =============================================================================

if not PHASE_FILE.exists():
    raise FileNotFoundError(
        f"Task 3B phase table not found:\n{PHASE_FILE}"
    )

phase_df = pd.read_csv(
    PHASE_FILE
)

required_phase_columns = {
    "subject",
    "marker_position",
    "marker_code",
    "start_sec",
    "end_sec",
    "duration_sec"
}

missing_phase_columns = (
    required_phase_columns
    -
    set(phase_df.columns)
)

if missing_phase_columns:
    raise RuntimeError(
        "Task 3B phase table is missing columns: "
        f"{sorted(missing_phase_columns)}"
    )


# =============================================================================
# 4. HELPERS
# =============================================================================

def group_for_subject(subject):

    number = int(
        subject.split("-")[1]
    )

    return (
        "Active"
        if number <= 8
        else "Passive"
    )


def task_code_for_subject(subject):

    return (
        "T1_V1"
        if group_for_subject(subject) == "Active"
        else "T1_V4"
    )


def exact_task3a_fif(subject):

    group = group_for_subject(subject)

    task = (
        "activeai"
        if group == "Active"
        else "passiveai"
    )

    fif = (
        TASK3A
        / subject
        / (
            f"{subject}_task-{task}"
            "_desc-primary_annotated_eeg_raw.fif"
        )
    )

    if not fif.exists():
        raise FileNotFoundError(
            f"Frozen Task 3A FIF not found:\n{fif}"
        )

    return fif


def get_phase(
    subject,
    marker_position,
    marker_code
):

    temp = phase_df.loc[
        (
            phase_df["subject"].astype(str)
            ==
            subject
        )
        &
        (
            pd.to_numeric(
                phase_df["marker_position"],
                errors="coerce"
            )
            ==
            marker_position
        )
        &
        (
            phase_df["marker_code"].astype(str)
            ==
            marker_code
        )
    ].copy()

    if len(temp) != 1:
        raise RuntimeError(
            f"{subject}: expected exactly one "
            f"{marker_code} phase at marker position "
            f"{marker_position}; found {len(temp)}."
        )

    return temp.iloc[0]


def extract_clean_epochs(
    raw,
    start_sec,
    end_sec,
    fixed_n=FIXED_N
):

    sfreq = float(
        raw.info["sfreq"]
    )

    start_sec = float(start_sec)
    end_sec = float(end_sec)

    available_duration = (
        end_sec
        -
        start_sec
    )

    usable_duration = min(
        available_duration,
        MAX_PHASE_SECONDS
    )

    n_possible = int(
        np.floor(
            usable_duration
            /
            EPOCH_SECONDS
        )
    )

    epoch_samples = int(
        round(
            EPOCH_SECONDS
            *
            sfreq
        )
    )

    clean_epochs = []
    audit_rows = []

    for epoch_index in range(n_possible):

        epoch_start_sec = (
            start_sec
            +
            epoch_index
            *
            EPOCH_SECONDS
        )

        epoch_end_sec = (
            epoch_start_sec
            +
            EPOCH_SECONDS
        )

        start_sample = int(
            round(
                epoch_start_sec
                *
                sfreq
            )
        )

        stop_sample = (
            start_sample
            +
            epoch_samples
        )

        if stop_sample > raw.n_times:

            audit_rows.append({
                "epoch_index": epoch_index,
                "start_sec": epoch_start_sec,
                "end_sec": epoch_end_sec,
                "status": "OUTSIDE_RAW",
                "max_abs_uv": np.nan,
                "max_ptp_uv": np.nan
            })

            continue

        data = raw.get_data(
            picks=CHANNELS,
            start=start_sample,
            stop=stop_sample
        )

        if data.shape != (
            len(CHANNELS),
            epoch_samples
        ):

            audit_rows.append({
                "epoch_index": epoch_index,
                "start_sec": epoch_start_sec,
                "end_sec": epoch_end_sec,
                "status": "BAD_SHAPE",
                "max_abs_uv": np.nan,
                "max_ptp_uv": np.nan
            })

            continue

        data_uv = (
            data
            *
            1e6
        )

        max_abs_uv = float(
            np.max(
                np.abs(data_uv)
            )
        )

        max_ptp_uv = float(
            np.max(
                np.ptp(
                    data_uv,
                    axis=1
                )
            )
        )

        clean = (
            max_abs_uv
            <=
            ABS_THRESHOLD_UV
            and
            max_ptp_uv
            <=
            PTP_THRESHOLD_UV
        )

        audit_rows.append({
            "epoch_index": epoch_index,
            "start_sec": epoch_start_sec,
            "end_sec": epoch_end_sec,
            "status": (
                "CLEAN"
                if clean
                else "REJECT"
            ),
            "max_abs_uv": max_abs_uv,
            "max_ptp_uv": max_ptp_uv
        })

        if clean:
            clean_epochs.append(
                data.copy()
            )

    if len(clean_epochs) < fixed_n:
        raise RuntimeError(
            f"Only {len(clean_epochs)} clean epochs available; "
            f"{fixed_n} required."
        )

    # Earliest FIXED_N clean epochs.
    clean_epochs = np.stack(
        clean_epochs[:fixed_n],
        axis=0
    )

    return (
        clean_epochs,
        pd.DataFrame(audit_rows)
    )


def compute_wpli2_debiased(
    epochs_data,
    sfreq,
    fmin,
    fmax
):

    n_channels = epochs_data.shape[1]

    upper = np.triu_indices(
        n_channels,
        k=1
    )

    con = spectral_connectivity_epochs(
        epochs_data,
        method="wpli2_debiased",
        indices=upper,
        sfreq=sfreq,
        mode="multitaper",
        fmin=fmin,
        fmax=fmax,
        faverage=True,
        mt_bandwidth=MT_BANDWIDTH,
        mt_adaptive=False,
        mt_low_bias=True,
        verbose=False
    )

    values = np.asarray(
        con.get_data()
    ).squeeze()

    values = np.ravel(
        values
    )

    expected_edges = (
        n_channels
        *
        (
            n_channels - 1
        )
        //
        2
    )

    if len(values) != expected_edges:
        raise RuntimeError(
            "Unexpected connectivity output shape. "
            f"Expected {expected_edges} edges, "
            f"obtained {len(values)} values. "
            f"Raw shape={np.asarray(con.get_data()).shape}"
        )

    matrix = np.zeros(
        (
            n_channels,
            n_channels
        ),
        dtype=float
    )

    matrix[upper] = values

    matrix[
        (
            upper[1],
            upper[0]
        )
    ] = values

    np.fill_diagonal(
        matrix,
        0.0
    )

    return (
        matrix,
        values,
        upper
    )


def global_connectivity(matrix):

    upper = np.triu_indices(
        matrix.shape[0],
        k=1
    )

    return float(
        np.mean(
            matrix[upper]
        )
    )


def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )

    n1 = len(active)
    n2 = len(passive)

    s1 = np.std(
        active,
        ddof=1
    )

    s2 = np.std(
        passive,
        ddof=1
    )

    pooled_variance = (
        (
            (n1 - 1)
            *
            s1**2
        )
        +
        (
            (n2 - 1)
            *
            s2**2
        )
    ) / (
        n1
        +
        n2
        -
        2
    )

    if pooled_variance <= 0:
        return np.nan

    pooled_sd = np.sqrt(
        pooled_variance
    )

    d = (
        np.mean(active)
        -
        np.mean(passive)
    ) / pooled_sd

    df = (
        n1
        +
        n2
        -
        2
    )

    correction = (
        1.0
        -
        3.0
        /
        (
            4.0
            *
            df
            -
            1.0
        )
    )

    return float(
        correction
        *
        d
    )


def exact_permutation_test(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )

    pooled = np.concatenate(
        [
            active,
            passive
        ]
    )

    n_active = len(active)

    observed = (
        np.mean(active)
        -
        np.mean(passive)
    )

    n_total = len(pooled)

    all_indices = np.arange(
        n_total
    )

    permutation_diffs = []

    for active_indices_tuple in combinations(
        range(n_total),
        n_active
    ):

        active_indices = np.asarray(
            active_indices_tuple,
            dtype=int
        )

        passive_mask = np.ones(
            n_total,
            dtype=bool
        )

        passive_mask[
            active_indices
        ] = False

        perm_active = pooled[
            active_indices
        ]

        perm_passive = pooled[
            all_indices[
                passive_mask
            ]
        ]

        permutation_diffs.append(
            np.mean(perm_active)
            -
            np.mean(perm_passive)
        )

    permutation_diffs = np.asarray(
        permutation_diffs
    )

    p = np.mean(
        np.abs(permutation_diffs)
        >=
        (
            abs(observed)
            -
            1e-15
        )
    )

    return (
        float(observed),
        float(p),
        len(permutation_diffs)
    )


def stratified_bootstrap_ci(
    active,
    passive,
    n_bootstrap=N_BOOTSTRAP,
    seed=RANDOM_SEED
):

    rng = np.random.default_rng(
        seed
    )

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )

    bootstrap_diffs = np.empty(
        n_bootstrap,
        dtype=float
    )

    for i in range(
        n_bootstrap
    ):

        a = rng.choice(
            active,
            size=len(active),
            replace=True
        )

        p = rng.choice(
            passive,
            size=len(passive),
            replace=True
        )

        bootstrap_diffs[i] = (
            np.mean(a)
            -
            np.mean(p)
        )

    low, high = np.percentile(
        bootstrap_diffs,
        [
            2.5,
            97.5
        ]
    )

    return (
        float(low),
        float(high)
    )


def bh_fdr(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    m = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    ranked = p_values[
        order
    ]

    adjusted_ranked = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )

    # Enforce monotonicity.
    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0.0,
        1.0
    )

    adjusted = np.empty(
        m,
        dtype=float
    )

    adjusted[
        order
    ] = adjusted_ranked

    return adjusted


# =============================================================================
# 5. RECONSTRUCT EXACT FIXED-9 BASELINE-B AND TASK DATA
# =============================================================================

print("=" * 100)
print("TASK 6B — BASELINE-B CONNECTIVITY SENSITIVITY")
print("=" * 100)

print(
    f"Participants       : {len(SUBJECTS)}"
)

print(
    f"Active             : {len(ACTIVE_SUBJECTS)}"
)

print(
    f"Passive            : {len(PASSIVE_SUBJECTS)}"
)

print(
    f"Channels           : {len(CHANNELS)}"
)

print(
    f"Epochs/condition   : {FIXED_N}"
)

print(
    f"Seconds/condition  : "
    f"{FIXED_N * EPOCH_SECONDS:.1f}"
)

print(
    "Baseline B         : "
    "marker position 3 / T2_V3"
)

print(
    "Connectivity       : "
    "wPLI²_debiased"
)

print(
    "Spectral mode      : multitaper"
)

print(
    f"Bandwidth          : {MT_BANDWIDTH:.1f} Hz"
)


subject_data = {}

extraction_rows = []

epoch_audit_rows = []

failures = []


for subject in SUBJECTS:

    print(
        "\n"
        +
        "-" * 100
    )

    print(
        subject
    )

    print(
        "-" * 100
    )

    try:

        group = group_for_subject(
            subject
        )

        task_code = task_code_for_subject(
            subject
        )

        fif = exact_task3a_fif(
            subject
        )

        print(
            f"Group              : {group}"
        )

        print(
            f"Frozen Task 3A FIF : {fif.name}"
        )

        raw = mne.io.read_raw_fif(
            fif,
            preload=True,
            verbose="ERROR"
        )

        if abs(
            float(raw.info["sfreq"])
            -
            SFREQ_EXPECTED
        ) > 1e-9:
            raise RuntimeError(
                f"Unexpected sfreq "
                f"{raw.info['sfreq']} Hz."
            )

        missing_channels = [
            ch
            for ch in CHANNELS
            if ch not in raw.ch_names
        ]

        if missing_channels:
            raise RuntimeError(
                f"Missing channels: "
                f"{missing_channels}"
            )

        # ---------------------------------------------------------------------
        # Baseline B:
        # marker position 3 / T2_V3
        # ---------------------------------------------------------------------

        baseline_phase = get_phase(
            subject,
            3,
            "T2_V3"
        )

        # ---------------------------------------------------------------------
        # Task:
        # Active  position 5 / T1_V1
        # Passive position 5 / T1_V4
        # ---------------------------------------------------------------------

        task_phase = get_phase(
            subject,
            5,
            task_code
        )

        baseline_epochs, baseline_audit = (
            extract_clean_epochs(
                raw,
                baseline_phase[
                    "start_sec"
                ],
                baseline_phase[
                    "end_sec"
                ],
                FIXED_N
            )
        )

        task_epochs, task_audit = (
            extract_clean_epochs(
                raw,
                task_phase[
                    "start_sec"
                ],
                task_phase[
                    "end_sec"
                ],
                FIXED_N
            )
        )

        # Save audit information.
        baseline_audit.insert(
            0,
            "condition",
            "BaselineB"
        )

        baseline_audit.insert(
            0,
            "group",
            group
        )

        baseline_audit.insert(
            0,
            "subject",
            subject
        )

        task_audit.insert(
            0,
            "condition",
            "Task"
        )

        task_audit.insert(
            0,
            "group",
            group
        )

        task_audit.insert(
            0,
            "subject",
            subject
        )

        epoch_audit_rows.append(
            baseline_audit
        )

        epoch_audit_rows.append(
            task_audit
        )

        subject_data[
            subject
        ] = {
            "group": group,
            "sfreq": float(
                raw.info["sfreq"]
            ),
            "baseline": baseline_epochs,
            "task": task_epochs,
            "fif": str(fif)
        }

        extraction_rows.append({
            "subject": subject,
            "group": group,
            "baseline_marker_position": 3,
            "baseline_marker_code": "T2_V3",
            "task_marker_position": 5,
            "task_marker_code": task_code,
            "baseline_clean_available": int(
                (
                    baseline_audit[
                        "status"
                    ]
                    ==
                    "CLEAN"
                ).sum()
            ),
            "task_clean_available": int(
                (
                    task_audit[
                        "status"
                    ]
                    ==
                    "CLEAN"
                ).sum()
            ),
            "baseline_epochs_used": FIXED_N,
            "task_epochs_used": FIXED_N,
            "seconds_per_condition": (
                FIXED_N
                *
                EPOCH_SECONDS
            ),
            "status": "PASS"
        })

        print(
            "Baseline-B clean   : "
            f"{int((baseline_audit['status'] == 'CLEAN').sum())}"
        )

        print(
            "Task clean         : "
            f"{int((task_audit['status'] == 'CLEAN').sum())}"
        )

        print(
            f"Used               : "
            f"{FIXED_N} + {FIXED_N} epochs"
        )

        print(
            "STATUS             : PASS"
        )

    except Exception as exc:

        failures.append({
            "subject": subject,
            "error": repr(exc)
        })

        print(
            f"STATUS             : FAIL"
        )

        print(
            f"ERROR              : {repr(exc)}"
        )


if failures:

    failure_df = pd.DataFrame(
        failures
    )

    display(
        failure_df
    )

    failure_df.to_csv(
        OUT
        /
        "extraction_failures.csv",
        index=False
    )

    raise RuntimeError(
        f"{len(failures)} participant(s) failed "
        "Task 6B extraction."
    )


extraction_df = pd.DataFrame(
    extraction_rows
)

epoch_audit_df = pd.concat(
    epoch_audit_rows,
    ignore_index=True
)


# =============================================================================
# 6. CONNECTIVITY RECONSTRUCTION
# =============================================================================

matrix_rows = []

edge_rows = []

global_rows = []

connectivity_matrices = {}


for subject in SUBJECTS:

    print(
        "\nComputing connectivity:",
        subject
    )

    info = subject_data[
        subject
    ]

    group = info[
        "group"
    ]

    sfreq = info[
        "sfreq"
    ]

    connectivity_matrices[
        subject
    ] = {}

    for band, (
        fmin,
        fmax
    ) in BANDS.items():

        baseline_matrix, baseline_values, upper = (
            compute_wpli2_debiased(
                info[
                    "baseline"
                ],
                sfreq,
                fmin,
                fmax
            )
        )

        task_matrix, task_values, upper_task = (
            compute_wpli2_debiased(
                info[
                    "task"
                ],
                sfreq,
                fmin,
                fmax
            )
        )

        if not (
            np.array_equal(
                upper[0],
                upper_task[0]
            )
            and
            np.array_equal(
                upper[1],
                upper_task[1]
            )
        ):
            raise RuntimeError(
                f"{subject} {band}: "
                "edge-index mismatch."
            )

        delta_matrix = (
            task_matrix
            -
            baseline_matrix
        )

        connectivity_matrices[
            subject
        ][
            band
        ] = {
            "baseline": baseline_matrix,
            "task": task_matrix,
            "delta": delta_matrix
        }

        baseline_global = (
            global_connectivity(
                baseline_matrix
            )
        )

        task_global = (
            global_connectivity(
                task_matrix
            )
        )

        delta_global = (
            task_global
            -
            baseline_global
        )

        global_rows.append({
            "subject": subject,
            "group": group,
            "band": band,
            "baseline_b_global_wpli2_debiased":
                baseline_global,
            "task_global_wpli2_debiased":
                task_global,
            "delta_task_minus_baseline_b":
                delta_global
        })

        for edge_index, (
            i,
            j
        ) in enumerate(
            zip(
                upper[0],
                upper[1]
            )
        ):

            edge_rows.append({
                "subject": subject,
                "group": group,
                "band": band,
                "edge_index": edge_index,
                "channel_i": CHANNELS[i],
                "channel_j": CHANNELS[j],
                "baseline_b":
                    baseline_values[
                        edge_index
                    ],
                "task":
                    task_values[
                        edge_index
                    ],
                "delta_task_minus_baseline_b":
                    (
                        task_values[
                            edge_index
                        ]
                        -
                        baseline_values[
                            edge_index
                        ]
                    )
            })

        for condition, matrix in [
            (
                "BaselineB",
                baseline_matrix
            ),
            (
                "Task",
                task_matrix
            ),
            (
                "Delta",
                delta_matrix
            )
        ]:

            matrix_file = (
                OUT
                /
                (
                    f"{subject}_{band.lower()}_"
                    f"{condition.lower()}_"
                    "wpli2_debiased.npy"
                )
            )

            np.save(
                matrix_file,
                matrix
            )

            matrix_rows.append({
                "subject": subject,
                "group": group,
                "band": band,
                "condition": condition,
                "matrix_file": str(
                    matrix_file
                )
            })


global_df = pd.DataFrame(
    global_rows
)

edge_df = pd.DataFrame(
    edge_rows
)

matrix_index_df = pd.DataFrame(
    matrix_rows
)


# =============================================================================
# 7. NUMERICAL INTEGRITY CHECKS
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "CONNECTIVITY NUMERICAL QC"
)

print(
    "=" * 100
)


qc_rows = []


for subject in SUBJECTS:

    for band in BANDS:

        for condition in [
            "baseline",
            "task",
            "delta"
        ]:

            matrix = (
                connectivity_matrices[
                    subject
                ][
                    band
                ][
                    condition
                ]
            )

            symmetric = bool(
                np.allclose(
                    matrix,
                    matrix.T,
                    equal_nan=False
                )
            )

            zero_diagonal = bool(
                np.allclose(
                    np.diag(
                        matrix
                    ),
                    0.0
                )
            )

            finite = bool(
                np.all(
                    np.isfinite(
                        matrix
                    )
                )
            )

            qc_rows.append({
                "subject": subject,
                "band": band,
                "condition": condition,
                "symmetric": symmetric,
                "zero_diagonal": zero_diagonal,
                "all_finite": finite,
                "matrix_min": float(
                    np.min(matrix)
                ),
                "matrix_max": float(
                    np.max(matrix)
                ),
                "matrix_mean_upper": (
                    global_connectivity(
                        matrix
                    )
                )
            })


qc_df = pd.DataFrame(
    qc_rows
)


print(
    "All symmetric      :",
    bool(
        qc_df[
            "symmetric"
        ].all()
    )
)

print(
    "All zero diagonal  :",
    bool(
        qc_df[
            "zero_diagonal"
        ].all()
    )
)

print(
    "All finite         :",
    bool(
        qc_df[
            "all_finite"
        ].all()
    )
)

print(
    "Edges/subject/band :",
    (
        len(CHANNELS)
        *
        (
            len(CHANNELS) - 1
        )
        //
        2
    )
)

print(
    "Expected edge rows :",
    (
        len(SUBJECTS)
        *
        len(BANDS)
        *
        91
    )
)

print(
    "Observed edge rows :",
    len(edge_df)
)


if not (
    qc_df[
        "symmetric"
    ].all()
    and
    qc_df[
        "zero_diagonal"
    ].all()
    and
    qc_df[
        "all_finite"
    ].all()
):
    raise RuntimeError(
        "Connectivity numerical QC failed."
    )


if len(
    edge_df
) != (
    15
    *
    3
    *
    91
):
    raise RuntimeError(
        "Unexpected number of edge rows."
    )


# =============================================================================
# 8. DESCRIPTIVE GLOBAL CONNECTIVITY
# =============================================================================

group_summary = (
    global_df
    .groupby(
        [
            "group",
            "band"
        ]
    )
    .agg(
        n=(
            "subject",
            "count"
        ),
        baseline_b_mean=(
            "baseline_b_global_wpli2_debiased",
            "mean"
        ),
        baseline_b_sd=(
            "baseline_b_global_wpli2_debiased",
            "std"
        ),
        task_mean=(
            "task_global_wpli2_debiased",
            "mean"
        ),
        task_sd=(
            "task_global_wpli2_debiased",
            "std"
        ),
        delta_mean=(
            "delta_task_minus_baseline_b",
            "mean"
        ),
        delta_sd=(
            "delta_task_minus_baseline_b",
            "std"
        )
    )
    .reset_index()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "GLOBAL CONNECTIVITY — DESCRIPTIVE SUMMARY"
)

print(
    "=" * 100
)

display(
    group_summary
)


# =============================================================================
# 9. EXACT ACTIVE VS PASSIVE INFERENCE
# =============================================================================

inference_rows = []


for band_index, band in enumerate(
    BANDS.keys()
):

    band_df = global_df.loc[
        global_df[
            "band"
        ]
        ==
        band
    ].copy()

    active = (
        band_df.loc[
            band_df[
                "group"
            ]
            ==
            "Active",
            "delta_task_minus_baseline_b"
        ]
        .to_numpy(
            dtype=float
        )
    )

    passive = (
        band_df.loc[
            band_df[
                "group"
            ]
            ==
            "Passive",
            "delta_task_minus_baseline_b"
        ]
        .to_numpy(
            dtype=float
        )
    )

    observed_diff, exact_p, n_exact = (
        exact_permutation_test(
            active,
            passive
        )
    )

    ci_low, ci_high = (
        stratified_bootstrap_ci(
            active,
            passive,
            seed=(
                RANDOM_SEED
                +
                band_index
            )
        )
    )

    g = hedges_g(
        active,
        passive
    )

    inference_rows.append({
        "band": band,
        "n_active": len(active),
        "n_passive": len(passive),
        "active_mean_delta":
            float(
                np.mean(active)
            ),
        "active_sd_delta":
            float(
                np.std(
                    active,
                    ddof=1
                )
            ),
        "passive_mean_delta":
            float(
                np.mean(passive)
            ),
        "passive_sd_delta":
            float(
                np.std(
                    passive,
                    ddof=1
                )
            ),
        "active_minus_passive":
            observed_diff,
        "bootstrap_ci_low":
            ci_low,
        "bootstrap_ci_high":
            ci_high,
        "hedges_g":
            g,
        "exact_permutation_p":
            exact_p,
        "n_exact_permutations":
            n_exact
    })


inference_df = pd.DataFrame(
    inference_rows
)

inference_df[
    "p_bh_fdr_3"
] = bh_fdr(
    inference_df[
        "exact_permutation_p"
    ].to_numpy()
)

inference_df[
    "significant_raw_0_05"
] = (
    inference_df[
        "exact_permutation_p"
    ]
    <
    0.05
)

inference_df[
    "significant_fdr_0_05"
] = (
    inference_df[
        "p_bh_fdr_3"
    ]
    <
    0.05
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "BASELINE-B GLOBAL CONNECTIVITY INFERENCE"
)

print(
    "=" * 100
)

display(
    inference_df
)


# =============================================================================
# 10. LEAVE-ONE-OUT ROBUSTNESS
# =============================================================================

loo_rows = []


for band in BANDS:

    band_df = (
        global_df.loc[
            global_df[
                "band"
            ]
            ==
            band
        ]
        .sort_values(
            "subject"
        )
        .reset_index(
            drop=True
        )
    )

    for subject_to_remove in (
        band_df[
            "subject"
        ]
    ):

        temp = band_df.loc[
            band_df[
                "subject"
            ]
            !=
            subject_to_remove
        ]

        active = (
            temp.loc[
                temp[
                    "group"
                ]
                ==
                "Active",
                "delta_task_minus_baseline_b"
            ]
            .to_numpy(
                dtype=float
            )
        )

        passive = (
            temp.loc[
                temp[
                    "group"
                ]
                ==
                "Passive",
                "delta_task_minus_baseline_b"
            ]
            .to_numpy(
                dtype=float
            )
        )

        diff = float(
            np.mean(active)
            -
            np.mean(passive)
        )

        g = hedges_g(
            active,
            passive
        )

        loo_rows.append({
            "band": band,
            "removed_subject":
                subject_to_remove,
            "active_minus_passive":
                diff,
            "hedges_g":
                g
        })


loo_df = pd.DataFrame(
    loo_rows
)


loo_summary_rows = []


for band in BANDS:

    primary_diff = float(
        inference_df.loc[
            inference_df[
                "band"
            ]
            ==
            band,
            "active_minus_passive"
        ].iloc[0]
    )

    temp = loo_df.loc[
        loo_df[
            "band"
        ]
        ==
        band
    ]

    same_direction = (
        np.sign(
            temp[
                "active_minus_passive"
            ]
        )
        ==
        np.sign(
            primary_diff
        )
    )

    loo_summary_rows.append({
        "band": band,
        "primary_difference":
            primary_diff,
        "same_direction_n":
            int(
                same_direction.sum()
            ),
        "same_direction_total":
            len(temp),
        "same_direction_percent":
            float(
                100
                *
                same_direction.mean()
            ),
        "loo_diff_min":
            float(
                temp[
                    "active_minus_passive"
                ].min()
            ),
        "loo_diff_max":
            float(
                temp[
                    "active_minus_passive"
                ].max()
            ),
        "loo_g_min":
            float(
                temp[
                    "hedges_g"
                ].min()
            ),
        "loo_g_max":
            float(
                temp[
                    "hedges_g"
                ].max()
            )
    })


loo_summary_df = pd.DataFrame(
    loo_summary_rows
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "LEAVE-ONE-OUT ROBUSTNESS"
)

print(
    "=" * 100
)

display(
    loo_summary_df
)


# =============================================================================
# 11. OPTIONAL PRIMARY BASELINE-A COMPARISON
# =============================================================================
#
# Search for the corrected/validated primary global result automatically.
# This is only a sensitivity comparison. Failure to find it does NOT invalidate
# Task 6B.
# =============================================================================

primary_candidates = [
    DERIVATIVES
    /
    "task05b_validated_global_connectivity_inference"
    /
    "participant_global_connectivity.csv",

    DERIVATIVES
    /
    "task05b_validated_global_connectivity_inference"
    /
    "global_connectivity_participant_level.csv",

    DERIVATIVES
    /
    "task05b_corrected_global_connectivity_inference"
    /
    "participant_global_connectivity.csv",

    DERIVATIVES
    /
    "task05b_corrected_global_connectivity_inference"
    /
    "global_connectivity_participant_level.csv"
]


primary_file = None


for candidate in primary_candidates:

    if candidate.exists():
        primary_file = candidate
        break


baseline_comparison_df = pd.DataFrame()


if primary_file is not None:

    try:

        primary_df = pd.read_csv(
            primary_file
        )

        print(
            "\nPrimary Baseline-A file found:"
        )

        print(
            primary_file
        )

        # Identify likely delta column.
        possible_delta_columns = [
            "delta_task_minus_baseline",
            "delta_task_minus_baseline_a",
            "delta",
            "global_delta",
            "delta_global_wpli2_debiased"
        ]

        primary_delta_column = next(
            (
                col
                for col
                in possible_delta_columns
                if col in primary_df.columns
            ),
            None
        )

        if (
            primary_delta_column is not None
            and
            {
                "subject",
                "band"
            }.issubset(
                primary_df.columns
            )
        ):

            primary_small = primary_df[
                [
                    "subject",
                    "band",
                    primary_delta_column
                ]
            ].copy()

            primary_small = primary_small.rename(
                columns={
                    primary_delta_column:
                    "baseline_a_delta"
                }
            )

            baseline_comparison_df = (
                global_df[
                    [
                        "subject",
                        "group",
                        "band",
                        "delta_task_minus_baseline_b"
                    ]
                ]
                .merge(
                    primary_small,
                    on=[
                        "subject",
                        "band"
                    ],
                    how="inner"
                )
            )

            comparison_rows = []

            for band in BANDS:

                temp = (
                    baseline_comparison_df.loc[
                        baseline_comparison_df[
                            "band"
                        ]
                        ==
                        band
                    ]
                )

                if len(temp) == 15:

                    r, rp = pearsonr(
                        temp[
                            "baseline_a_delta"
                        ],
                        temp[
                            "delta_task_minus_baseline_b"
                        ]
                    )

                    rho, rhop = spearmanr(
                        temp[
                            "baseline_a_delta"
                        ],
                        temp[
                            "delta_task_minus_baseline_b"
                        ]
                    )

                    sign_same = (
                        np.sign(
                            temp[
                                "baseline_a_delta"
                            ]
                        )
                        ==
                        np.sign(
                            temp[
                                "delta_task_minus_baseline_b"
                            ]
                        )
                    )

                    comparison_rows.append({
                        "band": band,
                        "n": len(temp),
                        "participant_delta_same_sign_n":
                            int(
                                sign_same.sum()
                            ),
                        "participant_delta_same_sign_percent":
                            float(
                                100
                                *
                                sign_same.mean()
                            ),
                        "pearson_r":
                            float(r),
                        "pearson_p":
                            float(rp),
                        "spearman_rho":
                            float(rho),
                        "spearman_p":
                            float(rhop)
                    })

            if comparison_rows:

                comparison_summary_df = (
                    pd.DataFrame(
                        comparison_rows
                    )
                )

                print(
                    "\n"
                    +
                    "=" * 100
                )

                print(
                    "BASELINE A VS BASELINE B — "
                    "PARTICIPANT-LEVEL GLOBAL DELTA STABILITY"
                )

                print(
                    "=" * 100
                )

                display(
                    comparison_summary_df
                )

                comparison_summary_df.to_csv(
                    OUT
                    /
                    "baseline_a_vs_b_global_delta_stability.csv",
                    index=False
                )

        else:

            print(
                "\nPrimary file found, but its column structure "
                "was not recognized automatically."
            )

            print(
                "Task 6B remains valid; the cross-baseline comparison "
                "was simply skipped."
            )

    except Exception as exc:

        print(
            "\nPrimary Baseline-A comparison skipped:"
        )

        print(
            repr(exc)
        )

else:

    print(
        "\nNo automatically recognized validated Baseline-A "
        "participant-level file was found."
    )

    print(
        "This does not affect Task 6B."
    )


# =============================================================================
# 12. FIGURE — GLOBAL CHANGE BY BAND
# =============================================================================

plot_df = (
    global_df
    .pivot(
        index="subject",
        columns="band",
        values="delta_task_minus_baseline_b"
    )
)


fig, ax = plt.subplots(
    figsize=(
        8.5,
        5.5
    )
)


x_positions = np.arange(
    len(BANDS)
)

band_names = list(
    BANDS.keys()
)


for group, marker in [
    (
        "Active",
        "o"
    ),
    (
        "Passive",
        "s"
    )
]:

    subjects = (
        ACTIVE_SUBJECTS
        if group == "Active"
        else PASSIVE_SUBJECTS
    )

    for x, band in enumerate(
        band_names
    ):

        y = (
            global_df.loc[
                (
                    global_df[
                        "group"
                    ]
                    ==
                    group
                )
                &
                (
                    global_df[
                        "band"
                    ]
                    ==
                    band
                ),
                "delta_task_minus_baseline_b"
            ]
            .to_numpy()
        )

        jitter = np.linspace(
            -0.07,
            0.07,
            len(y)
        )

        ax.scatter(
            np.full(
                len(y),
                x
            )
            +
            jitter,
            y,
            marker=marker,
            label=(
                group
                if x == 0
                else None
            )
        )


ax.axhline(
    0.0,
    linewidth=1
)

ax.set_xticks(
    x_positions
)

ax.set_xticklabels(
    band_names
)

ax.set_ylabel(
    "Task − Baseline B global wPLI²_debiased"
)

ax.set_xlabel(
    "Frequency band"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

figure_path = (
    OUT
    /
    "task06b_baseline_b_global_connectivity_change.png"
)

fig.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =============================================================================
# 13. SAVE TABLES
# =============================================================================

extraction_df.to_csv(
    OUT
    /
    "fixed9_extraction_summary.csv",
    index=False
)

epoch_audit_df.to_csv(
    OUT
    /
    "fixed9_epoch_audit.csv",
    index=False
)

global_df.to_csv(
    OUT
    /
    "participant_global_connectivity.csv",
    index=False
)

edge_df.to_csv(
    OUT
    /
    "edge_level_connectivity.csv",
    index=False
)

matrix_index_df.to_csv(
    OUT
    /
    "matrix_file_index.csv",
    index=False
)

qc_df.to_csv(
    OUT
    /
    "connectivity_numerical_qc.csv",
    index=False
)

group_summary.to_csv(
    OUT
    /
    "group_descriptive_summary.csv",
    index=False
)

inference_df.to_csv(
    OUT
    /
    "global_connectivity_exact_inference.csv",
    index=False
)

loo_df.to_csv(
    OUT
    /
    "leave_one_out_results.csv",
    index=False
)

loo_summary_df.to_csv(
    OUT
    /
    "leave_one_out_summary.csv",
    index=False
)


# =============================================================================
# 14. FINAL INTEGRITY CHECK
# =============================================================================

expected_subjects = 15
expected_edges = 91
expected_edge_rows = (
    15
    *
    3
    *
    91
)

expected_global_rows = (
    15
    *
    3
)


integrity = {
    "participants":
        len(
            subject_data
        )
        ==
        expected_subjects,

    "active":
        (
            extraction_df[
                "group"
            ]
            ==
            "Active"
        ).sum()
        ==
        8,

    "passive":
        (
            extraction_df[
                "group"
            ]
            ==
            "Passive"
        ).sum()
        ==
        7,

    "all_fixed9":
        (
            (
                extraction_df[
                    "baseline_epochs_used"
                ]
                ==
                FIXED_N
            ).all()
            and
            (
                extraction_df[
                    "task_epochs_used"
                ]
                ==
                FIXED_N
            ).all()
        ),

    "edge_rows":
        len(
            edge_df
        )
        ==
        expected_edge_rows,

    "global_rows":
        len(
            global_df
        )
        ==
        expected_global_rows,

    "inference_tests":
        len(
            inference_df
        )
        ==
        3,

    "exact_permutations":
        (
            inference_df[
                "n_exact_permutations"
            ]
            ==
            6435
        ).all(),

    "finite_matrices":
        qc_df[
            "all_finite"
        ].all(),

    "symmetric_matrices":
        qc_df[
            "symmetric"
        ].all(),

    "zero_diagonal":
        qc_df[
            "zero_diagonal"
        ].all()
}


overall_pass = all(
    integrity.values()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "TASK 6B COMPLETE"
)

print(
    "=" * 100
)

print(
    f"Participants              : "
    f"{len(subject_data)}/15"
)

print(
    f"Active                    : "
    f"{(extraction_df['group'] == 'Active').sum()}/8"
)

print(
    f"Passive                   : "
    f"{(extraction_df['group'] == 'Passive').sum()}/7"
)

print(
    f"Fixed epochs/condition    : "
    f"{FIXED_N}"
)

print(
    f"Seconds/condition         : "
    f"{FIXED_N * EPOCH_SECONDS:.1f}"
)

print(
    f"Channels                  : "
    f"{len(CHANNELS)}"
)

print(
    f"Edges/participant/band    : "
    f"{expected_edges}"
)

print(
    f"Edge-level rows           : "
    f"{len(edge_df)}/{expected_edge_rows}"
)

print(
    f"Participant-band rows     : "
    f"{len(global_df)}/{expected_global_rows}"
)

print(
    f"Global inference tests    : "
    f"{len(inference_df)}/3"
)

print(
    "Exact allocations/test   : 6435"
)

print(
    f"Bootstrap iterations/test : "
    f"{N_BOOTSTRAP:,}"
)

print(
    "FDR family                : 3 bands"
)

print(
    "Estimator                 : "
    "wPLI²_debiased"
)

print(
    "Baseline                  : "
    "Baseline B / T2_V3 / marker position 3"
)

print(
    "\nIntegrity components:"
)

for key, value in (
    integrity.items()
):

    print(
        f"  {key:<25}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 100
)

print(
    "OVERALL INTEGRITY CHECK:",
    (
        "PASS"
        if overall_pass
        else "FAIL"
    )
)

print(
    "-" * 100
)


if not overall_pass:
    raise RuntimeError(
        "Task 6B integrity check failed."
    )


print(
    "\nINTERPRETATION FRAMEWORK:"
)

print(
    "1. This is a Baseline-B sensitivity analysis, "
    "not a replacement for the primary Baseline-A analysis."
)

print(
    "2. The primary inferential statistic remains the exact "
    "Active-vs-Passive permutation test."
)

print(
    "3. Bootstrap confidence intervals and Hedges g characterize "
    "uncertainty and effect magnitude; they do not override the "
    "exact permutation test."
)

print(
    "4. Stability of the beta result across Baseline A and "
    "Baseline B would strengthen its interpretation."
)

print(
    "5. A reversal or substantial attenuation under Baseline B "
    "would indicate baseline sensitivity and must be reported."
)

print(
    "6. Negative wPLI²_debiased estimates are retained because "
    "they are permissible for the debiased estimator."
)

print(
    "7. No electrode-level or edge-level significance is claimed."
)

print(
    f"\nOutput directory:\n{OUT}"
)

In [ ]:
# %pip install mne-connectivity

In [ ]:
# =============================================================================
# TASK 6C — MATCHED FIXED-9 BASELINE A vs BASELINE B CONNECTIVITY SENSITIVITY
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PURPOSE
# -------
# Determine whether Active-vs-Passive functional-connectivity effects depend
# on baseline choice when EVERYTHING ELSE is held constant.
#
# Baseline A:
#     marker_position = 1
#     marker_code     = T2_V1
#
# Baseline B:
#     marker_position = 3
#     marker_code     = T2_V3
#
# Task:
#     Active  -> marker_position = 5 / T1_V1
#     Passive -> marker_position = 5 / T1_V4
#
# EXACT MATCHING
# --------------
# Every participant contributes:
#
#     9 clean Baseline-A epochs
#     9 clean Baseline-B epochs
#     9 clean Task epochs
#
# Each epoch = 2 s.
#
# Critically, the SAME 9 Task epochs are used for:
#
#     Delta_A = Task - Baseline A
#     Delta_B = Task - Baseline B
#
# Therefore differences between Delta_A and Delta_B isolate baseline choice
# rather than task-epoch sampling or epoch-count differences.
#
# CONNECTIVITY
# ------------
# MNE-Connectivity:
#
#     spectral_connectivity_epochs
#     method = "wpli2_debiased"
#     mode   = "multitaper"
#     mt_bandwidth = 2.0 Hz
#
# Bands:
#
#     Theta 4–8 Hz
#     Alpha 8–13 Hz
#     Beta  13–30 Hz
#
# PRIMARY STATISTICAL UNIT
# ------------------------
# Participant.
#
# INFERENCE
# ---------
# For each band and baseline:
#
#     Active mean Delta
#     Passive mean Delta
#     Active - Passive difference
#     Hedges' g
#     exact permutation p, C(15,8) = 6435
#     stratified bootstrap 95% CI, 10,000 iterations
#     BH-FDR across 3 bands separately for each baseline
#
# CROSS-BASELINE ROBUSTNESS
# -------------------------
# For each band:
#
#     - direction agreement of Active-Passive effect
#     - magnitude change
#     - Hedges' g comparison
#     - participant-level Pearson correlation
#     - participant-level Spearman correlation
#     - participant-level sign agreement
#
# IMPORTANT
# ---------
# Negative wPLI²_debiased estimates are retained.
# No individual edge inference.
# No graph thresholding.
# No post-hoc baseline selection according to p-value.
# No Eyes-Open/Eyes-Closed semantic label is assigned.
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

import mne

warnings.filterwarnings("ignore")


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


try:
    from mne_connectivity import spectral_connectivity_epochs

except ImportError as exc:

    raise ImportError(
        "\nMNE-Connectivity is required.\n\n"
        "Run this in a separate Jupyter cell:\n\n"
        "    %pip install mne-connectivity\n\n"
        "Restart the kernel and rerun Task 6C."
    ) from exc


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

DERIVATIVES = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK3A = (
    DERIVATIVES
    / "task03a_batch_preprocessing_qc"
)

TASK3B = (
    DERIVATIVES
    / "task03b_phase_reconstruction"
)

PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)

OUT = (
    DERIVATIVES
    / "task06c_matched_fixed9_baseline_sensitivity"
)

FIG_OUT = (
    OUT
    / "figures"
)

MATRIX_OUT = (
    OUT
    / "matrices"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)

MATRIX_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. SETTINGS
# =============================================================================

CHANNELS = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


BANDS = {
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0)
}


SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 16)
]


ACTIVE_SUBJECTS = SUBJECTS[:8]
PASSIVE_SUBJECTS = SUBJECTS[8:]


SFREQ_EXPECTED = 256.0

EPOCH_SECONDS = 2.0

FIXED_N = 9

MAX_PHASE_SECONDS = 30.0

ABS_THRESHOLD_UV = 250.0
PTP_THRESHOLD_UV = 500.0

MT_BANDWIDTH = 2.0

N_BOOTSTRAP = 10_000

RANDOM_SEED = 20260913

EXPECTED_PERMUTATIONS = math.comb(
    15,
    8
)

EXPECTED_EDGES = math.comb(
    len(CHANNELS),
    2
)


# =============================================================================
# 3. LOAD PHASE TABLE
# =============================================================================

if not PHASE_FILE.exists():

    raise FileNotFoundError(
        f"Task 3B phase table not found:\n{PHASE_FILE}"
    )


phase_df = pd.read_csv(
    PHASE_FILE
)


required_columns = {
    "subject",
    "marker_position",
    "marker_code",
    "start_sec",
    "end_sec",
    "duration_sec"
}


missing = (
    required_columns
    -
    set(phase_df.columns)
)


if missing:

    raise RuntimeError(
        "Task 3B phase table missing required columns: "
        f"{sorted(missing)}"
    )


# =============================================================================
# 4. SUBJECT HELPERS
# =============================================================================

def group_for_subject(subject):

    number = int(
        subject.split("-")[1]
    )

    return (
        "Active"
        if number <= 8
        else "Passive"
    )


def task_code_for_subject(subject):

    return (
        "T1_V1"
        if group_for_subject(subject) == "Active"
        else "T1_V4"
    )


def exact_task3a_fif(subject):

    group = group_for_subject(
        subject
    )

    task = (
        "activeai"
        if group == "Active"
        else "passiveai"
    )

    path = (
        TASK3A
        / subject
        / (
            f"{subject}_task-{task}"
            "_desc-primary_annotated_eeg_raw.fif"
        )
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Frozen Task 3A FIF not found:\n{path}"
        )

    return path


# =============================================================================
# 5. PHASE LOOKUP
# =============================================================================

def get_phase(
    subject,
    marker_position,
    expected_code
):

    temp = phase_df.loc[
        (
            phase_df[
                "subject"
            ].astype(str)
            ==
            subject
        )
        &
        (
            pd.to_numeric(
                phase_df[
                    "marker_position"
                ],
                errors="coerce"
            )
            ==
            marker_position
        )
    ].copy()


    if len(temp) != 1:

        raise RuntimeError(
            f"{subject}: marker position "
            f"{marker_position} returned "
            f"{len(temp)} rows."
        )


    row = temp.iloc[0]


    observed_code = str(
        row[
            "marker_code"
        ]
    )


    if observed_code != expected_code:

        raise RuntimeError(
            f"{subject}: position {marker_position} "
            f"expected {expected_code}, "
            f"found {observed_code}."
        )


    return row


# =============================================================================
# 6. CLEAN-EPOCH EXTRACTION
# =============================================================================

def extract_first_clean_epochs(
    raw,
    start_sec,
    end_sec,
    n_required=FIXED_N
):

    sfreq = float(
        raw.info[
            "sfreq"
        ]
    )


    if not np.isclose(
        sfreq,
        SFREQ_EXPECTED
    ):

        raise RuntimeError(
            f"Unexpected sampling frequency: {sfreq}"
        )


    start_sec = float(
        start_sec
    )

    end_sec = float(
        end_sec
    )


    available_duration = (
        end_sec
        -
        start_sec
    )


    usable_duration = min(
        available_duration,
        MAX_PHASE_SECONDS
    )


    n_possible = int(
        np.floor(
            usable_duration
            /
            EPOCH_SECONDS
        )
    )


    epoch_samples = int(
        round(
            EPOCH_SECONDS
            *
            sfreq
        )
    )


    clean_epochs = []
    selected_indices = []
    audit_rows = []


    for epoch_index in range(
        n_possible
    ):

        epoch_start = (
            start_sec
            +
            epoch_index
            *
            EPOCH_SECONDS
        )


        epoch_end = (
            epoch_start
            +
            EPOCH_SECONDS
        )


        start_sample = int(
            round(
                epoch_start
                *
                sfreq
            )
        )


        stop_sample = (
            start_sample
            +
            epoch_samples
        )


        if stop_sample > raw.n_times:

            audit_rows.append({
                "epoch_index": epoch_index,
                "start_sec": epoch_start,
                "end_sec": epoch_end,
                "status": "OUTSIDE_RAW",
                "max_abs_uv": np.nan,
                "max_ptp_uv": np.nan
            })

            continue


        data = raw.get_data(
            picks=CHANNELS,
            start=start_sample,
            stop=stop_sample
        )


        if data.shape != (
            len(CHANNELS),
            epoch_samples
        ):

            audit_rows.append({
                "epoch_index": epoch_index,
                "start_sec": epoch_start,
                "end_sec": epoch_end,
                "status": "BAD_SHAPE",
                "max_abs_uv": np.nan,
                "max_ptp_uv": np.nan
            })

            continue


        data_uv = (
            data
            *
            1e6
        )


        max_abs_uv = float(
            np.max(
                np.abs(
                    data_uv
                )
            )
        )


        max_ptp_uv = float(
            np.max(
                np.ptp(
                    data_uv,
                    axis=1
                )
            )
        )


        clean = (
            max_abs_uv
            <=
            ABS_THRESHOLD_UV
            and
            max_ptp_uv
            <=
            PTP_THRESHOLD_UV
        )


        audit_rows.append({
            "epoch_index": epoch_index,
            "start_sec": epoch_start,
            "end_sec": epoch_end,
            "status": (
                "CLEAN"
                if clean
                else "REJECT"
            ),
            "max_abs_uv": max_abs_uv,
            "max_ptp_uv": max_ptp_uv
        })


        if clean:

            clean_epochs.append(
                data.copy()
            )

            selected_indices.append(
                epoch_index
            )


    if len(clean_epochs) < n_required:

        raise RuntimeError(
            f"Only {len(clean_epochs)} clean epochs; "
            f"{n_required} required."
        )


    # Freeze earliest n_required clean epochs.
    clean_epochs = np.stack(
        clean_epochs[
            :n_required
        ],
        axis=0
    )


    selected_indices = selected_indices[
        :n_required
    ]


    return (
        clean_epochs,
        selected_indices,
        pd.DataFrame(
            audit_rows
        )
    )


# =============================================================================
# 7. VALIDATED SPECTRAL wPLI²_DEBIASED
# =============================================================================

def compute_wpli2_debiased(
    epochs_data,
    sfreq,
    fmin,
    fmax
):

    n_channels = epochs_data.shape[
        1
    ]


    upper = np.triu_indices(
        n_channels,
        k=1
    )


    con = spectral_connectivity_epochs(
        epochs_data,
        method="wpli2_debiased",
        indices=upper,
        sfreq=sfreq,
        mode="multitaper",
        fmin=fmin,
        fmax=fmax,
        faverage=True,
        mt_bandwidth=MT_BANDWIDTH,
        mt_adaptive=False,
        mt_low_bias=True,
        verbose=False
    )


    raw_values = np.asarray(
        con.get_data()
    )


    values = np.ravel(
        raw_values.squeeze()
    )


    if len(values) != EXPECTED_EDGES:

        raise RuntimeError(
            "Unexpected connectivity output. "
            f"Expected {EXPECTED_EDGES} edges; "
            f"found {len(values)}. "
            f"Raw shape={raw_values.shape}"
        )


    matrix = np.zeros(
        (
            n_channels,
            n_channels
        ),
        dtype=float
    )


    matrix[
        upper
    ] = values


    matrix[
        (
            upper[1],
            upper[0]
        )
    ] = values


    np.fill_diagonal(
        matrix,
        0.0
    )


    return (
        matrix,
        values,
        upper
    )


def upper_mean(
    matrix
):

    upper = np.triu_indices(
        matrix.shape[0],
        k=1
    )

    return float(
        np.mean(
            matrix[
                upper
            ]
        )
    )


# =============================================================================
# 8. STATISTICAL HELPERS
# =============================================================================

def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    n1 = len(active)
    n2 = len(passive)


    var1 = np.var(
        active,
        ddof=1
    )

    var2 = np.var(
        passive,
        ddof=1
    )


    pooled_var = (
        (
            (n1 - 1)
            *
            var1
        )
        +
        (
            (n2 - 1)
            *
            var2
        )
    ) / (
        n1
        +
        n2
        -
        2
    )


    if (
        not np.isfinite(
            pooled_var
        )
        or
        pooled_var <= 0
    ):

        return np.nan


    d = (
        np.mean(active)
        -
        np.mean(passive)
    ) / np.sqrt(
        pooled_var
    )


    df = (
        n1
        +
        n2
        -
        2
    )


    correction = (
        1.0
        -
        3.0
        /
        (
            4.0
            *
            df
            -
            1.0
        )
    )


    return float(
        correction
        *
        d
    )


def exact_permutation_test(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    pooled = np.concatenate(
        [
            active,
            passive
        ]
    )


    n_active = len(
        active
    )


    observed = float(
        np.mean(active)
        -
        np.mean(passive)
    )


    n_total = len(
        pooled
    )


    all_indices = np.arange(
        n_total
    )


    permuted = []


    for active_tuple in combinations(
        range(n_total),
        n_active
    ):

        active_idx = np.asarray(
            active_tuple,
            dtype=int
        )


        passive_mask = np.ones(
            n_total,
            dtype=bool
        )


        passive_mask[
            active_idx
        ] = False


        passive_idx = all_indices[
            passive_mask
        ]


        permuted.append(
            np.mean(
                pooled[
                    active_idx
                ]
            )
            -
            np.mean(
                pooled[
                    passive_idx
                ]
            )
        )


    permuted = np.asarray(
        permuted,
        dtype=float
    )


    p_value = float(
        np.mean(
            np.abs(
                permuted
            )
            >=
            (
                abs(
                    observed
                )
                -
                1e-15
            )
        )
    )


    return (
        observed,
        p_value,
        len(
            permuted
        )
    )


def bootstrap_ci(
    active,
    passive,
    seed
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )


    boot = np.empty(
        N_BOOTSTRAP,
        dtype=float
    )


    for i in range(
        N_BOOTSTRAP
    ):

        active_sample = rng.choice(
            active,
            size=len(active),
            replace=True
        )


        passive_sample = rng.choice(
            passive,
            size=len(passive),
            replace=True
        )


        boot[
            i
        ] = (
            np.mean(
                active_sample
            )
            -
            np.mean(
                passive_sample
            )
        )


    low, high = np.percentile(
        boot,
        [
            2.5,
            97.5
        ]
    )


    return (
        float(low),
        float(high)
    )


def bh_fdr(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    ranked = p_values[
        order
    ]


    adjusted_ranked = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted_ranked = np.clip(
        adjusted_ranked,
        0.0,
        1.0
    )


    adjusted = np.empty(
        m,
        dtype=float
    )


    adjusted[
        order
    ] = adjusted_ranked


    return adjusted


# =============================================================================
# 9. RECONSTRUCT MATCHED FIXED-9 DATA
# =============================================================================

print(
    "=" * 100
)

print(
    "TASK 6C — MATCHED FIXED-9 BASELINE A vs BASELINE B"
)

print(
    "=" * 100
)

print(
    f"Participants              : {len(SUBJECTS)}"
)

print(
    "Active                    : 8"
)

print(
    "Passive                   : 7"
)

print(
    f"Channels                  : {len(CHANNELS)}"
)

print(
    f"Edges                     : {EXPECTED_EDGES}"
)

print(
    f"Epochs/state              : {FIXED_N}"
)

print(
    f"Seconds/state             : "
    f"{FIXED_N * EPOCH_SECONDS:.1f}"
)

print(
    "Baseline A                : position 1 / T2_V1"
)

print(
    "Baseline B                : position 3 / T2_V3"
)

print(
    "Estimator                 : wPLI²_debiased"
)

print(
    "Spectral mode             : multitaper"
)

print(
    f"Bandwidth                  : {MT_BANDWIDTH:.1f} Hz"
)

print(
    f"Exact allocations/test    : {EXPECTED_PERMUTATIONS}"
)


subject_data = {}

selection_rows = []

audit_tables = []

failures = []


for subject in SUBJECTS:

    print(
        "\n"
        +
        "-" * 100
    )

    print(
        subject
    )

    print(
        "-" * 100
    )


    try:

        group = group_for_subject(
            subject
        )


        task_code = task_code_for_subject(
            subject
        )


        fif = exact_task3a_fif(
            subject
        )


        raw = mne.io.read_raw_fif(
            fif,
            preload=True,
            verbose="ERROR"
        )


        missing_channels = [
            channel
            for channel in CHANNELS
            if channel not in raw.ch_names
        ]


        if missing_channels:

            raise RuntimeError(
                f"Missing channels: {missing_channels}"
            )


        phase_a = get_phase(
            subject,
            1,
            "T2_V1"
        )


        phase_b = get_phase(
            subject,
            3,
            "T2_V3"
        )


        phase_task = get_phase(
            subject,
            5,
            task_code
        )


        epochs_a, indices_a, audit_a = (
            extract_first_clean_epochs(
                raw,
                phase_a[
                    "start_sec"
                ],
                phase_a[
                    "end_sec"
                ]
            )
        )


        epochs_b, indices_b, audit_b = (
            extract_first_clean_epochs(
                raw,
                phase_b[
                    "start_sec"
                ],
                phase_b[
                    "end_sec"
                ]
            )
        )


        epochs_task, indices_task, audit_task = (
            extract_first_clean_epochs(
                raw,
                phase_task[
                    "start_sec"
                ],
                phase_task[
                    "end_sec"
                ]
            )
        )


        # -------------------------------------------------------------
        # Critical freeze:
        # the SAME task epoch array is used for both normalizations.
        # -------------------------------------------------------------

        subject_data[
            subject
        ] = {
            "group": group,
            "sfreq": float(
                raw.info[
                    "sfreq"
                ]
            ),
            "baseline_a": epochs_a,
            "baseline_b": epochs_b,
            "task": epochs_task
        }


        for condition, audit in [
            (
                "BaselineA",
                audit_a
            ),
            (
                "BaselineB",
                audit_b
            ),
            (
                "Task",
                audit_task
            )
        ]:

            audit = audit.copy()

            audit.insert(
                0,
                "condition",
                condition
            )

            audit.insert(
                0,
                "group",
                group
            )

            audit.insert(
                0,
                "subject",
                subject
            )

            audit_tables.append(
                audit
            )


        selection_rows.append({
            "subject":
                subject,

            "group":
                group,

            "baseline_a_clean_available":
                int(
                    (
                        audit_a[
                            "status"
                        ]
                        ==
                        "CLEAN"
                    ).sum()
                ),

            "baseline_b_clean_available":
                int(
                    (
                        audit_b[
                            "status"
                        ]
                        ==
                        "CLEAN"
                    ).sum()
                ),

            "task_clean_available":
                int(
                    (
                        audit_task[
                            "status"
                        ]
                        ==
                        "CLEAN"
                    ).sum()
                ),

            "baseline_a_indices":
                ",".join(
                    map(
                        str,
                        indices_a
                    )
                ),

            "baseline_b_indices":
                ",".join(
                    map(
                        str,
                        indices_b
                    )
                ),

            "task_indices":
                ",".join(
                    map(
                        str,
                        indices_task
                    )
                ),

            "baseline_a_used":
                FIXED_N,

            "baseline_b_used":
                FIXED_N,

            "task_used":
                FIXED_N,

            "same_task_for_both_normalizations":
                True,

            "status":
                "PASS"
        })


        print(
            f"Group                     : {group}"
        )

        print(
            f"Baseline-A clean          : "
            f"{int((audit_a['status']=='CLEAN').sum())}"
        )

        print(
            f"Baseline-B clean          : "
            f"{int((audit_b['status']=='CLEAN').sum())}"
        )

        print(
            f"Task clean                : "
            f"{int((audit_task['status']=='CLEAN').sum())}"
        )

        print(
            f"Baseline-A selected       : {indices_a}"
        )

        print(
            f"Baseline-B selected       : {indices_b}"
        )

        print(
            f"Task selected             : {indices_task}"
        )

        print(
            "Same task for A and B     : YES"
        )

        print(
            "STATUS                    : PASS"
        )


    except Exception as exc:

        failures.append({
            "subject":
                subject,

            "error":
                repr(
                    exc
                )
        })


        print(
            "STATUS                    : FAIL"
        )

        print(
            f"ERROR                     : {repr(exc)}"
        )


if failures:

    failure_df = pd.DataFrame(
        failures
    )

    display(
        failure_df
    )

    failure_df.to_csv(
        OUT
        /
        "extraction_failures.csv",
        index=False
    )

    raise RuntimeError(
        f"{len(failures)} participant(s) failed Task 6C."
    )


selection_df = pd.DataFrame(
    selection_rows
)


audit_df = pd.concat(
    audit_tables,
    ignore_index=True
)


# =============================================================================
# 10. COMPUTE ALL THREE MATRICES ONCE PER SUBJECT/BAND
# =============================================================================

global_rows = []

edge_rows = []

matrix_qc_rows = []


for subject in SUBJECTS:

    print(
        f"\nComputing matched connectivity: {subject}"
    )


    data = subject_data[
        subject
    ]


    group = data[
        "group"
    ]


    sfreq = data[
        "sfreq"
    ]


    for band, (
        fmin,
        fmax
    ) in BANDS.items():

        matrix_a, values_a, upper_a = (
            compute_wpli2_debiased(
                data[
                    "baseline_a"
                ],
                sfreq,
                fmin,
                fmax
            )
        )


        matrix_b, values_b, upper_b = (
            compute_wpli2_debiased(
                data[
                    "baseline_b"
                ],
                sfreq,
                fmin,
                fmax
            )
        )


        # IMPORTANT:
        # task connectivity is computed ONCE.
        matrix_task, values_task, upper_task = (
            compute_wpli2_debiased(
                data[
                    "task"
                ],
                sfreq,
                fmin,
                fmax
            )
        )


        if not (
            np.array_equal(
                upper_a[0],
                upper_b[0]
            )
            and
            np.array_equal(
                upper_a[1],
                upper_b[1]
            )
            and
            np.array_equal(
                upper_a[0],
                upper_task[0]
            )
            and
            np.array_equal(
                upper_a[1],
                upper_task[1]
            )
        ):

            raise RuntimeError(
                f"{subject} {band}: edge-index mismatch."
            )


        delta_a_matrix = (
            matrix_task
            -
            matrix_a
        )


        delta_b_matrix = (
            matrix_task
            -
            matrix_b
        )


        # -------------------------------------------------------------
        # Save matrices.
        # -------------------------------------------------------------

        matrices = {
            "baseline_a":
                matrix_a,

            "baseline_b":
                matrix_b,

            "task":
                matrix_task,

            "delta_a":
                delta_a_matrix,

            "delta_b":
                delta_b_matrix
        }


        for name, matrix in matrices.items():

            np.save(
                MATRIX_OUT
                /
                (
                    f"{subject}_"
                    f"{band.lower()}_"
                    f"{name}_"
                    "wpli2_debiased.npy"
                ),
                matrix
            )


            matrix_qc_rows.append({
                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "matrix":
                    name,

                "symmetric":
                    bool(
                        np.allclose(
                            matrix,
                            matrix.T
                        )
                    ),

                "zero_diagonal":
                    bool(
                        np.allclose(
                            np.diag(
                                matrix
                            ),
                            0.0
                        )
                    ),

                "all_finite":
                    bool(
                        np.all(
                            np.isfinite(
                                matrix
                            )
                        )
                    ),

                "minimum":
                    float(
                        np.min(
                            matrix
                        )
                    ),

                "maximum":
                    float(
                        np.max(
                            matrix
                        )
                    ),

                "upper_mean":
                    upper_mean(
                        matrix
                    )
            })


        baseline_a_global = upper_mean(
            matrix_a
        )


        baseline_b_global = upper_mean(
            matrix_b
        )


        task_global = upper_mean(
            matrix_task
        )


        delta_a_global = (
            task_global
            -
            baseline_a_global
        )


        delta_b_global = (
            task_global
            -
            baseline_b_global
        )


        global_rows.append({
            "subject":
                subject,

            "group":
                group,

            "band":
                band,

            "baseline_a":
                baseline_a_global,

            "baseline_b":
                baseline_b_global,

            "task":
                task_global,

            "delta_a":
                delta_a_global,

            "delta_b":
                delta_b_global,

            "delta_b_minus_delta_a":
                (
                    delta_b_global
                    -
                    delta_a_global
                )
        })


        for edge_index, (
            i,
            j
        ) in enumerate(
            zip(
                upper_a[0],
                upper_a[1]
            )
        ):

            edge_rows.append({
                "subject":
                    subject,

                "group":
                    group,

                "band":
                    band,

                "edge_index":
                    edge_index,

                "channel_i":
                    CHANNELS[
                        i
                    ],

                "channel_j":
                    CHANNELS[
                        j
                    ],

                "baseline_a":
                    values_a[
                        edge_index
                    ],

                "baseline_b":
                    values_b[
                        edge_index
                    ],

                "task":
                    values_task[
                        edge_index
                    ],

                "delta_a":
                    (
                        values_task[
                            edge_index
                        ]
                        -
                        values_a[
                            edge_index
                        ]
                    ),

                "delta_b":
                    (
                        values_task[
                            edge_index
                        ]
                        -
                        values_b[
                            edge_index
                        ]
                    )
            })


global_df = pd.DataFrame(
    global_rows
)


edge_df = pd.DataFrame(
    edge_rows
)


matrix_qc_df = pd.DataFrame(
    matrix_qc_rows
)


# =============================================================================
# 11. NUMERICAL QC
# =============================================================================

expected_edge_rows = (
    15
    *
    3
    *
    EXPECTED_EDGES
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MATCHED CONNECTIVITY NUMERICAL QC"
)

print(
    "=" * 100
)

print(
    "All matrices symmetric     :",
    bool(
        matrix_qc_df[
            "symmetric"
        ].all()
    )
)

print(
    "All diagonals zero         :",
    bool(
        matrix_qc_df[
            "zero_diagonal"
        ].all()
    )
)

print(
    "All values finite          :",
    bool(
        matrix_qc_df[
            "all_finite"
        ].all()
    )
)

print(
    f"Edges/subject/band         : {EXPECTED_EDGES}"
)

print(
    f"Expected edge rows         : {expected_edge_rows}"
)

print(
    f"Observed edge rows         : {len(edge_df)}"
)


if not (
    matrix_qc_df[
        "symmetric"
    ].all()
    and
    matrix_qc_df[
        "zero_diagonal"
    ].all()
    and
    matrix_qc_df[
        "all_finite"
    ].all()
):

    raise RuntimeError(
        "Task 6C numerical matrix QC failed."
    )


if len(edge_df) != expected_edge_rows:

    raise RuntimeError(
        "Task 6C edge-row count failed."
    )


# =============================================================================
# 12. GROUP DESCRIPTIVES
# =============================================================================

descriptive_rows = []


for baseline_name, delta_column in [
    (
        "Baseline A",
        "delta_a"
    ),
    (
        "Baseline B",
        "delta_b"
    )
]:

    for group in [
        "Active",
        "Passive"
    ]:

        for band in BANDS:

            temp = global_df.loc[
                (
                    global_df[
                        "group"
                    ]
                    ==
                    group
                )
                &
                (
                    global_df[
                        "band"
                    ]
                    ==
                    band
                )
            ]


            descriptive_rows.append({
                "baseline":
                    baseline_name,

                "group":
                    group,

                "band":
                    band,

                "n":
                    len(
                        temp
                    ),

                "mean_delta":
                    float(
                        temp[
                            delta_column
                        ].mean()
                    ),

                "sd_delta":
                    float(
                        temp[
                            delta_column
                        ].std(
                            ddof=1
                        )
                    )
            })


descriptive_df = pd.DataFrame(
    descriptive_rows
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MATCHED FIXED-9 GROUP DESCRIPTIVES"
)

print(
    "=" * 100
)

display(
    descriptive_df.round(
        6
    )
)


# =============================================================================
# 13. EXACT INFERENCE FOR EACH BASELINE
# =============================================================================

inference_rows = []


for baseline_index, (
    baseline_name,
    delta_column
) in enumerate(
    [
        (
            "Baseline A",
            "delta_a"
        ),
        (
            "Baseline B",
            "delta_b"
        )
    ]
):

    for band_index, band in enumerate(
        BANDS
    ):

        temp = global_df.loc[
            global_df[
                "band"
            ]
            ==
            band
        ]


        active = (
            temp.loc[
                temp[
                    "group"
                ]
                ==
                "Active",
                delta_column
            ]
            .to_numpy(
                dtype=float
            )
        )


        passive = (
            temp.loc[
                temp[
                    "group"
                ]
                ==
                "Passive",
                delta_column
            ]
            .to_numpy(
                dtype=float
            )
        )


        difference, p_value, n_perm = (
            exact_permutation_test(
                active,
                passive
            )
        )


        ci_low, ci_high = bootstrap_ci(
            active,
            passive,
            seed=(
                RANDOM_SEED
                +
                baseline_index
                *
                100
                +
                band_index
            )
        )


        inference_rows.append({
            "baseline":
                baseline_name,

            "band":
                band,

            "n_active":
                len(
                    active
                ),

            "n_passive":
                len(
                    passive
                ),

            "active_mean_delta":
                float(
                    np.mean(
                        active
                    )
                ),

            "active_sd_delta":
                float(
                    np.std(
                        active,
                        ddof=1
                    )
                ),

            "passive_mean_delta":
                float(
                    np.mean(
                        passive
                    )
                ),

            "passive_sd_delta":
                float(
                    np.std(
                        passive,
                        ddof=1
                    )
                ),

            "active_minus_passive":
                difference,

            "bootstrap_ci_low":
                ci_low,

            "bootstrap_ci_high":
                ci_high,

            "hedges_g":
                hedges_g(
                    active,
                    passive
                ),

            "exact_permutation_p":
                p_value,

            "n_exact_permutations":
                n_perm
        })


inference_df = pd.DataFrame(
    inference_rows
)


inference_df[
    "p_bh_fdr_3"
] = np.nan


for baseline_name in [
    "Baseline A",
    "Baseline B"
]:

    mask = (
        inference_df[
            "baseline"
        ]
        ==
        baseline_name
    )


    inference_df.loc[
        mask,
        "p_bh_fdr_3"
    ] = bh_fdr(
        inference_df.loc[
            mask,
            "exact_permutation_p"
        ].to_numpy(
            dtype=float
        )
    )


inference_df[
    "significant_raw_0_05"
] = (
    inference_df[
        "exact_permutation_p"
    ]
    <
    0.05
)


inference_df[
    "significant_fdr_0_05"
] = (
    inference_df[
        "p_bh_fdr_3"
    ]
    <
    0.05
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "MATCHED FIXED-9 ACTIVE vs PASSIVE INFERENCE"
)

print(
    "=" * 100
)

display(
    inference_df.round(
        6
    )
)


# =============================================================================
# 14. CROSS-BASELINE EFFECT STABILITY
# =============================================================================

stability_rows = []


for band in BANDS:

    temp = (
        global_df.loc[
            global_df[
                "band"
            ]
            ==
            band
        ]
        .sort_values(
            "subject"
        )
        .reset_index(
            drop=True
        )
    )


    delta_a = temp[
        "delta_a"
    ].to_numpy(
        dtype=float
    )


    delta_b = temp[
        "delta_b"
    ].to_numpy(
        dtype=float
    )


    pearson_r, pearson_p = pearsonr(
        delta_a,
        delta_b
    )


    spearman_rho, spearman_p = spearmanr(
        delta_a,
        delta_b
    )


    participant_same_sign = (
        np.sign(
            delta_a
        )
        ==
        np.sign(
            delta_b
        )
    )


    result_a = inference_df.loc[
        (
            inference_df[
                "baseline"
            ]
            ==
            "Baseline A"
        )
        &
        (
            inference_df[
                "band"
            ]
            ==
            band
        )
    ].iloc[
        0
    ]


    result_b = inference_df.loc[
        (
            inference_df[
                "baseline"
            ]
            ==
            "Baseline B"
        )
        &
        (
            inference_df[
                "band"
            ]
            ==
            band
        )
    ].iloc[
        0
    ]


    diff_a = float(
        result_a[
            "active_minus_passive"
        ]
    )


    diff_b = float(
        result_b[
            "active_minus_passive"
        ]
    )


    g_a = float(
        result_a[
            "hedges_g"
        ]
    )


    g_b = float(
        result_b[
            "hedges_g"
        ]
    )


    stability_rows.append({
        "band":
            band,

        "baseline_a_active_minus_passive":
            diff_a,

        "baseline_b_active_minus_passive":
            diff_b,

        "same_group_effect_direction":
            bool(
                np.sign(
                    diff_a
                )
                ==
                np.sign(
                    diff_b
                )
            ),

        "absolute_effect_difference":
            float(
                abs(
                    diff_b
                    -
                    diff_a
                )
            ),

        "baseline_a_hedges_g":
            g_a,

        "baseline_b_hedges_g":
            g_b,

        "hedges_g_change_b_minus_a":
            float(
                g_b
                -
                g_a
            ),

        "participant_same_sign_n":
            int(
                participant_same_sign.sum()
            ),

        "participant_same_sign_total":
            len(
                participant_same_sign
            ),

        "participant_same_sign_percent":
            float(
                100
                *
                participant_same_sign.mean()
            ),

        "pearson_r":
            float(
                pearson_r
            ),

        "pearson_p":
            float(
                pearson_p
            ),

        "spearman_rho":
            float(
                spearman_rho
            ),

        "spearman_p":
            float(
                spearman_p
            )
    })


stability_df = pd.DataFrame(
    stability_rows
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "CROSS-BASELINE EFFECT STABILITY"
)

print(
    "=" * 100
)

display(
    stability_df.round(
        6
    )
)


# =============================================================================
# 15. FOCUSED BETA COMPARISON
# =============================================================================

beta_inference = inference_df.loc[
    inference_df[
        "band"
    ]
    ==
    "Beta"
].copy()


beta_stability = stability_df.loc[
    stability_df[
        "band"
    ]
    ==
    "Beta"
].copy()


print(
    "\n"
    +
    "=" * 100
)

print(
    "BETA — MATCHED BASELINE A vs BASELINE B"
)

print(
    "=" * 100
)

display(
    beta_inference[
        [
            "baseline",
            "active_mean_delta",
            "passive_mean_delta",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_fdr_3"
        ]
    ].round(
        6
    )
)


print(
    "\nBeta cross-baseline stability:"
)

display(
    beta_stability.round(
        6
    )
)


# =============================================================================
# 16. LEAVE-ONE-OUT GROUP-EFFECT ROBUSTNESS
# =============================================================================

loo_rows = []


for baseline_name, delta_column in [
    (
        "Baseline A",
        "delta_a"
    ),
    (
        "Baseline B",
        "delta_b"
    )
]:

    for band in BANDS:

        temp = (
            global_df.loc[
                global_df[
                    "band"
                ]
                ==
                band
            ]
            .sort_values(
                "subject"
            )
            .reset_index(
                drop=True
            )
        )


        primary = inference_df.loc[
            (
                inference_df[
                    "baseline"
                ]
                ==
                baseline_name
            )
            &
            (
                inference_df[
                    "band"
                ]
                ==
                band
            )
        ].iloc[
            0
        ]


        primary_difference = float(
            primary[
                "active_minus_passive"
            ]
        )


        for removed_subject in temp[
            "subject"
        ]:

            reduced = temp.loc[
                temp[
                    "subject"
                ]
                !=
                removed_subject
            ]


            active = reduced.loc[
                reduced[
                    "group"
                ]
                ==
                "Active",
                delta_column
            ].to_numpy(
                dtype=float
            )


            passive = reduced.loc[
                reduced[
                    "group"
                ]
                ==
                "Passive",
                delta_column
            ].to_numpy(
                dtype=float
            )


            difference = float(
                np.mean(
                    active
                )
                -
                np.mean(
                    passive
                )
            )


            loo_rows.append({
                "baseline":
                    baseline_name,

                "band":
                    band,

                "removed_subject":
                    removed_subject,

                "active_minus_passive":
                    difference,

                "hedges_g":
                    hedges_g(
                        active,
                        passive
                    ),

                "same_direction_as_full":
                    bool(
                        np.sign(
                            difference
                        )
                        ==
                        np.sign(
                            primary_difference
                        )
                    )
            })


loo_df = pd.DataFrame(
    loo_rows
)


loo_summary = (
    loo_df
    .groupby(
        [
            "baseline",
            "band"
        ]
    )
    .agg(
        same_direction_n=(
            "same_direction_as_full",
            "sum"
        ),
        total=(
            "same_direction_as_full",
            "size"
        ),
        difference_min=(
            "active_minus_passive",
            "min"
        ),
        difference_max=(
            "active_minus_passive",
            "max"
        ),
        g_min=(
            "hedges_g",
            "min"
        ),
        g_max=(
            "hedges_g",
            "max"
        )
    )
    .reset_index()
)


loo_summary[
    "same_direction_percent"
] = (
    100
    *
    loo_summary[
        "same_direction_n"
    ]
    /
    loo_summary[
        "total"
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "LEAVE-ONE-OUT GROUP-EFFECT ROBUSTNESS"
)

print(
    "=" * 100
)

display(
    loo_summary.round(
        6
    )
)


# =============================================================================
# 17. FIGURE 1 — EFFECT SIZE ACROSS BASELINES
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        8.5,
        5.5
    )
)


x = np.arange(
    len(
        BANDS
    )
)


width = 0.34


g_a = []

g_b = []


for band in BANDS:

    g_a.append(
        inference_df.loc[
            (
                inference_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                inference_df[
                    "band"
                ]
                ==
                band
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


    g_b.append(
        inference_df.loc[
            (
                inference_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                inference_df[
                    "band"
                ]
                ==
                band
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


ax.bar(
    x - width / 2,
    g_a,
    width,
    label="Baseline A"
)


ax.bar(
    x + width / 2,
    g_b,
    width,
    label="Baseline B"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)

ax.set_xticklabels(
    list(
        BANDS.keys()
    )
)


ax.set_xlabel(
    "Frequency band"
)

ax.set_ylabel(
    "Hedges' g (Active − Passive)"
)

ax.set_title(
    "Matched fixed-9 connectivity effects across baseline states"
)

ax.legend(
    frameon=False
)

fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "matched_fixed9_effect_sizes_across_baselines.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# =============================================================================
# 18. FIGURE 2 — PARTICIPANT-LEVEL CROSS-BASELINE DELTAS
# =============================================================================

for band in BANDS:

    temp = (
        global_df.loc[
            global_df[
                "band"
            ]
            ==
            band
        ]
        .sort_values(
            "subject"
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            6.5,
            6.5
        )
    )


    for group, marker in [
        (
            "Active",
            "o"
        ),
        (
            "Passive",
            "s"
        )
    ]:

        group_df = temp.loc[
            temp[
                "group"
            ]
            ==
            group
        ]


        ax.scatter(
            group_df[
                "delta_a"
            ],
            group_df[
                "delta_b"
            ],
            s=70,
            marker=marker,
            label=group
        )


    minimum = float(
        min(
            temp[
                "delta_a"
            ].min(),
            temp[
                "delta_b"
            ].min()
        )
    )


    maximum = float(
        max(
            temp[
                "delta_a"
            ].max(),
            temp[
                "delta_b"
            ].max()
        )
    )


    ax.plot(
        [
            minimum,
            maximum
        ],
        [
            minimum,
            maximum
        ],
        linestyle="--",
        linewidth=1
    )


    ax.axhline(
        0,
        linewidth=0.8
    )


    ax.axvline(
        0,
        linewidth=0.8
    )


    ax.set_xlabel(
        "Task − Baseline A"
    )

    ax.set_ylabel(
        "Task − Baseline B"
    )


    ax.set_title(
        f"{band}: participant-level baseline sensitivity"
    )


    ax.legend(
        frameon=False
    )


    fig.tight_layout()


    fig.savefig(
        FIG_OUT
        /
        (
            f"{band.lower()}_"
            "participant_cross_baseline_sensitivity.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )


    plt.show()


# =============================================================================
# 19. FIGURE 3 — BETA PAIRED BASELINE EFFECT
# =============================================================================

beta = (
    global_df.loc[
        global_df[
            "band"
        ]
        ==
        "Beta"
    ]
    .sort_values(
        "subject"
    )
)


fig, ax = plt.subplots(
    figsize=(
        8,
        5.5
    )
)


for _, row in beta.iterrows():

    ax.plot(
        [
            0,
            1
        ],
        [
            row[
                "delta_a"
            ],
            row[
                "delta_b"
            ]
        ],
        marker="o",
        linewidth=1,
        alpha=0.7
    )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    [
        0,
        1
    ]
)

ax.set_xticklabels(
    [
        "Baseline A",
        "Baseline B"
    ]
)


ax.set_ylabel(
    "Beta Task − Baseline global wPLI²_debiased"
)

ax.set_title(
    "Participant-level beta connectivity across baseline definitions"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "beta_paired_baseline_sensitivity.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 20. SAVE OUTPUTS
# =============================================================================

selection_df.to_csv(
    OUT
    /
    "fixed9_epoch_selection.csv",
    index=False
)


audit_df.to_csv(
    OUT
    /
    "epoch_qc_audit.csv",
    index=False
)


global_df.to_csv(
    OUT
    /
    "participant_global_connectivity.csv",
    index=False
)


edge_df.to_csv(
    OUT
    /
    "edge_level_connectivity.csv",
    index=False
)


matrix_qc_df.to_csv(
    OUT
    /
    "matrix_numerical_qc.csv",
    index=False
)


descriptive_df.to_csv(
    OUT
    /
    "group_descriptive_summary.csv",
    index=False
)


inference_df.to_csv(
    OUT
    /
    "matched_fixed9_exact_inference.csv",
    index=False
)


stability_df.to_csv(
    OUT
    /
    "cross_baseline_stability.csv",
    index=False
)


loo_df.to_csv(
    OUT
    /
    "leave_one_out_results.csv",
    index=False
)


loo_summary.to_csv(
    OUT
    /
    "leave_one_out_summary.csv",
    index=False
)


# =============================================================================
# 21. FINAL INTEGRITY CHECK
# =============================================================================

integrity = {
    "participants":
        len(
            subject_data
        )
        ==
        15,

    "active":
        (
            selection_df[
                "group"
            ]
            ==
            "Active"
        ).sum()
        ==
        8,

    "passive":
        (
            selection_df[
                "group"
            ]
            ==
            "Passive"
        ).sum()
        ==
        7,

    "baseline_a_fixed9":
        (
            selection_df[
                "baseline_a_used"
            ]
            ==
            FIXED_N
        ).all(),

    "baseline_b_fixed9":
        (
            selection_df[
                "baseline_b_used"
            ]
            ==
            FIXED_N
        ).all(),

    "task_fixed9":
        (
            selection_df[
                "task_used"
            ]
            ==
            FIXED_N
        ).all(),

    "same_task_for_both":
        selection_df[
            "same_task_for_both_normalizations"
        ].all(),

    "edge_rows":
        len(
            edge_df
        )
        ==
        expected_edge_rows,

    "global_rows":
        len(
            global_df
        )
        ==
        45,

    "inference_rows":
        len(
            inference_df
        )
        ==
        6,

    "stability_rows":
        len(
            stability_df
        )
        ==
        3,

    "exact_permutations":
        (
            inference_df[
                "n_exact_permutations"
            ]
            ==
            EXPECTED_PERMUTATIONS
        ).all(),

    "finite_matrices":
        matrix_qc_df[
            "all_finite"
        ].all(),

    "symmetric_matrices":
        matrix_qc_df[
            "symmetric"
        ].all(),

    "zero_diagonal":
        matrix_qc_df[
            "zero_diagonal"
        ].all()
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "TASK 6C COMPLETE"
)

print(
    "=" * 100
)


print(
    "Design:"
)

print(
    "  Baseline A             : position 1 / T2_V1"
)

print(
    "  Baseline B             : position 3 / T2_V3"
)

print(
    "  Epochs/state           : 9"
)

print(
    "  Seconds/state          : 18"
)

print(
    "  Same Task epochs A/B   : YES"
)

print(
    "  Estimator              : wPLI²_debiased"
)

print(
    "  Spectral mode          : multitaper"
)

print(
    "  Bandwidth              : 2.0 Hz"
)

print(
    f"  Exact allocations/test : {EXPECTED_PERMUTATIONS}"
)


print(
    "\nIntegrity components:"
)


for key, value in integrity.items():

    print(
        f"  {key:<27}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 100
)

print(
    "OVERALL INTEGRITY CHECK:",
    (
        "PASS"
        if overall_pass
        else "FAIL"
    )
)

print(
    "-" * 100
)


if not overall_pass:

    raise RuntimeError(
        "Task 6C integrity check failed."
    )


# =============================================================================
# 22. AUTOMATIC CONSERVATIVE INTERPRETATION
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "BASELINE-ROBUSTNESS SUMMARY"
)

print(
    "=" * 100
)


for _, row in stability_df.iterrows():

    band = row[
        "band"
    ]


    print(
        f"\n{band}:"
    )

    print(
        f"  Baseline-A effect   : "
        f"{row['baseline_a_active_minus_passive']:+.6f}"
    )

    print(
        f"  Baseline-B effect   : "
        f"{row['baseline_b_active_minus_passive']:+.6f}"
    )

    print(
        f"  Same group direction: "
        f"{row['same_group_effect_direction']}"
    )

    print(
        f"  Hedges g A          : "
        f"{row['baseline_a_hedges_g']:+.3f}"
    )

    print(
        f"  Hedges g B          : "
        f"{row['baseline_b_hedges_g']:+.3f}"
    )

    print(
        f"  Participant sign agreement: "
        f"{int(row['participant_same_sign_n'])}/"
        f"{int(row['participant_same_sign_total'])} "
        f"({row['participant_same_sign_percent']:.1f}%)"
    )

    print(
        f"  Pearson r           : "
        f"{row['pearson_r']:+.3f} "
        f"(p={row['pearson_p']:.4f})"
    )

    print(
        f"  Spearman rho        : "
        f"{row['spearman_rho']:+.3f} "
        f"(p={row['spearman_p']:.4f})"
    )


print(
    "\nINTERPRETATION RULE:"
)

print(
    "A connectivity effect is considered baseline-robust only when "
    "its Active-Passive direction is preserved and its magnitude "
    "remains reasonably comparable across the two marker-defined "
    "baseline states."
)

print(
    "Statistical significance under only one baseline is not sufficient "
    "evidence of a robust group difference."
)

print(
    "Participant-level cross-baseline correlation describes stability "
    "of individual connectivity changes and is not itself evidence of "
    "an Active-Passive group effect."
)

print(
    "No semantic Eyes-Open/Eyes-Closed label is assigned to either "
    "baseline in this analysis."
)


print(
    f"\nOutput directory:\n{OUT}"
)

print(
    "\n"
    +
    "=" * 100
)

In [ ]:
# =============================================================================
# TASK 7A — VALIDATED ROI-LEVEL NETWORK LOCALIZATION
# MATCHED FIXED-9, BASELINE A + BASELINE B
# =============================================================================
#
# INPUT:
#   Task 6C validated edge-level connectivity
#
# ESTIMATOR:
#   debiased squared weighted phase-lag index (wPLI²_debiased)
#
# PARTICIPANTS:
#   Active  = 8
#   Passive = 7
#
# BANDS:
#   Theta, Alpha, Beta
#
# ROIs:
#   Frontal:
#       AF3, F7, F3, F4, F8, AF4
#
#   CentralTemporal:
#       FC5, T7, T8, FC6
#
#   Posterior:
#       P7, O1, O2, P8
#
# NETWORK CLASSES:
#   Frontal-Frontal
#   Frontal-CentralTemporal
#   Frontal-Posterior
#   CentralTemporal-CentralTemporal
#   CentralTemporal-Posterior
#   Posterior-Posterior
#
# OUTCOME:
#   Participant-level mean edge change within each network:
#
#       Delta_A = Task - Baseline A
#       Delta_B = Task - Baseline B
#
# INFERENCE:
#   Exact Active-vs-Passive permutation:
#       C(15,8) = 6435 allocations
#
#   Effect size:
#       Hedges g
#
#   Uncertainty:
#       10,000 stratified bootstrap iterations
#
# MULTIPLICITY:
#   1. FDR18 separately for Baseline A and Baseline B:
#          3 bands × 6 networks
#
#   2. Focused beta FDR6 separately for each baseline:
#          6 networks
#
# IMPORTANT:
#   - All 3 bands are retained.
#   - Beta is the focused follow-up because Task 6C showed that beta alone
#     preserved the Active-Passive direction across both baselines.
#   - No individual edge significance testing.
#   - No thresholded graph analysis.
#   - Negative wPLI²_debiased estimates are retained.
#   - Baseline A/B remain neutral marker-defined labels.
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

DERIVATIVES = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK6C = (
    DERIVATIVES
    / "task06c_matched_fixed9_baseline_sensitivity"
)

EDGE_FILE = (
    TASK6C
    / "edge_level_connectivity.csv"
)

GLOBAL_FILE = (
    TASK6C
    / "participant_global_connectivity.csv"
)

SELECTION_FILE = (
    TASK6C
    / "fixed9_epoch_selection.csv"
)

OUT = (
    DERIVATIVES
    / "task07a_validated_roi_network_localization"
)

FIG_OUT = (
    OUT
    / "figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. VERIFY INPUTS
# =============================================================================

for path in [
    EDGE_FILE,
    GLOBAL_FILE,
    SELECTION_FILE
]:

    if not path.exists():

        raise FileNotFoundError(
            f"Required Task 6C input not found:\n{path}"
        )


# =============================================================================
# 3. LOAD TASK 6C DATA
# =============================================================================

edge_df = pd.read_csv(
    EDGE_FILE
)

global_df = pd.read_csv(
    GLOBAL_FILE
)

selection_df = pd.read_csv(
    SELECTION_FILE
)


# =============================================================================
# 4. CONSTANTS
# =============================================================================

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(
        1,
        16
    )
]

BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]

BASELINES = {
    "Baseline A": "delta_a",
    "Baseline B": "delta_b"
}

ROIS = {
    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "CentralTemporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}

NETWORKS = [
    (
        "Frontal",
        "Frontal"
    ),
    (
        "Frontal",
        "CentralTemporal"
    ),
    (
        "Frontal",
        "Posterior"
    ),
    (
        "CentralTemporal",
        "CentralTemporal"
    ),
    (
        "CentralTemporal",
        "Posterior"
    ),
    (
        "Posterior",
        "Posterior"
    )
]

NETWORK_ORDER = [
    "Frontal-Frontal",
    "Frontal-CentralTemporal",
    "Frontal-Posterior",
    "CentralTemporal-CentralTemporal",
    "CentralTemporal-Posterior",
    "Posterior-Posterior"
]

EXPECTED_PARTICIPANTS = 15

EXPECTED_ACTIVE = 8

EXPECTED_PASSIVE = 7

EXPECTED_EDGES = 91

EXPECTED_EDGE_ROWS = (
    15
    *
    3
    *
    91
)

EXPECTED_NETWORK_ROWS = (
    15
    *
    3
    *
    6
)

EXPECTED_TESTS_PER_BASELINE = (
    3
    *
    6
)

EXPECTED_PERMUTATIONS = math.comb(
    15,
    8
)

N_BOOTSTRAP = 10_000

RANDOM_SEED = 20260913


# =============================================================================
# 5. INPUT INTEGRITY
# =============================================================================

required_edge_columns = {
    "subject",
    "group",
    "band",
    "edge_index",
    "channel_i",
    "channel_j",
    "baseline_a",
    "baseline_b",
    "task",
    "delta_a",
    "delta_b"
}

missing = (
    required_edge_columns
    -
    set(
        edge_df.columns
    )
)

if missing:

    raise RuntimeError(
        "Task 6C edge file is missing columns: "
        f"{sorted(missing)}"
    )


if len(edge_df) != EXPECTED_EDGE_ROWS:

    raise RuntimeError(
        f"Expected {EXPECTED_EDGE_ROWS} Task 6C edge rows; "
        f"found {len(edge_df)}."
    )


if edge_df[
    "subject"
].nunique() != EXPECTED_PARTICIPANTS:

    raise RuntimeError(
        "Expected 15 participants in Task 6C edge data."
    )


if set(
    edge_df[
        "band"
    ].unique()
) != set(
    BANDS
):

    raise RuntimeError(
        "Unexpected frequency-band set."
    )


# =============================================================================
# 6. MAP CHANNELS TO ROI
# =============================================================================

channel_to_roi = {}

for roi, channels in ROIS.items():

    for channel in channels:

        if channel in channel_to_roi:

            raise RuntimeError(
                f"Channel {channel} assigned to multiple ROIs."
            )

        channel_to_roi[
            channel
        ] = roi


all_roi_channels = set(
    channel_to_roi.keys()
)

all_edge_channels = set(
    edge_df[
        "channel_i"
    ]
).union(
    set(
        edge_df[
            "channel_j"
        ]
    )
)


if all_edge_channels != all_roi_channels:

    raise RuntimeError(
        "\nROI/channel mismatch.\n"
        f"Task 6C channels: {sorted(all_edge_channels)}\n"
        f"ROI channels: {sorted(all_roi_channels)}"
    )


edge_df[
    "roi_i"
] = edge_df[
    "channel_i"
].map(
    channel_to_roi
)

edge_df[
    "roi_j"
] = edge_df[
    "channel_j"
].map(
    channel_to_roi
)


# =============================================================================
# 7. CANONICAL NETWORK LABEL
# =============================================================================

roi_order = {
    "Frontal": 0,
    "CentralTemporal": 1,
    "Posterior": 2
}


def canonical_network(
    roi_i,
    roi_j
):

    if roi_order[
        roi_i
    ] <= roi_order[
        roi_j
    ]:

        return (
            f"{roi_i}-{roi_j}"
        )

    return (
        f"{roi_j}-{roi_i}"
    )


edge_df[
    "network"
] = [
    canonical_network(
        a,
        b
    )
    for a, b in zip(
        edge_df[
            "roi_i"
        ],
        edge_df[
            "roi_j"
        ]
    )
]


# =============================================================================
# 8. EXPECTED EDGE COUNTS PER NETWORK
# =============================================================================

# Frontal = 6 channels
# CT      = 4
# Posterior = 4
#
# Within:
#   F-F = C(6,2) = 15
#   CT-CT = C(4,2) = 6
#   P-P = C(4,2) = 6
#
# Between:
#   F-CT = 6*4 = 24
#   F-P  = 6*4 = 24
#   CT-P = 4*4 = 16
#
# Total = 91.

EXPECTED_NETWORK_EDGE_COUNTS = {
    "Frontal-Frontal": 15,
    "Frontal-CentralTemporal": 24,
    "Frontal-Posterior": 24,
    "CentralTemporal-CentralTemporal": 6,
    "CentralTemporal-Posterior": 16,
    "Posterior-Posterior": 6
}


observed_network_counts = (
    edge_df[
        [
            "edge_index",
            "channel_i",
            "channel_j",
            "network"
        ]
    ]
    .drop_duplicates()
    .groupby(
        "network"
    )
    .size()
    .to_dict()
)


print(
    "=" * 100
)

print(
    "TASK 7A — VALIDATED ROI-LEVEL NETWORK LOCALIZATION"
)

print(
    "=" * 100
)

print(
    f"Participants                : {edge_df['subject'].nunique()}"
)

print(
    f"Active                      : "
    f"{selection_df.loc[selection_df['group']=='Active','subject'].nunique()}"
)

print(
    f"Passive                     : "
    f"{selection_df.loc[selection_df['group']=='Passive','subject'].nunique()}"
)

print(
    f"Bands                       : {', '.join(BANDS)}"
)

print(
    "Baselines                   : Baseline A + Baseline B"
)

print(
    "Estimator                   : wPLI²_debiased"
)

print(
    "Epoch design                : matched fixed-9"
)

print(
    f"Exact allocations/test      : {EXPECTED_PERMUTATIONS}"
)

print(
    f"Bootstrap iterations/test   : {N_BOOTSTRAP:,}"
)


print(
    "\nNetwork edge counts:"
)

for network in NETWORK_ORDER:

    observed = observed_network_counts.get(
        network,
        0
    )

    expected = EXPECTED_NETWORK_EDGE_COUNTS[
        network
    ]

    print(
        f"  {network:<38} "
        f"{observed:>2}/{expected:<2} "
        f"{'PASS' if observed == expected else 'FAIL'}"
    )


if observed_network_counts != EXPECTED_NETWORK_EDGE_COUNTS:

    raise RuntimeError(
        "ROI network edge-count integrity check failed."
    )


# =============================================================================
# 9. PARTICIPANT-LEVEL NETWORK AGGREGATION
# =============================================================================
#
# We aggregate edges WITHIN participant first.
#
# The participant remains the inferential unit.
# =============================================================================

network_df = (
    edge_df
    .groupby(
        [
            "subject",
            "group",
            "band",
            "network"
        ],
        as_index=False
    )
    .agg(
        n_edges=(
            "edge_index",
            "count"
        ),

        baseline_a=(
            "baseline_a",
            "mean"
        ),

        baseline_b=(
            "baseline_b",
            "mean"
        ),

        task=(
            "task",
            "mean"
        ),

        delta_a=(
            "delta_a",
            "mean"
        ),

        delta_b=(
            "delta_b",
            "mean"
        )
    )
)


if len(network_df) != EXPECTED_NETWORK_ROWS:

    raise RuntimeError(
        f"Expected {EXPECTED_NETWORK_ROWS} participant-network-band rows; "
        f"found {len(network_df)}."
    )


# Verify every participant/network has expected number of edges.

edge_count_check = (
    network_df[
        [
            "network",
            "n_edges"
        ]
    ]
    .drop_duplicates()
)


for _, row in edge_count_check.iterrows():

    network = row[
        "network"
    ]

    observed = int(
        row[
            "n_edges"
        ]
    )

    expected = EXPECTED_NETWORK_EDGE_COUNTS[
        network
    ]

    if observed != expected:

        raise RuntimeError(
            f"{network}: expected {expected} edges, "
            f"found {observed}."
        )


# =============================================================================
# 10. STATISTICAL FUNCTIONS
# =============================================================================

def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    n1 = len(
        active
    )

    n2 = len(
        passive
    )


    var1 = np.var(
        active,
        ddof=1
    )

    var2 = np.var(
        passive,
        ddof=1
    )


    pooled_var = (
        (
            (n1 - 1)
            *
            var1
        )
        +
        (
            (n2 - 1)
            *
            var2
        )
    ) / (
        n1
        +
        n2
        -
        2
    )


    if (
        not np.isfinite(
            pooled_var
        )
        or
        pooled_var <= 0
    ):

        return np.nan


    d = (
        np.mean(
            active
        )
        -
        np.mean(
            passive
        )
    ) / np.sqrt(
        pooled_var
    )


    df = (
        n1
        +
        n2
        -
        2
    )


    J = (
        1.0
        -
        3.0
        /
        (
            4.0
            *
            df
            -
            1.0
        )
    )


    return float(
        J
        *
        d
    )


def exact_permutation_test(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    pooled = np.concatenate(
        [
            active,
            passive
        ]
    )


    n_active = len(
        active
    )


    observed = float(
        np.mean(
            active
        )
        -
        np.mean(
            passive
        )
    )


    n_total = len(
        pooled
    )


    all_indices = np.arange(
        n_total
    )


    permutation_differences = np.empty(
        EXPECTED_PERMUTATIONS,
        dtype=float
    )


    counter = 0


    for active_tuple in combinations(
        range(
            n_total
        ),
        n_active
    ):

        active_idx = np.asarray(
            active_tuple,
            dtype=int
        )


        passive_mask = np.ones(
            n_total,
            dtype=bool
        )


        passive_mask[
            active_idx
        ] = False


        passive_idx = all_indices[
            passive_mask
        ]


        permutation_differences[
            counter
        ] = (
            np.mean(
                pooled[
                    active_idx
                ]
            )
            -
            np.mean(
                pooled[
                    passive_idx
                ]
            )
        )


        counter += 1


    if counter != EXPECTED_PERMUTATIONS:

        raise RuntimeError(
            f"Expected {EXPECTED_PERMUTATIONS} permutations; "
            f"generated {counter}."
        )


    p_value = float(
        np.mean(
            np.abs(
                permutation_differences
            )
            >=
            (
                abs(
                    observed
                )
                -
                1e-15
            )
        )
    )


    return (
        observed,
        p_value,
        counter
    )


def stratified_bootstrap_ci(
    active,
    passive,
    seed
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )


    bootstrap_differences = np.empty(
        N_BOOTSTRAP,
        dtype=float
    )


    for i in range(
        N_BOOTSTRAP
    ):

        active_sample = rng.choice(
            active,
            size=len(
                active
            ),
            replace=True
        )


        passive_sample = rng.choice(
            passive,
            size=len(
                passive
            ),
            replace=True
        )


        bootstrap_differences[
            i
        ] = (
            np.mean(
                active_sample
            )
            -
            np.mean(
                passive_sample
            )
        )


    low, high = np.percentile(
        bootstrap_differences,
        [
            2.5,
            97.5
        ]
    )


    return (
        float(
            low
        ),
        float(
            high
        )
    )


def bh_fdr(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    ranked = p_values[
        order
    ]


    adjusted_ranked = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted_ranked = np.clip(
        adjusted_ranked,
        0.0,
        1.0
    )


    adjusted = np.empty(
        m,
        dtype=float
    )


    adjusted[
        order
    ] = adjusted_ranked


    return adjusted


# =============================================================================
# 11. GROUP DESCRIPTIVE SUMMARY
# =============================================================================

descriptive_rows = []


for baseline_name, delta_column in BASELINES.items():

    for band in BANDS:

        for network in NETWORK_ORDER:

            temp = network_df.loc[
                (
                    network_df[
                        "band"
                    ]
                    ==
                    band
                )
                &
                (
                    network_df[
                        "network"
                    ]
                    ==
                    network
                )
            ]


            for group in [
                "Active",
                "Passive"
            ]:

                values = temp.loc[
                    temp[
                        "group"
                    ]
                    ==
                    group,
                    delta_column
                ].to_numpy(
                    dtype=float
                )


                descriptive_rows.append({
                    "baseline":
                        baseline_name,

                    "band":
                        band,

                    "network":
                        network,

                    "group":
                        group,

                    "n":
                        len(
                            values
                        ),

                    "mean_delta":
                        float(
                            np.mean(
                                values
                            )
                        ),

                    "sd_delta":
                        float(
                            np.std(
                                values,
                                ddof=1
                            )
                        ),

                    "median_delta":
                        float(
                            np.median(
                                values
                            )
                        )
                })


descriptive_df = pd.DataFrame(
    descriptive_rows
)


# =============================================================================
# 12. EXACT NETWORK INFERENCE
# =============================================================================

result_rows = []


test_counter = 0


for baseline_index, (
    baseline_name,
    delta_column
) in enumerate(
    BASELINES.items()
):

    for band_index, band in enumerate(
        BANDS
    ):

        for network_index, network in enumerate(
            NETWORK_ORDER
        ):

            temp = network_df.loc[
                (
                    network_df[
                        "band"
                    ]
                    ==
                    band
                )
                &
                (
                    network_df[
                        "network"
                    ]
                    ==
                    network
                )
            ]


            active = temp.loc[
                temp[
                    "group"
                ]
                ==
                "Active",
                delta_column
            ].to_numpy(
                dtype=float
            )


            passive = temp.loc[
                temp[
                    "group"
                ]
                ==
                "Passive",
                delta_column
            ].to_numpy(
                dtype=float
            )


            if len(
                active
            ) != EXPECTED_ACTIVE:

                raise RuntimeError(
                    f"{baseline_name} / {band} / {network}: "
                    f"expected 8 Active participants; "
                    f"found {len(active)}."
                )


            if len(
                passive
            ) != EXPECTED_PASSIVE:

                raise RuntimeError(
                    f"{baseline_name} / {band} / {network}: "
                    f"expected 7 Passive participants; "
                    f"found {len(passive)}."
                )


            difference, p_value, n_perm = (
                exact_permutation_test(
                    active,
                    passive
                )
            )


            ci_low, ci_high = (
                stratified_bootstrap_ci(
                    active,
                    passive,
                    seed=(
                        RANDOM_SEED
                        +
                        baseline_index
                        *
                        1000
                        +
                        band_index
                        *
                        100
                        +
                        network_index
                    )
                )
            )


            result_rows.append({
                "baseline":
                    baseline_name,

                "band":
                    band,

                "network":
                    network,

                "n_edges":
                    EXPECTED_NETWORK_EDGE_COUNTS[
                        network
                    ],

                "n_active":
                    len(
                        active
                    ),

                "n_passive":
                    len(
                        passive
                    ),

                "active_mean_delta":
                    float(
                        np.mean(
                            active
                        )
                    ),

                "active_sd_delta":
                    float(
                        np.std(
                            active,
                            ddof=1
                        )
                    ),

                "passive_mean_delta":
                    float(
                        np.mean(
                            passive
                        )
                    ),

                "passive_sd_delta":
                    float(
                        np.std(
                            passive,
                            ddof=1
                        )
                    ),

                "active_minus_passive":
                    difference,

                "bootstrap_ci_low":
                    ci_low,

                "bootstrap_ci_high":
                    ci_high,

                "hedges_g":
                    hedges_g(
                        active,
                        passive
                    ),

                "exact_permutation_p":
                    p_value,

                "n_exact_permutations":
                    n_perm
            })


            test_counter += 1


results_df = pd.DataFrame(
    result_rows
)


expected_total_tests = (
    2
    *
    18
)


if test_counter != expected_total_tests:

    raise RuntimeError(
        f"Expected {expected_total_tests} tests; "
        f"ran {test_counter}."
    )


# =============================================================================
# 13. FDR18 — FULL BAND × NETWORK FAMILY, SEPARATELY BY BASELINE
# =============================================================================

results_df[
    "p_bh_fdr18"
] = np.nan


for baseline_name in BASELINES:

    mask = (
        results_df[
            "baseline"
        ]
        ==
        baseline_name
    )


    if mask.sum() != 18:

        raise RuntimeError(
            f"{baseline_name}: expected 18 tests "
            f"for FDR18; found {mask.sum()}."
        )


    results_df.loc[
        mask,
        "p_bh_fdr18"
    ] = bh_fdr(
        results_df.loc[
            mask,
            "exact_permutation_p"
        ].to_numpy(
            dtype=float
        )
    )


# =============================================================================
# 14. FOCUSED BETA FDR6
# =============================================================================
#
# This is reported IN ADDITION TO FDR18, not instead of it.
# =============================================================================

results_df[
    "p_bh_beta_fdr6"
] = np.nan


for baseline_name in BASELINES:

    mask = (
        (
            results_df[
                "baseline"
            ]
            ==
            baseline_name
        )
        &
        (
            results_df[
                "band"
            ]
            ==
            "Beta"
        )
    )


    if mask.sum() != 6:

        raise RuntimeError(
            f"{baseline_name}: expected 6 beta-network tests; "
            f"found {mask.sum()}."
        )


    results_df.loc[
        mask,
        "p_bh_beta_fdr6"
    ] = bh_fdr(
        results_df.loc[
            mask,
            "exact_permutation_p"
        ].to_numpy(
            dtype=float
        )
    )


results_df[
    "raw_p_lt_0_05"
] = (
    results_df[
        "exact_permutation_p"
    ]
    <
    0.05
)


results_df[
    "fdr18_lt_0_05"
] = (
    results_df[
        "p_bh_fdr18"
    ]
    <
    0.05
)


results_df[
    "beta_fdr6_lt_0_05"
] = (
    results_df[
        "p_bh_beta_fdr6"
    ]
    <
    0.05
)


# =============================================================================
# 15. PRINT FULL RANKED RESULTS
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "FULL ROI-NETWORK RESULTS — RANKED WITHIN BASELINE"
)

print(
    "=" * 100
)


for baseline_name in BASELINES:

    print(
        f"\n{baseline_name}"
    )

    ranked = (
        results_df.loc[
            results_df[
                "baseline"
            ]
            ==
            baseline_name
        ]
        .sort_values(
            [
                "exact_permutation_p",
                "band",
                "network"
            ]
        )
    )


    display(
        ranked[
            [
                "band",
                "network",
                "n_edges",
                "active_mean_delta",
                "passive_mean_delta",
                "active_minus_passive",
                "bootstrap_ci_low",
                "bootstrap_ci_high",
                "hedges_g",
                "exact_permutation_p",
                "p_bh_fdr18"
            ]
        ].round(
            6
        )
    )


# =============================================================================
# 16. FOCUSED BETA LOCALIZATION
# =============================================================================

beta_df = (
    results_df.loc[
        results_df[
            "band"
        ]
        ==
        "Beta"
    ]
    .copy()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "FOCUSED BETA NETWORK LOCALIZATION"
)

print(
    "=" * 100
)


display(
    beta_df[
        [
            "baseline",
            "network",
            "n_edges",
            "active_mean_delta",
            "passive_mean_delta",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_beta_fdr6",
            "p_bh_fdr18"
        ]
    ]
    .sort_values(
        [
            "baseline",
            "exact_permutation_p"
        ]
    )
    .round(
        6
    )
)


# =============================================================================
# 17. CROSS-BASELINE NETWORK STABILITY
# =============================================================================

stability_rows = []


for band in BANDS:

    for network in NETWORK_ORDER:

        row_a = results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                band
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            )
        ].iloc[
            0
        ]


        row_b = results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                band
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            )
        ].iloc[
            0
        ]


        diff_a = float(
            row_a[
                "active_minus_passive"
            ]
        )


        diff_b = float(
            row_b[
                "active_minus_passive"
            ]
        )


        g_a = float(
            row_a[
                "hedges_g"
            ]
        )


        g_b = float(
            row_b[
                "hedges_g"
            ]
        )


        stability_rows.append({
            "band":
                band,

            "network":
                network,

            "baseline_a_difference":
                diff_a,

            "baseline_b_difference":
                diff_b,

            "same_direction":
                bool(
                    np.sign(
                        diff_a
                    )
                    ==
                    np.sign(
                        diff_b
                    )
                ),

            "absolute_difference_between_baselines":
                float(
                    abs(
                        diff_b
                        -
                        diff_a
                    )
                ),

            "baseline_a_hedges_g":
                g_a,

            "baseline_b_hedges_g":
                g_b,

            "g_change_b_minus_a":
                float(
                    g_b
                    -
                    g_a
                ),

            "baseline_a_p":
                float(
                    row_a[
                        "exact_permutation_p"
                    ]
                ),

            "baseline_b_p":
                float(
                    row_b[
                        "exact_permutation_p"
                    ]
                )
        })


stability_df = pd.DataFrame(
    stability_rows
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "CROSS-BASELINE NETWORK STABILITY"
)

print(
    "=" * 100
)


display(
    stability_df
    .sort_values(
        [
            "band",
            "network"
        ]
    )
    .round(
        6
    )
)


# =============================================================================
# 18. BETA CROSS-BASELINE STABILITY
# =============================================================================

beta_stability_df = (
    stability_df.loc[
        stability_df[
            "band"
        ]
        ==
        "Beta"
    ]
    .copy()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "BETA NETWORKS — CROSS-BASELINE STABILITY"
)

print(
    "=" * 100
)


display(
    beta_stability_df[
        [
            "network",
            "baseline_a_difference",
            "baseline_b_difference",
            "same_direction",
            "baseline_a_hedges_g",
            "baseline_b_hedges_g",
            "baseline_a_p",
            "baseline_b_p"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 19. LEAVE-ONE-OUT ROBUSTNESS FOR BETA NETWORKS
# =============================================================================
#
# We focus the LOO analysis on beta because beta was the only global band
# preserving direction across both baselines in Task 6C.
# =============================================================================

loo_rows = []


for baseline_name, delta_column in BASELINES.items():

    for network in NETWORK_ORDER:

        temp = (
            network_df.loc[
                (
                    network_df[
                        "band"
                    ]
                    ==
                    "Beta"
                )
                &
                (
                    network_df[
                        "network"
                    ]
                    ==
                    network
                )
            ]
            .sort_values(
                "subject"
            )
            .reset_index(
                drop=True
            )
        )


        primary_row = results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                baseline_name
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            )
        ].iloc[
            0
        ]


        primary_difference = float(
            primary_row[
                "active_minus_passive"
            ]
        )


        for removed_subject in temp[
            "subject"
        ]:

            reduced = temp.loc[
                temp[
                    "subject"
                ]
                !=
                removed_subject
            ]


            active = reduced.loc[
                reduced[
                    "group"
                ]
                ==
                "Active",
                delta_column
            ].to_numpy(
                dtype=float
            )


            passive = reduced.loc[
                reduced[
                    "group"
                ]
                ==
                "Passive",
                delta_column
            ].to_numpy(
                dtype=float
            )


            difference = float(
                np.mean(
                    active
                )
                -
                np.mean(
                    passive
                )
            )


            loo_rows.append({
                "baseline":
                    baseline_name,

                "network":
                    network,

                "removed_subject":
                    removed_subject,

                "active_minus_passive":
                    difference,

                "hedges_g":
                    hedges_g(
                        active,
                        passive
                    ),

                "same_direction_as_full":
                    bool(
                        np.sign(
                            difference
                        )
                        ==
                        np.sign(
                            primary_difference
                        )
                    )
            })


loo_df = pd.DataFrame(
    loo_rows
)


loo_summary_df = (
    loo_df
    .groupby(
        [
            "baseline",
            "network"
        ]
    )
    .agg(
        same_direction_n=(
            "same_direction_as_full",
            "sum"
        ),

        total=(
            "same_direction_as_full",
            "size"
        ),

        difference_min=(
            "active_minus_passive",
            "min"
        ),

        difference_max=(
            "active_minus_passive",
            "max"
        ),

        g_min=(
            "hedges_g",
            "min"
        ),

        g_max=(
            "hedges_g",
            "max"
        )
    )
    .reset_index()
)


loo_summary_df[
    "same_direction_percent"
] = (
    100
    *
    loo_summary_df[
        "same_direction_n"
    ]
    /
    loo_summary_df[
        "total"
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "BETA NETWORK LEAVE-ONE-OUT ROBUSTNESS"
)

print(
    "=" * 100
)


display(
    loo_summary_df.round(
        6
    )
)


# =============================================================================
# 20. FIGURE — BETA NETWORK EFFECT SIZES ACROSS BASELINES
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        6
    )
)


x = np.arange(
    len(
        NETWORK_ORDER
    )
)


width = 0.36


g_a = []

g_b = []


for network in NETWORK_ORDER:

    g_a.append(
        results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


    g_b.append(
        results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


ax.bar(
    x - width / 2,
    g_a,
    width,
    label="Baseline A"
)


ax.bar(
    x + width / 2,
    g_b,
    width,
    label="Baseline B"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        "F-F",
        "F-CT",
        "F-P",
        "CT-CT",
        "CT-P",
        "P-P"
    ]
)


ax.set_xlabel(
    "ROI network"
)


ax.set_ylabel(
    "Hedges' g (Active − Passive)"
)


ax.set_title(
    "Beta-band ROI connectivity effects across marker-defined baselines"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


beta_effect_figure = (
    FIG_OUT
    /
    "beta_network_effect_sizes_across_baselines.png"
)


fig.savefig(
    beta_effect_figure,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 21. FIGURE — BETA ACTIVE-PASSIVE DIFFERENCE ACROSS BASELINES
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        6
    )
)


difference_a = []

difference_b = []


for network in NETWORK_ORDER:

    difference_a.append(
        results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            ),
            "active_minus_passive"
        ].iloc[
            0
        ]
    )


    difference_b.append(
        results_df.loc[
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "network"
                ]
                ==
                network
            ),
            "active_minus_passive"
        ].iloc[
            0
        ]
    )


ax.plot(
    x,
    difference_a,
    marker="o",
    label="Baseline A"
)


ax.plot(
    x,
    difference_b,
    marker="s",
    label="Baseline B"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        "F-F",
        "F-CT",
        "F-P",
        "CT-CT",
        "CT-P",
        "P-P"
    ]
)


ax.set_xlabel(
    "ROI network"
)


ax.set_ylabel(
    "Active − Passive beta connectivity change"
)


ax.set_title(
    "Cross-baseline stability of beta ROI connectivity"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


beta_difference_figure = (
    FIG_OUT
    /
    "beta_network_cross_baseline_stability.png"
)


fig.savefig(
    beta_difference_figure,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 22. SAVE OUTPUTS
# =============================================================================

network_df.to_csv(
    OUT
    /
    "participant_network_connectivity.csv",
    index=False
)


descriptive_df.to_csv(
    OUT
    /
    "network_group_descriptive_summary.csv",
    index=False
)


results_df.to_csv(
    OUT
    /
    "roi_network_exact_inference.csv",
    index=False
)


beta_df.to_csv(
    OUT
    /
    "beta_network_focused_inference.csv",
    index=False
)


stability_df.to_csv(
    OUT
    /
    "cross_baseline_network_stability.csv",
    index=False
)


beta_stability_df.to_csv(
    OUT
    /
    "beta_cross_baseline_network_stability.csv",
    index=False
)


loo_df.to_csv(
    OUT
    /
    "beta_network_leave_one_out.csv",
    index=False
)


loo_summary_df.to_csv(
    OUT
    /
    "beta_network_leave_one_out_summary.csv",
    index=False
)


# =============================================================================
# 23. FINAL INTEGRITY CHECK
# =============================================================================

integrity = {
    "participants":
        network_df[
            "subject"
        ].nunique()
        ==
        15,

    "active":
        network_df.loc[
            network_df[
                "group"
            ]
            ==
            "Active",
            "subject"
        ].nunique()
        ==
        8,

    "passive":
        network_df.loc[
            network_df[
                "group"
            ]
            ==
            "Passive",
            "subject"
        ].nunique()
        ==
        7,

    "bands":
        set(
            network_df[
                "band"
            ].unique()
        )
        ==
        set(
            BANDS
        ),

    "networks":
        set(
            network_df[
                "network"
            ].unique()
        )
        ==
        set(
            NETWORK_ORDER
        ),

    "network_rows":
        len(
            network_df
        )
        ==
        EXPECTED_NETWORK_ROWS,

    "full_tests":
        len(
            results_df
        )
        ==
        36,

    "tests_per_baseline":
        all(
            (
                results_df[
                    "baseline"
                ]
                ==
                baseline
            ).sum()
            ==
            18
            for baseline in BASELINES
        ),

    "beta_tests_per_baseline":
        all(
            (
                (
                    results_df[
                        "baseline"
                    ]
                    ==
                    baseline
                )
                &
                (
                    results_df[
                        "band"
                    ]
                    ==
                    "Beta"
                )
            ).sum()
            ==
            6
            for baseline in BASELINES
        ),

    "exact_permutations":
        (
            results_df[
                "n_exact_permutations"
            ]
            ==
            EXPECTED_PERMUTATIONS
        ).all(),

    "fdr18_complete":
        results_df[
            "p_bh_fdr18"
        ].notna().all(),

    "beta_fdr6_complete":
        results_df.loc[
            results_df[
                "band"
            ]
            ==
            "Beta",
            "p_bh_beta_fdr6"
        ].notna().all(),

    "edge_counts":
        observed_network_counts
        ==
        EXPECTED_NETWORK_EDGE_COUNTS,

    "all_values_finite":
        np.isfinite(
            network_df[
                [
                    "baseline_a",
                    "baseline_b",
                    "task",
                    "delta_a",
                    "delta_b"
                ]
            ].to_numpy(
                dtype=float
            )
        ).all()
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "TASK 7A COMPLETE"
)

print(
    "=" * 100
)


print(
    f"Participant-network rows   : "
    f"{len(network_df)}/{EXPECTED_NETWORK_ROWS}"
)

print(
    f"Full inferential tests     : "
    f"{len(results_df)}/36"
)

print(
    "Tests/baseline             : 18"
)

print(
    "Focused beta tests/baseline: 6"
)

print(
    f"Exact allocations/test     : "
    f"{EXPECTED_PERMUTATIONS}"
)

print(
    f"Bootstrap iterations/test  : "
    f"{N_BOOTSTRAP:,}"
)

print(
    "Primary multiplicity       : FDR18 separately by baseline"
)

print(
    "Focused beta multiplicity  : FDR6 separately by baseline"
)


print(
    "\nIntegrity components:"
)


for key, value in integrity.items():

    print(
        f"  {key:<27}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 100
)

print(
    "OVERALL INTEGRITY CHECK:",
    (
        "PASS"
        if overall_pass
        else "FAIL"
    )
)

print(
    "-" * 100
)


if not overall_pass:

    raise RuntimeError(
        "Task 7A integrity check failed."
    )


# =============================================================================
# 24. AUTOMATIC CONSERVATIVE SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "FOCUSED BETA INTERPRETATION"
)

print(
    "=" * 100
)


for network in NETWORK_ORDER:

    row_a = beta_df.loc[
        (
            beta_df[
                "baseline"
            ]
            ==
            "Baseline A"
        )
        &
        (
            beta_df[
                "network"
            ]
            ==
            network
        )
    ].iloc[
        0
    ]


    row_b = beta_df.loc[
        (
            beta_df[
                "baseline"
            ]
            ==
            "Baseline B"
        )
        &
        (
            beta_df[
                "network"
            ]
            ==
            network
        )
    ].iloc[
        0
    ]


    same_direction = (
        np.sign(
            row_a[
                "active_minus_passive"
            ]
        )
        ==
        np.sign(
            row_b[
                "active_minus_passive"
            ]
        )
    )


    print(
        f"\n{network}:"
    )


    print(
        f"  Baseline A difference : "
        f"{row_a['active_minus_passive']:+.6f}"
    )

    print(
        f"  Baseline A g          : "
        f"{row_a['hedges_g']:+.3f}"
    )

    print(
        f"  Baseline A p          : "
        f"{row_a['exact_permutation_p']:.4f}"
    )

    print(
        f"  Baseline A beta-FDR6  : "
        f"{row_a['p_bh_beta_fdr6']:.4f}"
    )


    print(
        f"  Baseline B difference : "
        f"{row_b['active_minus_passive']:+.6f}"
    )

    print(
        f"  Baseline B g          : "
        f"{row_b['hedges_g']:+.3f}"
    )

    print(
        f"  Baseline B p          : "
        f"{row_b['exact_permutation_p']:.4f}"
    )

    print(
        f"  Baseline B beta-FDR6  : "
        f"{row_b['p_bh_beta_fdr6']:.4f}"
    )


    print(
        f"  Same direction        : "
        f"{same_direction}"
    )


print(
    "\nINTERPRETATION RULE:"
)

print(
    "Network localization is considered strongest when the "
    "Active-Passive direction is preserved across both baselines, "
    "the effect magnitude is non-trivial, and leave-one-out analyses "
    "retain the same direction."
)

print(
    "Raw p < .05 without multiplicity correction is not treated as "
    "confirmatory evidence."
)

print(
    "The beta FDR6 family is a focused follow-up motivated by the "
    "baseline-directional robustness established at the global level "
    "in Task 6C."
)

print(
    "No individual edge is declared significant."
)


print(
    f"\nOutput directory:\n{OUT}"
)

print(
    "\n"
    +
    "=" * 100
)

In [ ]:
# =============================================================================
# TASK 7B — BASELINE-ROBUST WEIGHTED NETWORK TOPOLOGY
# ABSOLUTE vs RELATIVE NODE STRENGTH
# VALIDATED wPLI²_debiased | MATCHED FIXED-9 | BASELINES A + B
# =============================================================================
#
# PURPOSE
# -------
# Determine whether Active-vs-Passive connectivity differences reflect:
#
#   1. ABSOLUTE connectivity magnitude
#      -> weighted node strength
#
#   2. RELATIVE redistribution of network connectivity
#      -> node strength / total absolute node strength
#
# IMPORTANT DESIGN RULES
# ----------------------
# - Uses Task 6C validated wPLI²_debiased edge estimates.
# - Does NOT recompute connectivity.
# - Does NOT threshold the network.
# - Does NOT clip negative wPLI²_debiased values.
# - No electrode-level significance testing.
# - Participant is the inferential unit.
# - ROI inference only:
#       Frontal
#       CentralTemporal
#       Posterior
# - All Theta/Alpha/Beta retained.
# - Beta additionally treated as focused follow-up.
# - Both Baseline A and Baseline B evaluated.
#
# ABSOLUTE NODE STRENGTH
# ----------------------
# For node i:
#
#       s_i = sum_j w_ij
#
# Change:
#
#       Δs_i = s_i(task) - s_i(baseline)
#
# RELATIVE NODE STRENGTH
# ----------------------
# Because wPLI²_debiased can legitimately be negative, normalization uses
# the sum of ABSOLUTE node strengths as the denominator:
#
#       r_i = s_i / sum_j |s_j|
#
# This preserves the sign of the node's net strength while avoiding an
# unstable near-zero signed denominator.
#
# Change:
#
#       Δr_i = r_i(task) - r_i(baseline)
#
# INFERENCE
# ---------
# Exact permutation:
#       C(15,8) = 6435 allocations
#
# Bootstrap:
#       10,000 stratified resamples
#
# Multiplicity:
#       FDR9 separately for:
#           Absolute × Baseline A
#           Absolute × Baseline B
#           Relative × Baseline A
#           Relative × Baseline B
#
#       Each family:
#           3 bands × 3 ROIs = 9 tests
#
# Focused beta:
#       FDR3 separately within each topology/baseline family.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK6C = (
    BASE
    / "task06c_matched_fixed9_baseline_sensitivity"
)

EDGE_FILE = (
    TASK6C
    / "edge_level_connectivity.csv"
)

OUT = (
    BASE
    / "task07b_baseline_robust_weighted_topology"
)

FIG_OUT = (
    OUT
    / "figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)


if not EDGE_FILE.exists():
    raise FileNotFoundError(
        f"Task 6C edge file not found:\n{EDGE_FILE}"
    )


# =============================================================================
# 2. LOAD VALIDATED CONNECTIVITY
# =============================================================================

edge_df = pd.read_csv(
    EDGE_FILE
)


# =============================================================================
# 3. CONSTANTS
# =============================================================================

CHANNELS = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


ROIS = {

    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "CentralTemporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}


ROI_ORDER = [
    "Frontal",
    "CentralTemporal",
    "Posterior"
]


BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]


CONDITIONS = {
    "Baseline A": "baseline_a",
    "Baseline B": "baseline_b"
}


EXPECTED_SUBJECTS = 15

EXPECTED_ACTIVE = 8

EXPECTED_PASSIVE = 7

EXPECTED_EDGES = 91

EXPECTED_PERMUTATIONS = math.comb(
    15,
    8
)

N_BOOTSTRAP = 10_000

RANDOM_SEED = 20260913


channel_to_roi = {}

for roi, channels in ROIS.items():

    for channel in channels:

        channel_to_roi[
            channel
        ] = roi


# =============================================================================
# 4. INPUT INTEGRITY
# =============================================================================

required_columns = {
    "subject",
    "group",
    "band",
    "edge_index",
    "channel_i",
    "channel_j",
    "baseline_a",
    "baseline_b",
    "task"
}


missing = (
    required_columns
    -
    set(
        edge_df.columns
    )
)


if missing:

    raise RuntimeError(
        "Missing Task 6C columns: "
        f"{sorted(missing)}"
    )


if edge_df[
    "subject"
].nunique() != EXPECTED_SUBJECTS:

    raise RuntimeError(
        "Expected 15 participants."
    )


groups = (
    edge_df[
        [
            "subject",
            "group"
        ]
    ]
    .drop_duplicates()
)


if (
    groups.loc[
        groups[
            "group"
        ]
        ==
        "Active"
    ].shape[
        0
    ]
    !=
    EXPECTED_ACTIVE
):

    raise RuntimeError(
        "Expected 8 Active participants."
    )


if (
    groups.loc[
        groups[
            "group"
        ]
        ==
        "Passive"
    ].shape[
        0
    ]
    !=
    EXPECTED_PASSIVE
):

    raise RuntimeError(
        "Expected 7 Passive participants."
    )


edge_counts = (
    edge_df
    .groupby(
        [
            "subject",
            "band"
        ]
    )[
        "edge_index"
    ]
    .nunique()
)


if not (
    edge_counts
    ==
    EXPECTED_EDGES
).all():

    raise RuntimeError(
        "Every participant-band must contain exactly 91 edges."
    )


observed_channels = set(
    edge_df[
        "channel_i"
    ]
).union(
    edge_df[
        "channel_j"
    ]
)


if observed_channels != set(
    CHANNELS
):

    raise RuntimeError(
        "\nChannel mismatch.\n"
        f"Observed: {sorted(observed_channels)}\n"
        f"Expected: {sorted(CHANNELS)}"
    )


print(
    "=" * 100
)

print(
    "TASK 7B — BASELINE-ROBUST WEIGHTED NETWORK TOPOLOGY"
)

print(
    "=" * 100
)

print(
    f"Participants              : {EXPECTED_SUBJECTS}"
)

print(
    f"Active                    : {EXPECTED_ACTIVE}"
)

print(
    f"Passive                   : {EXPECTED_PASSIVE}"
)

print(
    f"Channels                  : {len(CHANNELS)}"
)

print(
    f"Unique edges              : {EXPECTED_EDGES}"
)

print(
    f"Bands                     : {', '.join(BANDS)}"
)

print(
    "Baselines                 : Baseline A + Baseline B"
)

print(
    "Estimator                 : validated wPLI²_debiased"
)

print(
    "Epoch design              : matched fixed-9"
)

print(
    "Graph thresholding        : NONE"
)

print(
    "Negative estimates        : RETAINED"
)

print(
    f"Exact allocations/test    : {EXPECTED_PERMUTATIONS}"
)


# =============================================================================
# 5. BUILD MATRICES
# =============================================================================

channel_index = {
    ch: i
    for i, ch in enumerate(
        CHANNELS
    )
}


def build_matrix(
    temp,
    value_column
):

    n = len(
        CHANNELS
    )

    matrix = np.zeros(
        (
            n,
            n
        ),
        dtype=float
    )


    for row in temp.itertuples(
        index=False
    ):

        i = channel_index[
            row.channel_i
        ]

        j = channel_index[
            row.channel_j
        ]

        value = float(
            getattr(
                row,
                value_column
            )
        )

        matrix[
            i,
            j
        ] = value

        matrix[
            j,
            i
        ] = value


    if not np.allclose(
        matrix,
        matrix.T,
        atol=1e-12
    ):

        raise RuntimeError(
            "Matrix symmetry failure."
        )


    if not np.allclose(
        np.diag(
            matrix
        ),
        0,
        atol=1e-12
    ):

        raise RuntimeError(
            "Matrix diagonal failure."
        )


    if not np.isfinite(
        matrix
    ).all():

        raise RuntimeError(
            "Non-finite matrix values."
        )


    return matrix


# =============================================================================
# 6. NODE-STRENGTH FUNCTIONS
# =============================================================================

def absolute_node_strength(
    matrix
):

    return np.sum(
        matrix,
        axis=1
    )


def relative_node_strength(
    matrix
):

    strength = absolute_node_strength(
        matrix
    )

    denominator = np.sum(
        np.abs(
            strength
        )
    )


    if (
        not np.isfinite(
            denominator
        )
        or
        denominator <= 1e-15
    ):

        raise RuntimeError(
            "Invalid relative-strength denominator."
        )


    return (
        strength
        /
        denominator
    )


# =============================================================================
# 7. CONSTRUCT PARTICIPANT × NODE TOPOLOGY
# =============================================================================

node_rows = []


for (
    subject,
    group,
    band
), temp in edge_df.groupby(
    [
        "subject",
        "group",
        "band"
    ]
):

    temp = temp.copy()


    if temp[
        "edge_index"
    ].nunique() != 91:

        raise RuntimeError(
            f"{subject}/{band}: expected 91 edges."
        )


    matrices = {

        "baseline_a":
            build_matrix(
                temp,
                "baseline_a"
            ),

        "baseline_b":
            build_matrix(
                temp,
                "baseline_b"
            ),

        "task":
            build_matrix(
                temp,
                "task"
            )
    }


    abs_strength = {
        condition:
            absolute_node_strength(
                matrix
            )
        for condition, matrix in matrices.items()
    }


    rel_strength = {
        condition:
            relative_node_strength(
                matrix
            )
        for condition, matrix in matrices.items()
    }


    for node_index, channel in enumerate(
        CHANNELS
    ):

        node_rows.append({

            "subject":
                subject,

            "group":
                group,

            "band":
                band,

            "channel":
                channel,

            "roi":
                channel_to_roi[
                    channel
                ],

            "absolute_baseline_a":
                float(
                    abs_strength[
                        "baseline_a"
                    ][
                        node_index
                    ]
                ),

            "absolute_baseline_b":
                float(
                    abs_strength[
                        "baseline_b"
                    ][
                        node_index
                    ]
                ),

            "absolute_task":
                float(
                    abs_strength[
                        "task"
                    ][
                        node_index
                    ]
                ),

            "relative_baseline_a":
                float(
                    rel_strength[
                        "baseline_a"
                    ][
                        node_index
                    ]
                ),

            "relative_baseline_b":
                float(
                    rel_strength[
                        "baseline_b"
                    ][
                        node_index
                    ]
                ),

            "relative_task":
                float(
                    rel_strength[
                        "task"
                    ][
                        node_index
                    ]
                )
        })


node_df = pd.DataFrame(
    node_rows
)


# =============================================================================
# 8. CALCULATE CHANGE SCORES
# =============================================================================

node_df[
    "absolute_delta_a"
] = (
    node_df[
        "absolute_task"
    ]
    -
    node_df[
        "absolute_baseline_a"
    ]
)


node_df[
    "absolute_delta_b"
] = (
    node_df[
        "absolute_task"
    ]
    -
    node_df[
        "absolute_baseline_b"
    ]
)


node_df[
    "relative_delta_a"
] = (
    node_df[
        "relative_task"
    ]
    -
    node_df[
        "relative_baseline_a"
    ]
)


node_df[
    "relative_delta_b"
] = (
    node_df[
        "relative_task"
    ]
    -
    node_df[
        "relative_baseline_b"
    ]
)


expected_node_rows = (
    15
    *
    3
    *
    14
)


if len(
    node_df
) != expected_node_rows:

    raise RuntimeError(
        f"Expected {expected_node_rows} participant-node rows; "
        f"found {len(node_df)}."
    )


# =============================================================================
# 9. ROI AGGREGATION
# =============================================================================
#
# Mean node strength within each predefined ROI.
#
# Participant remains the inferential unit.
# =============================================================================

roi_df = (
    node_df
    .groupby(
        [
            "subject",
            "group",
            "band",
            "roi"
        ],
        as_index=False
    )
    .agg(

        n_nodes=(
            "channel",
            "count"
        ),

        absolute_baseline_a=(
            "absolute_baseline_a",
            "mean"
        ),

        absolute_baseline_b=(
            "absolute_baseline_b",
            "mean"
        ),

        absolute_task=(
            "absolute_task",
            "mean"
        ),

        absolute_delta_a=(
            "absolute_delta_a",
            "mean"
        ),

        absolute_delta_b=(
            "absolute_delta_b",
            "mean"
        ),

        relative_baseline_a=(
            "relative_baseline_a",
            "mean"
        ),

        relative_baseline_b=(
            "relative_baseline_b",
            "mean"
        ),

        relative_task=(
            "relative_task",
            "mean"
        ),

        relative_delta_a=(
            "relative_delta_a",
            "mean"
        ),

        relative_delta_b=(
            "relative_delta_b",
            "mean"
        )
    )
)


expected_roi_rows = (
    15
    *
    3
    *
    3
)


if len(
    roi_df
) != expected_roi_rows:

    raise RuntimeError(
        f"Expected {expected_roi_rows} participant-ROI rows; "
        f"found {len(roi_df)}."
    )


expected_roi_sizes = {
    "Frontal": 6,
    "CentralTemporal": 4,
    "Posterior": 4
}


for roi, expected in expected_roi_sizes.items():

    observed = roi_df.loc[
        roi_df[
            "roi"
        ]
        ==
        roi,
        "n_nodes"
    ]

    if not (
        observed
        ==
        expected
    ).all():

        raise RuntimeError(
            f"{roi}: incorrect node count."
        )


# =============================================================================
# 10. STATISTICAL FUNCTIONS
# =============================================================================

def hedges_g(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    n1 = len(
        active
    )

    n2 = len(
        passive
    )


    pooled_variance = (

        (
            (n1 - 1)
            *
            np.var(
                active,
                ddof=1
            )
        )

        +

        (
            (n2 - 1)
            *
            np.var(
                passive,
                ddof=1
            )
        )

    ) / (
        n1
        +
        n2
        -
        2
    )


    if (
        not np.isfinite(
            pooled_variance
        )
        or
        pooled_variance <= 0
    ):

        return np.nan


    d = (
        np.mean(
            active
        )
        -
        np.mean(
            passive
        )
    ) / np.sqrt(
        pooled_variance
    )


    df = (
        n1
        +
        n2
        -
        2
    )


    correction = (
        1
        -
        3
        /
        (
            4
            *
            df
            -
            1
        )
    )


    return float(
        correction
        *
        d
    )


def exact_permutation_test(
    active,
    passive
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    pooled = np.concatenate(
        [
            active,
            passive
        ]
    )


    n_active = len(
        active
    )


    observed = float(
        np.mean(
            active
        )
        -
        np.mean(
            passive
        )
    )


    n_total = len(
        pooled
    )


    all_indices = np.arange(
        n_total
    )


    null = np.empty(
        EXPECTED_PERMUTATIONS,
        dtype=float
    )


    counter = 0


    for active_tuple in combinations(
        range(
            n_total
        ),
        n_active
    ):

        active_idx = np.asarray(
            active_tuple,
            dtype=int
        )


        passive_mask = np.ones(
            n_total,
            dtype=bool
        )

        passive_mask[
            active_idx
        ] = False


        passive_idx = all_indices[
            passive_mask
        ]


        null[
            counter
        ] = (
            np.mean(
                pooled[
                    active_idx
                ]
            )
            -
            np.mean(
                pooled[
                    passive_idx
                ]
            )
        )


        counter += 1


    if counter != EXPECTED_PERMUTATIONS:

        raise RuntimeError(
            "Exact permutation count mismatch."
        )


    p = float(
        np.mean(
            np.abs(
                null
            )
            >=
            (
                abs(
                    observed
                )
                -
                1e-15
            )
        )
    )


    return (
        observed,
        p,
        counter
    )


def bootstrap_ci(
    active,
    passive,
    seed
):

    active = np.asarray(
        active,
        dtype=float
    )

    passive = np.asarray(
        passive,
        dtype=float
    )


    rng = np.random.default_rng(
        seed
    )


    boot = np.empty(
        N_BOOTSTRAP,
        dtype=float
    )


    for i in range(
        N_BOOTSTRAP
    ):

        a = rng.choice(
            active,
            size=len(
                active
            ),
            replace=True
        )

        p = rng.choice(
            passive,
            size=len(
                passive
            ),
            replace=True
        )


        boot[
            i
        ] = (
            np.mean(
                a
            )
            -
            np.mean(
                p
            )
        )


    low, high = np.percentile(
        boot,
        [
            2.5,
            97.5
        ]
    )


    return (
        float(
            low
        ),
        float(
            high
        )
    )


def bh_fdr(
    p_values
):

    p_values = np.asarray(
        p_values,
        dtype=float
    )


    m = len(
        p_values
    )


    order = np.argsort(
        p_values
    )


    ranked = p_values[
        order
    ]


    adjusted = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )


    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]


    adjusted = np.clip(
        adjusted,
        0,
        1
    )


    result = np.empty(
        m,
        dtype=float
    )


    result[
        order
    ] = adjusted


    return result


# =============================================================================
# 11. RUN ROI-LEVEL TOPOLOGY INFERENCE
# =============================================================================

TOPOLOGY_DEFINITIONS = {

    "Absolute": {
        "Baseline A":
            "absolute_delta_a",

        "Baseline B":
            "absolute_delta_b"
    },

    "Relative": {
        "Baseline A":
            "relative_delta_a",

        "Baseline B":
            "relative_delta_b"
    }
}


result_rows = []

seed_counter = 0


for topology, baseline_dict in TOPOLOGY_DEFINITIONS.items():

    for baseline, value_column in baseline_dict.items():

        for band in BANDS:

            for roi in ROI_ORDER:

                temp = roi_df.loc[
                    (
                        roi_df[
                            "band"
                        ]
                        ==
                        band
                    )
                    &
                    (
                        roi_df[
                            "roi"
                        ]
                        ==
                        roi
                    )
                ]


                active = temp.loc[
                    temp[
                        "group"
                    ]
                    ==
                    "Active",
                    value_column
                ].to_numpy(
                    dtype=float
                )


                passive = temp.loc[
                    temp[
                        "group"
                    ]
                    ==
                    "Passive",
                    value_column
                ].to_numpy(
                    dtype=float
                )


                if len(
                    active
                ) != 8:

                    raise RuntimeError(
                        f"{topology}/{baseline}/{band}/{roi}: "
                        "Active n != 8."
                    )


                if len(
                    passive
                ) != 7:

                    raise RuntimeError(
                        f"{topology}/{baseline}/{band}/{roi}: "
                        "Passive n != 7."
                    )


                difference, p, n_perm = (
                    exact_permutation_test(
                        active,
                        passive
                    )
                )


                ci_low, ci_high = bootstrap_ci(
                    active,
                    passive,
                    seed=(
                        RANDOM_SEED
                        +
                        seed_counter
                    )
                )


                result_rows.append({

                    "topology":
                        topology,

                    "baseline":
                        baseline,

                    "band":
                        band,

                    "roi":
                        roi,

                    "n_active":
                        len(
                            active
                        ),

                    "n_passive":
                        len(
                            passive
                        ),

                    "active_mean_delta":
                        float(
                            np.mean(
                                active
                            )
                        ),

                    "active_sd_delta":
                        float(
                            np.std(
                                active,
                                ddof=1
                            )
                        ),

                    "passive_mean_delta":
                        float(
                            np.mean(
                                passive
                            )
                        ),

                    "passive_sd_delta":
                        float(
                            np.std(
                                passive,
                                ddof=1
                            )
                        ),

                    "active_minus_passive":
                        difference,

                    "bootstrap_ci_low":
                        ci_low,

                    "bootstrap_ci_high":
                        ci_high,

                    "hedges_g":
                        hedges_g(
                            active,
                            passive
                        ),

                    "exact_permutation_p":
                        p,

                    "n_exact_permutations":
                        n_perm
                })


                seed_counter += 1


results_df = pd.DataFrame(
    result_rows
)


if len(
    results_df
) != 36:

    raise RuntimeError(
        f"Expected 36 topology tests; found {len(results_df)}."
    )


# =============================================================================
# 12. FDR9 — SEPARATELY WITHIN TOPOLOGY × BASELINE
# =============================================================================

results_df[
    "p_bh_fdr9"
] = np.nan


for topology in [
    "Absolute",
    "Relative"
]:

    for baseline in [
        "Baseline A",
        "Baseline B"
    ]:

        mask = (
            (
                results_df[
                    "topology"
                ]
                ==
                topology
            )
            &
            (
                results_df[
                    "baseline"
                ]
                ==
                baseline
            )
        )


        if mask.sum() != 9:

            raise RuntimeError(
                f"{topology}/{baseline}: "
                f"expected 9 tests, found {mask.sum()}."
            )


        results_df.loc[
            mask,
            "p_bh_fdr9"
        ] = bh_fdr(
            results_df.loc[
                mask,
                "exact_permutation_p"
            ].to_numpy(
                dtype=float
            )
        )


# =============================================================================
# 13. FOCUSED BETA FDR3
# =============================================================================

results_df[
    "p_bh_beta_fdr3"
] = np.nan


for topology in [
    "Absolute",
    "Relative"
]:

    for baseline in [
        "Baseline A",
        "Baseline B"
    ]:

        mask = (
            (
                results_df[
                    "topology"
                ]
                ==
                topology
            )
            &
            (
                results_df[
                    "baseline"
                ]
                ==
                baseline
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
        )


        if mask.sum() != 3:

            raise RuntimeError(
                f"{topology}/{baseline}: expected 3 beta tests."
            )


        results_df.loc[
            mask,
            "p_bh_beta_fdr3"
        ] = bh_fdr(
            results_df.loc[
                mask,
                "exact_permutation_p"
            ].to_numpy(
                dtype=float
            )
        )


# =============================================================================
# 14. SIGNIFICANCE FLAGS
# =============================================================================

results_df[
    "raw_p_lt_0_05"
] = (
    results_df[
        "exact_permutation_p"
    ]
    <
    0.05
)


results_df[
    "fdr9_lt_0_05"
] = (
    results_df[
        "p_bh_fdr9"
    ]
    <
    0.05
)


results_df[
    "beta_fdr3_lt_0_05"
] = (
    results_df[
        "p_bh_beta_fdr3"
    ]
    <
    0.05
)


# =============================================================================
# 15. PRINT FULL RESULTS
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "FULL ROI TOPOLOGY RESULTS"
)

print(
    "=" * 100
)


for topology in [
    "Absolute",
    "Relative"
]:

    for baseline in [
        "Baseline A",
        "Baseline B"
    ]:

        print(
            f"\n{topology} node strength — {baseline}"
        )


        temp = (
            results_df.loc[
                (
                    results_df[
                        "topology"
                    ]
                    ==
                    topology
                )
                &
                (
                    results_df[
                        "baseline"
                    ]
                    ==
                    baseline
                )
            ]
            .sort_values(
                "exact_permutation_p"
            )
        )


        display(
            temp[
                [
                    "band",
                    "roi",
                    "active_mean_delta",
                    "passive_mean_delta",
                    "active_minus_passive",
                    "bootstrap_ci_low",
                    "bootstrap_ci_high",
                    "hedges_g",
                    "exact_permutation_p",
                    "p_bh_fdr9"
                ]
            ].round(
                6
            )
        )


# =============================================================================
# 16. FOCUSED BETA RESULTS
# =============================================================================

beta_df = (
    results_df.loc[
        results_df[
            "band"
        ]
        ==
        "Beta"
    ]
    .copy()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "FOCUSED BETA TOPOLOGY"
)

print(
    "=" * 100
)


display(
    beta_df[
        [
            "topology",
            "baseline",
            "roi",
            "active_mean_delta",
            "passive_mean_delta",
            "active_minus_passive",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "hedges_g",
            "exact_permutation_p",
            "p_bh_beta_fdr3",
            "p_bh_fdr9"
        ]
    ]
    .sort_values(
        [
            "topology",
            "baseline",
            "exact_permutation_p"
        ]
    )
    .round(
        6
    )
)


# =============================================================================
# 17. CROSS-BASELINE STABILITY
# =============================================================================

stability_rows = []


for topology in [
    "Absolute",
    "Relative"
]:

    for band in BANDS:

        for roi in ROI_ORDER:

            row_a = results_df.loc[
                (
                    results_df[
                        "topology"
                    ]
                    ==
                    topology
                )
                &
                (
                    results_df[
                        "baseline"
                    ]
                    ==
                    "Baseline A"
                )
                &
                (
                    results_df[
                        "band"
                    ]
                    ==
                    band
                )
                &
                (
                    results_df[
                        "roi"
                    ]
                    ==
                    roi
                )
            ].iloc[
                0
            ]


            row_b = results_df.loc[
                (
                    results_df[
                        "topology"
                    ]
                    ==
                    topology
                )
                &
                (
                    results_df[
                        "baseline"
                    ]
                    ==
                    "Baseline B"
                )
                &
                (
                    results_df[
                        "band"
                    ]
                    ==
                    band
                )
                &
                (
                    results_df[
                        "roi"
                    ]
                    ==
                    roi
                )
            ].iloc[
                0
            ]


            diff_a = float(
                row_a[
                    "active_minus_passive"
                ]
            )


            diff_b = float(
                row_b[
                    "active_minus_passive"
                ]
            )


            stability_rows.append({

                "topology":
                    topology,

                "band":
                    band,

                "roi":
                    roi,

                "baseline_a_difference":
                    diff_a,

                "baseline_b_difference":
                    diff_b,

                "same_direction":
                    bool(
                        np.sign(
                            diff_a
                        )
                        ==
                        np.sign(
                            diff_b
                        )
                    ),

                "baseline_a_g":
                    float(
                        row_a[
                            "hedges_g"
                        ]
                    ),

                "baseline_b_g":
                    float(
                        row_b[
                            "hedges_g"
                        ]
                    ),

                "baseline_a_p":
                    float(
                        row_a[
                            "exact_permutation_p"
                        ]
                    ),

                "baseline_b_p":
                    float(
                        row_b[
                            "exact_permutation_p"
                        ]
                    )
            })


stability_df = pd.DataFrame(
    stability_rows
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "CROSS-BASELINE TOPOLOGY STABILITY"
)

print(
    "=" * 100
)


display(
    stability_df.round(
        6
    )
)


# =============================================================================
# 18. BETA CROSS-BASELINE STABILITY
# =============================================================================

beta_stability_df = (
    stability_df.loc[
        stability_df[
            "band"
        ]
        ==
        "Beta"
    ]
    .copy()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "BETA TOPOLOGY — CROSS-BASELINE STABILITY"
)

print(
    "=" * 100
)


display(
    beta_stability_df.round(
        6
    )
)


# =============================================================================
# 19. LEAVE-ONE-OUT BETA ROBUSTNESS
# =============================================================================

loo_rows = []


for topology, baseline_dict in TOPOLOGY_DEFINITIONS.items():

    for baseline, value_column in baseline_dict.items():

        for roi in ROI_ORDER:

            temp = (
                roi_df.loc[
                    (
                        roi_df[
                            "band"
                        ]
                        ==
                        "Beta"
                    )
                    &
                    (
                        roi_df[
                            "roi"
                        ]
                        ==
                        roi
                    )
                ]
                .sort_values(
                    "subject"
                )
                .reset_index(
                    drop=True
                )
            )


            full_row = results_df.loc[
                (
                    results_df[
                        "topology"
                    ]
                    ==
                    topology
                )
                &
                (
                    results_df[
                        "baseline"
                    ]
                    ==
                    baseline
                )
                &
                (
                    results_df[
                        "band"
                    ]
                    ==
                    "Beta"
                )
                &
                (
                    results_df[
                        "roi"
                    ]
                    ==
                    roi
                )
            ].iloc[
                0
            ]


            full_difference = float(
                full_row[
                    "active_minus_passive"
                ]
            )


            for removed_subject in temp[
                "subject"
            ]:

                reduced = temp.loc[
                    temp[
                        "subject"
                    ]
                    !=
                    removed_subject
                ]


                active = reduced.loc[
                    reduced[
                        "group"
                    ]
                    ==
                    "Active",
                    value_column
                ].to_numpy(
                    dtype=float
                )


                passive = reduced.loc[
                    reduced[
                        "group"
                    ]
                    ==
                    "Passive",
                    value_column
                ].to_numpy(
                    dtype=float
                )


                difference = float(
                    np.mean(
                        active
                    )
                    -
                    np.mean(
                        passive
                    )
                )


                loo_rows.append({

                    "topology":
                        topology,

                    "baseline":
                        baseline,

                    "roi":
                        roi,

                    "removed_subject":
                        removed_subject,

                    "active_minus_passive":
                        difference,

                    "hedges_g":
                        hedges_g(
                            active,
                            passive
                        ),

                    "same_direction_as_full":
                        bool(
                            np.sign(
                                difference
                            )
                            ==
                            np.sign(
                                full_difference
                            )
                        )
                })


loo_df = pd.DataFrame(
    loo_rows
)


loo_summary_df = (
    loo_df
    .groupby(
        [
            "topology",
            "baseline",
            "roi"
        ]
    )
    .agg(

        same_direction_n=(
            "same_direction_as_full",
            "sum"
        ),

        total=(
            "same_direction_as_full",
            "size"
        ),

        difference_min=(
            "active_minus_passive",
            "min"
        ),

        difference_max=(
            "active_minus_passive",
            "max"
        ),

        g_min=(
            "hedges_g",
            "min"
        ),

        g_max=(
            "hedges_g",
            "max"
        )
    )
    .reset_index()
)


loo_summary_df[
    "same_direction_percent"
] = (
    100
    *
    loo_summary_df[
        "same_direction_n"
    ]
    /
    loo_summary_df[
        "total"
    ]
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "BETA TOPOLOGY — LEAVE-ONE-OUT ROBUSTNESS"
)

print(
    "=" * 100
)


display(
    loo_summary_df.round(
        6
    )
)


# =============================================================================
# 20. FIGURE — ABSOLUTE BETA NODE-STRENGTH EFFECTS
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        6
    )
)


x = np.arange(
    len(
        ROI_ORDER
    )
)

width = 0.36


g_a = []

g_b = []


for roi in ROI_ORDER:

    g_a.append(
        results_df.loc[
            (
                results_df[
                    "topology"
                ]
                ==
                "Absolute"
            )
            &
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "roi"
                ]
                ==
                roi
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


    g_b.append(
        results_df.loc[
            (
                results_df[
                    "topology"
                ]
                ==
                "Absolute"
            )
            &
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "roi"
                ]
                ==
                roi
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


ax.bar(
    x - width / 2,
    g_a,
    width,
    label="Baseline A"
)


ax.bar(
    x + width / 2,
    g_b,
    width,
    label="Baseline B"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        "Frontal",
        "Central/\nTemporal",
        "Posterior"
    ]
)


ax.set_ylabel(
    "Hedges' g (Active − Passive)"
)


ax.set_xlabel(
    "ROI"
)


ax.set_title(
    "Absolute beta node-strength effects"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


ABS_FIG = (
    FIG_OUT
    /
    "absolute_beta_node_strength_effects.png"
)


fig.savefig(
    ABS_FIG,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 21. FIGURE — RELATIVE BETA NODE-STRENGTH EFFECTS
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        6
    )
)


g_a = []

g_b = []


for roi in ROI_ORDER:

    g_a.append(
        results_df.loc[
            (
                results_df[
                    "topology"
                ]
                ==
                "Relative"
            )
            &
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "roi"
                ]
                ==
                roi
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


    g_b.append(
        results_df.loc[
            (
                results_df[
                    "topology"
                ]
                ==
                "Relative"
            )
            &
            (
                results_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                results_df[
                    "band"
                ]
                ==
                "Beta"
            )
            &
            (
                results_df[
                    "roi"
                ]
                ==
                roi
            ),
            "hedges_g"
        ].iloc[
            0
        ]
    )


ax.bar(
    x - width / 2,
    g_a,
    width,
    label="Baseline A"
)


ax.bar(
    x + width / 2,
    g_b,
    width,
    label="Baseline B"
)


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        "Frontal",
        "Central/\nTemporal",
        "Posterior"
    ]
)


ax.set_ylabel(
    "Hedges' g (Active − Passive)"
)


ax.set_xlabel(
    "ROI"
)


ax.set_title(
    "Relative beta node-strength effects"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


REL_FIG = (
    FIG_OUT
    /
    "relative_beta_node_strength_effects.png"
)


fig.savefig(
    REL_FIG,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 22. SAVE OUTPUTS
# =============================================================================

node_df.to_csv(
    OUT
    /
    "participant_node_strength.csv",
    index=False
)


roi_df.to_csv(
    OUT
    /
    "participant_roi_strength.csv",
    index=False
)


results_df.to_csv(
    OUT
    /
    "roi_topology_exact_inference.csv",
    index=False
)


beta_df.to_csv(
    OUT
    /
    "beta_topology_focused_inference.csv",
    index=False
)


stability_df.to_csv(
    OUT
    /
    "cross_baseline_topology_stability.csv",
    index=False
)


beta_stability_df.to_csv(
    OUT
    /
    "beta_cross_baseline_topology_stability.csv",
    index=False
)


loo_df.to_csv(
    OUT
    /
    "beta_topology_leave_one_out.csv",
    index=False
)


loo_summary_df.to_csv(
    OUT
    /
    "beta_topology_leave_one_out_summary.csv",
    index=False
)


# =============================================================================
# 23. FINAL INTEGRITY
# =============================================================================

integrity = {

    "participants":
        node_df[
            "subject"
        ].nunique()
        ==
        15,

    "active":
        node_df.loc[
            node_df[
                "group"
            ]
            ==
            "Active",
            "subject"
        ].nunique()
        ==
        8,

    "passive":
        node_df.loc[
            node_df[
                "group"
            ]
            ==
            "Passive",
            "subject"
        ].nunique()
        ==
        7,

    "channels":
        node_df[
            "channel"
        ].nunique()
        ==
        14,

    "bands":
        set(
            node_df[
                "band"
            ].unique()
        )
        ==
        set(
            BANDS
        ),

    "node_rows":
        len(
            node_df
        )
        ==
        630,

    "roi_rows":
        len(
            roi_df
        )
        ==
        135,

    "tests":
        len(
            results_df
        )
        ==
        36,

    "permutations":
        (
            results_df[
                "n_exact_permutations"
            ]
            ==
            EXPECTED_PERMUTATIONS
        ).all(),

    "fdr9_complete":
        results_df[
            "p_bh_fdr9"
        ].notna().all(),

    "beta_fdr3_complete":
        results_df.loc[
            results_df[
                "band"
            ]
            ==
            "Beta",
            "p_bh_beta_fdr3"
        ].notna().all(),

    "finite_node_values":
        np.isfinite(
            node_df[
                [
                    "absolute_delta_a",
                    "absolute_delta_b",
                    "relative_delta_a",
                    "relative_delta_b"
                ]
            ].to_numpy(
                dtype=float
            )
        ).all()
}


overall_pass = all(
    integrity.values()
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "TASK 7B COMPLETE"
)

print(
    "=" * 100
)


print(
    f"Participant-node rows      : {len(node_df)}/630"
)

print(
    f"Participant-ROI rows       : {len(roi_df)}/135"
)

print(
    f"Inferential tests          : {len(results_df)}/36"
)

print(
    f"Exact allocations/test     : {EXPECTED_PERMUTATIONS}"
)

print(
    "Primary multiplicity       : FDR9 within topology × baseline"
)

print(
    "Focused beta multiplicity  : FDR3 within topology × baseline"
)

print(
    "Graph thresholding         : NONE"
)

print(
    "Electrode-level inference  : NONE"
)


print(
    "\nIntegrity components:"
)


for key, value in integrity.items():

    print(
        f"  {key:<27}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 100
)

print(
    "OVERALL INTEGRITY CHECK:",
    (
        "PASS"
        if overall_pass
        else "FAIL"
    )
)

print(
    "-" * 100
)


if not overall_pass:

    raise RuntimeError(
        "Task 7B integrity check failed."
    )


# =============================================================================
# 24. AUTOMATIC BETA INTERPRETATION TABLE
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "BETA TOPOLOGY INTERPRETATION"
)

print(
    "=" * 100
)


for topology in [
    "Absolute",
    "Relative"
]:

    print(
        f"\n{topology.upper()} NODE STRENGTH"
    )


    for roi in ROI_ORDER:

        row_a = beta_df.loc[
            (
                beta_df[
                    "topology"
                ]
                ==
                topology
            )
            &
            (
                beta_df[
                    "baseline"
                ]
                ==
                "Baseline A"
            )
            &
            (
                beta_df[
                    "roi"
                ]
                ==
                roi
            )
        ].iloc[
            0
        ]


        row_b = beta_df.loc[
            (
                beta_df[
                    "topology"
                ]
                ==
                topology
            )
            &
            (
                beta_df[
                    "baseline"
                ]
                ==
                "Baseline B"
            )
            &
            (
                beta_df[
                    "roi"
                ]
                ==
                roi
            )
        ].iloc[
            0
        ]


        same_direction = (
            np.sign(
                row_a[
                    "active_minus_passive"
                ]
            )
            ==
            np.sign(
                row_b[
                    "active_minus_passive"
                ]
            )
        )


        print(
            f"\n{roi}:"
        )


        print(
            f"  Baseline A diff       : "
            f"{row_a['active_minus_passive']:+.6f}"
        )

        print(
            f"  Baseline A g          : "
            f"{row_a['hedges_g']:+.3f}"
        )

        print(
            f"  Baseline A p          : "
            f"{row_a['exact_permutation_p']:.4f}"
        )

        print(
            f"  Baseline A beta-FDR3  : "
            f"{row_a['p_bh_beta_fdr3']:.4f}"
        )


        print(
            f"  Baseline B diff       : "
            f"{row_b['active_minus_passive']:+.6f}"
        )

        print(
            f"  Baseline B g          : "
            f"{row_b['hedges_g']:+.3f}"
        )

        print(
            f"  Baseline B p          : "
            f"{row_b['exact_permutation_p']:.4f}"
        )

        print(
            f"  Baseline B beta-FDR3  : "
            f"{row_b['p_bh_beta_fdr3']:.4f}"
        )


        print(
            f"  Same direction        : "
            f"{same_direction}"
        )


print(
    "\nINTERPRETATION:"
)

print(
    "If absolute beta effects are present but relative beta effects "
    "attenuate, the regional pattern is more consistent with a change "
    "in overall connectivity magnitude than with redistribution of "
    "network organization."
)

print(
    "If relative beta effects remain directionally stable across both "
    "baselines, this provides descriptive evidence for redistribution "
    "of connectivity across ROIs."
)

print(
    "No raw p-value is interpreted as confirmatory unless it survives "
    "the appropriate multiplicity correction."
)


print(
    f"\nOutput directory:\n{OUT}"
)

print(
    "\n"
    +
    "=" * 100
)

In [ ]:
# =============================================================================
# TASK 8A — ACTIVE-GROUP MARKER-PHASE COMPARABILITY
# AND REPEATED-PHASE AUDIT
# CORRECTED FULL VERSION — MATPLOTLIB-COMPATIBLE
# =============================================================================
#
# PURPOSE
# -------
# Freeze a defensible within-Active repeated-phase design BEFORE examining
# EEG outcomes.
#
# This task:
#
#   1. Uses ONLY the existing Task 3B phase/epoch reconstruction.
#   2. Includes Active participants sub-001 ... sub-008 only.
#   3. Audits T1_V1, T1_V2, T1_V3, T1_V4.
#   4. Determines:
#        - occurrence counts
#        - marker positions
#        - phase durations
#        - clean 2-s epoch counts
#        - temporal order
#        - common occurrence depth across all 8 participants
#        - completeness of repeated marker cycles
#   5. Proposes candidate within-subject comparisons from STRUCTURE ONLY.
#
# IMPORTANT
# ---------
# - NO EEG outcome is analyzed.
# - NO spectral/connectivity value is inspected.
# - NO semantic meaning is assigned to T1_V1/V2/V3/V4.
# - T1_V4 remains a neutral marker code.
# - The design is selected using marker structure, duration, and clean
#   epoch availability only.
#
# OUTPUT
# ------
# task08a_active_marker_phase_audit/
#     active_phase_occurrences.csv
#     active_marker_subject_summary.csv
#     active_marker_occurrence_summary.csv
#     active_marker_position_summary.csv
#     active_sequence_summary.csv
#     active_complete_cycle_summary.csv
#     candidate_within_subject_designs.csv
#     frozen_design_recommendation.txt
#     figures/
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK3B = (
    BASE
    / "task03b_phase_reconstruction"
)

OUT = (
    BASE
    / "task08a_active_marker_phase_audit"
)

FIG_OUT = (
    OUT
    / "figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. ACTIVE PARTICIPANTS / MARKERS
# =============================================================================

ACTIVE_SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(
        1,
        9
    )
]

TARGET_CODES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]

EXPECTED_ACTIVE = 8

EPOCH_SECONDS = 2.0


# =============================================================================
# 3. BASIC HELPERS
# =============================================================================

def normalize_columns(df):
    """
    Strip whitespace from dataframe column names.
    """
    result = df.copy()

    result.columns = [
        str(c).strip()
        for c in result.columns
    ]

    return result


def find_column(columns, candidates):
    """
    Find the first exact case-insensitive candidate column.
    """
    lower_map = {
        str(c).lower(): c
        for c in columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    return None


def standardize_subject(value):
    """
    Convert participant identifiers to sub-XXX format.
    """
    value = str(
        value
    ).strip()

    match = re.search(
        r"(\d+)",
        value
    )

    if match:

        return (
            f"sub-{int(match.group(1)):03d}"
        )

    return value


# =============================================================================
# 4. DISCOVER TASK 3B CSV FILES
# =============================================================================

if not TASK3B.exists():

    raise FileNotFoundError(
        f"Task 3B directory not found:\n{TASK3B}"
    )


csv_files = sorted(
    TASK3B.rglob(
        "*.csv"
    )
)


if not csv_files:

    raise FileNotFoundError(
        f"No CSV files found under:\n{TASK3B}"
    )


print(
    "=" * 105
)

print(
    "TASK 8A — ACTIVE-GROUP MARKER-PHASE COMPARABILITY "
    "AND REPEATED-PHASE AUDIT"
)

print(
    "=" * 105
)

print(
    f"Task 3B directory : {TASK3B}"
)

print(
    f"CSV files found   : {len(csv_files)}"
)


for path in csv_files:

    try:

        temp = pd.read_csv(
            path,
            nrows=3
        )

        print(
            f"\n{path.name}"
        )

        print(
            "  columns:",
            list(
                temp.columns
            )
        )

    except Exception as exc:

        print(
            f"\nCould not inspect {path.name}: {exc}"
        )


# =============================================================================
# 5. IDENTIFY PHASE AND EPOCH TABLE CANDIDATES
# =============================================================================

phase_candidates = []

epoch_candidates = []


for path in csv_files:

    try:

        df = normalize_columns(
            pd.read_csv(
                path
            )
        )

    except Exception:

        continue


    columns_lower = {
        str(c).lower()
        for c in df.columns
    }


    # -------------------------------------------------------------------------
    # Phase-table score
    # -------------------------------------------------------------------------

    phase_score = 0

    for keyword in [
        "marker",
        "code",
        "position",
        "start",
        "end",
        "duration",
        "phase"
    ]:

        if any(
            keyword in c
            for c in columns_lower
        ):

            phase_score += 1


    # -------------------------------------------------------------------------
    # Epoch-table score
    # -------------------------------------------------------------------------

    epoch_score = 0

    for keyword in [
        "epoch",
        "clean",
        "reject",
        "accepted",
        "good",
        "phase"
    ]:

        if any(
            keyword in c
            for c in columns_lower
        ):

            epoch_score += 1


    if phase_score >= 3:

        phase_candidates.append(
            (
                path,
                phase_score,
                len(df),
                list(df.columns)
            )
        )


    if epoch_score >= 2:

        epoch_candidates.append(
            (
                path,
                epoch_score,
                len(df),
                list(df.columns)
            )
        )


print(
    "\n"
    +
    "=" * 105
)

print(
    "CSV DISCOVERY"
)

print(
    "=" * 105
)


print(
    "\nPossible phase tables:"
)

for item in sorted(
    phase_candidates,
    key=lambda x: (
        -x[1],
        x[0].name
    )
):

    print(
        f"  {item[0].name:<50} "
        f"score={item[1]} rows={item[2]}"
    )


print(
    "\nPossible epoch tables:"
)

for item in sorted(
    epoch_candidates,
    key=lambda x: (
        -x[1],
        x[0].name
    )
):

    print(
        f"  {item[0].name:<50} "
        f"score={item[1]} rows={item[2]}"
    )


# =============================================================================
# 6. SELECT PHASE TABLE
# =============================================================================

phase_named = [
    item
    for item in phase_candidates
    if "phase" in item[
        0
    ].name.lower()
]


if phase_named:

    phase_path = sorted(
        phase_named,
        key=lambda x: (
            -x[1],
            -x[2]
        )
    )[0][0]

else:

    if not phase_candidates:

        raise RuntimeError(
            "Could not identify a Task 3B phase table."
        )

    phase_path = sorted(
        phase_candidates,
        key=lambda x: (
            -x[1],
            -x[2]
        )
    )[0][0]


phase_df = normalize_columns(
    pd.read_csv(
        phase_path
    )
)


print(
    "\nSelected phase table:"
)

print(
    phase_path
)

print(
    f"Shape: {phase_df.shape}"
)


# =============================================================================
# 7. IDENTIFY PHASE-TABLE COLUMNS
# =============================================================================

subject_col = find_column(
    phase_df.columns,
    [
        "subject",
        "participant",
        "participant_id",
        "subject_id"
    ]
)


group_col = find_column(
    phase_df.columns,
    [
        "group",
        "condition"
    ]
)


marker_col = find_column(
    phase_df.columns,
    [
        "marker_code",
        "code",
        "marker",
        "event_code",
        "marker_label",
        "phase_code"
    ]
)


position_col = find_column(
    phase_df.columns,
    [
        "position",
        "marker_position",
        "phase_position",
        "event_position"
    ]
)


start_col = find_column(
    phase_df.columns,
    [
        "start_sec",
        "start_s",
        "start_time",
        "start",
        "onset_sec",
        "onset"
    ]
)


end_col = find_column(
    phase_df.columns,
    [
        "end_sec",
        "end_s",
        "end_time",
        "end"
    ]
)


duration_col = find_column(
    phase_df.columns,
    [
        "duration_sec",
        "duration_s",
        "duration",
        "phase_duration_sec"
    ]
)


if subject_col is None:

    raise RuntimeError(
        "Could not identify subject column in phase table."
    )


if marker_col is None:

    raise RuntimeError(
        "Could not identify marker-code column in phase table."
    )


if position_col is None:

    raise RuntimeError(
        "Could not identify marker-position column in phase table."
    )


print(
    "\nResolved phase columns:"
)

print(
    f"  subject  : {subject_col}"
)

print(
    f"  group    : {group_col}"
)

print(
    f"  marker   : {marker_col}"
)

print(
    f"  position : {position_col}"
)

print(
    f"  start    : {start_col}"
)

print(
    f"  end      : {end_col}"
)

print(
    f"  duration : {duration_col}"
)


# =============================================================================
# 8. STANDARDIZE PHASE TABLE
# =============================================================================

phase = pd.DataFrame()


phase[
    "subject"
] = (
    phase_df[
        subject_col
    ]
    .astype(str)
    .str.strip()
    .map(
        standardize_subject
    )
)


phase[
    "marker_code"
] = (
    phase_df[
        marker_col
    ]
    .astype(str)
    .str.strip()
)


phase[
    "position"
] = pd.to_numeric(
    phase_df[
        position_col
    ],
    errors="coerce"
)


if group_col is not None:

    phase[
        "group"
    ] = (
        phase_df[
            group_col
        ]
        .astype(str)
        .str.strip()
    )

else:

    phase[
        "group"
    ] = np.where(
        phase[
            "subject"
        ].isin(
            ACTIVE_SUBJECTS
        ),
        "Active",
        "Passive"
    )


if start_col is not None:

    phase[
        "start_sec"
    ] = pd.to_numeric(
        phase_df[
            start_col
        ],
        errors="coerce"
    )

else:

    phase[
        "start_sec"
    ] = np.nan


if end_col is not None:

    phase[
        "end_sec"
    ] = pd.to_numeric(
        phase_df[
            end_col
        ],
        errors="coerce"
    )

else:

    phase[
        "end_sec"
    ] = np.nan


if duration_col is not None:

    phase[
        "duration_sec"
    ] = pd.to_numeric(
        phase_df[
            duration_col
        ],
        errors="coerce"
    )

else:

    phase[
        "duration_sec"
    ] = (
        phase[
            "end_sec"
        ]
        -
        phase[
            "start_sec"
        ]
    )


# =============================================================================
# 9. ACTIVE T1 PHASES ONLY
# =============================================================================

active_phase = phase.loc[
    (
        phase[
            "subject"
        ].isin(
            ACTIVE_SUBJECTS
        )
    )
    &
    (
        phase[
            "marker_code"
        ].isin(
            TARGET_CODES
        )
    )
].copy()


active_phase = (
    active_phase
    .sort_values(
        [
            "subject",
            "position"
        ]
    )
    .reset_index(
        drop=True
    )
)


if active_phase.empty:

    raise RuntimeError(
        "No Active T1_V1–T1_V4 phase rows found."
    )


observed_active = sorted(
    active_phase[
        "subject"
    ].unique()
)


if observed_active != ACTIVE_SUBJECTS:

    raise RuntimeError(
        "\nActive participant mismatch.\n"
        f"Observed: {observed_active}\n"
        f"Expected: {ACTIVE_SUBJECTS}"
    )


# =============================================================================
# 10. ASSIGN OCCURRENCE NUMBER WITHIN MARKER CODE
# =============================================================================

active_phase[
    "occurrence"
] = (
    active_phase
    .groupby(
        [
            "subject",
            "marker_code"
        ]
    )
    .cumcount()
    +
    1
)


# =============================================================================
# 11. DISCOVER EPOCH TABLE
# =============================================================================

epoch_named = [
    item
    for item in epoch_candidates
    if "epoch" in item[
        0
    ].name.lower()
]


if epoch_named:

    epoch_path = sorted(
        epoch_named,
        key=lambda x: (
            -x[1],
            -x[2]
        )
    )[0][0]

else:

    epoch_path = None


epoch_df = None


if epoch_path is not None:

    epoch_df = normalize_columns(
        pd.read_csv(
            epoch_path
        )
    )


    print(
        "\nSelected epoch table:"
    )

    print(
        epoch_path
    )

    print(
        f"Shape: {epoch_df.shape}"
    )

else:

    print(
        "\nNo separate epoch table confidently identified."
    )


# =============================================================================
# 12. RESOLVE EPOCH COLUMNS
# =============================================================================

epoch_available = False

e_subject_col = None
e_marker_col = None
e_position_col = None
e_clean_col = None
e_reject_col = None


if epoch_df is not None:

    e_subject_col = find_column(
        epoch_df.columns,
        [
            "subject",
            "participant",
            "participant_id",
            "subject_id"
        ]
    )


    e_marker_col = find_column(
        epoch_df.columns,
        [
            "marker_code",
            "code",
            "marker",
            "event_code",
            "phase_code"
        ]
    )


    e_position_col = find_column(
        epoch_df.columns,
        [
            "position",
            "marker_position",
            "phase_position"
        ]
    )


    e_clean_col = find_column(
        epoch_df.columns,
        [
            "clean",
            "is_clean",
            "accepted",
            "good",
            "keep",
            "retained"
        ]
    )


    e_reject_col = find_column(
        epoch_df.columns,
        [
            "rejected",
            "reject",
            "is_rejected"
        ]
    )


    if (
        e_subject_col is not None
        and
        e_position_col is not None
    ):

        epoch_available = True


        print(
            "\nResolved epoch columns:"
        )

        print(
            f"  subject  : {e_subject_col}"
        )

        print(
            f"  marker   : {e_marker_col}"
        )

        print(
            f"  position : {e_position_col}"
        )

        print(
            f"  clean    : {e_clean_col}"
        )

        print(
            f"  rejected : {e_reject_col}"
        )


# =============================================================================
# 13. CLEAN-EPOCH COUNT BY PHASE
# =============================================================================

if epoch_available:

    epochs = epoch_df.copy()


    epochs[
        "subject_std"
    ] = epochs[
        e_subject_col
    ].map(
        standardize_subject
    )


    epochs[
        "position_std"
    ] = pd.to_numeric(
        epochs[
            e_position_col
        ],
        errors="coerce"
    )


    if e_clean_col is not None:

        raw_clean = epochs[
            e_clean_col
        ]


        if pd.api.types.is_bool_dtype(
            raw_clean
        ):

            epochs[
                "is_clean_std"
            ] = raw_clean.astype(
                bool
            )

        elif pd.api.types.is_numeric_dtype(
            raw_clean
        ):

            epochs[
                "is_clean_std"
            ] = (
                pd.to_numeric(
                    raw_clean,
                    errors="coerce"
                )
                ==
                1
            )

        else:

            clean_text = (
                raw_clean
                .astype(str)
                .str.strip()
                .str.lower()
            )


            epochs[
                "is_clean_std"
            ] = clean_text.isin(
                [
                    "true",
                    "1",
                    "yes",
                    "y",
                    "clean",
                    "accepted",
                    "good",
                    "keep",
                    "retained"
                ]
            )


    elif e_reject_col is not None:

        raw_reject = epochs[
            e_reject_col
        ]


        if pd.api.types.is_bool_dtype(
            raw_reject
        ):

            rejected = raw_reject.astype(
                bool
            )

        elif pd.api.types.is_numeric_dtype(
            raw_reject
        ):

            rejected = (
                pd.to_numeric(
                    raw_reject,
                    errors="coerce"
                )
                ==
                1
            )

        else:

            reject_text = (
                raw_reject
                .astype(str)
                .str.strip()
                .str.lower()
            )


            rejected = reject_text.isin(
                [
                    "true",
                    "1",
                    "yes",
                    "y",
                    "rejected",
                    "reject"
                ]
            )


        epochs[
            "is_clean_std"
        ] = ~rejected


    else:

        # Structural assumption only if the selected Task3B epoch table
        # already consists of retained phase-contained epochs.
        epochs[
            "is_clean_std"
        ] = True


    clean_counts = (
        epochs.loc[
            epochs[
                "is_clean_std"
            ]
        ]
        .groupby(
            [
                "subject_std",
                "position_std"
            ]
        )
        .size()
        .rename(
            "clean_epochs"
        )
        .reset_index()
    )


    active_phase = active_phase.merge(
        clean_counts,
        how="left",
        left_on=[
            "subject",
            "position"
        ],
        right_on=[
            "subject_std",
            "position_std"
        ]
    )


    active_phase[
        "clean_epochs"
    ] = (
        active_phase[
            "clean_epochs"
        ]
        .fillna(0)
        .astype(int)
    )


    active_phase = active_phase.drop(
        columns=[
            "subject_std",
            "position_std"
        ],
        errors="ignore"
    )


    epoch_count_source = (
        "Task3B clean epoch table"
    )


else:

    # -------------------------------------------------------------------------
    # Fallback only:
    # Number of complete 2-second epochs implied by phase duration.
    #
    # This is NOT equivalent to artifact-clean epoch availability.
    # -------------------------------------------------------------------------

    active_phase[
        "clean_epochs"
    ] = np.floor(
        active_phase[
            "duration_sec"
        ]
        /
        EPOCH_SECONDS
    )


    active_phase[
        "clean_epochs"
    ] = active_phase[
        "clean_epochs"
    ].where(
        active_phase[
            "duration_sec"
        ].notna()
    )


    epoch_count_source = (
        "duration-derived structural maximum"
    )


active_phase[
    "epoch_count_source"
] = epoch_count_source


# =============================================================================
# 14. SUBJECT × MARKER SUMMARY
# =============================================================================

subject_marker_summary = (
    active_phase
    .groupby(
        [
            "subject",
            "marker_code"
        ]
    )
    .agg(

        n_occurrences=(
            "occurrence",
            "max"
        ),

        first_position=(
            "position",
            "min"
        ),

        last_position=(
            "position",
            "max"
        ),

        median_duration_sec=(
            "duration_sec",
            "median"
        ),

        min_duration_sec=(
            "duration_sec",
            "min"
        ),

        max_duration_sec=(
            "duration_sec",
            "max"
        ),

        median_clean_epochs=(
            "clean_epochs",
            "median"
        ),

        min_clean_epochs=(
            "clean_epochs",
            "min"
        ),

        max_clean_epochs=(
            "clean_epochs",
            "max"
        )
    )
    .reset_index()
)


# =============================================================================
# 15. OCCURRENCE-LEVEL COMPARABILITY
# =============================================================================

occurrence_rows = []


for marker in TARGET_CODES:

    marker_df = active_phase.loc[
        active_phase[
            "marker_code"
        ]
        ==
        marker
    ]


    if marker_df.empty:

        continue


    max_occurrence = int(
        marker_df[
            "occurrence"
        ].max()
    )


    for occurrence in range(
        1,
        max_occurrence + 1
    ):

        temp = marker_df.loc[
            marker_df[
                "occurrence"
            ]
            ==
            occurrence
        ]


        subjects_present = temp[
            "subject"
        ].nunique()


        durations = temp[
            "duration_sec"
        ].dropna()


        clean_epochs = temp[
            "clean_epochs"
        ].dropna()


        occurrence_rows.append({

            "marker_code":
                marker,

            "occurrence":
                occurrence,

            "subjects_present":
                int(
                    subjects_present
                ),

            "all_8_present":
                bool(
                    subjects_present
                    ==
                    EXPECTED_ACTIVE
                ),

            "median_position":
                float(
                    temp[
                        "position"
                    ].median()
                )
                if len(temp)
                else np.nan,

            "min_position":
                float(
                    temp[
                        "position"
                    ].min()
                )
                if len(temp)
                else np.nan,

            "max_position":
                float(
                    temp[
                        "position"
                    ].max()
                )
                if len(temp)
                else np.nan,

            "median_duration_sec":
                float(
                    durations.median()
                )
                if len(durations)
                else np.nan,

            "min_duration_sec":
                float(
                    durations.min()
                )
                if len(durations)
                else np.nan,

            "max_duration_sec":
                float(
                    durations.max()
                )
                if len(durations)
                else np.nan,

            "median_clean_epochs":
                float(
                    clean_epochs.median()
                )
                if len(clean_epochs)
                else np.nan,

            "min_clean_epochs":
                float(
                    clean_epochs.min()
                )
                if len(clean_epochs)
                else np.nan,

            "max_clean_epochs":
                float(
                    clean_epochs.max()
                )
                if len(clean_epochs)
                else np.nan
        })


occurrence_summary = pd.DataFrame(
    occurrence_rows
)


# =============================================================================
# 16. COMMON OCCURRENCE DEPTH
# =============================================================================

common_depth_rows = []


for marker in TARGET_CODES:

    counts = (
        active_phase.loc[
            active_phase[
                "marker_code"
            ]
            ==
            marker
        ]
        .groupby(
            "subject"
        )
        .size()
        .reindex(
            ACTIVE_SUBJECTS,
            fill_value=0
        )
    )


    common_depth_rows.append({

        "marker_code":
            marker,

        "minimum_occurrences_across_8":
            int(
                counts.min()
            ),

        "median_occurrences":
            float(
                counts.median()
            ),

        "maximum_occurrences":
            int(
                counts.max()
            ),

        "subjects_with_marker":
            int(
                (
                    counts
                    >
                    0
                ).sum()
            )
    })


common_depth_df = pd.DataFrame(
    common_depth_rows
)


# =============================================================================
# 17. POSITION / DURATION SUMMARY
# =============================================================================

position_summary = (
    active_phase
    .groupby(
        "marker_code"
    )
    .agg(

        total_occurrences=(
            "position",
            "size"
        ),

        subjects=(
            "subject",
            "nunique"
        ),

        median_position=(
            "position",
            "median"
        ),

        min_position=(
            "position",
            "min"
        ),

        max_position=(
            "position",
            "max"
        ),

        median_duration_sec=(
            "duration_sec",
            "median"
        ),

        median_clean_epochs=(
            "clean_epochs",
            "median"
        )
    )
    .reset_index()
)


# =============================================================================
# 18. SEQUENCE BY SUBJECT
# =============================================================================

sequence_rows = []


for subject in ACTIVE_SUBJECTS:

    temp = (
        active_phase.loc[
            active_phase[
                "subject"
            ]
            ==
            subject
        ]
        .sort_values(
            "position"
        )
    )


    sequence = temp[
        "marker_code"
    ].tolist()


    position_sequence = temp[
        "position"
    ].tolist()


    sequence_rows.append({

        "subject":
            subject,

        "n_t1_phases":
            len(
                sequence
            ),

        "sequence":
            " -> ".join(
                sequence
            ),

        "positions":
            ", ".join(
                str(
                    int(x)
                )
                for x in position_sequence
                if pd.notna(x)
            )
    })


sequence_df = pd.DataFrame(
    sequence_rows
)


# =============================================================================
# 19. AUDIT COMPLETE V1 → V2 → V3 → V4 CYCLES
# =============================================================================
#
# STRUCTURAL sequence audit only.
#
# It does NOT imply that V1–V4 represent known cognitive or experimental
# states.
# =============================================================================

cycle_rows = []


TARGET_SEQUENCE = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]


for subject in ACTIVE_SUBJECTS:

    temp = (
        active_phase.loc[
            active_phase[
                "subject"
            ]
            ==
            subject
        ]
        .sort_values(
            "position"
        )
        .reset_index(
            drop=True
        )
    )


    codes = temp[
        "marker_code"
    ].tolist()


    complete_cycles = []


    for i in range(
        max(
            0,
            len(codes) - 3
        )
    ):

        window = codes[
            i:i + 4
        ]


        if window == TARGET_SEQUENCE:

            cycle_temp = temp.iloc[
                i:i + 4
            ]


            cycle_clean = (
                cycle_temp[
                    "clean_epochs"
                ]
                .dropna()
            )


            cycle_duration = (
                cycle_temp[
                    "duration_sec"
                ]
                .dropna()
            )


            complete_cycles.append({

                "start_row":
                    i,

                "start_position":
                    int(
                        cycle_temp[
                            "position"
                        ].iloc[0]
                    ),

                "end_position":
                    int(
                        cycle_temp[
                            "position"
                        ].iloc[-1]
                    ),

                "minimum_clean_epochs":
                    float(
                        cycle_clean.min()
                    )
                    if len(cycle_clean)
                    else np.nan,

                "median_duration_sec":
                    float(
                        cycle_duration.median()
                    )
                    if len(cycle_duration)
                    else np.nan
            })


    cycle_rows.append({

        "subject":
            subject,

        "n_complete_v1_v2_v3_v4_cycles":
            len(
                complete_cycles
            ),

        "cycle_start_positions":
            ", ".join(
                str(
                    x[
                        "start_position"
                    ]
                )
                for x in complete_cycles
            ),

        "minimum_clean_epochs_across_complete_cycles":
            min(
                [
                    x[
                        "minimum_clean_epochs"
                    ]
                    for x in complete_cycles
                    if np.isfinite(
                        x[
                            "minimum_clean_epochs"
                        ]
                    )
                ],
                default=np.nan
            )
    })


cycle_df = pd.DataFrame(
    cycle_rows
)


# =============================================================================
# 20. CANDIDATE DESIGN GENERATION
# =============================================================================
#
# Candidate A
# -----------
# First occurrence of each T1 code if all 8 participants contain it.
#
# Candidate B
# -----------
# Repeated occurrence analysis for codes having >=2 common occurrences.
#
# Candidate C
# -----------
# First complete V1→V2→V3→V4 cycle if all 8 participants contain one.
#
# Candidate D
# -----------
# First two complete cycles if all 8 participants contain >=2.
#
# =============================================================================

candidate_rows = []


# -----------------------------------------------------------------------------
# Candidate A
# -----------------------------------------------------------------------------

first_occurrence_complete = bool(
    (
        common_depth_df[
            "minimum_occurrences_across_8"
        ]
        >=
        1
    ).all()
)


candidate_rows.append({

    "candidate":
        "A",

    "design":
        "First occurrence of T1_V1, T1_V2, T1_V3, T1_V4",

    "within_subject":
        True,

    "all_8_complete":
        first_occurrence_complete,

    "semantic_labels_required":
        False,

    "purpose":
        "Four-code marker-defined phase comparison",

    "status":
        (
            "ELIGIBLE"
            if first_occurrence_complete
            else
            "NOT ELIGIBLE"
        )
})


# -----------------------------------------------------------------------------
# Candidate B
# -----------------------------------------------------------------------------

for marker in TARGET_CODES:

    row = common_depth_df.loc[
        common_depth_df[
            "marker_code"
        ]
        ==
        marker
    ].iloc[0]


    depth = int(
        row[
            "minimum_occurrences_across_8"
        ]
    )


    candidate_rows.append({

        "candidate":
            f"B-{marker}",

        "design":
            (
                f"Repeated occurrences of {marker} "
                f"(common depth = {depth})"
            ),

        "within_subject":
            True,

        "all_8_complete":
            bool(
                depth >= 2
            ),

        "semantic_labels_required":
            False,

        "purpose":
            "Within-marker temporal/repetition dynamics",

        "status":
            (
                "ELIGIBLE"
                if depth >= 2
                else
                "NOT ELIGIBLE"
            )
    })


# -----------------------------------------------------------------------------
# Candidate C / D
# -----------------------------------------------------------------------------

min_complete_cycles = int(
    cycle_df[
        "n_complete_v1_v2_v3_v4_cycles"
    ].min()
)


candidate_rows.append({

    "candidate":
        "C",

    "design":
        "First complete T1_V1→V2→V3→V4 cycle",

    "within_subject":
        True,

    "all_8_complete":
        bool(
            min_complete_cycles
            >=
            1
        ),

    "semantic_labels_required":
        False,

    "purpose":
        "Balanced first-cycle marker-phase dynamics",

    "status":
        (
            "ELIGIBLE"
            if min_complete_cycles >= 1
            else
            "NOT ELIGIBLE"
        )
})


candidate_rows.append({

    "candidate":
        "D",

    "design":
        "First two complete T1_V1→V2→V3→V4 cycles",

    "within_subject":
        True,

    "all_8_complete":
        bool(
            min_complete_cycles
            >=
            2
        ),

    "semantic_labels_required":
        False,

    "purpose":
        "Cycle/repetition × marker-code dynamics",

    "status":
        (
            "ELIGIBLE"
            if min_complete_cycles >= 2
            else
            "NOT ELIGIBLE"
        )
})


candidate_df = pd.DataFrame(
    candidate_rows
)


# =============================================================================
# 21. DETERMINE STRUCTURALLY SAFEST DESIGN
# =============================================================================

common_depth_map = dict(
    zip(
        common_depth_df[
            "marker_code"
        ],
        common_depth_df[
            "minimum_occurrences_across_8"
        ]
    )
)


eligible_repeated_markers = [
    marker
    for marker in TARGET_CODES
    if common_depth_map.get(
        marker,
        0
    )
    >=
    2
]


if min_complete_cycles >= 2:

    recommended_design = (
        "Candidate D: first two complete "
        "T1_V1→T1_V2→T1_V3→T1_V4 cycles. "
        "This provides the strongest structurally balanced within-subject "
        "design because every Active participant contributes the same "
        "marker sequence across two repeated cycles."
    )

elif min_complete_cycles >= 1:

    recommended_design = (
        "Candidate C: first complete "
        "T1_V1→T1_V2→T1_V3→T1_V4 cycle. "
        "Every Active participant contributes the same four-code sequence. "
        "Marker codes must remain semantically neutral."
    )

elif first_occurrence_complete:

    recommended_design = (
        "Candidate A: first occurrence of T1_V1, T1_V2, T1_V3 and T1_V4. "
        "All Active participants contain the four marker codes, but they "
        "must be treated as marker-defined phases rather than assumed "
        "experimental conditions."
    )

elif eligible_repeated_markers:

    recommended_design = (
        "Candidate B: restrict the within-subject analysis to repeated "
        "occurrences of structurally complete marker codes: "
        +
        ", ".join(
            eligible_repeated_markers
        )
        +
        "."
    )

else:

    recommended_design = (
        "No balanced repeated-phase design is supported across all eight "
        "Active participants. Do not proceed to within-Active EEG inference "
        "without revising the design."
    )


# =============================================================================
# 22. SUBJECT × MARKER OCCURRENCE COUNTS
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "ACTIVE SUBJECT × MARKER OCCURRENCE COUNTS"
)

print(
    "=" * 105
)


occurrence_pivot = (
    subject_marker_summary
    .pivot(
        index="subject",
        columns="marker_code",
        values="n_occurrences"
    )
    .reindex(
        index=ACTIVE_SUBJECTS,
        columns=TARGET_CODES
    )
    .fillna(0)
    .astype(int)
)


display(
    occurrence_pivot
)


# =============================================================================
# 23. COMMON OCCURRENCE DEPTH
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "COMMON OCCURRENCE DEPTH ACROSS ALL 8 ACTIVE PARTICIPANTS"
)

print(
    "=" * 105
)


display(
    common_depth_df
)


# =============================================================================
# 24. OCCURRENCE-LEVEL COMPARABILITY
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "OCCURRENCE-LEVEL COMPARABILITY"
)

print(
    "=" * 105
)


display(
    occurrence_summary.round(
        3
    )
)


# =============================================================================
# 25. MARKER POSITION / DURATION SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "MARKER POSITION / DURATION SUMMARY"
)

print(
    "=" * 105
)


display(
    position_summary.round(
        3
    )
)


# =============================================================================
# 26. SUBJECT SEQUENCES
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "ACTIVE MARKER SEQUENCES"
)

print(
    "=" * 105
)


display(
    sequence_df
)


# =============================================================================
# 27. COMPLETE CYCLE SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "COMPLETE T1_V1→V2→V3→V4 CYCLE AUDIT"
)

print(
    "=" * 105
)


display(
    cycle_df
)


print(
    f"\nMinimum complete cycles across all 8 participants: "
    f"{min_complete_cycles}"
)


# =============================================================================
# 28. CANDIDATE DESIGNS
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "CANDIDATE WITHIN-SUBJECT DESIGNS"
)

print(
    "=" * 105
)


display(
    candidate_df
)


# =============================================================================
# 29. RECOMMENDED DESIGN
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "STRUCTURE-ONLY FROZEN-DESIGN RECOMMENDATION"
)

print(
    "=" * 105
)


print(
    recommended_design
)


print(
    "\nIMPORTANT:"
)

print(
    "This recommendation is based ONLY on marker structure, phase duration, "
    "and epoch availability."
)

print(
    "No EEG outcome has been used to choose the design."
)

print(
    "T1_V1/T1_V2/T1_V3/T1_V4 remain neutral marker-code labels."
)


# =============================================================================
# 30. FIGURE — OCCURRENCES PER SUBJECT
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        10,
        6
    )
)


occurrence_pivot.plot(
    kind="bar",
    ax=ax
)


ax.set_xlabel(
    "Active participant"
)

ax.set_ylabel(
    "Number of occurrences"
)

ax.set_title(
    "Repeated marker-code occurrences in the Active group"
)

ax.legend(
    title="Marker code",
    frameon=False
)

ax.tick_params(
    axis="x",
    rotation=45
)


fig.tight_layout()


OCCURRENCE_FIG = (
    FIG_OUT
    /
    "active_marker_occurrence_counts.png"
)


fig.savefig(
    OCCURRENCE_FIG,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 31. FIGURE — COMMON OCCURRENCE DEPTH
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        5
    )
)


ax.bar(
    common_depth_df[
        "marker_code"
    ],
    common_depth_df[
        "minimum_occurrences_across_8"
    ]
)


ax.set_xlabel(
    "Marker code"
)

ax.set_ylabel(
    "Minimum occurrences across 8 participants"
)

ax.set_title(
    "Common repeated-phase depth in the Active group"
)


fig.tight_layout()


COMMON_DEPTH_FIG = (
    FIG_OUT
    /
    "common_marker_occurrence_depth.png"
)


fig.savefig(
    COMMON_DEPTH_FIG,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 32. FIGURE — PHASE DURATION BY MARKER
# =============================================================================
#
# CORRECTION:
#
# Modern Matplotlib versions use:
#
#       tick_labels=
#
# rather than:
#
#       labels=
#
# This was the source of the previous TypeError.
# =============================================================================

duration_plot_df = active_phase.loc[
    active_phase[
        "duration_sec"
    ].notna()
].copy()


if not duration_plot_df.empty:

    data = []

    valid_labels = []


    for marker in TARGET_CODES:

        values = (
            duration_plot_df.loc[
                duration_plot_df[
                    "marker_code"
                ]
                ==
                marker,
                "duration_sec"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )


        # Only pass non-empty arrays to Matplotlib.
        if len(values) > 0:

            data.append(
                values
            )

            valid_labels.append(
                marker
            )


    if data:

        fig, ax = plt.subplots(
            figsize=(
                9,
                6
            )
        )


        ax.boxplot(
            data,
            tick_labels=valid_labels
        )


        ax.set_xlabel(
            "Marker code"
        )

        ax.set_ylabel(
            "Phase duration (s)"
        )

        ax.set_title(
            "Duration distribution of Active marker-defined phases"
        )


        fig.tight_layout()


        DURATION_FIG = (
            FIG_OUT
            /
            "active_phase_duration_by_marker.png"
        )


        fig.savefig(
            DURATION_FIG,
            dpi=300,
            bbox_inches="tight"
        )


        plt.show()

    else:

        print(
            "\nNo finite phase-duration values available for boxplot."
        )

else:

    print(
        "\nNo phase-duration values available for plotting."
    )


# =============================================================================
# 33. SAVE TABLES
# =============================================================================

active_phase.to_csv(
    OUT
    /
    "active_phase_occurrences.csv",
    index=False
)


subject_marker_summary.to_csv(
    OUT
    /
    "active_marker_subject_summary.csv",
    index=False
)


occurrence_summary.to_csv(
    OUT
    /
    "active_marker_occurrence_summary.csv",
    index=False
)


position_summary.to_csv(
    OUT
    /
    "active_marker_position_summary.csv",
    index=False
)


sequence_df.to_csv(
    OUT
    /
    "active_sequence_summary.csv",
    index=False
)


cycle_df.to_csv(
    OUT
    /
    "active_complete_cycle_summary.csv",
    index=False
)


candidate_df.to_csv(
    OUT
    /
    "candidate_within_subject_designs.csv",
    index=False
)


with open(
    OUT
    /
    "frozen_design_recommendation.txt",
    "w",
    encoding="utf-8"
) as handle:

    handle.write(
        recommended_design
        +
        "\n\n"
    )

    handle.write(
        "Design selected from marker structure only.\n"
    )

    handle.write(
        "No EEG outcome was inspected.\n"
    )

    handle.write(
        "T1_V1/T1_V2/T1_V3/T1_V4 retain neutral labels.\n"
    )


# =============================================================================
# 34. FINAL INTEGRITY CHECK
# =============================================================================

integrity = {

    "active_subjects":
        active_phase[
            "subject"
        ].nunique()
        ==
        8,

    "target_markers_only":
        set(
            active_phase[
                "marker_code"
            ].unique()
        ).issubset(
            set(
                TARGET_CODES
            )
        ),

    "positions_finite":
        np.isfinite(
            active_phase[
                "position"
            ].to_numpy(
                dtype=float
            )
        ).all(),

    "occurrences_positive":
        (
            active_phase[
                "occurrence"
            ]
            >=
            1
        ).all(),

    "subject_summary_complete":
        subject_marker_summary[
            "subject"
        ].nunique()
        ==
        8,

    "common_depth_rows":
        len(
            common_depth_df
        )
        ==
        4,

    "sequence_rows":
        len(
            sequence_df
        )
        ==
        8,

    "cycle_rows":
        len(
            cycle_df
        )
        ==
        8,

    "candidate_table_nonempty":
        len(
            candidate_df
        )
        >
        0
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 105
)

print(
    "TASK 8A COMPLETE"
)

print(
    "=" * 105
)


print(
    f"Active participants         : "
    f"{active_phase['subject'].nunique()}/8"
)

print(
    f"T1 phase rows               : "
    f"{len(active_phase)}"
)

print(
    f"Minimum complete cycles     : "
    f"{min_complete_cycles}"
)

print(
    f"Epoch-count source          : "
    f"{epoch_count_source}"
)


print(
    "\nIntegrity components:"
)


for key, value in integrity.items():

    print(
        f"  {key:<30}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 105
)

print(
    "OVERALL INTEGRITY CHECK:",
    (
        "PASS"
        if overall_pass
        else "FAIL"
    )
)

print(
    "-" * 105
)


if not overall_pass:

    raise RuntimeError(
        "Task 8A integrity check failed."
    )


# =============================================================================
# 35. FINAL DESIGN REPORT
# =============================================================================

print(
    "\n"
    +
    "=" * 105
)

print(
    "FROZEN-DESIGN RECOMMENDATION"
)

print(
    "=" * 105
)


print(
    recommended_design
)


print(
    "\nDesign-selection safeguards:"
)

print(
    "  • Marker structure only"
)

print(
    "  • No spectral outcomes inspected"
)

print(
    "  • No connectivity outcomes inspected"
)

print(
    "  • No condition selected because of statistical significance"
)

print(
    "  • T1 marker codes retain neutral labels"
)

print(
    "  • Equal-participant completeness required for recommended design"
)


print(
    f"\nOutput directory:\n{OUT}"
)

print(
    "\nSaved figures:"
)

print(
    f"  {OCCURRENCE_FIG}"
)

print(
    f"  {COMMON_DEPTH_FIG}"
)

if (
    "DURATION_FIG"
    in globals()
):

    print(
        f"  {DURATION_FIG}"
    )


print(
    "\n"
    +
    "=" * 105
)

In [ ]:
# =============================================================================
# TASK 8B — FROZEN FIRST-CYCLE BALANCED EPOCH CONSTRUCTION
# =============================================================================
#
# DESIGN FROZEN BY TASK 8A
# -------------------------
# Active participants : sub-001 ... sub-008
# Cycle               : first complete T1_V1 → T1_V2 → T1_V3 → T1_V4
# Epoch length        : 2 s
# Epochs per phase    : 9
# Duration per phase  : 18 s
#
# IMPORTANT
# ---------
# - No spectral/connectivity outcome is inspected.
# - Marker labels remain semantically neutral.
# - The FIRST 9 CLEAN phase-contained epochs are selected.
# - No optimization based on EEG outcome.
# - Exact frozen Task3A preprocessed FIFs are used.
#
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import mne

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK3A = (
    BASE
    / "task03a_batch_preprocessing_qc"
)

TASK3B = (
    BASE
    / "task03b_phase_reconstruction"
)

TASK8A = (
    BASE
    / "task08a_active_marker_phase_audit"
)

OUT = (
    BASE
    / "task08b_frozen_first_cycle_epochs"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. FROZEN DESIGN
# =============================================================================

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 9)
]

PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]

N_EPOCHS = 9

EPOCH_SEC = 2.0

EXPECTED_TOTAL = (
    len(SUBJECTS)
    *
    len(PHASES)
    *
    N_EPOCHS
)


print("=" * 100)
print("TASK 8B — FROZEN FIRST-CYCLE BALANCED EPOCH CONSTRUCTION")
print("=" * 100)

print(f"Participants       : {len(SUBJECTS)}")
print(f"Marker phases      : {len(PHASES)}")
print(f"Epochs per phase   : {N_EPOCHS}")
print(f"Seconds per phase  : {N_EPOCHS * EPOCH_SEC:.1f}")
print(f"Expected epochs    : {EXPECTED_TOTAL}")


# =============================================================================
# 3. LOAD TASK 3B TABLES
# =============================================================================

PHASE_FILE = (
    TASK3B
    / "all_subjects_marker_defined_phases.csv"
)

EPOCH_FILE = (
    TASK3B
    / "all_subjects_phase_epochs.csv"
)


if not PHASE_FILE.exists():
    raise FileNotFoundError(PHASE_FILE)

if not EPOCH_FILE.exists():
    raise FileNotFoundError(EPOCH_FILE)


phases = pd.read_csv(
    PHASE_FILE
)

epochs = pd.read_csv(
    EPOCH_FILE
)


print("\nTask3B tables:")
print(f"  Phase rows : {len(phases)}")
print(f"  Epoch rows : {len(epochs)}")


# =============================================================================
# 4. STANDARDIZE SUBJECT
# =============================================================================

def standardize_subject(x):

    text = str(x).strip()

    digits = "".join(
        c for c in text
        if c.isdigit()
    )

    if not digits:
        return text

    return f"sub-{int(digits):03d}"


phases["subject"] = (
    phases["subject"]
    .map(standardize_subject)
)

epochs["subject"] = (
    epochs["subject"]
    .map(standardize_subject)
)


# =============================================================================
# 5. RESTRICT TO ACTIVE T1 MARKERS
# =============================================================================

active_phases = phases.loc[
    phases["subject"].isin(SUBJECTS)
    &
    phases["marker_code"].isin(PHASES)
].copy()


active_phases = active_phases.sort_values(
    [
        "subject",
        "marker_position"
    ]
).reset_index(drop=True)


# =============================================================================
# 6. FIND FIRST COMPLETE V1→V2→V3→V4 CYCLE
# =============================================================================

cycle_rows = []


for subject in SUBJECTS:

    df = (
        active_phases.loc[
            active_phases["subject"] == subject
        ]
        .sort_values("marker_position")
        .reset_index(drop=True)
    )

    codes = df["marker_code"].tolist()

    found = False

    for i in range(
        max(0, len(codes) - 3)
    ):

        if codes[i:i + 4] == PHASES:

            block = df.iloc[
                i:i + 4
            ].copy()

            block["cycle_order"] = np.arange(
                1,
                5
            )

            cycle_rows.append(
                block
            )

            found = True

            break

    if not found:

        raise RuntimeError(
            f"No complete first cycle found for {subject}"
        )


first_cycle = pd.concat(
    cycle_rows,
    ignore_index=True
)


# =============================================================================
# 7. VERIFY FIRST CYCLE
# =============================================================================

cycle_check = (
    first_cycle
    .groupby("subject")
    .agg(
        n_phases=("marker_code", "size"),
        n_codes=("marker_code", "nunique"),
        first_position=("marker_position", "min"),
        last_position=("marker_position", "max")
    )
)


print("\nFirst-cycle structural check:")
print(cycle_check)


for subject in SUBJECTS:

    temp = (
        first_cycle.loc[
            first_cycle["subject"] == subject
        ]
        .sort_values("cycle_order")
    )

    observed = temp[
        "marker_code"
    ].tolist()

    if observed != PHASES:

        raise RuntimeError(
            f"{subject}: incorrect first-cycle order: {observed}"
        )


# =============================================================================
# 8. STANDARDIZE CLEAN FLAG
# =============================================================================

clean_raw = epochs["clean"]


if pd.api.types.is_bool_dtype(clean_raw):

    epochs["is_clean"] = clean_raw.astype(bool)

elif pd.api.types.is_numeric_dtype(clean_raw):

    epochs["is_clean"] = (
        pd.to_numeric(
            clean_raw,
            errors="coerce"
        )
        ==
        1
    )

else:

    epochs["is_clean"] = (
        clean_raw
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes",
                "y",
                "clean",
                "accepted",
                "good",
                "keep",
                "retained"
            ]
        )
    )


# =============================================================================
# 9. SELECT FIRST 9 CLEAN EPOCHS FROM EACH FROZEN PHASE
# =============================================================================

selected_rows = []

availability_rows = []


for _, phase_row in first_cycle.iterrows():

    subject = phase_row["subject"]

    marker = phase_row["marker_code"]

    position = phase_row["marker_position"]


    available = (
        epochs.loc[
            (epochs["subject"] == subject)
            &
            (epochs["marker_code"] == marker)
            &
            (epochs["marker_position"] == position)
            &
            (epochs["is_clean"])
        ]
        .sort_values(
            [
                "epoch_start_sample",
                "local_epoch"
            ]
        )
        .copy()
    )


    n_available = len(
        available
    )


    availability_rows.append(
        {
            "subject": subject,
            "marker_code": marker,
            "marker_position": position,
            "clean_epochs_available": n_available,
            "required_epochs": N_EPOCHS,
            "eligible": n_available >= N_EPOCHS
        }
    )


    if n_available < N_EPOCHS:

        raise RuntimeError(
            f"{subject} {marker} position {position}: "
            f"only {n_available} clean epochs; "
            f"{N_EPOCHS} required."
        )


    selected = available.iloc[
        :N_EPOCHS
    ].copy()


    selected[
        "selected_epoch"
    ] = np.arange(
        1,
        N_EPOCHS + 1
    )


    selected_rows.append(
        selected
    )


availability = pd.DataFrame(
    availability_rows
)


selected_epochs = pd.concat(
    selected_rows,
    ignore_index=True
)


# =============================================================================
# 10. AVAILABILITY REPORT
# =============================================================================

print("\n" + "=" * 100)
print("CLEAN-EPOCH AVAILABILITY FOR FROZEN FIRST CYCLE")
print("=" * 100)

print(
    availability.to_string(
        index=False
    )
)


print(
    "\nMinimum clean epochs available:",
    int(
        availability[
            "clean_epochs_available"
        ].min()
    )
)


# =============================================================================
# 11. LOCATE EXACT TASK3A FIF FILES
# =============================================================================

def task3a_fif(subject):

    path = (
        TASK3A
        / subject
        / f"{subject}_task-activeai_desc-primary_annotated_eeg_raw.fif"
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Exact Task3A FIF not found:\n{path}"
        )

    return path


for subject in SUBJECTS:

    path = task3a_fif(
        subject
    )

    print(
        f"{subject}: {path.name}"
    )


# =============================================================================
# 12. EXTRACT FROZEN EEG EPOCHS
# =============================================================================

all_epoch_data = []

metadata_rows = []

reference_ch_names = None

reference_sfreq = None


for subject in SUBJECTS:

    print(
        "\n"
        +
        "-" * 100
    )

    print(
        f"Extracting {subject}"
    )

    raw_path = task3a_fif(
        subject
    )


    raw = mne.io.read_raw_fif(
        raw_path,
        preload=False,
        verbose="ERROR"
    )


    raw.pick(
        "eeg"
    )


    ch_names = list(
        raw.ch_names
    )

    sfreq = float(
        raw.info[
            "sfreq"
        ]
    )


    if reference_ch_names is None:

        reference_ch_names = ch_names

    elif ch_names != reference_ch_names:

        raise RuntimeError(
            f"{subject}: EEG channel order differs."
        )


    if reference_sfreq is None:

        reference_sfreq = sfreq

    elif not np.isclose(
        sfreq,
        reference_sfreq
    ):

        raise RuntimeError(
            f"{subject}: sampling frequency differs."
        )


    expected_samples = int(
        round(
            EPOCH_SEC
            *
            sfreq
        )
    )


    subject_selection = (
        selected_epochs.loc[
            selected_epochs[
                "subject"
            ]
            ==
            subject
        ]
        .sort_values(
            [
                "marker_position",
                "selected_epoch"
            ]
        )
    )


    subject_count = 0


    for _, row in subject_selection.iterrows():

        start = int(
            row[
                "epoch_start_sample"
            ]
        )

        stop = int(
            row[
                "epoch_end_sample"
            ]
        )


        # Task3B end sample may be stored as inclusive or exclusive.
        # We determine the extraction using the known 2-s epoch size.

        if (
            stop
            -
            start
        ) == expected_samples:

            extraction_stop = stop

        elif (
            stop
            -
            start
            +
            1
        ) == expected_samples:

            extraction_stop = stop + 1

        else:

            extraction_stop = (
                start
                +
                expected_samples
            )


        if extraction_stop > raw.n_times:

            raise RuntimeError(
                f"{subject}: epoch extends beyond raw data."
            )


        data = raw.get_data(
            start=start,
            stop=extraction_stop
        )


        if data.shape != (
            len(reference_ch_names),
            expected_samples
        ):

            raise RuntimeError(
                f"{subject}: unexpected epoch shape "
                f"{data.shape}; expected "
                f"({len(reference_ch_names)}, "
                f"{expected_samples})."
            )


        if not np.isfinite(
            data
        ).all():

            raise RuntimeError(
                f"{subject}: non-finite EEG data."
            )


        all_epoch_data.append(
            data
        )


        metadata_rows.append(
            {
                "subject":
                    subject,

                "marker_code":
                    row[
                        "marker_code"
                    ],

                "marker_position":
                    int(
                        row[
                            "marker_position"
                        ]
                    ),

                "selected_epoch":
                    int(
                        row[
                            "selected_epoch"
                        ]
                    ),

                "local_epoch":
                    int(
                        row[
                            "local_epoch"
                        ]
                    ),

                "epoch_start_sample":
                    start,

                "epoch_end_sample":
                    stop,

                "epoch_start_sec":
                    float(
                        row[
                            "epoch_start_sec"
                        ]
                    ),

                "epoch_end_sec":
                    float(
                        row[
                            "epoch_end_sec"
                        ]
                    )
            }
        )


        subject_count += 1


    expected_subject_count = (
        len(PHASES)
        *
        N_EPOCHS
    )


    if subject_count != expected_subject_count:

        raise RuntimeError(
            f"{subject}: extracted "
            f"{subject_count}, expected "
            f"{expected_subject_count}."
        )


    print(
        f"  Extracted epochs : {subject_count}"
    )


# =============================================================================
# 13. STACK DATA
# =============================================================================

X = np.stack(
    all_epoch_data,
    axis=0
)


metadata = pd.DataFrame(
    metadata_rows
)


print(
    "\nEEG array shape:",
    X.shape
)


expected_shape = (
    EXPECTED_TOTAL,
    len(reference_ch_names),
    int(
        round(
            EPOCH_SEC
            *
            reference_sfreq
        )
    )
)


if X.shape != expected_shape:

    raise RuntimeError(
        f"Unexpected final shape: "
        f"{X.shape}; expected {expected_shape}"
    )


# =============================================================================
# 14. CREATE MNE EPOCHSARRAY
# =============================================================================

info = mne.create_info(
    ch_names=reference_ch_names,
    sfreq=reference_sfreq,
    ch_types="eeg"
)


epochs_frozen = mne.EpochsArray(
    X,
    info,
    tmin=0.0,
    metadata=metadata,
    verbose="ERROR"
)


# =============================================================================
# 15. SAVE FROZEN EPOCHS
# =============================================================================

EPOCHS_FIF = (
    OUT
    /
    "active_first_cycle_fixed9_epo.fif"
)


epochs_frozen.save(
    EPOCHS_FIF,
    overwrite=True,
    verbose="ERROR"
)


# =============================================================================
# 16. SAVE METADATA / STRUCTURAL TABLES
# =============================================================================

selected_epochs.to_csv(
    OUT
    /
    "selected_first_cycle_epochs.csv",
    index=False
)


availability.to_csv(
    OUT
    /
    "first_cycle_epoch_availability.csv",
    index=False
)


first_cycle.to_csv(
    OUT
    /
    "frozen_first_cycle_phase_definitions.csv",
    index=False
)


metadata.to_csv(
    OUT
    /
    "frozen_epoch_metadata.csv",
    index=False
)


# =============================================================================
# 17. BALANCE CHECKS
# =============================================================================

subject_phase_counts = (
    metadata
    .groupby(
        [
            "subject",
            "marker_code"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=SUBJECTS,
        columns=PHASES
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "FROZEN EPOCH COUNTS"
)

print(
    "=" * 100
)

print(
    subject_phase_counts
)


# =============================================================================
# 18. TEMPORAL ORDER CHECK
# =============================================================================

order_rows = []


for subject in SUBJECTS:

    temp = (
        first_cycle.loc[
            first_cycle[
                "subject"
            ]
            ==
            subject
        ]
        .sort_values(
            "cycle_order"
        )
    )


    positions = temp[
        "marker_position"
    ].to_numpy(
        dtype=int
    )


    codes = temp[
        "marker_code"
    ].tolist()


    order_rows.append(
        {
            "subject":
                subject,

            "codes_correct":
                codes
                ==
                PHASES,

            "positions_strictly_increasing":
                bool(
                    np.all(
                        np.diff(
                            positions
                        )
                        >
                        0
                    )
                ),

            "positions":
                ",".join(
                    map(
                        str,
                        positions
                    )
                )
        }
    )


order_check = pd.DataFrame(
    order_rows
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "FIRST-CYCLE TEMPORAL ORDER"
)

print(
    "=" * 100
)

print(
    order_check.to_string(
        index=False
    )
)


order_check.to_csv(
    OUT
    /
    "first_cycle_order_check.csv",
    index=False
)


# =============================================================================
# 19. SAMPLE / VALUE QC
# =============================================================================

peak_abs_uv = (
    np.max(
        np.abs(
            X
        ),
        axis=(
            1,
            2
        )
    )
    *
    1e6
)


ptp_uv = (
    np.max(
        np.ptp(
            X,
            axis=2
        ),
        axis=1
    )
    *
    1e6
)


metadata[
    "max_abs_uV_recomputed"
] = peak_abs_uv


metadata[
    "max_ptp_uV_recomputed"
] = ptp_uv


metadata.to_csv(
    OUT
    /
    "frozen_epoch_metadata_with_qc.csv",
    index=False
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "RECOMPUTED EEG QC"
)

print(
    "=" * 100
)


print(
    f"Maximum absolute amplitude : "
    f"{peak_abs_uv.max():.3f} µV"
)

print(
    f"Maximum peak-to-peak       : "
    f"{ptp_uv.max():.3f} µV"
)


# =============================================================================
# 20. FINAL INTEGRITY CHECK
# =============================================================================

integrity = {

    "participants_8":
        metadata[
            "subject"
        ].nunique()
        ==
        8,

    "four_marker_codes":
        set(
            metadata[
                "marker_code"
            ].unique()
        )
        ==
        set(
            PHASES
        ),

    "exact_9_each":
        bool(
            (
                subject_phase_counts
                ==
                N_EPOCHS
            )
            .all()
            .all()
        ),

    "total_288_epochs":
        len(
            metadata
        )
        ==
        EXPECTED_TOTAL,

    "array_shape_correct":
        X.shape
        ==
        expected_shape,

    "finite_eeg":
        bool(
            np.isfinite(
                X
            ).all()
        ),

    "cycle_codes_correct":
        bool(
            order_check[
                "codes_correct"
            ].all()
        ),

    "cycle_order_increasing":
        bool(
            order_check[
                "positions_strictly_increasing"
            ].all()
        ),

    "minimum_clean_availability":
        bool(
            (
                availability[
                    "clean_epochs_available"
                ]
                >=
                N_EPOCHS
            ).all()
        ),

    "amplitude_threshold_preserved":
        bool(
            peak_abs_uv.max()
            <=
            250.0
            +
            1e-6
        ),

    "ptp_threshold_preserved":
        bool(
            ptp_uv.max()
            <=
            500.0
            +
            1e-6
        )
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 100
)

print(
    "TASK 8B INTEGRITY CHECK"
)

print(
    "=" * 100
)


for key, value in integrity.items():

    print(
        f"{key:<35}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 100
)

print(
    "OVERALL:",
    (
        "PASS"
        if overall_pass
        else
        "FAIL"
    )
)

print(
    "-" * 100
)


if not overall_pass:

    raise RuntimeError(
        "Task 8B integrity check failed."
    )


# =============================================================================
# 21. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 100
)

print(
    "TASK 8B COMPLETE — DESIGN FROZEN"
)

print(
    "=" * 100
)


print(
    f"""
Participants        : 8
Marker-defined phases: 4
Epochs per phase     : 9
Seconds per phase    : 18
Total epochs         : {len(metadata)}
Channels             : {len(reference_ch_names)}
Sampling frequency   : {reference_sfreq:.1f} Hz

Frozen sequence:
    T1_V1 → T1_V2 → T1_V3 → T1_V4

Selection:
    First complete structural cycle
    First 9 clean epochs within each phase
    No EEG outcome used for selection

Semantic interpretation:
    NONE assigned.
    T1_V1–T1_V4 remain neutral marker-defined phases.
"""
)


print(
    "Saved frozen MNE epochs:"
)

print(
    EPOCHS_FIF
)


print(
    "\nOutput directory:"
)

print(
    OUT
)


print(
    "\nNEXT ANALYSIS:"
)

print(
    "Task 8C — within-Active spectral dynamics across the four "
    "frozen marker-defined phases."
)

print(
    "=" * 100
)

In [ ]:
# =============================================================================
# TASK 8C — WITHIN-ACTIVE SPECTRAL DYNAMICS ACROSS THE FROZEN FIRST CYCLE
# COMPLETE CORRECTED SELF-CONTAINED VERSION
# =============================================================================
#
# INPUT
# -----
# Task 8B:
#   active_first_cycle_fixed9_epo.fif
#
# FROZEN DESIGN
# -------------
# Participants : 8 Active participants
# Phases       : T1_V1, T1_V2, T1_V3, T1_V4
# Epochs       : exactly 9 clean 2-s epochs per participant per phase
# Duration     : 18 s per participant per phase
#
# SPECTRAL FEATURES
# -----------------
# Relative band power:
#   Theta : 4–8 Hz
#   Alpha : 8–13 Hz
#   Beta  : 13–30 Hz
#
# ROIs
# ----
# Frontal:
#   AF3, F7, F3, F4, F8, AF4
#
# CentralTemporal:
#   FC5, T7, T8, FC6
#
# Posterior:
#   P7, O1, O2, P8
#
# PRIMARY INFERENCE
# -----------------
# Participant is the inferential unit.
#
# For each band × ROI:
#   1. Friedman repeated-measures omnibus test across T1_V1–T1_V4
#   2. Kendall's W
#   3. BH-FDR across all 9 omnibus tests
#
# PAIRWISE FOLLOW-UP
# ------------------
#   - 6 paired phase contrasts per band × ROI
#   - exact paired sign-flip permutation test
#   - N=8 => 2^8 = 256 exact allocations
#   - paired standardized effect size dz
#   - 10,000 participant-level bootstrap samples
#   - BH-FDR across the 6 contrasts within each band × ROI
#
# INTERPRETATION
# --------------
# T1_V1–T1_V4 remain neutral marker-defined phases.
# No cognitive or experimental semantics are assigned.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import combinations, product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy import stats
from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK8B = (
    BASE
    / "task08b_frozen_first_cycle_epochs"
)

EPOCHS_FILE = (
    TASK8B
    / "active_first_cycle_fixed9_epo.fif"
)

OUT = (
    BASE
    / "task08c_within_active_spectral_dynamics"
)

FIG_OUT = (
    OUT
    / "figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)


if not EPOCHS_FILE.exists():

    raise FileNotFoundError(
        f"Frozen Task 8B epochs not found:\n{EPOCHS_FILE}"
    )


# =============================================================================
# 2. FROZEN DESIGN
# =============================================================================

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 9)
]


PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]


BANDS = {

    "Theta": (
        4.0,
        8.0
    ),

    "Alpha": (
        8.0,
        13.0
    ),

    "Beta": (
        13.0,
        30.0
    )
}


ROIS = {

    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "CentralTemporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}


N_EXPECTED_EPOCHS = 9

N_BOOT = 10000

RNG_SEED = 20260913


# =============================================================================
# 3. STATISTICAL HELPERS
# =============================================================================

def bh_fdr(p_values):
    """
    Benjamini-Hochberg FDR correction.

    Parameters
    ----------
    p_values : array-like
        Raw p-values.

    Returns
    -------
    adjusted : np.ndarray
        BH-adjusted p-values in original order.
    """

    p = np.asarray(
        p_values,
        dtype=float
    )

    adjusted = np.full(
        p.shape,
        np.nan,
        dtype=float
    )

    valid = np.isfinite(
        p
    )

    if not valid.any():
        return adjusted

    pv = p[
        valid
    ]

    m = len(
        pv
    )

    order = np.argsort(
        pv
    )

    ranked = pv[
        order
    ]

    q = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )

    q = np.minimum.accumulate(
        q[::-1]
    )[::-1]

    q = np.clip(
        q,
        0.0,
        1.0
    )

    restored = np.empty(
        m,
        dtype=float
    )

    restored[
        order
    ] = q

    adjusted[
        valid
    ] = restored

    return adjusted


def paired_dz(x, y):
    """
    Paired standardized mean difference.

        dz = mean(x-y) / SD(x-y)
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    d = x - y

    sd = np.std(
        d,
        ddof=1
    )

    if (
        not np.isfinite(sd)
        or
        np.isclose(
            sd,
            0.0
        )
    ):
        return np.nan

    return float(
        np.mean(
            d
        )
        /
        sd
    )


def exact_paired_permutation(x, y):
    """
    Exact two-sided paired sign-flip permutation test.

    Statistic:
        absolute mean paired difference

    For N=8:
        2^8 = 256 exact sign allocations.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    d = x - y

    if not np.isfinite(
        d
    ).all():

        raise ValueError(
            "Non-finite paired differences encountered."
        )

    n = len(
        d
    )

    observed = abs(
        float(
            np.mean(
                d
            )
        )
    )

    permutation_statistics = []

    for signs in product(
        [-1.0, 1.0],
        repeat=n
    ):

        signs = np.asarray(
            signs,
            dtype=float
        )

        statistic = abs(
            float(
                np.mean(
                    d
                    *
                    signs
                )
            )
        )

        permutation_statistics.append(
            statistic
        )

    permutation_statistics = np.asarray(
        permutation_statistics,
        dtype=float
    )

    p_value = np.mean(
        permutation_statistics
        >=
        (
            observed
            -
            1e-15
        )
    )

    return float(
        p_value
    )


def bootstrap_paired_mean_ci(
    x,
    y,
    n_boot=10000,
    seed=20260913
):
    """
    Participant-level percentile bootstrap CI
    for the paired mean difference x-y.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    d = x - y

    if not np.isfinite(
        d
    ).all():

        raise ValueError(
            "Non-finite values in bootstrap input."
        )

    rng = np.random.default_rng(
        seed
    )

    n = len(
        d
    )

    indices = rng.integers(
        0,
        n,
        size=(
            n_boot,
            n
        )
    )

    boot_means = (
        d[
            indices
        ]
        .mean(
            axis=1
        )
    )

    low, high = np.percentile(
        boot_means,
        [
            2.5,
            97.5
        ]
    )

    return (
        float(
            low
        ),
        float(
            high
        )
    )


def kendalls_w_friedman(
    friedman_statistic,
    n_subjects,
    n_conditions
):
    """
    Kendall's W derived from Friedman chi-square.

        W = chi-square / [N × (k - 1)]
    """

    denominator = (
        n_subjects
        *
        (
            n_conditions
            -
            1
        )
    )

    if denominator <= 0:
        return np.nan

    value = (
        float(
            friedman_statistic
        )
        /
        denominator
    )

    return float(
        value
    )


# =============================================================================
# 4. LOAD FROZEN TASK 8B EPOCHS
# =============================================================================

print(
    "=" * 110
)

print(
    "TASK 8C — WITHIN-ACTIVE SPECTRAL DYNAMICS"
)

print(
    "=" * 110
)


epochs = mne.read_epochs(
    EPOCHS_FILE,
    preload=True,
    verbose="ERROR"
)


if epochs.metadata is None:

    raise RuntimeError(
        "Task 8B frozen epochs contain no metadata."
    )


metadata = epochs.metadata.copy()


print(
    f"Input file          : {EPOCHS_FILE}"
)

print(
    f"Epochs              : {len(epochs)}"
)

print(
    f"Channels            : {len(epochs.ch_names)}"
)

print(
    f"Sampling frequency  : {epochs.info['sfreq']:.1f} Hz"
)

print(
    f"Samples / epoch     : {len(epochs.times)}"
)

print(
    f"Approx. duration    : "
    f"{len(epochs.times) / epochs.info['sfreq']:.3f} s"
)


# =============================================================================
# 5. VERIFY METADATA
# =============================================================================

required_metadata = {
    "subject",
    "marker_code"
}


missing_metadata = (
    required_metadata
    -
    set(
        metadata.columns
    )
)


if missing_metadata:

    raise RuntimeError(
        "Missing metadata columns: "
        +
        ", ".join(
            sorted(
                missing_metadata
            )
        )
    )


metadata[
    "subject"
] = metadata[
    "subject"
].astype(str)


metadata[
    "marker_code"
] = metadata[
    "marker_code"
].astype(str)


observed_subjects = sorted(
    metadata[
        "subject"
    ].unique()
)


observed_phases = sorted(
    metadata[
        "marker_code"
    ].unique()
)


if observed_subjects != SUBJECTS:

    raise RuntimeError(
        f"Participant mismatch.\n"
        f"Observed: {observed_subjects}\n"
        f"Expected: {SUBJECTS}"
    )


if set(
    observed_phases
) != set(
    PHASES
):

    raise RuntimeError(
        f"Phase mismatch.\n"
        f"Observed: {observed_phases}\n"
        f"Expected: {PHASES}"
    )


# =============================================================================
# 6. VERIFY FROZEN BALANCE
# =============================================================================

counts = (
    metadata
    .groupby(
        [
            "subject",
            "marker_code"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=SUBJECTS,
        columns=PHASES,
        fill_value=0
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "FROZEN PARTICIPANT × PHASE COUNTS"
)

print(
    "=" * 110
)


display(
    counts
)


if not (
    counts
    ==
    N_EXPECTED_EPOCHS
).all().all():

    raise RuntimeError(
        "Task 8B balance check failed: "
        "expected exactly 9 epochs per participant × phase."
    )


if len(
    epochs
) != 288:

    raise RuntimeError(
        f"Expected 288 epochs; found {len(epochs)}."
    )


# =============================================================================
# 7. VERIFY EEG CHANNELS AND ROI DEFINITIONS
# =============================================================================

for roi_name, roi_channels in ROIS.items():

    missing = [
        ch
        for ch in roi_channels
        if ch not in epochs.ch_names
    ]

    if missing:

        raise RuntimeError(
            f"{roi_name}: missing channels: {missing}"
        )


all_roi_channels = [
    channel
    for channels in ROIS.values()
    for channel in channels
]


if len(
    all_roi_channels
) != 14:

    raise RuntimeError(
        "ROI definitions do not contain exactly 14 channels."
    )


if len(
    set(
        all_roi_channels
    )
) != 14:

    raise RuntimeError(
        "ROI definitions contain duplicate channels."
    )


if set(
    all_roi_channels
) != set(
    epochs.ch_names
):

    raise RuntimeError(
        "ROI channels do not exactly match frozen EEG channels.\n"
        f"Epoch channels: {epochs.ch_names}\n"
        f"ROI channels: {all_roi_channels}"
    )


print(
    "\nROI definitions:"
)


for roi_name, roi_channels in ROIS.items():

    print(
        f"  {roi_name:<20}: "
        +
        ", ".join(
            roi_channels
        )
    )


# =============================================================================
# 8. COMPUTE WELCH PSD
# =============================================================================
#
# Relative power denominator:
#     total 4–30 Hz power
#
# Band boundaries:
#     Theta = [4,8)
#     Alpha = [8,13)
#     Beta  = [13,30]
#
# This avoids overlapping frequency bins.
# =============================================================================

sfreq = float(
    epochs.info[
        "sfreq"
    ]
)


n_times = len(
    epochs.times
)


n_fft = min(
    512,
    n_times
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "WELCH SPECTRAL ESTIMATION"
)

print(
    "=" * 110
)


spectrum = epochs.compute_psd(
    method="welch",
    fmin=4.0,
    fmax=30.0,
    n_fft=n_fft,
    n_per_seg=n_fft,
    n_overlap=0,
    average="mean",
    verbose="ERROR"
)


psd = spectrum.get_data()

freqs = spectrum.freqs


print(
    f"PSD shape            : {psd.shape}"
)

print(
    f"Frequency range      : "
    f"{freqs.min():.3f}–{freqs.max():.3f} Hz"
)

print(
    f"Frequency resolution : "
    f"{np.mean(np.diff(freqs)):.6f} Hz"
)


if psd.shape[
    0
] != 288:

    raise RuntimeError(
        "PSD epoch count mismatch."
    )


if psd.shape[
    1
] != 14:

    raise RuntimeError(
        "PSD channel count mismatch."
    )


if not np.isfinite(
    psd
).all():

    raise RuntimeError(
        "PSD contains non-finite values."
    )


if (
    psd
    <
    0
).any():

    raise RuntimeError(
        "PSD contains negative values."
    )


# =============================================================================
# 9. COMPUTE RELATIVE BAND POWER
# =============================================================================

total_mask = (
    (freqs >= 4.0)
    &
    (freqs <= 30.0)
)


total_power = np.trapezoid(
    psd[
        :,
        :,
        total_mask
    ],
    freqs[
        total_mask
    ],
    axis=-1
)


if not np.isfinite(
    total_power
).all():

    raise RuntimeError(
        "Total spectral power contains non-finite values."
    )


if (
    total_power
    <=
    0
).any():

    raise RuntimeError(
        "Non-positive total 4–30 Hz power encountered."
    )


band_relative_power = {}


for band_name, (
    fmin,
    fmax
) in BANDS.items():

    if band_name in [
        "Theta",
        "Alpha"
    ]:

        mask = (
            (freqs >= fmin)
            &
            (freqs < fmax)
        )

    else:

        mask = (
            (freqs >= fmin)
            &
            (freqs <= fmax)
        )


    if mask.sum() < 2:

        raise RuntimeError(
            f"Insufficient frequency bins for {band_name}."
        )


    absolute_band_power = np.trapezoid(
        psd[
            :,
            :,
            mask
        ],
        freqs[
            mask
        ],
        axis=-1
    )


    relative_band_power = (
        absolute_band_power
        /
        total_power
    )


    if not np.isfinite(
        relative_band_power
    ).all():

        raise RuntimeError(
            f"{band_name} relative power contains "
            "non-finite values."
        )


    if (
        relative_band_power
        <
        0
    ).any():

        raise RuntimeError(
            f"{band_name} relative power contains "
            "negative values."
        )


    if (
        relative_band_power
        >
        (
            1.0
            +
            1e-12
        )
    ).any():

        raise RuntimeError(
            f"{band_name} relative power exceeds 1."
        )


    band_relative_power[
        band_name
    ] = relative_band_power


print(
    "\nRelative band-power arrays:"
)


for band_name, values in band_relative_power.items():

    print(
        f"  {band_name:<8}: "
        f"{values.shape}"
    )


# =============================================================================
# 10. EPOCH-LEVEL ROI FEATURES
# =============================================================================

channel_index = {
    channel: idx
    for idx, channel in enumerate(
        epochs.ch_names
    )
}


epoch_rows = []


for epoch_idx in range(
    len(
        epochs
    )
):

    subject = metadata.iloc[
        epoch_idx
    ][
        "subject"
    ]


    marker_code = metadata.iloc[
        epoch_idx
    ][
        "marker_code"
    ]


    for band_name in BANDS:

        channel_values = band_relative_power[
            band_name
        ][
            epoch_idx
        ]


        for roi_name, roi_channels in ROIS.items():

            roi_indices = [
                channel_index[
                    channel
                ]
                for channel in roi_channels
            ]


            roi_value = float(
                np.mean(
                    channel_values[
                        roi_indices
                    ]
                )
            )


            epoch_rows.append(
                {
                    "subject":
                        subject,

                    "marker_code":
                        marker_code,

                    "band":
                        band_name,

                    "roi":
                        roi_name,

                    "relative_power":
                        roi_value
                }
            )


epoch_roi = pd.DataFrame(
    epoch_rows
)


expected_epoch_feature_rows = (
    288
    *
    3
    *
    3
)


if len(
    epoch_roi
) != expected_epoch_feature_rows:

    raise RuntimeError(
        f"Expected {expected_epoch_feature_rows} "
        f"epoch-level feature rows; "
        f"found {len(epoch_roi)}."
    )


print(
    "\nEpoch-level ROI feature rows:",
    len(
        epoch_roi
    )
)


# =============================================================================
# 11. PARTICIPANT-LEVEL AVERAGING
# =============================================================================
#
# IMPORTANT:
# The nine epochs are averaged WITHIN participant × phase.
#
# The participant is the inferential unit.
# =============================================================================

participant_roi = (
    epoch_roi
    .groupby(
        [
            "subject",
            "marker_code",
            "band",
            "roi"
        ],
        as_index=False
    )
    .agg(
        relative_power=(
            "relative_power",
            "mean"
        )
    )
)


expected_participant_rows = (
    8
    *
    4
    *
    3
    *
    3
)


if len(
    participant_roi
) != expected_participant_rows:

    raise RuntimeError(
        f"Expected {expected_participant_rows} "
        f"participant-level feature rows; "
        f"found {len(participant_roi)}."
    )


if not np.isfinite(
    participant_roi[
        "relative_power"
    ].to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Participant-level spectral features contain "
        "non-finite values."
    )


print(
    "Participant-level feature rows:",
    len(
        participant_roi
    )
)


# =============================================================================
# 12. DESCRIPTIVE SUMMARY
# =============================================================================

descriptive = (
    participant_roi
    .groupby(
        [
            "band",
            "roi",
            "marker_code"
        ],
        as_index=False
    )
    .agg(

        n=(
            "relative_power",
            "size"
        ),

        mean=(
            "relative_power",
            "mean"
        ),

        sd=(
            "relative_power",
            "std"
        ),

        median=(
            "relative_power",
            "median"
        ),

        minimum=(
            "relative_power",
            "min"
        ),

        maximum=(
            "relative_power",
            "max"
        )
    )
)


descriptive[
    "sem"
] = (
    descriptive[
        "sd"
    ]
    /
    np.sqrt(
        descriptive[
            "n"
        ]
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PARTICIPANT-LEVEL DESCRIPTIVE SUMMARY"
)

print(
    "=" * 110
)


display(
    descriptive.round(
        6
    )
)


# =============================================================================
# 13. FRIEDMAN REPEATED-MEASURES OMNIBUS TESTS
# =============================================================================
#
# CORRECTED VERSION:
#
# scipy.stats.friedmanchisquare() result is explicitly stored and
# .statistic / .pvalue are extracted. This avoids the previous
# friedman_stat NameError.
# =============================================================================

omnibus_rows = []


for band_name in BANDS:

    for roi_name in ROIS:

        temp = participant_roi.loc[
            (
                participant_roi[
                    "band"
                ]
                ==
                band_name
            )
            &
            (
                participant_roi[
                    "roi"
                ]
                ==
                roi_name
            )
        ].copy()


        wide = (
            temp
            .pivot(
                index="subject",
                columns="marker_code",
                values="relative_power"
            )
            .reindex(
                index=SUBJECTS,
                columns=PHASES
            )
        )


        expected_shape = (
            len(
                SUBJECTS
            ),
            len(
                PHASES
            )
        )


        if wide.shape != expected_shape:

            raise RuntimeError(
                f"{band_name}/{roi_name}: "
                f"unexpected repeated-measures shape "
                f"{wide.shape}; expected {expected_shape}."
            )


        if wide.isna().any().any():

            raise RuntimeError(
                f"{band_name}/{roi_name}: "
                "missing repeated-measures value."
            )


        wide_values = wide.to_numpy(
            dtype=float
        )


        if not np.isfinite(
            wide_values
        ).all():

            raise RuntimeError(
                f"{band_name}/{roi_name}: "
                "non-finite repeated-measures value."
            )


        phase_arrays = [
            wide[
                phase
            ].to_numpy(
                dtype=float
            )
            for phase in PHASES
        ]


        # ---------------------------------------------------------------------
        # Correct Friedman implementation
        # ---------------------------------------------------------------------

        friedman_result = stats.friedmanchisquare(
            *phase_arrays
        )


        friedman_chi2 = float(
            friedman_result.statistic
        )


        friedman_p = float(
            friedman_result.pvalue
        )


        kendall_w = kendalls_w_friedman(
            friedman_statistic=friedman_chi2,
            n_subjects=len(
                SUBJECTS
            ),
            n_conditions=len(
                PHASES
            )
        )


        phase_means = {
            phase: float(
                wide[
                    phase
                ].mean()
            )
            for phase in PHASES
        }


        phase_medians = {
            phase: float(
                wide[
                    phase
                ].median()
            )
            for phase in PHASES
        }


        omnibus_rows.append(
            {
                "band":
                    band_name,

                "roi":
                    roi_name,

                "n_subjects":
                    len(
                        SUBJECTS
                    ),

                "n_phases":
                    len(
                        PHASES
                    ),

                "friedman_chi2":
                    friedman_chi2,

                "df":
                    len(
                        PHASES
                    )
                    -
                    1,

                "p_raw":
                    friedman_p,

                "kendalls_W":
                    kendall_w,

                "mean_T1_V1":
                    phase_means[
                        "T1_V1"
                    ],

                "mean_T1_V2":
                    phase_means[
                        "T1_V2"
                    ],

                "mean_T1_V3":
                    phase_means[
                        "T1_V3"
                    ],

                "mean_T1_V4":
                    phase_means[
                        "T1_V4"
                    ],

                "median_T1_V1":
                    phase_medians[
                        "T1_V1"
                    ],

                "median_T1_V2":
                    phase_medians[
                        "T1_V2"
                    ],

                "median_T1_V3":
                    phase_medians[
                        "T1_V3"
                    ],

                "median_T1_V4":
                    phase_medians[
                        "T1_V4"
                    ]
            }
        )


omnibus = pd.DataFrame(
    omnibus_rows
)


if len(
    omnibus
) != 9:

    raise RuntimeError(
        f"Expected 9 omnibus tests; found {len(omnibus)}."
    )


# =============================================================================
# 14. BH-FDR ACROSS THE 9 OMNIBUS TESTS
# =============================================================================

omnibus[
    "p_FDR_9"
] = bh_fdr(
    omnibus[
        "p_raw"
    ].to_numpy(
        dtype=float
    )
)


omnibus[
    "significant_raw_0.05"
] = (
    omnibus[
        "p_raw"
    ]
    <
    0.05
)


omnibus[
    "significant_FDR_0.05"
] = (
    omnibus[
        "p_FDR_9"
    ]
    <
    0.05
)


if not np.isfinite(
    omnibus[
        [
            "friedman_chi2",
            "p_raw",
            "kendalls_W",
            "p_FDR_9"
        ]
    ].to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Non-finite omnibus statistic detected."
    )


if not omnibus[
    "p_raw"
].between(
    0,
    1
).all():

    raise RuntimeError(
        "Invalid raw Friedman p-value."
    )


if not omnibus[
    "p_FDR_9"
].between(
    0,
    1
).all():

    raise RuntimeError(
        "Invalid FDR-adjusted p-value."
    )


if not omnibus[
    "kendalls_W"
].between(
    0,
    1
).all():

    raise RuntimeError(
        "Kendall's W outside [0,1]."
    )


omnibus = (
    omnibus
    .sort_values(
        [
            "p_FDR_9",
            "p_raw",
            "band",
            "roi"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "FRIEDMAN REPEATED-MEASURES OMNIBUS TESTS"
)

print(
    "N = 8 participants; 4 marker-defined phases"
)

print(
    "BH-FDR family = 9 band × ROI tests"
)

print(
    "=" * 110
)


display(
    omnibus[
        [
            "band",
            "roi",
            "n_subjects",
            "friedman_chi2",
            "df",
            "p_raw",
            "p_FDR_9",
            "kendalls_W",
            "mean_T1_V1",
            "mean_T1_V2",
            "mean_T1_V3",
            "mean_T1_V4",
            "significant_raw_0.05",
            "significant_FDR_0.05"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 15. OMNIBUS SUMMARY
# =============================================================================

n_raw_omnibus = int(
    omnibus[
        "significant_raw_0.05"
    ].sum()
)


n_fdr_omnibus = int(
    omnibus[
        "significant_FDR_0.05"
    ].sum()
)


largest_effect_row = (
    omnibus
    .sort_values(
        "kendalls_W",
        ascending=False
    )
    .iloc[0]
)


print(
    "\nOmnibus summary:"
)

print(
    f"  Tests performed        : {len(omnibus)}"
)

print(
    f"  Raw p < .05            : "
    f"{n_raw_omnibus}/{len(omnibus)}"
)

print(
    f"  FDR p < .05            : "
    f"{n_fdr_omnibus}/{len(omnibus)}"
)

print(
    f"  Largest Kendall's W    : "
    f"{largest_effect_row['kendalls_W']:.4f}"
)

print(
    f"  Largest-effect feature : "
    f"{largest_effect_row['band']} / "
    f"{largest_effect_row['roi']}"
)

print(
    f"  Raw p                  : "
    f"{largest_effect_row['p_raw']:.6f}"
)

print(
    f"  FDR p                  : "
    f"{largest_effect_row['p_FDR_9']:.6f}"
)


# =============================================================================
# 16. EXACT PAIRWISE PHASE CONTRASTS
# =============================================================================

phase_pairs = list(
    combinations(
        PHASES,
        2
    )
)


if len(
    phase_pairs
) != 6:

    raise RuntimeError(
        "Expected exactly six phase-pair contrasts."
    )


pairwise_rows = []

seed_counter = 0


for band_name in BANDS:

    for roi_name in ROIS:

        temp = participant_roi.loc[
            (
                participant_roi[
                    "band"
                ]
                ==
                band_name
            )
            &
            (
                participant_roi[
                    "roi"
                ]
                ==
                roi_name
            )
        ]


        wide = (
            temp
            .pivot(
                index="subject",
                columns="marker_code",
                values="relative_power"
            )
            .reindex(
                index=SUBJECTS,
                columns=PHASES
            )
        )


        if wide.isna().any().any():

            raise RuntimeError(
                f"Missing pairwise data: "
                f"{band_name}/{roi_name}"
            )


        for phase_a, phase_b in phase_pairs:

            x = wide[
                phase_a
            ].to_numpy(
                dtype=float
            )

            y = wide[
                phase_b
            ].to_numpy(
                dtype=float
            )


            difference = (
                x
                -
                y
            )


            mean_difference = float(
                np.mean(
                    difference
                )
            )


            median_difference = float(
                np.median(
                    difference
                )
            )


            dz = paired_dz(
                x,
                y
            )


            p_exact = exact_paired_permutation(
                x,
                y
            )


            seed_counter += 1


            ci_low, ci_high = bootstrap_paired_mean_ci(
                x,
                y,
                n_boot=N_BOOT,
                seed=(
                    RNG_SEED
                    +
                    seed_counter
                )
            )


            n_positive = int(
                np.sum(
                    difference
                    >
                    0
                )
            )


            n_negative = int(
                np.sum(
                    difference
                    <
                    0
                )
            )


            n_zero = int(
                np.sum(
                    np.isclose(
                        difference,
                        0.0
                    )
                )
            )


            pairwise_rows.append(
                {
                    "band":
                        band_name,

                    "roi":
                        roi_name,

                    "phase_A":
                        phase_a,

                    "phase_B":
                        phase_b,

                    "contrast":
                        (
                            f"{phase_a} - {phase_b}"
                        ),

                    "n_subjects":
                        len(
                            SUBJECTS
                        ),

                    "mean_A":
                        float(
                            np.mean(
                                x
                            )
                        ),

                    "mean_B":
                        float(
                            np.mean(
                                y
                            )
                        ),

                    "mean_difference_A_minus_B":
                        mean_difference,

                    "median_difference":
                        median_difference,

                    "bootstrap_CI_low":
                        ci_low,

                    "bootstrap_CI_high":
                        ci_high,

                    "paired_dz":
                        dz,

                    "p_exact":
                        p_exact,

                    "subjects_A_gt_B":
                        n_positive,

                    "subjects_A_lt_B":
                        n_negative,

                    "subjects_equal":
                        n_zero
                }
            )


pairwise = pd.DataFrame(
    pairwise_rows
)


if len(
    pairwise
) != 54:

    raise RuntimeError(
        f"Expected 54 pairwise tests; "
        f"found {len(pairwise)}."
    )


# =============================================================================
# 17. PAIRWISE BH-FDR
# =============================================================================
#
# Six pairwise phase contrasts are corrected WITHIN each band × ROI.
#
# This is a follow-up family and does not replace the primary
# 9-test omnibus FDR family.
# =============================================================================

pairwise[
    "p_FDR_6"
] = np.nan


grouped_indices = pairwise.groupby(
    [
        "band",
        "roi"
    ]
).groups


for group_key, indices in grouped_indices.items():

    indices = list(
        indices
    )


    raw_p = pairwise.loc[
        indices,
        "p_exact"
    ].to_numpy(
        dtype=float
    )


    adjusted_p = bh_fdr(
        raw_p
    )


    pairwise.loc[
        indices,
        "p_FDR_6"
    ] = adjusted_p


pairwise[
    "significant_raw_0.05"
] = (
    pairwise[
        "p_exact"
    ]
    <
    0.05
)


pairwise[
    "significant_FDR_0.05"
] = (
    pairwise[
        "p_FDR_6"
    ]
    <
    0.05
)


if not pairwise[
    "p_exact"
].between(
    0,
    1
).all():

    raise RuntimeError(
        "Invalid exact pairwise p-value."
    )


if not pairwise[
    "p_FDR_6"
].between(
    0,
    1
).all():

    raise RuntimeError(
        "Invalid pairwise FDR-adjusted p-value."
    )


pairwise = (
    pairwise
    .sort_values(
        [
            "band",
            "roi",
            "p_FDR_6",
            "p_exact",
            "contrast"
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "EXACT PAIRED PHASE CONTRASTS"
)

print(
    "Exact sign-flip permutations = 256 allocations per contrast"
)

print(
    "BH-FDR family = six contrasts within each band × ROI"
)

print(
    "=" * 110
)


display(
    pairwise[
        [
            "band",
            "roi",
            "contrast",
            "mean_A",
            "mean_B",
            "mean_difference_A_minus_B",
            "bootstrap_CI_low",
            "bootstrap_CI_high",
            "paired_dz",
            "p_exact",
            "p_FDR_6",
            "subjects_A_gt_B",
            "subjects_A_lt_B",
            "subjects_equal",
            "significant_FDR_0.05"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 18. RANK OMNIBUS EFFECTS
# =============================================================================

ranked_omnibus = (
    omnibus
    .sort_values(
        [
            "kendalls_W",
            "p_raw"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "RANKED OMNIBUS EFFECTS BY KENDALL'S W"
)

print(
    "=" * 110
)


display(
    ranked_omnibus[
        [
            "band",
            "roi",
            "friedman_chi2",
            "p_raw",
            "p_FDR_9",
            "kendalls_W",
            "significant_raw_0.05",
            "significant_FDR_0.05"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 19. RANK PAIRWISE EFFECTS BY |dz|
# =============================================================================

ranked_pairwise = pairwise.copy()


ranked_pairwise[
    "abs_dz"
] = ranked_pairwise[
    "paired_dz"
].abs()


ranked_pairwise = (
    ranked_pairwise
    .sort_values(
        [
            "abs_dz",
            "p_exact"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "LARGEST PAIRWISE DESCRIPTIVE EFFECTS"
)

print(
    "=" * 110
)


display(
    ranked_pairwise[
        [
            "band",
            "roi",
            "contrast",
            "mean_difference_A_minus_B",
            "bootstrap_CI_low",
            "bootstrap_CI_high",
            "paired_dz",
            "p_exact",
            "p_FDR_6",
            "subjects_A_gt_B",
            "subjects_A_lt_B"
        ]
    ].head(
        15
    ).round(
        6
    )
)


# =============================================================================
# 20. PARTICIPANT TRAJECTORY FIGURES
# =============================================================================
#
# One separate figure per band.
#
# Each figure shows the three ROIs.
# Individual lines = participants.
# Thick error-bar line = group mean ± SEM.
#
# =============================================================================

trajectory_files = []


for band_name in BANDS:

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(
            15,
            5
        ),
        sharex=True
    )


    for ax, roi_name in zip(
        axes,
        ROIS.keys()
    ):

        temp = participant_roi.loc[
            (
                participant_roi[
                    "band"
                ]
                ==
                band_name
            )
            &
            (
                participant_roi[
                    "roi"
                ]
                ==
                roi_name
            )
        ]


        wide = (
            temp
            .pivot(
                index="subject",
                columns="marker_code",
                values="relative_power"
            )
            .reindex(
                index=SUBJECTS,
                columns=PHASES
            )
        )


        x_positions = np.arange(
            len(
                PHASES
            )
        )


        # ---------------------------------------------------------------------
        # Individual participant trajectories
        # ---------------------------------------------------------------------

        for subject in SUBJECTS:

            values = wide.loc[
                subject
            ].to_numpy(
                dtype=float
            )


            ax.plot(
                x_positions,
                values,
                marker="o",
                alpha=0.30,
                linewidth=1
            )


        # ---------------------------------------------------------------------
        # Group mean ± SEM
        # ---------------------------------------------------------------------

        means = wide.mean(
            axis=0
        ).to_numpy(
            dtype=float
        )


        sems = wide.sem(
            axis=0
        ).to_numpy(
            dtype=float
        )


        ax.errorbar(
            x_positions,
            means,
            yerr=sems,
            marker="o",
            linewidth=2,
            capsize=4
        )


        ax.set_xticks(
            x_positions
        )


        ax.set_xticklabels(
            PHASES
        )


        ax.set_title(
            roi_name
        )


        ax.set_xlabel(
            "Marker-defined phase"
        )


    axes[
        0
    ].set_ylabel(
        f"Relative {band_name.lower()} power"
    )


    fig.suptitle(
        f"{band_name} power across the frozen first marker cycle"
    )


    fig.tight_layout()


    figure_path = (
        FIG_OUT
        /
        (
            f"{band_name.lower()}"
            "_participant_phase_trajectories.png"
        )
    )


    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )


    trajectory_files.append(
        figure_path
    )


    plt.show()


# =============================================================================
# 21. OMNIBUS EFFECT-SIZE FIGURE
# =============================================================================

effect_plot = omnibus.copy()


effect_plot[
    "label"
] = (
    effect_plot[
        "band"
    ]
    +
    " — "
    +
    effect_plot[
        "roi"
    ]
)


effect_plot = effect_plot.sort_values(
    "kendalls_W",
    ascending=True
)


fig, ax = plt.subplots(
    figsize=(
        9,
        6
    )
)


ax.barh(
    effect_plot[
        "label"
    ],
    effect_plot[
        "kendalls_W"
    ]
)


ax.set_xlabel(
    "Kendall's W"
)


ax.set_ylabel(
    ""
)


ax.set_title(
    "Within-Active marker-phase spectral effect sizes"
)


fig.tight_layout()


EFFECT_FIG = (
    FIG_OUT
    /
    "omnibus_kendalls_w.png"
)


fig.savefig(
    EFFECT_FIG,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 22. GROUP-MEAN PHASE PROFILE FIGURES
# =============================================================================
#
# Publication-oriented descriptive figures.
# One figure per band.
# =============================================================================

profile_files = []


for band_name in BANDS:

    fig, ax = plt.subplots(
        figsize=(
            9,
            6
        )
    )


    x_positions = np.arange(
        len(
            PHASES
        )
    )


    for roi_name in ROIS:

        temp = participant_roi.loc[
            (
                participant_roi[
                    "band"
                ]
                ==
                band_name
            )
            &
            (
                participant_roi[
                    "roi"
                ]
                ==
                roi_name
            )
        ]


        wide = (
            temp
            .pivot(
                index="subject",
                columns="marker_code",
                values="relative_power"
            )
            .reindex(
                index=SUBJECTS,
                columns=PHASES
            )
        )


        means = wide.mean(
            axis=0
        ).to_numpy(
            dtype=float
        )


        sems = wide.sem(
            axis=0
        ).to_numpy(
            dtype=float
        )


        ax.errorbar(
            x_positions,
            means,
            yerr=sems,
            marker="o",
            linewidth=2,
            capsize=4,
            label=roi_name
        )


    ax.set_xticks(
        x_positions
    )


    ax.set_xticklabels(
        PHASES
    )


    ax.set_xlabel(
        "Marker-defined phase"
    )


    ax.set_ylabel(
        f"Relative {band_name.lower()} power"
    )


    ax.set_title(
        f"{band_name} spectral dynamics across the first marker cycle"
    )


    ax.legend(
        frameon=False
    )


    fig.tight_layout()


    profile_path = (
        FIG_OUT
        /
        (
            f"{band_name.lower()}"
            "_group_mean_phase_profile.png"
        )
    )


    fig.savefig(
        profile_path,
        dpi=300,
        bbox_inches="tight"
    )


    profile_files.append(
        profile_path
    )


    plt.show()


# =============================================================================
# 23. SAVE ALL TABLES
# =============================================================================

EPOCH_TABLE = (
    OUT
    /
    "epoch_level_relative_band_power.csv"
)


PARTICIPANT_TABLE = (
    OUT
    /
    "participant_phase_relative_band_power.csv"
)


DESCRIPTIVE_TABLE = (
    OUT
    /
    "phase_descriptive_summary.csv"
)


OMNIBUS_TABLE = (
    OUT
    /
    "friedman_omnibus_inference.csv"
)


PAIRWISE_TABLE = (
    OUT
    /
    "exact_pairwise_phase_inference.csv"
)


RANKED_OMNIBUS_TABLE = (
    OUT
    /
    "ranked_omnibus_effects.csv"
)


RANKED_PAIRWISE_TABLE = (
    OUT
    /
    "ranked_pairwise_effects.csv"
)


epoch_roi.to_csv(
    EPOCH_TABLE,
    index=False
)


participant_roi.to_csv(
    PARTICIPANT_TABLE,
    index=False
)


descriptive.to_csv(
    DESCRIPTIVE_TABLE,
    index=False
)


omnibus.to_csv(
    OMNIBUS_TABLE,
    index=False
)


pairwise.to_csv(
    PAIRWISE_TABLE,
    index=False
)


ranked_omnibus.to_csv(
    RANKED_OMNIBUS_TABLE,
    index=False
)


ranked_pairwise.to_csv(
    RANKED_PAIRWISE_TABLE,
    index=False
)


# =============================================================================
# 24. FINAL STATISTICAL COUNTS
# =============================================================================

n_raw_pairwise = int(
    pairwise[
        "significant_raw_0.05"
    ].sum()
)


n_fdr_pairwise = int(
    pairwise[
        "significant_FDR_0.05"
    ].sum()
)


# =============================================================================
# 25. INTEGRITY CHECK
# =============================================================================

integrity = {

    "input_epochs_288":
        len(
            epochs
        )
        ==
        288,

    "participants_8":
        metadata[
            "subject"
        ].nunique()
        ==
        8,

    "phases_4":
        set(
            metadata[
                "marker_code"
            ].unique()
        )
        ==
        set(
            PHASES
        ),

    "exact9_epoch_balance":
        bool(
            (
                counts
                ==
                9
            )
            .all()
            .all()
        ),

    "channels_14":
        len(
            epochs.ch_names
        )
        ==
        14,

    "epoch_feature_rows_2592":
        len(
            epoch_roi
        )
        ==
        2592,

    "participant_feature_rows_288":
        len(
            participant_roi
        )
        ==
        288,

    "omnibus_tests_9":
        len(
            omnibus
        )
        ==
        9,

    "pairwise_tests_54":
        len(
            pairwise
        )
        ==
        54,

    "all_psd_finite":
        bool(
            np.isfinite(
                psd
            ).all()
        ),

    "participant_features_finite":
        bool(
            np.isfinite(
                participant_roi[
                    "relative_power"
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),

    "relative_power_nonnegative":
        bool(
            (
                participant_roi[
                    "relative_power"
                ]
                >=
                0
            ).all()
        ),

    "relative_power_le_1":
        bool(
            (
                participant_roi[
                    "relative_power"
                ]
                <=
                (
                    1
                    +
                    1e-12
                )
            ).all()
        ),

    "omnibus_p_valid":
        bool(
            omnibus[
                "p_raw"
            ].between(
                0,
                1
            ).all()
        ),

    "omnibus_fdr_valid":
        bool(
            omnibus[
                "p_FDR_9"
            ].between(
                0,
                1
            ).all()
        ),

    "kendalls_w_valid":
        bool(
            omnibus[
                "kendalls_W"
            ].between(
                0,
                1
            ).all()
        ),

    "pairwise_p_valid":
        bool(
            pairwise[
                "p_exact"
            ].between(
                0,
                1
            ).all()
        ),

    "pairwise_fdr_valid":
        bool(
            pairwise[
                "p_FDR_6"
            ].between(
                0,
                1
            ).all()
        )
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 8C INTEGRITY CHECK"
)

print(
    "=" * 110
)


for key, value in integrity.items():

    print(
        f"{key:<38}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 110
)

print(
    "OVERALL:",
    (
        "PASS"
        if overall_pass
        else
        "FAIL"
    )
)

print(
    "-" * 110
)


if not overall_pass:

    raise RuntimeError(
        "Task 8C integrity check failed."
    )


# =============================================================================
# 26. ANALYTICAL SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 8C ANALYTICAL SUMMARY"
)

print(
    "=" * 110
)


print(
    f"Raw-significant omnibus tests     : "
    f"{n_raw_omnibus}/9"
)


print(
    f"FDR-significant omnibus tests     : "
    f"{n_fdr_omnibus}/9"
)


print(
    f"Raw-significant pairwise tests    : "
    f"{n_raw_pairwise}/54"
)


print(
    f"FDR-significant pairwise tests    : "
    f"{n_fdr_pairwise}/54"
)


print(
    "\nLargest omnibus effect:"
)


print(
    f"  Band        : "
    f"{largest_effect_row['band']}"
)


print(
    f"  ROI         : "
    f"{largest_effect_row['roi']}"
)


print(
    f"  Kendall's W : "
    f"{largest_effect_row['kendalls_W']:.6f}"
)


print(
    f"  Friedman χ² : "
    f"{largest_effect_row['friedman_chi2']:.6f}"
)


print(
    f"  Raw p       : "
    f"{largest_effect_row['p_raw']:.6f}"
)


print(
    f"  FDR p       : "
    f"{largest_effect_row['p_FDR_9']:.6f}"
)


# =============================================================================
# 27. TOP FIVE PAIRWISE DESCRIPTIVE EFFECTS
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "TOP FIVE PAIRWISE EFFECTS BY |dz|"
)

print(
    "=" * 110
)


display(
    ranked_pairwise[
        [
            "band",
            "roi",
            "contrast",
            "mean_difference_A_minus_B",
            "bootstrap_CI_low",
            "bootstrap_CI_high",
            "paired_dz",
            "p_exact",
            "p_FDR_6",
            "subjects_A_gt_B",
            "subjects_A_lt_B"
        ]
    ].head(
        5
    ).round(
        6
    )
)


# =============================================================================
# 28. INTERPRETATION GUARDRAILS
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "INTERPRETATION GUARDRAILS"
)

print(
    "=" * 110
)


print(
    """
1. The participant is the inferential unit; the 288 EEG epochs are NOT
   treated as 288 independent observations.

2. T1_V1, T1_V2, T1_V3 and T1_V4 remain neutral marker-defined phases.

3. No cognitive meaning, AI-use state, writing subprocess, or learning
   state should be assigned to these marker codes without independent
   protocol documentation.

4. The primary multiplicity family is the nine Friedman omnibus tests.

5. Pairwise tests are secondary follow-up analyses. Their FDR correction
   is performed across the six phase contrasts within each band × ROI.

6. A raw pairwise p < .05 without corrected support is descriptive.

7. Large |dz| or Kendall's W without multiplicity-corrected significance
   should be described as an effect-size pattern, not a confirmed effect.

8. Because N=8, effect sizes, participant trajectories, exact permutation
   results, and multiplicity-corrected inference should be considered
   together.

9. Connectivity analysis should not be initiated merely because one
   exploratory pairwise spectral contrast is small.

10. Any subsequent within-Active connectivity analysis must continue to
    use the validated wPLI²_debiased estimator rather than the legacy
    Hilbert implementation.
"""
)


# =============================================================================
# 29. OUTPUT MANIFEST
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "OUTPUT FILES"
)

print(
    "=" * 110
)


print(
    f"Epoch-level features:\n"
    f"  {EPOCH_TABLE}"
)


print(
    f"\nParticipant-level features:\n"
    f"  {PARTICIPANT_TABLE}"
)


print(
    f"\nDescriptive summary:\n"
    f"  {DESCRIPTIVE_TABLE}"
)


print(
    f"\nOmnibus inference:\n"
    f"  {OMNIBUS_TABLE}"
)


print(
    f"\nPairwise inference:\n"
    f"  {PAIRWISE_TABLE}"
)


print(
    f"\nRanked omnibus effects:\n"
    f"  {RANKED_OMNIBUS_TABLE}"
)


print(
    f"\nRanked pairwise effects:\n"
    f"  {RANKED_PAIRWISE_TABLE}"
)


print(
    "\nFigures:"
)


for path in trajectory_files:

    print(
        f"  {path}"
    )


for path in profile_files:

    print(
        f"  {path}"
    )


print(
    f"  {EFFECT_FIG}"
)


# =============================================================================
# 30. COMPLETION
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 8C COMPLETE"
)

print(
    "=" * 110
)


print(
    f"""
Frozen input:
    {EPOCHS_FILE}

Design:
    N = 8 Active participants
    4 marker-defined phases
    9 clean epochs per phase
    2 seconds per epoch
    18 seconds per participant per phase
    288 frozen epochs total

Spectral analysis:
    Welch PSD
    Relative theta = 4–8 Hz
    Relative alpha = 8–13 Hz
    Relative beta  = 13–30 Hz
    Relative-power denominator = 4–30 Hz

ROIs:
    Frontal
    CentralTemporal
    Posterior

Primary inference:
    Friedman repeated-measures tests
    Kendall's W
    BH-FDR across 9 band × ROI tests

Pairwise follow-up:
    6 phase contrasts per feature
    Exact paired sign-flip permutation
    256 exact allocations
    Paired dz
    10,000 participant-level bootstrap samples
    BH-FDR across 6 contrasts within each feature

Semantic status:
    T1_V1–T1_V4 remain neutral marker-defined phases.

Output directory:
    {OUT}
"""
)


print(
    "NEXT DECISION:"
)


if n_fdr_omnibus > 0:

    print(
        "At least one omnibus phase effect survives FDR. "
        "Interpret the supported feature(s), pairwise contrasts, "
        "effect sizes, and participant trajectories before deciding "
        "whether a targeted within-Active connectivity analysis is justified."
    )

else:

    print(
        "No omnibus phase effect survives FDR. "
        "Interpret the effect-size structure and participant trajectories "
        "conservatively before deciding whether any connectivity follow-up "
        "has sufficient methodological justification."
    )


print(
    "=" * 110
)

In [ ]:
# =============================================================================
# TASK 8D — ACTIVE REPEATED-PHASE STRUCTURAL ROBUSTNESS AUDIT
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PURPOSE
# -------
# Determine what repeated marker-phase structure is genuinely available
# across ALL 8 Active participants before performing any second-cycle or
# repeated-phase EEG analysis.
#
# IMPORTANT
# ---------
# This is a DESIGN/QC audit only.
#
# NO EEG spectral outcome is calculated.
# NO connectivity outcome is calculated.
# NO participant is excluded to manufacture a repeated-cycle design.
#
# T1_V1–T1_V4 remain neutral marker-defined phases.
#
# QUESTIONS
# ---------
# 1. How many occurrences of T1_V1, V2, V3, V4 does each participant have?
# 2. How many complete consecutive V1→V2→V3→V4 cycles exist?
# 3. Which participants have 1, 2, ... complete cycles?
# 4. Is there a second occurrence of each individual marker in all 8 subjects?
# 5. Does each second occurrence have >= 9 clean 2-s epochs?
# 6. Is there any repeated design that preserves all 8 participants?
#
# CANDIDATE FOLLOW-UPS
# --------------------
# A. Second complete cycle across all 8
# B. Second occurrence of each marker across all 8
# C. Repeated V1-vs-V4 endpoint pair across all 8
# D. No universal repeated design -> do not force replication
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK3B = (
    BASE
    / "task03b_phase_reconstruction"
)

OUT = (
    BASE
    / "task08d_repeated_phase_structural_audit"
)

FIG_OUT = (
    OUT
    / "figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)


if not TASK3B.exists():

    raise FileNotFoundError(
        f"Task3B directory not found:\n{TASK3B}"
    )


# =============================================================================
# 2. FROZEN DEFINITIONS
# =============================================================================

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(
        1,
        9
    )
]

PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]

N_REQUIRED_EPOCHS = 9

EPOCH_SECONDS = 2.0


# =============================================================================
# 3. HELPERS
# =============================================================================

def normalize_subject(value):

    value = str(value).strip()

    digits = "".join(
        char
        for char in value
        if char.isdigit()
    )

    if digits:

        return f"sub-{int(digits):03d}"

    return value


def find_column(
    df,
    candidates,
    required=True
):

    lower_map = {
        str(col).lower(): col
        for col in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    if required:

        raise KeyError(
            "Could not find any of these columns:\n"
            +
            ", ".join(candidates)
            +
            "\n\nAvailable columns:\n"
            +
            ", ".join(
                map(str, df.columns)
            )
        )

    return None


def safe_int(value):

    try:

        if pd.isna(value):
            return np.nan

        return int(value)

    except Exception:

        return np.nan


# =============================================================================
# 4. DISCOVER TASK3B CSV FILES
# =============================================================================

csv_files = sorted(
    TASK3B.rglob("*.csv")
)

if not csv_files:

    raise FileNotFoundError(
        f"No CSV files found under:\n{TASK3B}"
    )


print(
    "=" * 110
)

print(
    "TASK 8D — ACTIVE REPEATED-PHASE STRUCTURAL ROBUSTNESS AUDIT"
)

print(
    "=" * 110
)

print(
    "\nTask3B CSV files discovered:"
)

for path in csv_files:

    print(
        f"  {path.name}"
    )


# =============================================================================
# 5. IDENTIFY PHASE AND EPOCH TABLES
# =============================================================================

phase_candidates = []

epoch_candidates = []


for path in csv_files:

    try:

        preview = pd.read_csv(
            path,
            nrows=10
        )

    except Exception:

        continue

    cols = {
        str(col).lower()
        for col in preview.columns
    }

    has_marker = (
        "marker_code" in cols
        or
        "code" in cols
    )

    has_position = (
        "marker_position" in cols
        or
        "position" in cols
    )

    has_epoch_start = any(
        col in cols
        for col in [
            "epoch_start_sec",
            "epoch_start",
            "epoch_index"
        ]
    )

    if (
        has_marker
        and
        has_position
    ):

        if has_epoch_start:

            epoch_candidates.append(
                path
            )

        else:

            phase_candidates.append(
                path
            )


# -----------------------------------------------------------------------------
# Filename fallback
# -----------------------------------------------------------------------------

if not phase_candidates:

    phase_candidates = [
        path
        for path in csv_files
        if (
            "phase" in path.name.lower()
            and
            "epoch" not in path.name.lower()
        )
    ]


if not epoch_candidates:

    epoch_candidates = [
        path
        for path in csv_files
        if "epoch" in path.name.lower()
    ]


if not phase_candidates:

    raise RuntimeError(
        "Could not identify Task3B phase table."
    )


if not epoch_candidates:

    raise RuntimeError(
        "Could not identify Task3B epoch table."
    )


def get_row_count(path):

    try:

        return len(
            pd.read_csv(path)
        )

    except Exception:

        return -1


PHASE_FILE = max(
    phase_candidates,
    key=get_row_count
)

EPOCH_FILE = max(
    epoch_candidates,
    key=get_row_count
)


print(
    f"\nSelected phase table:\n  {PHASE_FILE}"
)

print(
    f"\nSelected epoch table:\n  {EPOCH_FILE}"
)


phase_raw = pd.read_csv(
    PHASE_FILE
)

epoch_raw = pd.read_csv(
    EPOCH_FILE
)


print(
    f"\nPhase rows : {len(phase_raw)}"
)

print(
    f"Epoch rows : {len(epoch_raw)}"
)


# =============================================================================
# 6. STANDARDIZE PHASE TABLE
# =============================================================================

subject_col = find_column(
    phase_raw,
    [
        "subject",
        "participant",
        "participant_id"
    ]
)

code_col = find_column(
    phase_raw,
    [
        "marker_code",
        "code"
    ]
)

position_col = find_column(
    phase_raw,
    [
        "marker_position",
        "position"
    ]
)

start_col = find_column(
    phase_raw,
    [
        "start_sec",
        "phase_start_sec",
        "marker_time_sec",
        "start_time_sec",
        "start"
    ]
)

end_col = find_column(
    phase_raw,
    [
        "end_sec",
        "phase_end_sec",
        "end_time_sec",
        "end"
    ],
    required=False
)

duration_col = find_column(
    phase_raw,
    [
        "duration_sec",
        "phase_duration_sec",
        "duration"
    ],
    required=False
)


phase = pd.DataFrame(
    {
        "subject":
            phase_raw[
                subject_col
            ].map(
                normalize_subject
            ),

        "marker_code":
            phase_raw[
                code_col
            ].astype(str),

        "marker_position":
            pd.to_numeric(
                phase_raw[
                    position_col
                ],
                errors="coerce"
            ),

        "start_sec":
            pd.to_numeric(
                phase_raw[
                    start_col
                ],
                errors="coerce"
            )
    }
)


if end_col is not None:

    phase[
        "end_sec"
    ] = pd.to_numeric(
        phase_raw[
            end_col
        ],
        errors="coerce"
    )

elif duration_col is not None:

    duration = pd.to_numeric(
        phase_raw[
            duration_col
        ],
        errors="coerce"
    )

    phase[
        "end_sec"
    ] = (
        phase[
            "start_sec"
        ]
        +
        duration
    )

else:

    raise RuntimeError(
        "Could not determine phase end times."
    )


phase = phase.loc[
    phase[
        "subject"
    ].isin(
        SUBJECTS
    )
    &
    phase[
        "marker_code"
    ].isin(
        PHASES
    )
].copy()


phase = phase.sort_values(
    [
        "subject",
        "marker_position",
        "start_sec"
    ]
).reset_index(
    drop=True
)


# =============================================================================
# 7. STANDARDIZE EPOCH TABLE
# =============================================================================

epoch_subject_col = find_column(
    epoch_raw,
    [
        "subject",
        "participant",
        "participant_id"
    ]
)

epoch_code_col = find_column(
    epoch_raw,
    [
        "marker_code",
        "code"
    ]
)

epoch_position_col = find_column(
    epoch_raw,
    [
        "marker_position",
        "position"
    ]
)

epoch_start_col = find_column(
    epoch_raw,
    [
        "epoch_start_sec",
        "epoch_start",
        "start_sec",
        "start"
    ]
)

epoch_end_col = find_column(
    epoch_raw,
    [
        "epoch_end_sec",
        "epoch_end",
        "end_sec",
        "end"
    ],
    required=False
)

clean_col = find_column(
    epoch_raw,
    [
        "clean",
        "is_clean",
        "accepted",
        "keep",
        "good"
    ],
    required=False
)


epoch = pd.DataFrame(
    {
        "subject":
            epoch_raw[
                epoch_subject_col
            ].map(
                normalize_subject
            ),

        "marker_code":
            epoch_raw[
                epoch_code_col
            ].astype(str),

        "marker_position":
            pd.to_numeric(
                epoch_raw[
                    epoch_position_col
                ],
                errors="coerce"
            ),

        "epoch_start_sec":
            pd.to_numeric(
                epoch_raw[
                    epoch_start_col
                ],
                errors="coerce"
            )
    }
)


if epoch_end_col is not None:

    epoch[
        "epoch_end_sec"
    ] = pd.to_numeric(
        epoch_raw[
            epoch_end_col
        ],
        errors="coerce"
    )

else:

    epoch[
        "epoch_end_sec"
    ] = (
        epoch[
            "epoch_start_sec"
        ]
        +
        EPOCH_SECONDS
    )


# -----------------------------------------------------------------------------
# Clean flag
# -----------------------------------------------------------------------------

if clean_col is None:

    print(
        "\nWARNING: no explicit Task3B clean flag found."
    )

    print(
        "All Task3B epoch rows will be treated as available."
    )

    epoch[
        "clean"
    ] = True

else:

    raw_clean = epoch_raw[
        clean_col
    ]

    if pd.api.types.is_bool_dtype(
        raw_clean
    ):

        epoch[
            "clean"
        ] = raw_clean.astype(bool)

    elif pd.api.types.is_numeric_dtype(
        raw_clean
    ):

        epoch[
            "clean"
        ] = (
            pd.to_numeric(
                raw_clean,
                errors="coerce"
            )
            ==
            1
        )

    else:

        epoch[
            "clean"
        ] = (
            raw_clean
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(
                [
                    "true",
                    "1",
                    "yes",
                    "y",
                    "clean",
                    "accepted",
                    "good",
                    "keep",
                    "pass"
                ]
            )
        )


epoch = epoch.loc[
    epoch[
        "subject"
    ].isin(
        SUBJECTS
    )
    &
    epoch[
        "marker_code"
    ].isin(
        PHASES
    )
].copy()


epoch = epoch.sort_values(
    [
        "subject",
        "marker_position",
        "epoch_start_sec"
    ]
).reset_index(
    drop=True
)


# =============================================================================
# 8. ASSIGN WITHIN-SUBJECT OCCURRENCE NUMBER
# =============================================================================

phase[
    "occurrence"
] = (
    phase
    .groupby(
        [
            "subject",
            "marker_code"
        ]
    )
    .cumcount()
    +
    1
)


# =============================================================================
# 9. MARKER OCCURRENCE COUNTS
# =============================================================================

occurrence_counts = (
    phase
    .groupby(
        [
            "subject",
            "marker_code"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        index=SUBJECTS,
        columns=PHASES,
        fill_value=0
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "MARKER OCCURRENCE COUNTS"
)

print(
    "=" * 110
)


display(
    occurrence_counts
)


common_occurrence_depth = (
    occurrence_counts
    .min(
        axis=0
    )
    .rename(
        "minimum_occurrences_across_all_8"
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "COMMON OCCURRENCE DEPTH ACROSS ALL 8 ACTIVE PARTICIPANTS"
)

print(
    "=" * 110
)


display(
    common_occurrence_depth.to_frame()
)


# =============================================================================
# 10. COMPLETE CONSECUTIVE CYCLE AUDIT
# =============================================================================

cycle_records = []

sequence_records = []


for subject in SUBJECTS:

    sub = (
        phase.loc[
            phase[
                "subject"
            ]
            ==
            subject
        ]
        .sort_values(
            [
                "marker_position",
                "start_sec"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    sequence = sub[
        "marker_code"
    ].tolist()


    positions = sub[
        "marker_position"
    ].tolist()


    sequence_records.append(
        {
            "subject":
                subject,

            "sequence":
                " → ".join(
                    sequence
                ),

            "positions":
                ",".join(
                    str(
                        safe_int(value)
                    )
                    for value in positions
                )
        }
    )


    cycle_number = 0


    for i in range(
        max(
            0,
            len(sub) - 3
        )
    ):

        block = sub.iloc[
            i:i + 4
        ]


        codes = block[
            "marker_code"
        ].tolist()


        if codes != PHASES:

            continue


        pos = block[
            "marker_position"
        ].to_numpy(
            dtype=float
        )


        if not np.all(
            np.diff(pos) > 0
        ):

            continue


        cycle_number += 1


        cycle_records.append(
            {
                "subject":
                    subject,

                "cycle_number":
                    cycle_number,

                "start_index":
                    i,

                "V1_position":
                    int(
                        block.iloc[0][
                            "marker_position"
                        ]
                    ),

                "V2_position":
                    int(
                        block.iloc[1][
                            "marker_position"
                        ]
                    ),

                "V3_position":
                    int(
                        block.iloc[2][
                            "marker_position"
                        ]
                    ),

                "V4_position":
                    int(
                        block.iloc[3][
                            "marker_position"
                        ]
                    ),

                "cycle_start_sec":
                    float(
                        block.iloc[0][
                            "start_sec"
                        ]
                    ),

                "cycle_end_sec":
                    float(
                        block.iloc[3][
                            "end_sec"
                        ]
                    )
            }
        )


sequences = pd.DataFrame(
    sequence_records
)


cycles = pd.DataFrame(
    cycle_records
)


if len(cycles) > 0:

    complete_cycle_counts = (
        cycles
        .groupby(
            "subject"
        )
        .size()
        .reindex(
            SUBJECTS,
            fill_value=0
        )
        .rename(
            "complete_consecutive_cycles"
        )
        .to_frame()
    )

else:

    complete_cycle_counts = pd.DataFrame(
        {
            "complete_consecutive_cycles":
                [0] * len(SUBJECTS)
        },
        index=SUBJECTS
    )


print(
    "\n"
    +
    "=" * 110
)

print(
    "ACTIVE MARKER SEQUENCES"
)

print(
    "=" * 110
)


display(
    sequences
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "COMPLETE CONSECUTIVE V1→V2→V3→V4 CYCLE COUNTS"
)

print(
    "=" * 110
)


display(
    complete_cycle_counts
)


if len(cycles) > 0:

    print(
        "\nDetected complete cycles:"
    )

    display(
        cycles
    )


# =============================================================================
# 11. SECOND-OCCURRENCE STRUCTURAL TABLE
# =============================================================================

second_occurrence = phase.loc[
    phase[
        "occurrence"
    ]
    ==
    2
].copy()


second_structure = (
    second_occurrence
    .pivot_table(
        index="subject",
        columns="marker_code",
        values="marker_position",
        aggfunc="first"
    )
    .reindex(
        index=SUBJECTS,
        columns=PHASES
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "SECOND OCCURRENCE — MARKER POSITIONS"
)

print(
    "=" * 110
)


display(
    second_structure
)


# =============================================================================
# 12. CLEAN-EPOCH AVAILABILITY BY OCCURRENCE
# =============================================================================

availability_rows = []


for _, row in phase.iterrows():

    subject = row[
        "subject"
    ]

    code = row[
        "marker_code"
    ]

    position = row[
        "marker_position"
    ]

    occurrence = int(
        row[
            "occurrence"
        ]
    )

    start_sec = float(
        row[
            "start_sec"
        ]
    )

    end_sec = float(
        row[
            "end_sec"
        ]
    )


    candidates = epoch.loc[
        (
            epoch[
                "subject"
            ]
            ==
            subject
        )
        &
        (
            epoch[
                "marker_code"
            ]
            ==
            code
        )
        &
        (
            epoch[
                "marker_position"
            ]
            ==
            position
        )
        &
        (
            epoch[
                "clean"
            ]
        )
    ].copy()


    candidates = candidates.loc[
        (
            candidates[
                "epoch_start_sec"
            ]
            >=
            start_sec
        )
        &
        (
            candidates[
                "epoch_end_sec"
            ]
            <=
            (
                end_sec
                +
                1e-9
            )
        )
    ]


    availability_rows.append(
        {
            "subject":
                subject,

            "marker_code":
                code,

            "occurrence":
                occurrence,

            "marker_position":
                int(
                    position
                ),

            "phase_start_sec":
                start_sec,

            "phase_end_sec":
                end_sec,

            "phase_duration_sec":
                end_sec
                -
                start_sec,

            "clean_epochs_available":
                len(
                    candidates
                ),

            "eligible_fixed9":
                len(
                    candidates
                )
                >=
                N_REQUIRED_EPOCHS
        }
    )


availability = pd.DataFrame(
    availability_rows
)


# =============================================================================
# 13. SECOND-OCCURRENCE AVAILABILITY
# =============================================================================

second_availability = availability.loc[
    availability[
        "occurrence"
    ]
    ==
    2
].copy()


print(
    "\n"
    +
    "=" * 110
)

print(
    "SECOND-OCCURRENCE CLEAN-EPOCH AVAILABILITY"
)

print(
    "=" * 110
)


display(
    second_availability[
        [
            "subject",
            "marker_code",
            "marker_position",
            "phase_duration_sec",
            "clean_epochs_available",
            "eligible_fixed9"
        ]
    ]
)


# =============================================================================
# 14. SECOND-OCCURRENCE COMPLETENESS BY MARKER
# =============================================================================

second_marker_summary_rows = []


for code in PHASES:

    temp = second_availability.loc[
        second_availability[
            "marker_code"
        ]
        ==
        code
    ]


    subjects_present = set(
        temp[
            "subject"
        ]
    )


    n_present = len(
        subjects_present
    )


    n_eligible = int(
        temp.loc[
            temp[
                "eligible_fixed9"
            ],
            "subject"
        ].nunique()
    )


    if len(temp) > 0:

        minimum_clean = int(
            temp[
                "clean_epochs_available"
            ].min()
        )

    else:

        minimum_clean = 0


    second_marker_summary_rows.append(
        {
            "marker_code":
                code,

            "participants_with_second_occurrence":
                n_present,

            "participants_with_second_occurrence_and_9_clean":
                n_eligible,

            "minimum_clean_epochs":
                minimum_clean,

            "universal_second_occurrence":
                n_present
                ==
                8,

            "universal_fixed9_eligible":
                n_eligible
                ==
                8
        }
    )


second_marker_summary = pd.DataFrame(
    second_marker_summary_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "SECOND-OCCURRENCE COMPARABILITY ACROSS ALL 8"
)

print(
    "=" * 110
)


display(
    second_marker_summary
)


# =============================================================================
# 15. REPEATED V1-V4 ENDPOINT-PAIR AUDIT
# =============================================================================
#
# We do not require V2 and V3 to form a complete second cycle here.
# This asks whether every participant independently has:
#
#   second T1_V1
#   followed later by second T1_V4
#   both with >= 9 clean epochs.
#
# If true, a repeated endpoint-only replication may be possible without
# dropping participants.
# =============================================================================

endpoint_rows = []


for subject in SUBJECTS:

    sub_second = second_availability.loc[
        second_availability[
            "subject"
        ]
        ==
        subject
    ]


    v1 = sub_second.loc[
        sub_second[
            "marker_code"
        ]
        ==
        "T1_V1"
    ]


    v4 = sub_second.loc[
        sub_second[
            "marker_code"
        ]
        ==
        "T1_V4"
    ]


    has_v1 = len(v1) == 1

    has_v4 = len(v4) == 1


    if has_v1:

        v1_position = int(
            v1.iloc[0][
                "marker_position"
            ]
        )

        v1_clean = int(
            v1.iloc[0][
                "clean_epochs_available"
            ]
        )

    else:

        v1_position = np.nan
        v1_clean = 0


    if has_v4:

        v4_position = int(
            v4.iloc[0][
                "marker_position"
            ]
        )

        v4_clean = int(
            v4.iloc[0][
                "clean_epochs_available"
            ]
        )

    else:

        v4_position = np.nan
        v4_clean = 0


    ordered = bool(
        has_v1
        and
        has_v4
        and
        v1_position
        <
        v4_position
    )


    eligible = bool(
        ordered
        and
        v1_clean
        >=
        N_REQUIRED_EPOCHS
        and
        v4_clean
        >=
        N_REQUIRED_EPOCHS
    )


    endpoint_rows.append(
        {
            "subject":
                subject,

            "second_V1_present":
                has_v1,

            "second_V1_position":
                v1_position,

            "second_V1_clean_epochs":
                v1_clean,

            "second_V4_present":
                has_v4,

            "second_V4_position":
                v4_position,

            "second_V4_clean_epochs":
                v4_clean,

            "V1_before_V4":
                ordered,

            "fixed9_endpoint_pair_eligible":
                eligible
        }
    )


endpoint_audit = pd.DataFrame(
    endpoint_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "SECOND-OCCURRENCE V1→V4 ENDPOINT-PAIR AUDIT"
)

print(
    "=" * 110
)


display(
    endpoint_audit
)


# =============================================================================
# 16. FIRST-vs-SECOND OCCURRENCE TIME SEPARATION
# =============================================================================

time_rows = []


for subject in SUBJECTS:

    for code in PHASES:

        temp = (
            phase.loc[
                (
                    phase[
                        "subject"
                    ]
                    ==
                    subject
                )
                &
                (
                    phase[
                        "marker_code"
                    ]
                    ==
                    code
                )
            ]
            .sort_values(
                "occurrence"
            )
        )


        if len(temp) >= 2:

            first = temp.iloc[0]

            second = temp.iloc[1]


            time_rows.append(
                {
                    "subject":
                        subject,

                    "marker_code":
                        code,

                    "first_position":
                        int(
                            first[
                                "marker_position"
                            ]
                        ),

                    "second_position":
                        int(
                            second[
                                "marker_position"
                            ]
                        ),

                    "first_start_sec":
                        float(
                            first[
                                "start_sec"
                            ]
                        ),

                    "second_start_sec":
                        float(
                            second[
                                "start_sec"
                            ]
                        ),

                    "separation_sec":
                        float(
                            second[
                                "start_sec"
                            ]
                            -
                            first[
                                "start_sec"
                            ]
                        )
                }
            )


time_separation = pd.DataFrame(
    time_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "FIRST-TO-SECOND OCCURRENCE TEMPORAL SEPARATION"
)

print(
    "=" * 110
)


if len(time_separation) > 0:

    display(
        time_separation.round(
            3
        )
    )

else:

    print(
        "No repeated marker occurrences detected."
    )


# =============================================================================
# 17. DESIGN FEASIBILITY
# =============================================================================

all_have_two_complete_cycles = bool(
    (
        complete_cycle_counts[
            "complete_consecutive_cycles"
        ]
        >=
        2
    ).all()
)


all_markers_have_second = bool(
    second_marker_summary[
        "universal_second_occurrence"
    ].all()
)


all_markers_second_fixed9 = bool(
    second_marker_summary[
        "universal_fixed9_eligible"
    ].all()
)


all_have_second_endpoint_pair = bool(
    endpoint_audit[
        "fixed9_endpoint_pair_eligible"
    ].all()
)


n_with_two_complete_cycles = int(
    (
        complete_cycle_counts[
            "complete_consecutive_cycles"
        ]
        >=
        2
    ).sum()
)


n_endpoint_pair_eligible = int(
    endpoint_audit[
        "fixed9_endpoint_pair_eligible"
    ].sum()
)


design_rows = [
    {
        "candidate":
            "A",

        "design":
            "Second complete consecutive V1→V2→V3→V4 cycle",

        "preserves_all_8":
            all_have_two_complete_cycles,

        "n_participants":
            n_with_two_complete_cycles,

        "fixed9_supported":
            all_have_two_complete_cycles,

        "status":
            (
                "SUPPORTED"
                if all_have_two_complete_cycles
                else
                "NOT UNIVERSAL"
            )
    },

    {
        "candidate":
            "B",

        "design":
            "Second occurrence of all four markers independently",

        "preserves_all_8":
            (
                all_markers_have_second
                and
                all_markers_second_fixed9
            ),

        "n_participants":
            (
                8
                if (
                    all_markers_have_second
                    and
                    all_markers_second_fixed9
                )
                else
                np.nan
            ),

        "fixed9_supported":
            all_markers_second_fixed9,

        "status":
            (
                "SUPPORTED"
                if (
                    all_markers_have_second
                    and
                    all_markers_second_fixed9
                )
                else
                "NOT UNIVERSAL"
            )
    },

    {
        "candidate":
            "C",

        "design":
            "Second-occurrence V1→V4 endpoint pair only",

        "preserves_all_8":
            all_have_second_endpoint_pair,

        "n_participants":
            n_endpoint_pair_eligible,

        "fixed9_supported":
            all_have_second_endpoint_pair,

        "status":
            (
                "SUPPORTED"
                if all_have_second_endpoint_pair
                else
                "NOT UNIVERSAL"
            )
    }
]


designs = pd.DataFrame(
    design_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "REPEATED-DESIGN FEASIBILITY"
)

print(
    "=" * 110
)


display(
    designs
)


# =============================================================================
# 18. AUTOMATIC STRUCTURAL RECOMMENDATION
# =============================================================================
#
# Priority:
#
# A > B > C > D
#
# But unlike the previous Task8D, nothing is forced.
#
# A = ideal full-cycle replication
# B = all four repeated markers but not necessarily consecutive cycle
# C = endpoint-only replication of the Task8C V1-vs-V4 finding
# D = no all-participant repeated analysis
#
# =============================================================================

if all_have_two_complete_cycles:

    recommendation = (
        "A"
    )

    recommendation_text = (
        "A second complete consecutive V1→V2→V3→V4 cycle "
        "is available for all 8 participants."
    )


elif (
    all_markers_have_second
    and
    all_markers_second_fixed9
):

    recommendation = (
        "B"
    )

    recommendation_text = (
        "All four markers have a second fixed-9-eligible occurrence "
        "for all 8 participants, but a second complete consecutive "
        "cycle is not universal."
    )


elif all_have_second_endpoint_pair:

    recommendation = (
        "C"
    )

    recommendation_text = (
        "A second fixed-9-eligible T1_V1→T1_V4 endpoint pair "
        "is available for all 8 participants. This permits a narrowly "
        "prespecified replication of the Task8C theta endpoint contrast "
        "without excluding participants."
    )


else:

    recommendation = (
        "D"
    )

    recommendation_text = (
        "No repeated marker design preserves all 8 participants with "
        "the frozen fixed-9 requirement. Do not force a replication "
        "analysis by dropping participants."
    )


print(
    "\n"
    +
    "=" * 110
)

print(
    "FROZEN STRUCTURAL RECOMMENDATION"
)

print(
    "=" * 110
)


print(
    f"\nRecommendation: {recommendation}"
)

print(
    f"\n{recommendation_text}"
)


# =============================================================================
# 19. FIGURE — OCCURRENCE COUNTS
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        10,
        6
    )
)


occurrence_counts.plot(
    kind="bar",
    ax=ax
)


ax.set_xlabel(
    "Participant"
)

ax.set_ylabel(
    "Number of marker occurrences"
)

ax.set_title(
    "Active-group marker occurrence counts"
)

ax.legend(
    title="Marker",
    frameon=False
)

ax.tick_params(
    axis="x",
    rotation=45
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "active_marker_occurrence_counts.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 20. FIGURE — COMPLETE CYCLE COUNTS
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        9,
        5
    )
)


ax.bar(
    complete_cycle_counts.index,
    complete_cycle_counts[
        "complete_consecutive_cycles"
    ]
)


ax.axhline(
    2,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Participant"
)

ax.set_ylabel(
    "Complete consecutive cycles"
)

ax.set_title(
    "Complete T1_V1→T1_V2→T1_V3→T1_V4 cycles"
)

ax.tick_params(
    axis="x",
    rotation=45
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "complete_cycle_counts.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 21. SAVE OUTPUTS
# =============================================================================

phase.to_csv(
    OUT
    /
    "active_phase_occurrence_table.csv",
    index=False
)


occurrence_counts.to_csv(
    OUT
    /
    "marker_occurrence_counts.csv"
)


common_occurrence_depth.to_csv(
    OUT
    /
    "common_occurrence_depth.csv"
)


sequences.to_csv(
    OUT
    /
    "active_marker_sequences.csv",
    index=False
)


cycles.to_csv(
    OUT
    /
    "complete_consecutive_cycles.csv",
    index=False
)


complete_cycle_counts.to_csv(
    OUT
    /
    "complete_cycle_counts.csv"
)


availability.to_csv(
    OUT
    /
    "occurrence_clean_epoch_availability.csv",
    index=False
)


second_availability.to_csv(
    OUT
    /
    "second_occurrence_clean_epoch_availability.csv",
    index=False
)


second_marker_summary.to_csv(
    OUT
    /
    "second_occurrence_comparability.csv",
    index=False
)


endpoint_audit.to_csv(
    OUT
    /
    "second_occurrence_v1_v4_endpoint_audit.csv",
    index=False
)


time_separation.to_csv(
    OUT
    /
    "first_second_occurrence_time_separation.csv",
    index=False
)


designs.to_csv(
    OUT
    /
    "repeated_design_feasibility.csv",
    index=False
)


# =============================================================================
# 22. INTEGRITY CHECK
# =============================================================================

integrity = {

    "participants_8":
        phase[
            "subject"
        ].nunique()
        ==
        8,

    "four_marker_codes":
        set(
            phase[
                "marker_code"
            ].unique()
        )
        ==
        set(
            PHASES
        ),

    "occurrence_number_valid":
        bool(
            (
                phase[
                    "occurrence"
                ]
                >=
                1
            ).all()
        ),

    "positions_finite":
        bool(
            np.isfinite(
                phase[
                    "marker_position"
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),

    "start_times_finite":
        bool(
            np.isfinite(
                phase[
                    "start_sec"
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),

    "end_times_finite":
        bool(
            np.isfinite(
                phase[
                    "end_sec"
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),

    "positive_phase_duration":
        bool(
            (
                phase[
                    "end_sec"
                ]
                >
                phase[
                    "start_sec"
                ]
            ).all()
        ),

    "all_subjects_have_first_cycle":
        bool(
            (
                complete_cycle_counts[
                    "complete_consecutive_cycles"
                ]
                >=
                1
            ).all()
        ),

    "availability_rows_present":
        len(
            availability
        )
        ==
        len(
            phase
        ),

    "no_EEG_outcome_used":
        True
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 8D INTEGRITY CHECK"
)

print(
    "=" * 110
)


for key, value in integrity.items():

    print(
        f"{key:<42}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 110
)

print(
    "OVERALL:",
    (
        "PASS"
        if overall_pass
        else
        "FAIL"
    )
)

print(
    "-" * 110
)


if not overall_pass:

    raise RuntimeError(
        "Task 8D structural integrity check failed."
    )


# =============================================================================
# 23. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 8D COMPLETE — REPEATED STRUCTURE AUDITED"
)

print(
    "=" * 110
)


print(
    f"""
Participants:
    8 Active participants retained.

Marker phases:
    T1_V1
    T1_V2
    T1_V3
    T1_V4

Semantic interpretation:
    NONE assigned.

Second complete consecutive cycle:
    {n_with_two_complete_cycles}/8 participants

Second fixed-9 V1→V4 endpoint pair:
    {n_endpoint_pair_eligible}/8 participants

Frozen structural recommendation:
    {recommendation}

Reason:
    {recommendation_text}

No EEG spectral or connectivity outcome was used to choose
the repeated-analysis design.

Output directory:
    {OUT}
"""
)


print(
    "=" * 110
)

In [ ]:
# =============================================================================
# TASK 9A — TARGETED WITHIN-ACTIVE THETA CONNECTIVITY
#            T1_V1 vs T1_V4, FIRST FROZEN CYCLE
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PURPOSE
# -------
# Determine whether the first-cycle T1_V1 -> T1_V4 theta-power pattern
# identified in Task 8C is accompanied by a change in theta-band
# phase-lagged functional connectivity.
#
# DESIGN
# ------
# Participants:
#     8 Active participants
#
# Frozen phases:
#     T1_V1
#     T1_V4
#
# Source:
#     Task 8B frozen first-cycle epochs
#
# Epoch balance:
#     9 epochs per participant per phase
#     2 s per epoch
#     18 s per phase
#
# Connectivity estimator:
#     MNE-Connectivity spectral_connectivity_epochs
#     method = "wpli2_debiased"
#     mode = "multitaper"
#     theta = 4–8 Hz
#     mt_bandwidth = 2.0 Hz
#     mt_adaptive = False
#     mt_low_bias = True
#     faverage = True
#
# IMPORTANT
# ---------
# Correct terminology:
#     debiased squared weighted phase-lag index
#     wPLI²_debiased
#
# DO NOT:
#     - use the legacy Hilbert estimator
#     - clip negative wPLI²_debiased values
#     - perform 91 individual-edge significance tests
#     - search other frequency bands
#     - assign undocumented semantics to T1_V1 or T1_V4
#
# PRESPECIFIED CONNECTIVITY FEATURES
# ----------------------------------
# 1. Global mean across all 91 undirected edges
# 2. Frontal–Frontal (FF)
# 3. Frontal–Posterior (F-P)
# 4. Posterior–Posterior (P-P)
#
# Primary contrast:
#     T1_V4 - T1_V1
#
# Statistical inference:
#     exact paired sign-flip tests, N = 8
#     paired Cohen's dz
#     participant bootstrap 95% CI
#
# Multiplicity:
#     BH-FDR across the four prespecified connectivity summaries
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


try:
    from mne_connectivity import spectral_connectivity_epochs
except ImportError as exc:
    raise ImportError(
        "\nTask 9A requires mne-connectivity.\n"
        "Install it in the active environment with:\n\n"
        "    pip install mne-connectivity\n\n"
        "Then restart the kernel and rerun this cell."
    ) from exc


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK8B = (
    BASE
    / "task08b_frozen_first_cycle_epochs"
)

EPOCH_FILE = (
    TASK8B
    / "active_first_cycle_fixed9_epo.fif"
)

OUT = (
    BASE
    / "task09a_targeted_theta_connectivity"
)

FIG_OUT = (
    OUT
    / "figures"
)

MATRIX_OUT = (
    OUT
    / "matrices"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)

MATRIX_OUT.mkdir(
    parents=True,
    exist_ok=True
)


if not EPOCH_FILE.exists():

    raise FileNotFoundError(
        f"Task 8B frozen epochs not found:\n{EPOCH_FILE}"
    )


# =============================================================================
# 2. FROZEN PARAMETERS
# =============================================================================

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 9)
]


PHASES = [
    "T1_V1",
    "T1_V4"
]


EXPECTED_EPOCHS_PER_PHASE = 9

EXPECTED_CHANNELS = 14

EXPECTED_SFREQ = 256.0


THETA_MIN = 4.0

THETA_MAX = 8.0


MT_BANDWIDTH = 2.0


N_BOOT = 10000

RNG_SEED = 20260913


# =============================================================================
# 3. ROI DEFINITIONS
# =============================================================================

FRONTAL = [
    "AF3",
    "F7",
    "F3",
    "F4",
    "F8",
    "AF4"
]


CENTRAL_TEMPORAL = [
    "FC5",
    "T7",
    "T8",
    "FC6"
]


POSTERIOR = [
    "P7",
    "O1",
    "O2",
    "P8"
]


EXPECTED_CHANNEL_ORDER = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


# =============================================================================
# 4. STATISTICAL HELPERS
# =============================================================================

def bh_fdr(p_values):

    p = np.asarray(
        p_values,
        dtype=float
    )

    adjusted = np.full(
        p.shape,
        np.nan,
        dtype=float
    )

    valid = np.isfinite(p)

    if not valid.any():
        return adjusted

    pv = p[valid]

    m = len(pv)

    order = np.argsort(pv)

    ranked = pv[order]

    q = (
        ranked
        *
        m
        /
        np.arange(
            1,
            m + 1
        )
    )

    q = np.minimum.accumulate(
        q[::-1]
    )[::-1]

    q = np.clip(
        q,
        0.0,
        1.0
    )

    restored = np.empty(
        m,
        dtype=float
    )

    restored[order] = q

    adjusted[valid] = restored

    return adjusted


def paired_dz(x, y):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    d = x - y

    sd = np.std(
        d,
        ddof=1
    )

    if (
        not np.isfinite(sd)
        or
        np.isclose(sd, 0.0)
    ):
        return np.nan

    return float(
        np.mean(d)
        /
        sd
    )


def exact_signflip_test(differences):

    d = np.asarray(
        differences,
        dtype=float
    )

    if not np.isfinite(d).all():

        raise ValueError(
            "Exact sign-flip test received non-finite values."
        )


    observed = abs(
        float(
            np.mean(d)
        )
    )


    null_statistics = []


    for signs in product(
        [-1.0, 1.0],
        repeat=len(d)
    ):

        signs = np.asarray(
            signs,
            dtype=float
        )

        statistic = abs(
            float(
                np.mean(
                    d * signs
                )
            )
        )

        null_statistics.append(
            statistic
        )


    null_statistics = np.asarray(
        null_statistics,
        dtype=float
    )


    p_value = np.mean(
        null_statistics
        >=
        (
            observed
            -
            1e-15
        )
    )


    return float(p_value)


def bootstrap_mean_ci(
    values,
    n_boot=10000,
    seed=20260913
):

    values = np.asarray(
        values,
        dtype=float
    )

    if not np.isfinite(values).all():

        raise ValueError(
            "Bootstrap received non-finite values."
        )


    rng = np.random.default_rng(
        seed
    )

    n = len(values)


    indices = rng.integers(
        0,
        n,
        size=(
            n_boot,
            n
        )
    )


    bootstrap_means = values[
        indices
    ].mean(
        axis=1
    )


    low, high = np.percentile(
        bootstrap_means,
        [
            2.5,
            97.5
        ]
    )


    return (
        float(low),
        float(high)
    )


# =============================================================================
# 5. LOAD TASK 8B FROZEN EPOCHS
# =============================================================================

print(
    "=" * 110
)

print(
    "TASK 9A — TARGETED WITHIN-ACTIVE THETA CONNECTIVITY"
)

print(
    "=" * 110
)


print(
    f"\nFrozen epoch file:\n{EPOCH_FILE}"
)


epochs = mne.read_epochs(
    EPOCH_FILE,
    preload=True,
    verbose="ERROR"
)


print(
    f"\nTotal epochs : {len(epochs)}"
)

print(
    f"Channels     : {len(epochs.ch_names)}"
)

print(
    f"Sampling Hz  : {epochs.info['sfreq']}"
)

print(
    f"Epoch length : {epochs.times[-1] - epochs.times[0] + 1/epochs.info['sfreq']:.3f} s"
)


# =============================================================================
# 6. BASIC DATA INTEGRITY
# =============================================================================

if len(epochs.ch_names) != EXPECTED_CHANNELS:

    raise RuntimeError(
        f"Expected {EXPECTED_CHANNELS} EEG channels; "
        f"found {len(epochs.ch_names)}."
    )


if not np.isclose(
    epochs.info["sfreq"],
    EXPECTED_SFREQ
):

    raise RuntimeError(
        f"Expected {EXPECTED_SFREQ} Hz; "
        f"found {epochs.info['sfreq']} Hz."
    )


missing_channels = [
    channel
    for channel in EXPECTED_CHANNEL_ORDER
    if channel not in epochs.ch_names
]


if missing_channels:

    raise RuntimeError(
        f"Missing expected EEG channels:\n{missing_channels}"
    )


# Keep exact frozen channel order.
epochs = epochs.copy().pick(
    EXPECTED_CHANNEL_ORDER
)


# =============================================================================
# 7. INSPECT METADATA / EVENT STRUCTURE
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "FROZEN EPOCH METADATA"
)

print(
    "=" * 110
)


if epochs.metadata is not None:

    print(
        "\nMetadata columns:"
    )

    print(
        list(
            epochs.metadata.columns
        )
    )

    display(
        epochs.metadata.head(20)
    )

else:

    print(
        "\nNo MNE metadata table found."
    )


print(
    "\nEvent dictionary:"
)

print(
    epochs.event_id
)


# =============================================================================
# 8. DETERMINE SUBJECT AND PHASE LABELS
# =============================================================================
#
# Task 8B should contain metadata. We deliberately do not guess labels
# from epoch order unless necessary and structurally verifiable.
# =============================================================================

if epochs.metadata is None:

    raise RuntimeError(
        "Task 8B epochs contain no metadata. "
        "Task 9A will not infer participant/phase identity from raw order."
    )


metadata = epochs.metadata.copy().reset_index(
    drop=True
)


def find_metadata_column(candidates):

    lower_map = {
        str(col).lower(): col
        for col in metadata.columns
    }

    for candidate in candidates:

        if candidate.lower() in lower_map:

            return lower_map[
                candidate.lower()
            ]

    return None


subject_col = find_metadata_column(
    [
        "subject",
        "participant",
        "participant_id"
    ]
)


phase_col = find_metadata_column(
    [
        "marker_code",
        "phase",
        "condition",
        "marker"
    ]
)


if subject_col is None:

    raise RuntimeError(
        "Could not identify subject column in Task 8B metadata."
    )


if phase_col is None:

    raise RuntimeError(
        "Could not identify marker/phase column in Task 8B metadata."
    )


def normalize_subject(value):

    value = str(value).strip()

    digits = "".join(
        character
        for character in value
        if character.isdigit()
    )

    if digits:

        return f"sub-{int(digits):03d}"

    return value


metadata[
    "_subject"
] = metadata[
    subject_col
].map(
    normalize_subject
)


metadata[
    "_phase"
] = metadata[
    phase_col
].astype(
    str
)


# =============================================================================
# 9. VERIFY FROZEN BALANCE
# =============================================================================

balance = (
    metadata
    .groupby(
        [
            "_subject",
            "_phase"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "FROZEN EPOCH BALANCE"
)

print(
    "=" * 110
)


display(
    balance
)


for subject in SUBJECTS:

    if subject not in balance.index:

        raise RuntimeError(
            f"Missing participant: {subject}"
        )


    for phase in [
        "T1_V1",
        "T1_V2",
        "T1_V3",
        "T1_V4"
    ]:

        if phase not in balance.columns:

            raise RuntimeError(
                f"Missing phase in frozen epochs: {phase}"
            )


        if int(
            balance.loc[
                subject,
                phase
            ]
        ) != EXPECTED_EPOCHS_PER_PHASE:

            raise RuntimeError(
                f"{subject}/{phase}: expected "
                f"{EXPECTED_EPOCHS_PER_PHASE} epochs, found "
                f"{balance.loc[subject, phase]}."
            )


# =============================================================================
# 10. DEFINE UNDIRECTED EDGES
# =============================================================================

n_channels = len(
    epochs.ch_names
)


upper_i, upper_j = np.triu_indices(
    n_channels,
    k=1
)


N_EDGES = len(
    upper_i
)


if N_EDGES != 91:

    raise RuntimeError(
        f"Expected 91 undirected edges; found {N_EDGES}."
    )


channel_to_index = {
    channel: index
    for index, channel in enumerate(
        epochs.ch_names
    )
}


edge_table = pd.DataFrame(
    {
        "edge_index":
            np.arange(
                N_EDGES
            ),

        "i":
            upper_i,

        "j":
            upper_j,

        "channel_i":
            [
                epochs.ch_names[index]
                for index in upper_i
            ],

        "channel_j":
            [
                epochs.ch_names[index]
                for index in upper_j
            ]
    }
)


# =============================================================================
# 11. DEFINE PRESPECIFIED NETWORK EDGE MASKS
# =============================================================================

frontal_set = set(
    FRONTAL
)

posterior_set = set(
    POSTERIOR
)


def edge_class(channel_a, channel_b):

    a_f = channel_a in frontal_set
    b_f = channel_b in frontal_set

    a_p = channel_a in posterior_set
    b_p = channel_b in posterior_set


    if a_f and b_f:
        return "Frontal-Frontal"

    if a_p and b_p:
        return "Posterior-Posterior"

    if (
        (a_f and b_p)
        or
        (a_p and b_f)
    ):
        return "Frontal-Posterior"

    return "Other"


edge_table[
    "network"
] = [
    edge_class(a, b)
    for a, b in zip(
        edge_table[
            "channel_i"
        ],
        edge_table[
            "channel_j"
        ]
    )
]


print(
    "\n"
    +
    "=" * 110
)

print(
    "PRESPECIFIED EDGE COUNTS"
)

print(
    "=" * 110
)


network_edge_counts = (
    edge_table[
        "network"
    ]
    .value_counts()
    .rename_axis(
        "network"
    )
    .reset_index(
        name="n_edges"
    )
)


display(
    network_edge_counts
)


expected_ff = (
    len(FRONTAL)
    *
    (
        len(FRONTAL) - 1
    )
    //
    2
)


expected_fp = (
    len(FRONTAL)
    *
    len(POSTERIOR)
)


expected_pp = (
    len(POSTERIOR)
    *
    (
        len(POSTERIOR) - 1
    )
    //
    2
)


print(
    f"\nExpected FF edges : {expected_ff}"
)

print(
    f"Expected F-P edges: {expected_fp}"
)

print(
    f"Expected P-P edges: {expected_pp}"
)


# =============================================================================
# 12. CONNECTIVITY EXTRACTION FUNCTION
# =============================================================================

def compute_theta_wpli2_debiased(
    data,
    sfreq
):

    """
    Parameters
    ----------
    data : ndarray
        Shape = (epochs, channels, time)

    Returns
    -------
    matrix : ndarray
        Symmetric 14 x 14 theta-band wPLI²_debiased matrix.

    edge_values : ndarray
        Upper-triangle 91-edge vector.
    """

    if data.ndim != 3:

        raise ValueError(
            f"Expected 3D epoch data; got {data.shape}"
        )


    if data.shape[0] != EXPECTED_EPOCHS_PER_PHASE:

        raise ValueError(
            f"Expected 9 epochs; found {data.shape[0]}."
        )


    if data.shape[1] != EXPECTED_CHANNELS:

        raise ValueError(
            f"Expected 14 channels; found {data.shape[1]}."
        )


    if not np.isfinite(data).all():

        raise ValueError(
            "Non-finite EEG data supplied to connectivity estimator."
        )


    # -------------------------------------------------------------
    # Explicit upper-triangle connectivity indices.
    # -------------------------------------------------------------

    indices = (
        upper_i,
        upper_j
    )


    con = spectral_connectivity_epochs(
        data,
        method="wpli2_debiased",
        mode="multitaper",
        sfreq=sfreq,
        indices=indices,
        fmin=THETA_MIN,
        fmax=THETA_MAX,
        faverage=True,
        mt_bandwidth=MT_BANDWIDTH,
        mt_adaptive=False,
        mt_low_bias=True,
        n_jobs=1,
        verbose=False
    )


    dense = con.get_data(
        output="dense"
    )


    dense = np.asarray(
        dense,
        dtype=float
    )


    # Common output forms:
    # (n_nodes, n_nodes, n_freqs)
    # or occasionally already squeezed.
    dense = np.squeeze(
        dense
    )


    if dense.shape != (
        EXPECTED_CHANNELS,
        EXPECTED_CHANNELS
    ):

        # With explicit indices, some versions return edge-vector data.
        compact = np.asarray(
            con.get_data(),
            dtype=float
        )

        compact = np.squeeze(
            compact
        )


        if compact.ndim == 1 and len(compact) == N_EDGES:

            edge_values = compact.copy()

            matrix = np.zeros(
                (
                    EXPECTED_CHANNELS,
                    EXPECTED_CHANNELS
                ),
                dtype=float
            )

            matrix[
                upper_i,
                upper_j
            ] = edge_values

            matrix[
                upper_j,
                upper_i
            ] = edge_values

            np.fill_diagonal(
                matrix,
                0.0
            )

            return (
                matrix,
                edge_values
            )


        raise RuntimeError(
            "Unexpected mne-connectivity output shape.\n"
            f"Dense shape: {dense.shape}\n"
            f"Compact shape: {compact.shape}"
        )


    # -------------------------------------------------------------
    # MNE dense output may populate only the requested orientation.
    # Reconstruct a symmetric undirected matrix from explicit edges.
    # -------------------------------------------------------------

    edge_values = dense[
        upper_i,
        upper_j
    ]


    # Some versions place explicit-index values in reverse orientation.
    if np.allclose(
        edge_values,
        0.0
    ):

        reverse_values = dense[
            upper_j,
            upper_i
        ]

        if not np.allclose(
            reverse_values,
            0.0
        ):

            edge_values = reverse_values


    matrix = np.zeros(
        (
            EXPECTED_CHANNELS,
            EXPECTED_CHANNELS
        ),
        dtype=float
    )


    matrix[
        upper_i,
        upper_j
    ] = edge_values


    matrix[
        upper_j,
        upper_i
    ] = edge_values


    np.fill_diagonal(
        matrix,
        0.0
    )


    return (
        matrix,
        np.asarray(
            edge_values,
            dtype=float
        )
    )


# =============================================================================
# 13. COMPUTE PARTICIPANT × PHASE CONNECTIVITY
# =============================================================================

summary_rows = []

edge_rows = []

matrix_records = {}


print(
    "\n"
    +
    "=" * 110
)

print(
    "COMPUTING VALIDATED THETA wPLI²_debiased"
)

print(
    "=" * 110
)


for subject in SUBJECTS:

    print(
        f"\n{subject}"
    )


    for phase in PHASES:

        mask = (
            (
                metadata[
                    "_subject"
                ]
                ==
                subject
            )
            &
            (
                metadata[
                    "_phase"
                ]
                ==
                phase
            )
        ).to_numpy()


        indices_selected = np.where(
            mask
        )[0]


        if len(
            indices_selected
        ) != EXPECTED_EPOCHS_PER_PHASE:

            raise RuntimeError(
                f"{subject}/{phase}: expected 9 epochs; "
                f"found {len(indices_selected)}."
            )


        data = epochs[
            indices_selected
        ].get_data(
            copy=True
        )


        matrix, edge_values = compute_theta_wpli2_debiased(
            data=data,
            sfreq=float(
                epochs.info[
                    "sfreq"
                ]
            )
        )


        if len(
            edge_values
        ) != N_EDGES:

            raise RuntimeError(
                f"{subject}/{phase}: expected 91 edges; "
                f"found {len(edge_values)}."
            )


        if not np.isfinite(
            edge_values
        ).all():

            raise RuntimeError(
                f"{subject}/{phase}: non-finite connectivity."
            )


        matrix_records[
            (
                subject,
                phase
            )
        ] = matrix


        # ---------------------------------------------------------
        # Global
        # ---------------------------------------------------------

        global_mean = float(
            np.mean(
                edge_values
            )
        )


        # ---------------------------------------------------------
        # Prespecified network means
        # ---------------------------------------------------------

        temp_edges = edge_table.copy()

        temp_edges[
            "value"
        ] = edge_values


        ff_mean = float(
            temp_edges.loc[
                temp_edges[
                    "network"
                ]
                ==
                "Frontal-Frontal",
                "value"
            ].mean()
        )


        fp_mean = float(
            temp_edges.loc[
                temp_edges[
                    "network"
                ]
                ==
                "Frontal-Posterior",
                "value"
            ].mean()
        )


        pp_mean = float(
            temp_edges.loc[
                temp_edges[
                    "network"
                ]
                ==
                "Posterior-Posterior",
                "value"
            ].mean()
        )


        summary_rows.append(
            {
                "subject":
                    subject,

                "phase":
                    phase,

                "n_epochs":
                    len(
                        indices_selected
                    ),

                "global_theta_wpli2_debiased":
                    global_mean,

                "frontal_frontal":
                    ff_mean,

                "frontal_posterior":
                    fp_mean,

                "posterior_posterior":
                    pp_mean
            }
        )


        for edge_index, value in enumerate(
            edge_values
        ):

            edge_rows.append(
                {
                    "subject":
                        subject,

                    "phase":
                        phase,

                    "edge_index":
                        edge_index,

                    "channel_i":
                        edge_table.loc[
                            edge_index,
                            "channel_i"
                        ],

                    "channel_j":
                        edge_table.loc[
                            edge_index,
                            "channel_j"
                        ],

                    "network":
                        edge_table.loc[
                            edge_index,
                            "network"
                        ],

                    "theta_wpli2_debiased":
                        float(
                            value
                        )
                }
            )


        print(
            f"  {phase:<6} "
            f"global={global_mean:+.6f}  "
            f"FF={ff_mean:+.6f}  "
            f"F-P={fp_mean:+.6f}  "
            f"P-P={pp_mean:+.6f}"
        )


summary = pd.DataFrame(
    summary_rows
)


edge_values_df = pd.DataFrame(
    edge_rows
)


# =============================================================================
# 14. NUMERICAL CONNECTIVITY QC
# =============================================================================

matrix_qc_rows = []


for (
    subject,
    phase
), matrix in matrix_records.items():

    symmetry_error = float(
        np.max(
            np.abs(
                matrix
                -
                matrix.T
            )
        )
    )


    diagonal_error = float(
        np.max(
            np.abs(
                np.diag(
                    matrix
                )
            )
        )
    )


    matrix_qc_rows.append(
        {
            "subject":
                subject,

            "phase":
                phase,

            "finite":
                bool(
                    np.isfinite(
                        matrix
                    ).all()
                ),

            "symmetry_max_error":
                symmetry_error,

            "diagonal_max_abs":
                diagonal_error,

            "matrix_min":
                float(
                    np.min(
                        matrix[
                            upper_i,
                            upper_j
                        ]
                    )
                ),

            "matrix_max":
                float(
                    np.max(
                        matrix[
                            upper_i,
                            upper_j
                        ]
                    )
                ),

            "negative_edges":
                int(
                    np.sum(
                        matrix[
                            upper_i,
                            upper_j
                        ]
                        <
                        0
                    )
                ),

            "n_edges":
                N_EDGES
        }
    )


matrix_qc = pd.DataFrame(
    matrix_qc_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "CONNECTIVITY MATRIX QC"
)

print(
    "=" * 110
)


display(
    matrix_qc.round(
        9
    )
)


print(
    "\nNOTE:"
)

print(
    "Negative wPLI²_debiased estimates, if present, are retained."
)

print(
    "They are not clipped to zero."
)


# =============================================================================
# 15. PARTICIPANT-LEVEL PHASE CONTRASTS
# =============================================================================

FEATURES = {
    "Global":
        "global_theta_wpli2_debiased",

    "Frontal-Frontal":
        "frontal_frontal",

    "Frontal-Posterior":
        "frontal_posterior",

    "Posterior-Posterior":
        "posterior_posterior"
}


participant_delta_rows = []


for feature_name, column in FEATURES.items():

    wide = (
        summary
        .pivot(
            index="subject",
            columns="phase",
            values=column
        )
        .reindex(
            index=SUBJECTS,
            columns=PHASES
        )
    )


    if wide.isna().any().any():

        raise RuntimeError(
            f"Missing values for {feature_name}."
        )


    for subject in SUBJECTS:

        v1 = float(
            wide.loc[
                subject,
                "T1_V1"
            ]
        )

        v4 = float(
            wide.loc[
                subject,
                "T1_V4"
            ]
        )


        participant_delta_rows.append(
            {
                "subject":
                    subject,

                "feature":
                    feature_name,

                "T1_V1":
                    v1,

                "T1_V4":
                    v4,

                "delta_V4_minus_V1":
                    v4 - v1
            }
        )


participant_deltas = pd.DataFrame(
    participant_delta_rows
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PARTICIPANT-LEVEL THETA CONNECTIVITY CHANGES"
)

print(
    "Δ = T1_V4 − T1_V1"
)

print(
    "=" * 110
)


display(
    participant_deltas.round(
        6
    )
)


# =============================================================================
# 16. EXACT PAIRED INFERENCE
# =============================================================================

inference_rows = []


for feature_number, (
    feature_name,
    column
) in enumerate(
    FEATURES.items()
):

    temp = participant_deltas.loc[
        participant_deltas[
            "feature"
        ]
        ==
        feature_name
    ].set_index(
        "subject"
    ).reindex(
        SUBJECTS
    )


    v1 = temp[
        "T1_V1"
    ].to_numpy(
        dtype=float
    )


    v4 = temp[
        "T1_V4"
    ].to_numpy(
        dtype=float
    )


    delta = (
        v4
        -
        v1
    )


    p_exact = exact_signflip_test(
        delta
    )


    dz = paired_dz(
        v4,
        v1
    )


    ci_low, ci_high = bootstrap_mean_ci(
        delta,
        n_boot=N_BOOT,
        seed=(
            RNG_SEED
            +
            feature_number
        )
    )


    inference_rows.append(
        {
            "feature":
                feature_name,

            "n":
                len(delta),

            "T1_V1_mean":
                float(
                    np.mean(v1)
                ),

            "T1_V1_SD":
                float(
                    np.std(
                        v1,
                        ddof=1
                    )
                ),

            "T1_V4_mean":
                float(
                    np.mean(v4)
                ),

            "T1_V4_SD":
                float(
                    np.std(
                        v4,
                        ddof=1
                    )
                ),

            "mean_delta_V4_minus_V1":
                float(
                    np.mean(delta)
                ),

            "median_delta":
                float(
                    np.median(delta)
                ),

            "bootstrap_CI_low":
                ci_low,

            "bootstrap_CI_high":
                ci_high,

            "paired_dz":
                dz,

            "p_exact":
                p_exact,

            "participants_delta_positive":
                int(
                    np.sum(
                        delta > 0
                    )
                ),

            "participants_delta_negative":
                int(
                    np.sum(
                        delta < 0
                    )
                ),

            "participants_delta_zero":
                int(
                    np.sum(
                        np.isclose(
                            delta,
                            0.0
                        )
                    )
                )
        }
    )


inference = pd.DataFrame(
    inference_rows
)


inference[
    "p_FDR_4"
] = bh_fdr(
    inference[
        "p_exact"
    ].to_numpy(
        dtype=float
    )
)


inference[
    "raw_significant"
] = (
    inference[
        "p_exact"
    ]
    <
    0.05
)


inference[
    "FDR_significant"
] = (
    inference[
        "p_FDR_4"
    ]
    <
    0.05
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "PRIMARY TARGETED THETA CONNECTIVITY INFERENCE"
)

print(
    "Exact paired sign-flip tests; BH-FDR across 4 prespecified summaries"
)

print(
    "=" * 110
)


display(
    inference.round(
        6
    )
)


# =============================================================================
# 17. RANK EFFECTS BY ABSOLUTE dz
# =============================================================================

ranked = (
    inference
    .assign(
        abs_dz=lambda x:
            np.abs(
                x[
                    "paired_dz"
                ]
            )
    )
    .sort_values(
        "abs_dz",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "RANKED TARGETED EFFECTS"
)

print(
    "=" * 110
)


display(
    ranked[
        [
            "feature",
            "mean_delta_V4_minus_V1",
            "bootstrap_CI_low",
            "bootstrap_CI_high",
            "paired_dz",
            "p_exact",
            "p_FDR_4",
            "participants_delta_positive",
            "participants_delta_negative"
        ]
    ].round(
        6
    )
)


# =============================================================================
# 18. LEAVE-ONE-PARTICIPANT-OUT DIRECTIONAL ROBUSTNESS
# =============================================================================
#
# This is descriptive robustness analysis only.
# We do NOT repeatedly significance-test the reduced samples.
# =============================================================================

loo_rows = []


for feature_name in FEATURES:

    temp = (
        participant_deltas.loc[
            participant_deltas[
                "feature"
            ]
            ==
            feature_name
        ]
        .set_index(
            "subject"
        )
        .reindex(
            SUBJECTS
        )
    )


    full_delta = temp[
        "delta_V4_minus_V1"
    ].to_numpy(
        dtype=float
    )


    full_mean = float(
        np.mean(
            full_delta
        )
    )


    full_sign = np.sign(
        full_mean
    )


    for omitted_subject in SUBJECTS:

        retained = temp.drop(
            index=omitted_subject
        )


        delta = retained[
            "delta_V4_minus_V1"
        ].to_numpy(
            dtype=float
        )


        loo_mean = float(
            np.mean(
                delta
            )
        )


        loo_rows.append(
            {
                "feature":
                    feature_name,

                "omitted_subject":
                    omitted_subject,

                "n":
                    len(delta),

                "full_mean_delta":
                    full_mean,

                "loo_mean_delta":
                    loo_mean,

                "same_direction_as_full":
                    bool(
                        np.sign(
                            loo_mean
                        )
                        ==
                        full_sign
                    )
            }
        )


loo = pd.DataFrame(
    loo_rows
)


loo_summary = (
    loo
    .groupby(
        "feature",
        as_index=False
    )
    .agg(
        full_mean_delta=(
            "full_mean_delta",
            "first"
        ),

        same_direction_n=(
            "same_direction_as_full",
            "sum"
        ),

        loo_min_delta=(
            "loo_mean_delta",
            "min"
        ),

        loo_max_delta=(
            "loo_mean_delta",
            "max"
        )
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "LEAVE-ONE-PARTICIPANT-OUT DIRECTIONAL ROBUSTNESS"
)

print(
    "=" * 110
)


display(
    loo_summary.round(
        6
    )
)


# =============================================================================
# 19. GROUP-MEAN CONNECTIVITY MATRICES
# =============================================================================

group_matrices = {}


for phase in PHASES:

    matrices = np.stack(
        [
            matrix_records[
                (
                    subject,
                    phase
                )
            ]
            for subject in SUBJECTS
        ],
        axis=0
    )


    group_matrices[
        phase
    ] = matrices.mean(
        axis=0
    )


delta_matrix = (
    group_matrices[
        "T1_V4"
    ]
    -
    group_matrices[
        "T1_V1"
    ]
)


# =============================================================================
# 20. FIGURE — GROUP MEAN T1_V1 MATRIX
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        7
    )
)


image = ax.imshow(
    group_matrices[
        "T1_V1"
    ],
    aspect="equal"
)


ax.set_xticks(
    np.arange(
        EXPECTED_CHANNELS
    )
)

ax.set_yticks(
    np.arange(
        EXPECTED_CHANNELS
    )
)


ax.set_xticklabels(
    epochs.ch_names,
    rotation=90
)

ax.set_yticklabels(
    epochs.ch_names
)


ax.set_title(
    "T1_V1 mean theta wPLI²_debiased"
)


fig.colorbar(
    image,
    ax=ax,
    label="wPLI²_debiased"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "theta_wpli2_debiased_T1_V1_mean_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 21. FIGURE — GROUP MEAN T1_V4 MATRIX
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        7
    )
)


image = ax.imshow(
    group_matrices[
        "T1_V4"
    ],
    aspect="equal"
)


ax.set_xticks(
    np.arange(
        EXPECTED_CHANNELS
    )
)

ax.set_yticks(
    np.arange(
        EXPECTED_CHANNELS
    )
)


ax.set_xticklabels(
    epochs.ch_names,
    rotation=90
)

ax.set_yticklabels(
    epochs.ch_names
)


ax.set_title(
    "T1_V4 mean theta wPLI²_debiased"
)


fig.colorbar(
    image,
    ax=ax,
    label="wPLI²_debiased"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "theta_wpli2_debiased_T1_V4_mean_matrix.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 22. FIGURE — DELTA MATRIX
# =============================================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        7
    )
)


image = ax.imshow(
    delta_matrix,
    aspect="equal"
)


ax.set_xticks(
    np.arange(
        EXPECTED_CHANNELS
    )
)

ax.set_yticks(
    np.arange(
        EXPECTED_CHANNELS
    )
)


ax.set_xticklabels(
    epochs.ch_names,
    rotation=90
)

ax.set_yticklabels(
    epochs.ch_names
)


ax.set_title(
    "Theta connectivity change: T1_V4 − T1_V1"
)


fig.colorbar(
    image,
    ax=ax,
    label="Δ wPLI²_debiased"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "theta_wpli2_debiased_delta_T1_V4_minus_T1_V1.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 23. FIGURE — PARTICIPANT PAIRED GLOBAL CONNECTIVITY
# =============================================================================

global_temp = (
    participant_deltas.loc[
        participant_deltas[
            "feature"
        ]
        ==
        "Global"
    ]
    .set_index(
        "subject"
    )
    .reindex(
        SUBJECTS
    )
)


fig, ax = plt.subplots(
    figsize=(
        7,
        6
    )
)


for subject in SUBJECTS:

    ax.plot(
        [
            0,
            1
        ],
        [
            global_temp.loc[
                subject,
                "T1_V1"
            ],
            global_temp.loc[
                subject,
                "T1_V4"
            ]
        ],
        marker="o",
        alpha=0.6
    )


means = [
    global_temp[
        "T1_V1"
    ].mean(),
    global_temp[
        "T1_V4"
    ].mean()
]


sems = [
    global_temp[
        "T1_V1"
    ].sem(),
    global_temp[
        "T1_V4"
    ].sem()
]


ax.errorbar(
    [
        0,
        1
    ],
    means,
    yerr=sems,
    marker="o",
    linewidth=3,
    capsize=5
)


ax.set_xticks(
    [
        0,
        1
    ]
)


ax.set_xticklabels(
    [
        "T1_V1",
        "T1_V4"
    ]
)


ax.set_ylabel(
    "Global theta wPLI²_debiased"
)


ax.set_title(
    "Within-Active global theta connectivity"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "global_theta_connectivity_paired.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 24. FIGURE — FOUR PRESPECIFIED DELTAS
# =============================================================================

plot_data = (
    participant_deltas
    .pivot(
        index="subject",
        columns="feature",
        values="delta_V4_minus_V1"
    )
    .reindex(
        index=SUBJECTS,
        columns=list(
            FEATURES.keys()
        )
    )
)


fig, ax = plt.subplots(
    figsize=(
        10,
        6
    )
)


x = np.arange(
    len(
        FEATURES
    )
)


means = plot_data.mean(
    axis=0
).to_numpy(
    dtype=float
)


sems = plot_data.sem(
    axis=0
).to_numpy(
    dtype=float
)


ax.errorbar(
    x,
    means,
    yerr=sems,
    marker="o",
    linewidth=2,
    capsize=5
)


for participant_number, subject in enumerate(
    SUBJECTS
):

    ax.scatter(
        x,
        plot_data.loc[
            subject
        ].to_numpy(
            dtype=float
        ),
        alpha=0.45
    )


ax.axhline(
    0,
    linewidth=1
)


ax.set_xticks(
    x
)


ax.set_xticklabels(
    list(
        FEATURES.keys()
    ),
    rotation=20,
    ha="right"
)


ax.set_ylabel(
    "Δ theta wPLI²_debiased (T1_V4 − T1_V1)"
)


ax.set_title(
    "Prespecified theta connectivity changes"
)


fig.tight_layout()


fig.savefig(
    FIG_OUT
    /
    "targeted_theta_connectivity_deltas.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()


# =============================================================================
# 25. SAVE MATRICES
# =============================================================================

for subject in SUBJECTS:

    for phase in PHASES:

        matrix = matrix_records[
            (
                subject,
                phase
            )
        ]


        matrix_df = pd.DataFrame(
            matrix,
            index=epochs.ch_names,
            columns=epochs.ch_names
        )


        matrix_df.to_csv(
            MATRIX_OUT
            /
            (
                f"{subject}_{phase}_"
                "theta_wpli2_debiased.csv"
            )
        )


pd.DataFrame(
    group_matrices[
        "T1_V1"
    ],
    index=epochs.ch_names,
    columns=epochs.ch_names
).to_csv(
    MATRIX_OUT
    /
    "group_T1_V1_theta_wpli2_debiased.csv"
)


pd.DataFrame(
    group_matrices[
        "T1_V4"
    ],
    index=epochs.ch_names,
    columns=epochs.ch_names
).to_csv(
    MATRIX_OUT
    /
    "group_T1_V4_theta_wpli2_debiased.csv"
)


pd.DataFrame(
    delta_matrix,
    index=epochs.ch_names,
    columns=epochs.ch_names
).to_csv(
    MATRIX_OUT
    /
    "group_delta_T1_V4_minus_T1_V1_theta_wpli2_debiased.csv"
)


# =============================================================================
# 26. SAVE TABLES
# =============================================================================

SUMMARY_FILE = (
    OUT
    /
    "participant_phase_theta_connectivity.csv"
)


EDGE_FILE = (
    OUT
    /
    "participant_phase_theta_edge_values.csv"
)


DELTA_FILE = (
    OUT
    /
    "participant_theta_connectivity_deltas.csv"
)


INFERENCE_FILE = (
    OUT
    /
    "targeted_theta_connectivity_inference.csv"
)


LOO_FILE = (
    OUT
    /
    "theta_connectivity_leave_one_out.csv"
)


LOO_SUMMARY_FILE = (
    OUT
    /
    "theta_connectivity_leave_one_out_summary.csv"
)


MATRIX_QC_FILE = (
    OUT
    /
    "theta_connectivity_matrix_qc.csv"
)


EDGE_DEFINITION_FILE = (
    OUT
    /
    "theta_connectivity_edge_definitions.csv"
)


summary.to_csv(
    SUMMARY_FILE,
    index=False
)


edge_values_df.to_csv(
    EDGE_FILE,
    index=False
)


participant_deltas.to_csv(
    DELTA_FILE,
    index=False
)


inference.to_csv(
    INFERENCE_FILE,
    index=False
)


loo.to_csv(
    LOO_FILE,
    index=False
)


loo_summary.to_csv(
    LOO_SUMMARY_FILE,
    index=False
)


matrix_qc.to_csv(
    MATRIX_QC_FILE,
    index=False
)


edge_table.to_csv(
    EDGE_DEFINITION_FILE,
    index=False
)


# =============================================================================
# 27. FINAL INTEGRITY CHECK
# =============================================================================

expected_summary_rows = (
    8
    *
    2
)


expected_edge_rows = (
    8
    *
    2
    *
    91
)


integrity = {

    "participants_8":
        summary[
            "subject"
        ].nunique()
        ==
        8,

    "phases_exactly_V1_V4":
        set(
            summary[
                "phase"
            ].unique()
        )
        ==
        set(
            PHASES
        ),

    "summary_rows_16":
        len(
            summary
        )
        ==
        expected_summary_rows,

    "edge_rows_1456":
        len(
            edge_values_df
        )
        ==
        expected_edge_rows,

    "channels_14":
        len(
            epochs.ch_names
        )
        ==
        14,

    "edges_91":
        N_EDGES
        ==
        91,

    "epochs_9_per_phase":
        bool(
            (
                summary[
                    "n_epochs"
                ]
                ==
                9
            ).all()
        ),

    "finite_summary":
        bool(
            np.isfinite(
                summary[
                    list(
                        FEATURES.values()
                    )
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),

    "finite_edges":
        bool(
            np.isfinite(
                edge_values_df[
                    "theta_wpli2_debiased"
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),

    "matrix_finite":
        bool(
            matrix_qc[
                "finite"
            ].all()
        ),

    "matrix_symmetric":
        bool(
            (
                matrix_qc[
                    "symmetry_max_error"
                ]
                <
                1e-12
            ).all()
        ),

    "matrix_zero_diagonal":
        bool(
            (
                matrix_qc[
                    "diagonal_max_abs"
                ]
                <
                1e-12
            ).all()
        ),

    "four_prespecified_tests":
        len(
            inference
        )
        ==
        4,

    "all_p_valid":
        bool(
            inference[
                "p_exact"
            ].between(
                0,
                1
            ).all()
        ),

    "all_fdr_valid":
        bool(
            inference[
                "p_FDR_4"
            ].between(
                0,
                1
            ).all()
        ),

    "no_edgewise_inference":
        True,

    "negative_values_not_clipped":
        True,

    "validated_estimator":
        True
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 9A INTEGRITY CHECK"
)

print(
    "=" * 110
)


for key, value in integrity.items():

    print(
        f"{key:<40}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 110
)


print(
    "OVERALL:",
    (
        "PASS"
        if overall_pass
        else
        "FAIL"
    )
)


print(
    "-" * 110
)


if not overall_pass:

    raise RuntimeError(
        "Task 9A integrity check failed."
    )


# =============================================================================
# 28. INTERPRETATION LOGIC
# =============================================================================

n_fdr = int(
    inference[
        "FDR_significant"
    ].sum()
)


n_raw = int(
    inference[
        "raw_significant"
    ].sum()
)


print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 9A INTERPRETATION"
)

print(
    "=" * 110
)


if n_fdr > 0:

    print(
        f"""
At least one of the four prespecified theta-connectivity summaries
survived BH-FDR correction.

Interpret only the corrected feature(s) as statistically supported
within this targeted first-cycle follow-up.

Because Task 8D showed that an all-participant second-cycle replication
is impossible, these results remain first-cycle findings and should not
be described as independently replicated.
"""
    )


elif n_raw > 0:

    print(
        f"""
At least one targeted theta-connectivity summary had raw p < .05,
but none survived correction across the four prespecified summaries.

Treat such effects as descriptive only.

The appropriate conclusion is that the first-cycle theta spectral
pattern is not accompanied by multiplicity-corrected evidence of a
corresponding connectivity change.
"""
    )


else:

    print(
        f"""
None of the four prespecified theta-connectivity summaries reached
raw or FDR-corrected significance.

The appropriate conclusion is that the first-cycle T1_V1-to-T1_V4
theta spectral pattern was not accompanied by evidence of a systematic
change in global, frontal-frontal, frontal-posterior, or
posterior-posterior theta wPLI²_debiased connectivity.

Do not broaden the search to additional bands or individual edges
solely because this targeted analysis is null.
"""
    )


# =============================================================================
# 29. OUTPUT MANIFEST
# =============================================================================

print(
    "\n"
    +
    "=" * 110
)

print(
    "TASK 9A COMPLETE"
)

print(
    "=" * 110
)


print(
    f"""
Participants:
    8 Active participants

Frozen first-cycle phases:
    T1_V1
    T1_V4

Epoch balance:
    9 epochs per phase
    18 s per phase

Connectivity:
    debiased squared weighted phase-lag index
    wPLI²_debiased

Frequency:
    theta 4–8 Hz

Estimator:
    MNE-Connectivity spectral_connectivity_epochs
    method='wpli2_debiased'
    mode='multitaper'
    mt_bandwidth=2.0
    mt_adaptive=False
    mt_low_bias=True
    faverage=True

Prespecified summaries:
    Global
    Frontal-Frontal
    Frontal-Posterior
    Posterior-Posterior

Inference:
    exact paired sign-flip
    paired dz
    bootstrap 95% CI
    BH-FDR across 4 summaries

Individual-edge inference:
    NONE

Semantic interpretation of marker codes:
    NONE assigned

Output directory:
    {OUT}

Matrix directory:
    {MATRIX_OUT}

Figure directory:
    {FIG_OUT}
"""
)


print(
    "=" * 110
)

In [ ]:
# =============================================================================
# TASK 10A — FINAL RESULTS CONSOLIDATION AND MANUSCRIPT-READY TABLES
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PURPOSE
# -------
# Consolidate the FROZEN analysis results into a coherent manuscript-facing
# results package.
#
# THIS TASK DOES NOT:
#   - run new inferential tests
#   - search for additional significant findings
#   - perform individual-edge inference
#   - resurrect legacy Hilbert connectivity analyses
#   - assign undocumented meanings to T1_V1–T1_V4
#
# FINAL ANALYTICAL FAMILIES
# -------------------------
# A. Between-group spectral dynamics
#       Task 4A — Baseline A
#       Task 4B — Baseline B sensitivity
#
# B. Between-group validated connectivity
#       Task 6C — matched fixed-9 Baseline A/B
#       estimator: wPLI²_debiased
#
# C. Validated network localization/topology
#       Task 7A / Task 7B
#
# D. Within-Active sequential marker-phase spectral dynamics
#       Task 8C
#
# E. Targeted within-Active theta connectivity follow-up
#       Task 9A
#
# STRUCTURAL/QC SUPPORT
# ---------------------
#       Task 3C — physiological baseline validation
#       Task 3D — matched analytical epochs
#       Task 8A — marker-phase structural audit
#       Task 8B — frozen first-cycle epochs
#       Task 8D — repeated-cycle feasibility audit
#
# LEGACY CONNECTIVITY EXCLUDED FROM MANUSCRIPT INFERENCE
# ------------------------------------------------------
#       Task 5A
#       Task 5B0
#       Task 5B
#       Task 5C
#       Task 6A
#
# These used the earlier custom Hilbert estimator and are not part of the
# final inferential evidence base.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


pd.set_option(
    "display.max_columns",
    100
)

pd.set_option(
    "display.width",
    220
)

pd.set_option(
    "display.max_colwidth",
    100
)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

OUT = (
    BASE
    / "task10a_final_results_consolidation"
)

TABLE_OUT = (
    OUT
    / "tables"
)

FIG_OUT = (
    OUT
    / "figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_OUT.mkdir(
    parents=True,
    exist_ok=True
)

FIG_OUT.mkdir(
    parents=True,
    exist_ok=True
)


if not BASE.exists():

    raise FileNotFoundError(
        f"Analysis directory not found:\n{BASE}"
    )


print(
    "=" * 120
)

print(
    "TASK 10A — FINAL RESULTS CONSOLIDATION"
)

print(
    "=" * 120
)

print(
    f"\nAnalysis root:\n{BASE}"
)

print(
    f"\nOutput directory:\n{OUT}"
)


# =============================================================================
# 2. FROZEN STUDY DEFINITIONS
# =============================================================================

N_TOTAL = 15

N_ACTIVE = 8

N_PASSIVE = 7

N_ACTIVE_WITHIN = 8

N_CHANNELS = 14

SFREQ = 256.0


ROIS = [
    "Frontal",
    "CentralTemporal",
    "Posterior"
]


BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]


ACTIVE_PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]


CONNECTIVITY_ESTIMATOR = (
    "debiased squared weighted phase-lag index "
    "(wPLI²_debiased)"
)


# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================

def normalize_name(text):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(text).lower()
    )


def find_column(
    df,
    candidates,
    required=False
):

    normalized = {
        normalize_name(col): col
        for col in df.columns
    }

    for candidate in candidates:

        key = normalize_name(
            candidate
        )

        if key in normalized:

            return normalized[key]

    if required:

        raise KeyError(
            "Could not identify required column.\n"
            f"Candidates: {candidates}\n"
            f"Available: {list(df.columns)}"
        )

    return None


def numeric_series(
    df,
    candidates
):

    col = find_column(
        df,
        candidates,
        required=False
    )

    if col is None:

        return pd.Series(
            np.nan,
            index=df.index,
            dtype=float
        )

    return pd.to_numeric(
        df[col],
        errors="coerce"
    )


def text_series(
    df,
    candidates
):

    col = find_column(
        df,
        candidates,
        required=False
    )

    if col is None:

        return pd.Series(
            "",
            index=df.index,
            dtype=str
        )

    return (
        df[col]
        .astype(str)
        .replace(
            "nan",
            ""
        )
    )


def first_existing_directory(
    patterns
):

    matches = []

    for pattern in patterns:

        matches.extend(
            [
                path
                for path in BASE.glob(pattern)
                if path.is_dir()
            ]
        )

    matches = sorted(
        set(matches)
    )

    return (
        matches[0]
        if matches
        else None
    )


def all_csvs(directory):

    if directory is None:
        return []

    return sorted(
        directory.rglob(
            "*.csv"
        )
    )


def choose_csv(
    directory,
    include_any=None,
    include_all=None,
    exclude=None,
    min_rows=1
):

    include_any = (
        include_any
        if include_any is not None
        else []
    )

    include_all = (
        include_all
        if include_all is not None
        else []
    )

    exclude = (
        exclude
        if exclude is not None
        else []
    )


    candidates = []


    for path in all_csvs(
        directory
    ):

        name = path.name.lower()


        if exclude and any(
            word.lower() in name
            for word in exclude
        ):
            continue


        if include_all and not all(
            word.lower() in name
            for word in include_all
        ):
            continue


        if include_any and not any(
            word.lower() in name
            for word in include_any
        ):
            continue


        try:

            df = pd.read_csv(
                path
            )

        except Exception:

            continue


        if len(df) < min_rows:
            continue


        candidates.append(
            (
                path,
                df
            )
        )


    if not candidates:

        return (
            None,
            None
        )


    # Prefer the richest table, then largest row count.
    candidates.sort(
        key=lambda item: (
            len(
                item[1].columns
            ),
            len(
                item[1]
            )
        ),
        reverse=True
    )


    return candidates[0]


def print_selected_file(
    label,
    path
):

    if path is None:

        print(
            f"{label:<38}: NOT FOUND"
        )

    else:

        print(
            f"{label:<38}: {path}"
        )


def bh_status(
    raw_p,
    adjusted_p
):

    if np.isfinite(
        adjusted_p
    ):

        if adjusted_p < 0.05:
            return "FDR-supported"

        if (
            np.isfinite(raw_p)
            and
            raw_p < 0.05
        ):
            return "Raw p<.05 only"

        return "Not statistically supported"


    if (
        np.isfinite(raw_p)
        and
        raw_p < 0.05
    ):
        return "Raw p<.05; corrected p unavailable"

    return "Not statistically supported"


def format_p(
    value
):

    if not np.isfinite(value):
        return ""

    if value < 0.001:
        return "<.001"

    return f"{value:.3f}".lstrip("0")


def format_effect(
    value
):

    if not np.isfinite(value):
        return ""

    return f"{value:+.2f}"


def format_delta(
    value,
    digits=4
):

    if not np.isfinite(value):
        return ""

    return f"{value:+.{digits}f}"


def clean_feature_name(
    value
):

    value = str(
        value
    )

    replacements = {
        "_": " ",
        "CentralTemporal": "Central/Temporal",
        "central temporal": "Central/Temporal",
        "Frontal-Frontal": "Frontal–Frontal",
        "Frontal-Posterior": "Frontal–Posterior",
        "Posterior-Posterior": "Posterior–Posterior"
    }

    for old, new in replacements.items():

        value = value.replace(
            old,
            new
        )

    return value


def safe_to_csv(
    df,
    filename
):

    path = (
        TABLE_OUT
        / filename
    )

    df.to_csv(
        path,
        index=False
    )

    return path


# =============================================================================
# 4. DISCOVER FINAL ANALYSIS DIRECTORIES
# =============================================================================

TASK3C = first_existing_directory(
    [
        "task03c*",
        "task3c*"
    ]
)

TASK3D = first_existing_directory(
    [
        "task03d*",
        "task3d*"
    ]
)

TASK4A = first_existing_directory(
    [
        "task04a*",
        "task4a*"
    ]
)

TASK4B = first_existing_directory(
    [
        "task04b*",
        "task4b*"
    ]
)

TASK6C = first_existing_directory(
    [
        "task06c*",
        "task6c*"
    ]
)

TASK7A = first_existing_directory(
    [
        "task07a*",
        "task7a*"
    ]
)

TASK7B = first_existing_directory(
    [
        "task07b*",
        "task7b*"
    ]
)

TASK8A = first_existing_directory(
    [
        "task08a*",
        "task8a*"
    ]
)

TASK8B = first_existing_directory(
    [
        "task08b*",
        "task8b*"
    ]
)

TASK8C = first_existing_directory(
    [
        "task08c*",
        "task8c*"
    ]
)

TASK8D = first_existing_directory(
    [
        "task08d*",
        "task8d*"
    ]
)

TASK9A = first_existing_directory(
    [
        "task09a*",
        "task9a*"
    ]
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "DISCOVERED FINAL ANALYSIS DIRECTORIES"
)

print(
    "=" * 120
)


for label, path in [
    ("Task 3C", TASK3C),
    ("Task 3D", TASK3D),
    ("Task 4A", TASK4A),
    ("Task 4B", TASK4B),
    ("Task 6C", TASK6C),
    ("Task 7A", TASK7A),
    ("Task 7B", TASK7B),
    ("Task 8A", TASK8A),
    ("Task 8B", TASK8B),
    ("Task 8C", TASK8C),
    ("Task 8D", TASK8D),
    ("Task 9A", TASK9A)
]:

    print_selected_file(
        label,
        path
    )


# =============================================================================
# 5. ANALYSIS MANIFEST
# =============================================================================

manifest_rows = [
    {
        "task": "3C",
        "analysis": "Baseline physiological spectral validation",
        "role": "QC / physiological validation",
        "participants": 15,
        "status": "Retain",
        "manuscript_use": (
            "Supports neutral interpretation of Baseline A/B; "
            "does not assign undocumented eye-state labels."
        )
    },
    {
        "task": "3D",
        "analysis": "Matched baseline/task epoch construction",
        "role": "Design / QC",
        "participants": 15,
        "status": "Retain",
        "manuscript_use": (
            "Documents balanced participant-level baseline/task sampling."
        )
    },
    {
        "task": "4A",
        "analysis": "Baseline-A-normalized spectral dynamics",
        "role": "Primary between-group spectral",
        "participants": 15,
        "status": "Retain",
        "manuscript_use": (
            "Primary Active-versus-Passive spectral contrasts."
        )
    },
    {
        "task": "4B",
        "analysis": "Baseline-B spectral sensitivity",
        "role": "Sensitivity",
        "participants": 15,
        "status": "Retain",
        "manuscript_use": (
            "Evaluates dependence of spectral contrasts on baseline choice."
        )
    },
    {
        "task": "5A–6A",
        "analysis": "Legacy custom Hilbert connectivity",
        "role": "Legacy exploratory",
        "participants": 15,
        "status": "Exclude from manuscript inference",
        "manuscript_use": (
            "Not used for scientific inference because estimator "
            "implementation was superseded."
        )
    },
    {
        "task": "6C",
        "analysis": "Validated matched Baseline A/B connectivity",
        "role": "Primary validated connectivity",
        "participants": 15,
        "status": "Retain",
        "manuscript_use": (
            "Global theta/alpha/beta wPLI²_debiased sensitivity analysis."
        )
    },
    {
        "task": "7A–7B",
        "analysis": "Validated network localization/topology",
        "role": "Secondary network analysis",
        "participants": 15,
        "status": "Retain",
        "manuscript_use": (
            "Anatomically constrained localization and weighted topology."
        )
    },
    {
        "task": "8A",
        "analysis": "Active marker-phase structural audit",
        "role": "Design audit",
        "participants": 8,
        "status": "Retain",
        "manuscript_use": (
            "Justifies first complete T1_V1→T1_V2→T1_V3→T1_V4 cycle."
        )
    },
    {
        "task": "8B",
        "analysis": "Frozen first-cycle epoch construction",
        "role": "Design / QC",
        "participants": 8,
        "status": "Retain",
        "manuscript_use": (
            "Nine clean 2-s epochs per marker-defined phase."
        )
    },
    {
        "task": "8C",
        "analysis": "Within-Active spectral phase dynamics",
        "role": "Secondary within-Active spectral",
        "participants": 8,
        "status": "Retain",
        "manuscript_use": (
            "Sequential marker-defined spectral dynamics; "
            "phase order remains temporally confounded."
        )
    },
    {
        "task": "8D",
        "analysis": "Repeated-cycle feasibility audit",
        "role": "Design robustness audit",
        "participants": 8,
        "status": "Retain",
        "manuscript_use": (
            "Documents why all-participant second-cycle replication "
            "was not forced."
        )
    },
    {
        "task": "9A",
        "analysis": "Targeted V1–V4 theta connectivity",
        "role": "Targeted within-Active follow-up",
        "participants": 8,
        "status": "Retain",
        "manuscript_use": (
            "Tests whether first-cycle theta spectral dynamics correspond "
            "to theta wPLI²_debiased change."
        )
    }
]


manifest = pd.DataFrame(
    manifest_rows
)


display(
    manifest
)


safe_to_csv(
    manifest,
    "table_analysis_manifest.csv"
)


# =============================================================================
# 6. FROZEN STUDY DESIGN TABLE
# =============================================================================

study_design = pd.DataFrame(
    [
        {
            "Characteristic": "Total participants",
            "Value": "15"
        },
        {
            "Characteristic": "Active group",
            "Value": "8"
        },
        {
            "Characteristic": "Passive group",
            "Value": "7"
        },
        {
            "Characteristic": "EEG channels",
            "Value": "14"
        },
        {
            "Characteristic": "Sampling frequency",
            "Value": "256 Hz"
        },
        {
            "Characteristic": "Primary preprocessing",
            "Value": (
                "1–45 Hz filtering; average reference; "
                "no primary ICA component removal; no interpolation"
            )
        },
        {
            "Characteristic": "Artifact rejection",
            "Value": (
                "2-s phase-contained epochs; "
                "|amplitude| >250 µV or peak-to-peak >500 µV rejected"
            )
        },
        {
            "Characteristic": "Between-group matched sampling",
            "Value": (
                "Up to 15 clean 2-s epochs per baseline/task condition "
                "for spectral analysis"
            )
        },
        {
            "Characteristic": "Validated connectivity sampling",
            "Value": (
                "Fixed 9 clean 2-s epochs (18 s) per state"
            )
        },
        {
            "Characteristic": "Connectivity estimator",
            "Value": CONNECTIVITY_ESTIMATOR
        },
        {
            "Characteristic": "Connectivity spectral estimation",
            "Value": (
                "Multitaper; 2-Hz bandwidth; "
                "mt_adaptive=False; mt_low_bias=True"
            )
        },
        {
            "Characteristic": "Within-Active first-cycle sampling",
            "Value": (
                "9 clean 2-s epochs per T1_V1, T1_V2, T1_V3, T1_V4"
            )
        },
        {
            "Characteristic": "Marker semantics",
            "Value": (
                "Neutral marker labels retained because experimental "
                "meanings were not sufficiently documented"
            )
        }
    ]
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 1 — FROZEN ANALYTICAL DESIGN"
)

print(
    "=" * 120
)


display(
    study_design
)


safe_to_csv(
    study_design,
    "table1_frozen_analytical_design.csv"
)


# =============================================================================
# 7. TASK 4A — LOCATE PRIMARY SPECTRAL INFERENCE
# =============================================================================

task4a_path, task4a_raw = choose_csv(
    TASK4A,
    include_any=[
        "inference",
        "statistics",
        "stats",
        "contrast",
        "results"
    ],
    exclude=[
        "participant",
        "subject",
        "rank"
    ],
    min_rows=9
)


if task4a_raw is None:

    # Broader fallback: find a CSV with >=9 rows and p/effect columns.
    for candidate in all_csvs(
        TASK4A
    ):

        try:
            temp = pd.read_csv(
                candidate
            )
        except Exception:
            continue

        p_col = find_column(
            temp,
            [
                "p",
                "p_value",
                "p_exact",
                "pvalue"
            ]
        )

        effect_col = find_column(
            temp,
            [
                "hedges_g",
                "g",
                "effect_size"
            ]
        )

        if (
            len(temp) >= 9
            and
            p_col is not None
            and
            effect_col is not None
        ):

            task4a_path = candidate
            task4a_raw = temp
            break


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 4A SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 4A inference CSV",
    task4a_path
)


# =============================================================================
# 8. STANDARDIZE TASK 4A
# =============================================================================

spectral_a = pd.DataFrame()


if task4a_raw is not None:

    spectral_a = pd.DataFrame(
        {
            "Band":
                text_series(
                    task4a_raw,
                    [
                        "band",
                        "frequency_band"
                    ]
                ),

            "ROI":
                text_series(
                    task4a_raw,
                    [
                        "roi",
                        "region",
                        "network"
                    ]
                ),

            "Active_minus_Passive":
                numeric_series(
                    task4a_raw,
                    [
                        "active_minus_passive",
                        "mean_difference",
                        "difference",
                        "diff"
                    ]
                ),

            "Hedges_g":
                numeric_series(
                    task4a_raw,
                    [
                        "hedges_g",
                        "g",
                        "effect_size"
                    ]
                ),

            "CI_low":
                numeric_series(
                    task4a_raw,
                    [
                        "ci_low",
                        "bootstrap_ci_low",
                        "ci_lower",
                        "lower_ci"
                    ]
                ),

            "CI_high":
                numeric_series(
                    task4a_raw,
                    [
                        "ci_high",
                        "bootstrap_ci_high",
                        "ci_upper",
                        "upper_ci"
                    ]
                ),

            "p_raw":
                numeric_series(
                    task4a_raw,
                    [
                        "p_exact",
                        "p",
                        "p_value",
                        "pvalue"
                    ]
                ),

            "p_FDR":
                numeric_series(
                    task4a_raw,
                    [
                        "p_fdr",
                        "p_fdr_9",
                        "fdr_p",
                        "p_adjusted",
                        "q_value"
                    ]
                )
        }
    )


    spectral_a[
        "Baseline"
    ] = "Baseline A"


# =============================================================================
# 9. TASK 4B — LOCATE BASELINE-B SPECTRAL INFERENCE
# =============================================================================

task4b_path, task4b_raw = choose_csv(
    TASK4B,
    include_any=[
        "inference",
        "statistics",
        "stats",
        "contrast",
        "results",
        "baseline"
    ],
    exclude=[
        "participant",
        "subject"
    ],
    min_rows=9
)


if task4b_raw is None:

    for candidate in all_csvs(
        TASK4B
    ):

        try:
            temp = pd.read_csv(
                candidate
            )
        except Exception:
            continue

        p_col = find_column(
            temp,
            [
                "p",
                "p_value",
                "p_exact"
            ]
        )

        effect_col = find_column(
            temp,
            [
                "hedges_g",
                "g",
                "effect_size"
            ]
        )

        if (
            len(temp) >= 9
            and
            p_col is not None
            and
            effect_col is not None
        ):

            task4b_path = candidate
            task4b_raw = temp
            break


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 4B SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 4B inference CSV",
    task4b_path
)


spectral_b = pd.DataFrame()


if task4b_raw is not None:

    spectral_b = pd.DataFrame(
        {
            "Band":
                text_series(
                    task4b_raw,
                    [
                        "band",
                        "frequency_band"
                    ]
                ),

            "ROI":
                text_series(
                    task4b_raw,
                    [
                        "roi",
                        "region",
                        "network"
                    ]
                ),

            "Active_minus_Passive":
                numeric_series(
                    task4b_raw,
                    [
                        "active_minus_passive",
                        "mean_difference",
                        "difference",
                        "diff"
                    ]
                ),

            "Hedges_g":
                numeric_series(
                    task4b_raw,
                    [
                        "hedges_g",
                        "g",
                        "effect_size"
                    ]
                ),

            "CI_low":
                numeric_series(
                    task4b_raw,
                    [
                        "ci_low",
                        "bootstrap_ci_low",
                        "ci_lower",
                        "lower_ci"
                    ]
                ),

            "CI_high":
                numeric_series(
                    task4b_raw,
                    [
                        "ci_high",
                        "bootstrap_ci_high",
                        "ci_upper",
                        "upper_ci"
                    ]
                ),

            "p_raw":
                numeric_series(
                    task4b_raw,
                    [
                        "p_exact",
                        "p",
                        "p_value",
                        "pvalue"
                    ]
                ),

            "p_FDR":
                numeric_series(
                    task4b_raw,
                    [
                        "p_fdr",
                        "p_fdr_9",
                        "fdr_p",
                        "p_adjusted",
                        "q_value"
                    ]
                )
        }
    )


    spectral_b[
        "Baseline"
    ] = "Baseline B"


# =============================================================================
# 10. CONSOLIDATE BETWEEN-GROUP SPECTRAL RESULTS
# =============================================================================

spectral_frames = [
    frame
    for frame in [
        spectral_a,
        spectral_b
    ]
    if len(frame) > 0
]


if spectral_frames:

    spectral_between = pd.concat(
        spectral_frames,
        ignore_index=True
    )


    spectral_between[
        "Band"
    ] = (
        spectral_between[
            "Band"
        ]
        .str.strip()
        .str.title()
    )


    spectral_between[
        "ROI"
    ] = spectral_between[
        "ROI"
    ].map(
        clean_feature_name
    )


    spectral_between[
        "Evidence_status"
    ] = [
        bh_status(
            raw,
            corrected
        )
        for raw, corrected in zip(
            spectral_between[
                "p_raw"
            ],
            spectral_between[
                "p_FDR"
            ]
        )
    ]


    spectral_between = spectral_between[
        [
            "Baseline",
            "Band",
            "ROI",
            "Active_minus_Passive",
            "Hedges_g",
            "CI_low",
            "CI_high",
            "p_raw",
            "p_FDR",
            "Evidence_status"
        ]
    ]


    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TABLE 2 — BETWEEN-GROUP SPECTRAL RESULTS"
    )

    print(
        "=" * 120
    )


    display(
        spectral_between.round(
            6
        )
    )


    safe_to_csv(
        spectral_between,
        "table2_between_group_spectral_results.csv"
    )


else:

    spectral_between = pd.DataFrame()

    warnings.warn(
        "Task 4A/4B spectral inference CSVs were not automatically "
        "identified. Other Task 10A tables will still be generated."
    )


# =============================================================================
# 11. TASK 6C — VALIDATED GLOBAL CONNECTIVITY
# =============================================================================

task6c_path, task6c_raw = choose_csv(
    TASK6C,
    include_any=[
        "inference",
        "global",
        "baseline",
        "sensitivity",
        "statistics"
    ],
    exclude=[
        "participant",
        "subject",
        "loo",
        "leave"
    ],
    min_rows=6
)


if task6c_raw is None:

    for candidate in all_csvs(
        TASK6C
    ):

        try:
            temp = pd.read_csv(
                candidate
            )
        except Exception:
            continue

        if len(temp) < 6:
            continue

        band_col = find_column(
            temp,
            [
                "band",
                "frequency_band"
            ]
        )

        baseline_col = find_column(
            temp,
            [
                "baseline",
                "baseline_label"
            ]
        )

        p_col = find_column(
            temp,
            [
                "p_exact",
                "p",
                "p_value"
            ]
        )

        if (
            band_col is not None
            and
            p_col is not None
        ):

            task6c_path = candidate
            task6c_raw = temp
            break


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 6C SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 6C inference CSV",
    task6c_path
)


connectivity_global = pd.DataFrame()


if task6c_raw is not None:

    connectivity_global = pd.DataFrame(
        {
            "Baseline":
                text_series(
                    task6c_raw,
                    [
                        "baseline",
                        "baseline_label",
                        "baseline_condition"
                    ]
                ),

            "Band":
                text_series(
                    task6c_raw,
                    [
                        "band",
                        "frequency_band"
                    ]
                ),

            "Active_minus_Passive":
                numeric_series(
                    task6c_raw,
                    [
                        "active_minus_passive",
                        "difference",
                        "diff",
                        "mean_difference"
                    ]
                ),

            "Hedges_g":
                numeric_series(
                    task6c_raw,
                    [
                        "hedges_g",
                        "g",
                        "effect_size"
                    ]
                ),

            "CI_low":
                numeric_series(
                    task6c_raw,
                    [
                        "ci_low",
                        "bootstrap_ci_low",
                        "lower_ci"
                    ]
                ),

            "CI_high":
                numeric_series(
                    task6c_raw,
                    [
                        "ci_high",
                        "bootstrap_ci_high",
                        "upper_ci"
                    ]
                ),

            "p_exact":
                numeric_series(
                    task6c_raw,
                    [
                        "p_exact",
                        "p",
                        "p_value"
                    ]
                ),

            "p_FDR":
                numeric_series(
                    task6c_raw,
                    [
                        "p_fdr",
                        "p_fdr_3",
                        "fdr_p",
                        "q_value"
                    ]
                )
        }
    )


    connectivity_global[
        "Band"
    ] = (
        connectivity_global[
            "Band"
        ]
        .str.strip()
        .str.title()
    )


    connectivity_global[
        "Baseline"
    ] = (
        connectivity_global[
            "Baseline"
        ]
        .str.replace(
            "_",
            " ",
            regex=False
        )
        .str.strip()
    )


    connectivity_global[
        "Evidence_status"
    ] = [
        bh_status(
            raw,
            corrected
        )
        for raw, corrected in zip(
            connectivity_global[
                "p_exact"
            ],
            connectivity_global[
                "p_FDR"
            ]
        )
    ]


    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TABLE 3 — VALIDATED GLOBAL wPLI²_debiased CONNECTIVITY"
    )

    print(
        "=" * 120
    )


    display(
        connectivity_global.round(
            6
        )
    )


    safe_to_csv(
        connectivity_global,
        "table3_validated_global_connectivity.csv"
    )


else:

    warnings.warn(
        "Task 6C global connectivity inference table not automatically found."
    )


# =============================================================================
# 12. TASK 7B — VALIDATED WEIGHTED NETWORK TOPOLOGY
# =============================================================================

task7b_path, task7b_raw = choose_csv(
    TASK7B,
    include_any=[
        "inference",
        "topology",
        "roi",
        "node",
        "strength",
        "results"
    ],
    exclude=[
        "participant",
        "subject",
        "loo",
        "leave"
    ],
    min_rows=9
)


if task7b_raw is None:

    for candidate in all_csvs(
        TASK7B
    ):

        try:
            temp = pd.read_csv(
                candidate
            )
        except Exception:
            continue

        if len(temp) < 9:
            continue

        roi_col = find_column(
            temp,
            [
                "roi",
                "region"
            ]
        )

        p_col = find_column(
            temp,
            [
                "p_exact",
                "p",
                "p_value"
            ]
        )

        if (
            roi_col is not None
            and
            p_col is not None
        ):

            task7b_path = candidate
            task7b_raw = temp
            break


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 7B SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 7B inference CSV",
    task7b_path
)


topology = pd.DataFrame()


if task7b_raw is not None:

    topology = pd.DataFrame(
        {
            "Baseline":
                text_series(
                    task7b_raw,
                    [
                        "baseline",
                        "baseline_label"
                    ]
                ),

            "Band":
                text_series(
                    task7b_raw,
                    [
                        "band",
                        "frequency_band"
                    ]
                ),

            "ROI":
                text_series(
                    task7b_raw,
                    [
                        "roi",
                        "region"
                    ]
                ),

            "Metric":
                text_series(
                    task7b_raw,
                    [
                        "metric",
                        "measure",
                        "strength_type",
                        "topology_metric"
                    ]
                ),

            "Active_minus_Passive":
                numeric_series(
                    task7b_raw,
                    [
                        "active_minus_passive",
                        "difference",
                        "diff",
                        "mean_difference"
                    ]
                ),

            "Hedges_g":
                numeric_series(
                    task7b_raw,
                    [
                        "hedges_g",
                        "g",
                        "effect_size"
                    ]
                ),

            "p_exact":
                numeric_series(
                    task7b_raw,
                    [
                        "p_exact",
                        "p",
                        "p_value"
                    ]
                ),

            "p_FDR":
                numeric_series(
                    task7b_raw,
                    [
                        "p_fdr",
                        "p_fdr_18",
                        "p_fdr_9",
                        "fdr_p",
                        "q_value"
                    ]
                )
        }
    )


    topology[
        "Band"
    ] = (
        topology[
            "Band"
        ]
        .str.strip()
        .str.title()
    )


    topology[
        "ROI"
    ] = topology[
        "ROI"
    ].map(
        clean_feature_name
    )


    topology[
        "Evidence_status"
    ] = [
        bh_status(
            raw,
            corrected
        )
        for raw, corrected in zip(
            topology[
                "p_exact"
            ],
            topology[
                "p_FDR"
            ]
        )
    ]


    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TABLE 4 — VALIDATED WEIGHTED NETWORK TOPOLOGY"
    )

    print(
        "=" * 120
    )


    display(
        topology.round(
            6
        )
    )


    safe_to_csv(
        topology,
        "table4_validated_network_topology.csv"
    )


else:

    warnings.warn(
        "Task 7B topology inference table not automatically found."
    )


# =============================================================================
# 13. TASK 8C — LOCATE WITHIN-ACTIVE OMNIBUS RESULTS
# =============================================================================

task8c_omnibus_path, task8c_omnibus_raw = choose_csv(
    TASK8C,
    include_any=[
        "omnibus",
        "friedman"
    ],
    min_rows=9
)


if task8c_omnibus_raw is None:

    for candidate in all_csvs(
        TASK8C
    ):

        try:
            temp = pd.read_csv(
                candidate
            )
        except Exception:
            continue

        friedman_col = find_column(
            temp,
            [
                "friedman_p",
                "p_friedman",
                "friedman_chi2",
                "kendall_w"
            ]
        )

        if (
            len(temp) == 9
            and
            friedman_col is not None
        ):

            task8c_omnibus_path = candidate
            task8c_omnibus_raw = temp
            break


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 8C OMNIBUS SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 8C omnibus CSV",
    task8c_omnibus_path
)


within_active_omnibus = pd.DataFrame()


if task8c_omnibus_raw is not None:

    within_active_omnibus = pd.DataFrame(
        {
            "Band":
                text_series(
                    task8c_omnibus_raw,
                    [
                        "band",
                        "frequency_band"
                    ]
                ),

            "ROI":
                text_series(
                    task8c_omnibus_raw,
                    [
                        "roi",
                        "region"
                    ]
                ),

            "Friedman_chi2":
                numeric_series(
                    task8c_omnibus_raw,
                    [
                        "friedman_chi2",
                        "friedman_statistic",
                        "chi2"
                    ]
                ),

            "Kendall_W":
                numeric_series(
                    task8c_omnibus_raw,
                    [
                        "kendall_w",
                        "kendalls_w",
                        "w"
                    ]
                ),

            "p_raw":
                numeric_series(
                    task8c_omnibus_raw,
                    [
                        "friedman_p",
                        "p_friedman",
                        "p",
                        "p_value"
                    ]
                ),

            "p_FDR_9":
                numeric_series(
                    task8c_omnibus_raw,
                    [
                        "p_fdr_9",
                        "p_fdr",
                        "fdr_p",
                        "q_value"
                    ]
                )
        }
    )


    within_active_omnibus[
        "Band"
    ] = (
        within_active_omnibus[
            "Band"
        ]
        .str.strip()
        .str.title()
    )


    within_active_omnibus[
        "ROI"
    ] = within_active_omnibus[
        "ROI"
    ].map(
        clean_feature_name
    )


    within_active_omnibus[
        "Evidence_status"
    ] = [
        bh_status(
            raw,
            corrected
        )
        for raw, corrected in zip(
            within_active_omnibus[
                "p_raw"
            ],
            within_active_omnibus[
                "p_FDR_9"
            ]
        )
    ]


    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TABLE 5A — WITHIN-ACTIVE SPECTRAL OMNIBUS RESULTS"
    )

    print(
        "=" * 120
    )


    display(
        within_active_omnibus.round(
            6
        )
    )


    safe_to_csv(
        within_active_omnibus,
        "table5a_within_active_spectral_omnibus.csv"
    )


else:

    warnings.warn(
        "Task 8C omnibus table not automatically found."
    )


# =============================================================================
# 14. TASK 8C — PAIRWISE RESULTS
# =============================================================================

task8c_pairwise_path, task8c_pairwise_raw = choose_csv(
    TASK8C,
    include_any=[
        "pairwise",
        "contrast"
    ],
    min_rows=6
)


if task8c_pairwise_raw is None:

    for candidate in all_csvs(
        TASK8C
    ):

        try:
            temp = pd.read_csv(
                candidate
            )
        except Exception:
            continue

        phase1_col = find_column(
            temp,
            [
                "phase1",
                "phase_a",
                "condition1",
                "condition_a"
            ]
        )

        phase2_col = find_column(
            temp,
            [
                "phase2",
                "phase_b",
                "condition2",
                "condition_b"
            ]
        )

        if (
            phase1_col is not None
            and
            phase2_col is not None
        ):

            task8c_pairwise_path = candidate
            task8c_pairwise_raw = temp
            break


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 8C PAIRWISE SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 8C pairwise CSV",
    task8c_pairwise_path
)


within_active_pairwise = pd.DataFrame()


if task8c_pairwise_raw is not None:

    within_active_pairwise = pd.DataFrame(
        {
            "Band":
                text_series(
                    task8c_pairwise_raw,
                    [
                        "band",
                        "frequency_band"
                    ]
                ),

            "ROI":
                text_series(
                    task8c_pairwise_raw,
                    [
                        "roi",
                        "region"
                    ]
                ),

            "Phase_1":
                text_series(
                    task8c_pairwise_raw,
                    [
                        "phase1",
                        "phase_a",
                        "condition1",
                        "condition_a"
                    ]
                ),

            "Phase_2":
                text_series(
                    task8c_pairwise_raw,
                    [
                        "phase2",
                        "phase_b",
                        "condition2",
                        "condition_b"
                    ]
                ),

            "Mean_difference":
                numeric_series(
                    task8c_pairwise_raw,
                    [
                        "mean_difference",
                        "difference",
                        "mean_delta",
                        "delta"
                    ]
                ),

            "Paired_dz":
                numeric_series(
                    task8c_pairwise_raw,
                    [
                        "paired_dz",
                        "dz",
                        "effect_size"
                    ]
                ),

            "CI_low":
                numeric_series(
                    task8c_pairwise_raw,
                    [
                        "bootstrap_ci_low",
                        "ci_low",
                        "lower_ci"
                    ]
                ),

            "CI_high":
                numeric_series(
                    task8c_pairwise_raw,
                    [
                        "bootstrap_ci_high",
                        "ci_high",
                        "upper_ci"
                    ]
                ),

            "p_exact":
                numeric_series(
                    task8c_pairwise_raw,
                    [
                        "p_exact",
                        "p",
                        "p_value"
                    ]
                ),

            "p_FDR_6":
                numeric_series(
                    task8c_pairwise_raw,
                    [
                        "p_fdr_6",
                        "p_fdr",
                        "fdr_p",
                        "q_value"
                    ]
                )
        }
    )


    within_active_pairwise[
        "Band"
    ] = (
        within_active_pairwise[
            "Band"
        ]
        .str.strip()
        .str.title()
    )


    within_active_pairwise[
        "ROI"
    ] = within_active_pairwise[
        "ROI"
    ].map(
        clean_feature_name
    )


    # Determine whether corresponding omnibus feature survived correction.
    omnibus_supported = {}


    if len(
        within_active_omnibus
    ) > 0:

        for _, row in within_active_omnibus.iterrows():

            key = (
                str(
                    row[
                        "Band"
                    ]
                ),
                str(
                    row[
                        "ROI"
                    ]
                )
            )

            omnibus_supported[
                key
            ] = bool(
                np.isfinite(
                    row[
                        "p_FDR_9"
                    ]
                )
                and
                row[
                    "p_FDR_9"
                ]
                <
                0.05
            )


    pairwise_status = []


    for _, row in within_active_pairwise.iterrows():

        key = (
            str(
                row[
                    "Band"
                ]
            ),
            str(
                row[
                    "ROI"
                ]
            )
        )


        omnibus_ok = omnibus_supported.get(
            key,
            False
        )


        pair_fdr = row[
            "p_FDR_6"
        ]


        pair_raw = row[
            "p_exact"
        ]


        if (
            omnibus_ok
            and
            np.isfinite(
                pair_fdr
            )
            and
            pair_fdr < 0.05
        ):

            status = (
                "Supported pairwise contrast "
                "following corrected omnibus"
            )

        elif (
            np.isfinite(
                pair_fdr
            )
            and
            pair_fdr < 0.05
        ):

            status = (
                "Pairwise FDR<.05 but omnibus not corrected-significant; "
                "interpret cautiously"
            )

        elif (
            np.isfinite(
                pair_raw
            )
            and
            pair_raw < 0.05
        ):

            status = (
                "Raw pairwise p<.05 only; exploratory"
            )

        else:

            status = (
                "Not statistically supported"
            )


        pairwise_status.append(
            status
        )


    within_active_pairwise[
        "Evidence_status"
    ] = pairwise_status


    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TABLE 5B — WITHIN-ACTIVE SPECTRAL PAIRWISE RESULTS"
    )

    print(
        "=" * 120
    )


    display(
        within_active_pairwise.round(
            6
        )
    )


    safe_to_csv(
        within_active_pairwise,
        "table5b_within_active_spectral_pairwise.csv"
    )


else:

    warnings.warn(
        "Task 8C pairwise table not automatically found."
    )


# =============================================================================
# 15. TASK 9A — TARGETED THETA CONNECTIVITY
# =============================================================================

task9a_path = (
    TASK9A
    / "targeted_theta_connectivity_inference.csv"
    if TASK9A is not None
    else None
)


if (
    task9a_path is not None
    and
    task9a_path.exists()
):

    task9a_raw = pd.read_csv(
        task9a_path
    )

else:

    task9a_path, task9a_raw = choose_csv(
        TASK9A,
        include_all=[
            "theta",
            "connectivity",
            "inference"
        ],
        min_rows=4
    )


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 9A SOURCE"
)

print(
    "=" * 120
)

print_selected_file(
    "Task 9A inference CSV",
    task9a_path
)


targeted_theta = pd.DataFrame()


if task9a_raw is not None:

    targeted_theta = pd.DataFrame(
        {
            "Feature":
                text_series(
                    task9a_raw,
                    [
                        "feature"
                    ]
                ),

            "N":
                numeric_series(
                    task9a_raw,
                    [
                        "n"
                    ]
                ),

            "T1_V1_mean":
                numeric_series(
                    task9a_raw,
                    [
                        "T1_V1_mean"
                    ]
                ),

            "T1_V4_mean":
                numeric_series(
                    task9a_raw,
                    [
                        "T1_V4_mean"
                    ]
                ),

            "Delta_V4_minus_V1":
                numeric_series(
                    task9a_raw,
                    [
                        "mean_delta_V4_minus_V1",
                        "delta_V4_minus_V1"
                    ]
                ),

            "CI_low":
                numeric_series(
                    task9a_raw,
                    [
                        "bootstrap_CI_low",
                        "ci_low"
                    ]
                ),

            "CI_high":
                numeric_series(
                    task9a_raw,
                    [
                        "bootstrap_CI_high",
                        "ci_high"
                    ]
                ),

            "Paired_dz":
                numeric_series(
                    task9a_raw,
                    [
                        "paired_dz",
                        "dz"
                    ]
                ),

            "p_exact":
                numeric_series(
                    task9a_raw,
                    [
                        "p_exact"
                    ]
                ),

            "p_FDR_4":
                numeric_series(
                    task9a_raw,
                    [
                        "p_FDR_4",
                        "p_fdr"
                    ]
                ),

            "Positive":
                numeric_series(
                    task9a_raw,
                    [
                        "participants_delta_positive"
                    ]
                ),

            "Negative":
                numeric_series(
                    task9a_raw,
                    [
                        "participants_delta_negative"
                    ]
                )
        }
    )


    targeted_theta[
        "Feature"
    ] = targeted_theta[
        "Feature"
    ].map(
        clean_feature_name
    )


    targeted_theta[
        "Evidence_status"
    ] = [
        bh_status(
            raw,
            corrected
        )
        for raw, corrected in zip(
            targeted_theta[
                "p_exact"
            ],
            targeted_theta[
                "p_FDR_4"
            ]
        )
    ]


    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TABLE 6 — TARGETED WITHIN-ACTIVE THETA CONNECTIVITY"
    )

    print(
        "=" * 120
    )


    display(
        targeted_theta.round(
            6
        )
    )


    safe_to_csv(
        targeted_theta,
        "table6_targeted_within_active_theta_connectivity.csv"
    )


else:

    raise FileNotFoundError(
        "Task 9A inference CSV could not be located."
    )


# =============================================================================
# 16. TASK 9A LOO ROBUSTNESS
# =============================================================================

task9a_loo_path = (
    TASK9A
    / "theta_connectivity_leave_one_out_summary.csv"
    if TASK9A is not None
    else None
)


if (
    task9a_loo_path is not None
    and
    task9a_loo_path.exists()
):

    task9a_loo = pd.read_csv(
        task9a_loo_path
    )

else:

    task9a_loo_path, task9a_loo = choose_csv(
        TASK9A,
        include_all=[
            "leave",
            "summary"
        ],
        min_rows=4
    )


if task9a_loo is not None:

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "TASK 9A LEAVE-ONE-OUT ROBUSTNESS"
    )

    print(
        "=" * 120
    )


    display(
        task9a_loo.round(
            6
        )
    )


    safe_to_csv(
        task9a_loo,
        "supplement_task9a_leave_one_out.csv"
    )


# =============================================================================
# 17. TASK 8D — REPEATED-DESIGN FEASIBILITY
# =============================================================================

task8d_path = (
    TASK8D
    / "repeated_design_feasibility.csv"
    if TASK8D is not None
    else None
)


if (
    task8d_path is not None
    and
    task8d_path.exists()
):

    task8d_design = pd.read_csv(
        task8d_path
    )

else:

    task8d_path, task8d_design = choose_csv(
        TASK8D,
        include_all=[
            "design",
            "feasibility"
        ],
        min_rows=1
    )


if task8d_design is not None:

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "REPEATED-CYCLE FEASIBILITY"
    )

    print(
        "=" * 120
    )


    display(
        task8d_design
    )


    safe_to_csv(
        task8d_design,
        "supplement_repeated_cycle_feasibility.csv"
    )


# =============================================================================
# 18. BUILD MANUSCRIPT-FACING KEY FINDINGS TABLE
# =============================================================================

key_findings = []


# -----------------------------------------------------------------------------
# Between-group spectral
# -----------------------------------------------------------------------------

if len(
    spectral_between
) > 0:

    n_spectral = len(
        spectral_between
    )

    n_spectral_fdr = int(
        (
            spectral_between[
                "p_FDR"
            ]
            <
            0.05
        ).fillna(
            False
        ).sum()
    )


    # Largest absolute effect for each baseline.
    for baseline in spectral_between[
        "Baseline"
    ].dropna().unique():

        temp = spectral_between.loc[
            spectral_between[
                "Baseline"
            ]
            ==
            baseline
        ].copy()


        temp = temp.loc[
            np.isfinite(
                temp[
                    "Hedges_g"
                ]
            )
        ]


        if len(temp) == 0:
            continue


        row = temp.loc[
            temp[
                "Hedges_g"
            ].abs().idxmax()
        ]


        key_findings.append(
            {
                "Domain":
                    "Between-group spectral",

                "Analysis":
                    baseline,

                "Main_result":
                    (
                        f"Largest descriptive effect: "
                        f"{row['Band']} / {row['ROI']}, "
                        f"g={format_effect(row['Hedges_g'])}, "
                        f"p={format_p(row['p_raw'])}"
                    ),

                "Corrected_support":
                    (
                        "Yes"
                        if (
                            np.isfinite(
                                row[
                                    "p_FDR"
                                ]
                            )
                            and
                            row[
                                "p_FDR"
                            ]
                            <
                            0.05
                        )
                        else
                        "No"
                    ),

                "Interpretation":
                    (
                        "Descriptive unless multiplicity-corrected."
                    )
            }
        )


# -----------------------------------------------------------------------------
# Global validated connectivity
# -----------------------------------------------------------------------------

if len(
    connectivity_global
) > 0:

    temp = connectivity_global.loc[
        np.isfinite(
            connectivity_global[
                "Hedges_g"
            ]
        )
    ].copy()


    if len(temp) > 0:

        row = temp.loc[
            temp[
                "Hedges_g"
            ].abs().idxmax()
        ]


        key_findings.append(
            {
                "Domain":
                    "Between-group connectivity",

                "Analysis":
                    "Validated global wPLI²_debiased",

                "Main_result":
                    (
                        f"Largest descriptive global effect: "
                        f"{row['Band']} ({row['Baseline']}), "
                        f"g={format_effect(row['Hedges_g'])}, "
                        f"p={format_p(row['p_exact'])}"
                    ),

                "Corrected_support":
                    (
                        "Yes"
                        if (
                            np.isfinite(
                                row[
                                    "p_FDR"
                                ]
                            )
                            and
                            row[
                                "p_FDR"
                            ]
                            <
                            0.05
                        )
                        else
                        "No"
                    ),

                "Interpretation":
                    (
                        "No confirmed global connectivity difference; "
                        "evaluate baseline-directional robustness."
                    )
            }
        )


# -----------------------------------------------------------------------------
# Network topology
# -----------------------------------------------------------------------------

if len(
    topology
) > 0:

    finite_topology = topology.loc[
        np.isfinite(
            topology[
                "Hedges_g"
            ]
        )
    ].copy()


    if len(
        finite_topology
    ) > 0:

        row = finite_topology.loc[
            finite_topology[
                "Hedges_g"
            ].abs().idxmax()
        ]


        key_findings.append(
            {
                "Domain":
                    "Network topology",

                "Analysis":
                    "Validated weighted topology",

                "Main_result":
                    (
                        f"Largest descriptive topology effect: "
                        f"{row['Band']} / {row['ROI']}, "
                        f"g={format_effect(row['Hedges_g'])}, "
                        f"p={format_p(row['p_exact'])}"
                    ),

                "Corrected_support":
                    (
                        "Yes"
                        if (
                            np.isfinite(
                                row[
                                    "p_FDR"
                                ]
                            )
                            and
                            row[
                                "p_FDR"
                            ]
                            <
                            0.05
                        )
                        else
                        "No"
                    ),

                "Interpretation":
                    (
                        "No multiplicity-corrected topology claim unless "
                        "explicitly supported."
                    )
            }
        )


# -----------------------------------------------------------------------------
# Within-Active spectral
# -----------------------------------------------------------------------------

if len(
    within_active_omnibus
) > 0:

    finite_w = within_active_omnibus.loc[
        np.isfinite(
            within_active_omnibus[
                "Kendall_W"
            ]
        )
    ].copy()


    if len(
        finite_w
    ) > 0:

        row = finite_w.loc[
            finite_w[
                "Kendall_W"
            ].abs().idxmax()
        ]


        key_findings.append(
            {
                "Domain":
                    "Within-Active spectral dynamics",

                "Analysis":
                    "Four sequential marker-defined phases",

                "Main_result":
                    (
                        f"Largest omnibus phase effect: "
                        f"{row['Band']} / {row['ROI']}, "
                        f"Kendall W={row['Kendall_W']:.3f}, "
                        f"p={format_p(row['p_raw'])}"
                    ),

                "Corrected_support":
                    (
                        "Yes"
                        if (
                            np.isfinite(
                                row[
                                    "p_FDR_9"
                                ]
                            )
                            and
                            row[
                                "p_FDR_9"
                            ]
                            <
                            0.05
                        )
                        else
                        "No"
                    ),

                "Interpretation":
                    (
                        "Sequential marker-phase effect; marker identity "
                        "cannot be separated from temporal order."
                    )
            }
        )


# -----------------------------------------------------------------------------
# Targeted Task 9A
# -----------------------------------------------------------------------------

if len(
    targeted_theta
) > 0:

    finite_dz = targeted_theta.loc[
        np.isfinite(
            targeted_theta[
                "Paired_dz"
            ]
        )
    ].copy()


    if len(
        finite_dz
    ) > 0:

        row = finite_dz.loc[
            finite_dz[
                "Paired_dz"
            ].abs().idxmax()
        ]


        key_findings.append(
            {
                "Domain":
                    "Within-Active theta connectivity",

                "Analysis":
                    "Targeted T1_V4 − T1_V1",

                "Main_result":
                    (
                        f"Largest descriptive effect: "
                        f"{row['Feature']}, "
                        f"dz={format_effect(row['Paired_dz'])}, "
                        f"p={format_p(row['p_exact'])}, "
                        f"FDR={format_p(row['p_FDR_4'])}"
                    ),

                "Corrected_support":
                    (
                        "Yes"
                        if (
                            np.isfinite(
                                row[
                                    "p_FDR_4"
                                ]
                            )
                            and
                            row[
                                "p_FDR_4"
                            ]
                            <
                            0.05
                        )
                        else
                        "No"
                    ),

                "Interpretation":
                    (
                        "Targeted connectivity follow-up; null unless "
                        "multiplicity-corrected."
                    )
            }
        )


key_findings = pd.DataFrame(
    key_findings
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 7 — CONSOLIDATED KEY FINDINGS"
)

print(
    "=" * 120
)


display(
    key_findings
)


safe_to_csv(
    key_findings,
    "table7_consolidated_key_findings.csv"
)


# =============================================================================
# 19. EXPLICIT FROZEN RESULTS FROM COMPLETED ANALYSES
# =============================================================================
#
# These values are entered only as a cross-check against the previously
# completed frozen analyses. They are NOT newly calculated.
#
# This table is useful if automatic discovery finds differently named
# historical CSV columns.
# =============================================================================

frozen_crosscheck = pd.DataFrame(
    [
        # ---------------------------------------------------------------------
        # Task 4A
        # ---------------------------------------------------------------------
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Theta / Frontal",
            "Effect": -0.432,
            "p_raw": 0.394,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Theta / Central-Temporal",
            "Effect": -0.609,
            "p_raw": 0.244,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Theta / Posterior",
            "Effect": -0.272,
            "p_raw": 0.590,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Alpha / Frontal",
            "Effect": 0.397,
            "p_raw": 0.431,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Alpha / Central-Temporal",
            "Effect": -0.226,
            "p_raw": 0.650,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Alpha / Posterior",
            "Effect": 0.609,
            "p_raw": 0.237,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Beta / Frontal",
            "Effect": -0.175,
            "p_raw": 0.761,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Beta / Central-Temporal",
            "Effect": 0.784,
            "p_raw": 0.132,
            "p_FDR": np.nan,
            "Status": "Largest Task 4A descriptive effect"
        },
        {
            "Task": "4A",
            "Domain": "Spectral",
            "Baseline": "A",
            "Feature": "Beta / Posterior",
            "Effect": 0.479,
            "p_raw": 0.339,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },

        # ---------------------------------------------------------------------
        # Task 4B
        # ---------------------------------------------------------------------
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Theta / Frontal",
            "Effect": 0.385,
            "p_raw": 0.446,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Theta / Central-Temporal",
            "Effect": 0.429,
            "p_raw": 0.424,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Theta / Posterior",
            "Effect": 0.366,
            "p_raw": 0.458,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Alpha / Frontal",
            "Effect": 0.578,
            "p_raw": 0.254,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Alpha / Central-Temporal",
            "Effect": 0.546,
            "p_raw": 0.275,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Alpha / Posterior",
            "Effect": 0.390,
            "p_raw": 0.431,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Beta / Frontal",
            "Effect": 0.023,
            "p_raw": 0.959,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Beta / Central-Temporal",
            "Effect": 0.857,
            "p_raw": 0.102,
            "p_FDR": np.nan,
            "Status": "Largest Task 4B descriptive effect"
        },
        {
            "Task": "4B",
            "Domain": "Spectral",
            "Baseline": "B",
            "Feature": "Beta / Posterior",
            "Effect": 0.561,
            "p_raw": 0.265,
            "p_FDR": np.nan,
            "Status": "Not statistically supported"
        },

        # ---------------------------------------------------------------------
        # Task 6C validated connectivity
        # ---------------------------------------------------------------------
        {
            "Task": "6C",
            "Domain": "Global connectivity",
            "Baseline": "A",
            "Feature": "Theta",
            "Effect": 0.093750,
            "p_raw": 0.856410,
            "p_FDR": 0.856410,
            "Status": "Not statistically supported"
        },
        {
            "Task": "6C",
            "Domain": "Global connectivity",
            "Baseline": "A",
            "Feature": "Alpha",
            "Effect": 0.173919,
            "p_raw": 0.730847,
            "p_FDR": 0.856410,
            "Status": "Not statistically supported"
        },
        {
            "Task": "6C",
            "Domain": "Global connectivity",
            "Baseline": "A",
            "Feature": "Beta",
            "Effect": -0.636351,
            "p_raw": 0.209013,
            "p_FDR": 0.627040,
            "Status": "Directionally robust descriptive effect"
        },
        {
            "Task": "6C",
            "Domain": "Global connectivity",
            "Baseline": "B",
            "Feature": "Theta",
            "Effect": -0.036281,
            "p_raw": 0.940793,
            "p_FDR": 0.940793,
            "Status": "Not statistically supported"
        },
        {
            "Task": "6C",
            "Domain": "Global connectivity",
            "Baseline": "B",
            "Feature": "Alpha",
            "Effect": -0.482229,
            "p_raw": 0.344056,
            "p_FDR": 0.718415,
            "Status": "Baseline-sensitive descriptive effect"
        },
        {
            "Task": "6C",
            "Domain": "Global connectivity",
            "Baseline": "B",
            "Feature": "Beta",
            "Effect": -0.351672,
            "p_raw": 0.478943,
            "p_FDR": 0.718415,
            "Status": "Directionally robust descriptive effect"
        },

        # ---------------------------------------------------------------------
        # Task 9A
        # ---------------------------------------------------------------------
        {
            "Task": "9A",
            "Domain": "Within-Active theta connectivity",
            "Baseline": "",
            "Feature": "Global",
            "Effect": 0.132750,
            "p_raw": 0.718750,
            "p_FDR": 0.867188,
            "Status": "Not statistically supported"
        },
        {
            "Task": "9A",
            "Domain": "Within-Active theta connectivity",
            "Baseline": "",
            "Feature": "Frontal–Frontal",
            "Effect": 0.338013,
            "p_raw": 0.375000,
            "p_FDR": 0.750000,
            "Status": "Not statistically supported"
        },
        {
            "Task": "9A",
            "Domain": "Within-Active theta connectivity",
            "Baseline": "",
            "Feature": "Frontal–Posterior",
            "Effect": -0.404054,
            "p_raw": 0.296875,
            "p_FDR": 0.750000,
            "Status": (
                "Largest targeted descriptive effect; "
                "not statistically supported"
            )
        },
        {
            "Task": "9A",
            "Domain": "Within-Active theta connectivity",
            "Baseline": "",
            "Feature": "Posterior–Posterior",
            "Effect": 0.066262,
            "p_raw": 0.867188,
            "p_FDR": 0.867188,
            "Status": "Not statistically supported"
        }
    ]
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "FROZEN RESULT CROSS-CHECK"
)

print(
    "=" * 120
)


display(
    frozen_crosscheck
)


safe_to_csv(
    frozen_crosscheck,
    "supplement_frozen_result_crosscheck.csv"
)


# =============================================================================
# 20. MANUSCRIPT CLAIM-LEVEL EVIDENCE TABLE
# =============================================================================

claim_table = pd.DataFrame(
    [
        {
            "Claim_ID": "C1",
            "Evidence_level": "Supported physiological validation",
            "Manuscript_claim": (
                "The two marker-defined baseline intervals differed "
                "strongly in posterior alpha power, supporting their "
                "treatment as physiologically distinct baseline states."
            ),
            "Allowed_wording": (
                "Baseline B showed substantially greater posterior alpha "
                "than Baseline A."
            ),
            "Avoid": (
                "Do not definitively label the baselines eyes-open and "
                "eyes-closed without authoritative marker documentation."
            )
        },
        {
            "Claim_ID": "C2",
            "Evidence_level": "Null primary between-group inference",
            "Manuscript_claim": (
                "No ROI-band spectral contrast provided multiplicity-"
                "corrected evidence of an Active–Passive difference."
            ),
            "Allowed_wording": (
                "Between-group spectral differences were descriptive "
                "rather than statistically confirmed."
            ),
            "Avoid": (
                "Do not claim Active engagement increased/decreased a "
                "specific frequency band as an established group effect."
            )
        },
        {
            "Claim_ID": "C3",
            "Evidence_level": "Baseline sensitivity",
            "Manuscript_claim": (
                "Several spectral and connectivity effect directions "
                "depended on the marker-defined baseline."
            ),
            "Allowed_wording": (
                "Baseline choice materially influenced some descriptive "
                "effect estimates."
            ),
            "Avoid": (
                "Do not select whichever baseline produces the larger effect."
            )
        },
        {
            "Claim_ID": "C4",
            "Evidence_level": "Directionally robust descriptive connectivity",
            "Manuscript_claim": (
                "Global beta wPLI²_debiased showed the same Active–Passive "
                "direction under both baseline definitions, but neither "
                "contrast was statistically significant."
            ),
            "Allowed_wording": (
                "Beta connectivity showed a baseline-directionally robust "
                "descriptive pattern."
            ),
            "Avoid": (
                "Do not describe this as a confirmed beta connectivity "
                "difference."
            )
        },
        {
            "Claim_ID": "C5",
            "Evidence_level": "Null network-topology inference",
            "Manuscript_claim": (
                "Weighted network topology provided no multiplicity-"
                "corrected evidence of Active–Passive differences."
            ),
            "Allowed_wording": (
                "Frontal and posterior beta absolute-strength effects were "
                "directionally stable but remained descriptive."
            ),
            "Avoid": (
                "Do not claim network reorganization or altered efficiency."
            )
        },
        {
            "Claim_ID": "C6",
            "Evidence_level": "Within-Active sequential analysis",
            "Manuscript_claim": (
                "The frozen first complete Active marker cycle permits "
                "assessment of spectral dynamics across T1_V1–T1_V4."
            ),
            "Allowed_wording": (
                "Sequential marker-defined phase dynamics."
            ),
            "Avoid": (
                "Do not assign undocumented cognitive or AI-interaction "
                "meanings to T1_V1–T1_V4."
            )
        },
        {
            "Claim_ID": "C7",
            "Evidence_level": "Design limitation",
            "Manuscript_claim": (
                "Marker phase and temporal order are inseparable in the "
                "within-Active first-cycle analysis."
            ),
            "Allowed_wording": (
                "Observed phase differences may reflect marker-associated "
                "state, temporal progression, adaptation, or fatigue."
            ),
            "Avoid": (
                "Do not make causal claims about marker-phase identity."
            )
        },
        {
            "Claim_ID": "C8",
            "Evidence_level": "Null targeted connectivity follow-up",
            "Manuscript_claim": (
                "The first-cycle theta spectral pattern was not accompanied "
                "by statistically supported changes in global, FF, F-P, or "
                "P-P theta wPLI²_debiased."
            ),
            "Allowed_wording": (
                "Theta spectral dynamics were not mirrored by detectable "
                "changes in the prespecified phase-lagged connectivity "
                "summaries."
            ),
            "Avoid": (
                "Do not conclude that connectivity was unchanged in an "
                "absolute sense."
            )
        },
        {
            "Claim_ID": "C9",
            "Evidence_level": "Structural replication limitation",
            "Manuscript_claim": (
                "A universal second complete marker cycle was unavailable, "
                "so a reduced-sample replication was not forced."
            ),
            "Allowed_wording": (
                "The repeated-cycle branch was terminated according to a "
                "structure-only stopping rule."
            ),
            "Avoid": (
                "Do not exclude participants post hoc to manufacture a "
                "second-cycle replication."
            )
        }
    ]
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 8 — MANUSCRIPT CLAIM-LEVEL EVIDENCE MAP"
)

print(
    "=" * 120
)


display(
    claim_table
)


safe_to_csv(
    claim_table,
    "table8_manuscript_claim_evidence_map.csv"
)


# =============================================================================
# 21. MANUSCRIPT-READY COMPACT RESULTS TABLE
# =============================================================================

compact_results = pd.DataFrame(
    [
        {
            "Analysis": (
                "Between-group spectral — Baseline A"
            ),
            "Key_result": (
                "No corrected theta, alpha, or beta ROI contrast"
            ),
            "Largest_or_most_stable_effect": (
                "Beta Central/Temporal: g=+0.784, p=.132"
            ),
            "Inference": "Not statistically supported"
        },
        {
            "Analysis": (
                "Between-group spectral — Baseline B"
            ),
            "Key_result": (
                "No corrected theta, alpha, or beta ROI contrast"
            ),
            "Largest_or_most_stable_effect": (
                "Beta Central/Temporal: g=+0.857, p=.102"
            ),
            "Inference": "Not statistically supported"
        },
        {
            "Analysis": (
                "Global validated connectivity — Baseline A"
            ),
            "Key_result": (
                "No theta, alpha, or beta wPLI²_debiased group difference"
            ),
            "Largest_or_most_stable_effect": (
                "Beta: g=-0.636, p=.209, FDR=.627"
            ),
            "Inference": "Descriptive"
        },
        {
            "Analysis": (
                "Global validated connectivity — Baseline B"
            ),
            "Key_result": (
                "No theta, alpha, or beta wPLI²_debiased group difference"
            ),
            "Largest_or_most_stable_effect": (
                "Alpha: g=-0.482; beta g=-0.352"
            ),
            "Inference": "Descriptive"
        },
        {
            "Analysis": (
                "Cross-baseline global connectivity"
            ),
            "Key_result": (
                "Theta and alpha directions reversed; beta remained negative"
            ),
            "Largest_or_most_stable_effect": (
                "Beta direction preserved under both baselines"
            ),
            "Inference": (
                "Baseline-directionally robust descriptive beta pattern"
            )
        },
        {
            "Analysis": (
                "Weighted network topology"
            ),
            "Key_result": (
                "No multiplicity-corrected Active–Passive topology difference"
            ),
            "Largest_or_most_stable_effect": (
                "Absolute frontal/posterior beta remained negative "
                "across both baselines and LOO analyses"
            ),
            "Inference": "Descriptive"
        },
        {
            "Analysis": (
                "Targeted within-Active theta connectivity"
            ),
            "Key_result": (
                "No corrected V1–V4 connectivity difference"
            ),
            "Largest_or_most_stable_effect": (
                "Frontal–Posterior: dz=-0.404, p=.297, FDR=.750"
            ),
            "Inference": "Not statistically supported"
        }
    ]
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "TABLE 9 — COMPACT MANUSCRIPT RESULTS"
)

print(
    "=" * 120
)


display(
    compact_results
)


safe_to_csv(
    compact_results,
    "table9_compact_manuscript_results.csv"
)


# =============================================================================
# 22. FINAL EVIDENCE HIERARCHY
# =============================================================================

evidence_hierarchy = pd.DataFrame(
    [
        {
            "Level": 1,
            "Category": "Primary",
            "Analyses": (
                "Task 4A between-group spectral; "
                "Task 6C validated global connectivity"
            ),
            "Interpretation_rule": (
                "Statistical claims require multiplicity-corrected support."
            )
        },
        {
            "Level": 2,
            "Category": "Sensitivity / robustness",
            "Analyses": (
                "Task 4B baseline sensitivity; Task 6C Baseline A/B; "
                "leave-one-out robustness"
            ),
            "Interpretation_rule": (
                "Used to assess stability, not to rescue null primary tests."
            )
        },
        {
            "Level": 3,
            "Category": "Secondary anatomically constrained",
            "Analyses": (
                "Task 7A/7B network localization and weighted topology"
            ),
            "Interpretation_rule": (
                "Corrected network results only; stable uncorrected effects "
                "remain descriptive."
            )
        },
        {
            "Level": 4,
            "Category": "Secondary within-Active",
            "Analyses": (
                "Task 8C first-cycle spectral dynamics"
            ),
            "Interpretation_rule": (
                "Sequential marker-defined dynamics; temporal order "
                "confounding must be stated."
            )
        },
        {
            "Level": 5,
            "Category": "Targeted follow-up",
            "Analyses": (
                "Task 9A T1_V1–T1_V4 theta connectivity"
            ),
            "Interpretation_rule": (
                "Tests convergence with spectral finding; no broad search."
            )
        },
        {
            "Level": 6,
            "Category": "Excluded legacy inference",
            "Analyses": (
                "Task 5A/5B0/5B/5C/6A custom Hilbert connectivity"
            ),
            "Interpretation_rule": (
                "May document methodological development only; "
                "not scientific evidence."
            )
        }
    ]
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "FINAL EVIDENCE HIERARCHY"
)

print(
    "=" * 120
)


display(
    evidence_hierarchy
)


safe_to_csv(
    evidence_hierarchy,
    "final_evidence_hierarchy.csv"
)


# =============================================================================
# 23. AUTOMATIC MANUSCRIPT-SAFE SUMMARY
# =============================================================================

summary_lines = []


summary_lines.append(
    "1. The analysis retained 15 participants for the between-group "
    "comparisons (Active n=8; Passive n=7) and all 8 Active participants "
    "for the frozen first-cycle within-group analysis."
)


summary_lines.append(
    "2. The two marker-defined baseline intervals were physiologically "
    "distinct, with substantially greater posterior alpha during "
    "Baseline B than Baseline A; the neutral baseline labels are retained "
    "because authoritative marker semantics remain unavailable."
)


summary_lines.append(
    "3. Between-group spectral analyses provided no multiplicity-corrected "
    "evidence of Active–Passive differences in theta, alpha, or beta "
    "relative power across frontal, central/temporal, or posterior ROIs."
)


summary_lines.append(
    "4. Validated global wPLI²_debiased connectivity likewise provided "
    "no statistically supported Active–Passive difference in theta, "
    "alpha, or beta."
)


summary_lines.append(
    "5. Theta and alpha global connectivity contrasts were sensitive to "
    "baseline choice. Beta retained the same Active–Passive direction "
    "under both baseline definitions but remained statistically "
    "nonsignificant and is therefore treated as a directionally robust "
    "descriptive pattern."
)


summary_lines.append(
    "6. Weighted network-topology analysis provided no multiplicity-"
    "corrected evidence of group differences. Directionally stable "
    "frontal and posterior beta absolute-strength effects remain "
    "descriptive."
)


summary_lines.append(
    "7. Within the Active group, the first complete "
    "T1_V1→T1_V2→T1_V3→T1_V4 cycle was analyzed using an identical "
    "fixed-nine-epoch design for every participant. Marker codes remain "
    "semantically neutral, and phase effects are intrinsically confounded "
    "with temporal order."
)


summary_lines.append(
    "8. A universal second-cycle replication was not available and was "
    "not forced by excluding participants."
)


summary_lines.append(
    "9. Targeted theta wPLI²_debiased analysis of T1_V1 versus T1_V4 "
    "showed no statistically supported change in global, frontal–frontal, "
    "frontal–posterior, or posterior–posterior connectivity."
)


summary_lines.append(
    "10. The final manuscript should therefore emphasize robust design, "
    "baseline sensitivity, effect-size estimation, and convergence or "
    "dissociation across spectral and connectivity measures rather than "
    "claims of widespread statistically significant neural differences."
)


manuscript_summary = "\n\n".join(
    summary_lines
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "MANUSCRIPT-SAFE RESULTS SUMMARY"
)

print(
    "=" * 120
)

print(
    "\n"
    +
    manuscript_summary
)


with open(
    OUT
    / "manuscript_safe_results_summary.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        manuscript_summary
    )


# =============================================================================
# 24. SOURCE FILE MANIFEST
# =============================================================================

source_manifest_rows = []


for task, directory in [
    ("3C", TASK3C),
    ("3D", TASK3D),
    ("4A", TASK4A),
    ("4B", TASK4B),
    ("6C", TASK6C),
    ("7A", TASK7A),
    ("7B", TASK7B),
    ("8A", TASK8A),
    ("8B", TASK8B),
    ("8C", TASK8C),
    ("8D", TASK8D),
    ("9A", TASK9A)
]:

    if directory is None:

        source_manifest_rows.append(
            {
                "Task":
                    task,

                "Directory":
                    "",

                "CSV_count":
                    0,

                "Status":
                    "Directory not found"
            }
        )

        continue


    csvs = all_csvs(
        directory
    )


    source_manifest_rows.append(
        {
            "Task":
                task,

            "Directory":
                str(
                    directory
                ),

            "CSV_count":
                len(
                    csvs
                ),

            "Status":
                "Found"
        }
    )


source_manifest = pd.DataFrame(
    source_manifest_rows
)


safe_to_csv(
    source_manifest,
    "source_directory_manifest.csv"
)


# =============================================================================
# 25. OUTPUT MANIFEST
# =============================================================================

output_files = sorted(
    TABLE_OUT.glob(
        "*.csv"
    )
)


output_manifest = pd.DataFrame(
    {
        "file":
            [
                path.name
                for path in output_files
            ],

        "path":
            [
                str(path)
                for path in output_files
            ]
    }
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "GENERATED TABLE FILES"
)

print(
    "=" * 120
)


display(
    output_manifest
)


output_manifest.to_csv(
    OUT
    / "output_manifest.csv",
    index=False
)


# =============================================================================
# 26. INTEGRITY CHECK
# =============================================================================

task9a_expected = {
    "Global",
    "Frontal–Frontal",
    "Frontal–Posterior",
    "Posterior–Posterior"
}


task9a_features = set(
    targeted_theta[
        "Feature"
    ].astype(str)
)


integrity = {

    "analysis_root_exists":
        BASE.exists(),

    "output_directory_exists":
        OUT.exists(),

    "total_participants_frozen":
        N_TOTAL == 15,

    "active_participants_frozen":
        N_ACTIVE == 8,

    "passive_participants_frozen":
        N_PASSIVE == 7,

    "channels_frozen":
        N_CHANNELS == 14,

    "task9a_found":
        task9a_raw is not None,

    "task9a_four_tests":
        len(
            targeted_theta
        )
        ==
        4,

    "task9a_features_correct":
        task9a_expected
        ==
        task9a_features,

    "task9a_no_fdr_significance":
        bool(
            (
                targeted_theta[
                    "p_FDR_4"
                ]
                >=
                0.05
            ).all()
        ),

    "legacy_connectivity_excluded":
        bool(
            manifest.loc[
                manifest[
                    "task"
                ]
                ==
                "5A–6A",
                "status"
            ].eq(
                "Exclude from manuscript inference"
            ).all()
        ),

    "claim_table_created":
        len(
            claim_table
        )
        >=
        9,

    "compact_results_created":
        len(
            compact_results
        )
        >=
        7,

    "evidence_hierarchy_created":
        len(
            evidence_hierarchy
        )
        ==
        6,

    "no_new_inferential_tests":
        True,

    "marker_semantics_remain_neutral":
        True
}


overall_pass = bool(
    all(
        integrity.values()
    )
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 10A INTEGRITY CHECK"
)

print(
    "=" * 120
)


for key, value in integrity.items():

    print(
        f"{key:<45}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print(
    "\n"
    +
    "-" * 120
)

print(
    "OVERALL:",
    (
        "PASS"
        if overall_pass
        else
        "FAIL"
    )
)

print(
    "-" * 120
)


if not overall_pass:

    raise RuntimeError(
        "Task 10A consolidation integrity check failed."
    )


# =============================================================================
# 27. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    +
    "=" * 120
)

print(
    "TASK 10A COMPLETE — FINAL RESULTS FROZEN"
)

print(
    "=" * 120
)


print(
    f"""
Final inferential evidence base
-------------------------------
Between-group spectral:
    Task 4A
    Task 4B sensitivity

Validated connectivity:
    Task 6C
    wPLI²_debiased only

Validated network analysis:
    Task 7A / Task 7B

Within-Active spectral:
    Task 8C

Targeted within-Active connectivity:
    Task 9A

Structural / QC support:
    Task 3C
    Task 3D
    Task 8A
    Task 8B
    Task 8D

Excluded from manuscript inference:
    Task 5A
    Task 5B0
    Task 5B
    Task 5C
    Task 6A
    (legacy custom Hilbert connectivity)

No new inferential tests:
    YES

Marker semantics:
    remain neutral

Output directory:
    {OUT}

Tables:
    {TABLE_OUT}

Next task:
    Task 10B — publication-ready figures
"""
)


print(
    "=" * 120
)

In [ ]:
# =============================================================================
# TASK 10B — PUBLICATION-READY FIGURES
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PAPER:
# Active Versus Passive Engagement With Generative AI:
# Neural Dynamics During Academic Writing
#
# PURPOSE
# -------
# Generate the final publication-ready figure set from FROZEN analyses.
#
# IMPORTANT
# ---------
# This cell:
#   ✓ performs NO new inferential tests
#   ✓ uses only frozen/validated results
#   ✓ excludes legacy Task 5A–6A custom Hilbert connectivity
#   ✓ preserves neutral marker labels
#   ✓ exports 600-dpi PNG and vector PDF
#   ✓ creates a figure manifest
#
# FIGURES
# -------
# Figure 1 — Study design and analytical workflow
# Figure 2 — Physiological validation of marker-defined baselines
# Figure 3 — Active–Passive spectral effect sizes and baseline sensitivity
# Figure 4 — Validated connectivity and weighted-network robustness
# Figure 5 — Within-Active sequential spectral dynamics
# Figure S1 — Targeted within-Active theta connectivity
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.lines import Line2D
from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK10A = (
    BASE
    / "task10a_final_results_consolidation"
)

TASK10A_TABLES = (
    TASK10A
    / "tables"
)

OUT = (
    BASE
    / "task10b_publication_ready_figures"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


if not BASE.exists():
    raise FileNotFoundError(
        f"Analysis root not found:\n{BASE}"
    )


print("=" * 110)
print("TASK 10B — PUBLICATION-READY FIGURES")
print("=" * 110)

print(f"\nAnalysis root:\n{BASE}")
print(f"\nOutput directory:\n{OUT}")


# =============================================================================
# 2. FROZEN STUDY CONSTANTS
# =============================================================================

N_TOTAL = 15
N_ACTIVE = 8
N_PASSIVE = 7
N_CHANNELS = 14
SFREQ = 256

PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]

ROIS = [
    "Frontal",
    "Central/Temporal",
    "Posterior"
]

BANDS = [
    "Theta",
    "Alpha",
    "Beta"
]


# =============================================================================
# 3. PLOTTING SETTINGS
# =============================================================================

plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }
)


# =============================================================================
# 4. HELPERS
# =============================================================================

def normalize(text):

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(text).lower()
    )


def find_column(
    df,
    candidates
):

    mapping = {
        normalize(c): c
        for c in df.columns
    }

    for candidate in candidates:

        key = normalize(candidate)

        if key in mapping:
            return mapping[key]

    return None


def numeric(
    df,
    candidates
):

    col = find_column(
        df,
        candidates
    )

    if col is None:

        return pd.Series(
            np.nan,
            index=df.index,
            dtype=float
        )

    return pd.to_numeric(
        df[col],
        errors="coerce"
    )


def textcol(
    df,
    candidates
):

    col = find_column(
        df,
        candidates
    )

    if col is None:

        return pd.Series(
            "",
            index=df.index
        )

    return df[col].astype(str)


def find_directory(
    patterns
):

    matches = []

    for pattern in patterns:

        matches.extend(
            [
                p
                for p in BASE.glob(pattern)
                if p.is_dir()
            ]
        )

    matches = sorted(
        set(matches)
    )

    return matches[0] if matches else None


def read_best_csv(
    directory,
    keywords=None,
    min_rows=1
):

    if directory is None:
        return None, None

    keywords = keywords or []

    candidates = []

    for path in directory.rglob("*.csv"):

        name = path.name.lower()

        score = sum(
            keyword.lower() in name
            for keyword in keywords
        )

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        if len(df) < min_rows:
            continue

        candidates.append(
            (
                score,
                len(df.columns),
                len(df),
                path,
                df
            )
        )

    if not candidates:
        return None, None

    candidates.sort(
        reverse=True,
        key=lambda x: (
            x[0],
            x[1],
            x[2]
        )
    )

    _, _, _, path, df = candidates[0]

    return path, df


def save_figure(
    fig,
    stem
):

    png = OUT / f"{stem}.png"
    pdf = OUT / f"{stem}.pdf"

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        pdf,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return png, pdf


def clean_roi(value):

    value = str(value)

    value = value.replace(
        "_",
        " "
    )

    value = value.replace(
        "CentralTemporal",
        "Central/Temporal"
    )

    value = value.replace(
        "Central Temporal",
        "Central/Temporal"
    )

    return value


def significance_label(
    p_fdr
):

    if not np.isfinite(p_fdr):
        return "Descriptive"

    if p_fdr < 0.05:
        return "FDR-supported"

    return "Not FDR-supported"


# =============================================================================
# 5. DISCOVER FROZEN DIRECTORIES
# =============================================================================

TASK3C = find_directory(
    ["task03c*", "task3c*"]
)

TASK4A = find_directory(
    ["task04a*", "task4a*"]
)

TASK4B = find_directory(
    ["task04b*", "task4b*"]
)

TASK6C = find_directory(
    ["task06c*", "task6c*"]
)

TASK7B = find_directory(
    ["task07b*", "task7b*"]
)

TASK8C = find_directory(
    ["task08c*", "task8c*"]
)

TASK9A = find_directory(
    ["task09a*", "task9a*"]
)


print("\nDiscovered directories:")

for label, path in [
    ("Task 3C", TASK3C),
    ("Task 4A", TASK4A),
    ("Task 4B", TASK4B),
    ("Task 6C", TASK6C),
    ("Task 7B", TASK7B),
    ("Task 8C", TASK8C),
    ("Task 9A", TASK9A)
]:

    print(
        f"{label:<10}: "
        f"{path if path else 'NOT FOUND'}"
    )


# =============================================================================
# 6. FIGURE MANIFEST CONTAINER
# =============================================================================

figure_manifest = []


# =============================================================================
# FIGURE 1
# STUDY DESIGN AND ANALYTICAL WORKFLOW
# =============================================================================

fig, ax = plt.subplots(
    figsize=(12, 6.7)
)

ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")


def workflow_box(
    x,
    y,
    w,
    h,
    title,
    body=""
):

    box = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.025,rounding_size=0.08",
        linewidth=1.3,
        facecolor="white",
        edgecolor="black"
    )

    ax.add_patch(box)

    ax.text(
        x + w / 2,
        y + h * 0.66,
        title,
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

    if body:

        ax.text(
            x + w / 2,
            y + h * 0.31,
            body,
            ha="center",
            va="center",
            fontsize=8.5,
            linespacing=1.25
        )


def arrow(
    x1,
    y1,
    x2,
    y2
):

    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.1,
            color="black"
        )
    )


workflow_box(
    4.55,
    5.65,
    2.9,
    0.9,
    "EEG dataset",
    "15 university students"
)

workflow_box(
    1.2,
    4.05,
    3.2,
    1.0,
    "Active engagement",
    "n = 8"
)

workflow_box(
    7.6,
    4.05,
    3.2,
    1.0,
    "Passive engagement",
    "n = 7"
)

arrow(
    5.4,
    5.65,
    3.3,
    5.05
)

arrow(
    6.6,
    5.65,
    8.7,
    5.05
)


workflow_box(
    4.25,
    2.75,
    3.5,
    1.05,
    "EEG preprocessing",
    "14 channels • 256 Hz • 1–45 Hz\naverage reference • artifact rejection"
)

arrow(
    2.8,
    4.05,
    5.0,
    3.8
)

arrow(
    9.2,
    4.05,
    7.0,
    3.8
)


workflow_box(
    0.25,
    0.75,
    2.55,
    1.2,
    "Spectral analysis",
    "Theta • Alpha • Beta\nBaseline sensitivity"
)

workflow_box(
    3.15,
    0.75,
    2.55,
    1.2,
    "Connectivity",
    "Validated\nwPLI²_debiased"
)

workflow_box(
    6.05,
    0.75,
    2.55,
    1.2,
    "Network analysis",
    "Anatomically constrained\nweighted topology"
)

workflow_box(
    8.95,
    0.75,
    2.8,
    1.2,
    "Within-Active dynamics",
    "T1_V1 → T1_V2 →\nT1_V3 → T1_V4"
)


for x in [
    1.5,
    4.4,
    7.3,
    10.3
]:

    arrow(
        6.0,
        2.75,
        x,
        1.95
    )


ax.text(
    6,
    0.18,
    "Marker labels are retained neutrally; undocumented cognitive meanings are not assigned.",
    ha="center",
    va="center",
    fontsize=8.5,
    style="italic"
)


png, pdf = save_figure(
    fig,
    "figure1_study_design_workflow"
)


figure_manifest.append(
    {
        "figure": "Figure 1",
        "title": "Study design and analytical workflow",
        "png": str(png),
        "pdf": str(pdf),
        "source": "Frozen study design",
        "new_inference": "No"
    }
)


# =============================================================================
# FIGURE 2
# BASELINE PHYSIOLOGICAL VALIDATION
# =============================================================================
#
# Frozen Task 3C values:
# Posterior relative alpha
# Baseline A mean = 0.125180
# Baseline B mean = 0.432688
# B-A = 0.307508
# B > A in 15/15 participants
# Wilcoxon p = 0.000061
# dz = 1.604584
#
# Channel B-A:
# O2 = 0.354647
# O1 = 0.349609
# P7 = 0.265623
# P8 = 0.260153
#
# No new test is performed below.
# =============================================================================

baseline_summary = pd.DataFrame(
    {
        "Baseline": [
            "Baseline A",
            "Baseline B"
        ],
        "Posterior relative alpha": [
            0.125180,
            0.432688
        ]
    }
)


posterior_channels = pd.DataFrame(
    {
        "Channel": [
            "P7",
            "P8",
            "O1",
            "O2"
        ],
        "B_minus_A": [
            0.265623,
            0.260153,
            0.349609,
            0.354647
        ]
    }
)


fig = plt.figure(
    figsize=(10.5, 4.5)
)


ax1 = fig.add_axes(
    [0.08, 0.17, 0.38, 0.72]
)

ax2 = fig.add_axes(
    [0.58, 0.17, 0.37, 0.72]
)


# Panel A

ax1.bar(
    baseline_summary["Baseline"],
    baseline_summary["Posterior relative alpha"],
    width=0.55,
    edgecolor="black",
    linewidth=1.0
)

ax1.set_ylabel(
    "Relative alpha power"
)

ax1.set_title(
    "A  Posterior alpha",
    loc="left",
    fontweight="bold"
)

ax1.text(
    0.5,
    0.96,
    "B > A in 15/15 participants",
    transform=ax1.transAxes,
    ha="center",
    va="top",
    fontsize=9
)

ax1.text(
    0.5,
    0.88,
    "dz = 1.60; p < .001",
    transform=ax1.transAxes,
    ha="center",
    va="top",
    fontsize=9
)

ax1.set_ylim(
    0,
    0.52
)


# Panel B

posterior_channels = posterior_channels.sort_values(
    "B_minus_A"
)

ax2.barh(
    posterior_channels["Channel"],
    posterior_channels["B_minus_A"],
    edgecolor="black",
    linewidth=1.0
)

ax2.axvline(
    0,
    linewidth=0.8,
    color="black"
)

ax2.set_xlabel(
    "Relative alpha difference\nBaseline B − Baseline A"
)

ax2.set_title(
    "B  Posterior channel differences",
    loc="left",
    fontweight="bold"
)


fig.text(
    0.5,
    0.02,
    "Marker-defined baseline labels are retained neutrally.",
    ha="center",
    fontsize=8.5,
    style="italic"
)


png, pdf = save_figure(
    fig,
    "figure2_baseline_physiological_validation"
)


figure_manifest.append(
    {
        "figure": "Figure 2",
        "title": "Physiological validation of marker-defined baselines",
        "png": str(png),
        "pdf": str(pdf),
        "source": "Task 3C frozen results",
        "new_inference": "No"
    }
)


# =============================================================================
# FIGURE 3
# ACTIVE–PASSIVE SPECTRAL EFFECT SIZES
# =============================================================================
#
# Uses frozen Hedges g values from Tasks 4A and 4B.
#
# No CI is invented here because the consolidated frozen summary available
# in this cell contains effect sizes and p-values but not a reliable complete
# CI set for all 18 contrasts.
# =============================================================================

spectral_effects = pd.DataFrame(
    [
        ["A", "Theta", "Frontal", -0.432, 0.394],
        ["A", "Theta", "Central/Temporal", -0.609, 0.244],
        ["A", "Theta", "Posterior", -0.272, 0.590],

        ["A", "Alpha", "Frontal", 0.397, 0.431],
        ["A", "Alpha", "Central/Temporal", -0.226, 0.650],
        ["A", "Alpha", "Posterior", 0.609, 0.237],

        ["A", "Beta", "Frontal", -0.175, 0.761],
        ["A", "Beta", "Central/Temporal", 0.784, 0.132],
        ["A", "Beta", "Posterior", 0.479, 0.339],

        ["B", "Theta", "Frontal", 0.385, 0.446],
        ["B", "Theta", "Central/Temporal", 0.429, 0.424],
        ["B", "Theta", "Posterior", 0.366, 0.458],

        ["B", "Alpha", "Frontal", 0.578, 0.254],
        ["B", "Alpha", "Central/Temporal", 0.546, 0.275],
        ["B", "Alpha", "Posterior", 0.390, 0.431],

        ["B", "Beta", "Frontal", 0.023, 0.959],
        ["B", "Beta", "Central/Temporal", 0.857, 0.102],
        ["B", "Beta", "Posterior", 0.561, 0.265]
    ],
    columns=[
        "Baseline",
        "Band",
        "ROI",
        "Hedges_g",
        "p_raw"
    ]
)


fig, axes = plt.subplots(
    1,
    3,
    figsize=(12.5, 4.8),
    sharex=True
)


for ax, band in zip(
    axes,
    BANDS
):

    temp = spectral_effects[
        spectral_effects["Band"] == band
    ].copy()


    y = np.arange(
        len(ROIS)
    )


    a = (
        temp[
            temp["Baseline"] == "A"
        ]
        .set_index("ROI")
        .loc[ROIS]
    )

    b = (
        temp[
            temp["Baseline"] == "B"
        ]
        .set_index("ROI")
        .loc[ROIS]
    )


    ax.axvline(
        0,
        color="black",
        linewidth=0.9
    )


    ax.scatter(
        a["Hedges_g"],
        y + 0.12,
        marker="o",
        s=55,
        label="Baseline A"
    )

    ax.scatter(
        b["Hedges_g"],
        y - 0.12,
        marker="s",
        s=55,
        label="Baseline B"
    )


    for i in range(
        len(ROIS)
    ):

        ax.plot(
            [
                a["Hedges_g"].iloc[i],
                b["Hedges_g"].iloc[i]
            ],
            [
                y[i] + 0.12,
                y[i] - 0.12
            ],
            linewidth=0.7,
            alpha=0.55
        )


    ax.set_yticks(
        y
    )

    ax.set_yticklabels(
        ROIS
    )

    ax.invert_yaxis()

    ax.set_title(
        band
    )

    ax.set_xlabel(
        "Hedges g\nActive − Passive"
    )

    ax.set_xlim(
        -1.05,
        1.05
    )


axes[0].legend(
    frameon=False,
    loc="lower left"
)


fig.text(
    0.5,
    0.02,
    "No ROI-band contrast was statistically supported after multiplicity correction.",
    ha="center",
    fontsize=9,
    style="italic"
)


fig.subplots_adjust(
    left=0.12,
    right=0.98,
    top=0.90,
    bottom=0.20,
    wspace=0.28
)


png, pdf = save_figure(
    fig,
    "figure3_spectral_effects_baseline_sensitivity"
)


figure_manifest.append(
    {
        "figure": "Figure 3",
        "title": "Active–Passive spectral effect sizes and baseline sensitivity",
        "png": str(png),
        "pdf": str(pdf),
        "source": "Tasks 4A and 4B frozen results",
        "new_inference": "No"
    }
)


# =============================================================================
# FIGURE 4
# VALIDATED CONNECTIVITY AND NETWORK ROBUSTNESS
# =============================================================================

global_connectivity = pd.DataFrame(
    [
        ["Theta",  0.093750, -0.036281],
        ["Alpha",  0.173919, -0.482229],
        ["Beta",  -0.636351, -0.351672]
    ],
    columns=[
        "Band",
        "Baseline_A_g",
        "Baseline_B_g"
    ]
)


beta_topology = pd.DataFrame(
    [
        ["Frontal", -0.490, -0.577],
        ["Central/Temporal", -0.731, 0.113],
        ["Posterior", -0.577, -0.363]
    ],
    columns=[
        "ROI",
        "Baseline_A_g",
        "Baseline_B_g"
    ]
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.8)
)


# Panel A — global connectivity

ax = axes[0]

y = np.arange(
    len(global_connectivity)
)

ax.axvline(
    0,
    color="black",
    linewidth=0.9
)

ax.scatter(
    global_connectivity["Baseline_A_g"],
    y + 0.12,
    marker="o",
    s=60,
    label="Baseline A"
)

ax.scatter(
    global_connectivity["Baseline_B_g"],
    y - 0.12,
    marker="s",
    s=60,
    label="Baseline B"
)


for i in range(
    len(global_connectivity)
):

    ax.plot(
        [
            global_connectivity[
                "Baseline_A_g"
            ].iloc[i],
            global_connectivity[
                "Baseline_B_g"
            ].iloc[i]
        ],
        [
            y[i] + 0.12,
            y[i] - 0.12
        ],
        linewidth=0.8,
        alpha=0.6
    )


ax.set_yticks(
    y
)

ax.set_yticklabels(
    global_connectivity["Band"]
)

ax.invert_yaxis()

ax.set_xlabel(
    "Hedges g\nActive − Passive"
)

ax.set_title(
    "A  Global wPLI²_debiased",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False
)


# Panel B — beta absolute topology

ax = axes[1]

y = np.arange(
    len(beta_topology)
)

ax.axvline(
    0,
    color="black",
    linewidth=0.9
)

ax.scatter(
    beta_topology["Baseline_A_g"],
    y + 0.12,
    marker="o",
    s=60
)

ax.scatter(
    beta_topology["Baseline_B_g"],
    y - 0.12,
    marker="s",
    s=60
)


for i in range(
    len(beta_topology)
):

    ax.plot(
        [
            beta_topology[
                "Baseline_A_g"
            ].iloc[i],
            beta_topology[
                "Baseline_B_g"
            ].iloc[i]
        ],
        [
            y[i] + 0.12,
            y[i] - 0.12
        ],
        linewidth=0.8,
        alpha=0.6
    )


ax.set_yticks(
    y
)

ax.set_yticklabels(
    beta_topology["ROI"]
)

ax.invert_yaxis()

ax.set_xlabel(
    "Hedges g\nActive − Passive"
)

ax.set_title(
    "B  Beta absolute node strength",
    loc="left",
    fontweight="bold"
)


for ax in axes:

    ax.set_xlim(
        -1.05,
        0.55
    )


fig.text(
    0.5,
    0.025,
    "Beta showed the most directionally stable descriptive pattern; no displayed effect was multiplicity-corrected significant.",
    ha="center",
    fontsize=8.7,
    style="italic"
)


fig.subplots_adjust(
    left=0.14,
    right=0.98,
    top=0.90,
    bottom=0.20,
    wspace=0.38
)


png, pdf = save_figure(
    fig,
    "figure4_validated_connectivity_network_robustness"
)


figure_manifest.append(
    {
        "figure": "Figure 4",
        "title": "Validated connectivity and weighted-network robustness",
        "png": str(png),
        "pdf": str(pdf),
        "source": "Tasks 6C and 7B frozen results",
        "new_inference": "No"
    }
)


# =============================================================================
# FIGURE 5
# WITHIN-ACTIVE SEQUENTIAL SPECTRAL DYNAMICS
# =============================================================================
#
# Frozen Task 8C descriptive means.
#
# Theta:
# Frontal:
# V1 .394886
# V2 .373115
# V3 .374824
# V4 .351713
#
# Posterior:
# V1 .324355
# V2 .295340
# V3 .307605
# V4 .288710
#
# Beta shown as contextual descriptive trajectory.
#
# IMPORTANT:
# No omnibus effect survived FDR.
#
# Secondary V1–V4 exact pairwise:
# Frontal theta:
#   difference V1-V4 = +0.043173
#   dz = 1.366142
#   p_exact = .007812
#   FDR6 = .046875
#   8/8 V1 > V4
#
# Posterior theta:
#   difference V1-V4 = +0.035646
#   dz = 1.584845
#   p_exact = .007812
#   FDR6 = .046875
#   8/8 V1 > V4
#
# =============================================================================

phase_means = pd.DataFrame(
    [
        ["Theta", "Frontal",
         0.394886, 0.373115, 0.374824, 0.351713],

        ["Theta", "Central/Temporal",
         0.296791, 0.295058, 0.304029, 0.290339],

        ["Theta", "Posterior",
         0.324355, 0.295340, 0.307605, 0.288710],

        ["Beta", "Frontal",
         0.311541, 0.326945, 0.332058, 0.338670],

        ["Beta", "Central/Temporal",
         0.446953, 0.432702, 0.426833, 0.426450],

        ["Beta", "Posterior",
         0.350204, 0.382817, 0.380876, 0.387537]
    ],
    columns=[
        "Band",
        "ROI",
        "T1_V1",
        "T1_V2",
        "T1_V3",
        "T1_V4"
    ]
)


fig, axes = plt.subplots(
    1,
    2,
    figsize=(11.5, 4.8)
)


# -------------------------------------------------------------------------
# Panel A — theta
# -------------------------------------------------------------------------

ax = axes[0]

theta = phase_means[
    phase_means["Band"] == "Theta"
]


for _, row in theta.iterrows():

    ax.plot(
        PHASES,
        [
            row[p]
            for p in PHASES
        ],
        marker="o",
        linewidth=1.8,
        label=row["ROI"]
    )


ax.set_ylabel(
    "Relative theta power"
)

ax.set_xlabel(
    "Sequential marker-defined phase"
)

ax.set_title(
    "A  Theta dynamics",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False
)


ax.annotate(
    "Frontal V1−V4\n"
    "dz = 1.37; FDR = .047",
    xy=(
        3,
        0.351713
    ),
    xytext=(
        1.72,
        0.408
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=0.8
    ),
    fontsize=8.5
)


ax.annotate(
    "Posterior V1−V4\n"
    "dz = 1.58; FDR = .047",
    xy=(
        3,
        0.288710
    ),
    xytext=(
        1.65,
        0.318
    ),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=0.8
    ),
    fontsize=8.5
)


# -------------------------------------------------------------------------
# Panel B — beta
# -------------------------------------------------------------------------

ax = axes[1]

beta = phase_means[
    phase_means["Band"] == "Beta"
]


for _, row in beta.iterrows():

    ax.plot(
        PHASES,
        [
            row[p]
            for p in PHASES
        ],
        marker="o",
        linewidth=1.8,
        label=row["ROI"]
    )


ax.set_ylabel(
    "Relative beta power"
)

ax.set_xlabel(
    "Sequential marker-defined phase"
)

ax.set_title(
    "B  Beta dynamics",
    loc="left",
    fontweight="bold"
)

ax.legend(
    frameon=False
)


fig.text(
    0.5,
    0.025,
    "No four-phase omnibus spectral test survived FDR correction. "
    "V1–V4 theta contrasts are secondary exact paired findings; "
    "phase identity is confounded with temporal order.",
    ha="center",
    fontsize=8.4,
    style="italic"
)


fig.subplots_adjust(
    left=0.09,
    right=0.98,
    top=0.90,
    bottom=0.22,
    wspace=0.30
)


png, pdf = save_figure(
    fig,
    "figure5_within_active_sequential_spectral_dynamics"
)


figure_manifest.append(
    {
        "figure": "Figure 5",
        "title": "Within-Active sequential marker-defined spectral dynamics",
        "png": str(png),
        "pdf": str(pdf),
        "source": "Task 8C frozen results",
        "new_inference": "No"
    }
)


# =============================================================================
# FIGURE S1
# TARGETED THETA CONNECTIVITY FOLLOW-UP
# =============================================================================

targeted = pd.DataFrame(
    [
        [
            "Global",
            0.0072,
            0.133,
            0.719,
            0.867
        ],
        [
            "Frontal–Frontal",
            0.0256,
            0.338,
            0.375,
            0.750
        ],
        [
            "Frontal–Posterior",
            -0.0182,
            -0.404,
            0.297,
            0.750
        ],
        [
            "Posterior–Posterior",
            0.0048,
            0.066,
            0.867,
            0.867
        ]
    ],
    columns=[
        "Feature",
        "Delta",
        "dz",
        "p_exact",
        "p_FDR"
    ]
)


fig, ax = plt.subplots(
    figsize=(7.5, 4.6)
)


y = np.arange(
    len(targeted)
)


ax.axvline(
    0,
    color="black",
    linewidth=0.9
)


ax.scatter(
    targeted["dz"],
    y,
    s=65
)


for i, row in targeted.iterrows():

    ax.text(
        row["dz"] + (
            0.035
            if row["dz"] >= 0
            else -0.035
        ),
        i,
        (
            f"dz={row['dz']:+.2f}; "
            f"FDR={row['p_FDR']:.3f}"
        ),
        va="center",
        ha=(
            "left"
            if row["dz"] >= 0
            else "right"
        ),
        fontsize=8.5
    )


ax.set_yticks(
    y
)

ax.set_yticklabels(
    targeted["Feature"]
)

ax.invert_yaxis()

ax.set_xlabel(
    "Paired effect size (dz), T1_V4 − T1_V1"
)

ax.set_title(
    "Targeted theta wPLI²_debiased follow-up",
    loc="left",
    fontweight="bold"
)

ax.set_xlim(
    -0.65,
    0.60
)


ax.text(
    0.5,
    -0.16,
    "None of the four prespecified connectivity contrasts survived FDR correction.",
    transform=ax.transAxes,
    ha="center",
    fontsize=8.5,
    style="italic"
)


fig.subplots_adjust(
    left=0.27,
    right=0.96,
    top=0.88,
    bottom=0.24
)


png, pdf = save_figure(
    fig,
    "figureS1_targeted_theta_connectivity"
)


figure_manifest.append(
    {
        "figure": "Figure S1",
        "title": "Targeted within-Active theta connectivity follow-up",
        "png": str(png),
        "pdf": str(pdf),
        "source": "Task 9A frozen results",
        "new_inference": "No"
    }
)


# =============================================================================
# 7. FIGURE CAPTIONS
# =============================================================================

captions = pd.DataFrame(
    [
        {
            "Figure": "Figure 1",
            "Caption":
                (
                    "Study design and analytical workflow. EEG recordings "
                    "from 15 participants were divided according to the "
                    "dataset-defined Active (n=8) and Passive (n=7) "
                    "engagement conditions. Following common preprocessing "
                    "and artifact rejection, analyses examined spectral "
                    "dynamics, validated debiased squared weighted "
                    "phase-lag connectivity (wPLI²_debiased), anatomically "
                    "constrained network measures, and sequential "
                    "marker-defined dynamics within the Active group. "
                    "Marker labels are retained neutrally because their "
                    "specific cognitive meanings are not sufficiently "
                    "documented."
                )
        },

        {
            "Figure": "Figure 2",
            "Caption":
                (
                    "Physiological validation of the two marker-defined "
                    "baseline states. Posterior relative alpha power was "
                    "greater during Baseline B than Baseline A in all "
                    "15 participants (mean difference=0.308; dz=1.60; "
                    "Wilcoxon p<.001). The largest channel-level differences "
                    "occurred over O1, O2, P7, and P8. These results establish "
                    "that the two marker-defined intervals represent "
                    "physiologically distinct baseline states; neutral "
                    "Baseline A/B terminology is retained."
                )
        },

        {
            "Figure": "Figure 3",
            "Caption":
                (
                    "Active–Passive spectral effect sizes under the two "
                    "marker-defined baseline definitions. Points show "
                    "Hedges g for baseline-normalized relative theta, alpha, "
                    "and beta power in frontal, central/temporal, and "
                    "posterior regions. Positive values indicate larger "
                    "task-related change in the Active group and negative "
                    "values indicate larger change in the Passive group. "
                    "No ROI-by-band contrast was statistically supported "
                    "after multiplicity correction. Several effect directions "
                    "varied with baseline definition, emphasizing baseline "
                    "sensitivity."
                )
        },

        {
            "Figure": "Figure 4",
            "Caption":
                (
                    "Validated functional-connectivity and weighted-network "
                    "effect sizes. Panel A shows global Active–Passive "
                    "differences in debiased squared weighted phase-lag "
                    "index (wPLI²_debiased) under Baselines A and B. "
                    "Theta and alpha changed direction across baseline "
                    "definitions, whereas beta remained negative under both "
                    "baselines. Panel B shows beta-band absolute node-strength "
                    "effects in frontal, central/temporal, and posterior "
                    "regions. Frontal and posterior effects retained a "
                    "negative direction under both baselines, whereas the "
                    "central/temporal effect was baseline-dependent. "
                    "None of these effects was multiplicity-corrected "
                    "significant."
                )
        },

        {
            "Figure": "Figure 5",
            "Caption":
                (
                    "Sequential spectral dynamics across the first complete "
                    "T1_V1→T1_V2→T1_V3→T1_V4 marker cycle within the "
                    "Active group. Curves show participant-averaged relative "
                    "theta and beta power from nine clean 2-s epochs per "
                    "phase. No four-phase omnibus spectral test survived "
                    "correction across the nine primary band-by-region tests. "
                    "Secondary exact paired comparisons showed lower theta "
                    "power at T1_V4 than T1_V1 in frontal "
                    "(dz=1.37, FDR=.047) and posterior "
                    "(dz=1.58, FDR=.047) regions, with the same direction "
                    "in all eight participants. Because marker identity and "
                    "temporal order are inseparable, these findings represent "
                    "sequential marker-defined dynamics rather than causal "
                    "phase-specific effects."
                )
        },

        {
            "Figure": "Figure S1",
            "Caption":
                (
                    "Targeted theta-band connectivity follow-up within the "
                    "Active group. Paired effect sizes quantify the "
                    "T1_V4−T1_V1 change in global, frontal–frontal, "
                    "frontal–posterior, and posterior–posterior "
                    "wPLI²_debiased. None of the four prespecified contrasts "
                    "survived FDR correction. The largest descriptive effect "
                    "was a frontal–posterior reduction (dz=-0.40), which "
                    "remained nonsignificant after correction."
                )
        }
    ]
)


captions.to_csv(
    OUT / "figure_captions.csv",
    index=False
)


print("\n" + "=" * 110)
print("PUBLICATION-READY FIGURE CAPTIONS")
print("=" * 110)

display(captions)


# =============================================================================
# 8. FIGURE MANIFEST
# =============================================================================

figure_manifest = pd.DataFrame(
    figure_manifest
)


figure_manifest.to_csv(
    OUT / "figure_manifest.csv",
    index=False
)


print("\n" + "=" * 110)
print("FIGURE MANIFEST")
print("=" * 110)

display(figure_manifest)


# =============================================================================
# 9. FINAL INTEGRITY CHECK
# =============================================================================

expected_stems = [
    "figure1_study_design_workflow",
    "figure2_baseline_physiological_validation",
    "figure3_spectral_effects_baseline_sensitivity",
    "figure4_validated_connectivity_network_robustness",
    "figure5_within_active_sequential_spectral_dynamics",
    "figureS1_targeted_theta_connectivity"
]


checks = {}


for stem in expected_stems:

    checks[
        f"{stem}_PNG"
    ] = (
        OUT
        / f"{stem}.png"
    ).exists()

    checks[
        f"{stem}_PDF"
    ] = (
        OUT
        / f"{stem}.pdf"
    ).exists()


checks[
    "figure_manifest"
] = (
    OUT
    / "figure_manifest.csv"
).exists()


checks[
    "figure_captions"
] = (
    OUT
    / "figure_captions.csv"
).exists()


checks[
    "six_figures_registered"
] = (
    len(
        figure_manifest
    )
    ==
    6
)


checks[
    "no_new_inferential_tests"
] = True


checks[
    "legacy_connectivity_not_used"
] = True


checks[
    "marker_semantics_neutral"
] = True


overall_pass = all(
    checks.values()
)


print("\n" + "=" * 110)
print("TASK 10B INTEGRITY CHECK")
print("=" * 110)


for key, value in checks.items():

    print(
        f"{key:<65}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print("-" * 110)

print(
    "OVERALL:",
    "PASS"
    if overall_pass
    else "FAIL"
)

print("-" * 110)


if not overall_pass:

    raise RuntimeError(
        "Task 10B figure-generation integrity check failed."
    )


# =============================================================================
# 10. COMPLETION SUMMARY
# =============================================================================

print("\n" + "=" * 110)
print("TASK 10B COMPLETE")
print("=" * 110)


print(
    f"""
Publication-ready figure set generated.

MAIN MANUSCRIPT
---------------
Figure 1
Study design and analytical workflow

Figure 2
Physiological validation of marker-defined baselines

Figure 3
Active–Passive spectral effect sizes and baseline sensitivity

Figure 4
Validated connectivity and weighted-network robustness

Figure 5
Within-Active sequential spectral dynamics


SUPPLEMENT
----------
Figure S1
Targeted theta wPLI²_debiased connectivity follow-up


IMPORTANT INTERPRETIVE FREEZE
-----------------------------
1. No new inferential tests were performed.

2. No legacy Task 5A–6A custom Hilbert connectivity result was used.

3. Between-group spectral findings remain statistically unconfirmed.

4. Global wPLI²_debiased findings remain statistically unconfirmed.

5. Beta connectivity is described only as a directionally robust
   descriptive pattern across baseline definitions.

6. No multiplicity-corrected network-topology difference is claimed.

7. The T1_V1–T1_V4 frontal and posterior theta contrasts are secondary
   exact paired findings because the corresponding four-phase omnibus
   tests did not survive FDR correction.

8. T1_V1–T1_V4 remain neutral marker labels.

9. Within-Active marker phase is confounded with temporal order.

10. Task 9A remains a null targeted connectivity follow-up.


Output directory:
{OUT}


NEXT:
Task 11 — Manuscript Results section
"""
)


print("=" * 110)


In [ ]:
# =============================================================================
# TASK 10C — PUBLICATION-READY EEG SCALP TOPOGRAPHY / HEATMAP PANELS
# CORRECTED COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PURPOSE
# -------
# Generate publication-ready descriptive EEG scalp topographies from the
# FROZEN Task 8B first-cycle dataset.
#
# IMPORTANT
# ---------
# The Task 8B FIF contains only:
#
#     epochs.event_id == {'1': 1}
#
# Therefore T1_V1–T1_V4 cannot be recovered from the MNE event IDs.
#
# Task 8B was frozen using the deterministic structure:
#
#     8 Active participants
#     × 4 ordered phases
#     × 9 epochs per phase
#     = 288 epochs
#
# Within each participant:
#
#     T1_V1 → T1_V2 → T1_V3 → T1_V4
#
# This cell reconstructs phase membership only after validating the frozen
# 288-epoch structure.
#
# ANALYTICAL GUARDRAILS
# ---------------------
# ✓ No new hypothesis tests
# ✓ No electrode-wise inference
# ✓ No cluster statistics
# ✓ No significance masking
# ✓ No legacy connectivity analysis
# ✓ Participant-level averaging before group averaging
# ✓ Neutral T1_V1–T1_V4 marker terminology
#
# OUTPUTS
# -------
# Figure 5C:
#     T1_V1 theta | T1_V4 theta | V4−V1 theta
#
# Figure S2:
#     Four-phase theta scalp sequence
#
# Figure S3:
#     Four-phase alpha scalp sequence
#
# Figure S4:
#     Four-phase beta scalp sequence
#
# Figure S5:
#     Theta changes relative to T1_V1
#
# Figure S6:
#     V4−V1 difference maps for theta, alpha, and beta
#
# CSV:
#     Reconstructed epoch labels
#     Group channel values
#     Subject-level channel values
#     Figure manifest
#     Figure captions
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)


plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "figure.titlesize": 12,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }
)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK8B = (
    BASE
    / "task08b_frozen_first_cycle_epochs"
)

OUT = (
    BASE
    / "task10c_scalp_topography_panels"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 110)
print("TASK 10C — PUBLICATION-READY EEG SCALP TOPOGRAPHY PANELS")
print("=" * 110)

print(
    f"\nTask 8B directory:\n{TASK8B}"
)

print(
    f"\nOutput directory:\n{OUT}"
)


if not TASK8B.exists():
    raise FileNotFoundError(
        f"Task 8B directory does not exist:\n{TASK8B}"
    )


# =============================================================================
# 2. FROZEN DESIGN CONSTANTS
# =============================================================================

N_SUBJECTS = 8

SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 9)
]

PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]

N_PHASES = len(
    PHASES
)

EPOCHS_PER_PHASE = 9

EXPECTED_EPOCHS_PER_SUBJECT = (
    N_PHASES
    *
    EPOCHS_PER_PHASE
)

EXPECTED_TOTAL_EPOCHS = (
    N_SUBJECTS
    *
    EXPECTED_EPOCHS_PER_SUBJECT
)


CHANNELS = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]

N_CHANNELS = len(
    CHANNELS
)

SFREQ_EXPECTED = 256.0


BANDS = {
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0)
}

DENOMINATOR = (
    4.0,
    30.0
)


print("\nFrozen analytical design:")

print(
    f"  Participants       : {N_SUBJECTS}"
)

print(
    f"  Phases             : {N_PHASES}"
)

print(
    f"  Epochs per phase   : {EPOCHS_PER_PHASE}"
)

print(
    f"  Expected epochs    : {EXPECTED_TOTAL_EPOCHS}"
)

print(
    f"  EEG channels       : {N_CHANNELS}"
)

print(
    f"  Expected sfreq     : {SFREQ_EXPECTED:.1f} Hz"
)


# =============================================================================
# 3. LOCATE THE FROZEN TASK 8B FIF
# =============================================================================

preferred_file = (
    TASK8B
    / "active_first_cycle_fixed9_epo.fif"
)


if preferred_file.exists():

    EPOCH_FILE = preferred_file

else:

    candidates = sorted(
        TASK8B.rglob(
            "*epo.fif"
        )
    )

    if len(candidates) == 0:

        raise FileNotFoundError(
            "\nNo Task 8B epochs FIF file was found under:\n"
            f"{TASK8B}"
        )

    print(
        "\nPreferred Task 8B filename was not found."
    )

    print(
        "Available epoch FIF files:"
    )

    for candidate in candidates:

        print(
            "  ",
            candidate
        )

    EPOCH_FILE = candidates[0]


print("\nFrozen Task 8B epoch file:")

print(
    EPOCH_FILE
)


# =============================================================================
# 4. LOAD THE FROZEN EPOCHS
# =============================================================================

epochs = mne.read_epochs(
    EPOCH_FILE,
    preload=True,
    verbose=False
)


print("\n" + "-" * 110)
print("FROZEN EPOCH OBJECT")
print("-" * 110)

print(
    epochs
)

print(
    "\nEvent IDs:"
)

print(
    epochs.event_id
)

print(
    "\nChannels:"
)

print(
    epochs.ch_names
)

print(
    f"\nNumber of epochs: {len(epochs)}"
)

print(
    f"Sampling frequency: {epochs.info['sfreq']:.3f} Hz"
)

print(
    f"Samples per epoch: {len(epochs.times)}"
)

print(
    f"Epoch duration: "
    f"{len(epochs.times) / epochs.info['sfreq']:.3f} s"
)


# =============================================================================
# 5. HARD STRUCTURAL VALIDATION
# =============================================================================

structure_checks = {}


structure_checks[
    "total_epochs_288"
] = (
    len(epochs)
    ==
    EXPECTED_TOTAL_EPOCHS
)


structure_checks[
    "channels_14"
] = (
    len(epochs.ch_names)
    ==
    N_CHANNELS
)


structure_checks[
    "expected_channel_names"
] = (
    set(
        ch.upper()
        for ch in epochs.ch_names
    )
    ==
    set(
        ch.upper()
        for ch in CHANNELS
    )
)


structure_checks[
    "sampling_frequency_256Hz"
] = np.isclose(
    epochs.info["sfreq"],
    SFREQ_EXPECTED
)


print("\n" + "=" * 110)
print("FROZEN STRUCTURE VALIDATION")
print("=" * 110)


for key, value in structure_checks.items():

    print(
        f"{key:<55}: "
        f"{'PASS' if value else 'FAIL'}"
    )


if not all(
    structure_checks.values()
):

    raise RuntimeError(
        "\nTask 8B does not match the frozen 8 × 4 × 9 design.\n"
        "Phase reconstruction has been stopped rather than guessed."
    )


print("\nFrozen structure confirmed:")

print(
    f"  {N_SUBJECTS} participants"
)

print(
    f"  × {N_PHASES} phases"
)

print(
    f"  × {EPOCHS_PER_PHASE} epochs per phase"
)

print(
    f"  = {EXPECTED_TOTAL_EPOCHS} epochs"
)


# =============================================================================
# 6. REORDER CHANNELS INTO THE FROZEN 14-CHANNEL ORDER
# =============================================================================

channel_lookup = {
    channel.upper(): index
    for index, channel in enumerate(
        epochs.ch_names
    )
}


channel_indices = [
    channel_lookup[
        channel.upper()
    ]
    for channel in CHANNELS
]


data = epochs.get_data(
    copy=True
)


data = data[
    :,
    channel_indices,
    :
]


print("\nRaw frozen data array after channel ordering:")

print(
    data.shape
)


if data.shape[0] != EXPECTED_TOTAL_EPOCHS:

    raise RuntimeError(
        "Unexpected epoch dimension after loading."
    )


if data.shape[1] != N_CHANNELS:

    raise RuntimeError(
        "Unexpected channel dimension after reordering."
    )


if not np.isfinite(
    data
).all():

    raise RuntimeError(
        "Non-finite values detected in frozen Task 8B EEG data."
    )


# =============================================================================
# 7. RECONSTRUCT FROZEN SUBJECT × PHASE × EPOCH STRUCTURE
# =============================================================================
#
# Task 8B was saved in the frozen order:
#
# sub-001:
#     T1_V1 × 9
#     T1_V2 × 9
#     T1_V3 × 9
#     T1_V4 × 9
#
# sub-002:
#     T1_V1 × 9
#     ...
#
# through sub-008.
#
# Therefore:
#
#     288 × 14 × samples
#
# becomes:
#
#     8 × 4 × 9 × 14 × samples
#
# No EEG outcome is used to determine phase membership.
#
# =============================================================================

n_times = data.shape[-1]


structured = data.reshape(
    N_SUBJECTS,
    N_PHASES,
    EPOCHS_PER_PHASE,
    N_CHANNELS,
    n_times
)


expected_shape = (
    N_SUBJECTS,
    N_PHASES,
    EPOCHS_PER_PHASE,
    N_CHANNELS,
    n_times
)


print("\nReconstructed frozen array:")

print(
    "subjects × phases × epochs × channels × samples"
)

print(
    structured.shape
)


if structured.shape != expected_shape:

    raise RuntimeError(
        f"\nUnexpected reconstructed shape:\n"
        f"{structured.shape}\n"
        f"Expected:\n"
        f"{expected_shape}"
    )


# =============================================================================
# 8. CREATE EXPLICIT RECONSTRUCTED EPOCH LABEL TABLE
# =============================================================================

label_rows = []

global_epoch_index = 0


for subject_index, subject in enumerate(
    SUBJECTS
):

    for phase_index, phase in enumerate(
        PHASES
    ):

        for within_phase_epoch in range(
            EPOCHS_PER_PHASE
        ):

            label_rows.append(
                {
                    "global_epoch_index":
                        global_epoch_index,

                    "subject":
                        subject,

                    "subject_index":
                        subject_index,

                    "phase":
                        phase,

                    "phase_index":
                        phase_index,

                    "epoch_within_phase":
                        within_phase_epoch + 1
                }
            )

            global_epoch_index += 1


labels = pd.DataFrame(
    label_rows
)


if len(labels) != EXPECTED_TOTAL_EPOCHS:

    raise RuntimeError(
        "Reconstructed epoch label table has the wrong number of rows."
    )


count_table = (
    labels
    .groupby(
        [
            "subject",
            "phase"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=PHASES
    )
)


print("\nCounts by subject and reconstructed phase:")

display(
    count_table
)


if not (
    count_table.values
    ==
    EPOCHS_PER_PHASE
).all():

    raise RuntimeError(
        "Not every reconstructed subject×phase cell contains exactly 9 epochs."
    )


labels.to_csv(
    OUT
    / "reconstructed_frozen_epoch_labels.csv",
    index=False
)


# =============================================================================
# 9. CREATE MNE SENSOR GEOMETRY
# =============================================================================

info = mne.create_info(
    ch_names=CHANNELS,
    sfreq=SFREQ_EXPECTED,
    ch_types="eeg"
)


montage = mne.channels.make_standard_montage(
    "standard_1020"
)


info.set_montage(
    montage,
    on_missing="raise"
)


sensor_positions = np.asarray(
    [
        channel_info["loc"][:3]
        for channel_info in info["chs"]
    ]
)


if sensor_positions.shape != (
    N_CHANNELS,
    3
):

    raise RuntimeError(
        "Unexpected sensor-position matrix."
    )


if not np.isfinite(
    sensor_positions
).all():

    raise RuntimeError(
        "Non-finite standard montage positions detected."
    )


print("\nStandard 10–20 sensor montage assigned successfully.")


# =============================================================================
# 10. RELATIVE BAND-POWER FUNCTION
# =============================================================================
#
# NUMPY 2.x COMPATIBILITY:
#
# np.trapezoid() is used instead of the removed np.trapz().
#
# =============================================================================

def relative_band_power_epochs(
    epoch_data,
    sfreq,
    band,
    denominator=(4.0, 30.0)
):
    """
    Calculate relative band power for each epoch and channel.

    Parameters
    ----------
    epoch_data : ndarray
        Shape:
            epochs × channels × samples

    sfreq : float
        Sampling frequency in Hz.

    band : tuple
        Frequency band, e.g. (4, 8).

    denominator : tuple
        Frequency interval used as denominator.

    Returns
    -------
    relative : ndarray
        Shape:
            epochs × channels
    """

    epoch_data = np.asarray(
        epoch_data,
        dtype=float
    )


    if epoch_data.ndim != 3:

        raise ValueError(
            f"Expected epochs × channels × samples; "
            f"received shape {epoch_data.shape}"
        )


    nperseg = min(
        512,
        epoch_data.shape[-1]
    )


    freqs, psd = welch(
        epoch_data,
        fs=sfreq,
        nperseg=nperseg,
        axis=-1
    )


    band_mask = (
        (freqs >= band[0])
        &
        (freqs < band[1])
    )


    denominator_mask = (
        (freqs >= denominator[0])
        &
        (freqs < denominator[1])
    )


    if band_mask.sum() == 0:

        raise RuntimeError(
            f"No frequency bins were found for band {band}."
        )


    if denominator_mask.sum() == 0:

        raise RuntimeError(
            f"No frequency bins were found for denominator {denominator}."
        )


    # ---------------------------------------------------------------------
    # NumPy 2.x compatible trapezoidal integration
    # ---------------------------------------------------------------------

    band_power = np.trapezoid(
        psd[
            ...,
            band_mask
        ],
        x=freqs[
            band_mask
        ],
        axis=-1
    )


    total_power = np.trapezoid(
        psd[
            ...,
            denominator_mask
        ],
        x=freqs[
            denominator_mask
        ],
        axis=-1
    )


    relative = np.divide(
        band_power,
        total_power,
        out=np.full_like(
            band_power,
            np.nan,
            dtype=float
        ),
        where=total_power > 0
    )


    return relative


# =============================================================================
# 11. CALCULATE SUBJECT-LEVEL TOPOGRAPHIES
# =============================================================================
#
# IMPORTANT:
#
# We first calculate relative power for each of the 9 epochs.
#
# Then:
#
#     average 9 epochs → one channel map per participant per phase
#
# Only after that:
#
#     average 8 participant maps → group scalp map
#
# Thus participants, not individual epochs, determine the group average.
#
# =============================================================================

subject_maps = {}


for band_name, band_range in BANDS.items():

    subject_maps[
        band_name
    ] = {}


    for phase_index, phase in enumerate(
        PHASES
    ):

        phase_subject_maps = []


        for subject_index in range(
            N_SUBJECTS
        ):

            subject_phase_data = structured[
                subject_index,
                phase_index,
                :,
                :,
                :
            ]


            if subject_phase_data.shape != (
                EPOCHS_PER_PHASE,
                N_CHANNELS,
                n_times
            ):

                raise RuntimeError(
                    f"{SUBJECTS[subject_index]} {phase}: "
                    f"unexpected data shape "
                    f"{subject_phase_data.shape}"
                )


            relative = relative_band_power_epochs(
                epoch_data=
                    subject_phase_data,

                sfreq=
                    SFREQ_EXPECTED,

                band=
                    band_range,

                denominator=
                    DENOMINATOR
            )


            if relative.shape != (
                EPOCHS_PER_PHASE,
                N_CHANNELS
            ):

                raise RuntimeError(
                    f"{SUBJECTS[subject_index]} {phase} {band_name}: "
                    f"unexpected relative-power shape {relative.shape}"
                )


            subject_channel_mean = np.nanmean(
                relative,
                axis=0
            )


            phase_subject_maps.append(
                subject_channel_mean
            )


        subject_maps[
            band_name
        ][
            phase
        ] = np.stack(
            phase_subject_maps,
            axis=0
        )


print("\nSubject-level topographies calculated successfully.")


# =============================================================================
# 12. VALIDATE SUBJECT-LEVEL MAPS
# =============================================================================

for band_name in BANDS:

    for phase in PHASES:

        arr = subject_maps[
            band_name
        ][
            phase
        ]


        if arr.shape != (
            N_SUBJECTS,
            N_CHANNELS
        ):

            raise RuntimeError(
                f"{band_name} {phase}: "
                f"unexpected map shape {arr.shape}"
            )


        if not np.isfinite(
            arr
        ).all():

            raise RuntimeError(
                f"{band_name} {phase}: "
                "non-finite relative-power values detected."
            )


print(
    "All subject-level topographies have shape 8 × 14 and contain finite values."
)


# =============================================================================
# 13. CALCULATE GROUP TOPOGRAPHIES
# =============================================================================

group_maps = {}


for band_name in BANDS:

    group_maps[
        band_name
    ] = {}


    for phase in PHASES:

        group_maps[
            band_name
        ][
            phase
        ] = np.mean(
            subject_maps[
                band_name
            ][
                phase
            ],
            axis=0
        )


# =============================================================================
# 14. DISPLAY GROUP THETA VALUES
# =============================================================================

theta_table = pd.DataFrame(
    {
        "Channel":
            CHANNELS,

        "T1_V1":
            group_maps[
                "Theta"
            ][
                "T1_V1"
            ],

        "T1_V2":
            group_maps[
                "Theta"
            ][
                "T1_V2"
            ],

        "T1_V3":
            group_maps[
                "Theta"
            ][
                "T1_V3"
            ],

        "T1_V4":
            group_maps[
                "Theta"
            ][
                "T1_V4"
            ]
    }
)


theta_table[
    "V4_minus_V1"
] = (
    theta_table[
        "T1_V4"
    ]
    -
    theta_table[
        "T1_V1"
    ]
)


print("\n" + "=" * 110)
print("GROUP THETA TOPOGRAPHY VALUES")
print("=" * 110)

display(
    theta_table.round(6)
)


theta_table.to_csv(
    OUT
    / "theta_topography_channel_values.csv",
    index=False
)


# =============================================================================
# 15. GENERIC TOPOGRAPHY PLOTTER
# =============================================================================

def plot_topography(
    values,
    ax,
    title,
    vlim=None,
    cmap=None
):
    """
    Plot a 14-channel scalp topography.
    """

    values = np.asarray(
        values,
        dtype=float
    )


    if values.shape != (
        N_CHANNELS,
    ):

        raise ValueError(
            f"Expected {N_CHANNELS} channel values; "
            f"received {values.shape}"
        )


    if not np.isfinite(
        values
    ).all():

        raise ValueError(
            f"Non-finite values supplied for {title}."
        )


    kwargs = {
        "axes":
            ax,

        "show":
            False,

        "sensors":
            True,

        "contours":
            6,

        "outlines":
            "head",

        "extrapolate":
            "head"
    }


    if vlim is not None:

        kwargs[
            "vlim"
        ] = vlim


    if cmap is not None:

        kwargs[
            "cmap"
        ] = cmap


    image, contours = mne.viz.plot_topomap(
        values,
        info,
        **kwargs
    )


    ax.set_title(
        title,
        fontsize=10,
        fontweight="bold",
        pad=8
    )


    return image


# =============================================================================
# 16. FIGURE SAVING FUNCTION
# =============================================================================

figure_manifest = []


def save_figure(
    fig,
    stem,
    figure_label,
    description
):

    png = (
        OUT
        / f"{stem}.png"
    )

    pdf = (
        OUT
        / f"{stem}.pdf"
    )


    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight"
    )


    fig.savefig(
        pdf,
        bbox_inches="tight"
    )


    plt.show()

    plt.close(
        fig
    )


    figure_manifest.append(
        {
            "figure":
                figure_label,

            "description":
                description,

            "png":
                str(png),

            "pdf":
                str(pdf),

            "new_inferential_test":
                "No"
        }
    )


    print(
        f"\nSaved:\n"
        f"  {png}\n"
        f"  {pdf}"
    )


# =============================================================================
# 17. FIGURE 5C — THETA V1, V4, AND DIFFERENCE
# =============================================================================

theta_v1 = group_maps[
    "Theta"
][
    "T1_V1"
]


theta_v4 = group_maps[
    "Theta"
][
    "T1_V4"
]


theta_difference = (
    theta_v4
    -
    theta_v1
)


absolute_values = np.concatenate(
    [
        theta_v1,
        theta_v4
    ]
)


theta_absolute_min = np.min(
    absolute_values
)

theta_absolute_max = np.max(
    absolute_values
)


theta_difference_limit = np.max(
    np.abs(
        theta_difference
    )
)


if theta_difference_limit <= 0:

    theta_difference_limit = 1e-6


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        10.8,
        3.9
    )
)


im1 = plot_topography(
    values=
        theta_v1,

    ax=
        axes[0],

    title=
        "A  T1_V1",

    vlim=(
        theta_absolute_min,
        theta_absolute_max
    )
)


im2 = plot_topography(
    values=
        theta_v4,

    ax=
        axes[1],

    title=
        "B  T1_V4",

    vlim=(
        theta_absolute_min,
        theta_absolute_max
    )
)


im3 = plot_topography(
    values=
        theta_difference,

    ax=
        axes[2],

    title=
        "C  T1_V4 − T1_V1",

    vlim=(
        -theta_difference_limit,
        theta_difference_limit
    ),

    cmap=
        "RdBu_r"
)


cbar_absolute = fig.colorbar(
    im2,
    ax=[
        axes[0],
        axes[1]
    ],
    shrink=0.72,
    pad=0.035
)


cbar_absolute.set_label(
    "Relative theta power"
)


cbar_difference = fig.colorbar(
    im3,
    ax=axes[2],
    shrink=0.72,
    pad=0.035
)


cbar_difference.set_label(
    "Change in relative theta power"
)


fig.text(
    0.5,
    0.025,
    "Descriptive scalp distributions. "
    "The difference map represents T1_V4 minus T1_V1; "
    "no electrode-wise statistical inference was performed.",
    ha="center",
    fontsize=8.3,
    style="italic"
)


fig.subplots_adjust(
    left=0.03,
    right=0.92,
    top=0.90,
    bottom=0.17,
    wspace=0.28
)


save_figure(
    fig=
        fig,

    stem=
        "figure5C_theta_scalp_V1_V4_difference",

    figure_label=
        "Figure 5C",

    description=
        "T1_V1 and T1_V4 theta scalp distributions and V4-minus-V1 difference."
)


# =============================================================================
# 18. FIGURE S2 — FOUR-PHASE THETA TOPOGRAPHY
# =============================================================================

theta_all_values = np.concatenate(
    [
        group_maps[
            "Theta"
        ][
            phase
        ]
        for phase in PHASES
    ]
)


theta_min = np.min(
    theta_all_values
)

theta_max = np.max(
    theta_all_values
)


fig, axes = plt.subplots(
    1,
    4,
    figsize=(
        12.4,
        3.45
    )
)


theta_images = []


for ax, phase in zip(
    axes,
    PHASES
):

    image = plot_topography(
        values=
            group_maps[
                "Theta"
            ][
                phase
            ],

        ax=
            ax,

        title=
            phase,

        vlim=(
            theta_min,
            theta_max
        )
    )


    theta_images.append(
        image
    )


cbar = fig.colorbar(
    theta_images[-1],
    ax=axes.tolist(),
    shrink=0.73,
    pad=0.025
)


cbar.set_label(
    "Relative theta power"
)


fig.text(
    0.5,
    0.025,
    "Sequential marker-defined phases. "
    "All panels use the same scale; phase identity is confounded with temporal order.",
    ha="center",
    fontsize=8.3,
    style="italic"
)


fig.subplots_adjust(
    left=0.025,
    right=0.91,
    top=0.88,
    bottom=0.17,
    wspace=0.20
)


save_figure(
    fig=
        fig,

    stem=
        "figureS2_four_phase_theta_topography",

    figure_label=
        "Figure S2",

    description=
        "Four-phase relative theta scalp topography."
)


# =============================================================================
# 19. FIGURE S3 — FOUR-PHASE ALPHA TOPOGRAPHY
# =============================================================================

alpha_all_values = np.concatenate(
    [
        group_maps[
            "Alpha"
        ][
            phase
        ]
        for phase in PHASES
    ]
)


alpha_min = np.min(
    alpha_all_values
)

alpha_max = np.max(
    alpha_all_values
)


fig, axes = plt.subplots(
    1,
    4,
    figsize=(
        12.4,
        3.45
    )
)


alpha_images = []


for ax, phase in zip(
    axes,
    PHASES
):

    image = plot_topography(
        values=
            group_maps[
                "Alpha"
            ][
                phase
            ],

        ax=
            ax,

        title=
            phase,

        vlim=(
            alpha_min,
            alpha_max
        )
    )


    alpha_images.append(
        image
    )


cbar = fig.colorbar(
    alpha_images[-1],
    ax=axes.tolist(),
    shrink=0.73,
    pad=0.025
)


cbar.set_label(
    "Relative alpha power"
)


fig.text(
    0.5,
    0.025,
    "Descriptive scalp distributions; no electrode-wise statistical inference was performed.",
    ha="center",
    fontsize=8.3,
    style="italic"
)


fig.subplots_adjust(
    left=0.025,
    right=0.91,
    top=0.88,
    bottom=0.17,
    wspace=0.20
)


save_figure(
    fig=
        fig,

    stem=
        "figureS3_four_phase_alpha_topography",

    figure_label=
        "Figure S3",

    description=
        "Four-phase relative alpha scalp topography."
)


# =============================================================================
# 20. FIGURE S4 — FOUR-PHASE BETA TOPOGRAPHY
# =============================================================================

beta_all_values = np.concatenate(
    [
        group_maps[
            "Beta"
        ][
            phase
        ]
        for phase in PHASES
    ]
)


beta_min = np.min(
    beta_all_values
)

beta_max = np.max(
    beta_all_values
)


fig, axes = plt.subplots(
    1,
    4,
    figsize=(
        12.4,
        3.45
    )
)


beta_images = []


for ax, phase in zip(
    axes,
    PHASES
):

    image = plot_topography(
        values=
            group_maps[
                "Beta"
            ][
                phase
            ],

        ax=
            ax,

        title=
            phase,

        vlim=(
            beta_min,
            beta_max
        )
    )


    beta_images.append(
        image
    )


cbar = fig.colorbar(
    beta_images[-1],
    ax=axes.tolist(),
    shrink=0.73,
    pad=0.025
)


cbar.set_label(
    "Relative beta power"
)


fig.text(
    0.5,
    0.025,
    "Descriptive scalp distributions; no electrode-wise statistical inference was performed.",
    ha="center",
    fontsize=8.3,
    style="italic"
)


fig.subplots_adjust(
    left=0.025,
    right=0.91,
    top=0.88,
    bottom=0.17,
    wspace=0.20
)


save_figure(
    fig=
        fig,

    stem=
        "figureS4_four_phase_beta_topography",

    figure_label=
        "Figure S4",

    description=
        "Four-phase relative beta scalp topography."
)


# =============================================================================
# 21. FIGURE S5 — THETA CHANGE RELATIVE TO T1_V1
# =============================================================================
#
# Difference maps:
#
#     T1_V2 − T1_V1
#     T1_V3 − T1_V1
#     T1_V4 − T1_V1
#
# These are descriptive maps only.
#
# =============================================================================

theta_changes = {
    phase:
        (
            group_maps[
                "Theta"
            ][
                phase
            ]
            -
            theta_v1
        )

    for phase in [
        "T1_V2",
        "T1_V3",
        "T1_V4"
    ]
}


theta_change_values = np.concatenate(
    [
        theta_changes[
            phase
        ]
        for phase in [
            "T1_V2",
            "T1_V3",
            "T1_V4"
        ]
    ]
)


theta_change_limit = np.max(
    np.abs(
        theta_change_values
    )
)


if theta_change_limit <= 0:

    theta_change_limit = 1e-6


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        10.4,
        3.55
    )
)


change_images = []


for ax, phase in zip(
    axes,
    [
        "T1_V2",
        "T1_V3",
        "T1_V4"
    ]
):

    image = plot_topography(
        values=
            theta_changes[
                phase
            ],

        ax=
            ax,

        title=
            f"{phase} − T1_V1",

        vlim=(
            -theta_change_limit,
            theta_change_limit
        ),

        cmap=
            "RdBu_r"
    )


    change_images.append(
        image
    )


cbar = fig.colorbar(
    change_images[-1],
    ax=axes.tolist(),
    shrink=0.73,
    pad=0.025
)


cbar.set_label(
    "Change in relative theta power"
)


fig.text(
    0.5,
    0.025,
    "Descriptive changes relative to T1_V1; "
    "channel-level statistical significance is not implied.",
    ha="center",
    fontsize=8.3,
    style="italic"
)


fig.subplots_adjust(
    left=0.025,
    right=0.91,
    top=0.88,
    bottom=0.17,
    wspace=0.22
)


save_figure(
    fig=
        fig,

    stem=
        "figureS5_theta_change_relative_to_V1",

    figure_label=
        "Figure S5",

    description=
        "Theta scalp changes at V2, V3, and V4 relative to V1."
)


# =============================================================================
# 22. FIGURE S6 — V4−V1 DIFFERENCE MAPS ACROSS ALL THREE BANDS
# =============================================================================
#
# This provides a compact cross-frequency comparison:
#
#     Theta V4−V1
#     Alpha V4−V1
#     Beta  V4−V1
#
# Each band gets its own symmetric scale because absolute magnitudes differ.
#
# =============================================================================

band_differences = {}


for band_name in BANDS:

    band_differences[
        band_name
    ] = (
        group_maps[
            band_name
        ][
            "T1_V4"
        ]
        -
        group_maps[
            band_name
        ][
            "T1_V1"
        ]
    )


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        10.4,
        3.6
    )
)


for ax, band_name in zip(
    axes,
    [
        "Theta",
        "Alpha",
        "Beta"
    ]
):

    values = band_differences[
        band_name
    ]


    limit = np.max(
        np.abs(
            values
        )
    )


    if limit <= 0:

        limit = 1e-6


    image = plot_topography(
        values=
            values,

        ax=
            ax,

        title=
            f"{band_name}: T1_V4 − T1_V1",

        vlim=(
            -limit,
            limit
        ),

        cmap=
            "RdBu_r"
    )


    cbar = fig.colorbar(
        image,
        ax=ax,
        shrink=0.67,
        pad=0.035
    )


    cbar.set_label(
        "Relative-power change",
        fontsize=8
    )


fig.text(
    0.5,
    0.025,
    "Band-specific descriptive V4−V1 difference maps. "
    "Color scales are symmetric within each band and should not be compared numerically across panels.",
    ha="center",
    fontsize=8.2,
    style="italic"
)


fig.subplots_adjust(
    left=0.025,
    right=0.98,
    top=0.88,
    bottom=0.17,
    wspace=0.30
)


save_figure(
    fig=
        fig,

    stem=
        "figureS6_V4_minus_V1_band_difference_topographies",

    figure_label=
        "Figure S6",

    description=
        "Theta, alpha, and beta V4-minus-V1 scalp difference maps."
)


# =============================================================================
# 23. EXPORT GROUP CHANNEL-LEVEL VALUES
# =============================================================================

group_rows = []


for band_name in BANDS:

    for phase in PHASES:

        group_values = group_maps[
            band_name
        ][
            phase
        ]


        participant_values = subject_maps[
            band_name
        ][
            phase
        ]


        for channel_index, channel in enumerate(
            CHANNELS
        ):

            group_rows.append(
                {
                    "band":
                        band_name,

                    "phase":
                        phase,

                    "channel":
                        channel,

                    "group_mean_relative_power":
                        group_values[
                            channel_index
                        ],

                    "participant_sd":
                        np.std(
                            participant_values[
                                :,
                                channel_index
                            ],
                            ddof=1
                        ),

                    "n_participants":
                        N_SUBJECTS
                }
            )


group_channel_values = pd.DataFrame(
    group_rows
)


group_channel_values.to_csv(
    OUT
    / "all_topography_group_channel_values.csv",
    index=False
)


# =============================================================================
# 24. EXPORT SUBJECT-LEVEL CHANNEL VALUES
# =============================================================================

subject_rows = []


for band_name in BANDS:

    for phase in PHASES:

        arr = subject_maps[
            band_name
        ][
            phase
        ]


        for subject_index, subject in enumerate(
            SUBJECTS
        ):

            for channel_index, channel in enumerate(
                CHANNELS
            ):

                subject_rows.append(
                    {
                        "subject":
                            subject,

                        "band":
                            band_name,

                        "phase":
                            phase,

                        "channel":
                            channel,

                        "relative_power":
                            arr[
                                subject_index,
                                channel_index
                            ]
                    }
                )


subject_channel_values = pd.DataFrame(
    subject_rows
)


subject_channel_values.to_csv(
    OUT
    / "subject_level_topography_values.csv",
    index=False
)


# =============================================================================
# 25. EXPORT V4−V1 DIFFERENCE VALUES
# =============================================================================

difference_rows = []


for band_name in BANDS:

    difference = band_differences[
        band_name
    ]


    for channel_index, channel in enumerate(
        CHANNELS
    ):

        difference_rows.append(
            {
                "band":
                    band_name,

                "channel":
                    channel,

                "T1_V4_minus_T1_V1":
                    difference[
                        channel_index
                    ]
            }
        )


difference_table = pd.DataFrame(
    difference_rows
)


difference_table.to_csv(
    OUT
    / "V4_minus_V1_topography_channel_differences.csv",
    index=False
)


# =============================================================================
# 26. FIGURE CAPTIONS
# =============================================================================

captions = pd.DataFrame(
    [
        {
            "Figure":
                "Figure 5C",

            "Caption":
                (
                    "Scalp distribution of relative theta power during "
                    "T1_V1 and T1_V4 of the first complete marker-defined "
                    "cycle in the Active group. The difference map shows "
                    "T1_V4 minus T1_V1. Relative power was first averaged "
                    "across the nine retained 2-s epochs within each "
                    "participant and phase and then across the eight "
                    "participants. The scalp maps are descriptive and do "
                    "not represent electrode-wise statistical inference."
                )
        },

        {
            "Figure":
                "Figure S2",

            "Caption":
                (
                    "Relative theta scalp topographies across the sequential "
                    "T1_V1, T1_V2, T1_V3, and T1_V4 marker-defined phases "
                    "of the first complete Active cycle. Each participant "
                    "contributed nine clean 2-s epochs per phase. All panels "
                    "use a common color scale. Marker identity is confounded "
                    "with temporal order."
                )
        },

        {
            "Figure":
                "Figure S3",

            "Caption":
                (
                    "Relative alpha scalp topographies across the four "
                    "sequential marker-defined phases in the Active group. "
                    "The maps show participant-averaged descriptive spatial "
                    "distributions; no electrode-wise statistical testing "
                    "was performed."
                )
        },

        {
            "Figure":
                "Figure S4",

            "Caption":
                (
                    "Relative beta scalp topographies across the four "
                    "sequential marker-defined phases in the Active group. "
                    "The maps are descriptive and no electrode-wise "
                    "statistical testing was performed."
                )
        },

        {
            "Figure":
                "Figure S5",

            "Caption":
                (
                    "Descriptive changes in relative theta power during "
                    "T1_V2, T1_V3, and T1_V4 relative to T1_V1. Difference "
                    "maps were calculated from participant-level phase "
                    "averages and are intended to visualize spatial "
                    "evolution rather than electrode-level statistical "
                    "significance."
                )
        },

        {
            "Figure":
                "Figure S6",

            "Caption":
                (
                    "Descriptive T1_V4-minus-T1_V1 scalp differences for "
                    "relative theta, alpha, and beta power. Symmetric color "
                    "scales are used within each frequency band. Because "
                    "each panel uses a band-specific scale, color intensity "
                    "should not be compared quantitatively across bands. "
                    "No electrode-wise inferential testing was performed."
                )
        }
    ]
)


captions.to_csv(
    OUT
    / "topography_figure_captions.csv",
    index=False
)


print("\n" + "=" * 110)
print("TOPOGRAPHY FIGURE CAPTIONS")
print("=" * 110)

display(
    captions
)


# =============================================================================
# 27. FIGURE MANIFEST
# =============================================================================

figure_manifest_df = pd.DataFrame(
    figure_manifest
)


figure_manifest_df.to_csv(
    OUT
    / "topography_figure_manifest.csv",
    index=False
)


print("\n" + "=" * 110)
print("FIGURE MANIFEST")
print("=" * 110)

display(
    figure_manifest_df
)


# =============================================================================
# 28. FINAL OUTPUT FILE CHECK
# =============================================================================

expected_figure_stems = [
    "figure5C_theta_scalp_V1_V4_difference",
    "figureS2_four_phase_theta_topography",
    "figureS3_four_phase_alpha_topography",
    "figureS4_four_phase_beta_topography",
    "figureS5_theta_change_relative_to_V1",
    "figureS6_V4_minus_V1_band_difference_topographies"
]


file_checks = {}


for stem in expected_figure_stems:

    file_checks[
        f"{stem}_PNG"
    ] = (
        OUT
        / f"{stem}.png"
    ).exists()


    file_checks[
        f"{stem}_PDF"
    ] = (
        OUT
        / f"{stem}.pdf"
    ).exists()


# =============================================================================
# 29. FINAL INTEGRITY AUDIT
# =============================================================================

final_checks = {}


final_checks[
    "source_epochs_288"
] = (
    len(epochs)
    ==
    EXPECTED_TOTAL_EPOCHS
)


final_checks[
    "8_participants"
] = (
    structured.shape[0]
    ==
    N_SUBJECTS
)


final_checks[
    "4_phases"
] = (
    structured.shape[1]
    ==
    N_PHASES
)


final_checks[
    "9_epochs_per_phase"
] = (
    structured.shape[2]
    ==
    EPOCHS_PER_PHASE
)


final_checks[
    "14_channels"
] = (
    structured.shape[3]
    ==
    N_CHANNELS
)


final_checks[
    "theta_maps_complete"
] = all(
    np.isfinite(
        group_maps[
            "Theta"
        ][
            phase
        ]
    ).all()
    for phase in PHASES
)


final_checks[
    "alpha_maps_complete"
] = all(
    np.isfinite(
        group_maps[
            "Alpha"
        ][
            phase
        ]
    ).all()
    for phase in PHASES
)


final_checks[
    "beta_maps_complete"
] = all(
    np.isfinite(
        group_maps[
            "Beta"
        ][
            phase
        ]
    ).all()
    for phase in PHASES
)


final_checks[
    "participant_first_aggregation"
] = True


final_checks[
    "numpy2_trapezoid_used"
] = True


final_checks[
    "no_new_inferential_tests"
] = True


final_checks[
    "no_electrode_level_testing"
] = True


final_checks[
    "no_cluster_statistics"
] = True


final_checks[
    "no_significance_masking"
] = True


final_checks[
    "legacy_connectivity_not_used"
] = True


final_checks[
    "neutral_marker_semantics"
] = True


final_checks.update(
    file_checks
)


overall_pass = all(
    final_checks.values()
)


print("\n" + "=" * 110)
print("TASK 10C FINAL INTEGRITY AUDIT")
print("=" * 110)


for key, value in final_checks.items():

    print(
        f"{key:<75}: "
        f"{'PASS' if value else 'FAIL'}"
    )


print("-" * 110)

print(
    "OVERALL:",
    "PASS"
    if overall_pass
    else "FAIL"
)

print("-" * 110)


if not overall_pass:

    failed = [
        key
        for key, value in final_checks.items()
        if not value
    ]

    raise RuntimeError(
        "\nTask 10C integrity audit failed.\n"
        f"Failed checks:\n{failed}"
    )


# =============================================================================
# 30. COMPLETION SUMMARY
# =============================================================================

print("\n" + "=" * 110)
print("TASK 10C COMPLETE")
print("=" * 110)


print(
    f"""
SOURCE RECONSTRUCTION
---------------------
The frozen Task 8B FIF contains:

    event_id = {epochs.event_id}

Phase identity was therefore not obtained from MNE event IDs.

Instead, the cell first verified the frozen Task 8B structure and then
reconstructed the predetermined ordering:

    8 participants
    × 4 phases
    × 9 epochs per phase
    = 288 epochs

Within each participant:

    T1_V1 → T1_V2 → T1_V3 → T1_V4


SPECTRAL DEFINITIONS
--------------------
Theta:
    4–8 Hz

Alpha:
    8–13 Hz

Beta:
    13–30 Hz

Relative-power denominator:
    4–30 Hz

Numerical integration:
    np.trapezoid()


GENERATED FIGURES
-----------------
Figure 5C
    T1_V1 theta
    T1_V4 theta
    T1_V4 − T1_V1 theta difference

Figure S2
    T1_V1 → T1_V2 → T1_V3 → T1_V4 theta scalp maps

Figure S3
    T1_V1 → T1_V2 → T1_V3 → T1_V4 alpha scalp maps

Figure S4
    T1_V1 → T1_V2 → T1_V3 → T1_V4 beta scalp maps

Figure S5
    T1_V2 − T1_V1
    T1_V3 − T1_V1
    T1_V4 − T1_V1
    theta difference maps

Figure S6
    Theta, alpha, and beta T1_V4 − T1_V1 difference maps


INTERPRETIVE FREEZE
-------------------
1. These scalp maps are descriptive visualizations.

2. No electrode-wise hypothesis tests were performed.

3. No cluster-based tests were performed.

4. No significance masking was applied.

5. Difference maps do not imply that individual electrodes are
   statistically significant.

6. The participant remains the aggregation unit:
       9 epochs → participant phase map → group map.

7. T1_V1–T1_V4 remain neutral marker labels.

8. Phase identity remains confounded with temporal order.

9. These visualizations do not alter the frozen Task 8C inference.

10. The previously established frontal and posterior T1_V1–T1_V4 theta
    findings remain secondary exact paired ROI-level findings because
    the four-phase omnibus tests did not survive FDR correction.


OUTPUT DIRECTORY
----------------
{OUT}
"""
)


print("=" * 110)

In [ ]:
# =============================================================================
# TASK 10D — COMPOSITE MULTI-PANEL EEG SCALP FIGURES
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# FIGURE 6
# --------
# 3 × 4 publication panel:
#
#             T1_V1      T1_V2      T1_V3      T1_V4
# Theta       A           B           C           D
# Alpha       E           F           G           H
# Beta        I           J           K           L
#
# FIGURE 7
# --------
# V4 − V1 difference maps:
#
# A Theta     B Alpha     C Beta
#
# IMPORTANT
# ---------
# Descriptive scalp visualization only.
# No new inferential testing.
# No electrode-wise significance.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "figure.titlesize": 12,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }
)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK8B = (
    BASE
    / "task08b_frozen_first_cycle_epochs"
)

OUT = (
    BASE
    / "task10d_composite_scalp_panels"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


EPOCH_FILE = (
    TASK8B
    / "active_first_cycle_fixed9_epo.fif"
)


if not EPOCH_FILE.exists():

    candidates = sorted(
        TASK8B.rglob("*epo.fif")
    )

    if not candidates:

        raise FileNotFoundError(
            f"No Task 8B epochs FIF found in:\n{TASK8B}"
        )

    EPOCH_FILE = candidates[0]


print("=" * 110)
print("TASK 10D — COMPOSITE EEG SCALP PANELS")
print("=" * 110)

print(f"\nInput:\n{EPOCH_FILE}")
print(f"\nOutput:\n{OUT}")


# =============================================================================
# 2. FROZEN DESIGN
# =============================================================================

N_SUBJECTS = 8
N_PHASES = 4
EPOCHS_PER_PHASE = 9
N_CHANNELS = 14
EXPECTED_EPOCHS = 288

PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]

BANDS = {
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0)
}

DENOMINATOR = (
    4.0,
    30.0
)

CHANNELS = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


# =============================================================================
# 3. LOAD EPOCHS
# =============================================================================

epochs = mne.read_epochs(
    EPOCH_FILE,
    preload=True,
    verbose=False
)


print("\nEpoch object:")
print(epochs)

print("\nEvent IDs:")
print(epochs.event_id)

print("\nChannels:")
print(epochs.ch_names)


# =============================================================================
# 4. STRUCTURAL VALIDATION
# =============================================================================

if len(epochs) != EXPECTED_EPOCHS:

    raise RuntimeError(
        f"Expected 288 frozen epochs, found {len(epochs)}."
    )


if not np.isclose(
    epochs.info["sfreq"],
    256.0
):

    raise RuntimeError(
        f"Expected 256 Hz, found {epochs.info['sfreq']}."
    )


if set(
    ch.upper()
    for ch in epochs.ch_names
) != set(
    ch.upper()
    for ch in CHANNELS
):

    raise RuntimeError(
        "Frozen EEG channels do not match expected 14-channel montage."
    )


# =============================================================================
# 5. REORDER CHANNELS
# =============================================================================

lookup = {
    ch.upper(): i
    for i, ch in enumerate(
        epochs.ch_names
    )
}


indices = [
    lookup[ch.upper()]
    for ch in CHANNELS
]


data = epochs.get_data(
    copy=True
)


data = data[
    :,
    indices,
    :
]


n_times = data.shape[-1]


# =============================================================================
# 6. RECONSTRUCT FROZEN ORDER
# =============================================================================

structured = data.reshape(
    N_SUBJECTS,
    N_PHASES,
    EPOCHS_PER_PHASE,
    N_CHANNELS,
    n_times
)


print(
    "\nFrozen reconstructed shape:"
)

print(
    structured.shape
)


if structured.shape[:4] != (
    8,
    4,
    9,
    14
):

    raise RuntimeError(
        "Frozen Task 8B structure is inconsistent."
    )


# =============================================================================
# 7. SENSOR GEOMETRY
# =============================================================================

info = mne.create_info(
    ch_names=CHANNELS,
    sfreq=256.0,
    ch_types="eeg"
)


montage = mne.channels.make_standard_montage(
    "standard_1020"
)


info.set_montage(
    montage,
    on_missing="raise"
)


# =============================================================================
# 8. RELATIVE BAND POWER
# =============================================================================

def relative_band_power(
    epoch_data,
    band,
    sfreq=256.0,
    denominator=(4.0, 30.0)
):

    nperseg = min(
        512,
        epoch_data.shape[-1]
    )


    freqs, psd = welch(
        epoch_data,
        fs=sfreq,
        nperseg=nperseg,
        axis=-1
    )


    band_mask = (
        (freqs >= band[0])
        &
        (freqs < band[1])
    )


    denominator_mask = (
        (freqs >= denominator[0])
        &
        (freqs < denominator[1])
    )


    band_power = np.trapezoid(
        psd[..., band_mask],
        x=freqs[band_mask],
        axis=-1
    )


    total_power = np.trapezoid(
        psd[..., denominator_mask],
        x=freqs[denominator_mask],
        axis=-1
    )


    relative = np.divide(
        band_power,
        total_power,
        out=np.full_like(
            band_power,
            np.nan,
            dtype=float
        ),
        where=total_power > 0
    )


    return relative


# =============================================================================
# 9. PARTICIPANT-FIRST AGGREGATION
# =============================================================================

subject_maps = {}

group_maps = {}


for band_name, band_range in BANDS.items():

    subject_maps[band_name] = {}

    group_maps[band_name] = {}


    for phase_index, phase in enumerate(
        PHASES
    ):

        participant_maps = []


        for subject_index in range(
            N_SUBJECTS
        ):

            subject_phase = structured[
                subject_index,
                phase_index,
                :,
                :,
                :
            ]


            relative = relative_band_power(
                subject_phase,
                band_range,
                sfreq=256.0,
                denominator=DENOMINATOR
            )


            participant_map = np.mean(
                relative,
                axis=0
            )


            participant_maps.append(
                participant_map
            )


        participant_maps = np.stack(
            participant_maps,
            axis=0
        )


        subject_maps[
            band_name
        ][
            phase
        ] = participant_maps


        group_maps[
            band_name
        ][
            phase
        ] = np.mean(
            participant_maps,
            axis=0
        )


print(
    "\nParticipant-first spectral aggregation complete."
)


# =============================================================================
# 10. GENERIC TOPOGRAPHY FUNCTION
# =============================================================================

def draw_topomap(
    values,
    ax,
    title,
    vlim,
    cmap=None
):

    kwargs = {
        "axes": ax,
        "show": False,
        "sensors": True,
        "contours": 6,
        "outlines": "head",
        "extrapolate": "head",
        "vlim": vlim
    }


    if cmap is not None:
        kwargs["cmap"] = cmap


    image, _ = mne.viz.plot_topomap(
        values,
        info,
        **kwargs
    )


    ax.set_title(
        title,
        fontsize=10,
        fontweight="bold",
        pad=5
    )


    return image


# =============================================================================
# 11. FIGURE 6 — 3 × 4 COMPOSITE PANEL
# =============================================================================

band_order = [
    "Theta",
    "Alpha",
    "Beta"
]


panel_letters = list(
    "ABCDEFGHIJKL"
)


fig, axes = plt.subplots(
    3,
    4,
    figsize=(
        12.5,
        9.0
    )
)


letter_index = 0


for row_index, band_name in enumerate(
    band_order
):

    # ---------------------------------------------------------------------
    # Common scale within each frequency band
    # ---------------------------------------------------------------------

    all_values = np.concatenate(
        [
            group_maps[
                band_name
            ][
                phase
            ]
            for phase in PHASES
        ]
    )


    vmin = np.min(
        all_values
    )

    vmax = np.max(
        all_values
    )


    row_images = []


    for column_index, phase in enumerate(
        PHASES
    ):

        ax = axes[
            row_index,
            column_index
        ]


        letter = panel_letters[
            letter_index
        ]

        letter_index += 1


        image = draw_topomap(
            values=
                group_maps[
                    band_name
                ][
                    phase
                ],

            ax=
                ax,

            title=
                f"{letter}  {phase}",

            vlim=(
                vmin,
                vmax
            )
        )


        row_images.append(
            image
        )


    # ---------------------------------------------------------------------
    # Row label
    # ---------------------------------------------------------------------

    axes[
        row_index,
        0
    ].text(
        -0.42,
        0.50,
        band_name,
        transform=axes[
            row_index,
            0
        ].transAxes,
        rotation=90,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold"
    )


    # ---------------------------------------------------------------------
    # One colorbar per frequency-band row
    # ---------------------------------------------------------------------

    cbar = fig.colorbar(
        row_images[-1],
        ax=axes[
            row_index,
            :
        ].tolist(),
        shrink=0.68,
        pad=0.025
    )


    cbar.set_label(
        f"Relative {band_name.lower()} power",
        fontsize=9
    )


fig.text(
    0.5,
    0.025,
    "Scalp maps show participant-averaged relative spectral power. "
    "Each frequency-band row uses a common scale across T1_V1–T1_V4. "
    "Maps are descriptive; no electrode-wise statistical inference was performed.",
    ha="center",
    va="bottom",
    fontsize=8.5,
    style="italic"
)


fig.subplots_adjust(
    left=0.075,
    right=0.90,
    top=0.97,
    bottom=0.09,
    hspace=0.28,
    wspace=0.18
)


FIG6_PNG = (
    OUT
    / "figure6_multiband_four_phase_scalp_panel.png"
)

FIG6_PDF = (
    OUT
    / "figure6_multiband_four_phase_scalp_panel.pdf"
)


fig.savefig(
    FIG6_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG6_PDF,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print(
    "\nSaved Figure 6:"
)

print(
    FIG6_PNG
)

print(
    FIG6_PDF
)


# =============================================================================
# 12. FIGURE 7 — V4 − V1 DIFFERENCE PANEL
# =============================================================================

differences = {}


for band_name in band_order:

    differences[
        band_name
    ] = (
        group_maps[
            band_name
        ][
            "T1_V4"
        ]
        -
        group_maps[
            band_name
        ][
            "T1_V1"
        ]
    )


fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        10.5,
        3.7
    )
)


difference_letters = [
    "A",
    "B",
    "C"
]


for ax, band_name, letter in zip(
    axes,
    band_order,
    difference_letters
):

    values = differences[
        band_name
    ]


    limit = np.max(
        np.abs(
            values
        )
    )


    if limit <= 0:
        limit = 1e-6


    image = draw_topomap(
        values=
            values,

        ax=
            ax,

        title=
            f"{letter}  {band_name}",

        vlim=(
            -limit,
            limit
        ),

        cmap=
            "RdBu_r"
    )


    cbar = fig.colorbar(
        image,
        ax=ax,
        shrink=0.72,
        pad=0.035
    )


    cbar.set_label(
        "T1_V4 − T1_V1",
        fontsize=8
    )


fig.text(
    0.5,
    0.025,
    "Descriptive T1_V4-minus-T1_V1 relative-power differences. "
    "Each frequency band uses its own symmetric scale; "
    "color magnitude should not be compared directly across bands.",
    ha="center",
    fontsize=8.4,
    style="italic"
)


fig.subplots_adjust(
    left=0.03,
    right=0.98,
    top=0.91,
    bottom=0.18,
    wspace=0.30
)


FIG7_PNG = (
    OUT
    / "figure7_V4_minus_V1_multiband_scalp_panel.png"
)

FIG7_PDF = (
    OUT
    / "figure7_V4_minus_V1_multiband_scalp_panel.pdf"
)


fig.savefig(
    FIG7_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG7_PDF,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print(
    "\nSaved Figure 7:"
)

print(
    FIG7_PNG
)

print(
    FIG7_PDF
)


# =============================================================================
# 13. FIGURE 8 — THETA-FOCUSED MAIN-MANUSCRIPT PANEL
# =============================================================================
#
# This is the most useful compact scalp figure for the main manuscript:
#
# A T1_V1
# B T1_V2
# C T1_V3
# D T1_V4
# E V4−V1
#
# =============================================================================

theta_values_all = np.concatenate(
    [
        group_maps[
            "Theta"
        ][
            phase
        ]
        for phase in PHASES
    ]
)


theta_min = np.min(
    theta_values_all
)

theta_max = np.max(
    theta_values_all
)


theta_difference = differences[
    "Theta"
]


theta_diff_limit = np.max(
    np.abs(
        theta_difference
    )
)


if theta_diff_limit <= 0:
    theta_diff_limit = 1e-6


fig, axes = plt.subplots(
    1,
    5,
    figsize=(
        14.0,
        3.4
    )
)


theta_letters = [
    "A",
    "B",
    "C",
    "D"
]


theta_images = []


for ax, phase, letter in zip(
    axes[:4],
    PHASES,
    theta_letters
):

    image = draw_topomap(
        values=
            group_maps[
                "Theta"
            ][
                phase
            ],

        ax=
            ax,

        title=
            f"{letter}  {phase}",

        vlim=(
            theta_min,
            theta_max
        )
    )


    theta_images.append(
        image
    )


difference_image = draw_topomap(
    values=
        theta_difference,

    ax=
        axes[4],

    title=
        "E  T1_V4 − T1_V1",

    vlim=(
        -theta_diff_limit,
        theta_diff_limit
    ),

    cmap=
        "RdBu_r"
)


cbar1 = fig.colorbar(
    theta_images[-1],
    ax=axes[:4].tolist(),
    shrink=0.70,
    pad=0.025
)


cbar1.set_label(
    "Relative theta power",
    fontsize=9
)


cbar2 = fig.colorbar(
    difference_image,
    ax=axes[4],
    shrink=0.70,
    pad=0.035
)


cbar2.set_label(
    "Change",
    fontsize=9
)


fig.text(
    0.5,
    0.02,
    "Sequential theta scalp dynamics across the first complete marker-defined cycle. "
    "Panel E shows T1_V4 minus T1_V1. "
    "Maps are descriptive and do not indicate electrode-wise statistical significance.",
    ha="center",
    fontsize=8.4,
    style="italic"
)


fig.subplots_adjust(
    left=0.025,
    right=0.94,
    top=0.91,
    bottom=0.18,
    wspace=0.16
)


FIG8_PNG = (
    OUT
    / "figure8_theta_phase_and_difference_panel.png"
)

FIG8_PDF = (
    OUT
    / "figure8_theta_phase_and_difference_panel.pdf"
)


fig.savefig(
    FIG8_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG8_PDF,
    bbox_inches="tight"
)


plt.show()

plt.close(fig)


print(
    "\nSaved Figure 8:"
)

print(
    FIG8_PNG
)

print(
    FIG8_PDF
)


# =============================================================================
# 14. EXPORT VALUES USED IN COMPOSITE FIGURES
# =============================================================================

rows = []


for band_name in band_order:

    for phase in PHASES:

        values = group_maps[
            band_name
        ][
            phase
        ]


        participant_values = subject_maps[
            band_name
        ][
            phase
        ]


        for channel_index, channel in enumerate(
            CHANNELS
        ):

            rows.append(
                {
                    "band":
                        band_name,

                    "phase":
                        phase,

                    "channel":
                        channel,

                    "mean_relative_power":
                        values[
                            channel_index
                        ],

                    "participant_sd":
                        np.std(
                            participant_values[
                                :,
                                channel_index
                            ],
                            ddof=1
                        ),

                    "n_participants":
                        N_SUBJECTS
                }
            )


panel_values = pd.DataFrame(
    rows
)


panel_values.to_csv(
    OUT
    / "composite_panel_channel_values.csv",
    index=False
)


# =============================================================================
# 15. CAPTIONS
# =============================================================================

captions = pd.DataFrame(
    [
        {
            "Figure":
                "Figure 6",

            "Caption":
                (
                    "Scalp distributions of relative theta, alpha, and beta "
                    "power across the four sequential marker-defined phases "
                    "of the first complete Active cycle. Each participant "
                    "contributed nine clean 2-s epochs per phase. Spectral "
                    "power was first averaged within participant and phase "
                    "and subsequently across participants. A common color "
                    "scale is used across the four phases within each "
                    "frequency band. The maps are descriptive; no "
                    "electrode-wise statistical inference was performed."
                )
        },

        {
            "Figure":
                "Figure 7",

            "Caption":
                (
                    "Scalp distributions of the T1_V4-minus-T1_V1 change "
                    "in relative theta, alpha, and beta power. Negative "
                    "values indicate lower relative power at T1_V4 than "
                    "T1_V1 and positive values indicate higher relative "
                    "power. Each frequency band uses a band-specific "
                    "symmetric scale. These maps provide descriptive "
                    "spatial localization and do not represent "
                    "electrode-wise statistical tests."
                )
        },

        {
            "Figure":
                "Figure 8",

            "Caption":
                (
                    "Sequential theta-band scalp dynamics during the first "
                    "complete T1_V1→T1_V2→T1_V3→T1_V4 marker-defined "
                    "cycle in the Active group. Panels A–D show relative "
                    "theta power at each phase, and Panel E shows the "
                    "T1_V4-minus-T1_V1 difference. The spatial maps "
                    "complement the prespecified ROI-level analysis and "
                    "should not be interpreted as electrode-level "
                    "significance maps. Marker identity is confounded with "
                    "temporal order."
                )
        }
    ]
)


captions.to_csv(
    OUT
    / "composite_figure_captions.csv",
    index=False
)


print("\n" + "=" * 110)
print("CAPTIONS")
print("=" * 110)

display(
    captions
)


# =============================================================================
# 16. INTEGRITY CHECK
# =============================================================================

checks = {
    "source_288_epochs":
        len(epochs) == 288,

    "reconstructed_8_subjects":
        structured.shape[0] == 8,

    "reconstructed_4_phases":
        structured.shape[1] == 4,

    "reconstructed_9_epochs_per_phase":
        structured.shape[2] == 9,

    "reconstructed_14_channels":
        structured.shape[3] == 14,

    "figure6_png":
        FIG6_PNG.exists(),

    "figure6_pdf":
        FIG6_PDF.exists(),

    "figure7_png":
        FIG7_PNG.exists(),

    "figure7_pdf":
        FIG7_PDF.exists(),

    "figure8_png":
        FIG8_PNG.exists(),

    "figure8_pdf":
        FIG8_PDF.exists(),

    "no_new_inferential_testing":
        True,

    "participant_first_aggregation":
        True,

    "no_electrode_level_inference":
        True,

    "neutral_marker_labels":
        True
}


print("\n" + "=" * 110)
print("TASK 10D INTEGRITY CHECK")
print("=" * 110)


for key, value in checks.items():

    print(
        f"{key:<60}: "
        f"{'PASS' if value else 'FAIL'}"
    )


overall_pass = all(
    checks.values()
)


print("-" * 110)

print(
    "OVERALL:",
    "PASS"
    if overall_pass
    else "FAIL"
)

print("-" * 110)


if not overall_pass:

    raise RuntimeError(
        "Task 10D integrity check failed."
    )


# =============================================================================
# 17. COMPLETE
# =============================================================================

print("\n" + "=" * 110)
print("TASK 10D COMPLETE")
print("=" * 110)


print(
    f"""
Three composite panel figures were generated.

FIGURE 6 — COMPLETE SPECTRAL PANEL
----------------------------------
        T1_V1   T1_V2   T1_V3   T1_V4

Theta     A       B       C       D
Alpha     E       F       G       H
Beta      I       J       K       L


FIGURE 7 — DIFFERENCE PANEL
---------------------------
A   Theta T1_V4 − T1_V1
B   Alpha T1_V4 − T1_V1
C   Beta  T1_V4 − T1_V1


FIGURE 8 — THETA-FOCUSED PANEL
------------------------------
A   T1_V1
B   T1_V2
C   T1_V3
D   T1_V4
E   T1_V4 − T1_V1


OUTPUT DIRECTORY
----------------
{OUT}


RECOMMENDED MANUSCRIPT USE
--------------------------
Use Figure 8 in the MAIN manuscript.

It directly supports the most interesting within-Active spectral result
without overwhelming the reader.

Use Figure 6 in the Supplementary Material to show the complete
theta/alpha/beta spatial pattern.

Figure 7 can either be supplementary or incorporated into Figure 8
if a more compact main figure is preferred.

No new inferential tests were performed.
"""
)


print("=" * 110)

In [ ]:
# =============================================================================
# FIGURE 8 — THETA-FOCUSED MAIN-MANUSCRIPT PANEL
# CORRECTED LAYOUT WITH EXTERNAL COLORBARS
# =============================================================================
#
# Layout:
#
# A      B      C      D      CB1      E      CB2
# V1     V2     V3     V4    theta    ΔV4-V1  change
#
# The colorbars now have dedicated axes and cannot overlap the scalp maps.
# =============================================================================

from matplotlib.gridspec import GridSpec


theta_values_all = np.concatenate(
    [
        group_maps["Theta"][phase]
        for phase in PHASES
    ]
)

theta_min = np.min(theta_values_all)
theta_max = np.max(theta_values_all)


theta_difference = (
    group_maps["Theta"]["T1_V4"]
    -
    group_maps["Theta"]["T1_V1"]
)


theta_diff_limit = np.max(
    np.abs(theta_difference)
)

if theta_diff_limit <= 0:
    theta_diff_limit = 1e-6


# -----------------------------------------------------------------------------
# Create figure with dedicated colorbar columns
# -----------------------------------------------------------------------------

fig = plt.figure(
    figsize=(15.2, 3.8)
)


gs = GridSpec(
    nrows=1,
    ncols=7,
    figure=fig,

    # Four scalp maps, narrow colorbar, difference map, narrow colorbar
    width_ratios=[
        1.0,
        1.0,
        1.0,
        1.0,
        0.055,
        1.0,
        0.055
    ],

    wspace=0.35
)


ax_A = fig.add_subplot(gs[0, 0])
ax_B = fig.add_subplot(gs[0, 1])
ax_C = fig.add_subplot(gs[0, 2])
ax_D = fig.add_subplot(gs[0, 3])

cax_theta = fig.add_subplot(gs[0, 4])

ax_E = fig.add_subplot(gs[0, 5])

cax_difference = fig.add_subplot(gs[0, 6])


phase_axes = [
    ax_A,
    ax_B,
    ax_C,
    ax_D
]


panel_letters = [
    "A",
    "B",
    "C",
    "D"
]


# -----------------------------------------------------------------------------
# A–D: four sequential theta scalp maps
# -----------------------------------------------------------------------------

theta_images = []


for ax, phase, letter in zip(
    phase_axes,
    PHASES,
    panel_letters
):

    image = draw_topomap(
        values=
            group_maps["Theta"][phase],

        ax=
            ax,

        title=
            f"{letter}  {phase}",

        vlim=(
            theta_min,
            theta_max
        )
    )

    theta_images.append(image)


# -----------------------------------------------------------------------------
# Shared theta colorbar in dedicated column
# -----------------------------------------------------------------------------

cbar_theta = fig.colorbar(
    theta_images[-1],
    cax=cax_theta
)


cbar_theta.set_label(
    "Relative theta power",
    fontsize=9,
    labelpad=8
)


cbar_theta.ax.tick_params(
    labelsize=8
)


# -----------------------------------------------------------------------------
# E: T1_V4 − T1_V1 difference
# -----------------------------------------------------------------------------

difference_image = draw_topomap(
    values=
        theta_difference,

    ax=
        ax_E,

    title=
        "E  T1_V4 − T1_V1",

    vlim=(
        -theta_diff_limit,
        theta_diff_limit
    ),

    cmap=
        "RdBu_r"
)


# -----------------------------------------------------------------------------
# Difference colorbar in dedicated column
# -----------------------------------------------------------------------------

cbar_difference = fig.colorbar(
    difference_image,
    cax=cax_difference
)


cbar_difference.set_label(
    "Change in relative theta power",
    fontsize=9,
    labelpad=8
)


cbar_difference.ax.tick_params(
    labelsize=8
)


# -----------------------------------------------------------------------------
# Footer
# -----------------------------------------------------------------------------

fig.text(
    0.5,
    0.035,

    "Sequential theta scalp dynamics across the first complete "
    "marker-defined cycle. Panel E shows T1_V4 minus T1_V1. "
    "Maps are descriptive and do not indicate electrode-wise "
    "statistical significance.",

    ha="center",
    va="bottom",
    fontsize=8.5,
    style="italic"
)


# -----------------------------------------------------------------------------
# Final spacing
# -----------------------------------------------------------------------------

fig.subplots_adjust(
    left=0.025,
    right=0.975,
    top=0.92,
    bottom=0.19
)


# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------

FIG8_PNG = (
    OUT
    / "figure8_theta_phase_and_difference_panel_corrected.png"
)

FIG8_PDF = (
    OUT
    / "figure8_theta_phase_and_difference_panel_corrected.pdf"
)


fig.savefig(
    FIG8_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    FIG8_PDF,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()

plt.close(fig)


print("\nCorrected Figure 8 saved:")

print(FIG8_PNG)
print(FIG8_PDF)

In [ ]:
# =============================================================================
# TASK 10E — PUBLICATION-READY POWER SPECTRUM PANEL
# COMPLETE SELF-CONTAINED CELL
# =============================================================================
#
# PANEL A — Frontal PSD across T1_V1–T1_V4
# PANEL B — Central/Temporal PSD across T1_V1–T1_V4
# PANEL C — Posterior PSD across T1_V1–T1_V4
# PANEL D — T1_V4 − T1_V1 spectral change for the three ROIs
#
# IMPORTANT
# ---------
# Descriptive visualization only.
# No frequency-bin-wise inference.
# Participant-first aggregation.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from scipy.signal import welch
from IPython.display import display


try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "legend.fontsize": 8.5,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "pdf.fonttype": 42,
        "ps.fonttype": 42
    }
)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data"
)

BASE = (
    ROOT
    / "derivatives"
    / "tine_analysis"
)

TASK8B = (
    BASE
    / "task08b_frozen_first_cycle_epochs"
)

OUT = (
    BASE
    / "task10e_power_spectrum_panels"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)


EPOCH_FILE = (
    TASK8B
    / "active_first_cycle_fixed9_epo.fif"
)


if not EPOCH_FILE.exists():

    candidates = sorted(
        TASK8B.rglob("*epo.fif")
    )

    if not candidates:

        raise FileNotFoundError(
            f"No Task 8B epoch FIF found under:\n{TASK8B}"
        )

    EPOCH_FILE = candidates[0]


print("=" * 110)
print("TASK 10E — POWER SPECTRUM PANEL")
print("=" * 110)

print(f"\nInput:\n{EPOCH_FILE}")
print(f"\nOutput:\n{OUT}")


# =============================================================================
# 2. FROZEN DESIGN
# =============================================================================

N_SUBJECTS = 8
N_PHASES = 4
EPOCHS_PER_PHASE = 9
N_CHANNELS = 14
EXPECTED_EPOCHS = 288
SFREQ = 256.0


SUBJECTS = [
    f"sub-{i:03d}"
    for i in range(1, 9)
]


PHASES = [
    "T1_V1",
    "T1_V2",
    "T1_V3",
    "T1_V4"
]


CHANNELS = [
    "AF3",
    "F7",
    "F3",
    "FC5",
    "T7",
    "P7",
    "O1",
    "O2",
    "P8",
    "T8",
    "FC6",
    "F4",
    "F8",
    "AF4"
]


ROIS = {

    "Frontal": [
        "AF3",
        "F7",
        "F3",
        "F4",
        "F8",
        "AF4"
    ],

    "Central/Temporal": [
        "FC5",
        "T7",
        "T8",
        "FC6"
    ],

    "Posterior": [
        "P7",
        "O1",
        "O2",
        "P8"
    ]
}


# =============================================================================
# 3. LOAD FROZEN TASK 8B EPOCHS
# =============================================================================

epochs = mne.read_epochs(
    EPOCH_FILE,
    preload=True,
    verbose=False
)


print("\nEpoch object:")
print(epochs)

print("\nEvent IDs:")
print(epochs.event_id)

print("\nChannels:")
print(epochs.ch_names)


# =============================================================================
# 4. STRUCTURAL VALIDATION
# =============================================================================

if len(epochs) != EXPECTED_EPOCHS:

    raise RuntimeError(
        f"Expected {EXPECTED_EPOCHS} epochs; found {len(epochs)}."
    )


if not np.isclose(
    epochs.info["sfreq"],
    SFREQ
):

    raise RuntimeError(
        f"Expected {SFREQ} Hz; "
        f"found {epochs.info['sfreq']} Hz."
    )


if set(
    ch.upper()
    for ch in epochs.ch_names
) != set(
    ch.upper()
    for ch in CHANNELS
):

    raise RuntimeError(
        "The frozen 14-channel montage does not match expectations."
    )


# =============================================================================
# 5. REORDER CHANNELS
# =============================================================================

lookup = {
    ch.upper(): i
    for i, ch in enumerate(
        epochs.ch_names
    )
}


indices = [
    lookup[ch.upper()]
    for ch in CHANNELS
]


data = epochs.get_data(
    copy=True
)


data = data[
    :,
    indices,
    :
]


n_times = data.shape[-1]


# =============================================================================
# 6. RECONSTRUCT FROZEN 8 × 4 × 9 STRUCTURE
# =============================================================================

structured = data.reshape(
    N_SUBJECTS,
    N_PHASES,
    EPOCHS_PER_PHASE,
    N_CHANNELS,
    n_times
)


print(
    "\nReconstructed array "
    "(subjects × phases × epochs × channels × samples):"
)

print(
    structured.shape
)


if structured.shape[:4] != (
    8,
    4,
    9,
    14
):

    raise RuntimeError(
        "Frozen Task 8B structure is inconsistent."
    )


# =============================================================================
# 7. ROI INDICES
# =============================================================================

channel_index = {
    channel: i
    for i, channel in enumerate(
        CHANNELS
    )
}


ROI_INDICES = {
    roi: [
        channel_index[ch]
        for ch in roi_channels
    ]

    for roi, roi_channels in ROIS.items()
}


print("\nROI definitions:")

for roi, roi_channels in ROIS.items():

    print(
        f"  {roi:<20}: "
        f"{', '.join(roi_channels)}"
    )


# =============================================================================
# 8. WELCH PSD
# =============================================================================
#
# Each epoch is 2 seconds = 512 samples.
#
# Welch is calculated for every epoch and channel.
# The resulting PSD is converted to µV²/Hz for readable EEG units.
#
# =============================================================================

freqs, psd = welch(
    structured,
    fs=SFREQ,
    nperseg=min(
        512,
        n_times
    ),
    axis=-1
)


# Convert V²/Hz -> µV²/Hz

psd_uv = (
    psd
    *
    1e12
)


print("\nPSD array shape:")

print(
    psd_uv.shape
)

print(
    "\nFrequency resolution:"
)

print(
    f"{freqs[1] - freqs[0]:.3f} Hz"
)


# =============================================================================
# 9. RESTRICT VISUALIZATION TO 4–30 Hz
# =============================================================================

frequency_mask = (
    (freqs >= 4.0)
    &
    (freqs <= 30.0)
)


freq_plot = freqs[
    frequency_mask
]


psd_plot = psd_uv[
    ...,
    frequency_mask
]


# =============================================================================
# 10. PARTICIPANT-FIRST ROI PSD
# =============================================================================
#
# For each participant × phase × ROI:
#
# 1. average the 9 retained epochs
# 2. average channels in the prespecified ROI
#
# Result:
#
#     8 participants × frequencies
#
# for each phase and ROI.
#
# =============================================================================

subject_roi_psd = {}


for roi_name, roi_idx in ROI_INDICES.items():

    subject_roi_psd[
        roi_name
    ] = {}


    for phase_index, phase in enumerate(
        PHASES
    ):

        participant_curves = []


        for subject_index in range(
            N_SUBJECTS
        ):

            x = psd_plot[
                subject_index,
                phase_index,
                :,
                roi_idx,
                :
            ]


            # x:
            # epochs × ROI channels × frequencies

            epoch_average = np.mean(
                x,
                axis=0
            )


            roi_average = np.mean(
                epoch_average,
                axis=0
            )


            participant_curves.append(
                roi_average
            )


        subject_roi_psd[
            roi_name
        ][
            phase
        ] = np.stack(
            participant_curves,
            axis=0
        )


# =============================================================================
# 11. GROUP MEANS AND SEM
# =============================================================================

group_psd = {}


for roi_name in ROIS:

    group_psd[
        roi_name
    ] = {}


    for phase in PHASES:

        arr = subject_roi_psd[
            roi_name
        ][
            phase
        ]


        mean = np.mean(
            arr,
            axis=0
        )


        sem = (
            np.std(
                arr,
                axis=0,
                ddof=1
            )
            /
            np.sqrt(
                N_SUBJECTS
            )
        )


        group_psd[
            roi_name
        ][
            phase
        ] = {
            "mean": mean,
            "sem": sem
        }


# =============================================================================
# 12. BAND SHADING HELPER
# =============================================================================

def add_band_background(
    ax
):

    # Very light background regions

    ax.axvspan(
        4,
        8,
        alpha=0.045
    )

    ax.axvspan(
        8,
        13,
        alpha=0.075
    )

    ax.axvspan(
        13,
        30,
        alpha=0.035
    )


    # Boundaries

    ax.axvline(
        8,
        linewidth=0.8,
        linestyle="--",
        alpha=0.55
    )

    ax.axvline(
        13,
        linewidth=0.8,
        linestyle="--",
        alpha=0.55
    )


    ylim = ax.get_ylim()


    y = (
        ylim[1]
        -
        0.04
        *
        (
            ylim[1]
            -
            ylim[0]
        )
    )


    ax.text(
        6,
        y,
        "θ",
        ha="center",
        va="top",
        fontsize=9
    )

    ax.text(
        10.5,
        y,
        "α",
        ha="center",
        va="top",
        fontsize=9
    )

    ax.text(
        21.5,
        y,
        "β",
        ha="center",
        va="top",
        fontsize=9
    )


# =============================================================================
# 13. FIGURE — 2 × 2 POWER SPECTRUM PANEL
# =============================================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(
        11.5,
        8.0
    )
)


axA = axes[0, 0]
axB = axes[0, 1]
axC = axes[1, 0]
axD = axes[1, 1]


roi_axes = {
    "Frontal": axA,
    "Central/Temporal": axB,
    "Posterior": axC
}


panel_letters = {
    "Frontal": "A",
    "Central/Temporal": "B",
    "Posterior": "C"
}


# =============================================================================
# 14. PANELS A–C — PSD BY PHASE
# =============================================================================

for roi_name, ax in roi_axes.items():

    for phase in PHASES:

        mean = group_psd[
            roi_name
        ][
            phase
        ][
            "mean"
        ]


        sem = group_psd[
            roi_name
        ][
            phase
        ][
            "sem"
        ]


        line = ax.plot(
            freq_plot,
            mean,
            linewidth=1.8,
            label=phase
        )[0]


        ax.fill_between(
            freq_plot,
            mean - sem,
            mean + sem,
            alpha=0.12
        )


    ax.set_xlim(
        4,
        30
    )


    ax.set_xlabel(
        "Frequency (Hz)"
    )


    ax.set_ylabel(
        r"PSD ($\mu V^2$/Hz)"
    )


    ax.set_title(
        f"{panel_letters[roi_name]}  {roi_name}",
        loc="left",
        fontweight="bold"
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


    ax.grid(
        alpha=0.15,
        linewidth=0.5
    )


    # Add frequency-band boundaries after data so labels use correct ylim

    add_band_background(
        ax
    )


# One legend only

axA.legend(
    frameon=False,
    ncol=2,
    loc="upper right"
)


# =============================================================================
# 15. PANEL D — V4 − V1 SPECTRAL CHANGE
# =============================================================================
#
# Participant-level difference first:
#
#     participant V4 PSD − participant V1 PSD
#
# Then group mean and SEM.
#
# =============================================================================

for roi_name in ROIS:

    v1 = subject_roi_psd[
        roi_name
    ][
        "T1_V1"
    ]


    v4 = subject_roi_psd[
        roi_name
    ][
        "T1_V4"
    ]


    difference = (
        v4
        -
        v1
    )


    difference_mean = np.mean(
        difference,
        axis=0
    )


    difference_sem = (
        np.std(
            difference,
            axis=0,
            ddof=1
        )
        /
        np.sqrt(
            N_SUBJECTS
        )
    )


    axD.plot(
        freq_plot,
        difference_mean,
        linewidth=1.8,
        label=roi_name
    )


    axD.fill_between(
        freq_plot,
        difference_mean - difference_sem,
        difference_mean + difference_sem,
        alpha=0.12
    )


axD.axhline(
    0,
    linewidth=1.0,
    linestyle="-"
)


axD.axvline(
    8,
    linewidth=0.8,
    linestyle="--",
    alpha=0.55
)


axD.axvline(
    13,
    linewidth=0.8,
    linestyle="--",
    alpha=0.55
)


axD.set_xlim(
    4,
    30
)


axD.set_xlabel(
    "Frequency (Hz)"
)


axD.set_ylabel(
    r"$\Delta$ PSD ($\mu V^2$/Hz)"
)


axD.set_title(
    "D  T1_V4 − T1_V1",
    loc="left",
    fontweight="bold"
)


axD.legend(
    frameon=False,
    loc="best"
)


axD.spines[
    "top"
].set_visible(
    False
)


axD.spines[
    "right"
].set_visible(
    False
)


axD.grid(
    alpha=0.15,
    linewidth=0.5
)


# Band labels

ylim = axD.get_ylim()

band_label_y = (
    ylim[1]
    -
    0.04
    *
    (
        ylim[1]
        -
        ylim[0]
    )
)


axD.text(
    6,
    band_label_y,
    "θ",
    ha="center",
    va="top"
)


axD.text(
    10.5,
    band_label_y,
    "α",
    ha="center",
    va="top"
)


axD.text(
    21.5,
    band_label_y,
    "β",
    ha="center",
    va="top"
)


# =============================================================================
# 16. FOOTER
# =============================================================================

fig.text(
    0.5,
    0.018,

    "Participant-averaged Welch power spectral density across sequential \n"
    "marker-defined phases.",

    ha="center",
    va="bottom",
    fontsize=8.5,
    style="italic"
)


fig.subplots_adjust(
    left=0.085,
    right=0.97,
    top=0.95,
    bottom=0.11,
    hspace=0.30,
    wspace=0.24
)


# =============================================================================
# 17. SAVE FIGURE
# =============================================================================

FIG_PNG = (
    OUT
    / "figure_power_spectrum_four_phase_panel.png"
)


FIG_PDF = (
    OUT
    / "figure_power_spectrum_four_phase_panel.pdf"
)


fig.savefig(
    FIG_PNG,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    FIG_PDF,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()

plt.close(
    fig
)


print("\nSaved:")

print(
    FIG_PNG
)

print(
    FIG_PDF
)


# =============================================================================
# 18. EXPORT NUMERICAL CURVES
# =============================================================================

rows = []


for roi_name in ROIS:

    for phase in PHASES:

        arr = subject_roi_psd[
            roi_name
        ][
            phase
        ]


        mean = np.mean(
            arr,
            axis=0
        )


        sem = (
            np.std(
                arr,
                axis=0,
                ddof=1
            )
            /
            np.sqrt(
                N_SUBJECTS
            )
        )


        for frequency, m, s in zip(
            freq_plot,
            mean,
            sem
        ):

            rows.append(
                {
                    "ROI":
                        roi_name,

                    "Phase":
                        phase,

                    "Frequency_Hz":
                        frequency,

                    "Mean_PSD_uV2_per_Hz":
                        m,

                    "SEM_PSD_uV2_per_Hz":
                        s,

                    "N":
                        N_SUBJECTS
                }
            )


psd_table = pd.DataFrame(
    rows
)


psd_table.to_csv(
    OUT
    / "power_spectrum_curves.csv",
    index=False
)


# =============================================================================
# 19. EXPORT V4 − V1 CURVES
# =============================================================================

difference_rows = []


for roi_name in ROIS:

    difference = (
        subject_roi_psd[
            roi_name
        ][
            "T1_V4"
        ]
        -
        subject_roi_psd[
            roi_name
        ][
            "T1_V1"
        ]
    )


    mean = np.mean(
        difference,
        axis=0
    )


    sem = (
        np.std(
            difference,
            axis=0,
            ddof=1
        )
        /
        np.sqrt(
            N_SUBJECTS
        )
    )


    for frequency, m, s in zip(
        freq_plot,
        mean,
        sem
    ):

        difference_rows.append(
            {
                "ROI":
                    roi_name,

                "Frequency_Hz":
                    frequency,

                "Mean_V4_minus_V1_PSD":
                    m,

                "SEM_V4_minus_V1_PSD":
                    s,

                "N":
                    N_SUBJECTS
            }
        )


difference_table = pd.DataFrame(
    difference_rows
)


difference_table.to_csv(
    OUT
    / "power_spectrum_V4_minus_V1_curves.csv",
    index=False
)


# =============================================================================
# 20. CAPTION
# =============================================================================

caption = (
    "Power spectral density across sequential marker-defined phases in the "
    "Active group. Panels A–C show participant-averaged Welch spectra for "
    "frontal, Central/Temporal, and posterior regions, respectively, across "
    "T1_V1–T1_V4. Shaded envelopes indicate ±1 SEM across eight participants. "
    "Vertical dashed lines mark the 8-Hz and 13-Hz boundaries separating the "
    "theta, alpha, and beta frequency ranges used in the prespecified spectral "
    "analysis. Panel D shows participant-level T1_V4-minus-T1_V1 spectral "
    "differences for each ROI. Spectral curves are descriptive and no "
    "frequency-bin-wise statistical inference was performed."
)


pd.DataFrame(
    {
        "Figure": [
            "Power spectrum panel"
        ],

        "Caption": [
            caption
        ]
    }
).to_csv(
    OUT
    / "power_spectrum_figure_caption.csv",
    index=False
)


# =============================================================================
# 21. INTEGRITY CHECK
# =============================================================================

checks = {

    "source_288_epochs":
        len(epochs) == 288,

    "8_participants":
        structured.shape[0] == 8,

    "4_phases":
        structured.shape[1] == 4,

    "9_epochs_per_phase":
        structured.shape[2] == 9,

    "14_channels":
        structured.shape[3] == 14,

    "three_prespecified_ROIs":
        len(ROIS) == 3,

    "frequency_range_4_to_30":
        (
            freq_plot.min() >= 4.0
            and
            freq_plot.max() <= 30.0
        ),

    "participant_first_aggregation":
        True,

    "no_frequency_bin_inference":
        True,

    "no_new_hypothesis_tests":
        True,

    "neutral_marker_semantics":
        True,

    "PNG_created":
        FIG_PNG.exists(),

    "PDF_created":
        FIG_PDF.exists()
}


print("\n" + "=" * 110)
print("TASK 10E INTEGRITY CHECK")
print("=" * 110)


for key, value in checks.items():

    print(
        f"{key:<60}: "
        f"{'PASS' if value else 'FAIL'}"
    )


overall_pass = all(
    checks.values()
)


print("-" * 110)

print(
    "OVERALL:",
    "PASS"
    if overall_pass
    else "FAIL"
)

print("-" * 110)


if not overall_pass:

    failed = [
        key
        for key, value in checks.items()
        if not value
    ]

    raise RuntimeError(
        f"Task 10E failed: {failed}"
    )


# =============================================================================
# 22. COMPLETE
# =============================================================================

print("\n" + "=" * 110)
print("TASK 10E COMPLETE")
print("=" * 110)

print(
    f"""
POWER-SPECTRUM PANEL
--------------------

A   Frontal
    T1_V1–T1_V4 PSD

B   Central/Temporal
    T1_V1–T1_V4 PSD

C   Posterior
    T1_V1–T1_V4 PSD

D   T1_V4 − T1_V1
    Frontal, Central/Temporal, Posterior


Frequency display:
    4–30 Hz

Band boundaries:
    Theta = 4–8 Hz
    Alpha = 8–13 Hz
    Beta  = 13–30 Hz

Shading:
    ±1 SEM across N=8 participants

Inference:
    NONE at individual frequency bins


Output:
{OUT}
"""
)

print("=" * 110)

In [ ]:
# =============================================================================
# PUBLICATION-READY 3D BETA NETWORK PANEL
# DATA-FAITHFUL RECOMPUTATION FROM FROZEN MATCHED EPOCHS
#
# Panel A:
#   Active - Passive beta task-related edge change
#   validated wPLI²_debiased
#
# Panel B:
#   Active - Passive beta absolute node-strength change
#
# Frozen methodology:
#   N = 15
#   Active  = sub-001 ... sub-008
#   Passive = sub-009 ... sub-015
#   14 EEG channels
#   exactly 9 matched epochs/state/participant
#   2 s/epoch
#   beta = 13–30 Hz
#   MNE-Connectivity spectral_connectivity_epochs
#   method = "wpli2_debiased"
#   mode = "multitaper"
#   mt_bandwidth = 2.0
#   mt_adaptive = False
#   mt_low_bias = True
#   faverage = True
#
# IMPORTANT:
#   * No legacy Hilbert estimator.
#   * No dependence on Task6C summary CSV column names.
#   * No invented edge values.
#   * No edge-wise inferential claims.
#   * Sensor-space visualization only.
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import mne
import re
import warnings

warnings.filterwarnings("ignore")

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass

try:
    from mne_connectivity import spectral_connectivity_epochs
except ImportError:
    raise ImportError(
        "mne-connectivity is required.\n"
        "Install it with:\n\n"
        "    pip install mne-connectivity\n"
    )


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/Users/olusolaolabanjo/Desktop/eeg-data/"
    "derivatives/tine_analysis"
)

TASK3D = ROOT / "task03d_matched_baseline_task_epochs"

OUT = ROOT / "task10b_publication_ready_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG_OUT = OUT / "figure_3d_validated_beta_network.png"
PDF_OUT = OUT / "figure_3d_validated_beta_network.pdf"

EDGE_OUT = OUT / "figure_3d_validated_beta_edges.csv"
NODE_OUT = OUT / "figure_3d_validated_beta_nodes.csv"
SUBJECT_OUT = OUT / "figure_3d_validated_beta_subject_summary.csv"


if not TASK3D.exists():
    raise RuntimeError(
        f"Frozen Task3D directory not found:\n{TASK3D}"
    )


# =============================================================================
# 2. FROZEN DESIGN
# =============================================================================

CHANNELS = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]

N_CH = len(CHANNELS)

TRIU = np.triu_indices(N_CH, 1)

N_EDGES = len(TRIU[0])

assert N_CH == 14
assert N_EDGES == 91


ACTIVE = [f"sub-{i:03d}" for i in range(1, 9)]
PASSIVE = [f"sub-{i:03d}" for i in range(9, 16)]

SUBJECTS = ACTIVE + PASSIVE


def group_of(subject):

    if subject in ACTIVE:
        return "Active"

    if subject in PASSIVE:
        return "Passive"

    raise ValueError(subject)


print("=" * 78)
print("FROZEN DESIGN")
print("=" * 78)

print("Participants :", len(SUBJECTS))
print("Active       :", len(ACTIVE))
print("Passive      :", len(PASSIVE))
print("Channels     :", N_CH)
print("Edges        :", N_EDGES)
print("Band         : beta 13–30 Hz")
print("Epochs/state : 9")
print("Estimator    : wPLI²_debiased")


# =============================================================================
# 3. EXACT TASK3D FILE DISCOVERY
# =============================================================================

epoch_files = {}


for subject in SUBJECTS:

    group = group_of(subject).lower()

    expected = (
        TASK3D
        / subject
        / f"{subject}_group-{group}_desc-matched_baseline_task_epo.fif"
    )

    if expected.exists():

        epoch_files[subject] = expected

        continue

    # Controlled fallback within Task3D only.
    candidates = list(
        TASK3D.rglob(
            f"{subject}*matched*epo.fif"
        )
    )

    if len(candidates) == 1:

        epoch_files[subject] = candidates[0]

    elif len(candidates) == 0:

        raise RuntimeError(
            f"No frozen Task3D epochs found for {subject}."
        )

    else:

        raise RuntimeError(
            f"Multiple Task3D epoch files found for {subject}:\n"
            + "\n".join(map(str, candidates))
        )


print("\n" + "=" * 78)
print("FROZEN EPOCH FILES")
print("=" * 78)

for subject in SUBJECTS:
    print(subject, "->", epoch_files[subject].name)


# =============================================================================
# 4. CONNECTIVITY FUNCTION
# =============================================================================

def beta_wpli2_debiased(data, sfreq):
    """
    data:
        shape = epochs × channels × samples

    Returns:
        symmetric 14 × 14 matrix of beta wPLI²_debiased.

    Connectivity is estimated jointly across epochs.
    """

    data = np.asarray(data, dtype=float)

    if data.ndim != 3:
        raise ValueError(
            f"Expected 3D data, got {data.shape}"
        )

    if data.shape[0] != 9:
        raise ValueError(
            f"Expected exactly 9 epochs, got {data.shape[0]}"
        )

    if data.shape[1] != 14:
        raise ValueError(
            f"Expected 14 channels, got {data.shape[1]}"
        )

    # Explicit 91 undirected channel pairs.
    indices = (
        TRIU[0].astype(int),
        TRIU[1].astype(int)
    )

    con = spectral_connectivity_epochs(
        data,
        method="wpli2_debiased",
        mode="multitaper",
        sfreq=float(sfreq),
        indices=indices,
        fmin=13.0,
        fmax=30.0,
        faverage=True,
        mt_bandwidth=2.0,
        mt_adaptive=False,
        mt_low_bias=True,
        n_jobs=1,
        verbose=False
    )

    dense = np.asarray(
        con.get_data()
    )

    # Expected shape usually 91 × 1.
    dense = np.squeeze(dense)

    if dense.ndim != 1:
        dense = dense.reshape(-1)

    if len(dense) != 91:
        raise RuntimeError(
            f"Expected 91 connectivity values; got {dense.shape}"
        )

    matrix = np.zeros(
        (14, 14),
        dtype=float
    )

    matrix[TRIU] = dense
    matrix[(TRIU[1], TRIU[0])] = dense

    np.fill_diagonal(matrix, 0.0)

    if not np.all(np.isfinite(matrix)):
        raise RuntimeError(
            "Non-finite connectivity value detected."
        )

    if not np.allclose(
        matrix,
        matrix.T,
        atol=1e-12
    ):
        raise RuntimeError(
            "Connectivity matrix is not symmetric."
        )

    return matrix


# =============================================================================
# 5. DETERMINE BASELINE/TASK EPOCHS FROM TASK3D
#
# Task3D contains matched baseline/task epochs.
#
# We do NOT rely on event names such as T1_V1 because the saved Epochs object
# may contain only {'1': 1}.
#
# Instead, Task3D construction froze equal matched baseline/task blocks.
# We inspect metadata/event structure first.
# =============================================================================

def split_matched_epochs(epochs, subject):
    """
    Robustly recover baseline and task blocks from the frozen Task3D file.

    Priority:
      1. metadata columns if available;
      2. event_id names if informative;
      3. frozen matched ordering: first half baseline, second half task.

    Returns first 9 matched epochs from each state.
    """

    n_total = len(epochs)

    # -------------------------------------------------------------------------
    # Option 1 — metadata
    # -------------------------------------------------------------------------

    if epochs.metadata is not None:

        md = epochs.metadata.copy()

        for col in md.columns:

            text = md[col].astype(str).str.lower()

            baseline_mask = (
                text.str.contains("baseline")
                |
                text.str.contains("t2_v1")
            )

            task_mask = (
                text.str.contains("task")
                |
                text.str.contains("t1_v1")
                |
                text.str.contains("t1_v4")
            )

            if (
                baseline_mask.sum() >= 9
                and
                task_mask.sum() >= 9
            ):

                baseline_idx = np.flatnonzero(
                    baseline_mask.to_numpy()
                )[:9]

                task_idx = np.flatnonzero(
                    task_mask.to_numpy()
                )[:9]

                return (
                    epochs[baseline_idx],
                    epochs[task_idx],
                    f"metadata:{col}"
                )

    # -------------------------------------------------------------------------
    # Option 2 — informative event names
    # -------------------------------------------------------------------------

    event_names = list(
        epochs.event_id.keys()
    )

    lower_names = [
        str(x).lower()
        for x in event_names
    ]

    baseline_names = [
        name
        for name, low in zip(
            event_names,
            lower_names
        )
        if (
            "baseline" in low
            or
            "t2_v1" in low
        )
    ]

    task_names = [
        name
        for name, low in zip(
            event_names,
            lower_names
        )
        if (
            "task" in low
            or
            "t1_v1" in low
            or
            "t1_v4" in low
        )
    ]

    if (
        baseline_names
        and
        task_names
    ):

        baseline_ep = epochs[
            baseline_names
        ]

        task_ep = epochs[
            task_names
        ]

        if (
            len(baseline_ep) >= 9
            and
            len(task_ep) >= 9
        ):

            return (
                baseline_ep[:9],
                task_ep[:9],
                "event_id"
            )

    # -------------------------------------------------------------------------
    # Option 3 — frozen Task3D matched ordering
    # -------------------------------------------------------------------------

    if n_total % 2 != 0:

        raise RuntimeError(
            f"{subject}: Task3D contains {n_total} epochs; "
            "cannot divide into equal matched baseline/task blocks."
        )

    n_each = n_total // 2

    if n_each < 9:

        raise RuntimeError(
            f"{subject}: only {n_each} matched epochs/state."
        )

    baseline_ep = epochs[:n_each]
    task_ep = epochs[n_each:]

    return (
        baseline_ep[:9],
        task_ep[:9],
        "frozen-half-order"
    )


# =============================================================================
# 6. COMPUTE SUBJECT MATRICES
# =============================================================================

subject_results = {}

summary_rows = []


for subject in SUBJECTS:

    print("\n" + "-" * 78)
    print(subject, group_of(subject))
    print("-" * 78)

    epochs = mne.read_epochs(
        epoch_files[subject],
        preload=True,
        verbose=False
    )

    # -------------------------------------------------------------------------
    # Channel integrity
    # -------------------------------------------------------------------------

    missing = [
        ch
        for ch in CHANNELS
        if ch not in epochs.ch_names
    ]

    if missing:

        raise RuntimeError(
            f"{subject}: missing channels {missing}"
        )

    epochs = epochs.copy().pick(
        CHANNELS
    )

    if epochs.ch_names != CHANNELS:

        raise RuntimeError(
            f"{subject}: channel ordering failure."
        )

    sfreq = float(
        epochs.info["sfreq"]
    )

    # -------------------------------------------------------------------------
    # Split matched baseline/task
    # -------------------------------------------------------------------------

    baseline_ep, task_ep, split_method = (
        split_matched_epochs(
            epochs,
            subject
        )
    )

    if len(baseline_ep) != 9:
        raise RuntimeError(
            f"{subject}: baseline != 9 epochs."
        )

    if len(task_ep) != 9:
        raise RuntimeError(
            f"{subject}: task != 9 epochs."
        )

    baseline_data = baseline_ep.get_data(
        copy=True
    )

    task_data = task_ep.get_data(
        copy=True
    )

    print(
        "split method :",
        split_method
    )

    print(
        "baseline     :",
        baseline_data.shape
    )

    print(
        "task         :",
        task_data.shape
    )

    # -------------------------------------------------------------------------
    # Validated beta connectivity
    # -------------------------------------------------------------------------

    baseline_matrix = beta_wpli2_debiased(
        baseline_data,
        sfreq
    )

    task_matrix = beta_wpli2_debiased(
        task_data,
        sfreq
    )

    delta_matrix = (
        task_matrix
        -
        baseline_matrix
    )

    subject_results[subject] = {
        "group": group_of(subject),
        "baseline": baseline_matrix,
        "task": task_matrix,
        "delta": delta_matrix,
        "split_method": split_method
    }

    global_baseline = np.mean(
        baseline_matrix[TRIU]
    )

    global_task = np.mean(
        task_matrix[TRIU]
    )

    global_delta = np.mean(
        delta_matrix[TRIU]
    )

    print(
        f"global beta baseline = "
        f"{global_baseline:+.6f}"
    )

    print(
        f"global beta task     = "
        f"{global_task:+.6f}"
    )

    print(
        f"global beta delta    = "
        f"{global_delta:+.6f}"
    )

    summary_rows.append({
        "subject": subject,
        "group": group_of(subject),
        "split_method": split_method,
        "n_baseline_epochs": 9,
        "n_task_epochs": 9,
        "global_beta_baseline":
            global_baseline,
        "global_beta_task":
            global_task,
        "global_beta_delta":
            global_delta
    })


summary_df = pd.DataFrame(
    summary_rows
)


# =============================================================================
# 7. VALIDATE SUBJECT COUNTS
# =============================================================================

assert len(subject_results) == 15

assert (
    summary_df["group"] == "Active"
).sum() == 8

assert (
    summary_df["group"] == "Passive"
).sum() == 7


print("\n" + "=" * 78)
print("SUBJECT-LEVEL GLOBAL BETA SUMMARY")
print("=" * 78)

print(
    summary_df.to_string(
        index=False
    )
)


# =============================================================================
# 8. CROSS-CHECK AGAINST FROZEN TASK6C GLOBAL BETA RESULT
#
# Frozen Baseline-A fixed9 values:
#
# Active mean delta  = -0.002037
# Passive mean delta = +0.015948
# Active-Passive     = -0.017985
#
# Because Task3D baseline is Baseline A (position1/T2_V1), our recomputation
# should approximately reproduce these values.
# =============================================================================

active_global = summary_df.loc[
    summary_df["group"] == "Active",
    "global_beta_delta"
].mean()

passive_global = summary_df.loc[
    summary_df["group"] == "Passive",
    "global_beta_delta"
].mean()

global_contrast = (
    active_global
    -
    passive_global
)


print("\n" + "=" * 78)
print("FROZEN TASK6C CROSS-CHECK")
print("=" * 78)

print(
    f"Recomputed Active mean  : "
    f"{active_global:+.6f}"
)

print(
    f"Frozen expected approx. : "
    f"{-0.002037:+.6f}"
)

print()

print(
    f"Recomputed Passive mean : "
    f"{passive_global:+.6f}"
)

print(
    f"Frozen expected approx. : "
    f"{+0.015948:+.6f}"
)

print()

print(
    f"Recomputed contrast     : "
    f"{global_contrast:+.6f}"
)

print(
    f"Frozen expected approx. : "
    f"{-0.017985:+.6f}"
)


# Tolerance allows minor library/version numerical differences.
TOL = 0.005


if abs(
    global_contrast
    - (-0.017985)
) > TOL:

    raise RuntimeError(
        "\nThe recomputed global beta contrast does not reproduce "
        "the frozen Task6C Baseline-A result closely enough.\n\n"
        "DO NOT DRAW THE NETWORK YET.\n\n"
        "This most likely means the Task3D baseline/task ordering differs "
        "from the assumed frozen-half ordering for one or more files.\n"
    )


print("\nFrozen global beta cross-check: PASS")


# =============================================================================
# 9. GROUP EDGE MATRICES
# =============================================================================

active_delta = np.stack([
    subject_results[s]["delta"]
    for s in ACTIVE
])

passive_delta = np.stack([
    subject_results[s]["delta"]
    for s in PASSIVE
])


ACTIVE_MEAN = active_delta.mean(
    axis=0
)

PASSIVE_MEAN = passive_delta.mean(
    axis=0
)

EDGE_CONTRAST = (
    ACTIVE_MEAN
    -
    PASSIVE_MEAN
)


assert EDGE_CONTRAST.shape == (
    14,
    14
)

assert np.allclose(
    EDGE_CONTRAST,
    EDGE_CONTRAST.T
)

assert np.allclose(
    np.diag(EDGE_CONTRAST),
    0
)


# =============================================================================
# 10. PARTICIPANT-FIRST NODE STRENGTH
#
# Absolute weighted node strength:
#
# s_i = sum_j w_ij
#
# computed from each participant's task-minus-baseline matrix.
# =============================================================================

active_node = np.stack([
    subject_results[s]["delta"].sum(
        axis=1
    )
    for s in ACTIVE
])

passive_node = np.stack([
    subject_results[s]["delta"].sum(
        axis=1
    )
    for s in PASSIVE
])


ACTIVE_NODE_MEAN = active_node.mean(
    axis=0
)

PASSIVE_NODE_MEAN = passive_node.mean(
    axis=0
)

NODE_CONTRAST = (
    ACTIVE_NODE_MEAN
    -
    PASSIVE_NODE_MEAN
)


print("\n" + "=" * 78)
print("NODE-STRENGTH CONTRASTS")
print("=" * 78)

for ch, value in zip(
    CHANNELS,
    NODE_CONTRAST
):

    print(
        f"{ch:>4s}  "
        f"{value:+.6f}"
    )


# =============================================================================
# 11. EXPORT ALL 91 ACTUAL EDGES
# =============================================================================

edge_rows = []


for i, j in zip(
    TRIU[0],
    TRIU[1]
):

    edge_rows.append({

        "channel_1":
            CHANNELS[i],

        "channel_2":
            CHANNELS[j],

        "active_mean_task_minus_baseline":
            ACTIVE_MEAN[i, j],

        "passive_mean_task_minus_baseline":
            PASSIVE_MEAN[i, j],

        "active_minus_passive":
            EDGE_CONTRAST[i, j]
    })


edge_df = pd.DataFrame(
    edge_rows
)

assert len(edge_df) == 91


# =============================================================================
# 12. DISPLAY THRESHOLD
#
# This is NOT statistical thresholding.
#
# We display the strongest 20% of absolute descriptive contrasts because
# drawing all 91 edges makes the network unreadable.
#
# Every edge is still exported to CSV.
# =============================================================================

DISPLAY_PERCENTILE = 80


threshold = np.percentile(

    np.abs(
        edge_df[
            "active_minus_passive"
        ]
    ),

    DISPLAY_PERCENTILE
)


edge_df[
    "displayed"
] = (

    np.abs(
        edge_df[
            "active_minus_passive"
        ]
    )

    >=

    threshold
)


print("\nEdges displayed:")
print(
    edge_df["displayed"].sum(),
    "/ 91"
)

print(
    "Display criterion: top 20% by absolute descriptive contrast."
)

print(
    "Inferential meaning: NONE."
)


# =============================================================================
# 13. USE MNE STANDARD_1020 FOR ACTUAL SENSOR GEOMETRY
#
# This is better than manually approximating the montage.
# =============================================================================

montage = mne.channels.make_standard_montage(
    "standard_1020"
)


montage_pos = montage.get_positions()[
    "ch_pos"
]


missing_pos = [
    ch
    for ch in CHANNELS
    if ch not in montage_pos
]


if missing_pos:

    raise RuntimeError(
        f"Missing standard_1020 positions: {missing_pos}"
    )


POS = np.array([
    montage_pos[ch]
    for ch in CHANNELS
])


# Normalize geometry for plotting.
POS = POS - POS.mean(
    axis=0
)

scale = np.max(
    np.linalg.norm(
        POS,
        axis=1
    )
)

POS = POS / scale


# =============================================================================
# 14. HEAD RENDERING
# =============================================================================

def draw_head(ax):

    # translucent wireframe scalp
    u = np.linspace(
        0,
        2 * np.pi,
        80
    )

    v = np.linspace(
        0,
        np.pi,
        55
    )

    x = np.outer(
        np.cos(u),
        np.sin(v)
    )

    y = np.outer(
        np.sin(u),
        np.sin(v)
    )

    z = 0.92 * np.outer(
        np.ones_like(u),
        np.cos(v)
    )

    ax.plot_wireframe(
        x,
        y,
        z,
        rstride=8,
        cstride=8,
        linewidth=0.25,
        alpha=0.08,
        color="0.25"
    )

    # nose
    ax.plot(
        [-0.09, 0.00, 0.09],
        [0.98, 1.10, 0.98],
        [0.05, 0.08, 0.05],
        linewidth=1.0,
        color="0.2"
    )

    # ears
    t = np.linspace(
        -np.pi / 2,
        np.pi / 2,
        60
    )

    for side in [-1, 1]:

        ax.plot(
            np.full_like(
                t,
                side
            ),
            0.09 * np.cos(t),
            0.17 * np.sin(t),
            linewidth=0.8,
            color="0.3"
        )

    ax.set_xlim(
        -1.15,
        1.15
    )

    ax.set_ylim(
        -1.15,
        1.15
    )

    ax.set_zlim(
        -1.05,
        1.05
    )

    ax.set_box_aspect(
        (1, 1, 0.9)
    )

    ax.set_axis_off()


# =============================================================================
# 15. COLOR NORMALIZATION
# =============================================================================

edge_values = edge_df[
    "active_minus_passive"
].to_numpy()


edge_lim = np.max(
    np.abs(
        edge_values
    )
)

node_lim = np.max(
    np.abs(
        NODE_CONTRAST
    )
)


if edge_lim == 0:
    edge_lim = 1e-12

if node_lim == 0:
    node_lim = 1e-12


edge_norm = colors.TwoSlopeNorm(
    vmin=-edge_lim,
    vcenter=0,
    vmax=edge_lim
)

node_norm = colors.TwoSlopeNorm(
    vmin=-node_lim,
    vcenter=0,
    vmax=node_lim
)

cmap = cm.RdBu_r


# =============================================================================
# 16. DRAW FIGURE
# =============================================================================

fig = plt.figure(
    figsize=(14, 7)
)


ax1 = fig.add_subplot(
    1,
    2,
    1,
    projection="3d"
)

ax2 = fig.add_subplot(
    1,
    2,
    2,
    projection="3d"
)


# =============================================================================
# PANEL A
# =============================================================================

draw_head(ax1)


display_df = edge_df[
    edge_df["displayed"]
]


for _, row in display_df.iterrows():

    i = CHANNELS.index(
        row["channel_1"]
    )

    j = CHANNELS.index(
        row["channel_2"]
    )

    value = row[
        "active_minus_passive"
    ]

    p1 = POS[i]
    p2 = POS[j]

    width = (
        0.8
        +
        3.4
        *
        abs(value)
        /
        edge_lim
    )

    ax1.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        [p1[2], p2[2]],
        color=cmap(
            edge_norm(value)
        ),
        linewidth=width,
        alpha=0.82
    )


ax1.scatter(
    POS[:, 0],
    POS[:, 1],
    POS[:, 2],
    s=35,
    facecolor="white",
    edgecolor="black",
    linewidth=0.8,
    depthshade=False
)


for i, ch in enumerate(
    CHANNELS
):

    x, y, z = POS[i]

    ax1.text(
        x * 1.10,
        y * 1.10,
        z * 1.06,
        ch,
        fontsize=8,
        ha="center",
        va="center"
    )


ax1.view_init(
    elev=24,
    azim=-72
)


ax1.text2D(
    0.02,
    0.96,
    "A",
    transform=ax1.transAxes,
    fontsize=15,
    fontweight="bold"
)


ax1.text2D(
    0.09,
    0.96,
    "Beta connectivity",
    transform=ax1.transAxes,
    fontsize=12,
    fontweight="bold"
)


# =============================================================================
# PANEL B
# =============================================================================

draw_head(ax2)


node_sizes = (
    90
    +
    650
    *
    np.abs(
        NODE_CONTRAST
    )
    /
    node_lim
)


node_colors = [
    cmap(
        node_norm(v)
    )
    for v in NODE_CONTRAST
]


ax2.scatter(
    POS[:, 0],
    POS[:, 1],
    POS[:, 2],
    s=node_sizes,
    c=node_colors,
    edgecolor="black",
    linewidth=0.8,
    depthshade=False,
    alpha=0.96
)


for i, ch in enumerate(
    CHANNELS
):

    x, y, z = POS[i]

    ax2.text(
        x * 1.10,
        y * 1.10,
        z * 1.06,
        ch,
        fontsize=8,
        ha="center",
        va="center"
    )


ax2.view_init(
    elev=24,
    azim=-72
)


ax2.text2D(
    0.02,
    0.96,
    "B",
    transform=ax2.transAxes,
    fontsize=15,
    fontweight="bold"
)


ax2.text2D(
    0.09,
    0.96,
    "Beta weighted node strength",
    transform=ax2.transAxes,
    fontsize=12,
    fontweight="bold"
)


# =============================================================================
# 17. COLORBARS
# =============================================================================

sm_edge = cm.ScalarMappable(
    norm=edge_norm,
    cmap=cmap
)

sm_edge.set_array([])


cax1 = fig.add_axes(
    [0.10, 0.095, 0.32, 0.025]
)


cb1 = fig.colorbar(
    sm_edge,
    cax=cax1,
    orientation="horizontal"
)


cb1.set_label(
    r"Active $-$ Passive beta "
    r"$\Delta\mathrm{wPLI}^{2}_{\mathrm{debiased}}$",
    fontsize=9
)


cb1.ax.tick_params(
    labelsize=8
)


sm_node = cm.ScalarMappable(
    norm=node_norm,
    cmap=cmap
)

sm_node.set_array([])


cax2 = fig.add_axes(
    [0.59, 0.095, 0.32, 0.025]
)


cb2 = fig.colorbar(
    sm_node,
    cax=cax2,
    orientation="horizontal"
)


cb2.set_label(
    "Active − Passive beta node-strength change",
    fontsize=9
)


cb2.ax.tick_params(
    labelsize=8
)


# =============================================================================
# 18. SMALL METHODOLOGICAL NOTE
# =============================================================================

fig.text(
    0.5,
    0.025,
    "Sensor-space descriptive visualization; edge display is based on "
    "contrast magnitude for visual clarity and not statistical significance.",
    ha="center",
    fontsize=9,
    style="italic"
)


plt.subplots_adjust(
    left=0.02,
    right=0.98,
    top=0.96,
    bottom=0.15,
    wspace=0.02
)


# =============================================================================
# 19. EXPORT
# =============================================================================

fig.savefig(
    PNG_OUT,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(
    PDF_OUT,
    bbox_inches="tight",
    facecolor="white"
)


plt.show()


# =============================================================================
# 20. EXPORT EXACT DATA BEHIND FIGURE
# =============================================================================

edge_df.to_csv(
    EDGE_OUT,
    index=False
)


node_df_export = pd.DataFrame({
    "channel":
        CHANNELS,
    "active_mean_beta_node_strength_change":
        ACTIVE_NODE_MEAN,
    "passive_mean_beta_node_strength_change":
        PASSIVE_NODE_MEAN,
    "active_minus_passive":
        NODE_CONTRAST
})


node_df_export.to_csv(
    NODE_OUT,
    index=False
)


summary_df.to_csv(
    SUBJECT_OUT,
    index=False
)


# =============================================================================
# 21. FINAL INTEGRITY REPORT
# =============================================================================

print("\n" + "=" * 78)
print("FINAL FIGURE INTEGRITY")
print("=" * 78)

checks = {

    "15 participants":
        len(subject_results) == 15,

    "8 Active":
        len(ACTIVE) == 8,

    "7 Passive":
        len(PASSIVE) == 7,

    "14 EEG channels":
        N_CH == 14,

    "91 unique edges":
        len(edge_df) == 91,

    "9 baseline epochs/participant":
        all(
            x == 9
            for x in summary_df[
                "n_baseline_epochs"
            ]
        ),

    "9 task epochs/participant":
        all(
            x == 9
            for x in summary_df[
                "n_task_epochs"
            ]
        ),

    "validated wPLI²_debiased":
        True,

    "multitaper 2-Hz bandwidth":
        True,

    "beta 13–30 Hz":
        True,

    "symmetric edge matrix":
        np.allclose(
            EDGE_CONTRAST,
            EDGE_CONTRAST.T
        ),

    "zero diagonal":
        np.allclose(
            np.diag(
                EDGE_CONTRAST
            ),
            0
        ),

    "frozen Task6C global beta reproduced":
        abs(
            global_contrast
            -
            (-0.017985)
        ) <= TOL,

    "no legacy Hilbert estimator":
        True,

    "no significance-based edge selection":
        True,

    "no source localization":
        True
}


for name, passed in checks.items():

    print(
        f"{name:<45} "
        f"{'PASS' if passed else 'FAIL'}"
    )


overall = all(
    checks.values()
)


print("\nOVERALL:",
      "PASS"
      if overall
      else "FAIL")


print("\nSaved:")
print(PNG_OUT)
print(PDF_OUT)
print(EDGE_OUT)
print(NODE_OUT)
print(SUBJECT_OUT)

In [ ]:
# =============================================================================
# EXTRA-LARGE PUBLICATION VERSION
# 3D beta connectivity + weighted node strength
#
# VISUALIZATION ONLY:
# No numerical values, statistics, thresholds, or connectivity estimates change.
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("png")
except Exception:
    pass


plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# =============================================================================
# HEAD
# =============================================================================

def draw_large_head(ax):

    u = np.linspace(0, 2*np.pi, 100)
    v = np.linspace(0, np.pi, 70)

    x = np.outer(
        np.cos(u),
        np.sin(v)
    )

    y = np.outer(
        np.sin(u),
        np.sin(v)
    )

    z = 0.92 * np.outer(
        np.ones_like(u),
        np.cos(v)
    )

    # Very subtle scalp grid
    ax.plot_wireframe(
        x, y, z,
        rstride=10,
        cstride=10,
        linewidth=0.18,
        color="0.45",
        alpha=0.055
    )

    # Nose
    ax.plot(
        [-0.10, 0.00, 0.10],
        [0.98, 1.12, 0.98],
        [0.04, 0.08, 0.04],
        color="0.15",
        linewidth=1.0
    )

    # Ears
    t = np.linspace(
        -np.pi/2,
        np.pi/2,
        80
    )

    for side in [-1, 1]:

        ax.plot(
            np.full_like(t, side),
            0.10*np.cos(t),
            0.18*np.sin(t),
            color="0.30",
            linewidth=0.8
        )

    # Tight physical limits
    ax.set_xlim(-0.94, 0.94)
    ax.set_ylim(-0.94, 1.00)
    ax.set_zlim(-0.87, 0.87)

    # -------------------------------------------------------------
    # THIS is the important change.
    # zoom > 1 enlarges the rendered 3D object inside its axes.
    # -------------------------------------------------------------
    try:
        ax.set_box_aspect(
            (1, 1, 0.84),
            zoom=1.75
        )
    except TypeError:
        # Older Matplotlib fallback
        ax.set_box_aspect(
            (1, 1, 0.84)
        )

    try:
        ax.set_proj_type("ortho")
    except Exception:
        pass

    ax.set_axis_off()


# =============================================================================
# LABEL OFFSETS
# =============================================================================

LABEL_OFFSET = {

    "AF3": (-0.015,  0.045,  0.080),
    "AF4": ( 0.015,  0.045,  0.080),

    "F7":  (-0.050,  0.020,  0.035),
    "F3":  (-0.050,  0.000,  0.100),
    "F4":  ( 0.050,  0.000,  0.100),
    "F8":  ( 0.050,  0.020,  0.035),

    "FC5": (-0.065,  0.000,  0.040),
    "FC6": ( 0.065,  0.000,  0.040),

    "T7":  (-0.090,  0.000,  0.000),
    "T8":  ( 0.090,  0.000,  0.000),

    "P7":  (-0.075, -0.035,  0.010),
    "P8":  ( 0.075, -0.035,  0.010),

    "O1":  (-0.045, -0.065, -0.005),
    "O2":  ( 0.045, -0.065, -0.005),
}


def add_large_labels(ax):

    for i, ch in enumerate(CHANNELS):

        x, y, z = POS[i]

        dx, dy, dz = LABEL_OFFSET[ch]

        ax.text(
            x + dx,
            y + dy,
            z + dz,
            ch,
            fontsize=8.3,
            ha="center",
            va="center",
            color="0.08",
            zorder=50
        )


# =============================================================================
# SCALING
# =============================================================================

edge_values = edge_df[
    "active_minus_passive"
].to_numpy()


edge_lim = np.max(
    np.abs(edge_values)
)

if edge_lim <= 0:
    edge_lim = 1e-12


node_lim = np.max(
    np.abs(NODE_CONTRAST)
)

if node_lim <= 0:
    node_lim = 1e-12


node_sizes = (
    75
    +
    500
    *
    np.sqrt(
        np.abs(NODE_CONTRAST)
        /
        node_lim
    )
)


node_colors = [
    cmap(node_norm(v))
    for v in NODE_CONTRAST
]


# =============================================================================
# FIGURE
#
# Much shorter canvas.
# Heads occupy most of the available vertical space.
# =============================================================================

fig = plt.figure(
    figsize=(12.0, 4.1),
    facecolor="white"
)


# Left panel
ax1 = fig.add_axes(
    [0.015, 0.20, 0.475, 0.72],
    projection="3d"
)

# Right panel
ax2 = fig.add_axes(
    [0.510, 0.20, 0.475, 0.72],
    projection="3d"
)


# =============================================================================
# PANEL A — CONNECTIVITY
# =============================================================================

draw_large_head(ax1)


for _, row in display_df.iterrows():

    i = CHANNELS.index(
        row["channel_1"]
    )

    j = CHANNELS.index(
        row["channel_2"]
    )

    value = float(
        row["active_minus_passive"]
    )

    p1 = POS[i]
    p2 = POS[j]

    linewidth = (
        1.25
        +
        3.5
        *
        abs(value)
        /
        edge_lim
    )

    ax1.plot(
        [p1[0], p2[0]],
        [p1[1], p2[1]],
        [p1[2], p2[2]],
        color=cmap(
            edge_norm(value)
        ),
        linewidth=linewidth,
        alpha=0.90,
        solid_capstyle="round",
        zorder=5
    )


ax1.scatter(
    POS[:, 0],
    POS[:, 1],
    POS[:, 2],
    s=38,
    facecolor="white",
    edgecolor="black",
    linewidth=0.85,
    depthshade=False,
    zorder=40
)

add_large_labels(ax1)


ax1.view_init(
    elev=25,
    azim=-72
)


# =============================================================================
# PANEL B — NODE STRENGTH
# =============================================================================

draw_large_head(ax2)


ax2.scatter(
    POS[:, 0],
    POS[:, 1],
    POS[:, 2],
    s=node_sizes,
    c=node_colors,
    edgecolor="black",
    linewidth=0.85,
    depthshade=False,
    alpha=0.97,
    zorder=40
)

add_large_labels(ax2)


ax2.view_init(
    elev=25,
    azim=-72
)


# =============================================================================
# PANEL LABELS
# =============================================================================

fig.text(
    0.018,
    0.965,
    "A",
    fontsize=14,
    fontweight="bold",
    va="top"
)

fig.text(
    0.044,
    0.965,
    "Beta connectivity",
    fontsize=11.5,
    fontweight="bold",
    va="top"
)


fig.text(
    0.515,
    0.965,
    "B",
    fontsize=14,
    fontweight="bold",
    va="top"
)

fig.text(
    0.541,
    0.965,
    "Beta weighted node strength",
    fontsize=11.5,
    fontweight="bold",
    va="top"
)


# =============================================================================
# COLORBAR A
# =============================================================================

sm1 = cm.ScalarMappable(
    norm=edge_norm,
    cmap=cmap
)

sm1.set_array([])


cax1 = fig.add_axes(
    [0.080, 0.105, 0.350, 0.032]
)


cb1 = fig.colorbar(
    sm1,
    cax=cax1,
    orientation="horizontal"
)


cb1.ax.tick_params(
    labelsize=7.8,
    length=3,
    pad=2
)


cb1.set_label(
    r"Active $-$ Passive beta "
    r"$\Delta\mathrm{wPLI}^{2}_{\mathrm{debiased}}$",
    fontsize=8.8,
    labelpad=3
)


# =============================================================================
# COLORBAR B
# =============================================================================

sm2 = cm.ScalarMappable(
    norm=node_norm,
    cmap=cmap
)

sm2.set_array([])


cax2 = fig.add_axes(
    [0.575, 0.105, 0.350, 0.032]
)


cb2 = fig.colorbar(
    sm2,
    cax=cax2,
    orientation="horizontal"
)


cb2.ax.tick_params(
    labelsize=7.8,
    length=3,
    pad=2
)


cb2.set_label(
    "Active − Passive beta node-strength change",
    fontsize=8.8,
    labelpad=3
)


# =============================================================================
# EXPORT
# =============================================================================

PNG_LARGE = (
    OUT /
    "figure_beta_network_large.png"
)

PDF_LARGE = (
    OUT /
    "figure_beta_network_large.pdf"
)


fig.savefig(
    PNG_LARGE,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.025,
    facecolor="white"
)


fig.savefig(
    PDF_LARGE,
    bbox_inches="tight",
    pad_inches=0.025,
    facecolor="white"
)


plt.show()


print("=" * 70)
print("LARGE PUBLICATION NETWORK PANEL")
print("=" * 70)

print("3D camera zoom:          1.75×")
print("Head limits:             tightened")
print("Electrode markers:       enlarged")
print("Electrode labels:        enlarged")
print("Node markers:            enlarged")
print("Colorbars:               moved upward")
print("Numerical results:       unchanged")
print("Statistical inference:   unchanged")
print("Display criterion:       unchanged")

print("\nSaved:")
print(PNG_LARGE)
print(PDF_LARGE)